# Nutrelle — treino em GPU, sexta execução (seção 15: protocolo no recorte; uma única foto de cima)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py`; **não edite à mão** (edite `treino_gpu.py` e gere de novo). Lê a saída do kernel da etapa 0 (`nutrelle-deteccao-r5`, anexada em *Input*), `listas/conferencia_r5.json` (a regra e as bases da conferência a olho) e `listas/ref_r5_prato.csv` (R4c e H0c fora-do-fold da quinta execução, para a paridade). Refaz as features do So400m no recorte, a R0c, a R4c e a H0c (paridade), e treina o P3 (com a HP3), o F5c e a H0c-m de 15.6 até o orçamento; grava previsões cruas (fora-do-fold no treino e nas partes de registro das bases externas, **sem pontuar nada; nenhuma do teste2**), os arquivos para servir e `servico.json`.

**Versão do repositório: só bases liberadas** (Nutrition5k, Flavoria, LeFood-Set, GT Mensa). As configurações que usam OmniFood8K ou o Food Portion Benchmark ficam de fora e o manifesto diz por quê; o SNAPMe também. A versão completa é montada fora do repositório com `python3 monta_pacote.py push <pasta>`.

Kaggle: *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Instruções completas em `eval/treino/kaggle/LEIA-ME.md`.

In [ ]:
# Parâmetros (os padrões servem para o Kaggle)
import json, os
RODADA = '6'         # este pacote só traz as listas desta execução
ORCAMENTO_H = 4.5    # horas de relógio; o que não couber inteiro não começa (a ordem de prioridade decide o que fica de fora)
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_saida'   # no Colab, aponte para o Drive para retomar
EXTRA = []           # ex.: ['--so', 'ridge']; ['--dominios', 'flavoria,mensa']; ['--retoma', '<pasta>']; ['--espelho', '<pasta com o zip do OmniFood8K>']
# as variáveis de ambiente NUTRELLE_SAIDA e NUTRELLE_EXTRA (lista em JSON) substituem os valores acima sem editar o notebook
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
# treino_gpu.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/treino_gpu.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<FFy`{#^hOh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRyx75<tfLmlr9v8=rA^vK9U_wKSxGTYDxj-jWATV6)EKC(Hm
YNGRX;bmU+;6s%e2My;AdQ+%8O$p?ZCbjDzN$>!xfgl28Q=I#TvJTAmr^77~D@Zu{uA8j}=Ah_;`+(Wh7Pf7bceFsb$RGZ$;$m)?GKB?9<~?!B1vArY#MT#@
mIAho^Z=^{$g_MwVIU6R+-vlNl8?BI_hHwb%$T%+eNqIhigtVv!`j{?O7Q+YhT=3|SaqgO8$j!W`>k|m9zhNv0uDp5Xr3Q^6;J_^66184ImAO>Ppp50eUOUl
VD&>9$J+8n9FytF5)CM~Sc#tdYxtF`hWZz|<d_JaC{_Y4R^}Ij(EC{*VE_;CXsX}VhcYbuwO8~%Y#jlw@lc?7#bpik-7eRYpnBU+;&;cT?6a+Jgm6*e!rBs;
mKwl`A@b7S_1st7`0v2=*3SJ@q!Ng+Dz436NEB|wU%m8E96_)<P-l^1)8Z?<H?6y_C|Pe<q{q@wqXQX!dJ#C~>+^DJ`dry)O_26bgb{w+1LNuP?4__ckTBNH
8ZM%F+=GRyv@U9Xriv?F0cc*hfgBhv0A&Xfo0dg?gCwhr_Nv2Cvv2xm=Kg~d9_%egGaM2gHrbpRp<p_^7`FByodDM#@>~_1{YUjb^lk%zA9{jw^h9RP*#)bp
rqu7<)>u@JT`MoDkSEXn+zcToA|eb4jJ7QgSU)TBk`U1mEvwZ1d1Mo_5dBC793jB&#7;t7sK70M5}=QTo5t;EWX_fVIw*rY2fC=vCunV^cn(e~faHkK9yb>I
-ID}rWTL!B3<fDBL}uT+Am2>PTAH~+P5RN;$qv?)0KjM&1k&KG3`=wOe3?b46*LwXfw_$!Vv!Z7;tV|2_ZS5IwTQz68?1dAI~=u$OR7y;MGV+<P82%<so7rv
-GxjvyNMxuznP@-EK#>sRosEsNu)Cb1xW%JhNWLu8%%20`kE<9yU7d8@{j{|D6Hq}(Sxywk?hubJq9N4gaTO3$1JX%$W^`NwyMy(s4w1|3h|Jjie6d4YB~Q2
of6stzib(J%>po^qmT#yz3z>W;)88Gf{nX}+&Ym+aeZ+^Y?!J%A~&m$b6|@`AExp#Y%ozq@OA2TY%RD~p1ZZHQ6C>a|BD$+pI7iK5y#?tG9efCY4~UAm^w?C
OGEcuiTXKYQwMR3oZyhdL5qLcA%K2niCI?4&)(T|pJhn6IO}y!0v>kldb($B;WkYdtN}v~tEAWpqbx=NNUYM=>y_!`HMYg#tL7FsfKJGq9TkJH1e5+gtwYkp
jB!kU=|OcOdtkrqqD4H-7p5(EjZpXT8}cI4Pql@8Z@^u1d^XQV7C?Yc!3#W>99B+#I1A2|Xb||1>LcZ+tyg?kNlqg$-4WLW2R>3T$;d_T8HR&?CVMY*nq^RB
9yS6dF*U;T0wwZ%7OvJeg)K_p9ilWw$RyGtU91@qI@yf$P2civ_bRp1ys7pRn>B@~7yShI4Q&?SUCX<DVwkStv)|rJBr?eS4bd=He@kXdP`5)ax7&I_Fw-iB
2e{{U_hvt`P{yBka#iD3@BJIeaP8g`>8gEpxSl*+5bB%-u<>JkfGN!F=ZW12Echb0!#=a69`qoE5I=rHr8KO}`TOXY;*%}e=b47B*N0mlSXAZZFH{e=cd~}X
iAW=wv$>OC1EHtIqXxG*;?FbQ8uLSHZ;!tRXa3b*KAmyB_R_J>J`~jFp}$TN>ziKC;Dl$!ayk{@&-oWm%JoTAuOz3Pd2m+YksbbT%oa6Qt0XCJy1V3&OtYd#
n-R0w1kM{WRJ!S|LL8KftvrA8E$E<sjjMV<N8Ye>GKMM{)7{QoFeL?t87cO<sYGTba<<wZH2IMl%p<p7^ai(hQxey)wuGTd+DS&SU(2vYV&;<h@3yi&$^(jo
;K<E9ZE!>yioi(FNzhsYy|9ScmyQA12T6`Vgi0h1?o}T~@JT$k6_+;-$TFHtOK;i(>@pWz+*2z=Pv-1}+-R~rcK}LyVPSP1`rYkS<!}%dGRF(65<E>n#^0xr
q^$WO91}M%uGWupdWcuD=m!o;D;6l04;1!#w53s;#yv{!X$#7G*gm@eKLjrZfi@sviNY>kKGQ0EX&`b=1WwLGQ)zG))!P9xm_%oQvBJSy>``!3V2GQ}De~ns
>O4~Hfw?uXr4mzU2!d_H$*NQuyVAhh*jJkj0G-n~5MhQ!9M}0jgfs6Vdy1Fv9byO@<t1j3)H6pwaxaCC-+MS}6S)8KURSZC9{Brq3oH)=l!?6c9ajp-G4GX|
0n}J+0yAZGnkM_(Be&M%(6`i^Qw*6UGUT8Ev*ql(Sc50Bm}apCPfb?E%rNgPc%?b9TI7udl~+O~%ltQ%_)RQFW~x|>u6p>FUb|Vo%&@V<XC&Bk5(^*O<p!9o
qg5)q<YbL5g7jb|is7$th$%LF77QguoV#U4x6&pgXD2+F&Oo&VBSI{<%^onm8j}cCc=|Iyx<oqfOi;=4@uvlCw;x$K7~al{Xmy`bHemR5Zq+182Y+qt>x8N3
hI^hgk>=YA!uWIEw|O!wlAV9)FT4<$r5sn%&Otwt%o;FK)qqWrZOAtpB3}LlfuJPdw!m@vdq__VP+9MZ#F?|`OPt~83=+tk%|4(+sU#GkWk5}1>!)<m$8(7m
f$}tPq0E}CCs5=H4d^<7@J?nWIu;>5l<cIL9RSg{{a%}Fk6ouKs-X=!_wcHhwADSa&aq4X>t;b#i5cWt!J9jm@2i^DWFqS1D!^uuY8zQ=auvHTVaPa$S}yh(
W~AXBDLQRItIayu6FIn7ms#emS9@jOc|r;zm5Z>%@mxWSEMW?l!}T@V2!{|ExV@e3D4aG9&8u<&+@ak|(x<Q98xR`7d(jF<7g&&H2BtrAs+$0{{C)DABifSa
e9`SVYzg2Y#6=oSii#YFZ;41DB%MeyUE7!ccJsM*i4A<2W6Vf(G<4z}Ob3YeCxbf9w69tv`LN2y@Xf!+bqbne{i=+baOMq`Kmt8N5hzH1;{6kk;EN%a;g1{|
a#~RSl<bc{%c58GEEAA+XQ9)m5?<Szwmdd~8DLG_LaaUq$$80SkQtsl0e`Dy&z&}n6>Ma50P$(0J)}7%Y3QekKn`OGx-gYaM3?VKhen{h{trnQsv?<*FdwIH
=rVa3X$xf?@O}n5GfjS=RS%(k%WS+C6~5(1yC|$AD6~oG0A%*J&VegTF4}$y0b^x`?W%8cKX@~pZFEPD?nL~z`)oXf;wty&=<xbXj?=>&tCrubyxEcIC|Bfd
GbJ`<1|FPmT5j4MAT1=&X3{V66OPf{o*T=Lce=l=SBlrN7ixzX2J+vdCNah@XbESIoa6>Ds*I(b<x6_upyX~1PL^%D&YK%=wkgWCu%&y~x7|t!%j^px5u>9~
y>O-syynWAaY5O_$||3v8EOolx2|!vkk3?$`!CuuTpX)m=b<UCJ|-T-5srw`9v)p?g;BAHh5T&B3rxc|v{qA;ZuLLhpKhZce3}j4JVzCZA=INRR=1_E%;8l;
6&shc3!WV%;h+K>8jg_QcBj>Y4g2Ox-XEx>3*x^!LdPtx`gsMfO2?=Kf#=V#m9V3dx!Pp2D;uT8Mh4*ZP&sPiCUsUQ)9jlX=pT=rU}j=L+f}qcDAs;zf~jI*
vnK1)@z{h8lldod(;grZiQ9FT&vE)Z_Gu2)ty@Qv0L}v-6R8IQ!1dMS1;yJQuB9&|9DX)riTH{j_)P$_l8s*E#xlFdL*-R@K$gLNbMQeI`*~qSl>LFh^wYjv
o$9)DyrR6ERcwPPf`}DS8$P%tM|H$@^yyzm<j8@P%)cb3?=}+A2xW*ZkirtUjCe(~1;ArmN5ScgnRE+bWCWB-z0(RNBt^M3;h*`4T_Qe1EduC55L+ablaOOB
_dr%`pIuHaSL`F5M;6$FLF|E7gRUNu@ExD6B#)rujBcKH$6m*W<^Kxe=<<AZm*gr59{ONYUpi3s7GFR4*WxW56v+rJ4PAl&DLi7GQ%9h3rNKzWdlh|cSd`yk
*c4lbBTV6DRL<MdC}j7oZYN@+WvtoCy;1Lfj2~wH_0pRKpe69aLEO#BNLx1#RdTR%$bjnD#8j$)#Co#tfWA6AA1gNE@wVbw(UM|7q_0BtYnq(XMO#)eS_Ov)
+!vN^y$;1CPKDDp(I;cJa7yFmYxyJL2WGf?i9R8r&DeJg_#IiYr>}uU+Lo1Bj%p2v3FjLNqn-V>MT_C*u9He+)Lv-)3U4~7*4Swy0*cLG=E)cz%^R2+)xLiA
GVSx}d+1(+CN#u;4@tr}pYsc^WE&jw7ft}t+x-{%9@f|d?^FV33*4Bjs{c#-3HU8?-FSLC;zr#h7L4#l!caFpQxvtJRPspQ!~~>Oy-_9p<D%uiM_-46F#@cs
utv*mJdLE+1(xVyUp$G)zCXeL_DW9W%API6)*ttb@V|TdhKF+eeB<a32$E&y%^3QnLF?|n6z_dcv7-K(UQ2*-#(Tw#B?G~4FlvB+lHDk|0w)?%g}61@DbfNZ
vyJf6hIJ*xSis)0s8_!oi&7Lv_2&&wF3wN&PgtLc<wYf1!Av=bd6;KwN`Sz4bws|UVg-l?km>Vi-<yy#(Ya7m`3<QMqP{%uw^=exX3UY|Qp+RE0tRs5NI3~5
#XWDUvXU^X6)gT;FfeZF<JogDX`2o0mjc#mK(E-qfeZBv;aqVGQCVeU6NN)WAN8+(z19)n(M_IRdMNwk-GFVlVYmWu5I)fI!`^rR3I`%7MbiixdhBe~gduv7
R&rc`vO1s^%&Xu$9$0<sX)E6q$|ub`Il*+@Ni~nKjFebf)1)caMB<gXZzsom1jL4pCejx7?1!b2Gd0Ue>PSS_f^=4)1yK%|FMdMT&?kf?<J8a$%zY`KQjrGS
v2^&papSq?(ce0IROH$A{@DM`mdm$r23dM-B9eyT!%if;iv~tQYa+)e#Zl{P7^lUF<9`&@ht6u9qYOw18a*1J*m$#0YRG&<DthtWu;J>zPA(;pGwjspnDCN^
iD@WyLpx&llY@Zddgmyj1-Z<_Ryc{?h@q=@rjpP^q-g}H3nc4%9u2-tJ!a@#rka@W*wGUdBS!YMAV~B2=QNr%E+Ebhtl^MUc931GqBO2&BQKW{Thsbhk01pC
M3qN~)M<=pg~918<+dTO;>HMe#S*7Bv|D3Fl};yJW;00K;M4O_z{3qq6Eh#RlDwbT@7EB3qeDc>NO#^rFEFj#{h%%S_gKILRE2o39f!mkEZkzK`r1eS6MVUm
hk9BP-W^*=dSboeWk5(rLh~%LA;q1^1Yi(n+ySPIlqa{D0qw(9A>v9KE^^}`%#pdEY~eP^9Gyn}uw$anhQ2WkPjN~k^GAoNO<H6G+`*xKHmpt1H%nTr)I5Wh
>Vb|SVjYT3Y*w!7U(aud5j-O2HKm848g8<gR|xtLDpeP^;}H8JOSY)v0V1wH<^yWolvml>$?v;Yib>A8PaqEOsCf%N``qOG2cN$)FYspo&UJ>E(Lpg~x(4rz
`f)^?)#97xLX)L&t6Aqf{g0+uu96n*Jw@s1(C+zqXw61v*j}sa0TriZ%X^B(bziEG<M88uDce;|<ITF34g^^ROG4s_DFa65ir%t}?So+os1X1)D;qnV89t$+
TwRaO!Zkrxt#Tm&LRa95oZ5_N;;up9a=woHYiM<z@TzKYibW}rteiolMjFm0jrG!4w&UITGdocsp@Wtm`?w91=icRncgyZ=|8Tj9ffz^H1ax|b4&*9d?hgPN
LpLcoJM&rpgWe^Mdx>z<CG4*qYuqDj>}pp_E%+duN!l?hSs^POtHJvNZdXZTFg$xqtWl9X$nVO`I{kn>fUrJwzo|<>J>eZ~b8~`wzA6T}rhfD(@~b<0M#0{g
ASB&Xf<NyiP8B#TU*z6J25y>(UwI6nrOP37YYVaT0TX@YcciyrTCH&jzS~2JYYP9L#eKoCE8R^kFrd=93sOzk%RE6hYWCEs6>}Ri^etVL+)x3=hxm~1T*CYs
Uuq$!$M1L%qL<*gPsFaIX(b50NTML@9>&8-x+Vd&_ch{3)=fIyp$KkK!U_RbJ8VNb=wBcZ&2xa<vtMWf5Mb_+pES0uFskch7@>~Jm;iW4Mj@O{!>v#iXg#&e
t_!s-M<f!k|F+0g+?j=D!*aXx0quOWzJ+2!MRPz6FW*f;r~p~i0c62IPd;@ZE3h~&l)NCZcCjStX!+TA`Y6^0icw&l#rhiSy^p=;?JQr(4tTukR;Wv3iSV1p
p@;-rYs~|Hr|s*#>fNhtVi_qiyL{;~wc~wA@EFU>f&?(4YO8faJY>!}`$pPX9K8Q2sqsX1CFo7tG}lIRiyD){GyNWb7pSj9AJ;|n$gpp{5kX$)$wjB;_@w<Z
0&6`sFJPC)mo&;nNR{}GR_G8`S1iduf{Mn%O%n8o%W@$f2c9t<^N$m(I855xVx(OP%rESTkYsD8aUGr+p-uuk^|gZU1bkYKl{{s!8RIt@p!OX%I|;NJL#j@x
NABd?H&Ydi=$=k<Ogp}0|ChwB1X%0^9VZ=aNr5Tf44S7^cxIY`I{qpMQVJMjb9FdTRuYv?<>E4oN6UkIi}$pA-Mnh&IBYRu7@E=T%zPYI7a>4$^Gp$o6n_Jj
?~06Solx))GGR`56teY-$DI59IY3?>ki@{kIpOLI9Fr_Xq=#a)vhbgJ1``mwOCUcw2ti};;<I$wmNog>;dJ>D!fbhU@fLFGyFh(LVS?gy6sI5kviSEUMJSfJ
--JMVKgjFH?EOs4BYL2BDx0G~$$+)^T>Y!-2gXA+L@ArM*7PMqDatr@Ed5U4LNe>G<mVGqm_^P%xt-6S1aZzH^Je}MFrW5(``}pT@y07GX`zypKYy=EiWK7#
R!WL1z$(#erOoR_N}HZ}6$14>0p+yJqR*8U_>bes7p&o1BDpB4T*KU6S+9w51k4T}s8NIMM_B&a7uG^4O9p@el*|%%>$jp&5H><3%`fjnUIK@u3T=Un(rb4$
YE=^aRTu$wNUyM4*!_$7BFQb~*lY>ll6-$Kj=Wdi3TFexCb-k4TOP22l#UZOf8`USyyf;rvSOW{#v=y)xTEcru$=R3!aa$P_5ho?MAt#k!noFcN`n~>h16Sl
k-4T9uP>(!Axp?V)xc|{VLtGo;!2)mGmNOx1}<y87{DfUYT1f}G-qa$uT107ubgCq0_+ixf54~zBg?kaKOu>!*d;07FJu0Ga>9=ji57KyjDdbY;}`5%frN4c
M2jaaaHmVF-gi15e*<dyW2N)syTD_bgVHXIA<9&S|K96Xd&!RE+Se_h-7yY3E;%3bJrhp7>K+1U`xq4F<CdQ}Qsm+ob`pUz_oJ+LCT&C0ZTR2?^?scrqG*A&
%Z20))JBD+lwSLK@X1NpTuLPM(H@cSn*n8l<v_*$Bc#?1e9}coIuoZ6uv=n0sRYj0HrvUDjwuqv%XWSL**M+gDp-KhUJoWEi`EBv>dq7x^lfP6Rt$DuS+&t|
Fu_k9s4U+5x>q7~nwSbm<*~s|AbQz25Qyy}CU^P+J1U9>hmCa6f3$h>!a-w?OKa?;f?7P7``Tta5?u^tLFx4#xc9^=B}*W15#Q1D2s?=$xv7j@0oc1>KD25z
B`*{$@`;+AL^n{OA>Hkven_>%hlQ$~kX-Q*>+>7V@h-s9=kwuz!8e(2{b{c`b9^)Z{zjvO{RY0>3zRULytzE4`30`T4;!`7CzDL^?n?H8+9FcpJi-3xp;a-V
vL{{a?5pZ8#XsQx0^p5M(9PKVC@ZE$!KvoDQs$%kv+(%_1v^FJw%TA}f)X3F3Tk$_L-%!Oy!xH<y?_~f;-y2$#6og#TOq(zTiRmnr)D@P1s((7o9&EQvM6<9
PBqj}?!t@SHzzsr<UHpY+Ww`qBMTuce~gt=XQO!OUH*b5Ntk5>-xl&l7M5Jdv%q>fiEXy}#XTz`$tLP3i}sfPF2XTe8g#0<Q~wLUCx3IM7hoy5V{PJ&mAZ0!
?kiAJ=6--3EWwluIIv{3nlNvwJ_HRhUY^h3<{VeSGb;1Wo%O#cD!_LE6%mBET^5FWR}u^K<D(CQAgmdhkST9GV`FV4Ebp*$g1YKMEZ<WWqH?r1#^Dfl+POF$
%UWp_d4YYwvl%Dhz$y#U&v2xg9Cx|PCJ6!yYwJ=))AzMeZ-Nu+)Obxc!9ZM8a+B)pe<$7De@Zq~aYS6ru}g1i#Eo}qoFmxWS-<`Ggc)yWVko|L6mE6Ub4F8R
jVA`ClgP!vQiT%QH^B0)XL0~1x{dKbsu@?n$1n8-XKjm3u`Hf55%hgafh+dzI&;QM0k>hU6Uf|`))pXS2-X*Y?B#f=@4l%J^x7#5h3@P?QRt)SIWhHR#B3-j
wI@SZl5L${JucxJq$ZFD0A&2!Vm_75R!B$pPk2pn-?p^<vud8}<tVagCUs6;F2R$A;K!P2(s#3%#u}`lc#ETSW!^3fbhuM`(muwA*6A+@!^Zuw6O~PICHqz5
l7Gxrrsm;HDMV$5Um?Cxqgciz_JZn67XQf(tMew<7^gRNI!15TMs~OX0<a!=&5dkFnfW;?v*z=hKP)88>Ev03iCRUWA~}M6j5YS~En5`cd{CCUhTjC=7uW$?
&lB9ctX>yuW!weoJ|LIK<_uw8$xRL>+D$8W4=Uww%OeXI#90&;8XF(G(X0ssCJ7?Vf93Ha?VU_yM(@YA<%5&rTuKi<&L*tH{2rAF{de{T52rk4M&jmk4*jKV
IHx()d`e9RC}?m3rXOTs8;yq{=U?v2>$Rn^u`qD_Cjx|UT8AAASpXDh52>YkM9S3_!An7AmsOGo$q21uATO2<$yr@cpEt2N*3<hI=HOt~rpHOetQKq?JiS~k
TQB*w9~=90=GT^7)qs;4)d4})n}2vRyl4~e)cNBuJy!o4pM%R(0~)!W(h!^A!fr|cTY?WPL=el6X6+>mccN{q`mh;16LTqJ*2P_qo>@*pa=zt-5jW1*fa0Ti
C5du^$HTaIk9#ma6;3QCy|->5^~-WZrExl;=-_nOq^P*aaoi^de1HU*p{YvHn0TvvE{MuHgwAeCp%#q@;ZF=;EYE<E=@=?le&oMhc>@=UBTXD!?tfr5Q-PFG
1K%Oq#X;Y-Yx6vB!Z^jTxg7xaCZ!>QCn+6RKilgR+k%=g9pYOZeO`o71NKHQlQyD+<&^sI?zjYvAriwaAw+BvH%{p#CJip71o|!>(2-fM!Pkr3U(I?LvX)VT
w|fS{Ay&#5=xKHUDA|^N#MIa&0}Np#+U>5cPv?(_mk*T^5l4aTP%B}tBTL)>r5zjrGGB{0>v`>gX+Xw_M}u7((GU<lb9*E8E^{(E91c)&BKx`UHFs<-IyKlS
i%@g4llf0e2`mjTAn`2%2$wiNzwm2>^PlQgzCX2P6FrEk=Q7{9h=A@%fWyNchF-DB5LK)HUs?GG7#*KN@lPnx2f;qq07pM4RDfR;;~7)wlD!pUfbF2x6B?A&
AU>}(fgJwR>s~;-!k*abgBWS`RP8k*ox`FHXiiTK-0TV_{tYssX24RM#}S$oI;O&hQy9++f;<$4reb8%W5Nky1Id2|vd8%m6Tx}Vhg~?0NJ<ZE3Yg3=i=Gd>
y*7rhA`!-yjSb`oub)8J@^^~r$-iYf+N{`ET4nADh;*2Ip92Bp8;HxXAHk657KCdjM-Oc^*Ow>Du9Cql8bCt?Y1e#eI$Cd>WQ%GabOH^d7}&}0VkL182eUXI
!ICfDC;3xurdB9VcLLO%ZOr#oU-pz4h}*@7ziX#PU%j(Hs4K;!7v3?n;nnX1YCp#w;v>jyvPq-z3|(<RB|$j!i+45zX*)(MJ>NKkezVZ{^!WL%bw|WwJqw&Y
*vRk$?QDb?YxsAQbU~-JWY^r;!iRp-9a(Y~NNiA*0t-iSDz$_SK=ZY+_-{xQ%l#2vH$e))VVjPT{{2>9gE&R4VboaBdp?coZ2c(h7%i@H=f(;^YSo2gCzQB3
K8kvn-(-QNzlR~Fh#cLs4Z8E56AsTNhXo$lkmDl|H<%pO?237JuyT-JaHBVp$2gT+G^;iFm|RNgYsx!*xcR^FIyC|x&iFuTv}*?b?9uFL!&dQtyf$A36Ud3c
cFI-?6$KVt_<!Eml>4WJdmtSSx(y#5``wJZbld4qYPWOJf%JJd(2cfYi!7zkXE+-bU?pBn^>*&?TA2B%fjv|UBgS51)8%O!1#k#^Y1<1Z+Y6X5x@Z$Lu#i&~
gQK$!xBR9)t_p~Rz5-S~H0oM?&3()d2H%sekV)d}%8>n0+%ny!5;zuw!4lOd3b*Xl1P5Zv?^Ih62N-V<TB7g<t4RLkoT1ybS+F597+j#(!v95Dfrd!v3q^bR
Gss00&F09tkC||E&!BIgU37>nNZ5^yW73c&O#}jF;*Y`gR9c;7!(Ij@2qr9nG;;wicW`AX$Ptdm-I+d<R#~4Vh4OV+BXr0BgL6MXma4Qf&%P%ZH9fXARF^g_
_~|TlRT>!KL@Dmx)pzi#)#V%}T;#ftpH+21_X>VS3E(pV3YMPmxqm^0f!7hwO{LGLFp6LG-q-J=@Eji3jj67&SwICmSgW4WS!PU|5X%}oXL^)?FVF9Y(9!gi
TvCq+w+{)8I7A$pCEccw>f3bMC@z2roA_nRRNgtY!9uicDq(X2#yKYxOCZ@biDgFI4#d;a*jaMUofjALF;Wy;5k@cZ=JY*jPmgu4-32jiI6g&WtCJuByD`7c
jZ6^>5%uHz504M}+IzmSV;LU`4LDbVUH?EZ9af}Jb5##G=V|}HnP6b_*Qls;NkI&b|3&ED`+cdl#f|rx;TA>j1?7H3MC$~w(Cab83f_ybmc8gwB!MO9p#d*v
_yY3B;JrAfp!su!(YT);CT*n(6ySRZSzFodD>TLEt+Aaqfu>HK+8eM@Q3Wz{%wW6BxvS5tX9a~N4)M)lh3%d@y65Gms?4W0!D?J25bBs&`;@y@Gyl$g#PhKX
pt=vLvt%Yg7K^|zVY!JxJ*yX}i+FuzLmZd~`WDcV@e>nz2v{<A5VTRr2OH^s*N)hHx>($0Z|iys9Pcn}4-OeRG}i2?`Cs-w$Bo5^?%39=oN!*IR8*u*Bf3jE
wJBzCI_0$Z8*VKdO8;9u-DN|HEULRfv?>NWMYGZ^WRjS~O@&2RK9Nze>M^kYM+hJN|7N03?)>eF^`rY-!qH=Sci7q4r9Lky1O~!t(AXejC4^|nYox}VNeHIu
s}@%0;&=MsxG9{7@d27M?>)&S2<{oAe)Eh=k0kj@$Qp>Z$XaoU12rwlK(NgY>X;f#(;Dia3EB0S`%H~7p)oVMUHN0U-Gx;Wq#J3x_-oj3Ki1_sPA8*TsrtTr
c@ZZkmO;H#`lLLh<QkKY*>3fBz~WoE@qX@#^(wQ<J10>=XBAt9>^Qe`t<cnh3eXF~XwG7CqAh4&k$xOFQ!CXzRKTyTH9hBiWmHQ;yr@eWFYv*AJsNe;M(5rx
w`*_p<O3X{f_u57DO1)N#k!qNd(sDn5_$G`Wo)n4yPuy5YF;;++;CaUH0t7A&UHAG5XK&I-4sr#v$S1d&vw|AN;yi8cA$$LT6)67RnTdK;_|xVM$63Vq!V1F
s`}I?oo(IoAz8BjhD!lMXS5dh;2YnM7MJN6O;PiIfvj3xHGaHV#!7J0Uw1wCmcppPLR}?am>H<5_Z}%Xz`hF3*1+Av5)+thhN9_0SJPTOSy=N6jUJmlWZE;u
>i)x(U?LB0tMKuQLUkNL|8n#v_VW^|)=TtPXv#&n_Jp{J5dyw*DZ6ac$kI^gin_P3VPS?h7fM*>js`jWlF(}&9F<4!{Xb6z=j+^-8&-dJvB5~FE?fe0)5war
Q1a_$iY*cFk$jN|Q=A0!m=E}Imz$V#PH?6;5T24)N}?8#uh$DHmbLi0FNa4D785_v-fc$^rkmL*yQmr8bPJ=hp&~8Kb*>YS&N-uFG=U6yN_hQMAd~l%+Wu!*
f<<FNeujZf;R~WEquV@7pL0fv?2+dFDe$9ET4iZvuAoFu6Z5d4p_Jc}t)PxxFT;c~3)mG`If7xz&YJpTWn?ovE%$d7EBSub+(lZfC|fryYf#7xc$ycaP;GIH
JGC5oyB6??Ux8pxFRT+*0%RgNge+nNP@Iw;zjN;<4f;fgy4XVAb(u0|k3Z8WhET*=DYcCBsZ4?<UY!bfSJ^Q_>?#?5+mrW2uhjftWRW20s*cq`7s2twmYarU
y)U3ao;3@`*sTZqB6;9-sryJYR}2&oMUD?8q>t1#*gAcUU;~1pT{Z1;2C=`cLdY;(`EbFJ1Uh;O`^{@m@=IuAst_Q$mhYif$rq!vBpHM}T4={(B*XZDMN?M$
#~Y@xFP{I`ad&LruJS|DdcV-SefVnFtdP#=N7#E5gz1wB?^S*eO>pT{5c{WpLa%1tBxzou+`rB1;Q`3xV`09*oAG!?;;iU;hEt=)oxcYumRR_1;2d^6>Kb-p
00c8vFypQ%H*wfymsr+Zn99>u^yV!Lli5Dn-&fHm#_JP4NLLWWSJ-8u$|+^g+hHStAG!C@VvC!w5=Ns;$1;$WS+99;K4M{oAesVCbY<*Q-ab}`sPb40I*1{;
@dwb2r^0S7_8kACb(c0R(JUa%Aoi7W`RU2p!NOw3qh%JqYw+Q7Z~PV>&WyviMD;*_xC>HkzH6xy(J1?$Akb;o+3-}>z&jtj<(&liY6lmUndCGKagct=9+IhO
&u~Z${SBEp;_y}BbO@d=xq6);mBfMfF2#yEojs)D`~Z4dMVBT~DzM=3%cF?VuANcPOKIim4P4-#GvRliAftje?)Evy#Ocak#Ld<Dr)A}d`Tl4R-l(UZ(j7<Y
A5M4Pc4vTG=z$*w`CwUTNML$X0m^o8kNt4~COwA>m(Re1|AkvC>2*Ru-E_`+Ip#O4HN?Ezpv;Pf)8NWH`x};|y_=Vqwn~@%YwDo~9VqJ_PRDJ$dNYlcAfjjP
z&_sdPw(Er5qZ8>oN*YDd1l=?lVWnoR)eTNY{vX^<O<=nP0iY`W2(ED@7Wpap5!AEq0!rW+|DMa0B@jorxO{J)1+G)vA_BKd$e~Q@8z(TpQi|x>jhj3?Ghw#
!0=FvFC=w=43G${fWrJuwWL5}>onnm;DV5P1JbSqBL@zt^Z)t!0c;`g!^=wF1Y2}fgBkj8*JDJDimNUXm-xzVj!Q7*vLTkuge7&BSsT5_xI)d7S2Ab%xE<kv
FRwl{l!j45VOMSXlEldDF4oI_CXb2t_(gV(oH_a(50inwd;J(V`hcl`)3^}6@Wcv5XP$P~2*UlE##;4;OA@~0F3@bf76=|i=-IsE1>a8>gORDE76n<tDUW1)
nKr)gR6^T>aHX6-Zy-AG731JugIW1XBYIbSoDXWj9JEKeZ2X^l6XK8i5}#vncL39|)JXi#E647Vkm8fWxuMot26EHAxdgh<Lt^bh(!Wk)H}27*@g0fflY#lm
Vea=y+aKP#sfg_ygwm^K*qL>N=|^abELIkl&}qW3nBzC@Grb%D@&<L29eO}$Kf|rP2=B}RjG;TynRg{K%1SNmqBoi3n_D8__tfzPKu9p3yt#7W8~hs*T18yA
CEy}5cUUpiDpO#ZpQJZ%8-u8e*27sekR#ASp0vgdTx|{9T3bjuf^Yx!Yz^jn$m=bU7#G)9z?JO%@N;<#hk5W1&K@wZSiT&Zsz*673Iv3eA+&MOGB!@sKd~H-
r53`{P2Uqga2`V@7D#X!-6J?hiwX@$P&crj*>DUYkfLs&$czw^gs{+xZwsY>oAhC#>{O#lXHb!sw#d<l7+i;NyC&-slh<5EJF)UT-4GiIzyDB#0ZApSX9L1!
GZS^bgGLqAUrBZyxT-wccxv`A_>Pm_xoF%E)`C?;-@sv_sSf@^Rjl=uF*yhOQPhtqXxT1Q1Dl|_AW|ht_3Gr3M&WpmTz6&$HQ0yuBh+AU;sd>QYuk=q5g#3<
xf8wtYS+KFbqEOAv86U-)R0JM&$A82k|5+qZI;qpd6a<<n*3)V#)j#LtzN|-1bA0E-N2#;#sk|tW(^^w)ry-sUrwo1%N?V5LXWk3YtrlIpR%7k_gryO)Q4$d
x7XQItf=|-fCTG$3*u|@q+a)T9f~yt2J>h4lpTc!0YRac*n%8RNGdbhF;64O_hhO@6O)Hea+}QG*Xc4;@gD@C@5tk@SygH(Qd68-htVC8cdJ++r=QvH^OwC^
7?jbq;xe2`dmeV<L7Mp53dLhm`$EX%x3nXx<V5aA_1HTD?|$@aNibTAiu9N2AW|Am@zMehees0k4`}7@Xhq;p2(;`>&tZ>iv9Aw$(*F(!M=;XI-ecvbf`1`g
mCEwO4Hm;(Cyjea;Ddhk+;MmyAafJAavh%0go(d-`9!IO0-L0P>2Uz9B)dCl;{v`y;7l2bw$GRgU>`h7DU6cYQd>a5`;<4(@@$SS+a|@~$<R%wh+7u>c`yFm
#q!40igaG%fiObFlK&A?Yqi4W*>{~1>HerkLi`wk>|F=fvlMoZ+)q8H7jl?mp})l)6v8rwO(JQ?8}deggMNma04+mycT22he^6uTN76W@$0PI4uaK_+>U{y1
n{S5KL>861SyI{W!$%SWr3Q)?Rn)>42;`!XuRyk#*W)+i?NdX1v+JPQ0!$!bJL8u3Fl-tp>huf&>S*q`E+4RTJvW(JynI>70x^oj0!+p2c@&82Kp;(C9;sx1
W`B+SRMk)c-aYd#mv(%L!`zrGYfk13HUn=(PBvXr<w|*YK36kF)CUQZ4kFyQk&n+q5LG)3z30i|7k1^#wuI?yKK$Ei?K#UF)=j5MHMk%zYMi@Q-jP|k)V%>U
U&>O0d!k1@Li)qJUz=^zQzPd&(-%Fsz$5g)rD;?r8?a-ky||U!nq7bOiyH_b*2OBg&=SX0w_`h^4Ku?~wn=%fJTRp!wl<Vu3o%7&BCp)KC??|ixoKa7%yd(V
(bH3m^nXUIO4|jr=qS3mLbOQXt>Hz|)`E#9Xu?Zh;H0j|oSCTCpGUiCKIn;qBf1|%4=nx*!>&}fwaWsBP)^&@zhJqWEj)Z(<laBFXICs@<+2gF*JTHLjz@b*
1p5cMWRz?4hT*(Lox>gn=lYN-rDO^SbOqE)kW^SwrfN|vrN6-r)f7Wut{W1SjXX`iAF$yevvxr7QU6@_BUprag^&IGiti#dLav)q<h~smR2o%7`7Ko~)dim6
_<!G{#y=XQlfc6(@nJ`Q@|*c+ulZWhqJag>upA+-Sx0~fB%1Q+m5`1T6*Xkgj#ow<h#+;H5S{9x<9OrvozsdS@Tx`$359u_1vG#h!L`k1=X}@>!org|sbRFB
R;HC!kYIH@X@d8*?$92ZKue7Nq5(={akOWqdBr|(nfWh+=VhO^@WCXcVf~Gdv4|EHXu^C0>7GF3%u#(SD~?K6*b)0eroo0rpvc8>x68~HEW6y=%uH}_149j&
^5B4Oo+XW9VXx1UE%%v96i1*subM0!46_A93S&v9e@*7USy$nerhzOmpG(CjsSubSP3Thmpg+5{FJ}o2O*{(Y0l2IT^lw|?s!z<uH0t}}8>tetcBI8JK#&Q}
4lENQApoKjGABDO7EGaS-yPtcMPOS8)MQMVi+J{>e^JFvM-xs@hrI%qp>w^2Q*HvKe;xp9JzVJ_m-UCT|9^7{P7^GaXsUqeidjOEp4$hwCJO{}UHUh^B8cw-
nlr>H(aq7Z5oP=Y+Q>z4ogffy4X#?uSji7vJz9Zcuq`57krXc1XQAWfDwqKitdv>tEJCxet$1(?<5hS?Z#$vv0WyEio@Ek<mRD|QH%)7JN*f3qM43DP*A@%3
`k@Vbe1ncDS@a*8+2E1lZ%%R9SNo4g-9>yHd~EZPTTZwp@T=2&G9y_XET?pjgsYg|25%dnuCb`-A}#-tW{_b)eO#FQ6bx+t)l9bqb35}ZUb>2XkXV_&yrE9{
hoN;sj&2E_>&$s8KgYkQglgYha=T1q1pvcOP9>;fvl&leVzn$9G+e8--rYCY1<Tt3MR<+K{nc!!iI+??vEZ~xfvz1Ic28t<f0BTXo5=;vMJIH2Ul|;O>N63-
f6sO3^hw3z5;(H<Q&TvN`i22hJVqRymO?Kx)%R33o~<!dMB&=GtftO70a~C`u@*<RKThRg&1IHZ?8<k_9o}=oi{tEf0+1T)-2Pk62@9dC27GWl=;NG7_;FqV
p@G?rGZ)FuJu7{)_fK8260z_*sRhtn@(x9Jc#IAu3o^(YQN=x^xujj7y@yVGhN|^A|KuE=^1m+~{~JldVoT<A$$P8bpu-z*wVU>qn1bP4Z3<dFo;pQ#x%GV&
O*l(u#<~q4whN=#2M$E_z?Wi*Yc&t{FuK5RnA;v4t3<BTLRI<D_q(Cc<z49w;ifhA7*#8+h#-SIQNAD;@xrb9{3bB9OSAqhbK1+L-#5iK+O~O>j=CF25`xO)
RsJ1UUoozrH!S_WiWmt<%E8AKzwZHdttJHeN9}%rPIcx-2)Nv1ak|G2xRO)Eayu3gF|faa%Cxve<2}g471}ioLd{s+PRq<%5!(8cL|o<M4hS9?kI3)OV@xos
``uZM<Y1lk7yo?pe}Q)d7D?l(OAW-Un5`;3R(?*~w53Xf{Unbaxk0x*SUR0EFOl)#_Vqz0wTSc%#$C+4>-knwz?e&{@AV``9NJn%84<Q~BF_0+BnEMBb0O|W
bFwM&Fsl!{>q_Vl(NgerQse=*wWW4BdukPB+oBQWQ<P9U+oZW?nS?GYZI)J%SaEqwe~;b-89U7{22$oX2MylKzl+z?P+4ZuDiju|-|Vr=^BX7BG(@d0wXpv>
lhqP*Si!+lMUJob)To~1qF3d2Z_tXPXe=8z(5~tHDz|xN*;{Q`ZE=}hNJ{1<OE}AAF7Ls6BllSz+?{nxmdED7t4CuH&_l)pOHjSk)%fy;JnD_Lee#pJynEJH
33MH?s_kSdHjl3Is3eK&i^I4>IPfYul*D$9ex(p0Q=VZOeumG<VQu|^K<^c@rug)CwtdPULV}E8V&(c+CmL*@{{-+}GNOi+6+ap?NzuTmb%P1vhj;!w*Tw7q
f+B^jqK3=#clebxcU7Bdg4P>1aKiMV9|B=zFNr>qF#o^c=@POp>mjWZ9NfzM^uXkSq~gv>kyAQCT`bARYc@M+W-w2}6PaoT@5ax~w-ea#{XkMtH-F8f<LNk2
6Ye3nH$>NpY`oyazH)wc`U=WKFqH@K8l^fJ{^}ilWvF{?gRs4dEGM2i%?L1*(v8I}U)d+yjFPhj6|ayb;t@^#ak=yoYc0R3(QQ4Pm`Fyaj0#uF?oA11$i?h$
p}-j#!heHFkN+ijO5s^Y_pJyDTBY0mN{!ET#l~_seo5GZL|71(fkm-s**~jc<gJp>3xgoAY++jdnK=>rBF9m4u`1H7aZ+*>S8^6k4#71uN)C8oelOL^QLXxl
8$>k4e_nX@-=?ZaVZ;07cga_<vdnQ;kVnw_*?(ZzTe}~75esHIwL#%7p&u}1$y^E3vy!9*)4cY2jTC770a<li!k>}tnVbt*x59d?OzD5=h8X4n_h<qUriyBa
tW7yLU-f=5R&{<y_J(%EZ>!fmSmT0cg3PDk^G4C9v|E0igaBpXR!)7wpDtZxINhgS^;fw4SZ;pw_kj3l%zq_Q3t?VlWewBzhrUzj25yQebC0jbx*!nqCii7v
^p=d7!G8Q|j0Fjr!zie44dpK0-UUFeM;@#9`VWv)kHS`Gv)IOK#~GZ_Xq`&!vx<y=#I+&^?)Bv`BvoRsgb4)VZiu|FY)W%}UyhRf%zV&WYE8@iwnd6_-2z2u
zZE|NCXYm<A8@|@oc$uVVA{lTVuhJ9JvM?G&C8b$$vQK1X)`U_*D$QttPGcm6l)?w^nmud`?o`+l9H^?t>cD{yZge!ALfy^?0qbH2GyAztXNduHUmZpRlk*4
m`5u#j?%^D-9z#nc`<t57gbRvssmrf$u^71YRjK!U?n^9EI0^iW&Fmd*u3qMP=zB@ny>8Of5jxtshIXazE~g@i|~Yk8&(uWQ@zl&CMKg_yWx-W;hG43)Cr+a
$<@C;Rj`wv3yKc2>U>)hc7gHJcZKb#>zZ|5rq|=wy2?|@&p~{({m!5671Jb0<^Q5G?za4*siu>_9&K{+qGWzK(n6M3o$o<r+c89rUrGwbE2hUvXwSqJI!cCh
b1w#|OUKHK4n_8wLnK=gkdJ)*!?fy9o)th=ku7@03*_<~iYMbJg!^=JvNpx*H0bZa(TRjXZ9Q-j=O?)dvnw>uF`p9eZDk!$2D3Gh&3&Cph>dFF<z`ZcF7wze
Mu^L4+MZZRMCyxb!@geT%?@;|X_jO0cLYIy>Z^gsO*c^PCg_w{g0QykRhj%dC<!GGjAL4-hAZ{d(oW%Kgo_WaxF9<fzek>PdZc7scE^!cW$OGCuVG+DnrnwQ
>?saOtgleVvf1B^9yVaoG7~0z(Kgf#dk`Q*%D)XRv>7MuBd^Y`YyX)5CVv0QSZyyp_VHhIn~?uxYgz4x-V{43zAsf|n)%?48oZz>0e1O<wqCXSkyfqfS*&i7
&?3=YqcCK5pV7jV16fMa<zAf6(?EQp?Z!OQ92G@?dV07JDMHGXj&Hu=HBykUg3qDe<o|rxxY$~Z7#@usQ7$;@fCU+g^V~splptpCP#nqc5!Kt>_8`Lv*23nJ
1R{?6zJru~DI>-xaYZAHOAzj0lj!4cKC1-{=&Z9ocpYNU=Jsnww^JPX{ur<+5$I((7WWhywv|?5WFhYinp3iP#u!;=pCKx>13C}5&wQGL<#&xL3ErT^0`Yf9
={GuChv32O27nyk_0Y+j__?G<2^EoZTENx99=>=jU+orQP6WvD2B@)57OL$8*W?u!gf#}GhSUOMfGcgIsyYt+tvONIJNDwsoPkgFJ;u{9i*wAv?B-mAxGXgp
c$2|JhPo_GRjgvDsx6ns^|NQ=MLZ<)schA&ll?9B6u>dHR>>FaaHauY*go-uAy@NS*`Y(RXoO<fDSS|Zt%?Up8$S*mac350PNo5{gV@Gbzj3u^Q?lF{XBh4r
-!+4Pf~ZsC^h?Gf-RJ9-#FfZGSXk{!?_Bcxfcd?Y)ieP%+yZAhB*%de&M9?ed8{8Bj=(jm3ca<o;+8G3byNjF3rL<Sq%B-WQgP3jOtSnOR6aXV9-brcqprsm
-Jt!cr9kMeUIxZ!PvUgcNl#LXDtwvm`NqEs+Z)xPIXN7NyQRQ1Z1zmYbjM+Us_z6pi`HhX`ny~|mGcL_Ft&Q|lhLdo?h7pN26p{=DKOo;Fs_T)UL4Zb=BIT<
r)DolpcqoY;uFYm{Kk2~-+L<xa8U$%0EEewTdE%o+=>zn-Iv5?{DP7Wh9myKQe3_MP$Q_exkay*=-dJ+opu2Y+g+o_I?((wB5xi=lsj6#g!}K41%Blln$`vV
p&;3i?l*H@>9@K4w-$RKd54%A61J+QY2&bwCe#kQbh%T?ICIK~3&LOddx(%Ez?q9ok;<1agdDx?;{Fe~{;MV7Z`4iMytl4>f*vxn9t;t-rA>=DNeu%tJ|@^h
3&_x8an!KjCy(WAEuJLuz?ml~NU2N4W*tqJ0_K?X9@<|&hS=C`-WNEW1ZA{-!6ocv$fLAtAXmBwxu-a@>0vbiQ)OYr+m)q~d=lC3FbFdEZv^z<Zy}@q%IK*_
n`T)|Pc1^TWJkV8fnS-r{CF+#se2Ll*-YexMAm26zDO85(jQ!A)h>J%12w@dE-@Fo-5lYG2GHcoxzf;|>pAmNznD@FwJ%0T{USQ%oEEF=-d(syc6g);Mzj6`
Tt?I7ishN0tqHa}Z3(|&)%(gTM)d4MlaYdw-P<JYIcpU^|F72i4ABW*ViJ7Fx$U14r)Hgxh9C2_X}(?6hS%z}eENK`95@S!<6M<shD@koqp))eAVm<H)$kr5
bMz*Z?^<ajPB6hV`j`~2HJ$fn63r=u1;6u9Hf+cSe6MLwR;iE`BZiG2B+_ouoXqwhFIbPKHoD|pPtpF5j!8dt*s;U=gSTb?<0%O;_{>J#^*fK}b`%&)LqXvO
>m=3#x4JSNwAyhN;ZV@?Keo{P!%1n#%Mg`kTHVx-S%~S0Ojw7}O1qkuhdZU&Rr-tW=SDi!g_7a%LL)VRG42aT#B7RmbCkpy(}L?;qZCf6Lt&e1nb;#$Z~XHl
{n;FsT*uz&GTwNgWNPs-EQ^JoWwG|}%Ad=0!o2s}X;t4u!LfMji#;j76pZpmiS6JjR}NtcGNu@KL1Vt~i22VG8UN%tjzTb-&<RNTCh&33*RWC@qx9b?GUKvk
GSJZ+Ex7nwWg!Gm6Brbr;p#CbNAK$o*=33{;_n9KR9{|TU$r~YZ<w)6hkAG*T~Pe^e$%6BOaA1Je&%}kFWz7}hHz&Z^{cazQ-2+1!Sr0O*OG#wFI(oCaz48M
((nkP-RU&$WWT6>V<C&O!<#C0{%;?zFrw}(F)nU&n;8#_NraY&{oJXoIGBWcVMCJCMk!B;6t9LJp6RaVvb{B80m15nX)vDDbi8Lr>=LcSXHAL29MCY@CmH(Z
*byVMAh>TQQv@5^t?t`Sbn^O~1jETV1r-}PC5|At6sU+u@l^rCWT*z1h}YxB`<{hD<hD(iJ%jg5IWZ6s;ccM3PFlO}UEVf3rzPEjQ5)wCcMw3l_--BIvhBB%
CBCp8E$zC%?Z2kFm2__id=my7pv*opI!%XWgajbOcV*T~vuE3Ft;i95E-aF8C}%G}#>%fhHW$fjh;!60p9MV@)mqkYdQC$S-~%t@Dm~NPTLp3gY+5rA@VDKP
hb|B&(c9ec;j8xa#%3!>$P#O%G6Q2C+th^q93^S@-+~B;npcu*v{u0a2(*Cw@bM{GWvglG(vjlYJ>$)newwRNl*kN9{oRFrwlD7KG3ETbaz<*$Z}N*qZ^>O;
^+)+mQHGG;#uHxDCYT;#<tPW27Y_H9??gDStCCfZd)<=@Y~4SGITSR@+4#Nqw@EOxFo0l3h$~(bKOMcho$FtM9T!|c+Y%hR9I}@IbCr+~9K-A!z8~=s9bDro
ShF(fYL(=F_iW*S0G#*B>(7l{rvfS6^BWP`CKkBPJ^>Ldm)vG4mS6YhSbIpAEyQ`XZaCNyGzA2fh8%_$GnOOHfka0uy9dp3YQub;XvX^a0$eo*lTN01WLh(~
%JjJ*A(9H}q#pcSutkg1x#%SMK;>H9u_nqQA#;H|99<P|C@JrJS&a==-jkeW8-i}O>kUw(xvd3ZR13y%9p)ZS+XpJ}<Z|U*TeDWhp=P=@U<RozT<zVDH`sGD
iSy`T8ay#k+ZjhZYN-WLcU4;^PEAS4ji+3Io3=Hn)maKd66a6)AI`K${wd*jNnl_z*rD01bQYGyUzjM6P3EbfyW}A3mf}CUcw%gn<#|uQ8<1v^f2zC#4oTD6
*z@HzTS`g`p1de4)7WFQZu+N3DK|rxV;dm2B)ipJC)Ea3JUw#sZsT%x=!H!_Fd>^9(;Cg^(*g&|AI7&7j=afaflAG8eiFE&X3inYtlw&P2@)V!9+_VT|8U-=
VgSPvEoe%O5ZqnqfLmd!Zlg*Br8p3YoW<*^uvo@yu0LDQ%zg1j)$_1Td_`0HF?;??Fixrr;n_$3yM+l%IL`)MDbbu1CJqkYNXOFB%2A`93GQ<_bKe`*(%i@T
`gSMip=N<vZ~t2VwiwAjd-p)W_%qSLVECLCnOOOT&1we=rQzF9GQfI1c0tStQHr>2TEAO1a+JAWxyM|@I^?)4Ergm^B5Xf!1R4ib5V`ttNUc+An|tdB<HnZG
8wV6h(QJ5GaJvB!*5ibI>7a3c@XjNy{X2f4g8^9iV=Ajq+{&Scchr<PNIL}tOcK={6pHO8Jiy7iwQM4+UgStoj_`BuLEk(~KY<rZT~lAyS_NsrypZtB8t<$&
LwiNC3NV~c>z}E|YtaTxYJlOG*c@ANep139$S4z6-bJNG@6<KSLNh@GofBOUOfGUbxYnb0#6LxCpH>p00+L5mkm2vusR{K|ECtI!kTQvCwyiRa%VWD4L$gXf
PxcM=iz7E;?Xoy+9(oB@<W!_x{puigIBl)(sU&sm=zjvXaqItGR(w?_1Z^g!%7%=VEE$zSSzQKxZ<>dpN2=pfONiUnxRf<$$I;dJ79Ya|M=-`0>wB>B;bYTE
k#`9}h3;dE<f|Ut8p$P|B8e$SOuhV7Mi}dO7h{BK&q0jiFp%6fWNMwI_h_`r!!EbO*@XJ;&~lk%SYrKr#V4rQq&XPTkuDy<pcu&oGZ!SbZ(vZ%c-X2IEsgnp
7eb_fP|z^;chwDILMS+0$q2TvMC0fzcQ=GxNL+~~ue3Y46zXOF9)J|A{X>1g-Y=HR84egUZczHgFV3=jmMl!Vv%>8mCBhAN)Fr?LP!z9}w+>9#ki<VjQ3`@(
BH2D?zE65)5r|6}rX-}>55oo=MFR1Q+-@<?s^&L_41JEGcw80A_#GHOO`f1QcisL7Il)wMyOP*TXW*a>6~mciajZd+iizagV904m(^6&Qe4y+X%xY#p@J8j8
NpduDenQHk2#C}i=nuLQnG3fiFjI~eXQ3U4nXwK{;J}nhB)^9#wV{29H+3=u7R_oIpnK5T%u6ma5jJZyG8*#}qrs4vpJ7F%2#I2BJj8p-Soi97H1_3-Y)CH_
zUPGRI`S(HL;{Ul^`_k2d+LCu?J%^8vKlUHrIS11gZVwfPA8E#IK7VZJMaCzJ_e)bMhmTtwSs!qK7nL%iMq8Smx2M*J9F!R1a2k*=V5uGdW<qhh)|PAuZ&QZ
k4~*Wng9g%i0yP2aeHBr&)hiY=lWfF0y_$L+RO7cJS!#W)R-YH?bO{%zD!H2$Z!Z8&}R$irTF;mU0M52dpY}qMEZL0pInAFW~l3B=%wK#FoJC3xcnWFKtP$^
9%u``UEzQSi`|M#G!oPbN%ur*=g;w1m$YqVz(WI1@O5h4cE_i;CG;MuuROoHfO1Ot{k-bMpE7)p-w(Ep;D~#MA4Uj2m>ru{YnQBlSrZPAOI$7F5(6GltLmFU
1z{dEli3aYM|sIV+b}S!@08=TKfzs6cEXaMJ9Y%rFyED&X&e>`)3LDDn0)F_U&T-N!QDmnKwg4wzFsu0)iO=O0MmxGhEOzX{+hlCwN<A#{hGwC#H8j;NM*9#
Vc|~N_mYy688Easu{1EUgh%z950tGq!HawXEt*Z~NFG<PN+A?b=VjoFI)C=nd2F+!NQqa>Y8}9E<OK%_*ymmbEIS{s+v8*cV$(_v`v=guHzRR?<p`k~Y&4F`
WwyQZOA7(miN4~qTf%qhylLoZB>a`&j>YP-e4ra?KYoYfLwn!-x(|9Q%nN?MvJdfU_H`Ldm`{Knyk3`FE_^v9I+Tv7?_*`aX3@v`*Q|rz)X{02QQhCwW4{|k
#a4UbwINZm1eO|ufCTF}zQFd}9;^IBOn{I!jcP{SH~WCn0iy7xuHfx~k&A^!1m`FU_cwNU!rmy-?8d>wSTTLV;`!|mMpuT2f<qLO+j+FJ+3=*R#QyE>1c#!Q
Cc%l8K<-)^^33P85bwt*>6Rw5Fuf+l@z*!YB4lHr;>ll?eh_@db2;AQr8~K@gdD0t4}SkQOwq!my9J8tPB1*~vb|rm_?Z{BiujU%r=4Gg!)!<qXqmzP^rUQz
S6-+uuf)W+l%KF+-J1~;mAo*9T9VmNrOTUYe%<E4{VKY2ah#Er`i<TOzUmlN;V}+ahJ!<z$GoE7B2rLxXwkTtG{_((%3AbHiW7Ba{`9A_ZR^fuCZp40+S#8`
B(+b_q8*EYCtCN=SCQR7$5+$%R>X}smvMr278a2C`wnM`a?l^S*LxuT6)%wI`}R6#L38F)V`l;X3&aEC^(BSGTW9ePcls>!+fXA`jfFPEbcc+e8jBPZg{z|&
1gE=+25LF8El^_C4T#@3B^*1H^L8N7SFo&eY;@SKQPNOSDPCM^H<Y+jxm|P1^aCKD^RJe85uO06{oXm=p^;MWIdn-5MZg{A;*h=<2YZL}44&Q0@YY=MFqcOC
IKfXV%L)%RWEdP{Fv85V@_PNZo+AC9iEO2!7-m6<o-es|npdJsY)!1U5rY+dY*PNB<yU@8{Zr{xS1nz0@r=xyy=2y=f^fpNa%Y=)H~8<g))Y|sn}s*04jT9|
J70+&_=AkmPfj*Z!lB-rYzzRUVpJQ1GZ&2GYx`Y=RT+`S;Z*-^y|S45z0#Z@Cu9S)v9no{xyv_7+M0V<;^}Cp4?Yb8*$Gaa4-oxh&(%z!^Oe9-_5@#MB?PF9
Rd-NM;ux@%CetXcQ7BIUmzK$4xXxm9PJBz}5@x#QC6a0#`mX*Ve1ujOcp6C4U+k6t6^|VnL}nb7R#0N<Xm<4brb6S$SiZSQLEAP|r9j*1Kd<#gpQnkHx=M_~
x?bKVNGyS3O(4IIQQU<Kwzc?;7sSFPYw@F8evOTt#zRPBD)nMU0G?PqzxX$wpSIgZNyTHf2ag!IRqbVhw5fd=P5nQ}{#sz?>r?Qe89KT8X!kROJ-TdTy9{vI
%Iv>b<mR(|mDez**fuYLj~vn(ERger`Y7v{FbT;LVrMgAglgSD+3IKe=1m`CN5oqjN*1y#Z1I~&2F?IzB9v<6R5VpShwz>r%2w!6y?ZNy(o~>*+myi6&Wc<R
BVhg8y@;yYg#Ip<&;6$i;Ta?|l%011#a=znHVW>K)|v4<(nQl*8%_MwcTdgI`QxR$FTfrWM1{qKf0TQ>90=O?Aaq?hwPBI;5S%O;UYGj57OHtl^rM+3u;PCo
RoMY2%3kC4pBscbF2OhzI$NE6Sn<Sgfy1It?rbtGfvF`*_m)<9{~%*^mv5^+-u>VU!U25@l12THpw+gopsvP8bKrC?m$NKO`C;<Kq^G@0>B0Ly{$OF7hnly*
V^-|z&Y4g2w((c9h`O9L^g-209na52#$4{2;K;oFHW3ukD37w3TG%rxJgedLL}nYoUW1RN&Vzo>;V_Gpuf`8$mech~p!<_oYAT8O3E<gMTF1^_Si8Zm?~DmW
1nAuG^1JJw$+QHeh``cguf3{ds1wW|(E(1D0BJHAIzyzNuFVc9sXy9#&otv-<&(fJUQp1L!pyQ&sx4h*N08`MI8#TLwcS=bWVBT8s_4%|ocsMs*whL&md3_w
;UIhWU30D%`6bCYEP!|##D^87WO6yaAYxs7RzFYL=>ie2+L?WtvTSS?!N2NEqFBy-Rkh|*JE!OJkFV;m<-X7&4NmROsO;ks3V>Vy4zzKY3M*B3v3;rcm2h-3
Mr?}(tjs;N_r!EN*%}Es?A+l^a>CmFl=`hTBFBf4S?PH537a_`r`_}+vEf_M6WrWS(zrUcW+O?px0YN*&|CDY6am9@J;jziONHsb-&xXqb0^>V4;(419d&VZ
9G5fs2-e<-9piYV?*C%VR8WK`m*!Utp2TZG#;fd5f<5<q7n!T*=<3Z3{(7M#sfAlRouKeIu2jXlb{WBm`}gFV5;pJ5mk`uZ$Q#+K-Ben-zbQ@Q(CP|_2d{KJ
jC64@ht{79cEei=X|$7y%>~BRPQoqBbL|f<?YS*$SK3F2)3*>uN!Q&0Y|XRA%H?NFsho$NY%BSgo}*<(>vw+`iz`bbs7e-~>3ai32Iw7f!>Q}=f{VeHYG5a1
q}mYKUs)I9ozqzB?QUf|Si7Hkthu!DDJO?e<+^m9cruGc#;_HtgG?3K`0E@P$ZF4tVZeIgn`H!2ZdjG2y5sNuk+rc3f2$K8k(|dsb?0YuA5E6)0Do?T6=_{>
j8cB$MgY;C@9LnC7pph<Y(<+6wagJ~HFE+C2;MWG*^o33mXI*~Xy>g(S!|}VO`Wy#(g;!5QrfW@AR#0lZ{+liKVW0Id>DQr6gHQzZ0o~OpR}rK0|v4!935tn
)v2mDb-~k#x>@XU^pz>1!I}RxP!+zRFMgk`1Be>T)ZWj(7`JazS39`-nYQAPL#NUkiq85<{h*=9evyhm5t;+t>sFT6nsyQ3jXsP0HB_dqq&b&Zyplwoj(eXY
$uS`uD(8?MeVB#f-NT@uvXc3!KcMS_w9Z@1F#a{u%Yu%;s&;d3#~;tBae`+e>2XL_*Tfej%Y@oesyc0O*&1-qEQ=d)NIR+Gh$}=&I{^_yot)$a1f>rjgL!ck
pEh+rgu<gk`~4*~cXxmHa;6n=YR%m#JRI84>Wu?ElP%@r?X4B15>r1#<MwRy#2B=~wK+ERt?jle{z&DJ9OG6w8#!}vcmvd4auUZu9({jya?W3}qf5#e$e`ug
Q(lkO?OBPZk;nkEVzp~z_U}Gu9el+<NsY@~yesJ;0sXvv*$L<iL-q7COvZV5DSkkcU=2{o<@6l%ah4t#3cay9?dld_ABhTI{O-p`02-ZvHj8jz_bt$gCqsTB
{LwNiZdSAXk;7lIb1BrE!X!b`&xEMiFh`+?q%aP#>_|vqq_vg*)91+_T;p1?O8SLQ-(^Z_-Je*I#G@(M>b_x^DnrLnn%ex;7U7Dg+<nPsC0C0&EG35=-CtuO
{<Fe@d{CN&#F{=1@YNnIF!?|*`wBpZt$=7_*Og!fb0a|2nb0ZtU)=QjTVP@R;^BXw>h=ZKk6;lvH`1a~o3W?9{7*&RUJVaaZyB3N;2h&%kl`_rZZc81!zHA%
b=8HpN;6H&EjDnnS@J40IA74_N8#8gXq4DAp4^7WXcYP}mrXmhbaedz4Vze`=c#uY0gTH-m)(*Sv?g8m!5e^<(wZS84WIjWIanD#hdKX`NHPuIEAP$#XOtkG
is=m7Hw02D&?BmO3_}jeIlJ0QM)7DtTDP7rj68F8I9XTvKnzl~MD3jgY~vH4SbB@PO}mUlSShIq5+s{dy)AH6Jr~8btYF~z1*5DH1bCDK#L;AzD@@lRcbSv_
$yMrsKrTD0K_LUNvo1W4h)-ioTd}S7Yd$ZFKLSL<oqvRTQ*A)%^~M>fIgYf_9#_Es)q6K0OnW#A*}DNoJ?ncYi6!oo;EVCt>9cSg^H0S1G1|Z$18)6eo!Yb&
du!+CpaTQu_Qb){LU^n?%+w1qGNCN9{Hd5ZE(Z>!RCp^qqQw&ThS&}RQKm+-CP2~|9gS9y>u{T6&2G!|#;zvO<b)^Z<h^^Hxyfty?7c?J=x8Fym8K4AOSIWo
z&xVN8*lOOctEKIdpPz|G}>3PZL}Ij4C?~E71WIihB;TIB4J{1C90M-m)V6!7Y66|;b;xvGcNgvF|`gq!U$P(n=wPO=EoHzLe-`6?`0FL9btVvwxXtMO8dc+
{bmQuRDG0FwX?HO9pTpiavWL$qZENAwSz@U_JiDE_)=Gu4yZd|tG=0J^C1qK3moj;tp|t-qPtJr3rPFx(+J|N0DufNpC0w`)<*;xm!pP$5TI)Fn7^A|QSnu}
5`uIRcBl6j0-pjxm@E!{Qkksd%Zu$zz3Qr4e%Fd^bqbkhobcrzYJ;Gs*yFO83y%}0VXWw{2{DQcwgS6~>~<m40EL6SPV~yp;Gjt?h^tHu-Zl~+`#Oy){)LKZ
ovR|iP7-(@v*1CfxL@X16S6FO5F^xH@Bt475l^;Pc7~krgI(qGmKL?RA8)(z_=%;%h>#>laCIR7GR1B}-Zb#fCl>!%W!XWQ7y26x3HP~%Ks#Ksk6RUsd!K2D
@nY`(8&^ELfq*Q^6j1TnB1!7gkC33j$;jHgXWw*fwmmY^w8fSOg$Qtn<kvoCev3w)vLmCLw>eeb4dCxa_Lb8<Y-We)+U~X3sWfF3F<U||V>TG)h3<+9m{}HW
CPJNQ;~aP|();9Uoi-RX0E7pWH+6}WT>($)C>fTpIic<Qb=55uoOUO@Y?QmRgIdJ-0i=~dV_?4>nV|YAcwvqO4`uz-XRsSS{p*u6er}Hb+!N8##HUQi>Hyfq
`4O~>ZtNuYJLuJ~x}R=H(WD$Azjaq`7bkH@LTfdZS9Tkhmfrj(kX?(po%DMfyF{(z8y;bi=Cy!*`;HA~NCHZp5A>lDDmEqRVf0g@Jp1?GutLv^JAb1My1a~M
nUfkk5&Yj7*J2B0bu9;+94S=-96F&4bLhbaHLs|kF|tf+AS<_h2hkqT^Njxp!p<N>6l#<WG5927di2Hmid=$xh~JbMO2#3&-lhd(|0H3?@IQUB8fS|wR3emH
&ylI=3;C)AlY(W=T%!L3yG%q*(iP)WMiS<M$Rr!$)kk$%DJ=qWe&ZUz7f(<75Ty8$s+z^0JTp5MHh3nVoVrKJy{=9O;b@$#MC4h0hBGKK#s0X2t)K9!tsYfi
SagnT3#7V37+Y@8VvP;2y=ZZ9qQcLgID`c7b-T$`*t^N>E31(dDfrHh7DHDN<#b(n1D(Gwqna4fZ@aDk*h$T`mD<&KAzEiIZr2O2HhJ5H$Zq!HogUsH6pjLQ
RP5WbYzwk{XUE}pgoXi-j5BWz<`;@uC$6b-Sgo|!2GXO|=i7NX44_prfN=EnM-DhRVp>mvVY5N1t_<<!|F&Urx}Du^mYPDX>a=fJP0TPPBMij!Uk69wEOtsZ
HW=?q762!7<0q7_OkFFTZsu*Aabr!F_I#df8tUD1G&DBoCv4m2?`$MAd9f|Q=1Zl9WY$a9(%nLIj_{|_=m95ut{Wz$w-bo<@k>v0gAJo|@e<VnW>4uNc_##H
(3f)xXF+a{7GK2+w{y#m0=Q#v<X<lhG>@Xd3W9v1?V@KA1%rS5JOX@6;%|p0^X=!SMr@>Gk-jjb@Xh~>od!~xyLabKJIGUb_*;qcoFAG=|Myq;LQ>0<8R-e#
f2+|rI^h`~fQb~Vo{B}JdBo+i3^`DidOr>SUEf$#77$4B*va5SVM))9`bjUmmh1op%%}HaoX9N?V=+>spdlzDDmj!D%laKRnfWK*kn2unb1&|aLUk(ok<@#?
x1c3J3D^`xtH1#aVbJ+QAa&;X&;CmeugNwW7(uZLdEI|!jl=nZ!cN&Tn4tY})xCsKdwoQVq?&|UR5;(VCK}O9X6>LaNFMi8*zjC@H|h2tqJ)r-y>ys*;U9;@
`w!`h|FXVJah+bzxVtm1AWunO*LQjd7c5#vAgx4R+{5<Trewcl<`$~BrWU(PQfx*W_zi=ox-s!n^fV<MN*EL24dG!bl6g51-+tV>%1%dle-6b@=}X5$zmH#&
1%k@BlfF#eLNcRSp4{a$MxuCl(cg;ix!xw%$bS8NFWW1l1?RC>FC1D{gi^Z$+u@IeFEUf|gF<+u0{!>EzPIz_@^}Rs5}tHTVIu1gFr{@jj#1wz(zm(1^`ld@
mM+^cUE@6p;hW-6*Xb&IR4(4|oIRXyBLU0*TD~6H_p<-VR6KSYMtF&oSTP!l(j=D1)w0TKf@o8eMVer_&-Y>Cps92*-&O}bcc5OMOLAw3fzU`=b3!?TO^8O@
$EpoDwXB8Ja)-&-6|ORNj1N+;NXL;mGhsEkFX6R_3RqnngVwYnlZQ@0dVfwlMVNXu-(5o4;VJuyzY+EjW7;L&hJ^!#voVzBwB)uR4SS#O5!mCBNsuC8qI1Hp
c^Fo(xTerl8bqz_Hvz*Q#qIBN1As{dUiF<YojdfRJC>au<3n1GpQ0vjoMSsJ&!zOvmO5f4JBxm<N<f;E>I1;7PmRF#VW@Zm?Xy%}jIni3f-S)P&T2|u>SeHz
^^poJke(|&Rf#D>0#W_2(pbo)bp-Q1iCTR#*nlu(3ofUQ*}GB}M7%wPMz(>kTI;zoraME6a!K7$9l8zhM;?0_R#_b{Q<=u>9g|R#{^g7Q62I`BWTWq%H+^7M
HwgbgmH^HOOc_Z7eh$=n(~g(G+<9X$ly%<D{8OyG^|lm|?C8*fb?NpTCv`FzBR|W372te^(!4_~H$U<r_yAl89L|iUoH~WFre@*&nRrKA58sb&6-`-w7WboA
9bvrV{;&>;^Df{ZXk)(-OB`ZLv~b||Ah8|cnt<^3>+^$uF#k)D#!DrnQeqa~m_e+K$Q1kQeY-MQsTA9NLaLx$$>?fuN_lm@RW{@8+fTE1)`sArFf@50rDHod
dnSmkcgjFebaX`hu8(w_+$ToDgYLD*IXHHT*YFTTIX}Yw*>3`97O<<CVA!vuYczT6slc3dc0KB_k?|pxyuh|Q@LEXU81=#3T<?Q`FZAbX!paDg5LU=PiM1EA
i*}uqxHHE2j`3G6;`ow2<0CQB%rp)XgC5?CU8gGbe-%e^S`2av50wWRrMdP}(4Lbt@QEjc;7T2fy#`{#g#?CZp!&K;cC;+@$Jllm+bFNoUQw+*Jo|n3J?ZPK
nAWr-zq<d>VJ7vtJE~^;R>yv`wkk$aPqNF~DvkVr8_~F}OLwsbDw7@Cj}&?j;q<0Gh&K4bYvPU{SXX^}?|P7tX9-5=jPdW`De`QoPbXF*YJ1=ez;(zf?utwx
SUvpb;=`XOy4ZaMYw|@ak%p*zEXmaA2@J|vC<mf)Gc9N+r!UC{T1XOlbFa8tENJjiBX3`8kz4>e+gZz=W*G>5%uDBhA__H<RA?}tc4^tNlpHEmK8B78y;yc#
K`7DwcJgqplM?mmfN_Xvb{tmAS7_PYJz^ZdA4eCawt_YI0b#N6ZBDxGD}^<SVEcDDhYX3(^1S7OV6nTbx|Q-OLoJp9Q5QvATPLCSr9^Qq&Il`UJ%VvJ=*p`A
wo3mbQ-QfrP}Z9#Q~{6F%>cTC*0B#Q0-@6aols@IOZ07C4-@NhJ#A6(`y|xf@oBcw{`N6ZVe9_0Y?p2vloP4XHIw(xM&e-!xv5xph;B%)yor?nP`8@i?lMwU
g8i$4rW*5JUt1SSP?Zi*6R#IN+rNv9pi76P@4H)@y%n}&uy`AK&>gA75YbzKNU$L_RyyMi>kHZsW{*DNQ%DeCbt}h}?IdkP`sId$3az<QtEa(Tm}Vcd#644&
`NZIV<FuE)zgebwJ0Q-F;sNJ#H1=G)cdDOHoogvJvANWD7+;tD{Zld*bCFdjnV2#TXC^L!_&+#VAo<}L=3Q=aDnAxtRnv{-)G8>W{wdj9z8fb{uh9@`u8veh
+kCl@;(7G(RWl<0QBV^juB44xO^EN0TS?>idMT-#{bkoE(%+dzO4@*LA^7r9!~BUG5b~^Nu8MnSFy;1S{z1Al2S)>RS^dsv44ODyYt-JvtiH{B5C#~jbCD4$
Tt5$Gr2pdmH5x+x5Hq!Qma2}7raz^=&aq0Gds7DE;QD-y;dT`z4k|VrZ(2hszDI=23)_gh(>#-08?{@@uTW{%0GVkprK~M9XA`nQ(fj4rKBVesDD$N|uA`$i
We$e`&GsL*9U_VRGCz1weScAr(j4tqW*Ux=MH$6AB%@CksonZIML6daPEVz%)YJT3uBV>h7&fXzlJFn>JeT>g!nqh_St8l8lXDic&-WmXonw5T;4qPa5L@<=
xPo=e#1$^rUs%@B0v?Sv{|V;C69xOu>Oii9$>(zX2ux5iQY8Rupyymfu!dW224bsTe>b!;l-JyEa3=sDy9FJjZejkl`KLfbK&jn&4h2-UeS`dh<fCzPKd2yP
;jg5VoX{|O-)~s&SflE(imT&;i2n6;%I|dOTV?Tkm^zONya3YOFVQbRC9*C9-h?2q7D>A2S%Z+rPxkhu`u7e`$w1bvp`tl2<aSnVqC|S@d8i2Q>r2oFd^o*O
T&b@af4hVTemmi+;cSHL<}bT^zY0D+Ega=TGpTsBq-TlJ=Sg-qVxYu_NEyF;LJ(f6Y9r>3%?co<C}g#G&dckB9wC%|^>3?q0WIs#9<G#;xYSP&F7xcw^Z2t8
JJMT6keX-VZzDiZbGgoW$c^fcvKYHL@G^OJZ4~Ry1+D*5=)I(_@$GuPosa8vwG{F^E(-gfMO`Fz`myg{6Rh*`*Xz2n#D$aF4s$7hRPf}jvP~mtg$In;Nu-!(
=*?~&R76wdJ41DE%&+<a#*V6j!aZ#Uuc;Qt*X!fvvtCq++sTX5lnvfbu0@y6)iIA#{lmf*o8Vhj9#*0-<xH=ZaCFmPTW`oGTGy~bghqdR(zba0rr#z`*EDI4
`X+G4dhsFf94xVwsf>NfZT)ekX#z$G3C~FxLH7qvfQ;u5Y{UBOV55f?Ajz@mpxC+=4K+VI9L1}%t3b4UQt6uc(%LPh8G<vkUF4n*6er(qLl<?cgI2i_=M5p6
P9aXKZF<%E)Edyv1;~!8>&pAWe&vLH?kEk)O<c%v&{R4pE}YKEyr;<k8mTGgH0t|g#G4vrjJ<4m2&8m9?vtm3^vkY%2+qM5sqVSeB&(6Lk*KqDP4!-kUHvNW
I<l;>T>W@%aCvHGkFgi%64&BO-AN`pVO^0a`Z2}7q_up7?WmKckKOK_`2E4f+}arfe@;@HJ+Flw9FHx}z!1Ko1s$ESaI3NO`$!^!zU1W)sR&;0Jminz%SmeS
z8C+vuJu#xZhE^u<ZCr@1;|DrtrTN=uQ&?14CsWv4Il)kbaQggO^>?0sTO&SNR`1txW*k3XEzstsaf$Jd7|@Fcs03kq-g@#wNRX$XUxN!Ct-l2Ai!ZwvWaIv
cB_}gLf@^xM%q7u^?~S9zFg_Z7miK>WR`D6kw_ca^{y9FCzl*5yauVfIdctjq{1TvL@VZDGT7xXXv!6~vL*qtm@uJ&)%58ls3M!29d>F!#gE3VHH8Rrjffpk
)g1u)V_Y6w9m5qy6=K(3CUQ9_1A^96K@<}`dU_I;Zktcg{GQ~8;w;*dEOebhJQFg;VQ6ED&zssU+1T|1EPd;z!)>Y3T~Izc3AyH4^M!3oeq;D^|0fstAp}s(
jcFMJclm8gfCDaXI4UVxRn$Ec8^zZvmNbQCIh#KXHv*~|aO_OmTWni9TW+s&1V~Dv??IQRO082z5nsUosAd8o5Fc->BdUHjglVx;%;1^BDV0vDJRg~mMMe#_
%r}CSUi>Qp$durYJ3!>hyj%_~bW0$da<|tC%9&AwAK_UO1*{A1(KJR8PMV#zWwf+_Wx=^LH>uZ~mAh-FmS*>X$GQ=0L3+2r8LtipH;YdQjER^2Zr;(nb#+{a
I8JJ$Llj+ulQ?0(Tfmqd(@2(PxiBnH&Qr`ho#lE|+Vrq(p(+uN=^*yau!}8`*f?H7-U3slJ*!hmSqNwtS}`W@W2z3!=-U)L(+YY`*G56kNGp`i+K&^5TlTh8
wae=1l?6*hfgVM4SY<YV`{X-hQ+<2JM#R}A`wG3XHO_E$_k}d5N;8}{qT~bs+*kgvKKlS8nGE!NY;RkPYv+W&wJMR?&90dYyuvoN*9gEf@zL4-I$#K-X_slW
6l5nEQK_n8jQn9t$wbIb$S%NHNI1)>jSbDZK#aqqOw*~IwcO;)-CsZ4R8?ut@%Bz=9mGD1HdDj)8D`14%OF1dp7Yp%)(aV#0qgjoDBnAy*c$=)Et>iz?B*rG
9q*OPHC6x@D>!^DJq<MExiGonGEi|K8c&dfB(G7b5EV$`0K}#_zB4ddq;QrKLp-9R@G7#)B<BIY#!86mg>E8s#1-GZ)xfKHL>C=NAaV*rU9Z&4_l29H3tM(l
byVg7jLNcBS8$2ox=nvf!~!?qJP&2U!^dFIDyHYdtb94#Sa|L@44RhdYJw$u!DzP@u}n_cnI@*R@#?peD1Am>tnh0yiG78<oK%VBw6pqL$ZnkUmde!TAI`0x
l{=O?s&0t6J_iB8i;Q|x0$|3kW~GE@BhX&aKqa7~St{|kTXUBW@gR<W@V{ia@xe)^1SBJ8GI^qdb-pQHZwsk<>4GsH`(;ArEWJ3#f;^vM$6(=U^4*fQ_5D_*
m^pXjiHN$mB>V(`tjyReX^KqB0no9}wF@i~><z0%xiZr$w|eZyU!codm$UFW?}sm;Ir=8A?@PT67f$M4ny;lS9uu<nr#z)^w~ep$(S@fzhkw(GTV_7INYs`C
&75|tzDnfcSy`>5$@6<e-&dgdb<r&^ySt!3^^^&1*k|DijAIG_MS&n+;*M(!iH2JxYt6EN8iX0t{P!&rDeml%VDNIzL^;k|HpTU9utQOVBZ&Gn0SBWD{jkFa
p*;w^s;-*zaXwa7^BYr~b<6JeK13)qKN`NA&y-|ZNyf|@tvf$6VQ}cUd9N%$00|8-v2VcV_Jvbmzg+qZ!AU{hA!_w!9y_ar3^(O|vhV=7pjy7BQQzsv$55&9
3|R&5^U4n+SB?bD^w^kwK@=27?NrLfK@-n;rcln)N37J`<`loiRIYmVv;@LE!u478;G6f$BMW&ZKp>JqVQnmWU~4#$W<T!Zx`KCQ9yc3+YoAs2yYOrx<P28i
uzSk{RwTK&M1*%opX5UarZ&T*6|RbC!M_&xU_sgv1cx)@!BePXlV2_W!J_eMG0Yk3iT9ngjxR7QdLY<Bm6h5h6ovY?aE9gCFmoeN^&|Gq3(MjPmA<Y#DkRUO
zO?l0C$lcT9e~#VKTDZG;Xy2}JmG|q%ia)Js)$Akin(C`bnH$t-?a*;!4Uw_S_r)-MR<GE>d{60)Winlt5AI9VuX#4-j~1Q!j_DhudZxP$BW_7Y*O<?BOT{4
v|bPJD;5siA7YvlH(a9`^hmq-GWntign>3iOX>cAhKyZ)3h2Y<#p$9}x^42U5}^NFeQ#agT;=7hG6eA#Ri5C7C^KkNso*kPvc`4}nRZQHMd~yu!H(Ex7$VAI
*Amflgr>rzZP1*==$F%wFQ|{*;IB0Kl%ZR?&SnGyG$9Dp)G>V4n~Jx#k(<5e<4b4w-IHWEByU4)HOcCUx(xk*rr6BDr}l}t`{j0X(+=YGFmGnO3u=#eSQs|f
Ug(UjH*CS@#7-QdASM`TAm_d$1o2kVAzd!4phUU^#>Og9=tP0<CvfyImEiNC?nU?nwmZZJZ$fNFr>b7ulmC7beGKwSpf^3uTf(mhxa0ee?!?y^JS&!GgCfeZ
s-0-b9Qv}JHRr~vSAZ2JWV!B1t^sRsH+4^_qVUL=_v6WaA(3uKnAJlSNQC`UB#d%36_YQwOj^%E4sS;LE4qDPr;x;5O7!a{smize(i7rd8Hb@mxu}BDceP|J
mzq_S=PAT6YJ5Ei{**re3@L%$2a6sxc=H(EReQ8a>}~fiv}ki#0$xN{yN*tdjbW#Hv?CyBNmaA3q1K-eLihS&JahPcAJMbQk6(q%pYH{T0Ze*xX<)+NoAssX
Y8^Puw%oc4>Ma|;Ot}iDB7)}MibzR`4=t=KXkf+(`&mL?#A})5K+u|ROiIC+)ZSj!VJxu>4+_z+u+HZYaQERcXFUGEdLI0S+2O6xpAjX*Pq#31z*;Ifd{m4o
w`8wnpW?A8C1GIL@Ehu;oPRD_5<L>}Op~<y{PF99@35YVDGSUGA+nlp%awnJLy30LR=}=+@spY2OsNE`iao7(oGSVHV1xZ&->he30$oh=e(aK!SDxi^L<3Z}
De&+cz54)#_x-tih{8)~?0qlb>Au3=IZ=dMxk0A|*nOMGe*&|p$0EFb-Y$d++8ch?Qe@3+1jlVAg;jSo9S3~GgEHATE*;@8<7fC%*^O}EiS%6AxDZ_k3`ngh
A+4WfG-^+1=oaSXmi}L%V(<T=A{`+>RMZ7MX4{mPJ|(abwbb}-i%6}FP@h)TkXl)5JtP)q_t3e&XxzBAq;d}E(!UMV_`}X?kgQ~;4pLc=UkH$T&t^V^lY;hL
o&f+fHvWt{6F4&P24C+srSrzO!nHq1<W{TovYRks0(0ShOe75@_k=&J{%@j+__N9}F&2ENTOA{X>fO_NV~{6KZPe`}J@3Cq8p;l#NgabmfGSzx)}jMHtvJzB
<H2gFXb-1(E!;H1i_pGKgkqbM&y6ftlr;@zcX?Oe?APiSVNzLUeo?K95MwmY1$<UDu`Ze!nyy!v>Q;MbQ5ua-{hRVXel|ApHF#XXORY>utkyJ;@XT1FA@9B)
bcFcUYma2K6}nt-nDxrdBY|P~PHx&?wiNJDX;I)y<cPM*6=~ZBNM=2g=wU(@%;fWCKl7RlW(}&0#!TWhkSSJ+=19qlWZx1m&@&lAmc^5iufTKrj)Av?6O2?z
i0fxi1GdHSWaYHb{?IU9oleD2Q@%RL!f?x#_dDB%6FOcR!=g0Hdojf;SD8ZmCBu{7bZ^&POZR=wln&9^92>gniHOy**D?ccUhZnF7RXj@FXj8~44t>6Z0gy7
ItB71;`^BpmQd-fVW3>i9+?^fd7!xJUZziLY=qx6!9yv}-#Pimz*&*?S`#0ZT$rVFkE|_Ujv{FIE2Ffc8p8d2C~aunMw{@uhw%dNcTFuq=Zj&6gr5i@!B#B3
xm462AtCUJHK!kpoQVTn%0@^gv3n%%=sQHI5hL!a=2_&kF55LjDN!peo@M<W_x%^U2`L`8XN7pcP6c-2)n1S!I+4DbTg7{#XUC3<8azM2WkEO?JboN~Q})-W
G`%Ds6!iZW(gT2vAj0E<xZ`xcwJSz^mvQ$m5&6xzrsQnOR&3gSNOV&Ygm7Zh6#~r?Lo9aKe+~ZJ298n%_*0~!;Mlr6yKyuZmzOtG?)qNX<+EV=lg;R>i;)qu
1fI(Vhi^3-o5d!Rk}QQ$W0c)t^_Ze1VvuGKWd$vDx2sNuZJgiMV)XO-{S|p@w0Q2M8peWv4>u@*HXV~PwjdJ|<HR;VG}C$2q|0xs1#2`)N@|YXHV#ZZtqKCg
VC}X1d}qS(9z!)wzcI#tCoL^f`M*>WScxPi`&`<jsTwCP)<l3P9FAFoV%AxAR3tUghV<Cwc|g(4env}b%Qe|ZiwWC6IWp2Y_dYgvzc;Z%sz7K0q2x3>#bNFe
P;Lf$ix9aUNK6mUm4~d~8-CTqY-*@dx{$IwY~Z&PsYEqy=A!Xk$|!Dhh?;3D1^ZE0B0fsFgLcuFd{3YQI(?xB9?7ATR);R2@ppFcS9S^=rJlSFuTpsA{cr1l
*6T5nOYTT>GdV{F>-sJxN67>3D>8OkaljDJ$uUVVYsg5HiN1nMXpkR7a^}6RFP$&5AKiGO@1YWgG=;wlK0LsQ&oSATvQlqMvttu^&QcTz(uSn;<-2Is<kHAz
Fi!1V;*uX>MrDaGk_qxe;zjV)C)c#~^~S(iDrabW9?wxk{Gh6=_h3-`E7vpMna3<HE<Rn=v0_M8j~VkJLMSA6yU-QEqu~d%BnaaX9z};g6W<5=T=8lp+TdES
X#tq0-=$g%KXh{Uro`(izGsMoGRM9rC8HO$WjC95o!=q$2I2)EMdGXfE|YXODCj!CGwGUi3hdtwtgcrlEcb0Mgq^2$;w(_NJ8}j;dW&Ea!7;F#4FuPS{@TDR
TO$+i-E6URce3@<G5`Kt5$ba$(~*-dHf2?c^j|HZD{0r@JN3a1zXqJQSjRr}Jp{3^vejZ4@-)?F#n{c+V}#A)_RLOEB7<aeR?a;GBOMaJ#%DzX6HX8&q2}`g
aqdF&FU`yJM&y{K2OfIy6~;;+nA^Ee$mlXcOh((tTppZ{Xjo7#_cO!)JLvsYhk|UgS;o4)S40^X{-!;Rx)err_U>O~ATs4P$@*8Xh6h485NLb)Jb!Wi^2eF+
m?lI1v`k^3ur2fn`St9rIg2TRXAb$oczRj`BDv!r6CSSzT;n`B7l8tJaHkdG<Ex`}8fj;nYrE~HTNIn)8!%)(nQZ%}qESzhc;_rW3{X1xP{wple|R^v6b$+C
Q4xFXwolnphvpI5Hf4xCJr;`iKwuEfmqxpB&Xq8a$=E{|9Pk46G)~f*h1xKK!fJ0g>PS<HPaxCve8abAexy#t&#z3D1IsC7$BCdqf>k52uEkdFxz$AcH;1)F
jexK<*_nbu8TNV-SR6pAjj2(Bzfi<sc;n@oyzg>w1O{UfJ?pjG@#c%_so8=vk7ap<0d<UKSeuaz^*1<c6WCw;s+cwAk&Iq1O?F!nfsy>pg_6JqccFT>&S~N7
?DPO#9Qp>+_uUo)x5&)iMtsh3wDk?$M%*1!(PotZWv~m<GxUy6RNs|{qFWs5S}N{|u)vJ05gh*tINA<Z#)_>TGL{7A8XFhL73{|pYGEQth@fyg`c|yG*Ay-G
-D)H=(@>!OrG{fS=gpRmkpEzgU$zz)aVPjHMes0cz|IU-0BRoWrjzG4&XCvyuA(gwV}v&><dvO$qr;c)D6<-$*$*m5xd-3ME-t#@!Kt_o^~me)cpk`ssG8a;
CsIlS;YE*o(gCd@Jm-{15-jWZrP(V+-5}dN`iKMieVMtqx<!*r)>-1t0u+T8J&X2YLg^*qJXg~D)7J{C$ZeF2cA1fQ<sQ%kt@?pfGhR}e=A|js%ScZbAe56f
XU#di*c!jvB=4}}Qv*$Ys*ug&E8rR#p=a^($jDS!O#0%Z&hLO&e#fVjnNN`AS^%Q;6k8lEPx>&^ejq!5L!vM2XbK_h{EfaK;pN6u;Q>F@*<X(BSx<gU55GOa
E*2))e3g4D0VGxc!X0r+?9PEWlK=sjpCkD1#oP;D1Rr>N6L^;QA5on1fNI*^VHC0L?HOhnW{TW<VQcQ7jIt`~3oX754$J{Lp#{|}bfo34X)7z&khavnt2i2b
Aqjp0(TK%ksFXpo(y*{L*Yy&6ARWur*xvK(Xq~$#E+iO%Mfj5}x87bo0F}bh^vXQg2E8+O=pG1dUv($x=woViNlA<d`p{`pW=qJU8&)}$xQjG<%SbW{-Y>n<
6cSL}3)fx?g&ZpxKW35WJFvuj%qBLz)wgB6=n$0R)#81%Sd5UF5HdT7PD<eAq&7>619be(+f8$bh1jOccwq;67|0snFZn-N${NaIH8vqCM1Vl*$0p7{<M=L)
`MdIwFJe6i`w$^ZT(+iO;f>3)!U_v^z)EyXFPxabB~SlmYlU9SxUZC9%r%~u%n{cwCx!+L(Py^_+~}!w3OnBplH@J_kOfj3%usW0+CJJXV@cu7ljxU0i5+p%
Zh?y;@pi^i258%!go8<$I#e-<2dk=AD2D-)ZzGdmEe~5rQTHrouIZw$n5yb{>D(lB!O53vu0wNH@y9<dBZ#<TwN%$wzb&CSV@KJ29L=m7oD)F;!HG*|Lp?<2
q;P8LVeA%2401+YGpJRuV`R)Q>{BYLoOmZA6>98WFBL>c1n%(dyhw&kf%K<_WU=wu4$gsqzA$!YYB(2a*I*tR(5#SqnG`MzP;>$wofIz`jvd^~Xi+Cm9y-BS
KgZ<r3KR3aRsOxgxz?_!do3<?5`-j7cEI*uR^LaH^V=(sC!AaMy@TR5$w2O`4p{6chA)W<*L*br1=b2fS(<Ypr?7M$1D~?{*;JOw0=eKuPb%f(Mm_DK))dnf
dcqmL4AFW}nf9l+?W?GMe??fb1&mQ<vf&=a4FgcOT>h(I2Jsjxck}t`A&X%|SNNS%@KvSCb?!<-&(cI()d@jMC5WU$5>G<p%!%~2AzyaQh~)&`jp&D$>*6)X
m-LWcUe<P`f6Mh)+}2x7Y6}`Uv8lC%1NsQrppO(iO?x={u3L`_E6uguMSmuEWRUNQ2iw<X(`ti55<LbgZ1X6QM|;W&h^kk~HNWkGaNEe{>jw`Yy?lA)LX8)9
*_U)I<7V>+-#k!^SdO7rjjf6Mc0{ttO?9~~g)f)lc0yn9LxxG`J58M2ldeC3cE-2&Ivt30$UXujDE7u*=U3dBl38ZPfS#Wd+KSZ|n(nTpX-UgQLTu7OWNI_M
!W6jV`f~LoUWQIR&M@z`cm;)tl%C8AMFC$7fYwJ}@Ta~3bW3F5S`Jc$;iB>Uc93vphUhaAD9+0fbBDd$*+$phXrwWf@{ngWH1(mYe9j7CcZ!A7qyre)+bBLl
pf{i<>2F!A8pbe{^r0hD3bCCwoRs{&PKJn@J*fVz_|vNVbvmTkEHW@{zhCg4RR<1m^e7hp&8^Kf?dKFrBJQ&xm;(kAV^onB`>*ln#j03Dc^F7^&SIH%%md=L
RUT)MC(t&?VHrxnBC8#Jiepu!vDz<~kBc2#G_t~ew`*%#$tlQ*60sl**^0PXHLSxV157?+=%c=2&5U2&4EkfVv0C@NHQNI$ZP#Zm<PNYXHBOT;i+tqnucusF
^-iK3`RwD>KWepmwNG&_j!j-3v4f)bkh}f*=|_a>A#HlYvi;VUOPy1Bk26_hT&%4|o3tM%Sr5p7&Q>t9ga9GjUDJPTva6qoQ|9h}o3h9>n2t8AHwhfk$&R2p
hyhKd9NXv{p<rik38-E|nJbdOsVw&JeZu9iP7fRd?W+Iy4JTnOblYdBdg4QSe|o3_ib6GMe3rv!jdJ>h)%_T(T$X@!CbbpoE5Hg*`=|di%%fx{?=AxvfxwdI
>#KA0@L5wB4e=ky8-gw*&>fy`fK<BLAMMyD80|KtEqP~)b+B%S%ub{YTgj8z+?HG2PVFO)ElY)Nhur8qwV?+JKT)8mmb2CMA_-<$tI1<6<QrY0_e?{tlx08a
1x^P5(fJv9$sWKWITV(ng|v6s+_LY@R)y}z6B~haZnm3N!r(#;;O>0-?;s2>F9Jm72Bx{kG;6+BSlMhGzD9`8uDfQiRi&i7K+}(}l#|%^99$6oY<~xn54^lL
zjoXh$rQKsI^!4-Z_W$An-->u*^PG{=pG-jdfjqt_V=p9xCLD(?z{jO)IY4XwJF$$;-K1Hy~D@So>5GX_#8sMM(pw&F)J+POh7d!oMMQPY(h-`(LF%#r$k!?
OQ&)OXiqcfcV_m!+_|F9#Qa~pU3J_{JPeup9tt+p9CjR4i(~4rCs!CG@Oe25jzDXN&n?lqzRlAti#<G0YG1S`<?T?$E<M9|8R@h@lJMT@TLl`cu@j~kR`aM%
8s!M2)v=X4YoT_fWr_&}EP9*nN<dp-QQx1j9`%oOr=VgbZ^;lIB*t{9eyRM1C#evgPoINoH@T0^FGT9_NcTI46gNoYd9!*KErczd#y-y*vKVHqh&4bKyns8z
{ZM2}`CkZ?ih@~^HLvP+CV}{OvC^{7tDGP-1=xA{1!$ulIuYJn^N9N)=_5`rNq*VKYXmV<s7H&qGb+p9TGuRGg2iesQ#K<bE*Yo#0>c5I4`CHFExfS257nf~
$F+@M!E5dIYWuwM<>%rWWa_Z{{yuv%`*mjb0zD!czq}@?9h0*^n7D{lwr?X{+6B#t$g0#mlX#j-n`wncn-&q#jm_)9L2KQcWG^UtO%^AQ2F9S){K(ic9F7KL
F1bmAUQ_#3w`6W>4L(-JCqa%hqE>Hc6}nkzYZb-8*TsPX7X#S?30G2bL%JInHEe}aUI6uu71tQI^afyHI!ahpv*l0*YEX90HBlp(^l)is6{7g>(-2wt?E07q
Ln2z&4K3-(V@r<O{b3OL&EYyesvB2b%<<DLvjozK|AsOL2H1famZQ=iBOb6A5FnXx)DJ9ipn;|oO#PB(p<P0>!=}X4l{EfxjLUik{K!uJI7&!}()2qPA#N&u
OE3c@Rd8KbS3aWrvA{1TsX+7I7)x7Jc=4TaUBT6(7=V87B~8yh7VLZjdq5aF`3TJ*G|CLDxO^2T28qGV9WI{6;NBn73T%2lu>w8-s|NU+YeAjt-?_m}z)(it
M+V}oDT$MRuU-(d%@U1ZMii!Q{q#!UE}<koa_jM7e({B?oPfNo36d6R6MXT3F8C3D1#r{wI9=<EEjtZjkyvf5)z*moMqK-b;oPjlp!M?(AZe#KK?Z8fL(Pc{
oW+(L0502T<1tEQ7K*&g1k<aVPtZdut(a&Owt{ZCHjfB_biLgyyP{dkvdx1PAo2WPr(1;(c!9zD+m8y-6+4G4I6nu2N@RRzk+M|M4CTaLU+m`oozv@$FvGN*
{T@V6f>sArI?9BqMY4e!_q--9BUJAGy^SXcebam1x*KQuKa3_O@>bc?QV`<8-Z=7ojSFH%{rvC-yMLxOwb(TFd;14(%RrO#Xnw9fuoS`2)*xuWctiN&>@lG!
Y08qhhvE#eA(<gaa;bS*_9x-^)Vo0#RCm?>Y#}>2IG;%Yjq_wE(sbM0!4gNZ3D-)HZgGm)OWplBUjRBbd+j~gRxnkO^W#{VyOE~w8bhzM%iY5Cj$XEQJzz3X
#TPOji4a404$ra}dp<I$1om@8%>?1*t%E3WON-LAV*JE!RhN&ooy;ZI%Sqc|81<t;2n^3J<!g~Ga$5aj{HsH+{k&=EckjbqMzQsGnWh;2RW((8uvIlXE1-cE
y|$=~A8-dVOSHPcZ=V04uae3XM!qy9N9BPHCoFD{z4k*0T~8P701`mUWfel_g+S_Fl=KSNis>j-ujAKP%ASL#k%no^H|PNeV~0Q>iIy?}jLD>><NL1rRWOi3
lxopwFe>;uoIUw@p(Y}eS*zYx_m4O;wi?caFj??aT^EmS!#3gl-b(DKjF789Xo4uFL&B@o`B4h}hrzKWK&EKQ#KK8%?gxv!G!+8Q@4>$@%lL|$OQ|SRg%A!R
QGe{Ztk*Uo1~&U0^nL6r+Om>!kv239k`1bG{bUL~AaY<iw-5xtH}#&>2~)`-$K_<_A-w|EcX`>(S^FWBycikoT0!%jHXZ7rhbk1<@Pnx@F3u1;ZH-l?1u*5b
P+n%e3rno(3z4B}@{lUKh16a^`V}+{#Iu*xgW`u_)r!H=Dh<KCW)Ydb2UuM&6i7$Gt*;k`p;nMgZr%F5?xg>^fy76oq@$NrO9QFg(s)(P4J;>{ahWW&$5v59
-?QCm-}`HE!Os=z%tt$ZpZ^BTceoNiLoz$<A1EN?_r`{0$4tN2-R_8^=t^aDj$m60J<R%LLbNq18;(icCx|)v`+9iOn*CuC;_e?Sm$k9?Dftb$?j0${Q4XE6
M<Rhd9csFH+D|zoLK$Z@4&slAP7Uy%EO(`1Er5N{vc)?gjJe~eP=}421FxH@xb@3FMfi0eBo0=P6DQy>r`;oEd4!XVUQYncriA99hZ>m)?FbdIe>LZJbek^*
v2Zes!}8g2b9HND*32!~$>&|l;`O2JM=eo2c0vMr2iffYvL7TYCC;V`D`c^KDse)0+txkMtm0DTMRG;!lL*wJpiNH$tO5R{seLJYu4#qWE&Sz60Lhx(>JQ(q
9(Hx<r=F8ta*(?Cc~#S2WP~1vv7mzBRF76=U;XU<@62fC`iw(;rm#m*P5PtapHy^X65%ZoNvCRnVmh?LbY;Kk18%;SN<6U2s?$1yUM?Wd^H?gnWmljDXUY=M
ZAPGxl|rKkf;hT<(SM%ulM^m~h~-CN7E~Yaf3~v`Uf4ley*C1IkqqR#>HPE=rLeWq=OyU|62M~8tL&an2ZxoRO)<yZOKl#uC_AiJlw(=*hqWP5G2f&>8SnGC
ogO26`y=;(=lh?FwJ|WFp26hA{b=vRi*a4Pg1;Pp?4bM&a<BC(O?vPwF@XDJT$XGLf?nr4&S*L#FXuoS0N?k~PV6Mtzo8eH>yI$cJi&`8x=ie=Mj$(VC`zt+
G{I%I;Y24@klA+;nmChIqj~b3yR^$|8vkmCTR`kJ*^%(opOM@y?&Yf6^Q&;3J2y!Fnczl6sk5dFq71<FHFH6@Q#Mu2AnT-nSKwXO^%sa2XEy^2g^0#NQ+CD`
5I<@T?W(Y-DVN%MKXV;3k@N{`-*Ryq-Eu(sr_k`1q5I1pOTXVwrb!5>1#Xy6)SK|RBa&togAj!NRVcs<6)F^8%;RZ?#?c7sozHsyLi>QV#!hyojOChl0|Oha
!j;@v>?Xu~T$)O8ic01|=SHaSv`A~V(_^Bh*6Cb}uT||f=Oyp!oBi(EmkU1^G&7*_z+Y2p5QVy;GtU0ywn$E<mLLQaYK;Ex=9OHtzt576tI3R%QhJR)_UScJ
Wapk>0K0|l=EnrKdRDvQHDi+1a`$W<+exi#7ynjy5uiN?{w)h9uTeTWG7WWbp!s~Q#t|$>I5#PWk}`d~zToEq=Nai)c)!<c67HFV=T?w_m9!2dKo8f@>6ESa
K`=wz!N1*zR#e6W&A8~{np`vl8ii<wMIVKoE)ow63i9*}Tx?jkC#R}(!S>+$yJQVq2a0$aqzK4LHO{&y7*kd<m=+`*Ddbr7y2ERZ1eJpSj;I#h2CERQ+6C=b
&g?Z`#=9QE5k<E&hx3NhQxX(m{^^i!eI`o}Xe8T1NV}<DcK@P4`W~QZYMZ&aVrF<<|3vL?<^HUc`?Wn^e^iEa0Aja4e4GnYX;jbGmFE#6?%{t>sU|fXQVY;1
Z~QDD_>YOEd1TTlT*k_38fux@|KI$kcgQA^oyD<{80ktTIUVM)T-2SQdF#i6ULf#xAU4Uc83uGI4uja$vB&cP*MP<SNe05uA4FeNndbD;)}agj(*f?iPo9>z
ndF0_&i1!-S0vgn%{6QZM{ew+rumM9klP&>h#Z$bOppcfRSRQx<b+chx6~X8NRJMOZCerY2cw-#=`f0EPKyF%oe(nwB<%U-Ma3fvc$)jyA-aJUg7{rM@#u%0
U#G?kPVG|Qm4!mrSy*NW#^4U1?p`C)pC}g<XW`8qvG4}2tIjw?6^_sry9Q!AAbjH4!$^s3n6EBVy3U~J!ds);r^^bVd3XJ{()C#@dD?n0)!+Cj@9*FqS_uE1
Or=+@_U^YauwGxa{+a!t?ayuwTx%<f@3aGJ#NoVJGwpG!ec3-wPygJ~8F__h%tVzcxO72`E~Xu`e)n`6WeKf<lt+!Y;G$TTnFx+<m$rMO;@{v2Owce4#Ug<)
yT~EKWHup^`2A)}*%4L^00m_x)@~K}&23kfbPgD|Yl^x;;59&UCQ=?nAMqy^Yz>afODYv#`%xId{ObPf!R7LUO!GAG7S*l(&|d}{M$CyPzYBtn#J;&?jzgQ9
I3}L~V`)Li-m?mLkq%;zFViiM81>#w@79Uwbb}r)#FMrPfR>3)(W2i5QNCH;p?#xg`Ira)z<*yhOQ;n|Xr#g&L&#kPMyo#suiJZ{<`en6TAW?-PfW|>dHX<Z
%HjZ5eBn(?{PY0dWz4~~Nl}FiSlw{%$i+IJqQ)Y?r9a@y?0r4EMHvVvOt+I(HP--RC*hCjU_pPf@KgPb@Gh<stxD%%bG-=RBDF2-W8)P?u8hTog3pET#_l&h
L_5lJ3+Kta@*QP9x>O6q)^zvnhvaQiMI<e8JX?N)w?J0BOzRg(D_ND<*dhgh5J%R5e8dY^QLuPrI`qDx$tJ*;C1hbEfnG$-T#n_6|8v;&S|P*D#-x^#PbCFS
_^U^EfRCw@A>9J94bjpbU>{2}_ZNZBJ=5;GfkB(?DS_EjMm<t0;+0S6%k&=8lNGZ7G|8@CX!zTst6?efo)sc);aH$#P>EM}k-%h?JEXE2Rnh4wvypR_P@)=M
Luqf$gv28z)I3F~Jf9TeJ+J2E{8j#P6}cf8ANR;8GQR;13g8#%@fB)a!1pd0s-H`>1t*zg%UXBdkka)1?a_H6=!$^{+{cs{2&y3e2QA5%RjFcRMoQU4T)%ns
aVe`n_vP_}9P5ES&ehF8Q$Ii*IeES6LD|azp4Eo0Qx7HhvV{Aisg-Opg0Ug`oWT?vGIYd7q*A!N?jn?z^(~t!EI&nSIJY`P<F-Gg=!AgarmTt7aK%(R8#HfU
^r{LpUOvs=)u9DPQB+4BGuitwLOadtK5Dc0us*sFMEKPYemGO51}?mE#?Ew~%9%GfNT<x2Ah5OrFSr{RF-3CO(Eq{@Hg5y0F>fp#Vde3`CZQKJVTg5)7=@lw
JA>o@;NHZ@?NK3;+EU=MipHtDi8bEuLCJKt5;}%mIoo-5EKhajwaEI1i{G&Sp+*nDXeE*z)ym0YCoU0!&S)j2WCl_EszMgT90td7*fTHt@-)S5@u{+MLEgy+
A_S|#UIAm-i}cNQr7~i@k|Q-fJ2pG7KX*rKb!?cnlXKgH(i2i~53RW`QJ7keJccmN5;47B&=j6J$f8h7Iek=QS=W*pS}Yw7d{n`$=+~5m&xi3zXI!})1N#CG
K}xJU5Yh1z*yNMyx|onfLA^lCnVbp=9jr8KIx)-iQiNLCPRl<mWn)y=gw<{R2aF~oWJAm)HCoDPsLa0fBJLZSQHQ}pWNNSS5!AVH%rs#a@xsjXJps>$J~iGB
{a?W=Ohg3z|MIaBr&QS~+lsw3b}I_sHy_Pl6|9_a7EZ-B8AKL#lVgK!cS~*xU*AOKtjUgChy|v#0+I`cu7Pb>k5Bp5e=id<+h%+2lyay6`;mN+5M&vu%*LQb
WnI}PXVvMUfj|%9Cxf9t_R7ZZMMq|j22Y;7|KB}E@^#JLlUrQ%Bab!b0Gm#Fr735_=BrM`lrqyensMTSew_uJsz%;m%t5;~kbPCgf<A@&u@!}tM0!jSAr|aY
vrM+Kl-}SG5-nWmJA?)Gd2)t2!@Lx%R<MZK&Rc>i)=2d4A7txXyicwE@;RS!gG}+7BPWhuFhw&0{Xp-B8DBIPqCvNDWDLpfj$u6j8k+T16c%=8Uhj266qr3G
kq3}71>Xrp#FKRdfg#t(jE<=l1r+;eYm@ts$Hpt4XBkQ6evHGBqU#04@07a6f){s{TxhebdhhCwXZ3d07**DBRf7Bs1ys=x`5cuF70k}E)jL<){ML4OzR-%f
RXzppjXiH7XDh+Oc1YUE+O83UBSl7*H!n-XqbqrMT}Ehx`3(tg5&kf%iier<2x55U-#fupqHz{6MrIk;2%egZG+RL2IRx)=0z!qmw|k>fnS){7fyKRK#t`Gj
jYtVMewIXblt>gxwnL;}udKgL{@HSSSb@00nS-`D{%$vRvtDe#3$esW@){O#C=&-5y=Ec$-S!FpxyWgmEkwTb(%TaD0)o5dwDkn7OW;5NjU!pC>pyt2J+DY{
Ovi*R@PordS9yJ2^H}=14XYuU)m)T}a<$&p-eq_m%%x1XIIj%t&v-v;5M2}~HIeKLi5EPLIwyt_D^JM~QFr4!QQT~AajPzVH1PLqu8US+EDcRzz$D=ZT`_I|
n&J!?TEK6MoOE%CeE)YTdXfdNJUd59P~~iI*o<8u0Lp(|NO^{H#Tq=PD^M2qc6IuD=QLzP{0x6o9XE1OV5@|Mkbvo3{F+8=7GR|~1fQ?%iY7gO5FS?krJlFl
DV29K(5+zp>>(mzuu4-yo4Nl{jm;R~M^<DP&DqKY6tWHu1M}HGsyNs-a929~%7`SvpmUt$<tPg{{pPbPJ!mA7tB3Ht=~!}7Yy|KwDC{@$_(%*MUEXM_$sXWg
CX2^p@;!}M?+AT?G;zxRPAbGtdkx+^%vIKtn=l)i(qJU-#_~9Re~uby2daw4(Jq3=Ln%#Em3YJHHYF_Dx7jh;_}zFhR|uXLXJ6Ub@uNn4w27u`RX#t=(**)d
)biBlW5j~vcEH~_SoLD~e9rvnkH1_}dju#Xkh4##b_kz{18m)E;Wt`AYkM|XDCu^cV0rh^2!|Zxdhhmv+CTDT<%!WJrN<KO*^3Quj0w+P3N*f@z>xg<bJ8_I
P~kSiZZ!MuB{@GxsS&Pxc(SG%j_Rkk&8H#va1PpF&AH7W9K&zuwm1t>)T5<i%NIHn%EdEkmsjSm?Inyon<!V2P8oAMh+TUHM0ODC$oBsao7>mO*2PWM3_$^&
Y?&~B<>w4AL7-GHtLl@y5~ag=7)St-1hWqa01zwR?xGISWfuBEWYs<5*(=!??AE~*a5+P}{%J3Y+M{F5QeQxzfb@g$Pl^&dy+2VYJ3&Ra$qv}hTBykA2$>Rr
g+Xb>P*n<QG1U+bb0dD7`Xk?mg7BgE5D@k3o%V)L*c$|T=xSyK*O<w-+)D-Cp0m4RsfBk0#<U{(W!ua4FZ3s4%g&ml`HLSDhtgfsyADhA_jOxkE&p$CcssDy
&2H%w<^R|>!=lU30ur+H?-lYNTQ1%A)i`pmzbV*FdSH{Z7kd0C{k9EEW*K0X-Ku`YiJG)a-FOFs6vq%R#fUdW+`zmXK~~@nL;-E*WrqT%Btak?SX4vJy?M~r
p)Ef9;8#@&l@uP5fM4pD)Rj<ba-tlOjSsrlzYd#fPFidtb`8(kgEHaZVx9byMd5`<JI>ZkliN?--uX9{pvWIIy;jj~-%Pql+YEHUJ;S}~cxly^d`&9ES-7Ol
5rdWr_6Jx)brytq*^}Es3p?p=aA8KZF4I)pkCVVlI$J$XuP|GDmd)kj7hYUnSW<Z2rC@c(tR!XBfatrpGt2$@tq2VyF><YzRjm8>8Y=kdH30hs<2n$X?5o~u
c)0a6DRTRSAaM}OY#B^TwR5iza(Y{nnXg5xm{?Q)x)EZ8NSPz*l4O!`2u{eVSrbkK7Wl;%<8m;_qyS7}qzRoyxQG-8WE1~t*;E#>)0>D)90gRQ-^xQY)Dk|T
<Q%ga0oC(Kebzxq*BAE>Nkfm$p3K|>@yqrsHxqdEJoluJ2^38?VZuyYT&T^sC6phcHfP0Wdk#u(xtt+yejb_C6uX_r=tppVCAnN<KuNE1I4xV7cysB)Z|-Zh
bP`&pzD-vw6?nGvMXtX+2KE^i!W&+}BXb9oaBitif~i1&ma{3SV2Gqw*|xh|krdAAh5cUuo})rgR}SWw9j?PHn@y1Q2CC_wd)>tm(OInTC=~wY{ih$QFw(JH
T|u()TQe`<yGg-(>*xy>GG&MJ^RQN0Q+!ps?M`MQ#U#_#bCJ_ABd{Nr$^^?Wm&bnn!?hjzjz4QB${>;2-{=uwuS*3N3$k8)f_fx_eY|`KFI3iU+56Frvu7%K
)sQy|JMSe%@{o0HUFMR3FGcIS;MVFsm@B2q+|oo*8_z4~nBX~=CkdP4x>8Ne!`B=NYRp<GJ#wXTT8TFG@7WL+jQOa-K{TQ^DHjWv7qvQJOvLQIyW_r;DMs@<
UtCNf5H8)W01xMzoW`aTbI!|t*`zHC;{V>U69SF}Y|tHnILgY0JvhhgPN{a0q?MDEMhOu8s_f!2=~lPm2i*P0y(j0-7t+eTrLDn`fxF`?oM}o11;Sws!|F=J
^D#Br>zW`d$F*k<9iL>0qON;Jpg;(4!Ifh6DgvffRimc77NC(L^wx<ms&wd*!MJkL`vAhrBh#ma`jOnWoP-ZVn%d272+^$;MFGe3N<vgdLxt0Z+V4zYv*RuX
s%UO$=8ZW}M6vkg{F38mW+YUIDsr>!lVUsZ1#}XX0VGqssAu|_5`YsOwc3?@S%Hr8m;#+5^Zg7E-@-r*&4HlY*lX$HgngaxT3<mZmLaC%p_j?c&Mz0u10y<2
TRaL}7b=1gaN3j2&0EVBq?=A&Y7?E&X54SQVCC9Q8L~WAu)0}L%yi6^OL0y;&X#Bnk<Bbm;*^il>xG84uO~PozGy|jg~g=d8ss&bep(x{4meETDLXbkq{lMk
2?6On)V4CMGPIhT&4!zJo_=99@01d@yzNg<5EaMb=Uj$){ft~8J!k$=Ci`aizTP5b)*x@-AX6WIz>?t%cbxB>I`CjP1?~y)px+M|Wf50Z;W632KDGM$ZkUr<
{PQZJjAr2=FKJA|hvf8j><rPFf&0M(Ck#M12sz4c->b|Kc`6*28iJna;Mi9?{~QlmUzWQEN4|Kfqn#jvZMVW_iizg&0hC^MP@u8;?L^^DoS1NLgI<gkCQ_6G
>wL`szxlx0tTxc4gpd~xhOL+{2fILMT0kQ=(iNp6_C4A5Z6=%~Lr+oK*z1$qo;2F-4OL1(IX~#jHg#VDPOJ|pn|b!lx;$k-&$+`vZOweCAzBxkXabb@)pYCK
Y`($Kn}Cd>XA+-*KCS*Zx_*)9aZOKxmDdZ9ZXTWy=!gaLFXcs6zVQ2dO3YXGs(6L#T@)lU%-^0Y3Dzuz*U6x1BZ-GVZU@e%UNJ}fG6M2GWn(9$lChUiV>i{L
4kUeZ`=E*sFohS0dtf9Ue8M-=o!{r(iQ&iq@bpCox1E7W-dT}yD!^h0VT^c3+$2n`o2IdgmXp*tDwg}<FJPvi%);<NvN?B`+Q7JH4`0zIb>nE9`tkbP5!)?2
ppPAE>V@!L>^H<?K!Oy%0nf66U3co*mQunqj6KsZ{YyMK8eGcJl8!Z_rdEuoPp@dON*roozV@7Tbx`1&G4ulRy(GS-U^GNaawL|*4$oF5nnhos4i513mSOZx
W%I2)z7ilqSqC97h<_j7sSfl5-usg4eVpIAtUQ3Or{!n}#8LueC5HphM{f9OiakTp07wU3k_0}?g9A}!&TP7*vek}j!h*#@Wi~hPvkjQ_$B5vtb3B|#sYG$?
1dZrbB?pz{wmKGZB4)rfUaqoxC91<9UpnggknGjtEo}xm8$+fkyx0GYs|=k@vD4pw@RHYL1kf7>3i{wo_r41F_0f@Rr+Fdm4t}<QK&YCZ;qHcEHK*F;H`XbP
F34xDMmE1Bk)`0s`YXz<$Q@Af<ovwZBVcXz8b4UJBFt^@_GHT#_z1Twa=Yu~(%C+!YG2wbUvL7Jj$LGZx$yib5s@oJ*y&#G-7~B}4kOw(m*-MzOpO#2;J-Kb
HLLQPFK!SRs<c(UPVZ0c2Xe#%xOI0EySGc!`Aee+=(Wz?nPWw@go#k}LiIMB{JY*4PT^#ZIYoS1ztmfDfBYg{UBgv=i=kns;>%VH5oCShw>5m<R)UCtj%}Gn
W%kHs9QUr9F3o_j#o6Wm1Z0B=D{Xfx^eA2`>uw7y4VTwdzq0EkH_@@aU0sA~KJ0CSQN%;t_+u{0vt{{q2OF0D<96PyMcVH(Pw~kab(NN5-~+Z&pHRbmZFRle
bF=PV8TCDtA5wF6XB7)bHgtBd=#{VTh(TsvIMjmDLj!?W9@9Y&Pez@Fj?S(O^=;xMWQ&@4&>oYJggw12A?{@ASY~#OgN)trU(tnC8z?5?k$bn8)KpgrvxnZ>
8s`;B$yz(EL|MsXwYe@O;f71tMNhizOYO9Bt-WDDE|$SM0|dV7M*j@5G8nvdRH;81DmmFp9HXSby<I!W7*NW=gL9K=(io)x_r=5E?~{FN(6q-udlmK~EV+i@
5J?Z`;;XA3bQ6l8r;r1c8pM(GcA}UhQj7NycWB2c-ts?dO#jOEnkf|9Z8)Gc-X6viWXCG79^qFui4}^^a88m3*V%?RX{Qy~cc-6ppox#^f)Rx@^L;;d6n!-D
X5u?G?xv-u0&Sj}thaj=Ww4LxVk6d$&Kjc`NF9Z!JMZ>fT%ZG$^~xn4tlZLPe`Ui0L27G&p=b*>O>Kc7fQq$9@Ou{&KMlz0*j^}nm9c*=pPh5d-z2lg0e&SY
07Y*WEYPyA=$BGlt%31M5wl>StWG~LnR=eQ^KWg#7CI?YR3$ykOv!V}ky|V+pL0iQu;;HY+__{uc)_d38F$AD7IWRoCyIv%IgFX$;&F8(XmRD5?@Y^>4i*r6
BLvFQ!3q#ZhU<bP(YKT!(|a54JL@Ueb)Qa&cTJI>rj5PaRa3dAJ7m`v3p_kRK3iUDu5euzg5MOG$dwAgk3-akv)W-cYIK>49>c;KWdV#ht>~CpbKOkULm$NM
2ip>)`hXge&EnMb4t_{H<XXS$z>fP>CRMC6{Ik?m>ufWvAd+vv^yXKr(_d(8fEh~LOl(40Lzp>Pum?0eO^Z0UhuE8JAzCR!7hEt?k?6)c=BQlI$L0kt3q;X~
*{{%)o72f9r5c;i=l8_1hzbHDfsOB8|I!HvFeE_qEnWgjlGSNBt3@O?Erw{~Co)$ShtHzH1RyVkPK)aWY_D^n9Z!akDeO`{jh&6}TI$u>t(mZ(L68_n#UFqV
iiz~aMC7P3dvV?Jni=@vq`x4@=eW+XGAqgE6SykPUUuGge<I;N&4hKFG<q4-*_XtoeJKQLQ?^RPOM=>Fygnr@0cMvO`|<{Z$w$la=#Zla9QCLqisj=CA=%>n
9z^9m2%a_`Q(C6$!dV+hMXJ7~OSjm2iS8-ZAgj!4@UCqh&Z%SkxiVYQQq{o93NPx*2e&VVrks}MfBL~UJtc<`zH>egz93)_Q1q$YZ>QX$-aNY!B)<>l*SoGT
3TSs;6p$qw#|r}&I0ZjF{C}jsWn{2Yhq9vO=tL0IZEBH0m?_7dPUOJ2IM|2Y$?vV{`tLA`TOLw_to_{6Xr^w-XMjq5YufUnF!HnTZ6t!VTuAqVxp%x6$iMy@
E;`EHKT0f2W>uzfzt?N_Z<bUz3y4&wrU32gWQt%x5KE?lIrQX;n*I0U_SwQ6l<F)0f+WUR%E*AIF{Bvl_SZ#x^UAZOl<y6C#wtS^veQnm<Nqt@P4F{llE4v0
pVGZDVolbTliwbgh@{V3d>yA-aNNBvMn%$Imq|Uc*oDI7QRO}in{nXT)nw391=sm$eUG=Xik|yQ4)Fl^=2P8mJ?ED`r7OxU3%<DW2#J%I78o$Q^362AsBZNW
3<~BF)`azksNtO27=PZ@YcM;a*kjHvNsGAYM~4KvchoJzwj~p?M-vimMx;YN&N;yuL<$16XYrWbO!K;=^R4hmldD76{UsKt0YD*m(U5bZ&Br%1<wV}U@&{ID
Xc;&8q%Gb5sZ7a=3*KhOgX58RZqsreNMH9mVWywu9n*GZY~UhkAtC%kR$}I@;Ccs2f>x+QxtiZ4aK~sdN);z-t!6|hpyP&aLzsU{G!N1ic;tQ#i}Ykw!;%p4
-xuGM%g5*Du}uQKo`w=G-Rh#3)dQ$@DQ~X`-DI4f1<XN#GET(nZw9Chn*l4~a*dg%=tfa{X}0uS70?#>!mLH?J+0mDb?_u0o!M5AJ>G5Z^>zQ|*+Jr@*$eV@
SE);1$HgLG_Pu1pEkN*gTZga%3YNi!KvpK$ku|;)=)3tC`~}8^?9V^1PCT!X4DR9okK2w{O@_r$Yxcj+xOSO!>j9D!qQ8NtL^L@T$*x4`PD`FGPjSIv`u-y8
Bh-!d%j<pmEsm9e?Y)Ya9dKa)MPSivsy3xzZm+MY)n*Ie&Y$5XP1O5eF3+;LBV5mk9LuPw!er>YI*oq#!tpJr1gWxCQtL>@F(E-LYR^jvo}m65jsx(`2~?dm
!cZG-qx`jdn$TkTs>e@ay*;DXKBMLp@Gjpkv~2OO^tDrp`y$8g^BLF|LRNm?hwjwhRA<RP@o|V#aRP5lQ#!a$e5*%s5pS~jPAD0aC$}<fOd_~NLI&K!)h)6*
)bG(eR@qLp<4w*v1@XRX>8}hZ_<<vZ=p<%15YAhT0@Xgz=-m6b*rn@)x3W7;(UZV%H6r#A@xD5@L}vMtlZ|G7c}dwDx>(~H#8*7u__#^Ej}qDmWmYxkW&z+L
D=0>o5~z?f_HavxGE+RW=^-8=Np4ve5Lu<?V7PmEm^emAkCU{uwdK1#=IY}N{myber5|kwG&N@&NW{5?$?9FoJ2GJV8Lxy%IQnoZ(K`5PWxRGiXe3UoAow%3
cQI|Rac+D$EMrSj1R!j|EQGH0EeE{DWgOLHSTCkmoQ6(kRN}3aN<aO1Df0oHRFKiVv0PB`rlKD%Im#DtCnC+qBzC+VojO;ki)C4)@7hHcok1^bLjfsCEdUlF
whOr5%@!J*@#%zcW!vrQ@)5=f@RGU<y{vDP*Ak#r$b<`MD=Q9<5avQ3SnQJ^j|MDsy=r7H(Zrw3hzgNCE6Wo2sqNt*G@BZ&Dzk;BY2Uzolq1Ad%04&Kjpc`v
kx%)K5Cd<lN|r8-s)`m>gc7Uhw2a~9!c!Ep>JxvsN@0Tr-DhwBmhriH@);c=+$bnaCsW#SShD!nYZZ8v>pU6Za&1pVYL92o5))o6E8-Be3VS-`S2saT=llDT
I_Cbmz(3tF@W2K6h`q~1QB$xkGEPM#*B4|Y=v%Em)Sf?aXM;7|c!{MB+O6X%LAlgKQ>UvgZ71T%TxY@mq6*QgeJM5JGUhReB1CrKAjoc$_0%cP>1=3=$rv0T
7D*4465DBzks8nX<Vwmzt&&d_SE2f=WgteIUE3u^?W;^y)Dvl9x{>RKQ|;yXM$P4GNm-Tq?{-f1GgqG}K1Rjs5@Pa_E?D5X743N4*B9N4N9WJhNR`zzU%8|q
)m4Tz#5j;zUr5mlq)KP#8l;_Q&!<SE&pv+ea0L$z@juCERin#{$WJ0AE3QV)K|p>c7)tX{;t7A!*}59C2w8k-J7tHnFKC>-r<(_uW!eSWkq)}J2YG`o`l%m5
3cm!mAY(=-;atvb4x}cOd_!FhW!I+e<Ar1VfYUm{P}b^$`I3m$FZ0ZXLf>vrtHI-=_dA7*yyh7mR&2o%0)kPxeETHOxlMyTh#RKt@X~_!8J^3TdBCIPSgE&N
<q~4ej04ac@6y~q7}?~B0Bl-mu*&dm>gA`*cU9Xk>e^zx9I#<-!Zk1%m<l_aInlo}!4bkVsfNiJ0+E6rP7?^d^!o9UU1*y+gCoDjNk!WHh28d!n=ezBW8M~i
yJO7WZ?>cc5oiH7nN_jDy@FUC#=@iZVUiW#AEX;Wr0mZuPm^Gn#IoTZ;*cFbxW^u-q&nVe-iDqjpUZz5JI}-uZ-B3}f)O}8jStAw)UIw~i1Xs03#dn76z*jM
ATuPF!=Nfil-{Ug!6Kf|rrH`}wwB?`^8I?dfnUW*(!4*UBMnbH>1m-q%ZQgIsiUVOhngB-^Gor3_X{&`0D&Viliug47%xUQ#*=)FcdG*X?q^i4;h6oShvIrC
R_fM&>zJv)z-Eww5fJzq!YI>NwwmQ#d&&kEs;>^44m`KH8y3&`Ss4xo%X-g^I^j39^Bl?+XS?@uFN@yleK0$?r^y%;5LD7o5^RS+@TH$fT5D^N>6<Zs!FE{5
Ize2<baFKk{*=>D2@Q36x8PV&Td?8f-NjLqM=t5EJ4wC^HJoY&Dq?|3tx-DF%5PRSPe0_tVz&<#xgni_<O>uIEc3fyVn9ENpL=1?;uH0XqH_s%k}2xi678A@
8FrgY!WZ-sEWUQ&t%XDkVaL@=NH%4~#pS~li{F(KGerGM2hl{rLtx`ZH2DU?8b27k7tD*bVOiKt(JQt(yPJ+9dl)rt@Vt`{zvm~nWR1Wljw_0mgsXs%j$P=o
%=o{7`&P;udQuM+v4_-&*K1Hn4W*McRce3~wp4N9luvRk%kDi?KutOVS}PMQB?=BUIK(3l^^Wg3+sf{a1J++MtwylQQf)X<ZNwoREt_(03pvPu!{#;0EYz)j
+eJWZ;;dxZ2H=ufYw9!RK8VDJ_`~x*8Kk(U>-9&^o<hS1tP$e1aPz$*+g4ntWEdD=iyR~<TvRlym=_Y%)-@eaJyzKWz_5a!3Ew`DAA)9E0)3iFKE?Y!h1-;}
fz<VhV7-&{!q*%!i!8&jNr?PZwOBe#*3-5MIY-M{urc2#P%p&`dUE%IR3U)*8^YCcr?}*SJJU@f$hoEz5k6vEloL8KOe7O^3+0pQ>q>J^t3};4W12l8pa<C+
09|2yx$U|kyymvQr)sna?e0bYslvGnN%m#ChnM6t7+beCmn#up6km$ai;GvG`VPD~)XE~G*2UWKhi4TgK5fXe<e$hW0!rkxqI~rMhJR);vR^%Wl270*zt<NL
;H?KgNX*dOY$1k&<sI^F3Y(uW>;rRc!uI9JU5u!}`m&r=f(Hj%*8!W~5LyjrV*En8W5oo(F;<FF8H(q8NfSnr3zf~(&Tjn)_1vwlM4L%$|C)xVC41SVB9XuA
yF@<WPq`1pE;eLo9OH+$V`H;G_@mjvmt^FurG<w8leDwpGwjMK3s;B>@L~rx7ALKy_x^B?7HNlTi+wR7CPdXnwk>7vw*~{A@X_N#dQ*M0h!+3-Z**kbRB!iP
nMjnlzEFb|%a7s=RE2qgB-n-Rqqi&_*e24nVN&KsIo%Q;T$xET6eE<u(oGxoJVJtN64%%d<|&!S>RtT8mEQy7SUnYptAJR2OM4n(cN20gz!mwpZfQ`!W-gBT
gGOT31#Fqt^F@<Gg?aGz7@I~P5U+SO9fnF8JTp>{DkX<PX&D|5-y=`fNCZz?SC`Jc0}Dw-kf5I|3Ct*|!Q(E3x}G>dic@Zg#`Hb`ZC*)%4I?PLOdOPu=GCU!
_emKu1vzg7RX`4$?m!;%q-m9@{GT29^D53gm1)-zfbZ?QigC4hDBMTx)#*Yc&DSJ+l_XUv-a|@>8m`s{PR$R&GO#oc=C#V+ApwW<I2NhQi6J(5AGj&l>y0^z
j;`?Mo&l#<xe^>zjI2d5l>_(;0r38RvM!CNRoIeR3f?M-u00?bu;XpW5c@NVJ_TaSBYAMXTOlZx*hkXs{#5-<cnAfw<r*ep_Zf~?Z&&f%>9u_rn&BeCi}2X~
kqvtwbPjA{k=|Mjy*n^z^P|(nYCJcn@;(XD+2M<~^}KL4E_*C%qQyp6!EcnjhZV)4PyUWmOQ<Z;km5vlW^2@1=qN2ef#$^3&v!*VXZ!ZnkeXALRG+~Ym=rFM
ByNAW$W$z@9uYv;?jP=&a%dL1+)3(UAalpWWBo~qoJ8jW5)YCZWl``+#8Plr(PQ{xQq1nOd};r_jG|QF%jBx5gYvED`VgFA#;~6*Xw)+sc`c1S38_*hobnFx
i5*(C%ha@ljWT?`B8~rK+^II|E)`0W^zYix`)ZAw?8DVtXOc(o%)7Dtb=YSQ`v5A)60K$o3ep!dhEwinv<x<zpOdaEX20kPS)+@_3pH%$Hx^h*K%N*H0XrtT
BcbEE=Nzcg0%YvD9a~LY(WQ>+JU&~;(0tnCfVH*-AXY$;u=IyE@lVxlnchLMJ)GPO!FAqy@)#!ZWDWJv;{m7;Te8X82|ASAtpdblw8K5V;VFq8&(*w++pzyN
Q=HA~v}%-tCQ&EwgYu>o&(mll0h{LhwG_vsp71DQ6;JV<{))8a`>UtdW$sLJyQf^750EG_tXmRI;jlH8%G}rJaQZqx981r|Jym$NiLjGMz4?(i@8irmaJ6f{
X#4A1!~EbaKNUl#6E}w39(Zv-;=CH?20e)%77R*!6$hH*dk^=&k|4sIk`$mJzv;vxDMRM`dLEes`SRIbHK5_m@y5vMQBb77liFOU{upTsv4ai5&1bL@%k?*-
36}~wm3a1}-TD9-<VGXRP$X~^>^!<%3=8H}kPIE7H{jG1=%A&Z`+L|-QwAZRCi_YFEIgJATBH0lCby|HoGBs`8wNp9-+~8JP@F>xnD1alm<NPkUYSp1s&)x|
9dEvr>X*^Fbg`M0*~Vn9AwaW=$q>F~7>m~3b8Fi+u>mKM^(~j)1fxQvSdg{<6?J+<c;yZd##p4Erf<kH4pTqOGe{U6WvO8Yv{{%GEu-K9(|d&&@?_fQaq(L@
^1(2HDqPpVZF2&){@j32uIa5|xN`EWAi7*JD~GP6Lf7l|oeSp{+^Py!t5utN!{)6iB|(>cdz)V~yp6_5p4`u7naT)CkZKDnNf5BDVRmY9qVOwgYCRnO_KPk3
-5>6;sSk$Xg`am;uCx~Y5br(~#Y~mV&WgAZ`2;T(&-hTW$v_6bhsEcDpDsd#7UILPX|TZc9X4cAvyEI%bcjs$Jtw$09B{y#5+~9p3PH9LKENq<_MlKQPhQ1?
%x4vIF#a!t->)g<=)*F_oRtIl_te<VHobb%6raQ0hPGG9`_rW}6hW?;c|<U@zY&nzy%#P>8$tq)S_wm$bhOH4Hr|R!!>0wNqLi+e)+7#7vq@zT8p4eDCDOJ6
FpI#=Ao-4<=1>D3lfJl<V8jZq@e>bJP<4N#k}xQR<@i^W?w#Um>Wzk;&p7(EDTS3=;GhyFSeYk=KMu0#-VpFFzLOi(M)aLWK*`S34&nT7TC{KPLu-xEs*|z6
2OAU+;=?6)%0<W}iHFUKA&ZVyLFxsE9ikg&4YT%Pw`?hX(C^9+<7zs3a4%U*PyE;~G_xvxwzQ!qSW;L?QK($k)hyb57Gb$>04#WXVb3W#<~>Unyav7}7<h}j
Ii%F~0PuAmMW_G4&m1O?7A`)jmSo3TG%H*u)(t<nPMwh$FK~+^kF4GfLhbn|Dj|S1D}-+Od`U1i)j{sQi#i;}C4CP~=dn_Aql7&qOm)C6ICh5RGcSv!ZNOp_
DYeGNf<O_co{?i@RW_wmI+$H%e}#iNJXEhbVgC`Wzu;g@@`e|2N7;wVcn0K#`j?sxiasMEPd+Vg;x)Hu=_w#nL^Q1^5aZZmg!vePsR(g{DFC-)Ur>W$ck3P!
iR`UL^C8wdtw_j04F*WRB{&fQU;X@zt|gMeXh~}&(RkXtM}-GhqdkAZ6qjM3URGhJ?xc633~6Kdz>V&P0kVOMI0Ri}7$eUgI;eTHm{>o7k-m~>7dz=A-eTE>
cPQu9oC?Stcf(Ibex^>gKG6HONPNEh62mKOrt1lSd+EzT*u2Vy@UpEMn`Vlg(8N+OZzJQy3YPzr?RPIlc*^E>*sBEM796^$PaiX0D~^?4_SC96i|gT#oS(>a
-n=SQ8Bk`*WtEWdQrFmNN~y3zyR-Bm5D;K8i_oM*Fam~5UwEzix~DCXTLuZxn3r-1oziWE(Un@V<!)Dk4xc=}P-55VDw6Kp<Dk>^rR&fSA1!rwK(AEbNKCp)
lQ4i;yYYHslVrn<i-fPlhv3dbw1NT>a4n0G*6(>F?NKqu1xxE9>H0rFdB<mv20TEtgyz0t4wnP}@w+z{0xCI^&f;_wvR;*xVqJWej`$BVEukaAmlxzBq%>5m
?q|F09LA1OvSeA=ugpTirr#h?fk$-A!Zy$9d~E_D5x%3LCr)d(=hopKT+XPUFM>Dhx$uRixd!)F+fKakB!J|nQezk1!&>zrUf^N0rhqgD#EK&Wi)nrTWm9?H
>;`u81w(|89GNN){Tzy=)|@2sMTwl5dC@2`FIirUwk)Ei<-fSP#_7gCi=u^m+5E%6P@cvhK`faqTk-6mJhPw-{Az-s<XED8OE~AX{9bv>h#eioM3**ES)cp&
u{geT@Gqbf%1dPMioqV~|MAA~?9R2OWFvM!D$`uX%Lp7G=B(5D!Ab8m0JkldKvmExl^+TUiWVFtnlAS1ST}vMc1vADL#0UQ^Yq+)v9!<rjj_WK{k`3(tF8$P
;9@Uo)jHDupMVav5CP_%p;}ZLkmVMR&x%#>?NEF27_?(J?$zk)RvbW>4iLA&G|Rfc^?~|B^4CcLTZHHnV6@B}5iqYcPtz<%@#DLtb_TPdodq@uL(rad5Yka1
<+R=EY2D|&mYS1Qf1C2uFKu>g4(mS+5rYko@1r{x8`|5_p=ESMvH;|hk~}D&L$v3*>L54Zu4a%!G3G>YebXcG{p<sZER)Cp;sCCZZj(}PklVJl#VOE`AWA{?
WfGDB)rT(Imvr2IFsC$qWFZ<&!9G^r(7g46dtHuxQlk+#!;$ja#G6U!-+!zV#o_e#DVkZBBAh{njcs1h=Puct!zH$YT4?P6Y*TuAM*GUNB&5OavfH2G*}SHS
FdCNturQmN&AbE;Gp-~@Ias&d`OQm%m=i7FW9CE)VPi39x^zW!rCxja(S*i$^pVGHDsY}AJGwNWWg=95u~oqjY95BppuSC!2!SHSd(ZT9ayxDg0Zi)KD5Ufo
8?j;l=2uh3iFca%W0laPR$-$b=(rj`|FIokqwJ+sD6McTyxUa6T3mgT^E#|+rQjwWR4^o`+5c0@0~KtGk3TX6T9mrqX-nI$s?Azok<(Jh*El;@MUT0_iAgL-
bAxxQF(U?s!?(3dm0sr+A>Z>1DyGE|MdBfO=AB-okm&_c+gwjpi&(eOwkNTO$1W{3E{bm4wlE`Z@(CBu>s~MfM*AcD<8-$!=ph=iN?FVFJ8Es7=^zdy@5lUK
M0dG8d+aWB9ZWd@k4>)d3V8;wKF<mSLG`hoG73ztD(o2uKjHxYV7obPKLu3yDOywck&L&i({E6&mPIz#@b@r|IfeR##E!hQ3y#lD5nhz_@fj?D*A6<hoSujr
Krs@53$BNDUPQo3seti+sU|!EY+Zh#mH&KnogJIoP9dASbPug|TuIFK9KV9{X2WLaRlyj{ulnmC@3aKNv_cGAV)k)e1{Ffp!{SqIvy~UljKIZJ*Sz6K*M@C@
H(Ww*fm&Had44i$W?K<(*t~a+W88;>x)%o09eYDSCKFXWijU({1Eb^6Du3MWA%Gu1tJArM^O&I}Zq7+aC5}GuDmp02?AP{=(+8qrP8uOMY_GHtT3z#m*DMiw
2gU9|7_zZ~!yVzYzMt@IHcLzE|67fYsOVJ)Clev}fpzI0%-J*d!@$2gdBw>SS)$%wqj;l@zbIE2*CEVRYf>gj8ka$4+)1kLzRUyqM~6m%T6~3P7mI&odFb%x
MB+3SD&mIsM<!_yNESl+xa<fhJznqMlmu41IRw92&@)JK)0uC)mcoOf$BaoLg!}Z2uIwZ<j-fzAW~;nVz*y?M7lKPwXb^b%0-_rg70NSqQC9*eTW=zrQijK2
tL0bo5Lhnh9_D~BLv)LN6g55O{PHfr^>wpIdb@i*Y~LB2HShKN1TJh-y@FaWzrKB#rR>jZ@)w~G`h4)Y=KN=Ss81*<H4zV1k;swbgBFc`dl)&eJ&^td7~rgS
gRv_m9^#9@56%Om+Nf^Qa<|8k#q%}}&aQax<iC?09!<&h3Fs~t=6D&!RHCs!9hajL9QBdiPX{v+KRyJF;lfxu6~yM7WRcT8<dH<2BXapL;P<>J?xvZ>6Lfe3
yCFm9423NX)5eyqVCxVeE|>%4y5-~<cuO}>9*!sqHR^F<fO(xP0zrg4nDY&{_+Y*>3~cpKQQ2wp_>r5rxLqkEV>>NxAkM8gZ}IX#t!;wia$QTx@^j=q-MV(f
*Zd;QJCgUp_+3)r`}JGXgNci?p7drqOx8&FtBP7l>ZB@Q?_rVjppKH2yEU@I;&MgyNG%g(a!#&U$?-TH=lC~?JI1aP^K{#leu|Taiouuxse?Y)86n+|#1gj>
dW|<LF<!?(awgzn@khJ|bzm^)!8LugQynDNY|vM+yVYZ2t=Lf)W`WfDViKVmuqXDYq`Sy>E%4<&19(0~crsXTu4CQ<!<{Dk<xC|kK%_ATg+X9ww*;IwxEF2f
_`xR#AA3#`=F$dwIf9d?gr&%qmvg#%!N*h^pE?n}r3cg>OBoIM^pj|P)|crM@X$ne<+^UDw37!T^<kjJ6~{ZEqIP4%Og<ykuYEHfsWF4*H#GRRzy+d9rD+xT
Xn&Cqc0MwmB-hZMky{bh6Bi4XJss!>T~e+X?8W8d1%0*(Boc52+)B;;^k~aKLeTstikZiu8fDn7q8Wt^0kh|yUh%%{+f>o<TwkA&z>3HJv53y`N<^-3VIu!9
ZCZ_7-g8i%gpb^()cI_({vl^e;6T@jY1^!=?emL1NlZK&Oj(!yqJTsP<L(dd)ixU=U-T*^=1Ye~FwlJ?ol`KT)5FKGV#Vx1Wns%n^)>&3m9p|olw6{5XK5M2
AEMPxey7hP<dBMreWxT(52rT{KMs|125!w#!%nQ2n2flOgc(JOGY%i5imD<o`fD7rFttU4D~*-uw+_Jd_d6MeH`mA2rpw^SES%H};d<R-4fRpM{ygMfgJH}y
|Gnd{Cr?bJo(dy3Oe@O>B1t#{fqL!XEwv>JTFYA2%2vsj96baA8$n*9Fv<*1i*d)dyvYn65?1)7ty*oEeS5SWZ#gk8%BBO-q(A;hk*P@DH~`mY%fR<Jw)}`)
*wH8#J}$}_8OuT_o=-*5i?$K=Fv%M}xJx=-I`<H6se8*M)V_gX%i1F)X}dH(I`gJlHN8q4GbvW*hN55hZJYO&QIqyIWsw^iIuI~ecz1`!dt#LkEL=BPgX%rG
4wx!n8X$vz@%@Icih^paR<T&;=tpi`-^p~4Mh2)S4@`=8t8CBd#^ZjzXXqCBN8M;LUouGrI-cSQqhy5r+6u~TrxDJ{;0XcVM<5mQCDY5p39o8{vu?#$q!i$>
VKODd6*#L3Hob*}Rh*t!Sa#%9mW5b*y*A%N=VZ+=x%j7VXPj!iPsNl*<9E>84Q)oU#w^r7@i`M~C*{7X<YQ14x`cr_C$|;armnQhP2i=ZukvPzexTpfbm{Ej
rSELdO0%%yATBSW709aldFay7B*3~d)EUu+HN^zvH~#oL1~Qh@#cx=&Ap0A-q#p3odMT$o2p~yvsQ8HAa{f|+NqAB_H*)HKYdh5C89&XE<2C@>)HEV?vxs%;
QTpM#J*P$8u}^5M6UPG#`5tbZz~E6Z6U|-J&eL;LmxaYT(+wid+JCqHR;zy1Q4CvLbCV;lqz6i)>%Bam_jVCa3$xGj0`?8)!N_LWh^-G@L&48C(I#sS(}ss)
dgaHIWk<4USdFILRL^h8qeBLbM0Y5k0ID8^m+$2+8Fhq0@HB|e-MTPmIbe_s<k@GEV=Bt*A`xW2vO=W&(%M#bg8{{i2imTX(0rZWae~dHTGl0q`1SVC8Z~+Y
n>a{!<|rGq;v1YfTuRB<)zB{Lu|;5?Nr;|^(#6HtC4D@}Zc9}BVO>`}eph;3&aK`!%bo?N&n)<OOXTt*fl>*z+@3i0O3|zJD;aB9j<gYE1H7r;OZJ-f-53^>
ynzfYy68l7sEDGAQ;u&LIhJNC%Wb4?N=RYsFB}4~lL<1Bj-+rXVQJB_PXKmy!YG4Ujol&UfgD3PQdTZRk47w3Q+ff~9eA?&s2j+cM9UZ}95X1lZw_P8KfRXc
_f^(FUIX+s?DS&)SfG`H&MC}vxvj#6+G=afwXkItL5Iw+lpxp4Bw*SbjF~x6E-_0889H*hAu<Hh9!B!M2|Mt%2Fat2u_-wD*kGbDAlxdnFTqYWtk^hgiTtUw
x>#0DzT`Vt-1QO<8n0+`^Y(*5$~di#z5DkPC@(v1qjn9$LkVw`g*WMrgG4f~puljsy{sF|zvfJpKeWx32Ex+U7uwFeX*d4hxXzs+(5MOP*Qg^E)bF<_ri2e;
%#kgDy9{05IHCBuRecz^VazkN@H;8bP6hi>kB^Qhwzb?71d_Q^)Y@PVU2?vZwICgJ-8jd7%Dv#$9h%9Pso>m6L$9mELAM&L<I?Z>Tb2KRUYlVF0qw7Po^4qJ
hTU5D0)|2(Vk+G?jayVj&4BK^Fs$~wDp63T_0Y32G7A2DLcsYky_>jsrc%SmTwv9|q^5=4L7v`LV0@jRBypaB=k+K>I4l-3$oAEpSt;QEp4KN?*@)mPHhFcY
$a#0<-gF6RkNapfb?v@1-#)c_2}mkUsH2YoZu>;V%;xJgfXCrM^33E}Ee2r7_+FFTdx4!I-aj0m(a>heMLfI<G@VsQGt=FwET%>G&?Hges)*JQ1H$cfk?Wr>
;Df~wQ;Wi3Gyfe^ga;`)j?NU`XCLDjby+)J=WSYI$GP!tEs0iin@PY}95gIm4vxjRj7_~i#%gk0=okA|PD5w>V^>;(wi8D%BrHEP>Po2(2%-FF2KMZ=Rph~E
Yl$C`Eyjp|KA#m(Rl*v4b{V0QO>b>qNy$$m{~oi?8B)oRS}`F}m$W3f)f91Br0HJzok&YIwMu_iEu@+kSxJ2**omk*VZJycEi^JflyfYxdz3;^Zc5o9llS-~
l%?$=^;26)g550_chs<e;NG#@W`6npQ&;70Kc2TlSbp>=3OIT)*tg>=Q^aGZpV{0u;+^+ito$Fz_?BNll9-nv5|4R}xe0uuEeE0Orps?}mQbR0se+ZPd*IK6
#LUIs>3b!bS!2EfE3WTo@?psl*{rHD=zZal<dA()GOXRSgLf9ts#(SDIyjJ!jfXKScrF?bfBqPgOk^udgS2?X3F7vBa*G}u6z_gXfuajBwH5zfSKNZ#TL9;{
)=nSFvn?-}O?m)R4@d=+RwA>-0_>sZobUL{kU-?G8n|$m2V~bLY~&LIIBx7o;Le;9`$0w&P~(^q6jjEHjsS)$JIVLZ$V|XDEbodml;wQrwa9D=6RPiAxPMB%
_wnb2YF23(M5e`Lo+bcsSrEZ8`C3Ax2maM4sRTo0vf<czNgfLFm>1yiK9$(d2w#F8NqdTeQab(UAW53&hZFxB2aWJxsKg$WIc@O7)hw35<b9S0O_yqC(7v6F
wQ$D{GGD-|B?K$SvsRq|fUp=zyE?8fcxtXsL=+{ICw+L0T!F!<eJ;8s0Yhf7r>)Xc&ECo}<7Mrw;!i^1c557JKA&j+HfwB4ix9NedBLc-zb!K*&qvqvfq!7U
2t_lgb2t}pjsc;Q!8~Vb+FPyNDS^@5QpB6DO-rHo=$^^YsDcssSYQBhS@RckqRk}oN3JD?D#-jS%M$jrYZ?5zbxs^d8ne6P41o#dlj6hs(+O$sNdx%bSCur1
jG;3f6!HC+yE0uR0nh2z=bI)e5VT2w?i5^trzmmjzVH<iOg2ybh|ISUt7%U3%h9WE2JX7Bt6wb$ey!qK(j*lv5ODA_y37_3W8<G>&Oxui6-KfI{z1jTk!Gt#
p2jw$_VYM{00#%ePMv-5T_ZmPJWOtp<~^+UkIrC^ag{s=H3oiS`k8BLLn8`vWqDlqZbsH?6gETF$D!S1I6-ABesi;kcZM9DUEU_c=k5nc4>HUnw)a!qo*P{+
rMM8{Lx6bQqjUZnyqto{1Ag?^+*t6NL<r|BKwFoCx#zrCgFx>VKW)ybF*mtbDzQI_61|E9PqBB5k7K@_0+uErV$J32F?@EVzT}LNO`w?fVWvuJ^#5NF8Z-c|
|E$_W7k<2$j;2)nNl`8O;<i_$kjzG>_}=cCjypEKuN<uWb*4}xPK|46qGWrd(k;Ym@7CC83T&2pGd-flmDn~9AES%)v3*K0p)k|6t#8F2`oAp8_my)D4C61n
w&w0aZbTaWq!*8n@Pp;tS2K`l7-???Pyhg!c7ZTJ^3X^>K`K1m%hq8G49epmdhPM)J>h=s6d?`6o#QhpwnKb!v|U8L4ni>(tMA{bc}yE>zu!m^D~$IZLKn!D
8C|%5^#46E$qB(RkrWb@2?XH>IU4q8V^5(sk(DltHN$?<ZH@p~>?(6o^Y_u_G*GGPqjA0&G!wOFxWi`k4G8?3fGE6Qz|)Y(zXu=s*HkIg$O;if87w_bc+IvM
+W`BLpQ>cA*H9)hC3{x7SQ+(Gfi;mdY7gY^n@<ZLBK7xw;&D32Tg4(q1l5olPjN|v1<OnEu)8UDvPob*g(5~da5kV?(<p>`9B)Vdf|yo5F~CEFaFoccwCgQd
gldV9(PcO_wD?W(rk$div<O!yhd`a>Cxqw4aQ*2UHS7W#%KSKe{#{`ltu;w=vZD;u7knYrLF__KzcG@w!Hj>5xc27tXSA8p>V^Rakf~u5=m$U^4eCS2#o(X@
`Zn+YBtT}b1SDF!NO$1l+P_9N>Q@Lam+E8e3;%G+FOeCB95H4)<_GkpSY##ss(D@{*XCNGVP%aj9fRSh2B!@Py`LtecL2U`0)4@>-)@zlIykO9HIg_z4zG{n
PPh;~{>Hl!z!T>hE7#OU+jU|vL9-xv{66}Fz)5|ky|0?HJeVt0xlH*rknt{sm7O{HTG8S==794P6y3V(a5hjwWzPI)mmwF*FA43i06*VxE&8(RmwO@YQW^qi
i#ZEMWy#p9C`if%Vl^+0AeyR^q)yyew$7PzDo=2ygJuTJWviPZ%RC9qQE@Y1-Tl9yD6;V&75E&}HnTJmts2TUS@Y_%-?I$_k$}&RD7C|~HTy=rK4asC*zg~F
TjtC`3f$LWK=}k)Q<z7p;!QOJzSL)@cMOCqZ3smeJu1T62A&4Zw11^{8Eo2Qa77wI0B9iOVuBDL&N{UEC`h?Lvv25U)Fh$V9tjHTE4P{%S0?=du98I{UzE~B
I-$NyrYS<sY9uEwY>;%6VLL6^STgRUsLKx{v;iAgUpWl>Ayi@mo?!@xGbk+?isCW*QKfI|)4^1&A|@uQKhJ*aPK0^dZNfyQ>#vle`uWlo<6j<q#KCVSReDs4
xT@u$S>!@|_YY&Jr31RVtw*ftTGbuX>7ZO25k{GNK+A{~A|7EyEgD@Q56JRJf)H&pl|4~DoF<bu%M(VFmH;^OMkr9};%^UhnGv#b)hFQJ#3QDRmh65TTutt@
9mU3{6GgeSk}+$}NA!`<6tj?wHx@^Pt2q#v90j>Z?hd<AK($rCB_FcmBdCY-AS|~$x7UgO-^=_Id7pOH5j}Rs_l|=N>`phs`^>h(K&|e4kGCHlbtj<7sp$;P
8?ZNZRTk587gqTorqcfy>f@qkTa?%|10aXIKf0_F*^)FNdc#I_WOQM?er;skA{HtOCpQars{k$CPD4X<FJ05BL^tqzeN8=V&54rP*|exFGzv`5FC|<o_H;+N
{Edi34L!oM3&`F?w1LwB9k2e>MB`ST25h6bQ)BpYfywm#{?(C+n!&8ZIiw3SLZ9I337A_)2E0zFmu=^-XZMM5IReY?P|1f>jXu0SA;-xGP#>F94*G&f4QxWD
f#pQRolhtuZM~BGmTb}7s=ZsK!DJ`H{#)v@28ilL@z?h}%VG$W!5Hy9OTc8j_7A@(ebjb5L9SZb+NhShO>IL9WF61fX=ND_#Z#6-vH6(Jb4%?HTDZ-}zRM;C
lYD%jHi-~-#~zpXeIe7Fer#3m%O#7nSXDRsn%nu(yoAhWWJ}+O8U%Uy;4=21rj_%1X0pw3=iHy$$ju)JNF7P8fJ%#Gr~DRSBz%OFY|~jo%m60UOybH{O$0Dw
6?zyqlu_3&DeqFwBn#a2V$1>IO!&1fB*?Bci%ZQ83_|_Rc{RpBpimGya^@N~*P?@`L`Vx1-t38v=6oOP!eA{z>0y#1gK+p6RlIcKar>%*N&^Lpf8Tc5e?vL`
8wEs}BY$={t(n#n`qBxGp!(H8Xo>W<@LFPoF4FlrpSIQNcvzH=<`%J#?Nu<PDxtN|)Ef*EobuVMG~-o)utLN+(q(Mu%#8xVD#Ohbx>@vTm&RC_xNZHWmIX{3
?F6muRJ$DzF1cDT=bopH?FNW}hT%2?hAjA`2G&f}+=3&Rfx`otCoI_;s&74KJ?;xP-K=A?-Xra)HEI(7g&PsY{hco+#~%JB6k&&dp@5hEYsuiIUtSt1t*nIq
BB(y&Y68je*WdX({&4y6L~V{h=;~CebOHfK6HjdaRbY5GYX9nQc1>g+sSL;{;HB@qN7N1cgyA<bEeTSL&cif-&kfKu<ar2*{Gd?(rbMCes0Y3@fJ`V}=k_hV
h%IimS<PRR-HhTo;lmaUf-*(d-yX`Pl$x7IMUkdGP`H!J7W>~yKR?+kVl>?f$GxWSmFjV4RgCyfcuDat?xz+siF!AE@A6UMoQ+764YLX$R233<-8$Kxi`WsK
>RS2&XRF7UP}cfH9k~;h`c-*GMlLU-zwG6-_Du|D=rJ_Owf*6K6*{&169;pen8dUFtmqN1c}YVpq;nW=)Z9x&N||h=`fxxf$7cNc`J|A`&nKwp3WCMPsbv7t
o2%Z~29hqYHU*Kq$tA8W2i%=!(`O72WV>h_gJw;UQ$@T5`8y3a1GGn$Ar^m<fwHTgB4_~j>(BE(+OBT}DNWi+J!+n64-C;lMAw*TySsejDun~u5aqm1RP4%W
tj4277ir+ikmRTih1d66u$fNJ7{Uw$tBW<|AS~=JvoCTo8E~&2qqG)IP75gve#hDG1H;@<Hfy6+2n8apm~wO7q_u8I8Qc9Z=B8j1WrqA;djC%buhD*Iq7W-@
IIn58Xy)Np)B6aDDSsCS>E&c1NQI%&!rG@nXve!qBTd2DwzSgOO_l*4o8Z>ePj-HnC)x=eL%6X0!iK)uEtSIbz>K2ff)a#>FoKTsYup6@_%I`fVI*gbMxNBd
3G9`-!!nRTOzhxQYD7m14!->dZ*uY;FPVexLy55Z9)6{Ui<glg4q?^aWU^9l31UTQ`yt5?bJF*&%uH^m5cpX@`LNLMa*W}9E@@G07(snC0i6Q`q;Num>6V}x
f!j9QTD^YU9Wb$PFO00>Tr|K?X_#;`|5!v};qQ84jAl=klE>u&<OpdT4$~*&5xj`$Xq_vy+ns(6aHo#MF?M88On9rUqw~r#kX=_I9ej0-@5IvyH4W3_JPW^W
%8ZTJBPY0Vc|_wlwJA$q(bHNVi2cE5BVq8?lnh8PEv*rD3A?Jiy)9oO&Ua{#A>zap@Ex9J%9<ZlFFlJt%ZLA~c|R7QPY@uH-vo-X4egurfu9Ni*ONg-?ObH(
UC&rZ_^r|(gRo7Zk@LJuCxZDpzQ9W|#nPlul=^nF+|zZFQy3k8XtWZ|+$d%m;2H18LSyK(yRr+i`?b4`Sy10T70-OW{0jjM{xdh~Wqf4fT0-4J_>CSpATaH#
#vcAt@=xafFSYzHj4m|wnULq}US+6KcvpV4Pb)fhYmEiFph|!z#MveorlJ_=^Un31i&&_qula{nMOztOXZblnSU7;H<g2Q8yYG46HDBpoR+)6oNQN8!TqGl#
6}e!tSIGfL8Zk9KpD-&ce|n}Z8u=z>EA@&E*<HR@7wk;KFSfnz|3(H<^(&(`SW}R0h_FIC2Zi!R!Y5DxrRc-X#-ap&V3vl{a;?eyo^a~26d9seZiwBZ^;7NF
FD8&|I|9beWDTUknU3i);zEvYwq8VTl$%&jiJj+T09l0ANTs}Z`|<o-!AjvlUZ>XzC|&W)zsj_GTOSz}K1pjr5_1u+63x9+rOZA3n{|HD88<25rvu%96#9%(
@R=qww@WicN?Cbp=fWWO>a_A?hTYLxnDHX69MG^fFqLab4H)>rYpdJU4F$0t!%ni_t!~{tb=U6pT0s2j{C`ryJbsOIdbZ(^RPl$^J|M@p)IF?-AjHq7@)`v>
Q?}&B58}obiK^zba=GfEn>-CVBT|$3LPAMQT2Rn|MOy82Z4_@e!}LnkxP(|}hq%P0>|{CSam7xdIgvJD?gNq2vY+rWQqj+<N2qlvWio31tL3CfV1biutz1eA
Ecv*!HNri4(e8_C|6K~*Jt#Pv0(gy6zmL@f^qIE|Ug)oH+=Dj)O>Z`!PNUp?7ak{#;!dFtjAnsPzxb!TY5xm)33jF9^NENw24RgH5bKqeS4L|x;SQ--N=N(B
9U0+$M*M%M-ZE_>hAY=cYISR;Hun-5BDoH`u#a8J8s>=uWE|DLnUfd_2j20cPM$cAIlIUN0I0#vHd|w<shD#hZqIZYE821)E99KIt}o(S1imY1&-~q5D-Kg`
?Mj!SVa%M^`ytqh8)N+=KR~5s_zSsvbfsLe=CFqv=)^Z0s?L!{p;4@a6Z_0_?CoA<f4Vt8TFC`fQG<yXbC_^{p3VTcQvhN#RV*zID_fB%5BrOk{9P@qVl}v4
;=fazz`g;)R`neDgQczGrfmp$2Yi4A<7Q9lr6l|v<EZaESQZkulq_wRz&bG+$pEvfsuC6s75h0(cJ-76>Ir?QA2wv`^z-$q17`~a@FW==?Ah(Zv$0yg^54b5
e1NH|cbxEE)%ul{0=}bJ0QHzZ(LzoRp-kJhKaC2VP&O87y{zUN`_VSLQF2D~^N2tBW{IvfRP?EO!xMb8`ch_Hc9DP}=IK+~BQ($ixvY$y@43aLON4mIU+sCa
bgk-0C4@k|MZD)X`l8JZb_&ZXJyaT{yQe7jtl%+E5>V>W77jSmse<E`L<VYbQe^pW!SROst&F7!b&WMLCEDiubZnp)w6W0Ld;+XTE7E{yrdi+1DuPCJ9<ReO
3y&y_zF<mlQYUpW`v)}Ne(VC9<OJNH1ncp08*)}s%Ek?6D}#2a^ykI&fMI}oT~!KqzjX$4bR_Npa+D@unZPDSFt>;pA|CIIJIrKp3@239_dRkz+*;at9>;%(
{BRbgzAT!J*J<3fKmym?3W_|<6*FDQsH+%|wIZ%3R)yEd>%^3#z=VuMsX$vJDl4nP3$VIMcjVHbiDH}X7^nO7DnLryn?7k?_A&AM)LayMPvtSq3>0ZhaP(o}
VHB@m)Cw{XU%SilSZpikQCOgGg>9*8eMCPmYka)3(JTrxtfT&!;jOw;Xg4w)prN?L#_8rYK`ayiBN1NbG7|}^0m9_ED=e6w+P)l7I>)!KK=lrndc1E_y|KQF
&7GOcVdvKF?aPIj5(5`QPOBk_TU+-*T($$ur@H`bP||}jlO(6*1NaO^fO+2+#iE9!sih#&nen6P0D$h!=un9hM&eC%KC1)DVt^3xNRZa)`#+;b>VhdQvZ>)+
f`94Di|;+(TdF`jA{7OtD-j1uj$r>y^|of>Vi?UzTo;72DBP3-Q|{fo!4$U5WF4U_1Zh+HPff|9M4|#cc+EkBj2I6)`Jge_Zmaa!8@fc5R7NhzF?ocAYBS4?
gds5;sQEU~xRIFpy1#f4%ScDHV_-K%&HfN_c7-GuD0V{ujyWLLpC=G*u|4RUh-6`ZA4&DG1E-0k3tbH@s#_^UQdK#da~%CWY#ft03L5RvcWI(A{7_>%&J%#2
bWh>fyCr1=zRzcsG@U<w%he6+_Th-|BV>Ktd}@+2^-{8TTno`d$!(PE<Ua_>0auyucu6RVNN~4G1m0#F|I?q7p>z|xR`QC*1f_kSad4xxnLd1qKn(|e>m&_J
Pv|i(&|%e<>BkI6WvgQzS&E<HpDZ;_$r;OmebH9!cXWP;T0|Jx%iLv#Gu)5#pOTgSLh00r)H1Sbl}gL@>Y}aeouAjj`@5%{zTq~E$>eu~*RqvS?j}DFUxzi0
;a?&wFt#_&eX~sJ1_$OC6_!Lz2NK!fT9(eA>%xcMF3DOtT+#Qr{rq$Zv$A9ez#9W{R6l?iV80>?j8quU+L9*0uE~zjn<o&u%>T54nfZjOdO#N%xP6OlcoaCo
O<5qp?@KMb(k<`;!}GzkWotEM_TN$~GXk_fk{<1zVfLBW^%eJ6=M18K*=c=IUnXIy^RA6AI5$kM#WPz=H!#HK!n_TC&Mp9pP^{_U*=?2jU5e#Vl6|*z(=#cG
j-q`F^gC3{-n)<>sVCRmM4GOF$1!qIG-y%5)GSJ$!oK&|+m^Pk;tICla3_#DMUf<=OvCC*yokwmC%>?eT0uN*kQh#FX&qM9rI4{ldfQn7?SL2iF3&|K4IAU@
?hLqQoB1vU$B6Wb(nV1vxX~(2sp{p^#Rqe0K6;C&&DR@#?r*<R&Kz?P#dC5i9niiqA8v;hV8$7OSGz$^HOnuK*Rt%tq#)(rn1^a!-tm+OQ|Ue|T&muyR6!ZW
$MlNsHkN?m-YI`t$$^)Bq+*7cqco@)Pd2wIs|7~>oJH17jOo5M@dF}qV^P_gg;^?eg?Or9bozHNKF56$>SAtVa6Wd!?F~+su3OEuy}e5I&Y=I0j1hv}vJ|#^
&B=TDc<$-6CjxFxX;#B@`OQ5k?2z~4rff6nxj1Gac<KsGNRY-X&W{iFuyfgG5X$Uqmxao(_y}~yoCSZlto8qy21Ecm>Y)|jtvW&wM9{;&G$sF6%Z}T#WKP>~
Qv@7J#?pK~1l_&zMx+ox+5yrW=y(?*#%QmqcT9Iep?a9?k(MX<(tFzj0VX`Tv<ZNt8y*4<hNHae1@7S5>r+o~!mfz}@Z)Tzve>dE(i=dHl_aRh<1`<&V4B*X
zoU5fH)*}y3{jI^QGHNuetVdQ8eF^Z6@dZw>rbp|(C(s@KG)hmt|OHWD_<xaY4w7yrq`o<#TZ6k&vuw86i&<U(g9<h$s@AD{ocfueQy8kgqUGSHttC=2a}tw
M|IJMjrg^K|DAsv{nU7QMk{Mj2eeIfZU(+qOss+nB;q@hlHuc>s~EdB%VVuc7TNAsLirmEcWp3LD<t#~IaQl)>pJ?Xx0w(gRXu7seWP$X=jl9w+efhA;Pa<u
(OGirAGKWwSl-WZT9EP?209;=mugl+3isXgattJ@Toz7p!NqhSyv=*6R%)v18q5t8pvqp<n@6wjwqPZDvokMX@Fg?GF)I8}Z>WN}KRMh|Ab3@P!<;g$Za+G#
hb->R)#4`K!p;zbpYxj=_*j5>T>3FsCheM#@77_6HTB^9$0;NTih}<hh;}vhQ7nqYT1BZHlvMFrAml-Ik^N=XL(I)iERUf`H>IVuf&lqYC@WcfR{_kcX;h>w
1+}@x@3bokt->z)vjB4;OXUtWFcdD4l@H@nm3+k_eMJbl=71*>3YSnD=Xro)Xi!&d=QasnzY_u%KuJZ?$3W_cj)&hr>jPCUCA~k@!E}_2a?FLNH0t50;brFZ
H$Ka5Lp8oOxcnubM{iJSCss_!v-=>FUqLVNPg%0Ho~<f;SdL?)CBVuw4=lyV1C@DT@2TP&Le()YiBmdS+n6KIROKCH9ct-K1+#cN^9#`3^MJ&cvH`%8=alNy
DgiX_lPedwThgQKYBk=+ySK2x$z3QCsqWZF-)cQ2=|wcEzFYoY)erXK^LxNJWjI|A=nB|9F5apJ)F*y{g?;GtO(ROXUSg=S?Z6pv&8^OI*TME(u~u*ahgFO&
3cF^2oPPFVH+y;suQOn`27SXP<%g=jI|8Py2dT5&!(5M4@Ct0qczBw)7HFj1!CoZSf3{o6cvOy&c8?{AsK4A)C7~&MgGMW=)|+tITFa7F>hM-Z+pf36iEkF~
OF@NeN|JHSAQ#A|ZPSo$tL9+PQGsN}CR`qOhY5)Yia<ML*HK`2D3wz)JCmV6DpvOGGVKq&>^_fG=Atzsa=&HiWHio3HS##GziR782?pMaRhvii1%j<~JgYd|
q+6N*s_tP{^_LWp#%qYl_Ke6V78|(U4@EX7S|dA2G+>B7+cOl(z3YnVWsxlZZ=Dg7D^X19m!=Q-=qn?fGe7{_Ov^Pb85RpBM|Ux9ex>K0TLC2*Oaow>)>IO(
fP27bi39vvS9CKvZQp_|@E|(6FO0=KKsO>&g@B5M(H&Xp^J)rZVD7H=6V<xy?;A9ulWx`}knX39rO?_>firFa=?g{&zQty7FVoC-Y<UN{71%#{fLH<zkQ_9z
N-2e5bHD_1ZHRKh-zC6|g`%a)H%<~#=+2(Qx7#(g*mpG!oTsYwJ}5}9G?+6yjo$k9&bCvA<y+A9Ab#%XY4{xyRaNWw<A@)9`${l~Zv4d7udu4I*WEtLs&>ZS
!3jqZbxvqW5}C}u4^oEtbF2y%G51YczdfX_FQJJhb-@o{&G!IGBC8F3{3Qv!L$LV&OW(T@JCnrC;t{F*WNflZ;)AIrR26N3qIThh-3vsJIy>SfH3@&i+Mo$-
Ht43#l7KtY$@+NVTwvIq=I_Sy1he4FfA(iJp8&3Efr2;4ouz(N@;ehS@T&UT)tb`S-rLti?7xf|wj^T{2qr|*_Al@@s{O%ZF=M%>i&dD}G9Y=_8_0b%W(@|L
LTg2$JFUrratCU6<OIR_>On`0b{p0*N1`!{`v`AW<W^~Y96LyPTdJJC6e-F|5-MdqE1msnD7B^B4$tuwWQj`xL$-=WI6}|{OMCZN_INAdfFR$*D^i}Gr>>I#
LmJ<JX?R6J+wS?suYz^raIggk1`~~58H&mgQQ>M1nIHR8&AP>Z`aBM{u(c)RI_F{R>CO|wgLXvAlOh?$MAqJ|vcSQng<shGp<iI%Jfx0hW0erpV-HkD5ENL@
1Wl5*@eoM)0nc+R_}XOjWKTd5wyX7IU<1;DG7|#29rnlkEt{<Z3W^7bLZr`<r<P7b$(74YCSvJQ9;z~UoUQRJY?9_m(1IHMA4gO}-%!58(XkTrFcc?tO=fM!
C%&gyz0C2Ylt~sKv~%Epzx-bxuP9lECd3GqB#;|ep3Y9+sEIcSbk^$<b~fZkBl(zL2KT9iqFP$eO4%#&*;Ge6ZIu#c87+;;>7hJ{3c|I*ZdI;!b}Ft)7w8j?
$EZFH`Vs3gQo2Ge2rQ~`Ht61qMo(4>(UP7CDA79#8e`;WehC^7;^OqP$cb($@4C73!F&032EgdFNO_7`u~)>xl|WE6#BuO4IVf~_?A#kJk=Y-kwv9<ihVlSk
zCYU1yhB{ntrZ>Cpr>rE12%^GV8l8T5mk<rb!W84r{>94U1>3tNO4iw)3<GDFgFe$IcYZ`hkTAaSE;<=dK(%Qpg{;T<VoVf*U1x$+NigjB5yQT4*3(ijnb4(
-qcs!1&u4x_S{dC@$PG5+mi<eQ5m~#2urj*LW7}RBclp89Z^*Y4;h3U1dJgfX)Buz1DwxXB$f{pmr!I_MDJ();X@eytt~4ld)<s)U|J`}h=Xo82DxV$(yAt>
%e}`J!ARaurH7Fyyp3U^u00S(Z-(~Kh5f1*ARx02-7UI7vJNKLgfsvP_k08UK8n@ZYv%h;8ImOe$tb)Mw93c<;3}M1O4<4am!=+U`ia(I=!w-A=M1nk?v1?e
gnY65!7n3t!CdbuJ_4mNr1jOvXPUAETU`NZG;cib&8vbNXW@Q{OCiy`VWDJ@Pa*3Jp$s~|&{r*2_NPVHlzS_I5>PZ#E|t6}7G$E-(Z7Vth#XdG_rhlz`711H
y3=Hrq7l)3RtH3XIdw=x!xSe{DgZ7>!O|(QCVY~XNIMBS4fOHdO*8PAe>C#sK+E<OV|!+-QB~;vPoXsgGep9Va04(|v*+3&i}*%0X|h~U)v4IOgu&n~yYEHa
eoCN_t4eI9qBh8YkJL;<vFnm2p%lbq2uT9&Es`OWH<)2uMjoY!D)*3l0otxLOs1UG6k~PL)9j>SZAJWT_}wRpdD-UmS-v5mLSv~)99kPiK6c7w$t_zWznNm`
@ka<2Pn=Ik8yVuKkE!3aMddU<E25zmAet32tg{dVtx-oa=U_@ybwR(SMg}x$sZQ=&xG5{;7{C~w9`JT!V%r!*wI$@5R={>@ytdAD>?zDwQi3G2AlGx`9>1u&
Kjk)IOSNllL4{j3O-QbDxgutOuYjS|0!5>X&1V|LwwuHk+iG`-xSM6|3z-L#f~LySVpu7rQ=^K_m8IuQ9~MB_H0+W9VG-Dl9(Zu{Cq%nhtDu0o1eebiEoY!~
AqP5y_yB0*v9{;Xm>GSPGXI#5hbgAj71LQWs07XamB~P~N^*oH9&hekq66ZDG$SdbhTeT_QT93Ljo~)@<w9H^vv&6BJO0FKFIXJ62U;zW>bB0Y)EizYPQd=R
X^ef`N<3n^K$Id}OykO$Ktc4DYt7-nYFqcG)6kCZyD)ad1cggo)4z{mZYmW(FdKpZg(i3Ym9Qa;CTsxd91m^(Xrjm0FxXw_Mti#IL@7L!1i@ox%(dAuZz~Sc
7AV)V83?f7X1%&}$L~jEfRr`#a-9O29M5|Nj!uMxTWs!$gmGq7ha`MfvPrP>x;04yXi(gTy|7eu3HJ+v{lAX0FZ>vk79mwbNJ252{sN{l9)_b_t~<qq#X-^A
cGrwLB;tj9Okh)~JV7){rIG5^2Cs@!lou(xvB`kSdq~@SRW`Yd(P&Ogw*kg6w>B1UtuPzzinK>-o=vGq;n)`UM>>eYp?2fRqIAJpmjx+}0s<M4n8_tMe!=a`
aU4wEgJr&`kGI%+XplWM_}L6Y#Qa}!ezr^c<4W<V8P>*}fq7A}1m0-<u8lb$Je}&>YI*DV8^N3b;TR4u$un$2QPML;v<VHyqX>_vHN7~XfBF@j`A()96Df3>
;Nr0kwC(#*>#$W0a{fGTFx4djL!5mo`F9w_aS27I1A?8m>*KF8RDR-uqepxnNjBf00#|4Bdx0r#bD|>QPb9{M#2#JO`fE1j+5r=}2Xhd>?={HB9mx?y&;9GL
XJ>+<{;O{F@%c<lN%x?&LixsMf)Z5(l<?%||CdJViaW<IQ}oynJJ5%`-+X*xZ*J-%9CA>=a?W82V09EeC*AONb5*9*YqC$aw*uG{We*Q8lgZ-m5NG-EkfP9z
G45aas-Ou0g?9~Q*tuw}C>J$%h*dP@>L?U6#n2*%g?O-@42zqM-x~w*v_+>IQ}CLy355~9G7`p6UyjWgmgb!dd&UnJ=di&HM+PCB`(|R9@Ks=-cG$6~XPyss
F~D!vZ<0hV)!KfW)FC6H_7+lo(O3*f99+?GytsS?60Nq>?_!<!obgPOcmvlf<d0hTVZ>TZ<ZB;V*V*k4P$IY|G+_5Z==J}oQ0JZ7O)|!(H~cqh<Eqsa_Gh8t
sK!nAWg3_QwaS#t?hs772YR$!`m1T^$C9gkmWx*1v_IBN4(sZOEu|<_sAb7FoHWXeX~_Bu0})rZlcD|ih-G?&4Xe8#h?3|_aq-Wdp=Mo?{WGlK#k0d$f!r;S
e*7}rY0dyFMTH0XD-+uq3-X|GdQ>)^C6fOe^weg-%Dv1P5Ch6)pgjB0{D0YhREr8g5%z$EGWaQ%{ijC1elhv29So97uGg%=-$<!`fS<EFY5w(g&03DozU_Ml
+BbCdKAhTf^7Z)iAi3?Z=hcS&=U>|=r)2U%+rNo7VRK3$TmpJ?>5r3Ua<bSNAtwAC=GPI?3%gAm76o&S8Q)iJ^ZaqkvlM+T{$7DjHx&-iU~`-*(|f_Bf&)Xq
{Fn8wvy-_rAmY<*{7=7{!#5CL1P)j6&JO^(IU}Mo#Q<Lr|JAr6#v3w|F`3LU?2^uKtY-fiAr1g8K4NoKlna`X+}S{){kq?mRF2o>iCmlIl6WCn9=f;ml__Gl
>sV#c8%1<D$O96lFL2iVwkfpt)Vy}yiBGJIlZWW?r>NZJt>;)Z!YsIIl}na#Xf6$ACd(~}oO5>#r-b@%a;llFXSG5$y^Uoh^wtUn?ReCCxnwX1{?bLiB+Tcq
ot3+=__SU;h+Ldf!TO*DNwKr0fZs)mS3QOuUAq1_1W?C&0LeDEvg!3EUPJ7Ih`?LJB~@1m1wEanp6I69<~3xVz_R|_*(VplAp}OX(L=bo)YOmu-Z=iBMse?*
Z~84rdkWV2Jk5Rf<-F77yYN}b&J&?|J5#{-FsO|ScqhoI<LD8`4~SBIzBSGSIx;%}-7*}{Uh#?v3&PN;uRD@Bp$si0vV5_0>aqneGUpg4>0ia^WxWm8C@e7C
<&&s6wv6+{qVG|>ZJ7R%^&tVcm2i^{50q5)ih>}U!e1nt3UnTwJ})CsREr%P;^_Vsl~DT;dVl*0lAc1yv4+hKpib_Dnn_=0-#?|d)!he$E_y<maZCI#fw<8#
)PHGRNoq`WRBZnQ6wFW#xW$fX(q8v}ve|_hV8uWQuCxgAGm?L?lXc)I*8Wu6jCO@@^%KAfy$XU2`sa!^EF*(tyId~LmY0*7Hi*+_zZ!7;!yRFv-zl3e-Q{(%
M<{VPZk#%+uD#c0)9&+Y+mRTcD*-F_=LKx}Ka?6zgZ~Sy+3oK`Z+<wlrSD&M<ZVnmHiVX@w}#qnS4$6bj5h;+rbe{WkslH7c_*EsCUGl%UYuSA1`j5qRt#&7
oAt@8i|ddzpO4RV<$2swi=qQol#IPS5DBm&oU~sMEHTpHlIK!`vwi++{LRkvsS}A~Ke1k@WAiBIpq|Q1B&t@@5m9x}nP5=S|0HPyvW~wT&MSL=2>Jm+@V*rx
l2K$L#+KqgW@RtQBQl!2Ty`bv%<5ZoJo=w(ubS>F2%b1H<jANljFj`l$No`gRx3jS*jD|pgVx$(>O$fsQ!-ul^m}O+9d{CRb69G?GQV6*v?A#R*sitwT;&pp
Mt_%N!YC+Q=)vaNJgpC9;|J@;Xa$bSG4@ikZRC{3+X?9^DG^c?%WNbw7-2s8raI3gH;K!k=bG(QExPpHdcM8@0QL=`G<I3bE$)J3=<PqgQ4u_BGWy_`9-!~t
+8Is6k><^UkzXl3as{Y%)$=afSELd~OLIt9Z$4GEBjRE|35z?e@W;6?PD(UZCZyhlu+JzmeFZFC!Z*dB(A~ykbm;lF;nd&XdA?xv6)~umvEjV?E}6W^z`;*e
jHM9by{8MpzKt_jvhH?APlLXNNtqTc2(_1g-F`ZU6K?Sk3a=p6qAYvW-b(P#6=??Wt@B=+o@tnpnR-U29306!Wl%@e!mk#_0lhLHLgIdmok$LKu5T#mtEXQ5
aS2W;rk;=O4A!TF<szu1aBsy9<&RoU^lpZKxfK(Ww5!sjG+!KG@sS_$CU?+|)}w+Jh6tbS;P}@PL148p2Q*(s@MZA``T8mtZfKN0_M~W`<yOr5kb>RSFT&X~
Qfwk|H!mHLsSTfD`{<_DP10)ZjqOtkc%dZ*gyu?)xtHZ69joLY>ad-)Z!}rrZEDC@{=Qjcir_L{wd`n(qfU+ok~JYH#n$doRW9*+;oWkG>sHbR_hz6k*8sN!
X<B!`>pw*2-4!+Fs5^R~9&Gi$EB$|%d}F&vh1xFkeI7n)q!Z#{K{k0$Xphy=7>MOP6Wx~i_hy5A2xT_yygO$!ts$5iFqR{v#QYlMz>%*p)S5k8+N8il5%AZ)
-C4^pQx_D;zB#=k4$|$$s&%u~aBYazV9YWZw(H~0ZwId7^s%|e!srqGF>j49`;6s}ERFW~;zL3T9j}uS^>t)KnTMG=-evKF907L1$q?rGS!Y%NE}ACHtQB>B
rU&}6j4L$lnH{MRHBM$dz6c4JE7#}3A^Z?Uwutr|bjV9EcHTJ@tCokUBtDkRKd%`OgLA8gui@UhcCVhz2+(2zAqi^lulAb4R{jF?<a_?cis+v03$hGK6PN&~
Z;;IovY05&yxvIoCO-NuDl@je4(VKwTBAA;Vnre`n<y9DCxg<PQfrZK@Q%a$-@~6##M`z)_WJewLz?Wu)Lhf^mSd@kZcYDx)G_6R>bk)+dTspO-pPajpY<$?
T&On~Dk2tDWtC;F4|dI}p){e#I9vT)-ffS2$aHX^6DiC?JRd}H`AZ0y_kaQ-6vv448Fa+s63PVyEK216S^&>nGl+7)_bej!FnWMToMV#T*Z%Ja?8R<giMr9M
#a@%*T`%2l1rR{iam~Bm+SS|gqYd7cwILfL&^N0fhT0`)%Xy~n23qau`#bS%LVD57NocptS#zG)T4ffKo-d?(VTYllMkTq&a*@M@Exg%02fH`%+hd*Z7@;E2
Z5L062Eo4dh<yGdv+?6mIBv-r5;yxIKF+hq{ckRe3GH304?1<9xZch%^`K&pg~Ywh5rN$b8EJX(kH!>W%YqCq=CA<^>-^ofYDx$P2rsVSU>=aU^y;d)$>v^x
QP|BwXxDJ9014Xq+9Xj?AL$&=JPhNpAe&1d(Hz}p{ZY$l-R4;Sl&LZdmDYN$j`1Nu?tW*0t*auKy@aVr^@&_V1EkC{Os9oOW_Pd^8aeto23tiP&JpXCaHNAV
Q^TiC%k<$06Oi32CQ-RaDf?%^X~$t{8_1<0UNYpX0!XRe{0a)A<Q}3G+Itfw!_bs4G*Tc4Zb;kn?4VKvFs3G)Bq=P5<zwMtOigrCt*I!GG-CyBWlU|2ic#Dk
0x)WuWKs7MZY<c{r+$qx<sP@r@*W(Zu$O7Qp(~QRftK)6qlF_})e+b2rC;^+<9`qV&p<1GjxVq+FF0(p6=ZVx|7i_f!6QaCE7)-yLlvA&4m$+j-;eOzRE{7b
Q$|ZLTBu8FCWC~^P7mlECg{wpK7g{{+Gldx2#{f@HqK6x#Ad4d<8}1D=0Y-$w*yW;X4G*=w|p5*OmQ<B{xrGXpDD882!W2Wc|PLN3+jMX|I-EuSFRX@;fzvt
+Qcn}BF)tC_O|DTUS47sslb4V#!(+|gCxM8C@{YgO$}n#{(i<xw>tx_jS=+|NC<5d$?5!!oRMC>jB3nYQ$pC&2`@rp&jJyX4q6p<1MX&?^i(F_Mej>9?<4?Q
38>p`iXT!kmXMW<EXHj_Gs<-}M9Vic!o88Lw;fuEM;tq?9*+Z@1G)GW!8#Vwe36f*k}2X9IL3Vo;jv`vfBGGFfnzxBMYqt<O8Fh~UI^^u8x()!lnBokBMu!T
A`84E@%z=6(oEe3@_b@8qhdIH3xz&hs$Rqmg^+I$tWvoEmdMvIb5-$Zk>Cx3rR*N&aEIKSkK4Y6H>HA`%>+CRlErlA?Hn%xp^&b}`vN8RT3p%ur@S^x^FKpF
gNR|j;r<l^E>SX<HSRc&oooGuGZ(B>BPOynD1ySbj}-tBS$l^Dxg$&9bz^t5T}gmU0!;B$;B>;2!f|7bHuc1tWOjb|1d6EFKlm%hS<i91OAYL8=680m;jg_a
I3~-_EU^z-*ZI{r`i&<<lS@!+7U$ViuhT!2)7cc$j%6>a<n^3iPg@+4B;vE1f#Ivb6*>=yPGK{~*BkbgO_lpdTiQk%fxvCz?RQq)1*>VIcdCucc=zu~0b1j>
7|-hol#1T>Z^*#i<9f}Gw=5*l)tM8yEuhZ3ho+!-D|m^49rseCQ+B=4foWnZ(?jB`mm5JE@PM8HF!xFcM;-V7+4jIhsY|BneyQ{kKK?=+okVULX^_+wsaAFJ
lUN5#w$UCv`S7TK)q9v1YWryS8t&^<-#q4EEtwU0YU`(=s5=X?)qAX)P?>XPlAh(yo-O7lF~vJ!?4*94=7rKEKxpWzU2)@Og%cpLuwj1zc{I3xzBKk55mS6m
y8l7wXx%-iFc9m`Jo6fIE}opWA4CdWf#@-G`~k@ZFZ`A4kBIi7E?41uPK7AI^4vtnpT$wkC?^IURVEOUE)sSLNC0n=lYiyB(}O0ZNUSqHex6y-*-}I&L_*<-
g#P!sF4@;z*y9mBlP7-gP1y63LADQ=*S~;}4kn@8as`}QneR6e1|g7}f;QM+*MoL+q6+P-Xh>9V&?}A(8eiWNxX|ue(>Wn(rC<S(0lqTUj`(hgNXXZ5yqaUm
D?`e@5jrSF&V4EvxZal;g2-*>_hVCdIL%{4Y>PjXB1@bfAu+S}59@=`!rp2HM}TRIZ-=dfGnj1L7*~z_pNJ?sDpScJVmi{GAeVab=I6J#q9UTJCy<8=jhx+*
PI^Vq^(kvN_dt#Y?SxzP$RGf!t3L~iLzZ+Ndeei%M$n^Z2Dm)7#a`XX{U)zhvyiYxM4V!>Z$8Wg9g!@5;L!W4J?&;V0Bhx|66QR}joDVDM;P4vMvl5(;%BuN
TC&2(km6c<8cAz^)znJ+^w<DZW^xT(t|UY{A(U@{t-<Z565&t}{<FYgP@O*SVA|MHqRFcgApISTt9+Mx@+JV}aKoi*e@zgjzxBlR2%RT@lCfGEH?Z<^4kBu*
Ic&8jrrn%!m7O!vWR`hZP~3tAlK=~6Us(NmLg?H3qiscDfxTbTJMq<k(_q_;{{Q?uA6~9WHW|ZlomwIvbd%bhS$4)5Z4X(fb}Kzax4~Mr+xB>tl7wHg8uv`e
A{hAB$g>4dPiNnYgLIF-4uUsHT)bmF-}jHk&8@kTYj(hsmZueyamI_;wY*2&Ezhxl-CE~CB#DNR?}xy|!h^ykoIr~HV&5tP30QKzTc<Xb2n%y$YDMX)Zf`LQ
Puc^kgJ6?hv5Z|wAW*F$F`<o^2$TngH~x52@;#7`o66cYwgP}}v6&tT{oK~S$EcD-JXQKhy07rKfz|EBVtiZie|7x|S)!nV=X%!@SFcSNiw1bGOZfL)H1$<;
&$SsYr-@`@PLp7V$2E1lm|2cvKH6d~p{|S{R=m2)MXLje5JZO?$R+qsht-luc~A)CC7H>65nk5u#~{%Cx*YJDbG~u*0~{WAe~Y?x)NIsJr}h#yLQ5K8E21y7
?Fwx@Cnv)rOsDlSD6_iuy&zm*4fR7aw3n_7GMIE&DU~*^itBV~_y~oE2%Xdu@SR#yM~V3VcdTdJ^*Z%c-FLg8!Te(8nzx@LPSbona~(%=n(LOo#XGzB?1kx%
tJO>~Kyt5bW31ku;zKe9fGQ+N9)Ny~SY~tWwh_T}ecZnx&Vb)IWF&jJ=6ufZb#8W$BDnbNF%<%)3z>8)eUj7p9G}QRsf1I4j?RhHlr>_Y7~!uI?AbuF`hFbb
V9N_K?N<4?4J;qT7*cS$Jm`HA*YPO)VP$f3v~j3uB<Dd*Q^OyJ^A?wT{r23_oiKw_2i1>Vz^`~H=ZV5YDtZ1*MQ{5j;#5kT<2V!8b_Qdh<EgP5LV=Srlbk`e
NZ54C)$FXz3xSOWxuaBw|3^X<W~D2o-ACD_$^8c@MloA<MlON!QW0qIKQr~I>Z?b0ra3_NBVQjgEV~m}f8gEgw12-FGm;cfZZMqE8k|Q$G9*9jiVmEPNuNAc
vZb^bZ>8bU^=-5O+eG2M|6M5F3v(ZpwM-H6Ra0k{#s~B|3>nv!Q}3)qk=HGD$GhQY2WPcQTCvOAuy^7x?fFDDaTBRX{DL&{cKS>S#*ktRk@*3tRH4S3W^2&M
Eb+EmUyW{GuHvW5$^U%x9~2Sf>7OX29G*$hAMPcFprrfoSgd{=x`@G(b+rXh4vY-Is21<Vuh*H?rv5LRb7lxnVwoh7Q@SHm3!j0ogn17P41&H8XJ80LR*+0-
Jh?AXL}WlKJ$+%!Kyw$}T51H+stf@hmF6RC3iu85KD<@5iQcp;${o5>Fp>WP&IO1oWOeCJ6{+SpB73f~Gc`F+EWtk|gdXs(B2JUPWztsx1x3SZEngrs?bLaz
y(MOeuSfojzvXA1+TapjWG<I=&EVl^alFTx6HG|*D+?=_P;pvcg6fcohVH}6NINFSKO7=`J7RcmMo3`hJxens$?$uavsYgHMR{-Ow;0i~vT-{5`oj#JE!Kvw
YhJ;^LU>(M1!QCFA7jsm2WISX>%IGB=d_$~=(`ChDi$`YpJErv+$Nh0dxz(;$rbS#QcZV{%D9Fppqozx;hJN{ik7)AqhBT2n}haaYF|C9p)NKP>qe$Ihc^eN
nFUlWY=((xVvLs3r{DXihkc_>=B*GlQAl#f*kb;v)@^1TZ=@R@dJShDU$V$IVW4jOi=XPm8Qnr+dHqfPw(-29_8C;R1WRaD2V%dv<{vNQh#2j@{RD({RkzSs
(wgIwkf+yOJqoSs&=D^1H+)!`@S+zSLV@-IwAZ168agP~J3fM~=wR*mqCEkDaqi|<2h+quL7;#uR_Z+QUOtZe@j_{qQg$~{RfI<Evjx>0C%q-o%FEo6KVO_t
vtD8##KOzhfTA?x<+hOE3AETtKCgj{;DM`=+`~}KNrNM$*;0#HepQqRbp$k0ad%MnLF8QN!7$zjFrKU6AZ_wEWl*6je!t;^%(75V;SP4yS5*H6X_Ft9#jbDn
AoI-R5EPja{0W)Tj(;Yvyn0r@;&P`v(?GfM;C{R>xhet*lWu1z@8O?+Hkn~xhJ2uFQbtJ#=L%u@#|2HAm0Vp*H#YM5c8BJd=p9Rg%PqKC8F92<s)+%4i-_OV
*YgOox};cQkNgJ2cN9#N>Mm^=YEQ>offAg*UBI0SfwiE>a9ChOEx*i-+AHyg1lCedmmdi?>So&=8O>(LOx9!%MMarNLwaym`dMmhVDP3iuf$oEJm=J}Un}_(
C>S7hUu-d15$aCD__0dK=P4O^yRxwT{ts4$TZDM3IW?_y5N%_&w4;6<=Q&eV+VZq{c}f#8K12Nb@!#Xmrno1#?jhXEC{^?2t^tw^u?Z!NVh=R(rP$*G>{hGC
CJeqV9c`UWv&s*@tPg0KVoKkElQ@{@cW31AoG6Ofilvt7JbOp&)H;oBPH*_r4A7lgEfTK7Q@JWyL|!lGmY>mhLYE+Sl3LLXhc0@#vSfYEA2CorO{wYPyouX;
7PzK~RDY%u0#p{vL1DXTkjIh-^UCnW&3<m@d0~AfV5`C>4=-34alFMRUj<o=6$b=<7@#08Y-<N9mQcumBp2PDVq>`@7Nr-v-w17-BFcUhGjkD$k2p1L++OyJ
{W;X^Og73qB_yF}mURM^B94Hmw6mE0XKb8=5o!YOsyO7%-K1KQzcXU+5o)wb_BjJ2GcMQ7q(-#{Z(Ue?y-V)tm&M5DZexKwTWL6<^{99|A{xXmbPKq0A^@H@
g`V=n84OI`$7&D)aJjB+j~s3Eu83Y9*tJZ4tV;F|=@^glY1f5aYYfN;oAW%#WdiMh#|;DrA<0M_yU=m!e2kI&hnTVwyM7>(S(Hvhg->d&pJ=9}4JFdGw$*)?
fHNunPjh{>hzfB76R7pMpTGJ4_J|!)nuisY^8T0mwdvAO2W=6kjrL5mv{6e*!~U#0gS01?8Qx<gl`#s$B==Ceaibq#)cPbY!P-i969XeZcejuIpAH_Vsgdsg
WF$<p6WHv85x4B*pEyccCq;QUy(}M`CwW(RE)1cuQ?z2q6!!{e1e9?<MvOHP{Fg+P`Ui!yCv{iB7N|NgpgJ^!3Xp?lrapdkJL_X40*2AzhW^T#CxGD?TOpqe
hu|l$tO5ML9Gf9jRfxVJJsls~>gfDbIqqY|wnx$s@j|LTEa(^!Z@rEwBwCwr--SM@<0r-4dmb35;e+`ESNHF-!)5#s+;6ITFl=&dE1F^u_=ZeLXlAa5#=WI)
KNJLfZzS1yVZ$)7(e!8ufj$wCFtE07#D6g<rTyLDe0JX#PXOvCW0|#kGvQPndr0H{jU4^jNCPkT7fGFgT-i<jX)vT1Vv2PH9z|*zjIqtj5|6li`#ti&?<yxe
@|P-`aK3x?TWp6}tQ2teWk$l7|L)w#JqCw|e5KCXVW{stMxZM<vi`Mg(y-@IdbD~G&<C6^eN7f&e!`PZV?uW8NHk-0N-H_=h+%DgKs3&oamW;th)9qbLz%qd
?<G=HQF!&xiqDQ9%@km#WZ$sAzM{LC@nH%6);mN>aD>yjzctm_T%05>Y^OvT_sdw{ZNf3Oq8WzEs^h=JYgL{P63!zFH|yJ#g}~k!Ek%Yf|J|59t)ua5B75e!
Vqk*m9>7Soz1v-XC>Z0h-`dl-2k`D*?tRoyJ&Voan2qDAY-xPiWabKTbQfA+Q$g{(vka9MHZA^$_d~NKEcDUMS3ov_Gf(iVMME*KEEc+}PhB;mqdf&YEl^Wa
emj89h(E6ZvZfFtf%1E?dxd2-+?DzVXpVjy;~$cn)%8z@to?d?C=%)#--mcx5VyTX=vsLxh)3bTw96y*unLI>*rckm^h<Z9QAwuY2i}D1GHqe;TY_!WqqeJN
8F$RwvG?Q2oMUdJNdXVLW<Z1d-T*y!>NzGEklUD-SF7~A-Q)JgneZ~D2=&^gbeaSb3B;e0n=wB{C><*3L#pT+6I!JpkrR<tk;fAh1xka9VE&SPV4yK?qt^5i
RqXt?e?-O4aoy51kgR9n;sgA$i9BTUFIIb-)NpX@G_BX)G;rtxogWWumM3CoL3IWpGV)bKbED6ux?PMxG}0!!zsUH13$oSg%l4;@eqzGkqp#@5kSEi<Jk;6L
zi`=%2?RNrj#D<6tMywTd^$hc7)1CI>Q@&u4;yl_9>+I?HO3B#Hq*~4Xe{JO_iL?K?ul>2D3>Gv=^)5(Wg;iG&sZ9>*YVZUZAv5>pr;uL+a)qd6LnyxyL2NC
2|QWC^wrH!WJh!Kh>3$H^akq1H5>RM8>t5st+P!1D<={Jd)fLNjq5Xio0v2}@9P0*SGHZi!!zw!R78nkElZY2z^}#bh>uGjZ;z7WvXRq-j<|XqDaYmZ3I5*M
b`$P<#p0U56q7EU$j9QXP1S6l_jIA0(DA-eL;x8;1M;vajY0J~Q8OJ9<)RBay#5VATRQa6L1ZSgo2|Iv_b?a>>hn+afYw3BCc<qA2;55)YZ>|?R>;1e>|3xC
Na{<f%)pM269P{|x%<6#MX>;#zZlv_6>u5bYY9gTx{u?v;&Tt}2RR_oc*RH`8ZujGPLOQ@_4Tl|i5_dxT->q4D_~KGi7}F&$YvGn(Mb}!EJ_1iF?ADWi<=E5
shzsEY8fgXd_Qs}aWk%=v<+6FfOwmf)3LyGQc>ZaoloLyG}IglO<FU3&~gq(yb1(reqE3co3wDWxtWgEp9{tj-+-6`G@7d++ossOSCAmI{!{(Mt>7dhcfPO6
n$y-2Qxd}96|Tua+0-z`s?YoP$E-M7L_hDkRIN+iUCrJPlYy9fAV=^?ips^sfmKV*x~)yX6cRmH^6pgN())%--H}0RVn|3ne3&Ls;LMJDx=aa)&bo)_wKA<0
zCu8C7wHCYzjY=aA`Wc@OagE&@5J{spRVIW)C_lO^=`%2llOuS6*F%7?^mJbF*SZeghE75N33UoIZJ=~4?h7TN-CC|crY{O^wPv)s#^n9Ss}lhi8COOTz!Rb
TA!8$Kg+>u6x)HDLhm#j9lssu1+$f!dCO&T?-4U1-%R&o<HzvwSyi)KwEWeJp+HpZ9{%Dy@xp9#BE8$DmN2ExNuPL-ikB2qsOulTY2Y3)@|fI+@!u5Lk`yIJ
Q7-;Ww-L9l-b4NvfmE;jmJ3SxV76HM6k7o?97lAwE6UNoI%&mbfTCk<BCLK_Rqg}!gIHyeWd3Xx^d#<MFwQOzBokXTGq+LBp`fu~TJ89<bgTB*n#pz<yfO`U
W7nF=4hIE*I>ICv@Bt7)sAG`vE^C8cOl(duW`)KLUQ)Ini|aR-e|WFoDpZcH6ty~@6u&bR2zJ~WfUr{exm%wIb^hLZUGYt*XZpLvZXCfFN*G7Ju$iD0i&GWm
t=Y!dJv6eSSyUuB{38tfh_3J0Xbug7xoW^YZJd`8=$y8H7Ky`fB6F6K7eh2jZSzV2hdXsa1aRm}E=ozR1^R@ZTUUcB0mot?tvgKXrtZUw*NETZw-MT_EYGSi
vIuhQX$nmQ$0kutmH!x6L^@n{0u|8`b@cO`{Zl$JUkmP$!LO*AbH3Wd0q3a;m(HDHxNtHS99amwK_P9Gl#^1&o7K!_0sUR*S?{ow{V-%#y%|-G%=xWs7s##+
+=|H0cON>l<Q)7A(eT?7U+Ii{$X0h2C!D47E{p??z{}MFMEXBpfoKmW(<-?)$fx^Z3{&;#SXfi1osOkb-VC(inh-}eBF4vOGTd~T3ulhkE5%xmmTjwB0NX#9
E`zp97nfLFElXERiqXmW?H)au=p{Mb7qldVAC&pZO|)9??Ua-KQeyUa!iit~3!@Q`O)}FR_9k;z9pGGrxgd$mp!K6m#-#`CZo9s^1g$kzN>ILr-2~aLHC3--
?Gf`=Jb;<L1w|hkjiSW^wp!2Zp{CyFaP?qcK!sDrwv*eok;2Ud&qlW9g{l}<Co6$nx?u5>tBvaDUB6;AR39Vp1coC(kXvBy<cxXw0F*apOuHNbtfFbTI$B=g
`a(17Et~#K+qqV1PAEHeC7;(7$wXqE1-h_YXRyi1zny7?p*jyk4ioWtBsxkaq6vAAJ?O}FFEHZ5chKvG5*z^)jQaxME5>Dgcd(D@Vkik1wN>nw0YQ;?#dw;Q
%q_9IWh_Xld5l}%;CPx6yt(>0yDx0!a$+==Rds;@j>WepC2&IBAK#T_qvN-;=XpD|-@<5t@fPzoME;90=M%(Y#pSnXek$CuMWm1`^aMU()LT7&kOB1v&YuMZ
s1*hdJSFwFH5uy*eYw=t*QK0SMXf{Vse_EzK^CfhFC0e2fz^4q0{M*qAS^tOU@Ucu*ej3DD@GSb0~4Z*T%jJE(5c{vB808%jmrGHJ;Yg35eVtWpA6CWqkT`Z
wzO2CoM|Q~aEWUZvkk?}D>FDOBF*y`>IyNzI=5GE=DuP_uJtfXe4o(QnS~w(Wajtz{*_9WLU1vP36+Uh9^<doP<P7jg)5@4_R->FgrIjftMmkqE?FrXS%xq2
oiR>7<Q={5FQlQu31q}1TDXy>Gm(<X?W!~AA7dq|<3suAoEw>evf=x}5^M-H6k^1uBd`_E90(13M10qVIAzrpE^E3dW5v9L0geMxzB@7S9GqyE&%Fi2r#DG6
x>=|;Rz5Cd%m3&WjCk95r(iihxulUE8?OLi&?CbI{F`tT)U#3mre{yOA=G;H)T>bh*su2x#OiNn-2Lbo_#+0>jHY_C{nfOF(D4q<kz%wg1adblKKq7D*C+SC
k-jy1+c>@X0i9A~!@k*aZfx-x;KRA1dH&sYz38X_;;7x$ruV1(VI?EJ<`wLcDCoG5@mKFqAfNl>x3Ht+`2v;)DjX4S6G3RKe+feW>mEXa<)hkkX&>PzH$=sA
rOdsf?5_5G@R8yd0+l+_c9Xqs=VSn=g)c1BqUzYKgwI;PDnJO6Sy!`h?xQ3Ifmj^vvE2aiNEYCJZ1DSDFqsZkW$}|pWH$YtdoDn;B*-g(S9PwGsRD4ahqPz~
O4I?(=(&c=p}`e?X&7rj9^vpN?jdJiz&3;=%rc|(Hv@$g^Zz9IiRaTpWF7eRVJw!jgq?%>J9ms>97|ps5XCLo+QX~H!xts(bsrUlrw!@q2UYbZd3d!dVEP5E
5f4o1wc9pbzLDKTsM6Jw;h2A!5gWUkB4DE+Dg_2uZC6)s0{0(V+4l^lA^}`jm^v+!ev0=j6RLnxKgwIn0SRfZ43_|PPSI-NY_!=<_yP~XCc9b4C0xl$Y_h!k
cCo=9vv#97JO;6``(@|6pg6zV$2X&Usn@T2E7<>F44uP(LP7{YRH6>TxQY*GO?v(9nF7HnX$Jp=ow1#>2RvR{{pR`%nKNt9JF7SW0h6N!>O3j%n{+!+=0ZUS
O9NR?b7Fw8nS1Y_uLLq?Lvy7T^jUID8Bt^VSaigiSr19k^oB=lU(jFS^@QIukEQPi<=mF*L5jf-J@Q@mXJqX%n8k^NdkVm5@8*|gaO?@PO&%bV94d~Z3~a84
Avj?O#AAqi&4r=Fyt#?HnbNP~j8F6VFS(E}<851)C5+Fq-)o`{kT!z_NaJUJNX66rJ-Nb^u)It@!mtLk9~f<gM=cO<mDF!9CDXf7PZPL;$`n-bAS+!!DWfYP
%UPCcu<MZxpElfz(i}Ae4nDA7q9gzp@<!GQ`p{f&<ErB`vP0T6AX1<RsL#b`56R|ZaUhlDZcjvM`$&AFD?AvF&3B0QP{xM=Uc0m9FH|YbcZJ%cCO-pSI_qbJ
%Utyq&MVPo2RM^)aCv8`SFnh@0Q!9r7+rshSx(kWYvtZtp@f**3v(f>xbq%Yc)XxMkNz@<`vpy@2Js$~6*n0;Wo*QfI6s_&loW*KGtLK%^CZe)gbsU?`QeFs
HF>vq7anW&#3aneX8OV}UW5|#elpKfnd@li!yAp&GK`d<sm>!<+EtuSPFm(y_!r6$;z7my5&n~!jE%JNHguqo*T~JYcDN=j&UEC7P`2;=T9ePEt&Sb(R$<(S
?2+6+Saw8L-}iUrH~h!M6^aBriY`l3<W;r~r1f#6hfkUbS0<%A`Im|S_RV9chha`P(KPi5n^{HqR8Yd^X{{AHGjwLpJkJaUA*D{|>j<O2tUuF<SnHj0lEch5
53BMIBw#1}v~Oj*d9jKDqAg}_H~I~Q<aPCoL}rl>6>7qF@}^wQR5$>0;&9gbNw;LN{^D}+jN&2X2DKK5{7m`I1^Lh`CR{CAct`n%3l%<Dt?Zq1dUFf8+r9Ix
Ki4fOFkO?xtL%FwdddaP|2K7NIs*w^n-ck&y4r~P0tr*+r~%gwS>j9wq6$}#o7n7|%Hi4KAot#4l=bOuK=fBN+lVT0)%Ar~CxdCmVBYm)H`nxyvEvE<?|f_>
)fFrt`M=>EkSl+ot^$LWpda?bz$IuMRsh!ypa=mWRx(^Wyao+r4bB8j%}9Lw7Z?Z`1D3&!5xs|={1UuAOR_5M4SL53f`&U(>e@yhx>jp_z}i;l0L(KnjgRPv
p_9^e;=1w#k9Od<L6;=11spt4=f{a|FJJ;^GO;25x5^;7HHlSQb?iDE>@^FW*Qk|v>e&zc{>Rs;vPvaY6o|ejunl9aC>X~*c|YA0VqW0yzsxA4e8UH~DqxrJ
D%e&Wj+za(<$v34OXwRe)hC5G_SPh{cTaI+7wuz;Q9IBySWux~KY&T_l8%hC;5+ZGRVB?6GklV0QG$T}_|vi)l&b}0sB;`}f^mWrR2xK(S8E{6`w0_4yPVKM
{c3odhn8%slNxL`$uO{D>TtqS5%#NwIF$|b*(?AjV{E=3Y^1`{j~V)BjmEbE#Flr;<)Hd|S8>9uG}^Vwf31!p4zmaVYiSZ~bahz?<0t%nyTU9K-;&v@1f#u7
!kNVmWtOK$@ffYMid>bhZ!!h{8LZ|(hWIEH6%$DMPy_c(L6I2KoqUthO4oL`X{sFn9#Dj(HJ5FtPhf}cc5Jo;;>W4docH}dY;Vg{uWPz){GJ?hFSrVLE(oVu
4mxgksw7uLbnqJebbgtQyUG!s!wux^ep>453N^FSbs0gVJ!II~DaHBpm)H|lDYvY+T$T2TqF<KN5mi3l<RC6)_rFWd?8!)8G8B}~Q3egGi3YmEn5JDyuq$hp
Vtn(&AP>VSNoYs-XTHJ9mp!BC{BUx3&a`zAcH_zRkEBII5+RdzCqLKRJ|E?J&!Jt;(E_H+`H(_L+;!uaqvSyN0cdFoR;WhS2O+=HI!)$3F)e8|xq`MMhA^Yg
owE#!8*_ip>Q-uJc`_}_C*-JFIqDwBxCy)C7=Q~cHgW>QaCg8ZC=jt7Ccx3BKT7Bces{70{RE62E(EOwh{3bBXs74kJ|J-bRP3Eci!Jcj=Y&u0&c{h<qAUm#
t~o9>{E(@Wt&8nPCuUGtIiES0l}o&db8?>I@mE2x{zGN)(&--h*?p25co43%<6eeq3}38UX&#S268@EaEADy4tqC5ixYBeBrTs@02b<NM#_|eyK6+xEja7zx
cdCxQ>+HI(5@jx*`fXd9NqsJ2;%KaV<2z0WmSEjfdW>1DC%W68AiT#@Xy%sO?N)XE{=r?$WZ`?#pc~(-B|Bp>GpLPYUO$|I(>_WbKXulRoH{{2_pMXG(7;$a
iCjo9Qm^_#b|gW!=vpS04Np?FFDV@MfGO4-NN9PztoC~)?%&5oBKX7MIj|;kL51~ko+5+keSvqeKx6j80oP^<r?BM@;0<C->o1=kNlMT~|8M_uZC}X=9P#2}
-v``O`7(q;fSN0a=haM{B&&X&S@;)Vku2{6t|3xhhF&hg0#~p65PdY!(9^BEY2)UtrSufVt;`yrvJ)XFrl+<fvcwN^b2_p5OPFWf;+i}+^4q?;x770b<~E>_
r*WYw=J44Ywwr;^95S@wIJ4>lj4JIW%}4vU_i@fYa$YlLhq7VefJJc26CO3!{G?U4r>`-llP?FeTAK$P4cgr5;b4xeozZB=d{Xd@9{f*<OH(QEf7;I=W#&no
Pvs1)iP(_a)Vi$+wW((dwVhhRHNFk-Wn}w^K*S)x0&EJ(DClUQT4PQNJ<uptlvt2u;Q|!3l`BTf6_{n*t<J#;rP#6A7u7}!1{*{rc;UXi5*k!}<dh6Qld)C8
NDL2o(Cyo-Tl)daW8e(E6>fw(I@&O5`0aw|4j`@u1nztw$FVb5){+!Xy<vYEu8SPqFGCb$rA3?&V0=+5QSlGoF8txWOC~8vZ2c?mF?ulIdigEm?`F^tS(V#!
QB_F95@({+8zm4D8@u<kp$HO9cocSdQ>JW+7q6B^3cQ>!L&ATns4y_x*^sr}-DNmUWKg;S>=lxEg0-GzEGr3(PIip;=27Xksz>x;y@ILnHEi!t_)4(LI<x=@
LD+N^`;8*Ax(E^qvORR<ZLZ~~3n9DqNQwC~o8I3VL=_7XUkDO!Rb#SRUoHcl9&~UCK`5h|N$rN%cG=Xv+US(2RPGnrmsg#7%D+q5NSRFi?F#TdI^By*WI=c~
nA*=)o$m<5+L@0O^{5!1o;&h~xAc9iBC8LvDi0G@1_~!JA=<0jiL&giEQs_;S64fc>xgW2Hv}<U;bzykQQp9CAW;IQgBh|P#%yvbb>EEaDgbCR?{%c7YH=->
m6V#C1SC}#zeiRr8^}hyXe}Kzn)NZ$dffL9-|%-Ay)CLmSep->Vd|(Rxqdq>WawY$5;v@ya8!v4DC}~@O{OiQHW}*92&|z*I<YQl!LR3^-d!M5sGkY&A0Ktl
BxTV4`snWH5)&^sT@x>=Ow_I_eTu*S^7Cs|MUppGx<V6n3Q=E-uo^Qi#s>j7ei>s2tRR;N9H}UGcgZ6YZ?x&$N(&kG-?Rj?;BPzBh(%`BfFS8nP<)qQZ!^|`
nYhEzdp%ledHvpS{BfQ^JhP-1-U_QmwW{zf`xp3!r{mxL9zh*D7pot?{a$WXC0{}>QGI(jj2_53b&(kdO<hkB(b~$+TXV^?Ypv{l#WYE$^_f`%_Y-=+ev_Bf
N>bpF^NZ3g8~=B0*<%ijG?RV_Qm2W0fk1e=LI4u#x3p>yGbQeNEH~WPJ}ZQvtLD&Zz+92H4OKmROCLoDlrkfnA>nog$?~%@$A?{*#u=UxWJ$oJuTSnPyDKE+
ay*wGaSG`-0>x~<BY31m4y1u%3FlRhyEaBoSub?TA|-=TAG&$l*Lp4OCF9Oxm+8+FL2?JlFPJ#p8&^s_R{rK$2Y}eAhOw@6@?BWl!XL~>v%v|5S;=hoGhE8V
VJ3JCH-`q<_5aX!F^qRDmd8^VTNp~upcxhq|1Jv*_ZgT#i5ndkfx#C39Ak4`=o3uO5X-KHuaNE#;m+${&vsMayojcz$Ug>{2+Dqn!pT668<5JKcCFUdtjcVW
<vJ3g@Hs3gC0#H<i0;d0fi;vh()c=|(M3eh1ByZvqEP7`NR^V_o`CH6D+ZPv{HrPMGysQ>T})V9811`iM6C6Y9U8oxwfB)a`%+M#ZkP<)Ex!@}$anFDGMSBx
jt?1{1Z%{61#@hw7=!(1k%ZE^xfe!^o2z*?icR0)=wQMEJBQk!Ves>V(c>W?u}+e%Vf4{~Uly_e9)J(P8Ziw=YX$!rg~s)Q^PHbKE|c&ajxn`U5e5^O1fHZN
ya-KvhT%@wDn_z`lYo`^OIX?F=X?wCdNmUwVSBQu#BSgU8wRJiZ1GVj3Av(i%LyVZynKJ1IMANit|oVckcwL9nzi_$`%DD8&s$Ia7k87$8sxmsBW^CeALNeO
fY^XoR6?onPCoci@VY(J(&D7REGXK`rsh!I_Hh`wfPAuH#*n9f3weeb`KaY{ZKHrS*k0DI{nhynAFB+=UYOpA2*Cis*{{Hr2ccfjVN$-Fd!VC}1R_<gMhF${
YQI+}`t3H}%aoZo$VviGGR>AR6Q>B*vfYC!jPj_{ctE@2v$heAzlOi{XGSO`+{<5sdT-F@dn>DBqtazB%uNLv7;b&eu`-DfJIxj+ZYC)6t=Ai@9n6yg-!Cd?
hXAcZba9WF7L^Q6=8`pfNIyPpR&IP3&l$3ams!(UWdkjB|LW?}Z$r|@5|%<I_T#cI2^m|)ht!XUPg)z`E)jC$s0m8R-+4N{U}_Jhkwh#SweI0g$b%-2bQKXg
u<~1?F~uImNv{n}*pTOL4%#RZ5$?fzIOT+bTH2o$5q*xut&y!@>ecUFf5HC8|8NTA-_oicAK#)JSe0WD=k}j276<)U{LI(ei>f_oacN4QTmM&IFBS<Gdg#u^
xZ4PgOFMwH^iQEvC>AKzUl#qFvETlkafF%6OrsqVlQ(BNI}6YVrh~_%t!fmZRN>EAPjzK)awGNNmoiI~{ax|op^sRD;SZ(BDR9;i7en&G<&Kf9fPMiL_Td7l
efjLj#NiaBH9U+32}qrFQ)A12`m>ssZgaCB`n*KBk+0JwDTpbonc?|)l3=;5veJfMtteVvV&Z%xHP%%<213?ZRz5-vSM`Vuy7&!_Iujn{g42v9hms|CAJ~UM
L|sdwJF{xb&@Y@B*!S#A2Hjk3zo(~t1^wn5%%9Jv|4<g7fi{#2FBW~R?!qfw4qi=7o>-+RHA9rg4|npn)hwt@UBMjj5EJOkuN~}g&ugG7OJBKA6((cXmoG*e
N#=WnW3J7lDMYb5Ahz=Phtgwps|RJ1cH-_I*$_l=Btsz4LBSBUFb1Y5bNu0Z3UdAGD1aJ{q~V29TZ8)vH4|$=+QW|V0;>29<3~2W)rND4VNhWsS{!M>tncSF
pIavzHL`a%G9WB+{^cl&z1O*jvV$B3A1_q?lk;yax2xOUrmZZ?iK;xTJ2@FVqUxrXu@g5@eFp^(iiqn(tC<|4&Om4%Wf@8^T`uyQ2Lv8=OTV!ke@VEO&*P7f
hK&}*Ae&fdd};Us@mS{0ZlRko9SnGn!d}O|E%J!Q$N~rTHEBF)b40fn09~P7A}+t*mkG}LYKhjQ8WL@S^dkh37DKD1l&b!=Y{D*vjBKqVeXU%++A{7LM}C~(
baIPv0>XKx?nPDzsK?-PK@&f34nmbIj#<suhfayz>W)^L-6%sy_k<itY^r*@C>H>mLEfYWX%)8bUp5}#d_JdB=2EGz`smp)#U;-lKQi+hkBT~slu?DI6;g_T
r10fvB3Tqa!9I)ZD4=J9y=?Ps34G^RBkFZgJ%hNsE(yM22z}uD!>m8aEWQ#P&OkU~m~lxTK3E2Q;8pjUj=YU*ZTkJ+w-`Ihu>6`e;Y0#Fr9H?zrd`J2Q-rqr
I6Zy{BPV*v7tmA5WA{-lzja@zxs_!l8`R8QK26lYJ?Xk1@E{1CDsw<WXvgW=Y9Y{Ft%7hBC0|xp5J-+TOBh9&g6`my-ZzUb0yLHE4y=z@>A1hd4gqtv$&cV4
@y-#T;ujy?<y#jlIvN*G$I5wwYrXhdQ^XxPwK$q0n})~0va&alei^>LAWed-UJ;$8kb)SGMy01qLtT2X2)T&bpa)#$5n4_9AzKq$<tyuk$6NYXvKO%cs<=L#
C1@B;@mvN%{$G?180IPpOww^K(O2*vo$zZx3mjwCxG^miKf@$s2rDpIn*%p<Pbxu9bHzl|t%RJU5EIZ^fL^=gbz1~Ay>#J~r>u_tic{?Bq?=c6(qKJAX-9cR
lZMON42nS1PzOKU%#dwZ9e*qmVEZla8@UFJrSRV0(F!~15vS#WYU?_9pNWvE*B#STzMQ8j!0nl8nw(p#CB;h{iaGXmesSavkKOdugwU|ZwF2(PaFl=~moKnI
f-VUP#`DGBV$<@H+UQr9;~AmC7)Lv%&7?PNMS|}zXMuueUj3How$6`Pf3J|hi+vD#OKKi^A$35_tTZz*$N*a5CR;b*m|;Bw%%o0lWSFz!ibBQoVATB;FNE#W
K@3oxxHa~|INS~j!0K{ygS>R$A{dz4+`b_BZ?{BM1N0*FOeyQG6g7}Wwz-1*0{ww}%gybILqP2M{smniiKmq$EsScYDI5{(B}gM!`Vb5SpI{8v&!@$jwRUjR
cP5t=Dmxublf|6g*}wx<(K3R`Y&NP7tbZ}RD|%twtnOM;2R=~Gs5>V*6Q$0|^!d>~#}7W{ki=sw7Z${gpCft6klUmQ^L<5yQ$Vn8mVFn?JBrG(RTgF_>H)F(
8?~&`KjEGrc#s&=>c<M0uB0Wc%_dL<wtPnK<pf(u^QgE};c>zhXl?&SC<dyHQXWOB1n8+LH9CH%bIo*(tkMw`$2%_lW@D3PrS~}q63T%g(gso}H=)CTYByGV
1TnRnRMe->QrUDe!qD%tue_CnLT61Ee<S3Vpc@g%EF)(EJtc;?<n4B#gxBa4i}boN(d2n!Ig)o!sLkq2|BF-u*0LtqA=|om1a{>zT}fbQ&o3}a924lkk<fk?
+?%0g#mRzo?<2Zu_ko5y3$DfedJls#O=K%G1h=S1lb-Ul!`-M^;1|?zxJZ8J1RzJlz|w!VckwDV4`W;ekIxdYEjy?p3GglJ1?jr%G9qcUopZ^c9IAFZL7ndD
(Mww2(}&qZw22^BTxq}q2m2@m=s~VJI}*OK@a?y(x)9%h1l0Q(ZR1q<`Jy@`b?o;46ceXTu;Y&z!!r63rh9Xk>P``3_)^$QiQ}M&gcj@2a__2`5G&dCF|V&M
_9GDcH$T+f?h_PdA(flNh|(no443FoDqd{TN3Y8^XMUnc%#A2YDLFZ%!q86425Z&r`<S3TLv3Xl5SNmbffKb9BYNya)*xa&FH@F|MA1<~X~-fr0E2It5MDBk
@GdXNf4ISTu+c)HFo)-5-w9t%ty94=KR^&;#2kI2NSl+*dnoS?cGuoV{ghp{Q^(S^B=7{fm^-uEPPw2zOnazZo#{aa7x@Inj*K1zI{<By*mmr(r{-EjdhBt)
ejL3?o3KYIMIP+1-iipUxv`o4Zn4O0LF;T#dU-~>^T!Qei0O1EhKeIqgU4#GFI?wMX=AW?!$_AZC7UPY0^oCeJ80ucs;zSj2p@aax$Hjilnt3%-w}@4uV{TI
u$}ZcLVc>|p07bB1EcfP)kJ}YEZjM!pJ0Mj><cTI$!{EG{8F66Ye!DpujHpeG6pvio=A>e-H9A`lBtkY2nWknf+WC7B$qR?A29(t@s^XeOTCB0=f<geRCuaF
sk{|*I&%lx=#^)PjnV``22I{xdv?(fr0por9>^Ez7i}&{2zIoMuBPnAN^#Iz*`a9=l7;1t6AK2p?9s6e=oGPiL^<HQO1%o%7D#Uw{7S}S8w9I*#{JCv#cA*i
pr?x=I>Pty{Iv1`4U*5k)TXWBN7T`AW7lgOX1m)}{AJah<1P-%4r4F)p=)6;&s0sWKx;5rmzQR?WlHE`HGfg$FYbJ87pP0k0db$wDtq7TeCTZ4#tg%(B3r5A
%pfdAy)1vu*R8!iTawa?rAs_Oso6ec1&%-Jt<Ai$Gv7>m9hO*;|0zWTzXdx5nmouaaSFiIzp;ZSr5v%7RP^?=M!36QEL<8+mgUOk_H9lK4CSXwDpJnd#Q;_w
@-wE<{1QzX{I(o95!t@Q0e(*GxKmPwaQf6Bdo_Vv<}@Tc*~ff_yVIn+!!(p8xV=o5bozguyj4#HyuBW#k=Q+R<Vsf9@**I_oUPz(FUFx`Iw9H@(bqy@P55vE
_{2UfpRp)^03?F?sg1R+X-IYk3Z6U++}DqB>iMNTl0;Ht3Ruc(F@%3(&O`<`jBU(-ksqr_J500x$M1jxBQ=<GWCmK&l@_@nw~5+)JHD-OKjMaiWu>2=3-!T>
BbEd2Q3j>ZrH2_pzV}VLa8ey?A-wDk%ix;vIXF_RUAJi)yyWq5{TlPO8!$r*@*2{0w<u{Xo<NHb<)K>P({~z3V7fZiW$jjAR)p}fTo_%SL9F&DGibGMHhRSa
uH9btq6xy(HfccZkMW&e&y~Q^<n`&xB`3DN_L6BqRbcXZtaYtCeyWPX0%E#HVC_9VJfE2p;s^U;R<zytGhNL5-uePxrb%-ujq9ug;i34e{BrEDXogFJswRHJ
@xbk8867(=+2nTe&}5IYBw&&QC-<=r{$AD3R@KEH<(pm#v3~p5Qm)pdq_0NdDw&lObT42zs+#2l>~u;J<&-7wm<BA-G>as~QqKMWH;^*VQFyxk?EHZQkDmbq
AEZz`T{92)QAd&8`4G@{@bQ8*1yb+=e$Qskq7E?S)NudOpjI=?>($@(euqvILkNkp5qC`6uTR-RzI<?peG@Qj$J{pV6J~;xa%aDr`tJYU+-r)OG*@dB=|cU~
<H@_xP-lXfJALYj3?}c~yayDgwKBXCliG=^@1krSy=yHkZ@8m)sFkaA5f(|N-4e+~IC=E&qpn&knc|5BH$W3$k^W_POf5e)j3ra;@Stf-#0r?)T^bbdwk7|}
JXAFXmf|p%SMlloDMoJ!=w`SggF2D?MCe#)06M0)**d%4gr#_Y(c0-Ohi<EA{~AhNTBIo}H8+f-?d1?#tb4_DRfBr*rIw!iVBV+F7B}s%3f!mQ3!xcq+yDBe
=cHB3PNi!#nfJ-sDX})pGeS<iBLd6tc+AXqPic#P$7!Vam|ejD!r?J4D;r~Z-bj+z+$^HdAFg|zNMShLTORfvEYrZgVx*xWz7HJ^<)T%Ysjyo<P<z1o$h38c
$r51do8NKWf(h7GdWE>4tT@!P7H(sj{azyAgRYOS<47J8H3vS2`;yIy2=jLGeMQiA&($ieMEvQNh1$wUT>YQCm!Z?ppM(V@WoWj_87pGV5lgEmLyi1(Wv_6`
gwe8LWB&;>=NzQ=)FAM9wU0V5;D8s|gy^9c@Ob>0V1{}1W2ib`nBgdR9Sj-p=AKLRcWp|qe=NqG<(MpBFu+%r_dW{~+K*o&nYG==k%+26!k1q1=0yMf3Wovs
UCFfE3C?lS`hDGs>JCWgxPYSsuF*sL6G;yDPlZ7~BnG3`2J7~0C6E2h1heS->m#)bk2sEu&3@<430|hppQ%`g$n7MCZ0M=5yFo`EmOLF!kN|7p(>Vr^)t>tI
sn{xOu9!`p5E0*Wc|@7uaR^;psL;w+ld<H0BRY%Sremb!hGfNOQIRkgaQZ3uW=T&~fLKXB=g7~GH@r!maohU4<?R+AvC`oMiF?$$;Jif<bylf|0)T1eN`#CH
WC{G!sYeK>`(<>0=-gITRPmdmJ<IM{rs=K}P85SBzhSpVqV_k+ZUpSn=oDfFo3uO=krkR(0sJOFN_}s$(3$XHHp_ieo#y+4^D-2c=_!;=w5&Mr&<n&Fzl!#|
`5z=Sm776fR4tKIN(RguArZBR7!27oaGPoFN;Wt8TuWgpg+`-+Oyoaammrqsh1TdxvmY{)Ke>%Wi>$Une0(X9U(2f(OAkiPz?Pn6+l%wU$@;e{GJNY~C!<AH
by`o=1XyeaYJGK=7~2T+6ZvOrafD}*CwZ&$H3b*35x6P9vK2FRg#L7l<NXmZK)`xpSU&I(1ax<_=Wy;UvRvB=sz`lvSf=q%8Ou3<BcJIP2$_pXvc6zLytgKs
wtpjDo{0;3g_<r`O&>jwLNBKsN1+|BNJILFi#VY<lAfiseV*A?ls99E$y;%9?8LT@#NZ~r@Oi#wRVT2veKcDXbW}3Vtiz?#d%$3Q;IseVNij0;&@yzzUh$nO
b@Fi%$hJ(8M;t6@1wihRzQAGy@`ifM4iT)?88Pw<x>_qbWm#$(`S;c70uBy205csJv(G=XE4D5R+G1ryZ(~5l+C?2biqL7;YC#a40!iouRu#AJMJ@sBqDAd-
zX`wj;E0|pXqIFGN`XztBmZ;lPQUfCW>@ZrRjZTSir@Sjtsbs<8WmMu5-dmWIQa6n#+@J@4$A9oyU7Gd?r++smsjYOO7b)n;L`4}YEQ7p?T1Yf{oLKcS_@W7
?MHq@)}Vy5zN-iwLmhC{+rg9FW2OtLpKuujHKEs$r-H=p`lXG7J`1V;XS*iD=qfDXD*am`uUHlK<>TQTdY8M2*Uq*0KLJM=-juVUBV(i2G)yO^+`f4!iX&Sf
7)BDyHdM=h8&P_jcv_I|o&i%K>|Y#osPcF60VDOnH~xe&SNbNcu4V<G3amNeX}T)y6w|Z@&<*zI5|{j8ZipzQH|@7ow79b0S~F&<JOu4V&@tO!6u>Ww?#E!E
-GBlF;}rgd$wDl$LBNJQ10z%MfpEDDoTL0E5~ycM03gVz?$bZ4-8~1e+oL#%l{0wa>Z$dfzIS%=Rxn(g4`eP;OaXpvCr)1bQiR<X9}~FIE_>G)Dt4qQ`K4zX
<c-T*?qia;9Z8d?vtc}<63wo*QYospoz{{2HA?x4rff;SqMDiNS=wGmrXq_Z(vQYw7b8lI2|K+M_ymIsNJ*O_01Kq%_f-;wGyf(lwYFcR))b=*rxaBjR?d8?
1NypF1bn6mHrEbUlRO@mW3mo9BFAa}CT*Wt9CCxYO5Ps3_+U-wi!om#5iddc0AjqQtQ1|RA)0<wrNo=k)9R?>vT?^VU?PUTyREzgD1r<!tzsg*e=$eI`ebiL
vj194?(od~c2L38?tJFz2isd<`=1#yt?|8)krhZOX8pz>am~eMMD(wz9b!<?u_qr~_V>lAVcH9Ypoy+;h)QJR#nWj8Gb~VX6mKR)F30>&eCCC~6%+5(1k6t`
Tbt+=ia15TvMHd#DKF#slC3~tHn}lq!y;Z{9DHf7>n#sqjtZ>#FQBK;cnmPQFsJ4%y&etQG4YPANEM95X^Y<JfvE4@c)pOYBeD}lW#d7z+Uo%LbuU;6EF8xQ
sMiNLDCm{KM?S&dP85=lKaziVpB+m}`P*`$k<^@9*!3Ik-sLkbp~W(=BoMA$YXv3xb=mW+{WYR7hV#uJP&3>*lTTY@TbA?O=(J}fZ(>X~I;VydR5!IG7D9rL
D=zHo)*@zEPY=ThL^}~k({R6P(&JuoMewvHg?VTQ34p6>D@33_%dAC=10KQJW^gh;=Ab<&B5@1tUIAr%+J+wYGljjp@IEwKCcjqu8`Y%#^;qk6R2M4{0K8SO
lMoeWdk-x4_P}v9Kq`&exyn;?%>cyv4FS%EI5J(fdNqTWvYax>i#q;>V+kTo(_{Ayrqg2Mj@~T2EYT?ye9*(gWc>X7)2B_M1AviS3iY3I94}J7g(OP6%L6K@
AD9{c$VBR>MNms--2Y+8K@EGKY0XGr{ttOQW6Yyff7rU&@6?Ma#Cw{x1v3<&XiM}B6dX89UCN32m<JnmGwX8b*OI$~@hagf`T~-!Mo%E&uzj!eYn?sxUUyH9
lq5JLbN-&M<KzzCD3NH&lM;Iy>a)6t$Uqy|*`WEGch|)Q6A4#U@}-Arzo@rL&m~=ob8cYMEH@*&C&m|{Xy`|@kPK`4Kg}{4B#KjT&uWo$YtqW&>OZ>r1I>A`
p7Xp!>lr2MA3@q>N38jdnN+`v8R%yFjx>r~mtC&0yQIn?5XISVBrV^-{1;ilD@F8-KUUMaO%}Z_$mWtblbz7maA~dXWCmls58aKnp&7E{GBYzCftq7wd|A#p
pyy_#_<cfjb`ZnAV#<J=Y5UU@_!b%BhQNnM95wm)K3TXIh?MvT{AqjdVtW7K3;EDBKIv<(L9O)+A-y92s|F$RKco+pJW9W7L8anH5V>$q^27r>q}IGc=tQEu
1<M&#geuzBU(tz}J}E!LXOXO}Kxp6cR6+n>3gjuu)byftCofaY6&-yhL~Z%<$D2~?O*uitXdgK|`3*~4k+`vJ4hEN`A)jdzlMB56H?<5tEBVV$uX`<^3&m6W
FsfJ=#4@MC1}~h_aNiZ!ThSj@=|IKMZ`4M*IBXDnz>-}v`SBlU(GQrtQ=5nLxPGH-59gfB#e<Tp*WyF&zioWu3&U)UI<A~;<||C>C&G|IG|Rx5K|jST<aCV$
&|^qA=>6(4*OyjIP2<gr5cJ??xHNdrRgb?mCf(7lxI*Vs8V~yTQh_yN)U87yH@I8@Vv$z-h%|b9fuUCmR>cnLi=xQhVTaf0xgwmbG&zA1ADHNRi$w3oc$fMV
A&Y<E0M0HM>3J`CaMTzkzsMbZGi$zs+4C|L_kqtzP|9d_QJEgS;JTJ!I!g$OY3iA1XV!Wun}-$U(YU`!EQPz3Lqs`?O1h%ii875(gd3_$ZZYPZXb)boXIwQu
=us!-P*ZsKZ_nt=in%1#^C#Ns%K1rNM~47V!CQ7fOlPCmTtFhwic%~QA}#*M<0PFTY4&f~j+-1pru%l@gzl(|Rs@@M#vDkD2UWJUqHBcp295B|m0P`Y*>D#`
h~mjV#>v}0?v(wrc#=D1PNEP!U;=x#<A7B3;VfdFFl{lgAr7P613vZyfMK7RgHgluv1C&uB8St@7dT>x8vfX^vAYKL<L<wQw-uIo7BsfH6PLoxBygU1lQ=3>
*}{Y6OGtfk;-@jMy9RA_iuu8C+&l_z6zaNq*LnP34UT{~K>#~t1ya)Qz3r`P6~N|QU(w_a=#`q9ZH;UUzI6jM`#yi@6=@G+0j^(`@Ya$AbdD5hBf!oSNOx%a
g;r(}3gECzJR$8fuQ^tf2aOjp6FwRF-sQ}Aqc9l9UMWwz-i$#>esO>Prl}9tZSU#VXKS@DoH60IgvRPYED3ps-78H4CDsm#VlTA925A$;hy9ZiyW|Y}@&xlu
Vxj=GXmM9Y46+T!XAov*yVc?xPnwV2$<P@oBJ_fh#@}F@iU5VB7?<G%r0%_Q#xZ?z6meH?8jXK{rZOrdOjeS=T+-3x8<mKcOe<i(9Qv+tl9AJ`<zRB174)7f
OGXqv1wyd#Uvf&;^3Oa4LYjfU^eyM9Ua7yhoi3iU+(Jr*bR7N3gEQ!`r=<r3=i`MTtI@2}U)KCUm$s}_^<_k0v)%T0sH0E$S2n2QGE57QlcnQB$oafB(FRX7
&F)`oF~XDrH;EYx;7E9M9&!24D^`RX(Jw%>2{0lk#SIp%c2AB$wmqj*`JVe<o4>YEvx?A?V`78JdLP!Y$8CZvL9L^<^k_iDMvBEjIi6F|pd7XrJN}8-@wOTO
I<2SCpNE5#Rna7RJ$z+0>nyblfLqvW(F#Gk9mcp4*ME^rPHPu8&SoA{kTwTJT`8+U6WZo!?A4nXD3y4obxTgmn#yZdi3#uv<<sU-&rJY6XGT6Q?mobcWgT_k
pzGsyONGSL7M?(`oN9RKyU35S02ikLW9P-pSK-g{1^LF~MdxrG4>lH!F}JSxg@BASxj8^ZY>IS4KX__(HFg!&1;*>RvXXjA^>fJ@zm|MBSs>zU1LuNx@s`~W
ILq%Weilh|BsLkoM$w%^!@}oEE%j>WW7-_*;85k*R@^Q7pD{dV%-=N=^m$Che50yai|RA7u$`}^o^xi8zW^#xwzq_A8-8Lmd*<A9xt0{;pOBx)f3_(}D_)ob
?M%AN&a0piaFBGy^oD|<-LJY+VaE7Qi&Df5?Wc><Uo_ELaB>ewoFz2Oae8$<yq+iuDQjs-%au^zArd<%)*e}@+Z`=OoQtC#srUEmIPS!ZlNH7(rg^5ZFM2zt
_4#%DI3$NdPDbXYIZa#FEG`9x;lvxm@M_aF+cvZCPpeSd)Q^GpIB;%cZXMo_?DI8;LQU~CF#xCCw-d3mWI8bi5-gui6$Z~bHAXdc>QTRFs7Fen7&VW86A8+E
QXXeuX7<bIT<r}^fUc2e+J_pA)VyoI=$rILKUV!a;$p)Y2KkjBD%8}kL0qi0<Z?UejT5LgxsZslyAFa{wktg9rqqR5m+uepV%sU^n*l_?8F&C5;Qh{>OpEgl
fR&8}go$rcivLDUh$_{$G^AyL382^IBq;q*0Kd|ejEq|9+kGN73cJQUgg4gxQQr||!OZ)cWS#2aw>!htfGquNQ8^e7Y@r#?z40YgNF9Tcxqt>f1$Vy?L!e`b
$s0HSE^4VMED|Nc#cWCSq`H(0@N+oQ(hF+Lj`A51O$HkzZaA2609Xbgs~Q)@FIvX|Nk{VWl<SWu0p5daAYgk2<jrmf!k}y+ef#4ZUF{)MHxGZj`D(t<me<|2
i)89qRU=hOFwo%r@wXpRYE{nk9q(6-ZyEYxaWEC?$tKsew#RCyb;Lt(ucFG>lBp2135Ijqo?Cy-RY>>1S|~H+#c{37_v~chGpW8eDXBOc{Yf=?^-m&&|CsNe
J1|w^OGonbjrvK4Bo;+LG)I#)rO7qBr*jyHmT2dD7E?9eNRu3Un-;P&Kzv%%)|*yzcy8XAD_TYDh**h3_oSmrp5lufV0oTCk=gI0o{!1tUxXPK_i(U)#mg-e
<4ts%4G#IeNwz{TITJ^S*NoHMjFz#@@e=_=!z)gW?YEm5inV;5=L3rXRSx*_bY)xAC3*04#2_a1e-N9KG4Yn411=BiRIv<VXE=xI+dWe;ZT^IQ8YLhz6#<i=
XSA?-Wi(<hMN$D8e;NYmw5iS9C#@~dHSQzA<uqR&pU+QR!RAY+iVRJ7pjpKOoJnB+Vyt(O;AEx+QN}#ena|FUI!R5=K}u)~$x+MlaOQ=7jNy;$FrDCl5d_`6
WN&=7;R)nCQiIbq0FoZa&S}CQYj2YCivHtM%cNY?a!RIoHpMP_I#s$NzJc1eAqoGc0Pp?{(K8U-ExQ6BwEF9opziK8NM3*#d({B{j>Hd>$$|rdb>dAM1`5Kw
XCW;sK3QU-TPRnV8~K{(8p<2ZO;ca;Q|h3{?!x$?0&?;x3PimjpLFntcTc0H&(eUWiy46NL;Tu;sUJ4^B)-!b3>*B|^KJ5QSWg9&62*pDE1cZR(9pX!0Gme3
>v#J8BBe;Ny_ftOZ>LLwGr{b!U_VoHhYXDPZf`L8dWcpa-)}ziATXUS;$}2NBE1(1;cZ~XKT>O}e8=GZh#ahlh|Q!9RJ7KP=9F7$1}PYZ|J<dv^a^rjyIg%y
jb3F~x@^<Ds3CG5mg(hTjATAca6=_I5$dS}o=B&BgUOj;$kEv6++A#fVAK@D+^(oLQ0Lc#hs5+@=#_HTxiuA|`j4F%tYk^m9{3a)`C+p+C`|8`J&LMce5+d(
IgHu=4lZfjCgBD|q$ZJ6Pj*zp{QUDpH7P&g%bW8$-9Oh21kzi<ngn68fW$bnTb;D%?bn@FR^rlKy`%tF;Y!a;)LdZ(+vBZVk9<HELJ75P>a`?a6B;DUB?su4
LrrbreS$kV5l+JbJ2AK9$(XortbaLcp7sA%0n~Gt*+Am*FVuuXWKioJZ{r@GY<UlE1a+Gii0iU(0Q$3k!k<1R)e<$@!U+Yp`v9+uI;M~3USX_*%?ZVG4M84B
9~_t3UFj2;JG5ci;r?v`4{EnEVolK`f-Tt#L%1ECVaCDYDIhA`D>Ms56PG55@BD@Sqo5WyLhPx<W?r;*!bg*=A#`~6JGs<G;n_u?!pl3AZneQ%km9Hg<8z{>
6S)Ek`KKH0Z{v}W7l-wCY;brXunj9}QmdBoB^E^3Oi%na%6XGJjBE<I8bd4pJ8!c3hd8x|NUt>Dt16|v@i9IBmD0P;=Ih4nRf3s)bnM*+{dff9g9d4>70wO^
p0)Y%eT>?#krUhes3Ut9tlg?Ll5o3IOU&WhbrooQI0ti&>&hds0ltb0oBR{y9J@CRPk936gq~ReYpW*MBTIsrn-1)B%N-Z@^5iW=S_y1*HDPD0B2jl@*JPXR
s5HS71=9Xo4}Crj8GrR;pe}6>2<ofvgzqKa<Cd><xl$atxlx2tETEZp!FQR(He>TxJ6Lg-@t{Yt(t`weJZWCY(_8}F?j6yZR;$4KM~y_m^uVFAgvOe*2vmBE
JZSr{p5B!yB{fzj1uaL%<{nE8PP1d2GI`u1>lyj<baa%;>0Qe@tD_UEPfDlZcR_||ZwFecSir8*tEQMLf+{ixRoX^Y>?yiW(Q-0kn66&EP?&4hCR+R5y7lgt
cyX<X$|^kIQ{Hjop_U#1JIQpH?xFMTtM0&HQ6~=X&qu$XF|KLF5P9AKcayTeO-qSdPSxj2sjqDmt$n5pN)-|q%#_Gaw6PcvyA>FFWv*fPU=aSu-<pX9dc+f=
v~b<I2zvHj)0L4b7twy6J7ohi=7=v_;4^|m^(+Dp&f3XuC$6|v0*y00T4x_ZsEj4vD|sv>>zZM}fz${rI!bk_pLqhMRtbKi+OnC83C`tuCEME6j+2uQ6duzV
L#sfS*h|L4n`Ph0U`*zk?}e0ZjH_Jorr-~2y$iuAr4vp+9!yZ}H40i`Fd7#dqUU?Wfu8vSok0&-x|dvw)MNu0bzrgGiVxv&yNSJ_SoL~b8wsGvbtPqWS2}D6
ztYrUU4v#lU2_8SjVVC5a4p?#b!Pi2X7#Pm*%#<+CkHIbmQhJ@{DD;@lrxv`XP7*IWgfd5gdzdgy4rZsBDW2bkhcUnzMe5I4Re*T4K#`nGtMFvI#V%%7rI)x
q?q=8<>xIt86P~9_IBUbmrb_d{VE<ObQN64^c##=tSa%ZGLu`!$)iE9aeuxa6krEb>J2fQU{cohEUft<4FtXEp8Jd<$%ZKW9*_M7Mboc(LnFR*sC-}zJ{Pv2
D-^WjK&j<{G<W*2hEGpnn9FpGP#%)$MnO}jV25TVoE2VTpN_X7<btw_&vUtUPoL2jL5;DElZ#VuX;H7P|3SfEb?T8uVr@+y;|IB4cFR8MJI9np&WyGSSrGjC
NP2X#Gwlf@F7HcQ&|{}Hfp0L5^3$OoD<m@ikmTdkoLzrns2kkJ9b#KY^<ZVYbB$a+2Nr3Ex`Pv4-vuAUkZ@*TG*WjQA9%KM00#4~A;AW9)uP6P(YVd>^Kr@}
r}PF}Bq}q(3ddqvyg9s=h5nN&dOJK&7lKmSQK3Zsk^J4QG=wF80Kl*O7@2kAT$QRm9ZG)*=8Kbtdx#)BuNb10!;d1F3Rg<lfg->DU!**Y&c8b4h|7X4R}^S$
PMHlz1XVn8Xs$!xtj4YECBRP3-$lK_xgjWHV%WoN7P{Z$U{o18m?7}K6cg-gnM}qHAO$^&1HN3YM%ut^fD$h~J@CWFc2d&VsgorRS-L63qe82m(ekA6L6chc
rP5mQM7`7T$ns#IV=7#3msJGsffj1l4prl4iIx{s&5r$2PR_wF*10htGu1*CAqR>mcB|D)JLaMOph<+vwiT8e15mET<O=Asmwa9m4lp}C|1}Q@8l@c;t%==s
6$VK`ETkWkQN|_&Hk`XiMeLZ$04r;f9<(K#X!?uQFF(Mz85eEd#k|gR@NFLMH=$<3YRq%1B~!0g63%VCdyb0P%nn=#stFEfoWPRz^}L{x!vi|dO)eLRn1wl1
zcsLnnY11vQ2Rq@2n3b_t9&$}MO~BUt?C(S?u=KBaRBoOJ*>e^j-t!IRHUXI-{}=-!-xZ!#!G#i`?^Et(UB?#?2uZf18z*1yjLW00Ta&c9%Cx?qrWi`N+2A+
^1B^>A`%+$-F^e>NcevBsd7A5XsKa0XeeUqRFy~9TcY)hL^>oi4S>&F?~7Vi=&0HTO?8yw%;!kuB?3e%6?K)v`on?OV9m6DwV;I^dyj*e0TC4_0$L4GXO?@>
IkYj1_r(Souu~FOr586LB9|3~9>|p(wK>|Ym?T#>l|0dHsbo}H^tdC&KJ~cJwfl=YH+4I<;2xqG+Hy;`*t<=|R-&Udk=c6W^pIz9BL5vjZz}DNimg6#n2vPa
cY|+=B;yuL>ixo3NtquTcF=BSW5MDPtHU?;Iyl>7G9J=q#^DxtK7`sBGFG;Fe-u&xA$bidl{T8yY)wDojG!a)usTpVzrugUH&fYfd=nS^iXXpxBvrB+TE}7@
+6%D7OAy@ztDTd$V;KFjui(K}<g!_&t-LTp#&<Qhk)|!d>1X(RwR`5V1O*A+mLQt;o=3xC$UAvh00oRlg%-KgffBf&(l&d6TZ&5h(1R<Lhm<!Meu+ok*oA+R
d^RPPpi$<aWj{_8b_u)erFXyDNp9PIBjV?)c}gUBA>inN<9ks+qLrCQ>R?S#`8i^Ed^Q<g-iG<;$FXqBKh=M+D9=|ML0SmR8zQ>x*qGNXe25^!n;3%0pZ;^q
dF^L^?hE>HgL>8)>tg;p7rr3;db(J<j`Tt9$xDVioPy7BwN1MVu&*nDDl!KQUAuxpllZKQe|fQ8aq&E{Yjz3#DNBpXvF&k!2RXNvF58(+csLPim#p1V8@vbG
n_MCR17Ww%sd7FI9{s>rm3OcbfV(7)ct74Loo3WuqWG&39xr=jF1K74`}TV@WqdBlD()yB&{MhymvBaqtGW3!hw-iNV3ix2#{9|+2B^GW{hKgoZ|sX5S$yL`
l~-`D?`ecC3=l-t#k@hjye18^kr!J7p!Mq{<GqlI?}=?kUi8CE#xkWOP({Zef~eo-Klr<G@QVT#);a2)@FVT#W(riM+X$rH!MU$4a**}gzo27#nEw}W^DZ{c
y7K-}yQNQHjlI?rSaqHh=#+9W56K6-<Dhg@?=>zhmlPutGRtdnpsVVYx{bA@;^;msA&RSWxy=@u^KtpQInMdjd}=ORT*5fTJS^rBQF^krOQYDaJYdr}AS|dQ
jq0bj{Gt^j(~3%M?*6)Cf$qH9g?eqyUj3HGWOP2H)84@*N7Z({gkCNMpO<?1R1s3xr-7ey8aLXvJ-MDz59TWpZ*)}8VZV`tx-TcN#CCi~w&RUElbWyWY$O0(
tpw5rK9`D#BoQiM2ec7XqCwqEd9BsY+mr=L65gATUV8^Aq^g-f-HL8NlI1yR^IB%T&!(r$J4Y~md}u9`Y?i0YBtp$|0KAOLW1H_KBkbO1ewR<GIz|b-=|*S%
AX>@S+R-p*nnCz3FuYsHziH>`kD1!W_Q7nDuVx?Fl<M4&ei6>W_i9F}K;x$~Wglbqj)>UI&OQ>G+9CrgWkrt#vA%P+V8H4a6DjK$%{=q-5P(AhSKc3f5-24a
ppzVg3h=_b6+#iPYp6o%Am}3QCs-M?YqBv}T+QpOLJeLN`XGE1!eNd9v(^6zDAMEjp#G0T&?0nQ)sD)mBtYLh-oMC6Ud#5ON;SyKA5IK0zwe3e+O9)Rx8iqV
o&nhfSE*CAu(($yG6KphRqkx`37i&>9z#M2`^(udFs7o3fx0|v3b*03S`77)GtO(cq1<;kp0mkmZYAJi{6VhS$8>U*55i(?_iHnCxG##yEWIv}=xDe>%88$_
L6hJ$9?+*^2M(Q_5cv`j&5XDdNfh)oFsy0IFj8e^I$-)`8W}kAG(1JK{hY#laYc|YlJIc6po~5W0Vh&>x_lkz3gE*1e=;5x5H(AdDQa!c_;JTowRLZ_PJGCE
<@@TxH%vV$AY(?nPKnO(%4w1gvcZpl23LDfJR$!bDsQ$(%7T$0A~k+fl!I6+rZOX|p@myD{*0Wf`dg!X_FfY#wEz<e&C$?mYkwlHoC?0hsmM0=&UKYgcayr^
DQnCAqNPNJew03CV@sI9)>gH;?TR+4@u<4Q+gAxRwK#(Q5TrJ^qNySn;l_y8C2c8Aw=37hP_KmVY}B;*HPbR)|Aj`OL#f#zFtPoQ7*~3?txKK7M)5@hD`pWz
+*<0{LGI}Z_UQM~SQW3=+kavWMUdL+pphk|iPiF;7R|agp)Rw8kPTov!Z4drypGEO?5(BTueIFg9Wfr2b|*S~DUaLVFQEbE(PF3e@g)+6u~EWMx2=pO&Qc@}
&fhNC6<ckl*lFqqXs{%;MMd`Y3w#_Tp5Ya?DGHH%Y00g7C8D(R*mR)53abRfh)&O7neEE(uD%029LZY@x4*dvbYm;o^82ZA1zjRKXO3E42Ex=+BBVAm-TE%}
{y8P|Wcn>s%WQ#99*pjPKMTe1+DQ|b_T3aDop-hJx?<1d7uRVNcn-$<?#^2&9<~Da)j4p$RIw-seliP?!W6+kDka$m&LLSz|LgZHEzLKRzs6sl^xO2cL+PZb
lnY%kG@A*4Wfk2ISEkqy(kGb4W{?WcNK0Jym3vNN$H~a?!MzDS;B@LiT1+&qGnK1Vs70c#64V{#Bb0vLo?TNA)J6&_?W!xp+@WV%7Z7B`MWNq~JRUZ#dj5tK
B54cufcBX*8K3Z6Hb?ZJWRUSael)J}VD7DZXb|pyjPm>8MQM&SjWOM@{WcGDKfA#CE{ddzB5yJdGj86JdYABKb`1B%i<CnVf`<^OVOePo`$|2W{kYDmBewP{
waFJ@8()_MCFTy^f38}PWNvn;krey=lA!cYHD)WW$+bN9TyDEV5uS7WaZcYsDJS+<G^+#&lhRo`h$d&z10}e%Oo7*i^|RR;+z8-6!dXds(2=nc7BjF26EC`l
9G`fYHygO%3V1u7Bu!o6!$8TofO@=Q3d@X)gTLLMrClMxBdnE@e9dj|o%Fe<QF?D{OWeJqkse&E8lTHriMc+#iE59v4?M?j7l!BsSi)7NFL&1abU&)T=|?q)
v`f2?op|EHi;=m`l_~MJZI-?$Xm@-g`R}xtqEEFk1*=YVcv%VFmYGJB%|~QP7qK*A4}H}Xfh(&HFc$|y#{T5|WG)0|E`aSkO0;mFqmdQ8(=cO{iuY1<h~(@+
%1d`cZPj6T9&Cp8@sx06CEq4y(4ZS}-P(R!U#`ua&v*t3XX5vumd7?vyFNE#)ZUD()&*&8fkM#r&P$4w#?2}ar^i7auW>kV*lScw0~>=c^z=nKY|qKTZkKIH
Q}4F=UY{k)6#2+o%*cJ0*JF0)cb_LoWr@gMike-0F|oX;?SVtZ*C9%gYg+NC1~Cl6`v(?dCn=6Ocr<BFFsJ+oW^0Ba^;~G|$uuTgY-!*foz8=TycLRqGj%c(
)IB#mAi;lPlLpAX8EWef8c>RprWJ$bexc*YGe8Qllkbki&c(1hPPm)5HioA>4JR#)4lFI8NHaeukIKaIX$X`$%1Q0x*fuG%12E)3p(7PtOaR8b&XzL{@fpUQ
;t5oUoG6I66(H#wIrW5`;>+jGpw*NP2G+q!ZEm`DtxS3=^dz8Ju{3EDe$<gAEnEFH`ot;2EPbF|&izCYZ4(R!LMYP;ZtS=e$@47cA`1*cYrB5eEuRR)0-@Tx
f20n|bHwGV8N!6bhP0T~p+tHc{{e18!R{N{*WQVNYEZ1F`n^*mF~ru{NM3OJH;Gd1TAH5Au4KQLV8y=r=B`CIL<^?G@0BiWO9SF4I1bqxb77hkg&<4#vh?a}
trJ^h?mX5)5BzhHsIn)&#<S5ST%R$*A`Mm4IPjOau1xAok9-cA9A>Edsk3-cSS%=zGRNp^Ce<ivdP~8?0RmJU==r#&D(s*S$MD=ifvIIde5DSt8jhlwxsEUZ
N>_h5zJl%!ZuB;C)NEeaV(?w28`6Ctp1u#|a0j@CehvEsqlwXH@p|o2-WUpYv?qJIekbv@;g%faF`q0XA8^%?Nvdr>pO*)__&X}F9T=E7<)gj)*7S?R=4OxK
nbR^Q%U6fHZ6{?PyJ+~D<$I@{E9U3`_>0zo>p8zUrGDwwm#I|HSe-5T_5_)HRhpN|=SC5u!D+W8p>t!#q!$MiiBCJ2StNh=XZ~%twNQ_#tj{OM>fN6;4#kuO
17vk4(pA-8ESBXRiLSp2*Nqer)n$2k?kv2*^a=my`sfuFsMlw$l`9XAyeExi^eFeInGg)~2NEs_PP!hO@0)UAPM*1(fDr0oDLsj7?9y9Pm2-aY3P@asU5hQd
tKY<0fRtHYoZy#Y428GtoMuV16BupHibj8=p|oPYPtka?)$Wip6?4XA6$Gb0f-$A92K_=|ip5u21WzFO%=m{ZWO8*`y~S>;kgIF!ZD_J&_Vsqb*j?c<hX?N<
pmbF-=cJjtrv*9#Zrbr%g*|C6p!D{@k(D4mOqvNqYo1SH6YgFEkN#g4T(d7tq%6SsDq;I=DN{!481+=HN-}sl{EI4l6xlN8XsT^+%+q_eT{_Wgj)K;pl&mV=
;4fyvwrfmniS3;o{>nw6m-|O)(@YKap2$Vly<`hHr;(dk@1|Al1VI1j=0#r3Uv80`gjF$>Q|Rmmr<;J^sPn9w4kfV|`scfY**n_6NariN>$p*s@F@h#%d06s
d@7~NDQyGI*ge-PgtGrxCayaRf6Wj9|A>e?WucLYz237|I)JDsv%e7bo$5JC${P9QP8v7%jp7ocG?N3%=|dsHms#J-_7R156Ox@!_q(BITr(Xb8(>{GgM15)
e0op&<?6TQ7=l%T!&{0D_lm#6SEB$c({1A0;b3Fn+LjdgXWE4%kbjekr+qI8{mLU~KKiZ$e|EtI8b8ofT^^qxrj5k)bg$g#;L+%XXx%d-SR#b~4jU^Z=o{6@
jU%Jtn$_<r@X`o1o>>C*aDL7>wNY!IMk{)eQ)yokgV64d)<pr`cqp`{%8ylX1<wtRdl59rDPRw#o(sJ5Rcy(XTP);Xz)c9wJWqiMLQGLT3M^`5$zY`!fF`O$
p<%yi*?)hK`nTZ7YUqCZ2}QwF4A?i*XBSbaPIV4~%RfZZl%?YV;PA~NE)=q6j_9|_&5tVt#}<u&0+fF0Z+tgvDY+NIaxFa%KaBsh{1|9HV3BszzPTAKT7Zcd
r5hi_?Rq(_d{)6&$70N>-)_jcdLqnPC4QzfR}iun3Zz~6p}}}=#g4g>c}nFV*;HgTgdXHTDg(6gaUB%FxF<s<TDr7UYkR_0oXY$>Oal`GCLp5^US(2-R~4&=
Fj&#%tXj!2k%rC10SvZqbjms6=KCb4Vyuhrz54yhePpt9%PUt0)V*oPkEP6j^9@M9Nd06e@6}q7od=v~1oJM|->Bvij1amOf(c!C?dH6f6-Y11H5x5aQvhuz
iu!MmMDw1CE+H_+|87`8o9Qlw0ZZ27I7Sz@m;TOZyh7{BAZ+RH5?+zJP{hvJx05v{?v&a%C5k&IkSf+h6cg?4-vqeW5wCtX-DGct5r_Q6!lP?<qm!cdDQ=cO
;)?nwln|!_q%!k)ux<Lom(n7o^+Hl71_w=;r4Q|GFiVvs<BuDgl%y@qHz9Vo_m)P7Sb;4by15c0Sok2qPlp4AN#HW3h+7VN!Itv*==C`6^3O6jhY%w`ob5!h
lWv8+N~p|$Ly2zhdANeV;Z3mVBoAi&zQdx5Mv5$sju3{bo{B)l3g*wa2vG0|tiV8=NiIHV_B+e!P{UXYTA}O4#u^U`A<pX|x5}yQKKdOr&$mtO)%Ov60uX8X
b2w8Cm3UFow_hNX57vY)5=D2JlJwl8ndA817@eUUoJY495c!rvx*0FG7c?x!yn6S?Pu~}Ot~E5R2$S9OjNW5LUv{OcB$`JO=ITn<qJJ>9goP}Ytp+%qqB-G=
qehcx)1x`bmc?!<*{CHW{GdP_<^1>9p1?)=V%rNfF?SDno4hUqd|aY{NYrh4pL6B83il4Egs3fpKL~eEiA|?l0Y249Mp}aeJ^T-9)!t)}$hbqAZxLCS?_eLq
+bbeg#dltDsVpn*6{{$U_YwJG_?@7zs}an5c2l;t0}u;Tsw{+)cW!rOnk-dW4t$FZzL6FbQRpVm&TyDR$A9@hc~-u1em9sS(KMsAqYUPJpR>Z1+vp@idj;o;
p=T$}GQ9imqH^yzba=|{8N*6!tfgt7y(0qlI`@}a%nESDL-%%h1_n_NN)`nJ0%07rvJ0q|;)C59CBROl6RQnuUY5H`L!JTuc&K#KyuBpfTebD4Sm2W;pd8pj
@2yXD`0Of!cKzp@gDLwPCkAxQYiZ^!_W@cd{5$uPnOask)tI{KeRN!JJKzc*b^2HrV-NZj))=L16D(dp1h{G?S6QkjjXs|q5zb}J18<TmVybWp1jhVuJNS;8
J3Q|f0=!f=W34flay{i?UoK`9knun@Aj1Bxr82hY7(>gJ<=)DQb_)D*l)UFIsw@N8Y~J6i5ulE_wAi@px^FQQr%j(8tAH$++lve|3Rk4bxvL4)T$5|j^>cTz
N^7|?uHMx0aPx?Xr?z%Y#-fNLm-88mjhD8^WK#up+82$3a?{`>^{IsHlD$EeZ1dTixyWI-znM_&3h$Ib#!he(wM{NRRR|kFa`4%`mWR(lvr*v&=O{?y;L$2-
1o~%}-u)p?9hm9a+h0?%wqXtwMYJ}YpHVweyS);>aqpgKBQYn1e@H>o_&s7se?zrgsMOd-y|NFL5u$HJ?E)mi|D{c_tmD;qr3$;ajZ_j1&*)|j6>5iwO%zvV
Wyq^jiLI(6t*e(fFKBcM^AgAhgU^(972w9Zb8;?nwAD9CgR$S=>RryvB9)OL@9~_lyyRbhN~E9_0Ikby>};B|Qw~#|41MXOZS`QRMZgE=`#DjkyOclOv_#p6
?~p+s$9B$L`q8-roZ3tX@t{&)i3g*U3#h3;!vNwi9XhzX4VNoAAUt@}Qpa;M1;7vIp&zzzAU$SNF0vV4Mmz9Bx3!M{PO!pZ=C&!<b~<%%^<JEY^Dl)R`a5n-
`Pli<HRosc+Zx!u@g6hzz#M>dFnQ&|olMIdjp4da#--6FY1rBF&DiF*d#dUp=(OZ(0~eOXbS>q(3%tqgA%M@{Kh5Pe2ut-eov{91hg8Kf7!eE=3V|K!-52Mc
v(k&wCSghqn1W$$36W>{g%gir!NEOxIqf7sc1=VHB6ckF8P#U%;WWlp8ZozXXwCUqCXmW7^j(Y!UN`hKqjVdgfE-G3W|X%4J081_f7=5}hiiS{la-dvEZjR8
3Hz@dy1<N&SA<3@FrmYoIFaHsay(eRT9Nf4>$Bo~@A5Bj{}ZnnzqPEI0nnO6DyLt8QX;EhcA$nUlt1DPRrDMSFXCsm&;y4khm8Q`)kL!OyEEF1#g`1gHvU#O
aEXSUzk;B}IVlJ*(JY%{-s*e0Dn8Xv+s2Neg0KVJ%ZS3feDn&eWZ*!QG_hIi3bVf*uF7z?t=!3YNb9O`xF)g195l6)(n?VCRr1?H2uuiMqC0Nk@xZ|60XA7-
dTUK#qi;{|h~SVgP(ddKSwONaon-Fln{$UO=U!5)I>)4{!Br`cU!qfx<cL71S5c5(GWO-q_&9$M>IQ6jQg*I^9ZCyIe1<28Z&}&4pQKl-qV1g|jAHOgK}`~o
E1tT5N6nQpWHl}0yxQ>wa0>;HxYE`4tOw<!IQs*!iaJ}+t(k7F&uQd<bd-c(l{WynpUQ?yq2xPW=Kc~iB^z;5V_X>|(l7R7)k`(*V>h~>*lZ`1XMD_^z!-)B
FNqEKl(4}dL91gYak|ZY6lvF!n6q>)?d|*qzD;tnY7EgvRxK%B9_^bG3~Npg+AI?{yAHWeIv;p%;q3v1$%lGIz2p^+1h*YB6DM6@KxRl9X8p)rO}f(<pwu_o
TV5W&b*$8$Vwl?d>0wyy3-8FNzV{M)TH?>2>UzKWhRyMD)~otzk87`Byj-FVNZc}WKMrt|1bH|X0hsK(2@sWB>Y*%tnhlA3uo|U;Rd)$Z=1^0RbO+gJoo?V!
5!{JjxIP(*xO(mQC5Z@jG3h}F4hJ<}Z?3|HE9=-An5L|G6qo%zN>Yz5czN$wyS*FlxYp+y_>6D8Lk9ZN2>)=MK%~I8vaB{}tSFzQgS*+7d_hNx(tIiDY59|f
?CW?5JQzMj-!RA{G5jjNJw)4?uLIqk+L`(rlV)PGg7ww)x)ypn@+mqP&*k_pEBuB<r?wGh8ij5iQk8(UhH)6=R0m;+pB%TPGn7WhS#=L8@<PW<e2cpUeE9Xf
A?pN^{d`Dn!BBh*@vD?o9W{_$>NB|!={qdG>~u5v{MzedjY}~H4*<oE+4pGnM7C2XJ4)Em8lT-zv8WjnVP94ubO-De4`tVYhJfYe^nosrtZ;r(PZ~er4sfYh
50|SOomJk&QM3FNw%!$QDJCG+Im|Oud?4&vj3FF*`!-thOa2Dcg4-^OIO6wUOM~K)xRHt6*4!8Xq?jA}qB$G&Tuhy7f}*H0h#;}tQ894#EiZt@@w#C&C(EED
el0!Rbp?Xgf-BpVa*Bt%_|mB?S^}kQP|ZpqO35)o_Yr38=&F93OxQtxQ(N`~T>l^*9K0}dYmRud*LUov4=ZPzfZJ{9W%L&k-)m^VYX7l}8z-LTO_%<$9#<?o
W@7IcH#|!Up_(uwq(M|rv8MuRHkMf#!`vcxgc~G*IzmJ(_6!lu2>h&eZKCtQxgdoVcJ=)%Kv_RzKx>J`h%lPu;*e{S?3nclfErilL^)A21^@4|q{6AmdDq&h
muN~gLloAV1YL0=Ta&8W8SOKS4e(s$`haWRUsDqbV~B@{_-xV~`L3Z-h{k$8>3ri@#T<|hlcFdzT$|gO_d3HD&Wz0)KW%Hk;P;O$X8pSR%kYB?c9kuxlaK=c
(n?=XKHVzx$&$K<97!{R_1FCc(LTb7ra#giYsCJA3ye`x=U~1TiHdZ*ey24dh8yV^1aO9!>H-Op?b=J%`6qa}SBWa4`gDXNZO(-^ePCM>zGge%e2`KS)+#u3
(P&x?8M!@|WalpCEy=Cdq7LJgoH_NHH-hBGh)L?iW>MeF!_C=Do?!{=rx|%DHJe9B2+sj8?Oe@V$D4DmwbROslca=bFrIY}3TWleUW^|~<s#w%3dFH>0G~_J
QVJf)5n8`mM&vRNeX2rleUJjSwTh#xVY-TNyjvlL_&h+@3fv|7%>W1?p%&zE9B#{jHIjwrEe2NgHv>_!?SwX1b;J=q>VPgwr&CWIpd?dp-f$xLZWa*!ApO;O
MA+X$<w)Eta3Y}97tBv@+*(gN8Mf+D%7<MByjTCmWDUp%H+w1Ruulh6p{4M2todm>r8P&TK9Mm#rA?{))w|A9y<Ez;y39sG5w3fIv@+u#s+$|^=WY{!L;x?I
<^90dS@CSvo{v6GcQUt~{d(&kxn07~!&?|Lm5+C-{h^lDaR*b<R;{nbZLIN;b3yBg2ZmVfd1)-Z)ZWz5{DF9JML>wBdaz-$i8{|q%-8)XdR_3O8>VI7wMQd6
jPY^{A!;imR6qDMefmr<S@G1)As2!?r^ZvcYPu=(Hf10Tej_B(_P|}9X={1c+B@xiwTj8r!Z0d1IIVmpWl;<m%85`0%#6k=Y1m>;16?HRA~{kQiL@MY4~EyJ
J?v;|>jGlR8|c-6O_wIei4NgeB#}SF_syP#1Lp9Duygm<)<Q>t`<y<~y?jj<u^F_-o<*=!3g;8E3iRoe0K6JQhz=mTbYSI}nIEjHODSNCD=+LB1DNGF$@DPl
{eH9}-hcA<x1?wUOa)x2OWNajbBM#nH7ThErN6i}HKicpK96Qm0}Zo61OVnQlUIO<?`LA}e%!bsT@TRckl;2}OBRw_dWJwJFN-Ae*^$RCzZ0ss_Z{A6;)*Mp
3{>i45Wm-Ac<j`uvTUx5Hg(bf#P{+wk3Ww7%|RF;6T_8apwXS&**J8xJsp$=m|ega+7monuL<uWeoAehjEs2#-MPeQkH2P_{Pm;Go3scTRw{nX<J>9V+m=UV
7>>u~T-19dlmC&sb?=;v%ii)33cO2FsXsAcpjuy6nM5rOJ?33w{0~9mTi9%^#9I42H6tZgmGK)1sE}ZR_oKk5*-%F7*9EJOv>Y0gRQo<U{2c+tq^OG`?2e;S
m)H@07zGE+cU`XpBZ=+RKdwgoN4N9qjcOpZvC9Z(hid;>p^e>}rDy6o2`5U))rL6KA%VNQ1nPhG8cU~|vLE_NJ?<fb3SwNPIK_UW^;MG(l-H%tinsE1kqNwe
JJ%PD#S&%+3!ALH2l6aO8Ya!v;^O9D4-YbJ2|f!U8(?0@3_)9EM{(Rz5Bd`l$hZb9)?O~gNFB}RuuE-1$Jz`ZRd8}UqI1l0?!wSNa7opdEfby-bPC09FE=dq
AaXzd6UIJwt0u0oix0(^!)fqR(-A!*)^SWpFFfffzt%haQRq5E^JSx`E#%NTcL_mY7A$%_as#V;L9-fTg_gqYZ5ZryL0AEb;kIxJ@-%VjNGHF`&f*$*iklp?
9w->ws(kZ<1&(mL26&6(*mo3@sp>CVLij!sJ@ZtQ`HOqTK<bAh;kv%hRv8t)Y@E+jZ^#X)vWaC+J1Ex#uUf$hJ4CMR7$q@*VAd%<nPYBt9kn#vE8mye--G!-
z&BD>hler7W>IN{phZOwI;oYow0%y5$$t{%O}X{^tF&N^W*`y8l5p;MztWoCLSgZq)y|F64>!78B;)wZxy##|h5NNJ4rMYgAiPv)#(rnhjEjnGKglkl&%c7A
upgzR^I}Yd*oW+9#0X>ykRuW~kC>1uP42ECX?{o`3MLEI<Y2Sx0AB7b7z>8hH(x;5!xO}SxVZ^W$M~}VOI*`-o&aua`)2uK))QZL4)ag-r<+1!xt@ag5uR;?
x);Ut=j9gDLi@Ndi<Peaamu0<ASWC((jD7~^DxAdbL%tEj3?I0{||lf{XftuEd4sM4hh1P?xU%B1H~7$hVfgBlPK~p(qQJx2kY8X*j{wkPQl$W<7%5E-_H@Z
)7p7EKvw90flU333|2JcOJ9=r+Ba2e6dGEZB;nnLKES|&%3jw=+w1L?=fOTeW(%7j<vT|$VUz%Js6~Jsc)WCdHp7(bHV4uUp6>zC`6}nO7`O)pPdXoaVA(H*
uT3?txaKd#xm8fm*NNP0wY1upEZ_P}L2o|ohr1dQev#RK;U0C8N!Ypiew!s*+A)J(^E|Gsuet@b#`jUfI!%$E$!_E>pb$o1te-%oG2cmc0M=4`DZWfRa5I6R
l*uM2yJ<{R5#;<28|MaRHbPyc;44l6boesaTSj0eEh&)6wX`f7X~(L<@c@*(=^I_q)~9E`IduGAHjwtI`&kL{$z9yLk@$XE+TwK3q>@L<E(7y#OYu+b1;heW
&ylp&$JuH@!Y7Pim&LIXwqh-sknKu6-P1?JEwV1d0iINpxUG~9jO!$nWj9SbtLd8Dt*g%s4bAZ7h;}1#kv&_4z@zzS8@RJ2e(B(1Lu>EEQ^NIHjgxx3Xk0du
>Zua_g5l+E`DhuA*F~x|!p7|jVv@})I0MG(1Ia{?zYZbAq4MK@$`#kC>S_UKx&_Lwg40(ZTKn89$lEA}e?1%W8;u?xS5^&nXE}^`<ziX)MIh~AtQlv2@$*Jo
TPf$Ch}wfcQ=odh_<HA{&Rp8hkCyd?)jzKPZ$5Ynp`+b?#%jrpVJsi^vuB4E{=`N~0uf>ZH;{l!XS7a5BdVzyC#<Xhqx!br)<+(}Z6Tb#{wBI<ajOa+2HiFs
KmROo*5%!D&ckr&v#K`iRV1O%^M^ww+w|R)88XHG_rV99s}FEhW14XLv>zh_vRCb0E?+4#BA*)dG{2BHRqQsCT9^-^!mmJE7|vF!!6rosXLJn<LV=SEIuGxg
uXtjHThE6YD)5|*k<1n*&!q!Lt8kOq8+fQM*+;^<IKnU{l)Ahx|D}!2F;t@fFpv1$@$l}P8Ldbq4Y@!?_BkmQg_04*Ff9qQQ4y^dV+QQyP-609cWvFv@Ygby
BoiT@^BG+#=H&xY#P7e1d#jgYX6((kZG~U(6xPYUIuxD1TWtkvK1j0rJ%k5pe2_lgd5g(?vqFZFMT@F<6R~&g80}<PjpyOo1$clAquGc{jQKQs=vS}{PLlI`
fT*v~14?JT;||XjmTa)Tu;6q-#x1U`W@=|}X0>g<=E|2)))1ksHrDCttL!0{-{vM8(#CaWPX;4RJCY=ZBcSfKr~UO;Q3Ah~J;jQcHze<P-8KCD69s72+2=5F
vi9&g!Z`FcXYXTWWE-W0;U8Lm{}#5tPTOX{m+Uk((SoOKEtpX#pKioZPH$cjlYo`EAnQi9i?hfNq2BNT+gKTiS0+xnyH7}RyJ+m!5+vYIn^8S4UG+6)G*)#@
^P?kad(Tt=BQ_&eG|xY~wU`C5SDMnDSj$Wd?o|+O6gSAX@Un#?Q4MZU1wvVQe-eP^<olu-9`+}SK>qI+aRPTC+?<u)$VjKf>}Cd)?P;ehW#)2F**{a#2N!q2
s={$$<%2J6!T@X!AxP>maZnVP#-=jRj?#awVqndY4jI+VAJ9}CBdfJW3h=qf*Ab|Z0(^9dH{di>AmxH|7UCrWBkOenYfFP1iMmR7+`D+aKw@;otqxh{KWERc
T-6W~<DU%CO8qvf<_jG$Ck72i$AiV05L_4e1ta_IrM0~LyMi!Es!4wR>>fW<;0`*Q`#9PW|9*td-#x*W?y?#_@>PdMOJmOZ$lS1(P)La_S=;X!NO;_0Mro?e
-CC!xLPipu5pF560g+)0lkm|k)G254<QM2)0G<|H3ykYkoMJ`8oG>qr-ehTux_etqvLdVX*9v>3lct;mEp6G=Q};kr9!G?=(OlMAqgk@^Rp}{6^BIdU$fSXI
S#tBPIa9x7If=XKd;5c3+)eArX`4Fmynqv3geathZ~)R^)bYZ~ezl%n9naDso-Am(%Fv@TCEZB;2g855oV49qADeTec#=!z^X&g0nem(ab7U{=W4}W=kXD7R
QIb)Jyh<9kMsE#~c2JXe965N7Y_KNo=?{egQ-orw0C-F~v-KhTAxcv-mP3`A8$0A;v}dAhouZ(dN24B!3gDE2ezT#g7wQ3ga;oe*^WR7Oo-`~W08N2wr#e|t
z4^ulIt7P-9N6U<oG!bRtFVuAq7X?hyM8WSPqsdZp3792f6GqU8=Ll7=u%lhK5~_`4@+EtA$c$5OBqK1h2QzjIEVWA#WGwF;{M}SQ0&-E*Y{7^9(hSSFG<BS
2!z0{fUGwNQ$j!>?(D9A`>rYl6(juBS_Ns~BhKhU#XA@+x~319BnTv&OFTkFNPMi*Alp7{%;>afq2~&rH;4NDA-3!Ve_j=1tK-k_%gN*aUfx1gWi?V>?a)j-
ntQj;Iq?~2*yr#YZPGqwo;h~G{cgefE7kZ(h6Hr}?A69%sP|5+mwC{!iShMJ3F=@OV6x)EYpA;!(8$uVN&gKBn~75>O8eke+0?>&&MOr#$4mId9AI{^TJ~*t
BmR%-b=Lw*nP!*`e3aGJ?P2X!X+03m9Ers@T@Qtt;w#VWWiXprWB8E6iBZMz#xl`cTOlk>i~hL;qvldvs5BX8XqGTD<#ltB4zI?MASXc0s{YEpY&Y$}ON8^m
qNW1kv5q5O^n={AhEu&%T1lUi4^WoqazaaDQeDs5kSGLOSnM3lyb(Ih)hT>w36?|)kxMCDHt78<<6?_)kr6z3Ws9k`1y=XQv(eI^>NANVR}+z=U|bqq^8iXn
%j{u6J%uTp*Eu{Tusn>jNEFFCZ8@pE8U!ww_o4M9TepJ5xM8LcorNSwDXy$5fwUZrRb4b6NF*Z^Tml|Cj<9DuZjH~oKZ}(2F97d4XnyIhmr%?NC<VQy>dh2-
NhpWok_X4&qRe{NmiSel33IZumpc-s^3LpPj!*v0=u@yzO~C3aF173D%v`Daz4633>N~uX8BA1;x^1|Wyhr|BNL@1a>9R~$7FYoJz@1<7{nkVF>1FXfWM{!`
KXkjd#HE$S83j(Wm`sTy1Z~4~l3On$?ZOGbqx=S6CKLn7yE290r0u&&3tvqo0WK<$P@6Z+TnAKixx2I07^%UZOTs<HXa_NmBcgF*$gWD}!#YLL#F1%!(Nz}s
W8Mrv-`vL;hRQN(`JdYT>h&WpC8%bp)Vrf&C598eSb^!~VEE)Z|JWSf(x<WH1XkMgA@U-@zCfh?qeWT;*wausN@`b5ZM3o0CyTgpF<}8;bAd^YQVHGO4_BY<
f>=*^khIROvtDm#6*GPA#N6W_HIlE*7}Pi4KK_BLw8<tR=3NeX!uiE+G?eDD@cvnAP7fh=thj+U)&H@$(d*WU&$+x>Ne_=N0jyIgLRnwvJm&Axa(t2_!;y0L
23P`}^M-x=Kl{?(6@ZGg$yzAkcBamk+w2KMFHd0ISR>B78PwpYar*z(Q^HO+{3oM|s1Ty)nuIG__@eI2q%w{)<?`1jgmInagEE9sX%j!y?Mi(X<o{935VlXz
IU(n-DUkVA#E+G?Ht;HYvsOez8LPf9!CP}$XA>eYQoA@`Wm`3l4TwVC48#LOXioOgX;qn`{ZX}y9o^dsx<Wx)WAtks+kHQuS^gJ>FO=ls6Tm|=gwmY!s-j82
btO2H#uewmRA>0`+YwC`UFHDmCr;|jyyo}~@Q&6T&kCmUJi?c-o4N8?L!sj~ShEhTj^_klM7BxCyZQica+8<d96q<vQD{c^xpmVZgP^CwRadLDp#GkvTMAhK
wDlw7tk^mwEf@Zd{wu%k(uJ3laK%8n5@qAeK;Zy}zU<B@^g3j+$QJ)hu&3G@vE_x$UHss}z$?__9oGnlGtairMEWD6Cb53k%EHV`hqdfIN9;T1F_?V+0s6ov
U?9p-QUt+TQ*Cv4t3MFH=u$}nbvc&gs$kMj$)7l*=XoLA7k$#^x-1}LjMR?IsPEuNcO%5C>}=}&-aB--zMdq68gNSJ2!!09S7CLui)EjAIh0T){|Q!lFDJ!a
05?!>JmUW4C~KC@B2!Y`Vc`$Fi=qwusEaKnW&|o3Zg05?S`Bq&>GuG;-ob6`xh}tb9qBAIEeI{55x^(YWWz<+SL-+aNka1|Jj{Stq33zl`--JsQ!o6Ts0o-J
%f_-9x^)v*{6w(C`tmpBV9177b=U<|!0@fDp5HZP{E$d5hnQVP;E$tz@PY^sU`I*V6eJO;rpcglB1I4Fn|TvMNPOHAe|2lPaNv5e`=1p`Nz9Yzb@dSeOKS~;
A#^%M!9N?GY`wx9gjfWQ5lNl_{fFb5{Hy~ke7!KxU{OBdM}M9W6?ioV6$Wh@t;zXSUxV2MQF3ImVBR#nl^0xgYS#jyJ=0TTyF`q6Lt#!ezu{P$_Vwwy{~{ZB
-v~LVY?uUB4dI;k_Y)<fgOw#5bNBT4`jZHg?IrA74OPBG`(F8OJU=3er`FPL+aKs7r~~PkP5U~(=+K(rynQoMZ69Y`hcCr6Qp-#{{D7}nQ;x*C@N5g(KsJ{2
wMeLKeL+{~zn?3Fr;t$l0n^2MrFq;N0=O^lRR|om&Xg%J$!vIRhK7!=JkJ~t5&*PNvBoU&EPAR}DB#|>Dp6WAcQ%P!huT^!BHi!`)a~@zN@Is(lSSoYq~0vN
yXmny+P=o{Rf>gTangpD*Z}ET3=4vp1^ayP(l>;tB!d+U8{qx_GoOdquh&X~irU>wEsf`+>e5o3tU|F5{u<G_7C=V3)or1K=8LIXlHFg*Eg|C6sF7Zx|Br94
=z1=YXQbrDOq~)w(u%yw$CMEvx1jg}YuGS+oH%!p-qvq6qp1?=wiP&c!8zU-#m)+D_7Gy(aGNm1Q*(uu%ga>&>5H%V`u-mVC(){OwZ4~B|5I8_MGn%GfnIk^
i``nF9S)B}VB5*UF3kInPoby2tuyyMa2!k25R<+N)a-??C|CiiO{h40Igxr5_J6WC6*X-ZX3<m>zKJTJ?S0O+U>C(7DdJE@qOZ+6m>vIB=YL@D%U+6eREJUf
{HF*z=A*LG;kv_h<-009GA-Fvt!@DLvLqrCK&kGIo*%w}M$>vW5Qn)Sy@GDRR!q6$?g^lb+(<Gomy`76qarzd1U{qbPC8^67GPZF1F`24ktOEpO#y<(mxYV1
AAU-RWz^w6E0~v+lOQt@r6E~EngdUv>^n#|$aeIS;J<X3CT8eFY76bcpJrQ10U*{oZi0w76wfRudW-DX?wq@VK-0)qP}yY)NUs4;C>gq^_T`Rh`p-De%I4#7
{LfkoO%JRpVtHT(i0=6(U9VCA3OyIVF!*T!uL@CIugK-Ag(Q0*^h3nf<;#Y#GiducWT&@tq2~w7y+CgW&k~iOj%ighE&}v@wC^z~V8E+lK;w%?DJhj_Z<RBx
faQrL<p7iRCC()H3SYD7Tv<hCy5$u`4bV_7HI%RKK0F>|y}1faV#gyE$NFQGaur)xtbMat_`yEWpiDXUFo%8Ne)~st*>(g-lysfejVJl_TX4X2^fB~a+CZxj
2FzoBO0^??==Z?(m6RwfQ|h9a@CKsFctPlpdfJtzD0Ug+Q=jv;0)t>%L)4y@&q>^7`FO~c{B@)H*`o$3wBEa*i#Rlxsv<7E;7W(p^66FSaysxvO(_(U??RqI
>WkQanz@Mh?s9C@_2>^DpQ#X&*706foq?$bPLOhLfM2lVoJ&g}t+Pfl=p*==4sQZ2Rpz!fuj+;esQVKuP^D#u8ab76T?|sxQ|)h%V<K?tJE>I=sMC5)<(@T>
u&={GOpy(5awjl)Gr-uaPqr+>)sA2*2~GR9CS_2Wt3*Y%C=g15#q+HIa=Ws^5A?Vx<h@O+;(SXro`fI#^ZLX8Ke0I^d1mbu{ii51-=xI@8yjqd?RPzgcIL81
`4X{ca{Xs6K(^{}*(?=PP2Sc9+qeKo=UWUv&MHjfeOOMvQM(fABc%745KMXTPXkKiS;3EiC(iptSe;bEPAU)4#}8fLLZS8r>L!gq4yKJ1I^<X>+y6=SO$zX8
p<Q7h$W<F?D{|VSHnfE9+?l>9Q;>b#mk)#Blx^U+&B+zJZ@+%44jKR>F#E<{ihlB1t=>Z9xmlVh!wgC%H*4zh$-9GNLM)yi5-qL(wY^NejCRL*1Wax*RfQ^X
=J{By%N3@`ziA1AO;`5oK=BQGpMcgLE}v?IP#k1_#={1i>&H)jaO^F>NLN%`>E+Q%D)6#|{l(mn`q2|f=cP*$_|>$DYypL!NQFsHi;eMC5|Wzs>p*Sp?+Uh>
yzA&})M#dtMb`YoB-A?Xb<)yi-uJcHEADB9L@-YB)UTI#-MmZk(wZ&kt4i#tXb`Na?kpWtRRU23)!(qkih>6(Cik>VXNl-u`3bc|e&tHwO=P`qxy>vCwJPsW
tDXy4bkTC-ivC{6fg;-gvVX76AqaViW>O=whGXh@IZZFT`S1AkL4XIF-fWCLJ|UC5MgNHNb?y#^(CXr;`B3tKcH~Y6eez#hU^sxiL~2lO0OS=I3tJozJU#dd
tPk57X9m~o=<hDD!aCBs$`}KVv({e^-Da}{D$@F~=>vxSvA6a1Tl4t<DPdwqF!x#9gG)8u6tK@3;feR`39GDkmCYmiO$H|~K!rcj3${L~+-@@^oej#u!J^+}
^2q(mOS;vl04*keZ(nJZon}DQgzXA=DN0&2E*4eoZxiR=Dr=8&*{WML#tQDlmk>_>F69q1{TY6VsRRqh#EPfQXSpA<Iv&gS@&-g?!4*g){SQaWg-mx*im09&
Ca1M8;S-m+6W)^@(pk--SozfDf?MtYVn6u5(n(X|)v8Kk3mmcjIQK;#zGk7MI0r}U+C2oXRR$xLEiSH9w6*L{6@xn;bW$qx{xt^eQEwb7TIg3-$<#9Nh(nyy
<*-#%50XbzB5vNxnL)(6K<nzRo+ePfFNS6~HELh*^TruY=2IVR4n<f&n;#U5!w3yK>9lKlssMQh)+UsEMg`0O$@l60r-&sKK20VYC^rKmuZZ}lFH3owJ9jFP
G^K?x(b;9N&q;p|IF1YSRLQG|85zx93ne}iV1riCHn-d_lX2G2ptihcmHAK~kwe2V#J|ud(KKhS5#XI`S|2&LQ0|s{Y-h7IT%fZsw(FImq)ZdCzG})!3FQ4y
A!OHIvPm*g0RdW7w+$_%fvoDv9LXW9J_r?qAcow%DC6E!2!J3r3=T1y9Cwgr^SHDT$Xi*|0Hxzg7z1c)#6q$!)A+Ma8(<cMjx(qXj`@d1r2b$rXF1Eu^Y8nn
MGB{$xcI<|v#_|Y=o2<iusxxsFMv3U0&Kh4k9ukuv0ul1RxhI%P4Z%a*xRmspA8`yIYcW(1ztUDn{}a-j{|PnJK??2iai*BAoZfXj^6-lDL=PSBeK(~23l56
ooL{CaX}2yk!euXiag(>_L`aQ#H_2(YE)7vA{n|5A-u?HSXL796_9WS_oI!2Ja`tU=8SuQyK$qRN^3yySX9@~&YcDK&dp_&4trsKN{V%eN#R@=luy%>Rc`l~
H|~;wC;=Q-a$om0pat8H?s(X9JLZ~4>IKcX5+6^E*LWa9_*E>sVw%DAQJy6&3t^cE3$(K~U5OW|9qro#pSjqhnK-7v&%jUq>S^b7<!)C<;o$-t7NIFPuk0ll
w?uIw(O)<6`95blXy_PLB(s5h5Le_|$oDpo=ycperYwhj@S<V%etJ{XIx|fs;%oA266Y@Ns9zi`4lUA$KGj5ztcIbjPa%Yb1DE*zM>BXBl#5UY$vD1oe$LWg
W6<r(1?_C&kxhCr8NbFf<mO%kfXg@dpwwsV(+-MX#Wlx(mMp8SyrF>Z43Wqj)SQEmcPtk*rU%$IA`)ux(@U!(M(J9>Z}Bo7AHH!MI{)S`#|?CZ^aM#1PEt0k
^V|TZ>;ITqf<mnJ_9}sEcSi~|=UD+FyOeU;z!=+vI7Io5pkuOAZEhQA8^_%&fpE6(IpQI@$~@Ku!I?#y*es=%f6=+3(8y=9Qmj(TM@k&Dwd{CQZ6HN$95vW^
ut~jqx11~kQ<BrCsH`-(0&*A*>CK&`IQmS<h8<^<s_6}pC;-ndQwVt~-b$Sgg=!zPG9(r|F>>H4HE5}Tmf;Tu2Z4-$*|Ak?#fTIo{@R-G@0Rc?rn5TCqa`1X
BNY4%)4gP4W+FzbZ5#9f!m0ZAy9X1eS1$DTf7|;o9;W~hc17-Q<<MLRm0U!=UBAED51*ueo6XUawU8nxzKPV*z6;-Us&d-*Wj;P=B?kJdzHP|?leorPh(PN_
Y@PXGiSo`^6mh8ETDiWb0~8aVeqs90@jy!nk1bYhsG*in+a3OHm7!iZn-;B8{FM4q6I15630>4~SmZN86nKC|;G~U5Jj8q6JrI-wrBU7)oq?fka<Iq=R??e(
n0!PXt8${(MNfcz{%ec}FTsWftz_hgw`y6n9F1#2sZ?C1Q!;@RS<~(H`_gE(D%w>WyK0q7MZ!hLy8?4bjkAW_P%poutknymI*0-F-Q=8t-^A^@A~}?ZmlDMz
4_epQKFE>M*?uD)vK=F3>G8Xh(Dz6wA4m1IF}*WCjr>TUoW@sqgi5DgvcT!|B?w81U(9#IJLvr7RS*Xs7>TOdQE7*)NB3H&6zYSvB>u3K0$%Hv(&6+<etLgO
Rx2ZBVrUo*;*tjuY{7f;q`k5y>%KD|dlQD}c+Ls8nmXSeZVC97|F3jJpKM`fu`z>$c}cXpgZM;m1?7|p>8fpCyy&YBG<2O5#I3zRb<7(&G~U^XS9*z{wL`kU
ajD4Fe^`eeBZ)lGCiN*FL#%%kQ{}^R#6bD9wJr(JAQ}o<QFL!WJ&>Fv5vJP)bl8GG$0&e>iE^lw$`XKR1*N%5(HZz+)^fAPH<NT%>}^L2EBd857%pC&^h3-X
0~4SJpkoPaAg9z4Ks?HmRPuU4WuYpfs_lHqVWmp8z7+tg%>y&~3LkOzYff|}1mde68-$zhwFCoI5==Crn}&c%!$CKa(=Q0G1I5<t=d1GymYp9O2Q-FMN*kHC
U%`mE|2Rd^rY9W)BE$|XN^4hN2=D0AwyBpsrg8^~oZ3DoULy0dIsb5O{A@HzP7R9Dn+KHe{PUNM_@5iMAXH=;#vTq#`dCaSh!@;)W<rDRVDhmIp(A)pQNZDa
%^dxgYodj{xx-M|%F|yuCm09gr*p2IX|S3@#P{oa(*dW^0d?$xeJ@(7=h~UDXd7e?fiuO?E#@yOFAywl$B=i@MzGi9(aF@wo&J5enQ}Lc!V!iCz|8RYg?%n>
wq3vr=GyxMj#h+c-WUBLRZ@HCW)nxQF($&xfEIe?Tw2D&yRBiKXXRmQQ`Ee65>LeQn~7HA<3OPoc|t|rppaJpvPi;Y>#`_Kj9j{&2!V`gyZ-bNy(4${$Oy+y
D8^R{XdoP&LyIXB&#aO)VE{1$lcivy{Lg|3)I>zTG3bdY?%T7J{s@G{TM9U32t=^{*pd?rXwGu3Xw^t<2*%L}X*a&+FIexTfI^2|hjAZu;PW_sR91JgaGWYd
1{~87>|0U)`);BS1y;QI$B%Hm<2e_uCzL9Ug@`|#BF$FV3}jaBMuwp~fYS;^>Tm*-lQxg->3k`$@ly4|-(_D?UxP{IMo)aB508?S)60Q*7t%=e9EfYYYMDqo
{a~`Cccfn;uT&L1>g^cE^bimLWDYydeKM?<wCUAb1?UF%FR<P@)0m)6CKb?$4r^r~|J3QJi-IBwYh20k>QXYY54vWOkeFy0E(9kVxmiu!p7XV4a9O5o{~8V^
1pOcGb#|*?mx_N5$oYf_fGog{7C;APm?8(luoZd)mI(ckR}0gZD0tsiS}g)hrx;5Oy*{oZeZX6{Zd{%=T%ZjPAK9qyG9djhA2p5-gPC{DI*U~w*8f8Nsk?^L
^Zyp3az{L4fsi*d!^nLORv&NAshi{t305~i;R(A_?cslyK|L|*xl5S(WV6;)PfvZ>!*n0FGVoX3Mue3A4;$B6jRaaB3`>L@7bI+Tu(bH_*u=Y@`lw;<qVnV!
+)s>tQBc*S=N^>9WDWh#Cxv8XTSUWPm9-1%aFjaHGL=)rZ`@rFvHwP+A{NAGZg)(TtlO@s#2Pl6Pb=N^dTX+KxE!KPl4aYS^A6Hk4tPma9-r}p9%)u`E`Y{c
1*Zr+{&4{VX$1U-%dx(gNqizG)%P!`T|yOsS5N%QM1-|$%<GEb(yL0oN}HX#7QSwJK^`l{GRQZ@j|T^$q)6t8n_C}x+(E11Co{_%|FUX{w23x000000Hs{fv
42@v`00FiB1={5qnpgCWvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/treino_gpu.py')

In [ ]:
# deteccao_r5.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/deteccao_r5.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;HbYhzFhzjh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSQedqEo0ysxn_N%4%ns$4kyqM%f{Q8hcU>P?Zi%CUFlOyx%q6
<KX*wM+ojI<~Y~xyJd>0y@RxmSTd(@rP|(<Z&AW9kOu<*`m$&O2jikfTq)qU_QHJ33#ExTKx0|5N}g)zl%2yPeM6db1rHNsI-mrdPrT-;d!y(=YrJcz+y*!w
Go?}tH3*U4f$lf5pLAQ&F~+woghcn(c!afk)qCLDb8w3g5a?TRrFTITtwYOmI7kOm^dyqD1h%wpB}ziG?iX+}!<z|ShRx$F9oG}Yu_U<*D&dLB3Ob{-qPIlX
kjKO*m<)%M4M+?iLhwhA39*{Vhe%n+K%W54*|oj}3Hbl}_5;D7PtG^<?QZ0q8UD5L;6}XUhq0QV1+6y6qpCr@)s}DBpi)F+C*RJo?xK~guj%O6aw{daAw%UF
1cjkme4KiN-~90;+?nuL5dYh~hCj1-h&5^GK6@#>lpOn_&(EF*yky2J7UG8!0ujcBaL4lm@|yCWaSY7?1!83Qsg`QdzH5#&A=qTw4C@!&7ZJ8wi~~-0m9PQ(
VaGx~*j!d<dn{Rby9i|DN|z7w`R<d$u!!y=EZys)#m{}<<olU!m89dixB$UdihD?XOLv`w6I#JYs;=n-`0IG7NiR~4Uxy)oGwkhx=(~@~@^BKf41M>hFYMBY
7#)^x3H>nVNl^4vFffg9WA_0vOCwz+o+|IZL@ckrM&GCW_VoR!b2x~;4fhYpF+~&-t{BS}c{QL?-|yUDZ#^ZS&X$mcI{A&paXwn@jNn5u{0h@7FVd7YK6GtN
u_?u^ikF)v>$&Xp<!+np(m`JQ-xb&uA-VU;;P_~3xUU|x*`BP!^VA3<6^gr7fhilp5112e^=FH#xicXTo@Ak~vX1d7;^3ry=b&`B^a5X@e1M6bo{G-)xE+h$
g_NH5S*ZxzkQOX6ZZ~&v1E*g!DEVuYKCEnP1Y(w2^vB77H!w)ga(fK}?`r?zl@X>5ARl>^@OUs9VqC(XbAW&I;j9yAji0MSE#M*<{Tlm^npSw=8W2WsEvElL
3y{EZj2!R1X+pmJ%xzbsC|8LUtHV!U_S(sd@z-{6Z!Gr!^Macl!GW6a_Hef+#vKU|UHGAxi3*{Edm}Ns%5;M2A=VFxnpQo?-GGZd*=-)I(dqHBPSGzq`FhAx
7_yXe9EAhZ`6JZj(c!rc9~{aH19TLCajB?l#lLnXO@yVK`u0<4Lz_H6GRR^lE4Wnx)kMVilUrE3kaBs=7Mo_v6a{Q&zT|nc<SZ~Usu^d~%S&9azTjrvcX+I;
E8klnQ$sMTNBR8yH>JS6DPc?~y|UzvO?WlAq;`TB2!myy9-W{aZuPUMU2o?`>C_GvAtWa|z6!<J{WU9|(Bzu&9Jpb}p=%ivlf0lsI}`G{dS~xqDYF!(IAH^)
V(>WOa58eB3G!B#HRC5XR~w0~JV;FhP=h=p!_+F|XC5ylV@e{xUx1y%K-OSzSW8$zzKLq$L3Roro~U{Z64BUuAL-dIA9_>g)>zvVTY|NkkUffJ0&BK1R$HFF
hRF?Tq0Ds%n}wDF`7ov2&uOR6uV$>QY&&U$aetI7&4{t-_5xXy0R^7;kt}1BmlmvUd~BBp;YX)PeV-_DWxaHL-?}PB%&kprWE4C7ari>d34cvjxuGoHxhBkd
OwfQ(8r;W(U}st=ZYwL#aU}*!h3ssnB2A7IRsx?lJ0BXLTQ07M58Irjizu;mWCe}h=^1q#Qetma-&smnzCx(D_Mg3<Dsf#eXn1-CeJi}v2}0&@+vO5GjL&RA
+Dik=sV%Cj5ObK<eC8t~{@I1;tYWrY=+{{vyUWZW%Y{n1ZgdPH=HfhpEX5j1KLY}2GaCx4{%QW8y?93GbCBe`^B2G>?_h}hZte=NDI5UT1WL8zr3tyyDfDB5
i*~0uS*)H4{}!?m_De-8VTO}h+X`nl<zknB8->nRa>0m7koOPFmG)iQGOR}K4I+T{0}~LYbTyvKb<nkhh_|DT5x8~L)~6W91xf=5uwP9C@dRiYhgp#$Q+RDq
*#U~<1pgIF#)SnGXu#8Ir0QK1ywfI!o3gm=DQ?fl8&X7cJ~CXuZn}NDg5?`kL2ncA>Ae8j^5z#ykKEs*XJD0&vE0kGa!|Sg1FDczah?EJ;q-XJGv~MrIdYV`
Kv@b`?!m#65!>AUiLbSGtZn}ML|=t2YD&__apg^fy^mco-^Ro*wtQ1>x`W$qxO7-<ig!`Yxy5=WIXbytVblGta?5O&AL41O>JBI<e|Gb>(@)l4c`!IrwfVk>
VoChtD#GNCKu|=3Kl<7NIhpvsFgwmaAv6@(4xD;iC-b@vdY7sxLsagY(G?Z1>b3Ba#ew4bh-a#a>VQw3HleG<F0l1$I??FlabFk23Qv7Edf^&?NZy45kM0&_
anSPMGayik^{nHF+@x0d_Fh+JG?Jon$lMCTLH_B6IY$pC<ZH9h>Lx`K=~Heg%;6RKj(V+l2k-I-^A-FlJV&vijo77+oM|BGeeFiwn!piur8is(*m7G{RexXf
Sz!t{?1B~h+XUvplgE1{Cv6*d%O8Swt_j+3HVDQeh*6p<q?btobbn=fE*l0v)8Y26S8kcT;x27HQJCcQhTs0sCR8;(htO=5^NNJjzr(2<0oa|C{SQNPAV|qm
n=ggA4_F5G5H3@e3=wnDI^koCt?2&EoRef|6lH8cb`c=aVF_Rn^e({4m%aB*3OP029Mk8VGkcn&6ouw-;x9<=%WdtLA5~#+Y})~4_O;A&m65oMsii4E$(W*9
HSPuNp2TeNB<QXDEVWAM&HXs6R=wVi{M9T)tiBN#y}3#oFw-gCq)u}4%t+*<_WVA6jowMh_3Q7?|B80rckD+t1$b!x(I@yFjEH6SwQJFxOmPYjF)6Hlti}-D
88FVEfQ6ar`b_QA@q5OxAn)clLjZ{d(6Z+sT3g>WIPlQcw2q3ReqRm|GulYPgV>P9aAWa`?mHbvv5S#K4xaQRJ#6X~2NUGeKS+<AGMODa!z@4&#$F0nkC`30
%qh~~hPodXn@Zmjk_)i}GT9%slw(Zg*pF{+&HilMgp@djeRAja<_Xb^Rt^=m&pxoFI@dy-Yxj-EW1k&|JJrpWxhcDtMw=Rmx=;G)mez4R6BAQL4V{|Qh2~@$
a>y&}Ghkk>NtYr7ws1K^XQj2q?-OF7s{HVy+EvLLgE^pJF(|YL(UC48#Ejg3HrA9D=1lmU;)$+QH%FQ%uX4dtjHhDuk;?@;EZ*uP!u1LP^4Yj6`<6$pJV&03
ty4h2G`^=95D6kM@&ClB$A$A~A~s09sM(ywroya0w59`X-l#lRc|@k9K&FIXXut*4{yPKMyR70p0_h@_X~?%%XAAzaR}_5BZ_L+p4zeyLECsGYSZhO49DC?(
`~j0X%{}B*crSw?N&{WQC*eBRUcSCJTyHTXWJ#W~s?Wh2yvP_20?kkXD8*eblU)h&$JnBO-7tbmY|1vJ%<EQ6;wf^$uW^KEMP)@)^FL?qF;<(aM34fJL+iO>
$3lpw`c+*jZ5-1%`D;3fBCZT({OL3SO5-z9u#tE5ut@p%p99(IC&ESsy3~;&i0fq*5Y*^c&5NqBPPPZh)<|xaAEv5us?9?2@>XM2P4nu;j0y3fNF-wS#-tBE
ZW7*5l%BVMW>qmhx~^kdJ(byJuU>W9XPk^18m2;*)&X59MF*S8cDM++(iZ3|Q-D7}F|=v^Ui6@SG!14Wp`^2J*?b}MptV#%=^U4qfB~I8C)yVpK=DGoWKx{H
`o$Q8=TviM8O{Oe9&!lp>k`5zZ!0uiPF!h2M=^BaxLyyBc^N7{6ph2v&!(t)IGp=(@ySUPw*!?oReurls`W1IPM!6LWNN!0(`vb0m!%I)H&-e5hv^S|4jhxx
)-vbaCduf&wN#NZEC@}G_^~?+ri!4!DzQI8&?#%&hgBZ@6bpDu)a)mu_+)5tKL=PReD)9{i{d?C$qwJm$&}S20G%xWh`<cTs<Oa7t9WugA7y5;$qn)@u+juU
mbZVC6KQ~8k`!>6_0Kl|5x_nEz>wD5OzgVvxnM^hwhY+G%jAr)UD!$}KP>{J(9(y(clHDFVsgrd*|`Om^&s6Ps!J?jeKE|!h!~rrfyox$@ybviAbt=j{^bA9
>$xfrlA=bu)453#raoO>bxG#0%JYsbiYncT7BR*;qIVy`hGyNJs_;PuMWZN4Hmr0Nb_#0rZ3)M0Y9m`?nFLvT0Q@qCk48v*H^GcpTXN|sLIT$NNj%9Nj~*#|
5Ui<|`eBV~6t<)IjBtz=bi6xKYyUv3T-0eC5Ymt<NcIg=jcBobD0NKKHi0j5GWuVa80{g)Worp|u>SF9i93n#w=uM+$^f>8Af{KbiPU>vL#tH|V`UOpo%fAn
$)>cFDE(Qc?X?sd4U&YG2I-VF2v2LtOlm7o#7Rm1$9+?5g_Nz_7+%(aj1o?u+@fe#6o9Qq1sX@beLO6?(Krx7J~)N)kFjkFlQ5%M1@6NPOzn`(PJ^AkFt+8H
VZ$C->w8s;*ZF@W%gPQYg@SA%rUsJ(nKnh}pQLx<cT^Jxoz4}<1P6&d;FDS`EzRYhqYn))Xtx(t!=qH@mq!eoHF%7oF+xW}XBrof4IFv0M%zFVmHRgfh7cB{
=;}MNI86Cb*ETu8xx44Ho3j*SehXt{#F%9>3MF(D=lF;_RA<=4G^mlI85O<tkKj%)Pz$c>+igX`bUSD&n7Hnq4WO|Z`gonNio)aHN69u3Az-fbZ+_&eI#7k-
n@~gMZ9HdI9ZZ8aoEfA#srBzH*)(r?YBlbMI*xY+WY%tLC8;yMg8ZFPd&siqo)a*@JL*^_pyP1;QDyB_K9j9u>U4zOcFj<Zq!OZbqFc6$TZ;53g*0my-GYOv
4=ck@v7^FL5oOVBa8P==*??r|Ol#dD8fp~$3;;~)n5}U}m$0q(hNo|?LbODw%wvsV(?C0JLFpM7fE}wp7<rBZ2`*KAEEG~hlpNTXFBx52JePnM=nAB+Fci~u
BWzf}ONe066^O*$)3CqK3t<?RhCIf?2!392AuzysrHGVdVpTQwn1WmAW^Hs?P{iZuv-HFrh5WGw<CtBeey5paD?lN@g~lGLf6`I+Cj_?H5=4^N5<-Fw3Az73
G=_~%F4WmhZaQ6IB~Z2J83$w-@~y~<wX@t^ad<GGG=E60rFaOy!BxZn2U$53_)t!}ZKatqYnMOG6@bOWONb|hZX4fFpJ<g6XRDeV<!v-n!{(f34U0t9s@1aY
>vD#xR54}N_}NAbaVnG%!Hh&cu8t2KXuY@1WziDb*Di|#O!2%PWKgyM9VTf;g-9w5?#jd<i|C-?Y(;pxp{~+#3Dxf%xdSycufNR9tj9c831G0{vyuy3oE|I>
as-ex49L^977f`9Wk&fT3hYntL~qSLW&WCzb&=_ccxrxt)00IJ7GumEmQt$e`FVu$(P%LwxE#~HE=#B1J2E@df(njBT%}YeHsd6gt!~9Kd2i@`JN91;>n7BX
Z=}ap4b{%_(jiwA%3qpz5n7}mTT#sO)-Wx9_L8F1-PF%&v=~m?$3nBo{MjSvm+DY3LPPz9I1_Gv%!v2m6e6n19HEeQB~UY4=M2Z%z}N9|)aVU4?h^52sCyDX
mWWNu6^1#IAoPE~%Gz6-aL-!F)x+TsM0pJg0cX)4l~#rzZe=kOpHK`#^an-1k7WXt;U)LQC1^`QQ(q?T3G97Yi+mEsIyr4#^2>Jt#KiMSOP1_#DhV``YUF=i
pazF&)pSODI@1zhjL&+@M!{&Eb;H-Rq>+XXOtkVM%q$|o<y4<0^nvO&<xHjDK|bx-6p3NbH{>o94%7{5fCrRtio_Zf5~4pOMTPcnMJ1-F(O05c77C>-%V>pH
fJO%qHKr<t(G<78g#SE?Zwb3jqmEvt3vYN>3yBH^I{?YMXIGDlw%6F?D`5&vJ12&T&Lt67#@@is(FJC{>FyJ#%UwNjVjRF&X|97A(|SqB6b*9VmUk1$ijmo4
ND8ud3KNOzxQMEX1B_CJ5RQSV8~V$HAHEMTIywTXoj(|1kX#?dY4K+7LL%KN`SSSD8I5k(WDG9AD_dqSC80-Bn5jRH#XQi+1gQRYlFAdBZ}=#t>p$R>4$Bma
-rNDM)*;@ym=Ed(cK$o{pYVt=pxGKHNlO5srE9Ai(g(4XV%&mPP{9cV6$I2(55a2+_*kYWfZ(8z8N*~DcTr1%fq>t^SmbtE{`Yhu)<q8=s5Z=4I(WhE#tqv=
p0~$5|M-WZB~GY)+~{%g;0g{Fi;rkvhFsI=(v`NwjMhS^cGp>~*3)M3Z`^JdM2%M;%T+lSps8TS`Sf_}iHV8_sLSVw3vChh>c!bjQzP|s>c9Sf#N~qGBQ&$)
7(S$TeAP{`bMNF`eryWC%l5;nmj*)&0r3~BgOtvymE4rmYxU>7`qmjZ=xeq=sW@ez5cHX{GGxnAvN2ix;_tcZ97@@S4G+UO#&J|Ydlf<sSk>gjg6xZmb}M>X
{QC%SGkwWWT;pT+6e076??BABet_HnXH85tPX1S6Y_@)`sFklm%9!}CUv`+bND?mz(B*`!@X-P}e{`b3QZhg2;UzqzOc;WYCUGbbwQ^%uzh#P?ZX~8hy{k!z
C^9KwWNu}tFOb%c6X+eNZ671gOJDz&-2Hil>a88LwVETm_yoGKi-9S*Cse<B6g;%5$$iUzH0_V0!sS}HLN(8SrfR2k)uZqWxl|rCPwZ{^beZ_TtzUe~KS1Ch
<^nSFxd6Ba_!gKrEOHeDTKWkoLkvJiS2VtZ{KL28(50>=Kc{Ofvn|ZoPVDUuwU_OB-iQGVyyM7WE1pFH<GiJOib{rkthWM94_jkd72kuDPRw~&{8e|#(AR$g
;HuaP_jE{=fPILB5~_>M9=$0TV5hU+Tzu6xCTzRws+RF7rZtH$GLAa<qSRk*7HO;^bfJjE6+@;gCWOw$XCJgdP?mQ_ZhYj2>&Eq<jO}=^braR4-rzJZ!bJkN
VKff|SrU`bBb(asStbw*x{5o@Z+3E`5t_{g&ylUB?3DgMiLW1$cC(P>d<xn8+W2u=H%_GF6B1vIU}87#vCnZY?h#2X!}BH%@THC5B!fo}Wd&z*jGu9<)(=$9
VwR>`46HZW+fcd9$@I!i+x>=}p*N3M`ldf2xrlguryq#N0fV(Fe|jTJ&V@vySDcU40l&a_J7jX1tTB%n<SQhu!Y4=$XAoRUi{Q;DNryKc0!>B3YwXLsYVRVP
K~D&&2Qd}UFs-J<gA}&^w?i(4R4p4;P{+%EWXvE-!XChR7o7B3Du;KJIi37`8}TN+=?1ZivFM|Fp4Rt{uGA#@S|F^j)9o7D&dCK5hU=f8BvDc_bi|{*m2)=G
ZRshoQ9r4|8F5wJo%~!UXnOVgk;sg}rEflZS1Kx^?^sQ<<}%sAsVh|D^t0V~^_Q1nZHd;(8Bd7q7dBz`F-o;lQzJ-O>30=r3qIO9J*+4dp-~IUPvbaOi4vCP
{a%e90B`!*zOAxX#G~RtWVw;1FmV7V!Gr#~Dds$@z+Ul5SwIQElE|aqHMoXvJXPrN%=jfpc13;y3<qUA{n~UOIBAr&Cm72Mw&8m;ivZ<q`mS)TC0KleH}Aob
5JiOr?PCl()n9SMI48tDyy%;al;$)x7?hc{_g|`*5l4u>vOC?J^y4D&nDMR$z=&RkXX@uky;7Nu{!HK_!^0W@wDG0<C?;?U{BIm~xTL8zF#KZ@;)kwP$bn`2
<F1(3c)_tp^NgC_l~|NH*Rq+m1#}tZuW9N@6qH4|APOIDXVC5Iw|kzUQwcM4?0~YjzVfw8Q8oBZGppOj+a+JerYGS<Z89E+4#Tg+RL`DJxHD-ZXSN7*FW0}Z
<Zqz?Yq%}7=_Xm=`szmZHh!oq>|bI#)@<-efpch0qY!rYyfI-oSPN5=SQhZiK;7c)8hdi}nI0*OwMYlR#RAWL<4D1#GNOza4;3H?x}ZsxIIaK<O9w#0?L8Rn
2smBSh&012jFy37K|~dgc~MlZ2_~Md9!!a%wz`({?&Y#@kbnD=9lK#bXQ(V$p2=Vbo9kGzb_2Rs1=bIjUJC{GnA<IDG1a`g__nvKqO=&!6aWn>e4*M5H0&Uv
&M$!577s@YvDui%q^<Y1fbJgnOMy8bL>7OARG~(SQH#iHZ6xD)qm#ZM?6@XCT^OfQJ{p__slu#{>J1-y`hnL*Dvxc=>;*`iS$6tA;cj;hx-FcV>XA>bVgU0$
l!z4vpxkuz@}B=h+C-+$kXUHfA&GvA&hjP`z?$jC|M5SYpQcQakJtTl`>DN5=#!3l#ef6X&>&H-4OZr7*>n<ZKsxYSFl?vHCE?J<gu~!kWi}W1FSI(t=_lW;
ov5+mAauB7Tp|J*uAJXnYhuCF?$JTECeF+2ctPS4X#1h$gyq!v?_C5|Avf?a3u3l6GWN`1Wq;iIJU<eZV0NUIF_F{~7tR;WjU5f){Vi<I0IEHADO06z`op~?
4{cj|_o2|I^Na2-XsQ!k(15}`WXV=wwdi*%)+OgVp6nukzb^RCLCR5bcTH=G+8iC-%pYEdPp-iEhgN$J5x>E%m6u@jbvLHsP_g*7_F@OnsaJw1kF|;okVQvd
j#-^h*4sfKL*bmv`fWJir9x0avRf-S#*M6{AAYLQGQp;wL)GEknPlAa4*yBQ^j~1U&aD0va$%L@ZwX=+AIyun)Eo%1KE3D$Z!*|sf6(`@V2Ki6LoBm8J1)B+
TLQLe9sv%>tsd>anc?y=$nI38JK8f499#w!$_$B_4vd6?BvG_A#%8$f(U;P`vJTgCc8uK+AI=%IraJTf%iUsg&QpN3UDi!1gm%js@;}eOgK|ns1u`nptGOV;
=c~NlJs|m%J#gko!#|bj2+=n8AM-g_fSoEEU&?lUsdf^7C|yD{`};%2>y(RCQ58KAH69S~+jT)EUCS87&lam?OnssagLU|Nb`{TEHB$|nS{{iWA=zubbwf3$
Vl(`@VTQM}&*e*NU8u`v0FJGkHd>jlWdG1A;-A64$c=JZY77TdLQZGT%s6Mk;3=|&T1s$IN;vS9meXuiJtXD%+v`@Q1|OdTpWM0bhM0k#!2d!g$g3sd)RQRC
QJ<3)()gH3vbZ}7{$um$nQ;Ewc;P}64J=8&GEet&Dz0h1jc|1PjO0YCd)7$yl|y#jmGB}wGa>D850X6|$9xX(FI58=CL?7qsCoq}9#F5N&fB5%7moRNaa5iL
eIg)DQ78jrz8O{ZH4Do@ucD>R?+{zRCR4(zkLC8Z)$>6gdh~WytWcfgno3w+*DaO`raW&sO|+<A^Rm`$xfqeICH%)4_+qKff_Yjh<-jUVkr0WlUhwOmD2bh!
4Kt69+xu!+N?t@<TZx#l=MZ^5-AxDxd2EdX$BEVPoK$H*a$^i{ev2o%yEX{*4$+?KuA0C&U~~;73DS1k@E1O-<dmH0epFF28`y(UTQ@;M+8!YjM+}^ha}iyC
eqTOLC}SRw$t7*&%~d)TkAJUEU^-h}zAOMhi_?6!9<-H^qFoZ!r4y5JoVplR#ll$ii(HG6$FbWiLwDkU%;_S*W~UIyL0=(EY@8R6IQzL}#c!Di(Rv3E9ilpn
Q$9)ZJ;JNC$4@EzSX-Ec7iY{3!mB^I?n~GixtJ0F<?dobYBaJiuiF=M<QqAM*c5#+*zX|3-lE_h{v8e2Xgl**9!4F*f=^Io-y{^j&_+gYurSK1<jnTDFwXCw
U!!F#=z)dT8?*sDpoiKVdLQ2nT%<iW6o>7Mv&JFnW89`vWy*yRRIp2@4E&@_V5f(VA(2p>t)auN;6@~V8GH>M*C|0Di-fkJ_O;oj%V3j;oOqbyf3&r4EYI7m
-9I8bqF0pxS2d72c&@Oi-P{R+G8ZxOUF8eCz}>t`jf;J{@swJNQMJ8Rd}{;gV2Gw%YV73`B0hBoEiH5iTk;$Ecu%BB49LclkcK+jizPR|_B0D+enSp1mF&=K
Mt+k@K4|y|TYWJ>J8==X^A9q*Y>~(>UbP7o4^(AG%{?af@4XGwInqGAve1@KMfIr{>riFedyr|eH%{)fisX*L_7|M;qMaGYt_z;<gFBXa)P!k*w*Txc_T9yj
-c|z5rt5*qziD?lDewIwK%eC?+94+J<gNloGOrInlr!V~tVPxn-D^iD`>R&CT-md?!4h*J+laZ8OA$94l2+U?Rra&cv<~;L8R$uqmKuLI1wWAymreQGe@qRH
DdS_sQl3q6Z0c53&fM<_6;s{7bBe^bHMsb+;C*~YLbT3Mc2=mgeQ^37wfOb_MC$|Ii3PwP#A$g9;A$ObD}8w%^um|lSkI>uA|F~$VM_oe3k;Y7;7y)me1ym-
%6`;jK?@BwDrnjTbNzdo)x6*5cUDGwJkQ1p`sXp*oLs+&tAn?4oIAr#C5mFV>ov`v6uW5zDhN?N*Cf;t(Nn1jSvBC1mCfE+waHYHpsbu~yMy@Ey-i8IRL&mO
XxR)Ic?*c6N_tP~x721xwx(Tt)EHpe&2{0fL@r)R8agJ%VEpg`-&Z6W-w`d{r3OklRi6GyQM+Znnx?df(hCl&lWo4==Qo^G2-%@vBEh(TE&^9}NQPeo8!3Bb
JmitD^7-a(TwozY<RCBvd0MAi*hIo6rN8ft(w}yeTZm(WM9{7R4_5=OF~hj^J=e4<W*Z^UqQ<>aFtcc#gjeI@<Ur4-_=THJV$q!FPCx8Uv38nJ9D?kI?_k5n
5x*LzaX{3yvjfiG3l_$m46DeglF7$zNUj1rI3>8fAxL14$<1X29H`Dn`B^@uiD>h#3qC4GEj*TVh~yWzE|QI2PWK<{>y`ag`W@wuIvk?nFkTZvMK%obX~!+P
?YP$|_#Fj$pvvIC>_yM^6{cH`e!s`$J&0f)=f%L?watrUr}h{thjdw(j7_8#lAjGM=c8`6%}>Jm5`=^Vw(siwtB$)X>uobagSCzA05c4_w&ubi2X*8_eOe>1
>r(w+^KFj1-l7@4D-x&@uZ01-ISPH5CQUyvxNUUALnlku@QTu##^+kJQa4FQ5{C@5P5xa_R0i?W-1NrgY(fn|A^yI<2qHDrS*tsw(P@*-gDGY7#3}D%r$J5d
RZsq&C$4EO@N*W4$lF0Eh`AW>tEmh<9Kb)b`MF7S6<v9O@Nq_S`9awu+b?}=%^m%skkV>a@SU-NaH|bHdxRc%pn1#B=9;#8Q44hEuF&fQ9ah_aP`Z8ga;-;m
=Rza)z2zL_`0?Jo7(IF1_{p(!pf&?x1G(4|H><CH8l1e&H`58BG9oFH0BEY@fsAqlwIA1}pBbIlb4xp77CHqR<eJ#3V&}F68q*n<$cO?K{_Sq@lhmCDk<zPJ
Wcj4%0D$DOkmW#Qkle!?Z>qp;J{=0?_Bk6oqQEXzUsVk1a&p;);m^3A>qWvS<ajaERK#E0#p5g^y+!qjDC`NplRJNvXopCkc3koLBvsmC<cIaJ{UjNPsw?9B
d(5^r_+(o7AJ%IvPinRVunvDCIodv1Y{_phz=SRr<kPCsvy<;z{10Gjz+04f7Aoz^R#|dHpEV+jui>VZ(trwroAySqXtpP9Q1Km*lxxsAY(%{a54&($m{)Vd
;}n7W_o5x7;x`E!x=c+U7NudP#r<tshs@-GHTqRTfYH@Bn(J^PqBaw@6`06+{1*2W{^!c}^r$kRi*S8(6MDM|d5iwNFG3Qau+X2TFP{47vDDlp0auT><7L&D
CK7N&Q0Jv>&IRQ!{>}mZ!RGy?PAcnLPxZ2ocNJWC!(@LmAfEQz3)a|wyAk?q=7hVUFDZVSlLh+=kngzIs^FxDWg5yQ&~|9pHw-`wLaU7lp}Axzt+c_wq4fD!
mfUr)f@vnCu&59H8-=S<_+_ct<ozk&6U%PiCef@GWd^kb>B0r<lBZy{ZALg>@nrCmkJK(R2z3h)3S{LCn4=!Ds8{u1h%TCRNC^cUDDYHwC64#=*)^_+TXw{i
J#F04I}qEsG?Kfct6l`@mpxvGfchvBKl8#8AoR{;qSDE`1&k}&fayr<o*-zTj>QH9-%kNK!rll4y<NBur`Si13l;kSpMxukNE~O}GJG9zaao;q>%%*QTW%-0
|2ObU_RA9SY+1Y&;G=B2A7uh6xH(o?O=}&&HUoaSo?EyH<L#y0rP&OI{R@_=q?hZxHtzye$0WtpxPFT^FHAhBSQ4Ph8*+jj&QrnFOwr1$z4sS#MYp9i;;6Mt
m`-ChFs1M=lHL4qtmoiaz3jSR2THB5MmZeb(}6gw`q^^TDp~sai0BKJk;ZJIHj0M@OJzDS?|lTvZNbw0z=o(F!jha1oDOe;`C@J_s1F{MqCPEp1E(5Q&-!CU
%&Y^6n2B<Nbr`%RtYJHzN~O{tljm!W?Yb;E2WIdPivLv4#{Z~!d%JEX6iBn^FGXDR&yWGNAyY9t7A1u)DnqPTkRUssr|o$jVbmJi&-$5is=J0T$a{6NIV%9t
=pzG@y#>!k^Sd=Wy=La_umgH&&&ci8La9BSmfwu<+Y<G(LFINoy+={VeIcetl0IjJ{piz)aGGVZg9B_~LSX)=nA3hlz%CD1-LWxyl%+1!6;s+*^4WI(tSwD~
)!lSaF`^dO{;=p0MgjD%ba)md5=(tGnCEO;Q?d`q`4Re0KC@xr_AS3;t&uk5_n;!Rd`YOa5H@Eg9C`~Y&C<x$dywmobeDONbH%}~FOU1oFMf*-BbIgQx8{x_
sp7ES=!0vujw*a%r5q;%tie9BN)V^ijs=2D1E)G|^AY1CLY8#2f?ShL1F7F=R@rGNe)@}Rl0FGo_dYzX&e~oHR&@{-Y?I`2iy<gnfC||3&X-Ip^96gjY>bYc
+D`^UWA>-lMO!*^GjnHx{rJE<5->NjeW@&6wrlxL8=v#ed0BkQzbMW91M_^i4=Jt_`mR&*+4ob<G~*RmUQ~2auY)(nd+J;8o5+Xgt+|(^89Qe!_0qMt9Gx@3
Yy`#PWJ)xy8xn6ECC{g!9Kbs?js)nn6>4jRsI#!fuFnW!iEs)?Qt_?`++Hq;ee(aPL2DL0>Z5_)5MuUIV3y@=KZ4*jF7VN|{5(9)^?%(*`7U-wJNrFnna;=u
EUK$DF;AO9doUeSD1|0KCGBOgW!E++fqvS?sxlCP8oC@+JwQrMGM7FY&|>UJy^$Pg3+DRI@N)R$Zj}JQI3HUt8`mlef)qByZgSin@F)QE#+4onnyG9d<ryHY
>~M*MvtmZ6EPvykIc0S*2>7K*s--7dvdI^7^}+o|yiG_v6h_H<dG4(Wo?gF`p4zTvY)|r1%OlnVL(mLuPDl(^+34(lJN{6MH4HKnz(6!d&^ny>CuO@o{Qd8e
Uh$hWiSl>p+&*?ab@XgGdnpF3y^gO)N54V+G<wv_`^0ntD@E4FM2dz;lL`Jp{?sbj%04niIMI)1Ul-T?fjeOj-1yp<q@zRCg_pTD8E|UUWyFNqxZ4wz9G`z-
dZ6t*1=TJ4#7fAECc)$d`~ev%ecYNDwAg<n{Fs``wKgT1g(dBNLzOSA6{R7TZZEYj?~AhS2kpu`tbS3{TdzrtvDMK=VU%j+5&8V&H0-v`k;l<-V1Cugeb%=|
#zNkOV)5c_s-pPLmtaLO&}m|9z6>M!T#bYzt%;+dnfc%I1gi}}kFwxW?M2dbAj=&HN8UNijW}+^lbimPu`<;4+Px2?&C&G)y37gX`LB($s2efB^Tl{~DS3}y
@2miejv(w^Q%kUChnuZV-+tqy#J$ho5aPJviPLvK+W252KS;Aak3+h<3r$@W8do9sZ%p3FtvMBNYrjAhRC=d0!CA*ofTyIa$?;_-<b{q)7!yf1wizTeAv0-N
=*tR>ZK!V*Nf<GFMMn+&FSm$qT}~LuQ<3OW_RE_;sIfFRaLiZ}n~2HjKZn)I&Q);L*Gc-VC&RBgSKK`1;T2=R{JXt&CL1ojiB;IOZFs<WZpNQD#JbL~nvs7H
g@s2Pu+&DAlF5gziO!rv{EfuE`rQ;pJjMV1M`QH!RRY9b=8dW$Rxoc5OSNoGl|BvRxs40q@99$58<3^tCRUF}3D5ZWyX^ay#>-11YfY(|a81O0PVuaBs>VmP
$zez=Glu^3A=sb=>AC&iEQ2~pBL_<@YAemcmA|Und0mh5I)$oOWT`dB^5fs#*gK{z*WqDjZP5h1f*p`RHWW}hNI*SS5D0tfPYz)vjyzG3wW3IV6?w&4@#Jr|
6-mfRc)%w9JM!_15+w5T$Y55&!wvqs3^Rg933DqjGVa%#OrIrxq${QejVV07`s9Krq&9Nx!_C0gIK)KU`e0(G><35qr5IAIpZi@Q`&ZZAqa29?Z2xYqUjvS5
N$wEdsuhXH4>0f}uQ6s^1@09d9J|&F!f)0zxYM06*r|H-g;If?VV1X)Eggk*e(Sb)KDh~ejvXE`V14_uv79k)pG#F|SApH51wN0TjJpPdUn5QyHSI2}Cntg^
QV6OUSIIKsQa7!ihdHATq7br}{|<>ti$P3{UJ_ipu(&>#`?=Vjy7NfE?nCa7>Ahp;jS66@!z?n-7k+<ZcUqD(I<By`rPEsB+#Wj#j%Z3FlV`tZs<UNN-v#;3
W)i=DNYwvn!B${C3>YUM0iG&dbJFj=@&jT7ZQ!?u`z%N2P$ae|A}b+?otTd+UvPCEi>)>Y>jk)!gFha1No&#u2Ac>O_jN@$RI}NJXzamyRrW1l9{yJkt1-ZO
V<%;PU{JV}0oJ;<Sdz!%46<+YYHBZikdDgA<xa>@Uu<s2aDC4M#lCC=m`2fiW+CZ)aKl4j!&!DNf12Fj-Rgw;a=_S~g>Jqf?+Ful$^%qQw@wYEu-56;1HUkw
V_ZFSr?rr8wPzS?3&Bn2WA7~*RZldvWsafdxH<E(?Mdr;^TA`>AII_v7ji)_C`v)@*NHGmaHj<U413}O#+WrlEC60W1Ui$*{M>Twuyn&TOa{~P5EpWhzZ8JB
88yAEC!TZ7^7zB={1RBz&%4h92*02%FuV_AEs0>0<8*5aI$)`sUnUylqnsmF>>I9$vF#~1fDvaal|F}>U#~2ITiEG!zwq!db^^s6(R(|=r>Ny=MzlMp5E@vj
myrNPac+}eI%0X#!@(ID*5flPjXLp6jwcEuqy1)^3w)?-)*uR9OiaRn@nB$+#l`SbXo%kS=n$OOAyCX9+mApt^p8Jr%heTlx(;9pcuG&&b@=6eAX$2wZs)Da
jI%s+Oeea+rg*gHW+KQyT5=g5(g~<p#0nwclhq~Mz=c~*DsEoyh&;QJqq9bf_Zm@;yhlyhDUw%%4fa0D)R}O`W~Am7%%bG@EnNt=kSisHl5EBYYX#?C|6TE1
d!4^CD{hrfVvF)*cSpU)oiKnA#EGxXH3A!bXvO#)iQL&lcH~ht$$^p4X~e`XgKn^3K)8L!4@nn{<3UYu4%>(TguUUZ9n?PW{<ix}b>&=M)#ril?ozrNtjNOA
IyD9u;GO{Uuz32I1sT})aG7{iYvqEH3F6pjamiOi8nXZk<KlS29^w;!k9_^-YOZ9gcAu)6NRTpRD|5>><s(FSX>J@V3sh3+&l56iYUT~p1bkJJeiR*xdjRG$
@@s5s=;=f6E(yW@JXFp(98gHT8qRVLBhKN8DG9bEyXO1en3X3`^1szMU|+`BvG?jKFSeZUfZ2`>?l|_i=4#HItM$zF%!*Ld(Nq8dGd6e-UbKtH#iN=(aYW;4
krln9j3KF%K?AQ^Bdyb={n8=^ferDRYPucU*r0-I6;WUJCt>?~an+$VCwVGw4WZixmZnU_Iq1upxYLW+MnphODME4TCxN(^NoQTTzxNuI1B;069&jP5jHnn(
9px{}O>yyx$i!Ov?QLaP`WT;~i><F*GKC3e)ToM6dAmcM<OiveS{XN?0BC{4hX6I$tgVCg^^20gJ@gZTyM=?nqVY@9n;-dCq+g8WiJM`Vb?RqYw+;@v9HA6Q
?4TYYH!~nkExV3~;R~f#haQy!V9YD?*^1D*n$(+#Pj^9P4|bht|C~|LmiHFzwpwfm?91=)ArZz+9yt8mQOHOq2lmPdl&jIDm+4a1lQ^GIA_gnjs`#|iEpc7t
Xv$lt;O@jFS(&$$toO(<L7GV-fFs8oZDYKgg^Qwz&xewpB-y4S*_$ly=7D7nyF+f14Cb@z+{0K7_gA{!V&uno^53%wE9Uh)(jvvGE9$<6I+f<V`v<o8k1yd$
9n!oc7SoRC>{W@#{sJ{rx*@)5)%K>;xP^9?8NBppjUbECQStaN|4fCvB#jNB#~w}V<X3a;LS#L*J557SbbXx<&$=2%r9p#>vXk>?*<iv1F-kx(tQBzRlAXXi
FaBB7DWxl*j_Bw2@YvZ;6UhXxYDe3PsO2q*V`a$Ewc?(6sAYjxerW~(({)8X3<#}6=FOd_?o^8{kofqo;MxoVWF1)uwY0_b+VE~t9=QYB<>E}KYWG#?-v@<k
R!*b=wte8`CQ3Tspt-Q*roB&kczCQFn%s>_w1>bNR0n<h(S!K@Yr{=&n8QS`FO>bT=Ev}^s2`(sAs2zMJ>asi>2WsqYIjI%)J$jiFZF5Itoq{*y{Lx1(@Kzk
Y(4IRGrj@u&4{~cB*iD)JA^&-(I!v`ONHJgvKVxyTta#&)T<@`<$!y^%`_mDF8zi&_PS&@Pt6^3EHZv_d25d%yysdaMcQ<SMjmsR*>g9u6g0@*s4BA`pjnNt
Ui5S1B7pLq(U7?9vJN`1+iv@sjSSWT1Bj0?<*g*KTwbQrh4*vfS)&v@<(eQan9pM}iM1Pw*10)D#s?2;Kq;&&sH;#X{-?1L^`v!~I^uFD-b<duETl`Vlel7D
>>$J0YDxOz<J?o|n~HjYdPrddGb^723!QqMrRcPsVA7f+I;3B}bXg7=u<R{pQCzC!gj__IN{kD>s`38*D#6+gdIrJJ$cp`fVwX{>pmiWMX8Wb0?8R;580@$A
ULgm|2Tx<?lz(8YA{FNy>fhspcrp-JQhpEf$K4tnuTn0FgI*z?;}t1wdsIX*kEbU%ek(ke)~OreS)bO5UsQf?oc{>9KGAKMV<=q=qren9-GKiH-)MMMrX~`p
$pzsVGAihZjs$jY>{a|G;+M}k+nch_wD9Evi#vTK;~brqFy4l;)P6tD6BMd!VM}f7ue#343%RyfV0{3&>_)H~Uy47X3Wz4e)D{&9d;_JV4U}VdC7g&;0;9G_
vg0s1AFhqxUA}VNyG1tg_LBH^HNy+Hn`S&ViyNvuQcNYZ{YnSVy|_r{VoS26t1d{kyeKG^E2mJUV0P;Rh6G?blH61y`HNdS@@)qJs5xD|`O-@6(h@uOJ9rOJ
iXPM80Qc-z<>(?{lu9L5h$o@Nmz38!B86;w_YVeYZ-S(#>}%q*38FJ~O2>>EF?w<NkyJ=OiocQ?@L!M#lsIFnqooqofappGb1~bxi#F8|22=hHTde_(`(wdL
t!ZIfr#bncl7HNTQ<%hMu#&1?NhK4DP2g?A#%OO1UzZ#yFWzpm7r~)YzlOY*C{QLyEq76JUdaWl-P5Z-B>8Y&4+|PUQ?w;i#A>XEQ3GL`3uIe|c@Bs0-zv$T
w+Yg8uXPxxPW5CNd+1|Liy#8*c)a+Z1c`ql1i-2Ks>HWl9&pcMp-OckLvs*6MMDMfn2Yelt9dgkd1GYA%gt_7J$u5yVLQGnNv`kh+(hqI`B!qy^}QBLNr^ps
1KgkmGkdM{{<vXRN)hlAGEGvYU_05q(}t@NM5yt{eU|L7k|<<ap7nFrP?S>>cikNQzxuhU6tg3cD`Mt&&Z0J!LlrA#pxgb|iP}H>yOVc0Go!HK9z3{yRtx`v
tB6v3L^&HI8`Ai=H;$eHl|PKWAZUKHsHF0|)Ft8H(z*0%<_EYYC1RL=F#;712km`#o47>xewG|F8VwqwvI!bxe{Jtk^0K;3)B~kA7q<hre2v~yPWycqp9Q6`
?}2b7it_XuDHm}nkYk9|oB3XzbOO>YJqzEyF<Mj6001VNImeD=B^?g)_>+lD#PUf)P%h%BTUaH6sLSQG3`=hr+h!I>+lwuTlCGOO6$^ejHYAxAHMlKqr=(<9
Dv5}I690gwJsMc7WS2P<SZmYNrdbhP>F@{GRO}&OjH&r@4ve)v;kj@_#yhwCxZ>}Q)h|!}?cl2^NsE%a0NKCeJzOyZ{Vsv={&f78yPk3AWF#-g+{ssaTgO&#
L0xv<V@-6jItnWXWv!s%Ps4QHzI2kbtj%FLd3iN$i#bZ4B=p~q4^6nUzlju16Un~6Vu=_>X4RDKjAF~<hZE?E10|UZocYG2R-FdpZrSw!{|~ce-)qL?T_h92
^*EyB5v9kJJ$Tcp3;8d(?8LW&nX!EPI!5o8C$9|FMio~YtM~esiY~<blSL;}E{tII4A9MIq()W=Opc))W5GM9y@{EvzwM2Y#xHI2Ae*1f-p1YYci5$?umAu6
ZFs%EoZTv300G)>z|jH#i`b&0vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/deteccao_r5.py')

In [ ]:
# listas/conferencia_r5.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/conferencia_r5.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~!G8h+P0Zf`9=dMrM9yQU)ur3Ekve`7t=II+0Y5_<jf6z7xUw6At9e6Jb$7d(tQ-TtZoYcEf~iawj%bNb2cUPrAqA
7&sKD!rUsxEb^f#PPJ}I?2z7Zo35%(yuqGHjS!kCgh5~QaIMws{uco{`c~CSxyyNBJGl{YvEM6~D3P-9e5!V|R41`GQ<mJEA*U`4Re-v78^$XwzWr_=K|3(0
Gn(H2IbO#+<CfmJ4MFJ#t?Sef@%e$L!UL%CX=;B`t5IUb7~>oqACHyn|7|j*^=fXuX3pY(*vQ29$D~$jy23;l*djp@@!>L!y^9LPZM%)AE1Bk)=g3j(CI1M_
gb1?P{Da`QQnHWUYJ3cdW6gX8)fD%@9K0I|YkgvGAVUmqrHGV9eFeW-BSkJ9v+sm3s{rFT9;8*;>No+e>g6nhJ=G}+8`P1Q(XQL&EUw%J3&fn*s1uSzN0=x>
Fn`$c_G^9vqFUBbSJ{ej`_cKcK2CsJf8kt_Jk*t+DNan4JPAMmJ~@T-CtVAL00E=}&jkPgPQjfnvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/conferencia_r5.json')

In [ ]:
# listas/divisao_r4_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;!SNenOy)f8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7v-DGDY8X~*_v0-{=i^@umI~GHRNPa@`5dWMqjsoLnzp;lUd7Ji_mA)I0;G(WU`yfBq{}ZaufwqIWAMf7Qh>!WAa=OY
+>LTd77ID0ipvQ2TQZ4Xg|G~xnjstD*T&Y(KBc<;S@<Qs9U7X@p5cDxn>~aJ<5*U+=Fmd<`SQL*sRepR9+$*n*W8psfRPrkCKdNIrhQ?HzMoA8xQ+5wl0@;2
?a9e{Pk{c=5n=P=Rnd-yNNouIVpA{xrho;Y_Cvi+FrdpgrblY;QH_I!Fj@khlcwf)S;bumsVm;TNmjwTB2w@g<G=RZL4xkI?vq$JsSO`oN2rhW7`QW9--?&f
qoYFghxI~O^%eM*;4v8fUxwjwl_s={Hxyh3<SsPjT|kNg+`kZisyYxF($x41=E$gzbcg!uRESu&#r|PNsqWhV6&X5KC^+0#bPK>poZc$y$ImBdRvV2HQs@@j
vH~>Js_-Iq-gVLoGDj8MSLkdlSQU7x@Te#bn;qjqaX2qN97**TirM%M-m`isj4JE%f1!N`HrS$lJj4>hb^{oRCuxSU?o(!Lo3nnj&abd)Q*MJzV6CyYbiA2*
6aC7g**#P)O*LXZV56cjBLVQk_-|<M;FZl(f<<5sxnI#7_&KN4T@-Gv<xJ=IA1@_F-5Ewfx!@<tw7768EX8bic0^14+D@o#qe2g=WM3liL@cwO=Kot=#(_X?
K;jWG?muZaj<0)WrP@}TWd|`a_&9okoi?6fJILF|VV|!PRKx-jke(@04542%HgIYm6NT<_oJ@E6VOF|8RnSOj{c3cAIfNQGjL_j^W-!Cm<vKpsv^(kn-N?zZ
0g&%#gZo0xK){6?yEfw|1OU+M3VoCD@}<g>=Vim}&Uh~E+jfE_4!JhYvP#es%>)n0iOg3To+3u@N>qhTS=d6>N_tD905GXAki7oWM6*2rbc!_xyTu)k*?72e
uJC(FF{K_1&c9C>!+7qEb0x?D)9d4S@Rd{yeN(5`*}sy4jf~z7q`pIb5~jFq2k)XPthqBHd5E5EJMu@{H=OvI2)2d!@gX&^VQTV{W%J8?uyw?$j-lS>kNqeZ
5z~~TJ_^#-%GEbSE>|6e7=#`i&2QOQB-$-kVC-bR*3rY{%_#9`0PwI7?LDWfhB;ucm$!zZi8L{o6Vus|RL_$4)SkuEiuGYVo{rVA*^9Gt+{J@8x}@ir4@u<J
YVO?<d+SMFx6;(n{M4jnIoZY_fbT1dsn;g`zVhf2xBJgW-{F9*^ZXCA4Rfr+Ax{9}FBUT_cRuNM=EwKs)C~+zM}vUz?J6FHh}A+k6o`i?6VYb6y$MyB=x#nq
^(#FA(BJkQcjHvIR&Rb8-*7{HgdtRiGj4Cvl2tc@4VdWj+9Q{X5zyI=6|LWgg%vepws!pH>Bi53gII+P1cE+A&lLL&-EPx^7xLNWF5hKw*pbs>EW4<2=G4{U
0;cefFv}mfRV)|{sx2S3s6ZK|EkwO0iyY9boi(HUbpXS7XgF(Nf8mZDugM_`S$V!`i}SUvUmC?gfNY-fpv4P0YEa)I*xaEg_XfRYTWQc0LTKII3o{<yLW=)W
p+hA^Bf{k!qv=^!H=+AAk3fW3xTvPZoL7M&e;K>o3!?O(P2!t+lZ}%+8T&1PvZ5+M#V^{wWse@eV2sKa_4zU`-AHhrK<#KJ+Ro_<qGMVx+dfVX=#}XeIQy_(
#;vK!xPqOL&5@UYcYPFSjT9{81_2{N=9bK#qU_81uB6KS8!xwBn}JkD&UgZ&@p3-tN6f6VN!IC&uKDSD{vtrWb%6$c#|58-kA1fSmuOQkF;)0gby~<alA2)b
mHlJF4*Da=KN&I*QhB;Iz>i!VzfMVR<QCrb$@WqJ7S8m94u*N4*!9Iajew)kEQ%XxWeDyw>j93!%XVti^-3Bm=@-y~dW08i`jJcDU{qo=@g=;|&HD~T1PjWX
w-8Nb9vuWCjHGiMU$!e5W!n>J+5A5?3zl2^6}mD>2VMTyCU86=I1b)c>e+#?@O>A_d{Sm}b-Ip^o0#JHEk}#cF49zS_#?vMkOUQv=Toc@NAs?I!)!Cc<tMV+
pC079HkrQ<+N*dlLr(|#2=|T;%4GIq_vVB-lPwj5Jgt4P!5dD{4Wogfkj7|R@-sv=as_AQjAWs$tMePRNt4c7U|M!p8ij#^5UP3EY?X<fGfsV^@_S3Wdohzg
$WL3Qe~-ySbU0R0$o0wFtzRkQnXH}N|1A6}Rz{t;^p%CJR7nEc6CuFzO%BEJcOT}}xTNKO0MP=8m#pqq{hK<rctxfctj*iBeRoCcb)M7=9#XHep9$<4Puh>`
l*eTK%q}*sHgpg{cc}KI)C$;Ji8>w}3pTM79szg4QIh)61A+cmZ4W_|%ElV3+0DZwV_hmP1?7(ce!)!;#(V|(_`{k8Pn{hmpIj_i1JX}qWehQZVD!_1z#brB
FKBqS^{HCmcui7#o5h&U=FdmyOlinPV|&|O#Rb#|1G1~x|98Yxj}5cg={4f;GS}}$P|1qfJlh0aXl>q&Qls<XR0P7kS9jgE0ag_lP*(O|M|5#PD$&3zPx76#
07wyADPTE7A!QpSep4dT=XkN+xO8&HeCQ4gmj~WeC6e@jNoaeWM~&<frj$Agr@*fsI*$J$#c2tlG@ddTDh)CcK<;KTwpQ10Q;@xP3CEo$(nBXghkG76fCis^
tvoYl;Fd0f5z+D6KK^znR_o%6_ktK`?zzX?-TAt`4I_Y)hVmRYA|!nZ8&)-F7%tx;8yKxl6aL{%*0LGPxWoTw+8Rg)RqKi6n5{;TFBOA0!O!afI0<!?pTEbG
v`+adqsi@pxE~c;|M*r*nvi_i(DqJZ$_4J1t%WO!EG>&`DTMUI)>E|W`xEMQREE5F`gg#uTpkH}=otb|mG)KAc>XvnW((i$kI5uv35e&p0f3sQ=M01Rh)y>X
glwOkj>vfn;*S@4apc<QM+E`v*b9d1X|?g*^LZT8-i^0lkcCvn{XBL5QJnpfjv+rX+iPAeyY)p>-&K{ZU@*u-TCk{7(X-In&`4a@avEC97Uwk(qH%E$RsQqx
(Zrr_8KxiJ?NDZg;66Fxkznka^6IQfg)B`RkKIfPsP-6Sj9<&{4|ggm*qtND%TJ|SV0%=<P{66XRKVuU0>)42L#OPcW5dYrmGKOqTVrP){52j@pbL=?f~Bl;
jnEd?kU$y}TJ!GZV11{GA_^UQ&0tUH?A=f(yh!>&;Jf;%?lX@cEQ}h>o9_EO<Y4Kv{qdf!Fu+^BS*G_9{eHSaRbP>jID{paUIeJGH~;6Lts&dnvYc5ZJ1i>+
kHsq`Mf7&-q$FU06#@RnK7H>NXwxSq`ww5j>r^pqrptn{z3cR=T@e~(YJSfA>Y2Wd+5D!j`?~bELYJegM?HX6cd@W+@ljmFTa+D=yfrb?@p;P-hd>~-vz!Lx
=sCz(+H>gsm*{yF2Tgg)%KKCN(jA;bzs}I7wwEdO`hPkrTmE)gV=s=G$xzKi?P4nvSXG%stz8r<IA+8Jd~CvU+F1^12_HE%Pw}&aK#zuzCU=B~D1&@-EZk_&
915@n+3$+~iuOQ0e-R*kAiyzX)DPYI(QNva8LTC88t$nuX{T!5XRXhygQlrD7=9}N!grkcvxJE`<GZ<dI3t!AP{v=VRMZwb(d_;1a)RfW1{8TzZ24R{UOOn7
{))a$w?=|xLr0Dcvw5&eFhu;Ge{Y80Op4mUlgXFT(Ez=`PyG)(DysaNCOm+Dj3~H&z^z2JYx~UzNyEb9J@pat$Wx+9&f&W_YE{gVvfL4fXgPGDij~oB{-^RO
dz}cv)h!Ear3gSZjV<q&or+<ys8zDNNUXHEx*v|zy=`4%vC=`epkAY#oAd38#@-UJ5OuTNEzxtPUc!J>TJ8&p+#$&!c8L2Zp3MBb&#fidt|&%}Z819(*3`ex
E1OdioPQGZWV!6(_o|lrWvv!e7IKqN0za06_I(kkPq!(v<{<%p-%Z`-c$`u2)ob;iwzqHtm=dc`xwaX~t%4K`XO|A77z=<J#zLniX4JR;=_WtLPLJK0))|ma
?u(1ynFo)=R=QX1uSs5h4MFppperx;_rXOKMUm*cTn#|WV%naHBt_obcU%bQ<C>^3Ic3LQLkbZvJEziMJi_#5madBKLOTAP`?}=-S7`W#cPhox>`9(k6czYf
-i75R8K?0QWf$ETywGKyII;~a*MdoucF#y4rP1R>g6o)@l!oOK%_OM<Msa6&Z_bS)hO9jwmfVEWrr-HVkIUbolu&Yf>We=?%M%3>!9H-MfEJ%BALJ%h-D?0q
yoy>V{j$pE+JxQ&#)-e}%V`sPgoKN19BE?k4ZVEm8xLxjm@fAqa77HJTPHy%8_IpJi95K}W<@$gQjQzyL8}?G5Q49H$LtI_O^$?-s&q|h1}Nfc95t9!sQ|GR
Jq>V=lhbA&$Tyc+@^qdxrXOpxsGW0+r({3N%%f<ZuZpk2|5lk&_*%0zfj}}2wnDtld2FGORJ!1Ir8-XK^|KLi<J;dVE|(<%5d>t5`-AK(Tgw1a0|~A%?Mvzo
Y1Q+HX<73M!s6jJ)lSYjz`ui64X_(bvgstcy2f<C1>Wn$^?8k&J8g?%^mwCy<vfNH*Zw~(1WGIFdZR<$T2L?TTtE19@kluRLG_}Z6bgEZ2C&JuXUN1%sUc>n
_wGmpcf(7O7pO&k=DnS-#WHx0?nwU*4QC_)BiL#s7ENlj&;Vr2h~&#4RK1Zs{PtAx%#7bOPS#{VQ-a7$E6DU?Z!myVv$<b6saJt*L-%4{!KKI8=Y}p_=n?pO
twq+F56>Y8TwQLi(&(~sJ3@cucJ;nwqhz8bSUI@iABi|-xaTh@uI40+N$W-FpL^}P?%~=91-~(W0BJKH;p9L(xGgNOTE0RxLCE)PTi|8;Y%?W<Lo>C>!%<6g
3M1HJk<Qy>`WZYBUmmMbH-igXn9W|`Y*!T8XnloZ4e>-?im!(+Tx2_HnX9X|(O*y_Ts+@J#q2cX#-93V)S2A97M!d*<oMXRD92|r3RAOn92b+Jb-sKWFSdtI
?UT&2AFfV;@BBqyL=-)jj+rO*Orj07Sl|66&M~{+-bH%eA8TFZ=xNHkJhx|Mw)ZA!aFVXRf{s<bvavbiqVPCP)Cvgrpv)@s_UG^3a|0+#uf6l|h%SqLkBg+f
*>tUEIz0Jad>iDc7NTJN=DhJ870!jpUqP`$A+saLoY8_rw?=NaH%ptjTY3RBH=dl4T3oP%tC7+^Ci$IguvrX<TC^M<X|znt=UzzU`kgp_8mkQ)0jR*!av<SL
w|8vuyEn%N@gKwKt}qO5dBg*B!kw*&ZYw7gQ2y^Da0%VZlG!?!TZH$AI_@q{bqy3p{YpMQz!@+%4kD%pD1+%ScWCt{YCz^=z*82xkGRXLK;o#UOkDxY+FD+O
;3R7X*wZIEzu-oPe?laq|FTzFH#n%CHvJ<F*6{&P(f>ljHoL3-@h{L@ZByUr*d0TC7Pb~O?9h*hH$KC4Ba%0usK@a1i9*(QdB4mo%UtFx<kQI(wGwXHgUbl!
6t4T@y#E3?XJ@;6u^x)vsWXO9aAAzy)%t(DthGTtRzPbrD&;G<e)@$r1vOx-fHa|__8vJ(q{^tqM2as}s)a1B`<?xzDK|zH2It2H>gdOPGF{|$P6Ai*A;dh@
glDcHg<<yy+zOLdYCfScOb4i|#+<`jR}`ii7I@<wA?T6Awuo*^=Mg6>`vHxx52t?2_z?kr+Vys&icDZ5=05L;s1V63$U39PwTp<S=tA1t4F3ysq|Bv+oeF9X
wuy9yk{|;Zhn_SBo^#j(7vM(tnzi>e`7j&XZ6#Vrh-YhfPA{_rU0|c(OK&jswL$kyEC9Nx=mLj&`C2&b#BL6`|NQE3J~M0BK1CPh$gL4#3a*E%)AyPX)&pmz
qsyRLynE)=1v&Fp5LDh%ZC`wz7{=1>ix;be-Lmu}T()O~>w-3ehcvj$!ad<;z+Vq5A=YwKV4x1Z&M{evIg`6Y<^185p3?Aq^rW<3T$Ydkr}t{fsX4^9#GKi>
<gEElgQM?NsOK7jYcOtkGI-&Wn%7hllNXnKNCcJ%>3$sZ9M#n2QOQ%2TXooZ4L_uMjenUNd>)Qof@tcmwz4ok{-;&-El4n^P0}8^L)wZaf%F*uIqS7>r6zL<
l@G{fR`t7UxVH$AoI=+q8y$oa^1$owCWu1xbdadZqhrMOJfksUnxNKV0HX7w7f&T-@!wi6Z&a2C<u`rI5#Xp}sLK_J8d~xlFtGWRY|x$M@gGy&_KK>evWo^B
HVA};vx|=^diqH=B@v0-W9-?uEZBh5i61+AFJFCr{i2~f?}prp+n2B77qbfoEtezbFVx!Dz~d2SAX)8LqQa||25)ji1vQc>@zz$n6vsi&l%#E?TgQy;N3lWL
@ipBf=)}#z_$51X>Ev_#2(kcTNnwjJ0N&M=l+Ifb{?=`V0J5JwER3}V?Jlo?-{_}SQI<8i&oW)OC{(cqUi_c1hF@L$<p6jNa)ThyM`;&I@-*M5IHJ%rMw<zJ
ZvOM?nB(Ul`%Yag?NkN*_#6=1)Mju%uOH2Z`#vUx>A?Gk5jQ~ZUt=F2PB!o7+<=&G3OO}--knxN;u36EV^HdC-U!B#j;kQIig!@i@WukUgYZUwxcl?wm`u84
UU(;k@_br0^F2UFt|hxPZv;C$YPCS4jod$cddv9+EdZ&a<f*6twl#alxXg3dE*Tqe5KG5APaBF6E*4nUu6qniH!t#;92pRdH5}Ua+O0P8sTV%a>-^Wy)v03e
$f^j?m~W8xtdD&7Erwwnk_erla43nK@gIqa8(GQrqN76j6wYW^#^CGP2jW&YRB>%{iwz9No?U9GP<<8-(!J>gpataJ@W7u#nvY~gFirFZ_}r(()+-q(QxG!$
O`W=t&(ToA6@h)3Id0DGY!77C)^UuezDwUl*hPngHzs0$BjtRJmBvISQyQjb4Hgq}=<)Qgqh~Sg8JSbQ=aTIqv>Gh32&HfED$HVSNDh$AB4mN1A;Ov%N4?Ln
I+JRd7=cVeX{=!>Gd4lVUF`Sm|FZ@%-`v+;?lK?sAt5eiLmOG0pvKQ?9X68GFXe>fq9vH(Kc1Uxb+>!xTZUcyz;^MsimXA}0^DNX49>VG=ng1$SkM`y;e^uD
xBU@X;cR#T<L-T+-EEJj6V}_sO_TG-k1zFSF>$dP*nR57Erq`Q@sgbY7~$(@IACE(8UEW7?HyUf02Uz@P*uaQG}|AsZGUg0N(I2UsI;B>;I6GpniVGIzWGJk
5=LZVLTS#Eb4Kz7c88ac`L=F%<DD-UT+IFAt@Z1vvEWRzjg;^Pfh{R#+a1N`bp0qhwbfoc?K0E*gBHJtPLaHUcgzH-OL{df!MKqSyfh{PYCM5&jxuYS*(sN1
u3fauj|pNrnyG4bde@3wk3~y$bCcda9P;emk5O~ih<h~XJrP9aCqZ;tq}XsIfhu=W-ZY7sD7eCo<Z!q6iBVS?${+2EYT=yYx4LJV_(>x$ik7Oy{iQ!q-2JoV
GhR%=$Ay|D`=4U&(OyHP1Cl#zmQgIC1y>69Uu#D`Z1yq{C)*BF3v#$J%CONpAb>!{ddHLOnY@*e0n^1>!Vx(XEna+G`k8n9r?rd&I2BnSq^ctf^DD=R?$qso
5*azFQyC?2-XQ+di{dBIuYs${y;3014{M6NKQ?<7Kkqc#OkbN~wB#q)3~Zn+DXbo{G5j~H7Yu9>FjB$#WUIZ!WP{p^#PEV&&?PE4AB%1+dyJY@`={`&=vB);
;P|e{*?LHMdAb!@Xg`4QMizC92eeyRIeJpQbOrTPU-vB@P>DPt6pn~Qcq+9{zg?I2m(_i_a>W{Au-v>j^X%R~A=FswdD)-T*Ll67m90}o1&J!v0+T9^n7J$J
PI;dy7jF_WVc6y%khVdGd-EtRv}`MA+38g7J*ukd<&LDf>gEmouk&tc4%X+@A*sqOs&Ek^JM`zTu}|z@@W1A041Es2<fX>ciAbq}$vIANh!{DNKh}5Z!D$#l
heYijl4dmtKH&+0=!FRC&Wvr}yZZ1CgY{~wU^+j+D6Z`JcV2s55!i-4QoP?#5EbjcwBwv^pic=v-T+VHqjxSkz6e=rxZ2Z2=681+CwlxuLcS1{@Ri@yyTh0i
0h{TmiotG3heI#bYv`8{B&8;y5l_!O#Y!ZK#OGgwAkDcc)Mq9ictt&-kG>(3zp88$Np?BfU3{Gxrczwn3s`b&+uH#D{g#Q`L7Izkvu30$Gh1q_6FWi1{VBj0
I)9F$0pop;E*w$UnwGs0W~=4Ea2?TH)X0>;#_aM<N}m{qU?9|;s_YJ;YeIXhiW6+0`97K0oF!*%m)%C6j6Wi#*{-?e2v^Y`sBnY(><<lAy7LhrPNM3Dek_ve
<D}U)^oi0{lf*Q!fd&|OD3Rih+Cr$LlBorJ4WajI!9#Cd*!jS$bPEWuh|;CFEheCD?94RQm#fOra61Di28gp)NG3!m4L;7cwvIPjaV`GeM6)%9ArdL;wU{<`
fTtbk^p3y^X;gb&`=CKN&6}1nF%oGhQJe!e@S0AtFV&ow`~#^=BI$74MY;tu)w^kXTJF8wkk*@q&Gxk+#}&xibw6#ek`bv1dkf<fB(cUuus}mawDmsa#RR;>
?mE&xm~vgKS1UVzu4==Wfv|>$O72Qkq-(?&!L>=RQzG+@5e@Q;^R3$fP_gsomLvC&OhxUge$PQ%WEFQbC76cXubhG@KhRr<S1(pA535nl)>~&yP7~ZrK!S>^
Xg+=vN6Rz1gy5dLeY7#kX(XAM4i{hM$s~VlRjRazH?FDTo3@3v+TZbU(o4Kjb&O&%mnE+ewf)>OVh%6IGP`z=;TCV|$UNm#=%2?=Nhi6jkZw}T)H1+lVP2Jd
UVz{$IlLf@JITyb$8s@O5fgsl`viWOrD(mH^L#PCp9<!TAYT;(iLB4<ESy_O$&h*0(9M^Uq*ZH#@CKDV)CC^<>Hio@cRtO<+IlU^sRW_`7RLIu0ca9qixAby
^k5;JbZ_@ys509TXg%bxNC?LM{h5!tNk?$uP+$%$6r$*_o?UKn1-Vt-659I<{-7r2ar-kRJ2(SRKqNOe*HCU9>hT1|FAdr96N_dOzXaZaxF;JSKu~7t?$xwh
(7a;p7q$Gkx;+H2>t&2D-`(izIkgn*T?>EkoWG|qQm-|Nvcy(7B5ek@Fp~OzgNV1ARy-c{8jZLXGF*+}hD%Hot2``%4Ccea_#)dW5sQr{!Ha-Zzs?^7KK6;l
dWehol470LM0j@RrXbjwPk3bVSvolXih)G9En89yRQD%NC__Wu?-lvLY~9?08RO)(C`LOdLGfF*E!d>|7D`%|vC`wq_Y^dni|_$Vr)YNs(=QojOuG#uX69B+
P;ye|wy1^uJ$V+0<=aS;6@d<CRGHxHUj&6#m=n65T4OCPwQjInY-1cTELk1%uEb@+VCSV2)sM?n4?u^@Z?I@mX4$4SmQA(2dm^AO0t5Kv^g8UaZ~{k%`RhaN
^f5tgh>f@SW(C3YbI?8`TXG<ku-TX`qx1}(RvUXkzSF-vPY)7dm$RigCLVbwsCXAFi_uzt{4!LVBv9i;)gi*<9eo61v+@uhm`?0GSV9Z-GKM)~2&#kWuH#XU
7UhXKEj|2bH(GZ5iP8i)jCg)wIwT^-u2v%NYu`hg8&9U(boa>EzfMSxTEFw_5(!Dhcg_0`kU(I)0qxU|uwS5ps`LF|7G4b^nI3<qIFqf3Bs-hY7^&CI=2e03
?4Q*P0&>I{huQd-W(LN;@n&>71Ci%p0cZj?bVhtgk`|R7i>cdmZS5k~UGV^}s#GF2>*njxA^2JD35zX*;0T@7nO@D&pkC0GuQA`cNF65SwOA5>0s|oP(mX1F
LEkb!l>MKmZKHF9{_Xo{WZQPw40t@X;9W5t$3!}vC_IAoBEd~OJnKQc#7F+DF--;*J-HuVJ##7hq>7Y~vd(rH{SyWf&+q@WtrA=kT#5|T$rT$U33GrO+H{^~
06}#H<&Qyh>CFoROlBwj%qX(1X4EdQ5r^Aq8h|Jyaj6}F$YU%dLCUvcT|ve_R%T3_rcQ}Tz=A+m`>xsa6)2+JpQHy{TjpaR8U(~)WFBc0I*o=A!;XQi$uYad
x5iBkLT&(f#)f|{`Cf7Mp-k>$W$od(V<H=EL<!%OlX3bw!n@HupsWxpOikkgp5ku3TiojNP_R%;GM7sO{&3+BOD=H3@1aEOygrL0<XKUSO5xXPypo){06Hol
f_zToXiKgE)g!)9&?cKtwre>?(#wawP}*cs(wwE|E;!?ODOZJ-wc=lPwy&i*Y^sgO28txCz625};!$)7Qr>#lk5I6m6tJXwj-!gz;a+M3k-cal-6I1*xjL9X
C@jnqSC9pGwURAU0m(fFd)2Y6sg;fEnW3>a#$%Lganw}LSB<dSUxogCYOidptrdn+8S1l18;<`o8{O-Ox#e<An{(gmhk<TQCsle)hJ+jacL(UeqZ)E+`(--$
md|(DDskx+=(Vnqdc9aGIjj5VYHUt---&KL*^zCPAqH;W88jbx5wn|N1Cq!?miv%@U8Tbe!i0MYG^e5Mt33gHKuPKclw=S_BjqSAxnTiwzM|{!pY`n|lzZ?G
8MCl7^PXIxdXeXac}z^9@Fzq2R<rLMPEkJwK)6u1mkAE|);K+vs)8GXCbZX0;s8sa?b<`MLTi_-VNCUNd3HS<&9JN-SY29`Wf3z5SM8{90;z-+Z;ec+ad9}0
;bk*{nx}J`GiW3NiSK59I~4CqNxt}>P*J?HDRsXX7ca$Ib2Sfw%{Bnmcd^XLNKRJe7B*f`a*-uky0P`^AwIL+y<BC4b~JK$P9nr=%vcw!zJRTEq<e%ri0r#_
V5X54OE_OL1LH9yqCk0dsP}3JSLh>tWe<L#Msy3vSED?JV`kzs&;(epyS27mwGb{be_Kgef9RNu_<Il6QiC>Tk2x|mesI1?c~v%+9?Zs<snNHLKLU)M`1oLt
zFJocZqeQ%A!AYVRj+)_4@HK)ImLZ}cdmck_rTvDnaP>B$Y4_idsn8WBkCo%m5>_^W{^*=+o6&xx4LHD8#)xa>!IsN5IhoxgvqmlqR~6VQp`U#g;i7*QPPqu
{>Gu08Ds8gQK!EPlY07B;Ss*N^SLABov0_hMOm_1XDDosQJ>>)o}oAiN>k)>^ChAVbVch2M4KRB6D^)`@(Fr@u!E?r$ma4HyxR*^HBbH}%?KEQcS%g2MM<Ck
BMe8J7K7`Uy!qz-^LAg6;KMy^pv7hCisY+$p?g>a%ICnZxr;U7Lrvtzur#&5zt{LjMIcNOs#H{;3@a3;&vm8Tf#^<YgH3{9kPem&!FshJL4Huon-W#tltyAJ
=HJ{b=w!V@Tg}C9bMzb2>5(%yC|IWvoad;T9M>V8FZhRNBy{h&?b3lVm&1<$QOr+3=#{z`LDjA9P$RfxP#iqsc)jSYFzo1a{~3|)xJkq$%|4jF2=p8e{=#oK
8ba=vx-*-^#W0GU@Vi>6Xp#X&`$C2(<y&4k-kL~5U=vGLlbG^>%iz3x*P+cAByjyD+8xHid7Suw2g=Bh7~;KJ8OvI`iq408&=Vc{4qQM<&|rBI3l(&F9^m%;
oH&#da)LoDEQHhampvd>rV%)Ja+aplXj?#|dT&emv#~xs<7WidSt_doe?pbLrr|VUm}GORs{5GPni&1DS?T)=c~ah$7k87JveII6(7&j{KttG7iBYqpZ;Y(m
Go*&B1bReru14u26x+wrOOGLoOF_F|b3T<sFT)kGkSh9~S0wkwES=7A%?1UOu`}UL#S(`S-khh=<Ji>BAyYnvQT44|r|8{h>0IWQA_L->Z+HyGPBMTB)o!K#
{1DeRfQ8Eat28cINQSjs#vJ(*l+zV?2I|o}qB(|fRw4>zFmXMm8%B{#BSQ>B!HDp`A4ggsL&)d7HtUM-q4=gwq;j(95pRF#ku|I3=jehcnBU<V&T040q@yt^
?AxqpO_<?g0i(?aDDP+ID_z36pCNi}v8Q=DExYL2gM)OG_$}Wmg`>P&dhlQSDs_Ozt-<ZdE{ZnfwP30Yo#$Pu*9NK|=w?vJ81B8)@ZO`LK&;G4$N`X*Ru{kR
ZB{=>%}yf?-6>#C)=_WV03-N}vFC~;_<lEJxda@hV9Sh$^9uR3BhP{8txb(XdsoLiz&@Yin>V^5I_(8ih$mWC5R)Lxrl^Y1ZQ2acL35oeuVIKjk@HH3`pv?A
=AH5mmOj1qj}6xzWmO4J|IJ?XDs)}r%Zo(}Y6XVIm`<oIIlC6ota+q}U>_AesLd}_fSw^XQku!0FS8ch!n^^&M|@XEE?$=1)>Fn(34PZCU6pr}(^4R+|4~fb
LYWtA75sVMcX;vh6qsND_0O*7SO*F+xSgOjMy6W|C<O~w_N_uRi3mAPr=kA4=>lg;rZ|>jt~7#@NT!*XNvBB6sDn}ktJ=Uul-ecNL=^9cZIw7(RITEt%*=A0
c)$oolouuA40Sia7p*dMjG06ivR&cj(W*#-skdgs8}^nM?23Ew$nIGKt#=b*-psASV#&{y$!;#z{`Sl?Kgv@lzTZuRD6Dt4yP0nW_M!EcYNy8uj{*BeQlm8@
!Vf=>M$63P&dX@VS(!MMjia4ei$)Mu>l>;ku!(zH!D&dIJ9H^(1~sr28yt2YBo=kfnL7QuL2G_t`?Gqe$GkSi0Ct6QWaucu4UUx%dfv1&v`=APrW?lWBB`5d
D|=X<L!lWeR{O=K6yf@nr<of?raatU<n8df<^$Cc^95W3X-t+}mv0Gab911BkXO4zmn4U%pP{ZqEQAAFaj|H1V|I9-(-0+oEsgNqStj^SB9L}dkVNDu&fI90
m~RS{{UzWD*VnA&Q0pc1eXF|XOhX%}@mq!gEOs3OR<%<WLcB4|GJ+v^v7eVC);*X(5KEqRLhyHSUOz88bXwiU$p~nCAQ3f=UEV?KRm*xN4ilLmn*z58Gi#Kh
=cL#HdO#j~K9{h*IxjD%^-B>HJ4^4aUwOk;Ae&!tFG_eZU<NQ|_w&nqK%}|B%P2;rdNU1<(b_b*{d6qGdlPjwm^F>Hasu4y`(<`><&rbD$Xjn`p+8MX+;D#+
jL%kcSF{`2QnL;mGx<AGx1kcnJZ}A*%IR!D@)XeR3qSm%xfEgB=slLmXWP3Bpk)*F;(e9>kV6n~jx{V-5o<Wj_ApFJI5+E{?X7m~QY&H@Q$%!XJCu<3DYo?W
jSp2HT7w@~;b+SO7j>qd$ir%TT^ues!TDlohui1bHaze*f%a9F(I6PvbZ|ex@$;h~0#T(#FfVV-hMo7z1r2O8yH-Mgya(pZANX^hy&Qq~EC`p*^grYxnI;^)
BZ`axwzI^Mf5ebVQQ_tIrtZ1AseHm#mb#fh<tdD;wM}(PG3n%cDA7-sW?VY>uG633+JhLWChC&CI527LhFpXzVS|@RfI~q#M!yNDkJP%NZ0@xgs%t!h*hu<(
F1;srf{v#>PC8Biuep&-WCNmVxWv!us4XoTDNaEEmc7d&-Nzg2`U!$rxR5bj8X|c@h<azmFuxBiNpJh2<ZP<v+|V_!jP3W@2~G5e3%1QYJIryXpa$d2vigTK
(rFk=j~UySE0G5l?hqjn|4j!-NcZ!ohpV3}C4!5MkvBLc9!byMcG%5sAXibNbPN5G){x*^u>oCk%vdK25*ZCRQJhT`ATHPMKR>|l`l2#^Kp;<)!K7QpyqKBp
#dJZ;-)VxJKUK}<f@0Y6iFoPi#41!1{Fwt`gZ@DGJk7FMG=)~RQ*m0t@^Y>w2}%5uCXi`jF+Z3L@<v;47>Hzn`Z|_(a`+IjMg!vXvTJLtm0EEtyV8FmEs`fa
c^VV*&hnn|>;twP#zn~6ml9JAwObC;(&1J!MPdqN{npvF?_H1H<J|#=?7q4cvxys>oX%cSN8Q6G=1}58oJ%GQb-#8vDAFqjUjkv8**#J!8Rztk3t5UAF$jAh
b1_j&T_*k>fw0CfoxK+BPd`Is^#}i^+ZgK?zV!iDk*4=E$^&%`u0f0=7xxTqWz}MZ!|??tH5*fJ&YF2Hs7ab@qsq={AlDQ{2KG^D<7IOq=q!iWs;QHwJRJx*
Sf~(lxfROtW4~-+Lr^|F=op`~%f(>XJS8W{YB-gQ(3W@fMTLKlt`xu0THRf%{P?y2F&p=Hkp5LGsDpIKWQwI6G7Ixazzd^K{xA1eO*OWymc3q>Ol>TZ-rmJW
Q{~b4%|X=}%jY#4Y$gDq1N0C`Lu^v;RN{CCQgUz}Ogi-W1pf*$a*bS~SE17kAE@FRu8pOfT&lk*o;c9)kC2?fFmvdCLdtHv9O6aVvemap<0BU*bKh>qfJJRf
{L8BPOt|D@_s9J1zBbO15G<LM&`l5K7TtxXlwK5E{z|<k0Q2N{91}py#Hk}E{TLMXy&5F?R^GMg4A@b2<>#`i0S;kh<ttB$SedY&Y?yccGTdMX>x_q=W@#XQ
QyuSk6-g?anm1*`Qvo(~*Q8x87!ffzxYCC=g#syOTtWqXzqTi2^Rz)L><aG52x5g$_R>qGz}Xms=w#P*;gTeMf?s~pA&_I%Vh9>t6N|{fo&JG$jhHp)L=y{=
+FSrlHA`qr+6g*h`{I;9;jtzx=sBiE$Z;OT#W!(|QY&@Q4U_TJskNL#5{Udpe&UA;hlc-Kp$}P4(IyI>uD_e+otJOZ+tuu6_D1Q_-o8ybY+8k{2-tA4hct9V
!{w!B3~nnE&OdprQuXa)$ntNOS<l?>{8u4PR>H%!LREK;sN3bNYpTX!Hzk+qc$6W-N*`$Wn)3=>%q22yO4Hyp3|Plt*cTQsm(fybn5$?j0vj<!k8PuXTc#I$
p_d-1t_d&|sMzqV&#-#p;&p5_G23rX&HuZjc<N)Cf;5l7xi8e{K_D(tx%R7lLOJjTkrHez)WgKtfPh|V+O1LkyBHc!@OVM`6$f~w1^DrBdw)LX`9vT!Iy(NC
@RguhYMeWcP%iB=d#415Mn%WwEm++7t_=*3@2@3Ff7Xvmvy<b{Zb;;dQ&&$^x#y5xCr5kSjXM@nV~0WG<8^bw1uV1bH(sXSU|~0GDw~^aG6%#y8{F4Dxn*tM
FsBU~g!VT=h+Kv<2>wYT)iym^!jYuXboWE{Jp1POSF!Q%CvL3a)d=2}vs$7n?X7Mf!DaezE<rj>GE36p254ia;}xjtJcJaVr??JTz7Uq?|KR=TkALbcSV*C#
`w~1>WCZ(Yve%v;+nZ3FcZJSmx|AavlvFgrxe(0hxLL{w<p5r~)oeBTMha2Awl-)iyLaCgz^t)4De*iI4m5m68Mf1}rdOcMSgmrsYLJ>GI`@cyU2y887T34|
Ah<PR;Ut0w$F4HJ@AEky4qyps>yq(v2d7(^IaMRNJ><7hj8n$KI`OV>f0@Fq$hdEHZ7dm~)cyA;oR{zpzS3-<5~q0RQss#n^3COojz)9fpypdiO?C>(ZYHft
=A@*V)rj8*pBUOrqG)-faVIcfC07f8-jDLE+zl3;sZ$WGc53Hf{WLe7fdaS+b;i1O0gQjtEG<vG!4EC=bE?$oL@HTP>nIEIaky~q!r*&eR$av{qJw|e9;6?0
Y5Xa-cyWPWZ_ZNs+`3Udl<HGI6~WE2Is^_>XN_(u3?W+|Tzadu?CYb*tix497@D%HuBA}}jX7@WiMV;0&e!z4<jBNfA2AGaPP6^nu3`OLK(G8zAQEk5lQG%;
Hhz_V9*G71)RP!bRKLa)Z9|VRi$5^uv1iL0`>l!EfCw{X#IY?0R8<MfP`<Qtn$@auiM^EE_0YTh+7Eu=dDb|pzLRIf-XH-?mYFBh;M|yeFWZ(&nh!>bj%4<s
B(cGMCa@-wsxBf3uA$2&#V7tWTFOk^%?F=gjYf^}QGzNi@w0#2K)foE{2<;~TCJ`r8gga7*#g*c8&IGXz<-^#r8c}wgq)VBF%;2a%-K1g(3222x+BiK#~Ck6
DED4tKS}+`%Ea4yd#Fv^FKlWC7!qKyF~+`#PTdw}ZPL3D>#7Z*dYv@a+VD9NYXod{0q=-@)gi&fD;y9z1#Ue76B~$-%0+6`IqAd)zbn;!isWC^;Y{f3Nc0H<
)IzG6Ua{-A>)X}Gt=j>@l>xsbWZfVg->bkP&&d0jGbUHDQw6F0!kr+RAPR8n-uPqlT|;4|na|y_loL*aBm*Ih#QC3nH_QiXi=ijrxc$xAi1{!;%pq@IQrmV!
3inLZeh&S)M;amVv0;B-+ref^Ff&%G_e4FYn@EO}gT?@Q+-NW_s}$b~M_mfAT^T$@z)q-e3dm*sxEV?kc}RqCcJCrmek7}_e?Y!(s|o4(@O@--^uRsVGsSzx
(1H{TVIxZ`Q5o`yL1&%TwS0Uc^BnjA=nxvg{I0K5VN>ecbRQQ%VHYPegYwB;*%$jl+HBwthgpD^1;eSF&imqp+D!R+BmeMee3!2W;+k0sA{pfL>Qx8yx#W7o
5;K96A-Mu<T@W7ft#zc0<V^JOukICAcP+i@wc3PLI4b6s73}KtRpA9jNuEa6e7S!f`0M=(fZSuDj8ntNEg@XCzN(-Jp`OeJanify@CNTaVzSHu5iDO<Y5$3K
6p*g=hq10mo3USju6F<Ev2^v;eE#>3y=B~&Gt&XmAzs3YNRf8Hxlx*f^|b3pfDu7);yHLA^3euKc(F~6_o@jeLyk=Omctqr7_!M;ml;bd)=3P@z~6>lt!M;C
4O!MB&Aoc*j+wj&FMuohagtQJBE=pjrPG9yz2<1#A6pM^jz8*3W$lbZlQ2y@9N)U~zA7?Np%jiKIiaoBC@DTuRqWb-`trqVfbKLn@JSrJ<}t$n!@6Ihn-~%b
uo#<DB5mxw3d%6>v<FmSm%3gLx>&H28AX=<uL5LEQSN(e{kM<m03a~BKgS<vs$}Cey_$on_;0?;qx!REvu+Q?0$&(C7Prz60eN%-#Uh;}&m*aAN_-O{LY9`Q
B>lXif_2F-_qq>Y0V3e-da3KTR&F0Ai;F!7FByHKNYTSQ!t@uLpFV$&*V}t6p266$oP<`n^mo{l2!{ZsEKJt~rr_zDh5s>6n5~TyTHW~bB^ql*v5)O@P(pRv
E!0ct^iI;53%>tXd7mAg6Q-5T`^jrvjF6GfPnZ`SLjnNSSE8N=uM;f*#f2~aO?Vcd&FgNx0S*h>f$YsVgYw4+c5KBM2du#$)S9CrKLNRW#`E8BS%6sNZ$rS5
&-oy^QAer?KkKI29FYlQg>;B)9x3(G^?k8IsR}p=gASEsvSoH>nx@7p-3RMQMboQvxb1<Td$=qXZraWJqM_(ofztg3{dp^4&&J7&4XFzVLMI~eLp*y&^ECkc
eh8};n|@ErBov&X;ow_PG$1!#K3vY=qI8d)Z5NMUr(BMme#Q)ee?{98Mt;Of)-G@9<{xmCZGB21tn>zI7XI5pZS{3vOW?3Fg}qCT(ej;1n?VsqKFc^xL?5jZ
?dylRhv6r0caUseKn;K^M#6L8s-XTXxc$4(ji5=7nNI-_32g~}()<XN!Npw!?ir6piX<63b7)FEi8<sS?>1MC5d=(0hILCw0?TUN1!;=X<nD+?ooKp~KT`_6
h9cr6_Nm*adaACHQ2}sAv9Lf}dA1S&00000&(l&U%<+_Q00FgY?wSbzBcN7rvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv')

In [ ]:
# listas/divisao_r4_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;GF3VkX-;W8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wR^Z9<JW#Oex0-F69)mXKZ=wHqQ@7FPl+n8r+<lOj6-HXU%@EHSZlR1OGgQ7u=ipFZ-N)3U5&haau$1*Tl>2pRLDAw
{;Bw9$?h^~2gF>EaCZD83@v(sEYd2RfM2)p26HSA>GCIV>TOZu1sCwkpN(l#wCg{gVCV|*Zq&<L78E6u*rpwFO*!P;KYJ&4JoxH%z<!ouRIA1zQj$cgwm2$}
!^axxSZnvF1vLG!#wl2ELCD39fq1<6D+PPA$5q^D1q3(1K#^gMaY?Ij(eJY6R|mViVStAu#Dp%Z2||T#n*H(<|3+9T)?{uJoOEjq!Z*Nl@7HMG@21;+9={nZ
;_Vv1nz%Ps>8$+*qUg0}0(0{i_)qC=J1!2mQ<i&NNHy#<5qnSz4t$7L{@jaQ`xWSZ*1h6r6~yxD><SfWk!?&1DbyUTz~H!U3YBDqH&)^=4PkBV4F(6m8n|l*
@|hsWTu5I2vF6-ROVhNiyFStv#C3P~lPW3~9VxBK%|>{u`xBp<9PNSuEk4bqQGy@9+hA^C$P{7e+@$p9hsS-3UB-YJ%YhTPO>7~$M?T~+8u~rqWRWCJc>#IL
8>)_y-nEVpdm^#7(*w>zNcvG+c(-YN<KI?URvjbT+9tJ{ij~(LSG9nN)66%gLgp+db4*l|zlfJjRJsltwj4&@&;uZjgV6OpX%(%R^({r7Icy|*Q)M*Es&ZLp
;1Z|)xfpJGsS#PuAL<(oq7O5kobD(~Y}+o+40UKw&qS`Q8QkldX>J_sY(w}aWas_g)%+e{A}D--mgjC%NvJi!aP^4y65{}84ndQxS?)ng_xL|wbT;+Bo;{}a
OjQ%@wD`SOTk=&7^Mi`9ytXF|kW<;VS5-8#CuxSne_D*!p0R^`oQwudB|>G=e_*MH#MCj8H+P?@X17f<Q#+CK9=mLVl>wQb9@62D(4|#eBpyc6`CtiQ7S|^m
Zt)^-41l_vD+2bEIv3;b{$XrrN2y-`iE@pP@W{h7=y#yJ4rJrb@u1;r6Y-DAMnvSyQ|s;xOXXicis)2kx1cK93WM)0wXyhVbq9|dLdF3N{F(;1mq1>}tcxyh
Nf>;lZHQNi7yLfMh$Hairf%-O+MNpLU6LceUpSaUCFi@N;2GIM<~}86NkH6DC$LtX{2%}JQJ17t#zHYlyd>#9qbhhby2lPP7SU$_E5LC2L}ADvI@5X8b}p}w
0yh6~u-=9rr$5foaZFW7*yo@ESJ@=j=s`MRhh!1B!J<iqgPzG|<eU~BLZBLCi2nLm;Q$3Va3(nGTP1yH+=}fYK#xC;lan2O*l#%o=UfPs4a#|X!$zbPS427l
EYL?yOq};KiygtXu4oO|m(be5>f9dtCHcpop8N(>eavOi_5$~N(thF&v|GtNn;iT7w;7}`AF`PamiO$mJa-rO-$_Y09U_TRA+V|nn|?@G9O89W*e%Cr;6J|~
K^`=<7R~NB>I~%RC#_N11Ru-4@zHbV7*jjNHw=~s0KMCN)M$p|)9f@B`{<`xs|PTO?H9i&OzJ2qp$nXxh_kKc8fmT09_sb0$^y4(&a3k2#Pk!`MCF{R<P|G0
hecIunLORT@}G_jGTd_@NdRXuNSLy7e`!jWB3E#p$H#_+XznN$s~|>_hZrPjqy5uja-Z$NYBD(n24-cu8yohTdaWaP!e$>``HQ@(!6ab)%kZ~f{3AWy*8&wg
DPn!w0G;C{wX|gZj?GGSK|5%o>}#&W{8G`+1<-yX%0{ZhT$CBL6G<r*rcGGIaP&lyS`c~8{1y=IpZR}iU(xS6M`x~2&Jxa{-CJQQ%V6ZV1bEZnw{adI?1sN%
IrbeB!q%fQ^br*wEwY8=#h5pc<-vOWFfGzJUonQP9OB101xfGZ_nlm2Zg46}l(zhXKp;jYB7jE~ZcU;oe5yEg!1<LtkWEx9-S&`*nW1JS69axEkaz2?q=F|E
{v)grB1%{x<bg@3Do+H1F~L}LatrHHH{bJ3xSW0t<vw%cKO;+>!DK43l7D@LG4vvFSV82+kxt<_J_GycC^nTDt*&h`aby2D0koS<13v$)UVZ{|XbGc|p5$zO
`{v)?^Fl7;y<=Es*RuQx<cEKZD{UFA*vT_dIN;vR8FJVnoB+Ie4m04y6F+OW*TRrqf$M_xs?#Soix9jPywBnjP}|CkeUwzoE6HQ2!x<gFnzEnODWu{WWukW$
3&vu3eL#}jbDWmunuP-#cdMwWu&r4Bv)}?eR8rglT#BLAYMG_{N<V4XFGxWRZUng*C1Q?5Yt*)7>y->7AzrpKF^_p~J%6V^IpnmB1ZPKMk`!exREJ`qkaW7L
uUg`x7mikNo(#?PZ_dl~fIuqQAPjh)h(}+KzQdfFR^`?E+&<kM7EX4K7eqghZ5kTO23-Vc3K%LK?xjo_!vd8s#es80!_omIbs5{Aa8-)be(8zo=b4`rslCZN
%dmfVZ$N1H6OiYhpD9Yt?fn(b#yX{R{C^7-VC!P*8n5RI%u*0(DUHKFTApYtg+LqR7+*(dv3iz~C`Sc!OV5nucaVS=rq4mg#REHde^plnwy{&amMQZ8M581j
a0<=xINesWo_g!uFy0JjX=ph%hgAR`Z_J%3b*z=J!j<LghBp)v!Yu`ih4=nsdLcSa=`cXg0>N<`Jr(`X1QyKX`a~{U83UcOX+1(bp17apHvVYBI9BL6d!KoY
7khr6$*>l!5yN0G9CF=|*28ti_4PGHcqA+XPUi}cM2LF3pP<5!%!%cF7Hc<0QozoG<d@n=4DZiK>6#07K*!jvpX<%}uVb)gUSO8=r>!V_i&=F{AH?{9MUDK|
pV3)o)kI^?WSSq6kg%i}uj;}?FCPnL=w98IgQakYD#tNk%!nSjH-_qVe}^%^R_pW<HmeKfD~2J|O_5`CqO_@%+H{hqQ5o*}%tbUf#R@fHUlF=eoEHGodg`&_
4&uhPk(|Rsp=S%O6x(ml>7cJv+j$vHVf!T{``yR<{)v`*OHzuPyrMKW>rje32E&I5jSJc&CQZ-WZh>+dV76~Q%K5tb9kew+U==$X=#0OQLw}?$UrGdZUc-+>
`$rWt|M8&2L~hB=tN2_&QxM;(Z#`Z|_YG>p7(WpE1b?g&YK3v9l;k*CXXo?uy&VsFp@vvt#Owwtc*0T2A=<k+Gu5Qz_OU<I(_TD(a}3s`DmD$*d8U6k6TYgg
`BeKKI5<C!25C6_Dy~E1U58tRBDhnlgpNOY_11xqK;fQQdpDSIJ>FZ0EW`6oPQ8hljl(Jr^+{aoz`;<w!kgGPV8$V8ML(z$_EWPPgW1=gK*#x~u+&f4*>Zkb
6wSVeyJG)ZT*ipe^FqJ#IW&B~W7gUmWQ%@Oz@wSyarv%o$#pix5*J9O6E~Vk7L?{lw<IFNO=KiKB6PBYPM`<Ma}1l7!40P-hcC+VHph<p=El+5YJ!SOHJmbI
lZESpPs&-f<PksSHr#ftCurZlB%uQ5g~~gJgLaP3X#xESW<eWc5|Rb!dYAa(BK)nw_oN-X0Mvuh4<Aj+73#p_itkDFhAu<$L!)Pqw7w7}U0eg$D@SZs2e>v;
{q$2LZ*|O1ELXTcm~|sLh?vTf2iEd0jaF3sGAS&JbOMx=sZ(2-y1Po`&<-ZElCa5Y+~%JNOeCJRYw1j=(l_+r56@c1Q~nASK_!n}MaZeDJ959A=Iu`&m6r+h
qhQS4jU-tiz1!o>5&eZ0zG5E2CD4U*NYEHjA82o1FU$!R@4H1EZ@Jc`NMgCFCz&)l#}!4Wz|%`D{yq(2k0CIDV*9@GPEC7LJuT-@9ekRWWLqL9O^Zm4xmTw^
bOcB_TmXqg6c>X~O+7fO%&v>!%uW3Cz<3-;D;j89<;R621Et3zWRK7;WnNS_)XA!ty^Zwn&x==<2)8%mr(!#7dts*7^$aqGtCPOOaWm;ptHs!zcrpQqC@1v^
#G>RDrd4T}xT?|>r4bs_cuL#^yIN?9_$%5*;)WD`EKOuX0RWEQj({hRb!nU+zkYHUp`IGWn7OGWHx$#jrd@dGpu;-5@72g`Ex+$!W4Ct*U*NJ*zo=xCS9o$U
Q5)8=mp0OKtiqUp^$)M^wY;DWu2T6#;V8ZP@QNxVB=+E1TyJd9m*9Jvk<wY)cw@1jr|g3}*BQ7Og9Fz#ywu_i!3<YLyS?()Ns=O>Ci0iM4M-ts3yeU(oTUuz
K4CE6oo_f3Cv~0&bNx$WH^5G7&U0MyaKG4&T@#$E5Tq)P9U+9tf`m*OJT8$y^82yaO3^8&+Y2f&pbel={_q|xw{Aa&zdPh*%vp?A3AF_<=KLaqJW&XeN|*Ud
<wbJCF4;ocRa(iJe+4F9txrTA8W(%t%ML$bK}Y)gFDx)e4i;L8`#adtN!jKA4L4t4S2-I#00FEU>bU{{nx(+|vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv')

In [ ]:
# listas/divisao_r4_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;4Fp$&|LsB8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wY>kRM9dNEoAvshJ9cQ8otbPUZ&_o%o&?Z>B{wbQgN{wP;Bsxs3myQ1bEj~v8<}<0zJzr9YohB6&doFyX{6r&knfRV
VdP7k?cDR1Bj-}IvNfkoVFwZ&=hLc|o_Cq>@~;jhD|Sl^1C~RA`R%bl0dgRTv{L!K_Y{HoAXJO;y-H}fZw=i!OlRO1F%qX1d{;lZ9$PmL$q{aoz)+<;=y(k0
)kc9L0u-;$U(Iy;2IoLPC%i)0nvaT%6QW)ve|!KOb<}=<pJ{;)<HwUao29_)shEX|5nn3iV*Fsmh|~mfFh3nYvYCAxVZDHg?>1>>Md@3~(`P}6_Ohj`<<Eb?
y<oka>@C%0cJj-NEYV_g4R_H&EP2}WERPpGwfss;U-*cLz{==^UR^*|-*;th7ip4l<_U-B;xSJQ(Ne!2d=r_WD47PLnLKUHtPCnAfJ5#gtNN#2ek*2We3}kE
!=glfmxoB0_B<ibMs3SyFefaLapi6fb$#-AEwe{*XOpIhaf;P+l8%F5#q;(>frMG5Ie6L|XMy48)`lMn;yl^IVAoaaIr}*QSE@yV78-9EP5qsvvdnGK8M%C(
c@P}&WpdCIm09X?lWV4UauFs^Jww}bKF3QgKiOWmosR3k{M@1m-~82K@S+r_CbcmjC05NIC0NDDhnbJ6$ibLh?vqYxCHRN}=h#=r{O~HEQ$qB?+C)|Bhq&L)
+vzfQG>wcFBtdWsbdhRNI6jbpP?6OYmveKjLPK0Pxp{=yO4VY6zJ$58hlBcNpn4c}6ons<H;oSJ$i}idb60>#2FX-VD9J9Tyj?B;sYw*nP5>Dt00Ha;hgkps
=^M~_vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv')

In [ ]:
# listas/externo_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5=d{0^)m;EG8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppbIM9w`%CkyKXG=WZ3duqfbQ1z_g`
N2W}#Ui%oQe%&YTzlOL=Ph%`|Zq2Dm3%W2wqX%~!;@OW`NchuDTF`BJCUr*XV$#Qcu;Z?5h8Ah#NLWAV<TMBJ1(15_b2}_{&uDS+uP@I%DZe9%%k*?sd9Mcc
?qGL`Xt@s42wKQO5(W(R2o0gr&g%~-{E|Gg#r;haP)P@^Iyc+UEJI2T*SKP~a6ZO~#(VQSLBe6(Og}Z;2iQW)IbJl~9aDhwjADehrDV!{3*Io`co;w8PY{{p
p6Q9Z6@DYfuP@xEihMHfFTwm~@-qDgyPlg$GjJ^Jz6*dmGM{zu7-gfQHs{J&`jbx2V6L!HCu(&YH8$UqsPoavKkF1B;^irZ<uF;4Mj<lFYB3eAbB~<9JbU#Z
5#MkEB~+p@rn(y})~uOVUJ9`0B0n-Hi&nVvyGT!sI)5l-AcuEEuLJn{OE><CFPd%k=*ApwXx}Qt;wgc1{pM;!*I5I!kr+8CoOBm;Er1?;RbACx4kW54sVU)S
ekbdESzz*sSOk{Q0DsQCQe{LF!&)?e&1hKr|2N-Adm%%I@sy3+z%utA;aU8X4KHs3GjR0Y$jPSHCM|Mcwmw2vg?Bu@LDj))(W6}z5lCQPE$Iap|0qfy!D0jt
1B02O{4b`6Oy6*49UtnWekNUa^TRc+F)R{pWUA$#qk)M*Arj4rUFJMH%RTXNL!0M`B`kW2r-fiMLgk%ewh3<DyDrtj6uYi@Z=@A7?4$wZ_`Dp!@?})RNB1ek
fV|da_BLQ%`l1O>KJlG**Tkn&+?3&{o$ZE*i{y!EO7cEN0U7mF_OTYuXssL9toqTIv~Nc$j2;HU%OG3xDEuCEu|v8n+uHX3^d)TGCCc_M_4IR~=R9t(;jSJe
9ODMlp6B8TlFy};{<VtrB<njKZE$R#fVqUzb1LMLbf>9Qap{UBqbF`*GXkRqKO;I9Q>`2L0%BR99w+!eUoM47+l?l;_@qFM!>~(Q=<*xfMgof0y40D)X=me$
KpmpY5EtpMt6v%s5XfEVv~S;FE)4OBMr4~(_@Z2ASjiUjsrjW^kNexn17Km@4O{<jn;$t&oUsoUtTTgSaK2sN_tsDbH1>&5Rq+9JjRpAO#f75^GJ|L6ta<=f
2}LoPw0!A<>jp6odc?~=wE$q-6a1{Ms9JDnbbOBfct)#qe$azl_@K^-CO5xGt{X40L-c4=s>_i$IZx|L$k1r)A+J*84}8%g8d>z&(a$(-gyO`rhSJ3v5DE_C
ne8Y)EfR5AS(Zn_t%?RsO?(Y0<OSbC-po4ak?_0y8ON$m4>Ax)=2bVp&UeBnu~7~?2h&;L1s`*SxJ=NZ1(4<*wE5e{DGGdLZ)U<L>}M?EWhhfIep{!|YTj2M
g>Y?jj16)+^ZB*CMi{C1R~W}t^a2s5ITwE3g^Z|2FwZP8-Er;kj;Ab=J-bpR!jD4%l6=m24gH0(qiY1=4P$9_Y6q2N)ro3RCD)dk7jzI+*On)kI4&PYQ%vTo
llgMAg)5nhRX})mIe7D$OWqyK_b(9Fq{X8xr)V!y2ptkY3i{j_@aWu>fJ3f!DZRiZUd;{|#M;pzG1FGdq4se)%Cj{>GVFZw+TCjD&pFYLok2syP`tSp#6#9^
__o1MD_ugIY4KNFY-Koxi-_T~Oj=g{OsE6G#!_owerrJqp1rCw@e0$)LH9r8COtHn9fq^ciRIu(Ru*8%zgHtIJNT@xf1+ihz+uZea5`Y;-omSe85A%QF5$xd
l>~BbrxPdMi7bJiZTp5F28&<N_C@>-ii*bw$#HK#XoSwnyWyz1F>b>L6jkmGvZ(7s);wmcpUJ1UL-%~;6&<o*aJ-O4&Xksa%!=9wW_d3VRKPVaqg(pk+7!J)
tz6dH7|MV?@vZYm?>!-8bPsv^S;{H%l)`Etrp8J~Uoqo2ad8VJ-(jan;OW%Pateoeq@qJb%%=onH0Ubd$Ji$mknJ;4PyNC>oSc{A7IA&ZXQnzK7@ut&jaZxP
0!rdl8Hey%a94aad;3Qo?ER|)4HZTwpVyiZvqz4^b|W;%#wy+W5&`Xn0=sr<P39(FgIs`-(ZZ9F@pvLr<=a@3e@SZ}&uX}16c5v}o5*_*QUTo6(Z@HhVv)^S
9e?shQbvrFf+}^{XP7B!B_OJBJUfjcp+=4Z%;HzDx2!ik({UT!L(;|etj!Q6!SM;KzU&qkaEQQaO);u^mrRs*dfj1OPAK1T&ht?eL{N4$*>5zSuUO+n0ekZI
5WH;}7+Wg#?TxjUeplWVcac3sCN=tZKFXy!e;=@xB=`L!M;IWiu`ETwMbg}w0&s?HGdl}kChNF#x7O?<(T3q7Rg_rJkiq9eI(f^%BdP@foM==H+bn#S-GdcO
{|F@_sSP@%ZDe1oSL8$H<Ey}DSqV|kY+rloC8w3(*#4k>&4h#Kht>h$2qe<?@lKWbCoX>;Cm}}5VG@G7veAKg`{A}uSW`xwNww){Nwf(5!1=H%@vpN+wee1>
BGu|3!UV-aTz2f#xg%}&Y5J17$_FRSIo!<kN`ufu;+vB5?FW&o!gIC1-4ts7D+b^I$;^>V75BpG3_vD6hf%LxQQ8Y%hHQ3_bRIgIrU<S<_S9Qk@CHvI(FiAQ
Duv3x^KR5$_VPLya=U>giagbnqk^e<^i;^5!ihVYBhQm*;7@NmxlBlY1(skvaZUuU!JX==)2C1f>A604wj)8oF`4*p^XelB6sIVV*1MhNyF_9af%rx-qZY=&
<X4MptlWynHfn(t<2MZbCmkvDQQbR$<>}KJzqly<@X|(cgy)$_nbgibIVM3^P^t*1(nHgpOoZ)i3sj3f5#md5h$6bHRdnH1bhVae0tul2t|D>#ppB&3gS&co
Ba!jJM^J}HniS}z&uQdP0+1ntu&p@nvbXpu`mKHRCS}4k(rgCx^NIh5K`fc?*acVlIp<W$Ov{FQVM)Frh`}NGpWg&L`Jty-N_I6bKQZg{eq7}K&X_N^BczOF
Znq793Twr<iF-)e*+lK^Gr!})ZffdAwcEOmMhnd{P)Cr~nf&)I@1&j|!XG|iYSj4p5f=HhTCXkI%g<^yD++I~iu3~41zX52@23Rby`Z?C&pWeB{f}B<YJe}T
Y}B0$-Aaz4=%FEYwusO#fw$cG+?7Jf${hZ$c+6n7SfxLs^B%UCcaIkHl(TAcCU?jUfFU?0?qLn?3E-=&^)N9pHnugrB1<q8_qw)2@{f4=HRI7pzB3d!knOnD
9Af_HMN>$|SJ@m3>|w~BR|`jVV!h*+SsUfDQCs3gDY9f(&8-Fv|M$+=uJatHB>w7ze9Aol;h9RV?f_W5qpfRf$o|yF+s&lnJ;yn{R5jIk>eeh0uBUT8(iSV{
Ahh^9MhrYXPm{Z>{%^wwtD=lxn3GC6FiRu6x|uQ`8GyVjOsGs;m2zqqC->!R%p2~%TF~WgHONPP>!+|zQE?@;wSwYq1fT4j3R)T|^O^ib;?jgA3^M_VV3s>f
8bqqep2P8k@=4H;Xu!4?6W?@c@Nv*Tuxx@Sqw>sy7vc(}2LC2Q35&~sbf@l7Hq{Hkyf3FzrHO<N&!?oZs4__n?6UZ;a$-sl1)U8m_k>3+A4Wf!DGJ37^?{&K
l1&~Zm){>spD)SlK|ZH^rPT1(61o&i3QoI&Yq0fk4|6=!lg{|HMIf8_aVuachpfQSqx{&Z38gEH>wEaHmE+ORQ<QDTR!(rqx_ekMDF08X<lUBqfVmiTklx4r
wPL;Ylrqp<{-YX~kTFoi<TO_k>#Or=LK>3fo?Ah=THpCAyfwp`nsK1!>~bWet67&J$PkpmWSq0AF?PV}!b8A;?uEm~b9T-TF`{z3V6J$nbvBJlJ}_~lF#1va
(oj}ZaGo2)6w}!M9Z&kF4%aMN;+}y0)RRzw%@i^t32hzSQFD{~G$t!<aCta!DIE3w-iu8d)_y~5q?S)=Ix}a==RH?j%LrKUX-Xg7ax~m&%Iw8dd2DI<eLXc>
*ahdE5YCIQ)hXM|<~b>P&Ca+ld_M+5rPpO{l2KqLl0a3r<id@Kw=V<<VQdW%2sL7T`Fk9+cn-H=ARDOUzk5&n=WZWNl;(3qMWw3~ejsqM7&hSZ8_HPSunJ*b
$`YWe&Yr2}1otO&obtyoQP!$y>czXW(`(k{ktQ)m0yELQop*{<RZv?%Qd&s%ZG~vq!mz^qA3i=4u<C(9pKTL1G0vK*+6?HeNDE0B_vIb!{o}_VVkrTLA19m}
DN0ye%sw7MIT04n9R9FMvO`t;P{F!MSctO31>d;B5llbk+=UWjs6d^n;CEzmJL|9+!QgMfw_1=$oQcl@E1qvab~4${tXP<;ku9X}VTMI?LrI>x{cUG1Rx|q6
PRc9fg_RxCk7ynOa~xm2w^4P}8+(_4G0VBjp{Mn>ypO!Xd`;4DTkd~Np!G~noZ2~HscO;PM3W*&i+=t)4n9L<>+Y3Wi)pJ%!vU8h&mdO|m&B)k1PM=z8Um3~
TE_qz?!A16L7@Jxv>szZa)M9BKQO2$6I<5%4f1t5^`c=~Zqs&vh`!;x*Kr;YMJmEd!b@gMET1mP&8+nW^KuW{mZL7yqZ<}DVS+Io?lPS7?usFOzWn!O=h6Za
R2N*ag{|WBiVPeBy9F&M8^qW-6&t)jdem=Km&$L7P46S<$9hXs;r5WIpiJJgm}KdwgKUnQjJI9U5Y4zejw-6|c<RFG4*fSm&q9=j9z0xVH)~_{h};>*JOQgD
D~f9v4Ok$rD53oSn5PdoZ)2gp=PPQp@HPx{mK)o=!dBf_8+5&A`8Z89UH~>s2SGFrSbScTt}S38A?+ts9tmEGWRqh|QRDY1e7MU7(EX?MVl-IV*zd1}FM#RS
Q?9x;ur`jPy~5#Fpi&j@gQu!<u+yA<j0AzMS?@Z@gRMcj7x+cjo!4-81i>m!B))EzX@{3n(P`bz;4!cgy4fU8mrZip=HXTQ1VS$B4Ep;~lm??-X>w@mBZfw9
-94p5EHFqJrjkJJ1I0W)mMOt(B`oY9+J815l+W6dt|y*il4RnkEr(O}x2U$aF{#eyxmfX8Cw7G}&mbj8su#EQq_S@^JXdY@Lsftl9B(1<Mfv>k+0{iC#9g`M
O11;zyvNHTqK1Y?A=orB^iD!qm#*|v+G65u>HQ%<cTDM*6#xmtBzoLCrtw>B;&s;f3%;dJLX3Och`M<>6_%pl8JOobOLB`vOfjJ=tws8~>gp|@COYj1arAV%
=tmfVRBc|od#fNov{RP8MWQh8yu%HWmRl{!u<8Ik9&YnWEIdC{o!3qiI_qIwb2;5BLRkS6-ZJBY4F|pya2|e6B8>Sfw12z=r|K1>Rm<q*C3Yrq&mw83<-=Wx
F)^Y~4WoWLXh>|_Jwy0erjLZlbEfKyF(tJgD6*e1b1br}QjIm$C!SLhXcvRu^FtIFDo5i@$U%;SZy+-n$N6BxNL=8@9=GAD=&*0OpkF84xy3{Xq4DCx<*6r3
2V+#l0tGcQBkC0Egtp%6OXa8s&}O&v9t=f5H($z*{=g;KX6+n)yb)9YJ{V;HAti9sHoW=8FGY0~bhrnU`-o?cB(9^!t4Gjp5C2luf<l}g_s@PG1I%DBnmwob
sqE462`ky8Eo4e{5t$#vR*1fn?z&WGOP_76gvsJ0N|OO3l|Z_FNyB&${E}l=CqY8LNfpgofXxu$2wxm&vfS1)TO0qwPoB`+$#~1;l72`inx2v?%Uj$ohRh<{
mFVQN<p%GaQ~c;!B|KMfw)JCl;=te!86kwj^!9$#|5$0Lm|*5x2sYn^f<tTE2Q4D#QyU2j_R!nXpE-tXa-sF>yD&fElZa!{x&+-CyiHQsryiMx++(sus#ZVS
x!5dl({q*J#bRc^-Dm7$om3AFqmcyWpv!?nTF(1Z9vz%ofy3Z9TLmTkHHz$2)urnTc3ISzuP#qn+U65O-pVukVf6=5h8z=p)#*yYe%;H@I@w4qoGjwpU>`BQ
OjX{ldX~ecd%+&Y#7UT#I42D^5s&~dFqQrbNhHS2w-Qwut0`T5UhztQdfElyAHj9<5Yk`mt%jy=)rnJ;pDVht8c>iyQyeSh4H+x*3C%>I3wZxgFR3w0Ke6r(
40NoS`wX4O9dz6I^3T}1g69E*EImBH)R^Vd+6|L#Qxe)cj1HDWzJzIO(|XDc<jN!KV8=9nnCRR&<Dt&(psD-6IV+sy7@&#)uw~93x-dE1q_E~9E!Y&sXe+X_
UQ`u|woMdI;Ymb%j2PjUxfsB%)drnKiS=tETU^hgGd&5$qEA#)+f}OcyD;1Mq4$h$v)QuB!nxc?W;Q%0LCPu>t|!V!9+j^(j?JPP`+|LAd--|1YH<GadV4qF
i+9ZRcdE2xk)<922a#5d%jS3H%R!J}@IiF?VBnP9?jVrj6Um2U5`zBa^eIR-H|mr2+0yhEqqCqNmva1{ajIKpI5kihvdf4R<o>G3YR$_B;-H};TXoRb3eHE_
$QMqeKF~Z*f9sjs;n4%A*;Nl7`ZkQtc9=YS>dgWJ9wk#tDCTPlSOJe5aw*&%DBZD6Ld#c#uTV(J|6pH~+yI}s3|kmj5I5%jdGp`RN9zU8h=Ygf6&WQxI|&i(
-feGN6W}AIgD(UE@wl{sbZOUpLF=cxa{CnIwip1UFT+1kX3#rM1Ubd8&D~hWmM7?0k6OrK*-|WD=w~iuPmrR@<uow@Q#*2MF3_9&@cz<?@@?qcG%UXSaL|c~
*WGcB@VQ=K^-Sc%7s#>*{z4MH3K;i_x?R?fqU@f=QH7CH(;1yeQ1!50R1EenwS0GC0J&t?LS>E3k#Td}eZrH<tKQ6T&}s-?4wXRAj5BdQ6a<}IP&-zAD(iXu
Lz)6o&?B(tYbJtTYDi!&>r1yJK?}#!!0+}cK~Jia)lHge0}K=NwNZ{nW*ZHC+@l3lB`t{QE9a3VN|9Yk;3q%+4r`Ahd7!N~2N0o+qqH*)5PbG|*%}A3H)h}r
i7yIf(h<i7dUlpmx;YSyMfLSHhBPezf7Hc=jKsf(2wF==NIX*Atkpp+UnY24s8Dr<$`$^_J}%&-dqmYw+jJso07`+5Xd1TCX(5%{wfA|8;g+~0bD_mJ4_gCP
7D2W1+(zBkCF0i4!AT%hqh_gs2d7m5(IK=ukaYbvv7zDf$G<&AcVn5oGu4GGo%C%1p5%-q%%HlAW#9ugVviNZTY=YouOg9`x3*UD5UFp~a}@=jxop?@njpnk
4~QrmClC+===mdVV;~9K%gal8cT)l~CwWlM*UI_zIWl=WgCx{5(_(N;r)TZUm!;fT6nb0fjEMs-p3#EDjFT|;8dHxq$KPq1JHN@P{K!%or-@jtLOF2)3=<$!
kbC;?1|W+6SRy}s<4$d5X_=2DGY*OX@&x|qbt$v&MD8uEF=-(A>>~3~VIZtFawZRJM~g%Ur@G@eD3El_BpbXjJIbobz=s_THt5R>W)}WcwdR;0FUBTK2X{rp
|MG^twUH`qwy_qjw1ith92f>1AagKH&;3rL=1Jf<J3;W`#wDHyVL7#4x*pKRKIZ6p2jk(h-t&;iYdAr*FvjF|Jf;58aj}Quk4X-fDo2*BLnCzY9ZjK*b4a3D
-Mb3NuX|joQ(#Xo;yGGCQ_Y5ZxUB=)RxU>5JOHS^3=gOE6(CJ~3DCQ=IH)qKn;`yuh=G-UWIg?Qp+E<pNT&L68ghpm_7FP%i_X(^q8x{6$R~=FXRfs_On1{4
3HzTA@o0`%wVA7pEa|1?l4pm)-uG}8X0H?FqXS5jrQ-WV#fecWE+HNe*jVIGcji;hJvt@OacA9TZgm-dbzgrjE4g$JEAhZ9X{i5m!iZ!8Jz7?*fhuKcdV1L!
HDRh;*tomlY^2@4Vd^+N%Ee%&Zgzk^>7Vq-TBhZGj9C3ocfD^wkwp6%ELmU^&ndaeoq`VO*VLH=@qZSBs~Z6hG*JRP+oc4-m=TuD_@};R%MUdphfSY;(Kdfy
<tTc~xu*oKS>5KIi~5hfObz0rU7J-9)A0{-@;nrwu89D<=AoXppGtRhL#i%-Hxej8XI<YvV>^ZYg)6rrhKBg~@|rwqRSf`twlrg0+QqyPgmZj}^POkmJC8DK
M@7sPz2@-hpthCidIK5+!c14IKvY327;~sQ=g`}Vf!sc8w+G0MS{pT!fC}5jkcZ`3Hs9$il?)h9stVF3#;I)!S^(t`mnl}OavV5=FCxD%WQ>Zv(v}r@(hg3}
5ZIjfFGWLJk?>;X7WX3}oc-+aL9I}w+!l|@ziMtrV3k5)e<&0911Z$k<Z4mbfGYEB^1A}rzStL~hTYU2G_Oyk<&nDO+S6-OEEMj!ms2o!;AEkr6AO-Gu<2#v
OG;7$Fty=W_X_x<yod=NY6xA&HwI~S04zN)_wG^b2C#Xu{E9jx4xEyNM@g>X-n3*M2{Z5A5^DH}w@1tQGHVFX-jq`%)5%-k?+KX1!k03YeHWm^LeAydFH_G9
xXSmKWHVJ}_mDjHck+{WsmedEg5xI?CaD-PX<yTURZLAkYl7PD%-5(I-Kzp0-hmeyJqz%w36mnQbaxZZTOf+1lmyy4ooCg)tC4{)2G)kTnaUhK*dY<1W>llu
rSPQmD8WOS*C70OJ7N5D0NqOs9HX>zo%QoCTR)zMH$){6F-{Uhgb_yWK@-S%)j<xjlui|HTdr?$m|JgM$8PlrBcuNu+R`|A*yogzB57_OF`6*M5XKvQX-WtW
I(8cPW&~$r{qg?qraC$0sA76xmCsd0J#vzK=ESRQ@3we$1uID=@9*@Zz$gj;hI@~inA5Q2fUu!qx7)aL2@ulZ;!?d(MqZlFY!>((R4d5xth+pplalxer|UST
pNY~C{n(L&r!hU*$mM02NEUQo6u?^#q2^dS{1c2ru<u`UsA*6?r+&sY5;l{$r(2zbV+(<dX6QceitnYyimRQ93>>h;b$E7>xjYmB6NcbSS?SZ)fTIt8fdAdR
8od^ewFvX1rRSx)xtmoJ;4#2tALS>;d3x6fRlUmlLvltat@%NeT_vgJ@XS^VD|*7Eq1}BC`G*MeFxA)~-Vd@yxF2vK%?)<<2KX4yM1FFNh0Am=sX5j@mdbWM
;h8eL8Ps6ZN%~W&<(&MKffj4beFRs~!PuGAJmQwVse^;yu7w9LLl-|2t1eqjL+`eRrG%;60xDonVd3r97Omcxe#_jbZr5`ifRZRhfQKiVZ-%|ik%l!M|J11e
wIQYj$@mRh7`GvwHv=sk)O1Xi5Ilq)*mF5+Qe2|Jom$${u8B;BVCl4oeTTI;G3pc{jP2dLP0c-(I$JZf$R_xO5&_B@2Lb=(^yh+SlcR5vR6*%nkPrCo!x$?V
l5?hri?}qg$ou3?mjPDJ%K#Og^+DNSoO%TD5BDn57CYHXK;sdqFSGmFjWGqeeZeVt4RCS+N7i{!%K_$(A7$rrv7I&nBV4pr-eBp7>;YEeCa<J#CShhIo0%*x
ZhBLYom*dboIy73g-49@7_uB@8j>Qq8^}f_T{pItK6o<ffkj>_q#o56s#J(xK?5Jp*)F8jDPzc*?XdRV=X)S(9krX`+^Gy+X){EK3f>-w_6X+TZ~9j$bbTWY
$P8?MtMg|pxh10h^#_z>M+ZJUYyU)Go^$uIbo&zA)M=0$_qgU_A})dux;$<J+q|~s8oyx5p^-(sNoHybRCo3}WD?MmUzEvTmKeemLozzCER@a(1~25b;rL(6
@YF}2kjG8zKxaOlTQ<i|WkX<KP6o}RQzZiBXckjL=vz!U{MV<v?@%zF({w9DKw9Nt{`zM|xDpC=ZC_lz>nB4l@Z9XWYwDrs7cf`6B64|>%Dh}Aq|ayBejQa%
k-W;d4p`+n{xS#S-cCM06}C=L8&)+tvFM%BPIUQ5<e{CWNL?i{3Q<OlCW|!%%v<-rmY)@WH^z2nNHvjXlxEi-k8G6i#X$b=BogsZpBx<Ks8$P)FaocC$0F5?
8(a_&)O|zy+#MpIyd|uj1DT#yoK`NqqXV&Fk{Db3I4FlT3NR2wo<a@$C}lG-YQ>o-ucUJT5kh%2sNQE}C*MyDS3Dj-LHzSczjfjf^U}&8(q~t5OJR&=Gn6Kg
%sPOCEIs^+Xc>u$QFP1jZF{bV{!kSU{I<7Vr60A}G0p@JiBkKI8A%ac(O_)3(qa-!pLfWe$6Fy){G83Csd&wCrW6v`QL%^_Vwnk!FMgV%-e!qk0EiqxdGhe7
2Hi4ska^21ONG_|q5tc64`;BqH(SwR7=0=lE5A7a00rk{|C$}sh`20zx9&gp+&AEHx$YAVPR?_IP;$O9>91tCLCL1mfca@`;&g7vW7&f6oNsQJ;zD<b%rleY
rmF1e0;-6)Rm+1HR)$XzaduKBgBe7hz0e9@8vwUd-z7ZBYeS<u0;ZfEpyU?tD~WygFALmZUWO4^<#Q!ZLbBJxEtlAQjIIA$#laz`sd8qgtF`Ch@qEgjYn&TW
$16l(p?<dSeLpm2{L=W=<>9A^vTqgHvSgZ=FT#W>GLP;pH%cUwljs*Cik`?L0$D1pLfm1j-dCe(9kr84r|!murKf6OL>uyX0~J}c*z^qyzE*mY5&s=U)TNT9
FTr3~MeMoQR?M8BKrZe#^sW8}#kELruV0t!Ou$_cY;OQcUx0%^j9zeyF6Mx3hooysP-;jLufl!-6F*)*kcvK{Gy|%c5BrsHU6@4P@3qufIwj&FKKgt1qno_D
?FUl}-tFLbFhWjbfqB_3P3l$zL9&0q?+;E%a8uRCXybFSA+0b~Npk^m(qj`gUt6UU+9_w$FR&jpya+-U*&Pg4ZCL7<s7M_>AtQQ>FU;sz01bD#K(3fwuZ#^E
#<KocOZRo|NrLKBeQV(~L$1Z4w~%L5F5;GIoHWwVJNwTAFpQE38X$&uH>~SJ1UAtCxi$iTWKYZyl^fB}gKfGx@1rsZ$vbISu|voE6U$yeWn%1#DA2dP7t$Ux
Ewl1@q%0H4G|@0@LpcLhqFS(g24ZvARum3Jf(A@-M#?@p3CY?=FYRyriJ}4gVPb<t_{89)1@$R{)WxS`S<+f@Ar9`etP(hc>MXMnn;>&lj6`0y&H&Ct0W*X%
TDsYS+8F8pAY(JxwVZcAXnLfac%i^$WZ`wAy;<`isg!RiB#Im#NB53=({|RHSPeh)7XltPiVe3FL`FG=f=1P0&7$h^qe~%5VOnG2<OM9T!DgECR-i=0$IoiD
ld1$*l6I@K$0-t}+NvDb*MnuphnZ|J9-d0+oLJ_+nj6mog}uq}Q0=ZH1C6GvF>|YxUoGx_2%A(_7S7E7B$!z4h$>Hv6<r^sh_*qd?z0Il*`A%~p4ISbq7+T*
(~ZZqLbF;OkSYQkkd(j6Afsg237i_Tm+)EdLZ8aJK{z|3c;u(oS|h6V_wF&oL>xsMQS8(oZ9G%9>SkNjWsuzeDAu1`W*t64UxG@u_&V)Vvk64aN}cOI%X?_O
4h&z6)^+3>MSh1CKH<h1lh1@o^KI<tuEdQyA62kF-;F7H_t3}@?77Jrc|tG?PH`6fBx)cRt{pFr4gmJWNW-?gI>|)LSZ06te3H%0>^IReN_Jz0mPDKYV7G2n
jc=sR$=QWU2%Bff!v$vbtvP~+iRlyvfFxV_%=nrC3k<RK0b;VVoQ349C}JyCwoxnD-4$~-bZqTPq_2!UcqS2bAFwDoB&I)uT-?fBo@Z9;03ks|ZBW&-+3Lq&
SY&;D6SZY{^SsvT*})vbl~GO#Dnp>WzJ|j|P-H&UjG(QlHMxtMYMVks?lU3?G)ghNS<A%>C}5x_UCHgddtJZ-#u!(|d7+sx))p^b-MnXk&|}XwlG-aMw-k%g
G!~iF{2Y{MqmZmC{}uX3c|u8WLu<PT)rv$8CO@}He(gQY8SgS{ch(Ufg;3Kmhf^km{|)Dne$fW{iU0@pm>JI}lR+i}p`!iF3F$_28k5|fik9xsuA5MloxfDI
if=X7V~k+RBI9x)=*EQNXwd^oKhypAuE@c4qWqR3S`fZAB%sd@k2Vm5(rwi%!7p}VH|(@|MQD2k-N)bC5Jtawcdgs0k)kfy+19&jOhQgr+|$R06Ue%yN(MeR
-fFD{RO0kBSu|IaCRn<%tjO=X3ePmW2cTMcuRmqzo^v1=o(;d$+T?MCKCy}2^u1qhKa`TR->fbtK+z|YNpDnEv)yNIP?3&r3fGT9cSVk_&_T=Tf`C=%7y)Yn
vqi8v<Y@L=1scPM*(I)aGmT<~kfGSVymu$WsYmX@10#6?;7un{1rwaBr{pj;5GLb`MA?QmpPXMw)QsN@)c=mP8xBY@I_>F%P7HW1+;yVBNyeh12l|0K)p+?*
j=TdhOYYpp|NG5}f@wM+fGZsFIpG!E0s>2GSMSfr)@KGh%+e%ZLJC|s$x+ecNCXv00#+R8A3L=%fXb~@2pmz(^d`0N4x8CjrBT<Wb~yRl%FW%_RlSD6+GX(a
IyEf`g&Y-Q6U6A<^4D0NHM+u1IiixTdjG7TORU$k?M3r--BU}3(G<XDVmT@K3qlupdSI7^u4JNpL55G<zW7N6o+b2Gm7{MiXS5H?o;l@0b-?4YJwu*AmDO)O
VXW8Jk%ildqh1rMCPIEX_Y$GW@;xM6TMrd3zT7<^egg4jA!s60Rnv@J^}Em^8k}XqI?fKp;>x#NuIhMw-YI2NG1;}~#MBwW(<kA-aAILkwQQ1d?xI<4NMElb
PBy*FbxhZ=7*;Hhqquc)RBd|)C=50U`SyWefEiI31f6XQZd>uAk5juQAKOshmk+2bKF?gN9}}nFuRD12C6;6m?2rLKj=Im^R3ND^Ys57CsmT`5Xl0{UJ{vdZ
adTfrO7gf>aAc(i@X->2Ldjn^Fumoe9TBi1kn%y*2t&hr+>aF}7DAoRT(|dsn{xlnlD|X7D8U`H4{d+q>=3|MKXiu8HF%t*XBw;T4VHspJ^eOh&ruF-(~}mb
5{QOowCy-84(^Q?qo!mWSX6|qV&rs)`l%{K2HOnQ@`C{pD~kb7K(VfO+<4L|WGeQoYFL2tyAxhcaUxsI&){gkcL$}qI2!|*tR9H<Q?2a_fjI_=y{G&W7KqhM
QVI!6{GJb!!GJ#P8OJbQze$?p=eg12xx?xGXXeDfK_Sk9F2o=T-z6vq7W<7<+9|{bE6|;DZtILc8%RJw_|{<f$80H%D1fq&epiNG)M;?JxL&MpCMu*NSg*gF
qex+YGu=@;5)78E1HkX0Ks_=vPt^8AD`9ulv;&0(kfY^SM*hZ{!hg~gL~!L8gC=`G)af%5ZNUPWAg}V0)bIfxO83d`RUXVd=Y5tJxS2gv@Ycu&p;!e{9~z(S
ujTsIFy#%Y;xYldmlaIic?JV;IC;rsdQ*D^lv@&2zdmu<s52s$p{(eefl&GNe5=LC99Rg~W56R5_MW)1J-sB6Fyg(chOt%$3xKPMsTkbTyW(=MaI)naG{e0~
o}pG6(&l*6W^(KCsimL~nyeJZ828WR^{2FG&z!}N?cg6X-RKY9phueNKzh{~7^kCA?z|^MEOHBct{CY!J_6qyZ?a{&v**UU1PqXRU!iv-G(Of*up`!F(%{~H
40j@852BX+g|Dx^C=ob>{`osnccFi09|Z_+-yE~Z6aD!d6<WK7p%OlAw9Sce^92K}J^41^pL7XbC;l3dD<6Nh#!vhFz>S&XB>GH>#I@m{z8biOE*%??LI*P^
ZgzqQfPq`<AJ$+dj$<d!PWn3RdFN#0cOOnY`Lq=Ar5p}!OE>X(&gDjk*;!oc5#pXPjIl6y#jSRqD{|fDUBojP9xBPQN*8gdR*aJd-tA8BnqGxM$27i*)0)vd
m^{hB$(@)QWUxAm_J^m~MSF)CcZ0W)qvQ3LstfHoHajJj8sRm|x;wMx3w9&PX(d&|I}+ku^}g8%EVhXL+F)Z<X39~%Ums`}V+G;93r+~VoC^Ua4>7)1{B?W%
^sryl_8D0h)F-EL3M|5-I<*w+`cIIzYB6txJSPt6nm3Ozsuvw7+}^xoJ{pV_-H=ldZW((0f^H`&6e;UKO2?g>bY#<e0zghr>rb;uSBHYey8~gM@3+D-G56X9
C^H@q%rwI#A!$$R7#CoV0GA+rO1$WzUL1t0j5{E~yqUZWMHPi}V{YIApla#qzXfayN+5XHg<y_V+0GwZb^d-|g^IhtjrbyKI0|{Y;NI;<yUN@aE~hf1u0elT
p5i_=1zY11wIDg>72<0JT&l|X%@;}$n?wUo%5sB|u;Xe{hqP<{Da&#xjRGwmnCJdS6o}JuTZc~>3Iff9VZ+YzyP@*dR~%$DCaOG<blTzxME!Lhj>4BtC+yKO
;4zgY&(`$wi=Tvx(HquVqF89kDZ}A-t!UsB?*y8L+HQOP-qxiEHa0!sW*73O9Yi5OMYwcD1qR@lf9eY}+sH`Qr9MpPpRpTl*3@6Pfx*2rSP(9#f-#1{;NbjV
N9D$Ms`|{_gF)_>yR%Clcr(-=+0O&z@@XE8YyrSLhd6v$Qam<wQE_+sXv*&GOlyGwkjF-;Rf3DpCj1Q8f!(l{$iwh$;b5ZhCD=5T(3)Cvqb(A(kK+tlcJ^RN
=R6ButKa3(ycz(Ut=REYdFe;`uFa{AmyjdCe3tGy!A2KIMdcxplEYGFkB1cqPXqz9m|t2%1-RwiBlXX1@>Q0F{3$A+JgALJy&jVaKo%%4k`?PAOsY`nrs2RB
+~oZUgI!oE_;m;qrd*a(L(#}p#=<fyLoTYd*+j{n5z6uy(g^BFVqI$k*iutf;WW__t;B)uhE4JUL|-arXuMi*iGbl4;NHSGs{F!J|Jfy_n@{<Z-VZzb8l?s<
yz1xi6&YR^2&uCJ+MX-=N%GVTNb}g%<}qu_Kz5-ryZ`{<FHAE%`hGAFJg}Qrl}9*o&Sg2Qm#0iS`vS*u5V6rMnmplb|ES<U7IO%`i#d#{_U0tvpuaRh#?#H)
gikWqt2a}tzL8x*x{Tvd7YTCpgvd|9f5!}QUq%?uiQB4@>XBqt)<Pv~kM5aSdG552S*yA2jnm3i)-14a-KHxFR2U&WX9V+wQog#ftwO|K!yx#*w74$(WR(b`
26V$0t7xK?wJP2h*-<i!ScRn&2Tnn$uo@K2abA#$Gh$h}MB0J<SZT2k#^lzTux5f44n7wvel}3bNCk%k$J>u(tInCYzL{MM<905}j3fyInctQiC5mC`<GLFI
COC1Hy7J1WwQ##qK6l<gf$f+WM{Vqsa>aY?Od*u<H)+J#>9Vt9z<Lq_`~S|RZkNJ!VS!?Yu1qzzaS>X$tNC8((&nJIo@+eGL~Y&$hMNM!9{LzcSWXsq73Q@1
9yu{{6({9s9lsO6H7lo5Wh7O#jEQZJ$FU^p|7@^OoM(HU$;ZvluHRTODkZ$rw>%hiu_s6sguXFyCBp0s+bk*GfF2fvg1mx*3-Ca+VMi$YRQ&fI+fT)CnLqi)
lvs@R8H`SB$D)>AUQLH;6EYnkzxD>XNVM@eZ|Ht!za9lms~b9oEjR5YzaMKOb+qo!BAcWp?=K4l>44Aj9CrB1Bgcx*l>I&i+`EMGA-eq_cZVJ*PN(W1^pYkO
Z;H*2mFihNPQlP&(xg+QANVrF%_Mi97p+DFX>nGA_L8meR@6L)Z1mCYq^xS~p?B$=7)*!$_1@|9Tb8%2+B&G~@V`jd1h5I9UvXQ*b(5O4BH4{?Bz<_tBo>!7
e-WZB%G$m)nE;E2(L+suQ(P2D1Fc_azs-1U8h#NofOV@RG#yV8wp_CCFwCW0w3@)L(9RQpTfS+aL@FYvKANn-2-F3E5Xzg3V1u@+p|C+JU$&)M-zz02Xk5~v
dP{gapR(c8(t671>yv_HfIT=pQN<^F_VqPB5aAR$j!mx9|6*wBA#TS`e(K&@a1yLKy}(iD%6!MDuVMzm^M%f6wT9WyAI0`uJm85uZbJ5FB}RzEMvYGwB>LRE
HJxZd(Hn4lmJpHfsJa1sY?{zIwO5@vgwP!Z`)T>oK2xnaBb5sQnX*}tTa)Br(acsS&uxi>tYd`$V{wm4Xwdck|9Dn-a^U07L>x#O12GW0|LvJ?-Cf%!#sms#
=Iwl#pw|vwUzGn2Bp;<UEh_FNtd2XB67_h$uO#D+#qDFZPa$P4Wy>_n!9s+Hcv4W)i0Au49V$-z)WuxVVh@z#Kl#Q>AMtJ$FvECkt#N^==35vk`?mP5??WJR
ib*Q=N6@bU)hZ_aPOL$j?xSXmMNQNP{;51w&{%1ce4i1fnHxR&BiyhHPV*p<B*OVK5>_}C4%-e?E0qo|n)t^4;j$H@4e4|3By3hubTnuZjWGC%s}#y|(w}+l
w;Wna!>Io6`a9@32TxRIO?5|tv!+g+9fWoiJgV^_(*p6^XfeTTM`+!FsuqHz9GdLM^8B5#)z;X6UdhUQ**Y3}Ha*C1pUV{vxT6)FIEDoH`8uzs5fJIKn2@4y
EffCLMxn|p;18z0ngNWSXDlhQs;Sa>!1xYP5ycU~_b*4Bprvz#3)jFlGk1wxzdX-Vn_tWNpW6B>t?|H#Ut-3<#qYtYjrS~z{nats$Smq&gV07*{H}Ubt>r?^
A<?&Wau+7cITzj;65-tB0ZDOR3ppN*`}U9a*W{6NrP4YvxG$j-8mp5C6<$gcbEeRorLX&MDKkXup(Bm#E5DLO%)Ar2dn7?)*aY%>XWIK?L5>iA!E1U#Cvyk_
DTakVEA+zBtDkM3taoYGXfxEm3G)*TMW=yipWsCc5almqEx8N2#R$)0B}h7ONx6Q&F`;8mB%3kSdEro$Ltl44n13%CP2|x(d6wfG{sQ06NB;w?w#EV>B>3U%
Kd=64-I<PL@bHH^+y~DXq#Y0Eg5*Nxap#b0S2yfF$&mW;+f>7*Rw5?AwcEzDjVnqa6p4+gWD%^zKqth?m&kLW4-c?8Enz5NNQa}Lf~3*l!aiv8eU6O-g}9!t
D2BTf>e11l1nr^Mm{>*0`k}=X`dGn+s!tg%jI!fx<C}Q@);EcD5i^%@bVn`enhgQ$MwZF0Af?Dn&iI*?F+(h!gfVepK%F~A58t4uoc<)EIkfk^i?fz~f{?e2
Au?VFrsiJECz=%fvDYeTAuuq6FqE`H4>8N+aFB3l^W!m`Q%Xv?zMfi#S*T!KAQDU7O`~qS;8z$<uDk>lajgr^pt}d}<gZ0sJJb(mUimebZ%H0MQrHS$4HJDi
Q}H2n%oJN89#O1UuY0I<UZp6{-gt)kW?lzIIN2D;z5W0XnY?vbc>Mp~t;~tRfu&7d#kn;_DJv!<U#LgqB4o!V%qk2nL4U&;qY)~;#==zdEK!F5<sE5&dd{#|
<LF8tK1BLb`t|5uOHes45zUC=pz%uv_*O4W;-vx=SSYuL$MtR!^xq2Hy?925fTYy=@#cjcTzAeMwlNx(-|Vw{XSj?KPiq-r>=?GMpoT5<JLg4vX{(7+oFW=e
pN05vY}zKzVUN@&qCE_7c?a-V`uc2F_bvBtX+$o1-u68*(bHiP$dit6jdK6N0=&JwTOwFgH~a`4YZwkGxEV*)jF8$r<+oQR2?yiGnT!IFuRPbe$Nhz>8$0E<
X6rYK`afFSo~G)yY>%%AG7>qZts6d<a!m4hWm|00!o%7hMrQ3p^;uGJ_{J$Ll=bsn`Z9X~O9j=n{#v3J!eN#pRPE@g;N%)5aOT5A4a{J$2?pP)^&c9^wk#t0
RWspn+F#$Q>&XO=0x6c^;--sLK7ZzOc#`#W%bQu>+OR2cb$g-ijQNVNMDHoIw)u*xjP*Q?!4(!kYk69T?r`-2U|3%!As}NS2d8t%xl$w~K5JaAHyk3Nv`xW{
%o{jTO$)w@Xv8nu=8XsE`3pxi$ZVzZf#QzNVDqy1R{ZgRjJ_RYKwO3lfSANgUI9nObq`zzRZP6DOIcB?OanZY!huP8kr3QlG4Sg)wBV8j4;|$Owes-bz|7@m
5IG4GR}c6D@$T8V4wOMkP9L56D${bFXz7BaqHAL}r&86#RpQqP>t<+$iV<}{HF}t-*GxE<&^He+U)xGEeS5fxPs}>VqeIgQG54{nk5e8W=kRDhQ9FBuLNW%J
XpJ-Sk#`V9p?`ojRLBF*L)IkSpaxZYm9Z5+Y<x3?jpXJ?s@;UZNBdh_xFT*dkem)4&_gP3Ya=k2>3JZb=`{x-#lhyAqjQfxuT3)x>6yhoj5jw%)ugyHbubv&
>Q*9U+%3^@8TPl2G*BBI0VhN&j_^#n23$}Tj``E`7v^Y`{TR{h#L4XlnD1_@(BHqTiM2Vd_XwL{q@k5ORyKED@|jMsi0mDMz+#;sc{pRXtb;N~(vHoWT{(#m
Okf`JrsQSoa-HHk=Ge5xW?)N&0>c1k8-1viFvA_9`mP;WIGkLcv(2qz?dWFYI+W11lZ1%@(Eg=!_EK?w030ol8=VPPQHfiM@>Kf3H+9&@7DnP5lAg~=j*)3q
M!i`bms|PveV^xZw!XUTS$Pr(!}*G0aVzi;-CGlY3}2mwo>Ddj6*REeOwR0TtveinfvHf)Yz@p#8xEY6lGn4>o}g9!P(i*Y$8Er1BItS?%G;v%##7pWHY1#_
$fJ3tndw>A2eKHV8xrtY{q(2$fcyO|9cqfU$Ih{0vf>ICpzLHFzb^u~@7d(^2Yh?Gf&{J5Y4M;q`Ft>A50kh<Gr1KG{g6igNz%}cJ!ISxYOz?vQ24PU{pw%0
U1mPH(gMuG&5j=7u7l-q5P0H@ya2r`Xpf3vtW@i&vu~cvbszVtt9`AXaJPnG2CE=*PDvju{?AOx`m!`BBtxPaT&zr!*8ov2Ve%7%OL>|E{rUGBL5gdE(n7b#
n0=o2PQ?mcA>sVH%8fSdfLzv8%FF?^l9(6AxHSU}Vb?_5aFV@0%wGgAejx}=CjeWdcN=Wj*h8-$@>kn@A2b;TvGh;C0s_EptZA1f_y&-N(yZ2pex5;UcAREi
A>kOeE5zE8F+{0=YFQXU*(dNYz@v;=agwl2({%-W)Xji3^{6cv;bP>@+kAbfK=^OgL#vCWyR~TMA)tn(JBQ1slz)B<{h_aY_Hi}VF-_F=MLZ3$=o)BTv}Qgf
YIk&^^*P(y%i~-?VFQ?)ek57M{+{Q(7g7SI1o8{t6tcv96;~EwuP$#q-%Eai0hnux*E|wXnP)^uNUWVDBy`%?J!)#K9g7dkImE2Vc2c)u3$s2WJX)whcYq%C
P{Ae>GI;pIP7xT3_A;`hxJTONFtzc|*xN{4=dv~AdNu1xgmP;O>T2`~JF(NJ|J}IRe|<Wu_C-DsSUDvZ;OCfry)XEhRmw-=g)}1l3fAZ49DL%@Gg-d?sNv;R
S5Caal}Yl27zs#qKl2<HtD>AMstOGnQ)=mve*eXLsE>bW|CuOMbIJXp^l&RwZ5P{zLL>gZF(iD-SBvWhm|iL1+$CkBR&O9@CnZO{Izq%i^q3<ej5P6F;2QJ0
wFa?U8C(HqU?D}QVa7o(`T%k?*Tf6St8F)JOBtP>QVrkTUnB&`&(qt$jBz6vyBW?&{Lf@^<0GUSu{Hv5(eY8Bw}F{A_K={g(L-_Y(^D`h7tZ1|+g4<Bh7|dN
b<kOXfhlB+nDYZ%)3znkL3Nqx2*LL8S>6LQ2fC!#?_r9)`iS)uj}SCVhs*HE3Q@B&q*1YgE>jHYufe3>L9R4?#9fKXcexLknEH4`GIpoN>QGl6-e$(JSk$){
eZ%!f1VtS*8hK$LdqNxi(tyvpeSw5;F9R(g{B708JAyM_5S6{+w7RZOKR=!##g<Xc`yh}CfcP%O=3|Kx*tkCk^!H09(h7#qEqG$Mu$5f3e13RZDRL(>m#4(x
w<1aSG}OO$R>0%RRCF9i^XkGBv!d`PS~j0IgNb<e7|~xYAejAMv3$VD=u_bOR_GUU)C{-3f+{aQw@j9@FtZb{#prAivQDBpdgYL3R`vy8Xy=_RHv3i_u$mAX
ON-#X9@UlNJ~s+MTkm`8JhxQSk@R6T+&!Wzw`}g7t`~SCa|+LVh6ZJ~&H6Tg4%EmZC=2^6|9+PUFeY@UTw>Ci+F3A?r#V^iaWG@yGxxx2L_To@tF7XV1)fa2
I74~0k9fC=U2VCwv-nDk*zbahiYXn>_k%~!`$mhM`!d^%*W~if#v7R~&+p&OX7WUxp;)}uHIBLvI+T*Y{Zky<d4|EUgJsPLMMw<)M;=z1hDq<f)BMMyDSz%Q
2x@8=w4(nlnR0FXYGjKaFN-av_Bm@pIy5Me27MSm1aM9k0<AJ1<mskpH&$i}*IdqbtSO6=R-;aii*~yh>iHJIgiZnPu~|`-hWUN!$7xH0gsrA0EH81nrC;W;
FJ6)gce0Rg-jZwUfX@Jb)e>*Scib>wVLQu@$^aa2$~`A<P^JiV1HplHAoFjCF;6Z1Z|=R`a>!uL;B<<@KOn2?@NqwkYB+b^$=NzQ*PDA3WDU|WCzteGhjnU5
!z5{dh)6J5T8c1YISy;jm{82^X+OcM3Z-dVaUgVBSIKFvdOJQsxInt+8Dwr#u-u3P1s49_zpb`t9CT3B1!w6$lK+Ah;A$e^?-BWTQ?U&(1Ur|0;zWxirLgTx
HRDLPmfeXO6BW(k!_5)*tz6O5ml+VvykE(SfD4n^Ax^Lk;tX{QC7z2`p-hS+X6sVRHTq29T2iCcc<AAPHr`wO(v8bYxV^?IA}oYIYt^>%FJ3Ulrdd8}Z09z-
1A~Oiv&`4cU&tJD+bUf-T8JAKzg}1+*7rWTpD|4z+vZU8N1~>uub}r4pSRL-NfCsPF4o};{$P%0+ZAz(UTZ_=+Uf>o=V`u(|5!9?sKU0b8`gG<6j0YS5C-m*
5@zT1E5;7L)*99O-!n32qZ26rs)}aSdSB>6;1>Lb=s`Emx-7|bt(ha$gXj87#u#MGs|Zfk?Gq77yIocr>h=hqSOZnBz|@au;Xi~ofi5L5rCl!b15c`1WQ}kw
(*4zyD%1OoFhc6xX)aFw636~o>J6Qwo3aWSaGcxXX%20;ITN`%DVbTP+-O*AAS@~(WXwT{G@k|=BlcG`X>;-rXUg{5)Ha=^9w-G=PI&2lgB^i~E(zV|Iwsed
l=S=o6jp1;ePyv3_oSu0X!fE~#`n_GhlJ9+p=6&oWTC`w{z;RL*gEhzF>?|iY1wVNgwj_RxGH8_iL&L?8mYRUw;iFW%;iZ6_hh?LQk^|76iw{rHc-XDULl*U
ep5O=1X7X~cY@y9Osj@HZsD-%&oA9f^oG#_g#09)_qz$zi&$@+IWpJszDCT9TSHyc{h!84NZuV9ynsO+sz>okssnV$hX)L3N-q2V{*~^>`1=k2>Bg}OQ|f5t
H>;esp^2$-N$oDE)(H@TP+k)!2p^#bgHwp%{xILJ41H?r8Xv3ut9j}*)ezco>F#|7$16PTbqr0!!&Q2;s(g>`R4K~aah`1*9>hXj{}p%iYYv-gIjAoTQJ*nO
GOpt940AIl{O*r{4dNOKVB5dFG2c<`R@l8o4!y2!itOxaG$Visv)@37%qDd6>6!Vv!$T5l5AyGB47)<%se}mDlF>ghfOcSn;gezu`S}6Vir@Ih{Q65WJn<v-
-u~lgHwZ_wV0^w7L`1%&epTSZP{0u&x}tSLN!$nhWzSZU#o$k+s@xazQ<&2-ONn4}`corBfCD+ePOs4*wi`f=3MG1kbUJF>Yr%F2ws!1r)5-61DQL%}@eCg5
KOU~;x`kzJLr>-%CX8FC!dkc#^W_VYQyUzW%$YD&IZKR$1~8ZW1$2G*n2Unh+gI`VH5jO-7Q@0^ySHp|>g%)zzvB0%elwD{r(&M|@npv=a#DYw=9i?|G!ucq
GCe#@1+EQL!59QgG~4b@PFnr*WXP2VMzGo^mms9-qI?&P?GkxLCUWQ#776f+(xlCfe-RhTNJoznT^;_|M<p~=M_oSdTPLbbvtJ`JOwbxFe2}=~EDhNN)9ahQ
OzmOW;_vH%0(XTG$@LZNEaO$$@^yXLl%onK?kDXlh(5=Na>#J){O+@>@hTT$Wz0-CkJ`J`@?v^XkLx1ZHo{20b6vP;M6y8*X8u?I|8|XgQTC{7>;E#^KWZ>l
lktwxUQt<)F`l70!~77PKe6)xBE?O!C%BM9bt8cy?fYTnoLM<E{m%kmMIr(4w|*fpG83LE!D3kJRZ~8=(KKNx_cO?XcfUpO7~f#h_D-QmzZKpLwwgTZ9`R*<
=#m>N-%d`we*@kq10Kiq#YL?9O1m4_5#K2PjNER1vQO0Y>N@7vxvyjzlMq&aQo;QvDv24%smY~Z{T{tGrw3vHkFva1`6jo=+X>TxOTl&^Q0+Ebj%$&tiT5Uv
8)&7*gko&$2&`)b@GoxJI0noyQV^Py@=!^2F31@#X~6*mH`MoPzg4WbPi<JBI`NB{*a3)v<=*}AAmXxr__aYcV@k}cPYCx$Er)4MJAAH|vh9UsmjO}d=R4m(
?lUuVgY^HzgTD)mYP>+&S9C0lH5+iQ+gKQtExX{2T2t4YwF-n5D-DDD5}1s$8Cv=2<$R@=41WCx1yM7&2ncp&7?*R)pz0~V2U}SIoC(fczkM#N%K7r2`rqbw
neJWP?8v*+2ZIMS%;Ug@-r{|V`k1@;18c9co$nA+Sq<w>$c%qn`d9j<F-BjEcvWb7B2_Bp1yAcCNvl(NO-AaoT*ug;Pb}cb=I(TJ>~Uqcyd2;>I0sWhRe=Iu
6pKJ_!7HrxZAeTN88pP}=A^P6{z&fnN6%QQk&;M&Wl_-V!>HLmj`-mbJTzQl_Q43QnvAul^3X8Gf`7ak;m+`<a_(rI#TiV1fLph=R<Xo<U0@gWR0!0UuXyp*
^Yp4doi;=8@&tmoT2SWL)K4EnBXJyTA$@NiklI7Ax@#$UG+e9*&jW;Zq<I{#J*a^QI+L7oU%WW;q7}qxpnLYON}NiZB{^`lTnCo*{OG2edX--9*vpw`499lr
o@g%<owhr+oO(_qDX<r{w)NsYL_s8vy#NZV9FJ5uIaT-pbT2K&lI_HGfS;zQOROE!Ewn14>?)cz^gnd0Nb8rG)07L_7t|UQ=X?%8H%@|im6P?hKMM?lj-B{4
m9B=cM=<z~IzEQL)e!sFV>v-SI##UpdXWok3MvR3B*WC@XPy0?Jn#Q|lwxrCzR<K{Bc)aJ0}fU7Y)^=Du*3MuQIUn$&Jq?}B6zURPF8EL@-F`xwk?6pqVP8T
FlDq<ot1v|0Xm2bcP;2t?4`e;V7KqYNVj?WHH(>n73JB203uVFgs96Y)<K%kwm$Q*5Qb=mvohC7xLtdH)bd{$UiB$`qTK-8{ED*Cnmo6<&RE-3KiCE#bH&TX
e){{nw#=B&x(}5;J6+v{VR!kqF&(QT1XY-b1VjqX7s?l?s{T)8;0Zs0w94rt_Vn5f9>*g+lp@gFa9xU+ss-ws2A@voMI7f;xgpy^Z3m-yW<6&?PQiPW1-DMA
(P=e^E(RSp_BQK@aurExN000UESiuE@0@GvGGRi5;Qe)7IDzyU$!KexK8IkQ?Lw-<Hf{}U#N({ZcaukNHPn7xaQv`P%FR&FB7VV_m=VW`q!*WPsa43kFCWg_
W(C|Sr8c{aEW?*0v5wFN?m{IvRn++Vi_1@f@pA_=&n}EV!g@%ewW$U|W2N6)oE8bwsHI~<I~!^Ad)+wy^HYs6Wz}T=7c^>ViU_uGuSZWoTm>X%Y?k(DC?-%d
bRFH!Wo}Z-$iE%B6=Qqa`Qh@%HuMy53w5?7=imhP+Op~}F3{N$>`TNYP<2hOfV-^lIxFr?f%aRR0XTBM(*ibhzfnp|tgWC-Q#dBI;_w3cMV>7Eyt->aR{MZY
#LKj4cDif<l+1Rhtl?HE3ba$^LlmWqC#MMcwT2_r+`v)+;t(W=Tcz3Gx?qpDWu(X5gI7*M7-3!#=I5PlUI!HpKD1Z`&iwQtMfp$`azRId{fyG(`*PT})xDD6
+6v&B1?qUJT!SyMb(Yge;rISFXC?B~NMn|I(LB-viKa~>8yTfUXTB=l4Pq+`R9TJqA2us$UUUrxv7u1{MMH7wm&{e?eQ5Mk%q&+N_6K(-PC)PliG%FoQ8OVf
8LEp;DybyEH4yQQ!|+DOLyEq6s0P?i?O@sKxI|>GOMwE+U?abda%#xz7Q?aUrt7Mem>BWO!tw>7(Ebq^>s-_geS~O5IaI8bnWOvhs{ob1AiER~u)I-LBuTdM
%D7!|Q(G%lkmv6f+b%azTVAjB;d6<?zrw+TqUHTtr|>(l2VJJ`evj$NX_*zxV2_)|f8aC8uOcx?Oy_^YKq@v@{1c-mN!2K}1w}xWvAGP*_1fe7CJQk@q;Fff
n}Q}4kE+I7O<vYztZ_~ei|ZRu+ibU$i-;*!N@hW|kKU&$+1La_&E4Kjh)OEpg>;0<IC;^gMHY#Am8!0muRi(WpQH@XYdc&1wv?v0n+>hsSpBJ4X=2yq$VZsw
A8bJMAe0dpTujbA0=sbM-6lvmlW9#(AF6JMkW!6k2OXhobJLG={drMSz{V^k_5VK;8fI`-Q2gfvvu(7$qK498^m02x>VZNL72ft5=&0HKPu%f{m-ehBa#ZV7
zM!wN{S{eGue_w}UdoPDt;|S+##EE6Oev5O=o&v(B1uZxS7)(!58P_);lDr<vQqEy{E4zYL_K`eYNipli`3oX-%vClwRK1Gw**QK?QdpcJl?VV9ArbE3H>Zr
Lb88x-7zUR+s>|%@|5Lp%J506wbY$(3*qu@xe{+!#8!v81jnmC+i6nDBt&wQ7_VgkwG<r3`Yf=y1+jVr3#kl)2~_lWdR+yCquOhAaQ_|bM#3X`$Z+-u0|l+g
7Pt*lZ<oSuG*5O0+jywEp~^>(QN^K`!?0X<A7uCa^1Z9}X@FJdWN?GN38i3Kg`yFN?eSp^O6QSb0yMEmGH%i2jh;%@u_X*KsYt`>>GUH#l8;*I%DxFu{ah<e
Hynd4zS=^|dIc6jzYzl=OV#*J;(5Nm?M0h4z!;;Q9vRWK8JdL9|07u$G<#T*Hra;%`^uX^AS2&7IY$@dhlW>+l#CHkBLVV}*}Cx`SXJy=O@yR$20&&e8o)0g
4}Vzq?J^@hd$P#e0ZUu2cv@CI0omJdgkoxWi`r2)$FPe5BYyhtvq1Q|oChSL*8f7(h%kJI_cocW-FYre*08SX&8f~7Ipo?Y>%n((%1zGTKYi2~<zOa|fC$V_
(q;J0Cs2rjOiuCPX$owMo4Z@$FEZ&lw;Ri*_`J>1QeNi0Hbp0j(eUUr#O}jiEh<?GtO&~Gwy<))Gh=4u$I}+0vDrS_7j>vLHc~Q$!&o=sn2^sENKq1IEy9|)
&}GA7wC`ztTI&sQv+ZMm9M#HB*@2g!%!y&y6S$$CRd6{C)`|t$b>M@il3P~vv^e?gRBVb@e|*9o)qgd=JvGY_UJlqpypM@_%zig?jao#yh=nxlSdgto#UGDZ
Dz#n2A2a4o1rZo{Lje(RQ*4$9aML_>;M9)D{=Sa;#LB9oo08;bJZrpaLVmh^0aXithzy7HrJ>Jwbj29LG2${#PzTO;aG4&jXHu2gvF~%mZb{Ch06v$eY8cJ{
_xnaV(OJEZEb=7Ne~zwIiU^>IFdJV&CHPJfbFXDuA&0&6afi++a5@<Y9utFT-)tov0g;6MChP4%sCI|3uCpU(t&Y?}G7>Jp%{c9H`&Y7$+jG73@KV}Mp}0%I
(#^5!Rw$O=oHq6?l4dE8&}$i~1^J|m>%P`ncbV|9Knc3**K#Q_m>iQF?>LhhqAKA-9*4nJU0<79&OE_(_F3ZP=LVz=tL#a*GzMWj$kBTyydR)11;#BB+8foH
qFH9Kf)MFM!G=i;eD-o?wjQ3#vuJEcUpyqvjCEe&G9Yy0gNuxh(tDRyh2`m>5mZDViNqbuYWHnLC^2g<nW!M+UWeWO7GF!%#brKV63_>s{=(S2b0~$+mWBWL
CfSzPGZ;w^Mw%E@&KCSW@9Th*uyu7^?>u@<;fc}=?)1Sx))$v8rnR}#^)rYKigspy|JMT?VN-^@SZ1tGDm7+GtDc$|XZW8eX@?TM&z=RPW-(*v&yG@!agO2_
kVPq@#b9?NC1oI=2e`O<P_zD$MGo?=hM1)#!?5h%L6*!;5F~My_hX%rlc!!y8OT!@Qr$uXbexR#E5YmPAUPQaPAoB4GS1KX+Dp4K#~p!+Y$-k1u5dR9-8bfA
dR@jnTZOqJ&1mA!q{n6B`-cYghH8t_aTLFvOE&GPTaUmerYIyK65-j{v=u+srqoe4HV|Jbt(h!!S0BMup?HpRqImMS^v@id2A2LjhWQtC0aa-K+y6MjJX%%i
pVc}cke%xWCSEyKmBq!m4#^_HF2yfc#Xaot>KLxd^yI*7ZL}t)wP#g{AI11G&mb}$(n|c^`KYKOM{B5(DLzp2k=6C?B+zJl<!&9iKDT}r`$d0i3Z$&$$y4ia
A{$I4IFVrbJ<ot#_HB8&LY_2n(`wTkxM-cfngM{uVdvDfDoauhF`U8Y1?KRchQm3xCsXDT@6_a*r~0BGV5P^U@8C2B?+v%tKL9lmlss<9s*n2{C)+l#gj~*M
7mE@Hgv|F-P4{pt6BVU~^;xSJ;!RM)G-(aVbi#oMTv8!Z1>gcsDuqK>9i{}HN!eL0GzWA<$16gIBBQoB#TcDTouGmHZS}b?uck&4nPUqrN53kRp^s3bWUZRH
9eDqVT8XLy5L;%Jgv!q+b1j;+V8Jbc)Pn9XXR8D6|1)80$s}^TJ(wSnk~y;S)26nl1fO~Q`dKW{XV^~WPV=1S0S;7zY$68?vPuz5pR};@i#l|UQP~3-QotaW
wet$~ZZ+1EoMzbIx}%tUTQxD=5tNHZP<b9cnDC(xq^~%*JHscC5a;kFVf!8R87#rn1Bs{rMX^*r95g1p1DUP3^Y^q9ikKy!+s5Z|BGIrGYV?Uu#hx2u#Xow9
f>U{L1Gb;Z;{J=LwaE8ns%&=U$5b>30BRK$oP0KbYv{Mcpu}&yP^|zHs)_XzGRI5Ky3LP8rXG_@a6=b9&5Y-=N=D$vr=+qfNx~hlmR%L<(fek}eT1ODVclE?
M)lf~QjCR(3;ha{h)6AM`QL|3T1;*JW8N-tXV{2xwnJzav<abDcN#&Hd2d5yslYW`BS$O4v|7OH`IuC&t(DPnsZ4)c^+>GLIuF|9yXX3^LPP1U`?7!fIv4ah
*P>}LnJ^gwD=vj-d%uu(a6)Sgw=gS&Gjr|puLw&qTO|ih&F(m7X>y|g8JZa+Ty!famV4O5S0Kx~t!tr}g6GwC+O{x+7(udq5@InFEXdoUWTO|`%wwwxuygKJ
hf7u58VOy@+$mktw>0w35X^q$GjRxAtLEUTLoe0l!_N*7vn}Vs)hKOD#86~p|9hMk{-@dqUbgEN^Cw1AOM}41$k`bY&trM?mu5j!p3~|MEvDhMbT}q|!u=lx
hM&zc!gAA)aT7dAu+C^j<9cg;i1sRBYIx60-j(Tu&1D~z{YLVS^jvV~`g{GufR*PkZcy6CtX<Oa3n$)`Z#^J*cw}-fuyr)+=Xo)ED*MCPyd3mZ<;hK<aGiy8
9H52x0}S933QJwOer6CcMAWqkD#TC?*mAUISz}E<ly`hhSjmdTAxg+VD%V}>5fX1(pr3R`N~Z>uD<2?8rmhIBq%C2G@z%KKQrt?zy2D+6g`{&XP+uNrFpka#
g4Ur5C^a?pTgup-EA%@rYpT&G@=P{%SdC#xuP3lRiJ}U(!3-1xjZQWUPL6O(Bl919lYvy7b4{#P;G?Uw4wEZa4|1q`r-ie2KQ{|s)6;XxOb$UQS!UC_K2@;d
A&#;(%i}58Nh;0GKqK8!zlF-wXeFx+t?BwxiEU%zp0DKNs7e_Fq|o8W@|H0c7(qF?r@qG_MFBW6Ep{o$mz5``Fyuo$xMF4s?TLT3_{}CEhkrAs&j4Jfv{=zW
Tyzq7%2kj{u~En&YRGaHBsxoWvgg$xiSLq2g8Vy|&1uPX6iox-K;hHkysiCs(j{ZY02J_6*neOogVIdQV=wp0!i$Kw2eLW`X&XsnE%$Ml!e=~aRH=6@#yp1=
5XxIDLq?Z!I~}sC_QnunYd<x+CvX7N3lMQnI{UAW(5(S8`L9rtVXRYz+V8n)kXkDhV<if+-kOb@u+#~mbXGWef`fBUPM&)i6@Jd)h{Z^qtvO6@d@r4`zhlr7
HSe6-$?_zcm~pWr>p-DS?EvzVBQ=-bWU353FK#JIKy?l45-D(Vv{g2Wg!otiO!##m>eVjdtry@T#SJ>cP=G{=>T!GlD0De<Pg;!x!Ff#!s_6IjIF?0|P97dZ
;n6M6q$eoWwhFSVkEBv(`$Trpm@&zU^1?);z2P1bmQhm?oH#Gr10p{W<?GR`u0BO`f29%@F=mSY%|y+e?K@Fa6%oe{QzXS~L|8yXK^kX+q_Pe!zd4hL5dYJ*
gNcNnqL}ska~33`?a=NbtwaRt0!mXG{}7#{i*l6X40u749(xj|R`dq(s;T>oO+bntX#L!-+!~2vG|~jH{^Q1C+=g86Q(op=jY4g_L-qA+eUa4?^fu4ReeT9K
4VVW9LXSJrGhz$uh1DvhH4SA>HYtT;Ab{^JJ=Mbni8RQeiy)BaarmDj?4Z|$-cbP}?BO6?MbBAsRiGu$07?`yag?8%&**lSyoBso(7cTOe@aDgmyqBY>ubeH
q}U|_sqY<TBS6sQ5{VT-Y-$iUG|$_sKQZ<}mNrY=U}V<l<=Tz#oJ_N{wVY5BjkWBR;}LbmE)~X5-J@eW(q)&1iCj4g@guC+Xq0ZG?%h&OM9N*o@qAhjq-|%b
LM(2{Z(5Y{5Rv(+d#~6GCxz*_3VLy`wCbK!)HrbuC_es;DLasZKZoX2>Dv%W9T+fhJ{V{OuiuYDIYd^dKQO;UHBmnv&7P{VIc(DIS4zl}p;!;0V)1b(iPZa)
g5R*L>~GcyBx5L#m^M|4QJyOV708G5G{LSfw)$r}>916OU9;RJ(p!U$U>%j+rz9`)ordS)_pC3u6;kcENGA@J)epJDaE&)}>@b~MFuKEjEUjtWo!SmdqBEO9
q#P1z8S4j*hGdJ!r7G46Vm9e(y7VDbF{ux!5lQ|rpjhxG@<CWu0Aj$XrZ%OwI!d)Xmc>(ls<_iUzte08=?3|SG?hVtYexhz`WL52U%D5%@^=kcq*MNX`DxU(
OH)@}?^3qVx-s&nNOONE+-dbf84y<e<b3L(<1E@zg2<=8NC;)XYHR-wlCUamsC}{-*xR7oF!QaAyvpmG-pT|fG1k6{@o&W?r}_oH6U?hZw7wBE3=V)dkk;J~
K=9M_W_V3h|Ji2P)mu>M8q!q1)4~PN5?FZH4OGJb!)b=FNKadOU8lh>MR;}1eTvn#xg;7GL0%QAU$>xf`VR0JG*{}r6#G`Bnz6r8d{~!KVcUx`i2yRKA~HiS
-VWs(D@*mwU66x6yW@DHY0!*x*nbYc>J9-~4VERJ9?KKOuGMsnQhKYIUK(m+;J(G<O4?G5%o!)lU5j00CH$acHyFIWjMpO<uQ&$f7M;UH(_%g!O_)M}f>}2e
877KdRqcijva44;T9iJosZyv%8+>&TU7%qz=MI!hl|VI~r#0NxPO;!{KJ?j(1K({$By$PaVm{x4D2SK37p#}xlj-i8YRRFv!ho(+e^mQ~vLETn3L(6peRUYD
5CTLXBikJ9iwKPdMxM)Q1Juqv1E9k6F6}ik25*4V0LeoYE{uTVe%w|BBh7FzS_9&(@K?wZW%O&r(qhkT!6z7rZZX6T5D1lLpf$n%@o^Q}lI9CIHm-y4_s`K2
^W(PWNRv6~4Vn3SpP^dB!2OE(EkML6>`vJaOS+Wm;Ehl}b{z@BNZmtIF3AZ$apE9xN?jdlK+a9`p%rSU&oNL{_9DdD%;#%(tk=7<;$s|zBT0rg@ruBJ0p}Ey
>gVjhpKIpg<2qdKTDgzT5=8;+tdOYf!{m<<I$E{5Agbu>vQE2x-2NnKy?ugl95&KLphe5LpXEZ9BpcJ3*9xK1(6HflK{h($>B@JfTcWlGOtHZ}0iXv{NKAb0
b`nRcv6t5Ef4t{sx?|vEo7^X+^bbtY`TOnboe`GBasGbu0&|ZNowU!|L4UewkidUlAc1j{<Pi|LDIU^AxwdrWaS4+3>Wt}{PEskAs^)Ur-6gZAjq)^u*^&du
5e9N$th0F61WNp&$4QV9*l1Sa#b~hHt*g*dYx_lfR)<jU#gU+gvrlGmTzbVTY0g094VCHi1!%IX4amH;66cjKeuy=<`caez-lh9kuz}to4^|0NhDE;SL@gv+
m3|ApBBC3Um!OM!=ljI|cVJF?(%koPLW7I0xIC~FV-8wmp-|c!k>#~tWj=6aLtKIkKCC3u7owwm93qnMuA{+fx4Ks3@t1}{5;Ms>u;y|*f{50|M8F-8!x<KG
Fx-W^%u?h0&COt_eiFV7Fmxc}r4=h7-g0Rxf^+U+<k%^JI>w?cTQ}FGum|fV6{Yqe#P43sS2|t>2&eUYO6B54%j8`DE<pZ~n6)x~;h==&oD(!-pVN9y$Js>j
E=2k-O`T{6K8CM)&L(m1HIUy@KhduR9p3TZK+*xxN35h8;MF!$(P3MZ(70A>*OxV3aAiSyRUd;$;z04Fy@t~E&1lq!j;Tx8ZfZ7+Y&RR#Tlq@4?aF>ic|_I#
+iRqQZ6fAW+5t-C&fe|oD0yaR=DACeG08ks_A;;{$_h_Zay%BA79~ZS#KY@@+)<dQGlzYLT7RaXwQ;;cUmZX3dSe3*dG!!%)cRVWsdOv>$x2+)y<~v_l{f{J
HPy^Ol$IlcEASP5hD4iCeX<3mE(t4WrzOCL;%5-}9n0(4DNf=&zXq5FXZNxo#X2Ncw1nrr%&t`Lwz!f+XG$Txv!Sv7#X-2ZNb@M<Zqqr=F1ZTvUI~y*o@O)Z
a}X>ThtxKemlB6-MJRc6AF%&xurER>ZCOLnJ^}H6e8yMFVr7%YtTNOFFr#{Dn1iBAWSIwJ*E9Izm*gt{wW(LR1qYQ_pSp8+CgBtlVG@imwXR9-<SaO{(^09j
G)`BkHNF1r8jD;E+H`s-P{9HtgZc+X>9d#h^3qiPz=hW2h$qV{SvM@}@;^V!h3=aQ1Z0?Tg_mu@6jvuM6Y`V?eJ4<2INyivEsSCyUwh_giLDzK5Xj)1KsB&;
5PZodAU1e~vi9&PA+f#`@<k3i(stt<a}&^WkDCsK!VYTRE*x^g>KVgHxZ++<KM16(mx64>Au?S0+E%<n^5-&zKNe||C<wYUtf>!Azxsa^Nc$WdLaUs<@X?66
DK=jTu&B-z!(XmbP8YkL6SU~95_#Ub6jHr1AVg?krNfB)ZZv|i6B}9^5`(OV)oW$RqbYw!GkAh)g9Vy}f26MSbjgg&WsLOL6yNt$9ZBv|HW|9kUER^4*epCk
P4IeO8fjM0r4FSL-?iuOF`yW4=lgZ4gc#HA^e#mrj~~yLWMsT*hSyl&b(FQ~MP4U&>sUvafpJ)h%|{&bO|g5g_j@?;;Z6K*=rVza)dupYGN$e1v?%iUmu4}e
Doz#)V=qs3J7d5Usq=|EBLf3>VJL;gxP(A_3)&K`vutAFG1M1fzl+V(g3CbbBSc7o_qt2%>TGF4$v$U?<a@XXdsq_SlWGU${q3uDcLA6-6<Ej7J-eOmqVA%9
!W8eD^BoWO=oXAl96Tc*(dN#S<T|3z$&`|qZB^Me!cN+#9GVs6@CThk?f>~c_CmAuGY+N5{eh~*`mC)E(wrkeDy4W;M}Kj#Is6y{vIj?N!7#RRbthTK5SGiB
ZU-^^sW>TLI<+kpp$+!!n${%tvQJ&Opvr0L9Br@`j3*G_A-dp=GPP^_x9sA8X}QQ`;9fo2VJ`{tW@_nnneaht8M>1-VcvY|pQm+x=YHngLx#;4#iWjn=R1<N
ihW$q!X%m)j2UDEMRFIsu>?(=`%KqL3D!VC!Fy<Pcb>~Zi_xaQ$h;8I%fZi==VKvIal5;o33?;4X8(#psqG(DQ!%vN)hPaJrT!<!E0@$R%s#{<dY)TT4@iB4
sePn0jIk)=_v#C3TgX_(wlHI#?d#*j-=s-<dGSdHqw!QS_T#(YH80%4V3KW#r=4;=C1qQd@DZdbPtLHI#>e|eh61@m_2ylN9aavnlFo@nupn@y9(A5~PpX%~
00u})lPe-;{LZ~aWM^#>6rHaQZX#SMi--~mqG}(fu@TzUsp3`kg}Fq8a6oD2UaeX07|(?9^N&P85R;$kgpg006_;owsF$=&tUk5IPi9Vvbroq*g3t*o_!76-
g_}^{LIj!b<xc;0dtuFX!uoS9QrIyX%8uEHnVUy0{R)pd7@=(OAY!B_)K7Ykt0g}_k@<&X+Wrl()2F3X-)T<MlU6r(;oaaW>4P#k`H!h*-hP?(Z>qq0v>WR=
yiHjPs~w*}^Xbfo$-St><v9EjSo1neQF_4v_^V2nGfZ;*)jq8dMqe`zK>;oCcJ>NZ%lQ>0mVbWqhEZYM0Gd!IaU(E9zGg=Y;)K7e(Vmv0RvdQqy2q^Ht5!O-
=lI?`t6hS??B4n`Wu3{~s8anKYjKEYT0{t)&9wNh8F<0dvr@jx*Nu}LGQJ7EvM@~Qh)bN-)-a>S?y$KPZ)K@@lfqND%24HYgF>7n@JPI-_L%EZc}+!{$Ch55
0gmWZ#-!&>r#}-559X90nKW89dsN6q(9$MGyk%|&YF06;$2p=X>gygkX(VnDLk~eDCy-Lj;u9X*(nfH?Fw1p}SutVQ%ar%J0>5Y2Y|{P7rnOtiU2Z#%e@EwD
=?$syar9V}UCSH@5%K~f<m5SbMM188;s4;V{MNk<XmV*|wHbklcY<xD#x^GRGVrSr06pE1leZ6;@+Wm*R%}+5DZ~ijwO~YGxelrnV8Q~8ERY?y5P)^(rI618
WJv4UH|s1t(VNMiM8t?kpeoWr+Ew6Pj{$Vf{FC`W?lr(@Gk<p|?%+P*Vh@S_qMImPZTn{0(Xc}JM7esJ5W+vujhv+aSkhuvkiw-}j3o!Tkmr9S;q)wqbL5Q>
o3rsFxrtqEys?7Y0s4iilycr#H81iFdH;lk3>+eV=ujYib^xlBGRx>ybiALKe!q?C013fw$T;?>*7_bI%Kth>v4&qP11MjB-raxB*ui)oV3hc*t~^R?zflcr
V0MrL<U?y3mVWsZ&Z1(IS$<{%%e#q~nTY=U?#w3nHB-noXEx9aHH3OkCGejJ!)*U-`0I6q#4L6xdi`VND$CmvKv!8Eel<13eiKPj9((AIas%SM!Oh7n4Np4d
_pdyR`kvu>Q{=C6AmaGXZ``{S7W)O73D8)*i1#$85c@jn$9bOZB33oWhuYQ(m&`)AC!I;69=;B;?@E#swzpa`=7+!_t~Uo<9@?92B=4+IAWBR!K>Cl60%Qd{
BY<q1`!-{!^6Ckf`oa7zKm_Chij&(6TdxUKD|Q5ZVnJ}QB`F+k|Hev?;QGnZo2fV8S#=pTZ{CsRzymmZJz~dw&<_KE77#YeD;7(dsrr!EpT|)BbK}hYXxq~m
RwFIN3@R%6i}P~<&a1(yV;H48u`=EB1GXuHrsE>grs$14uwRXL*O=Q7#$Bt!8*AnT@rRj^+($?a9m}sr29GCO#!frCLl35?mEp+7$8mC(m}*|g34X0s7F#fc
jb3LY%>TD?X8+<5zN>xGJhw}2#1ZPe_fd>RTX~WrhZTY9J%{NuT!XdafJSukwa`BmZ|rOEW1#=G8)bR6-}@-s`3Yztu%JtBMu6^;v{3tjvly5F4GXh?d7xiR
L~?``eoMsaqeHrhC>uTlnPNLmIC|fQuP~@T7HjkcKtd0?N@Wv-wTKT4<3FzeeZBxMxgKgB!M2La+LKsV-em2O$@V+ltA)%<V{XD$5iJF9V0QWyeh|b(8k<XV
KCCHmMTx0v7zkd>ES{VT?e`@2XDgt&66B_I4Dqn%QG#%V)jOT9R;e?0U~?hv{$srjH)q0oT(AGA;3m{`goO7Qr{AG^vO0c}6uUR&zt&!76+fSM-99;Yvi_<G
8OubQmH6~xS{dsA+kuWM44V9#$!J7xfe$TLd}psL&#tsTxd;KpEIQ@U^i~khGI^xi$)cn|0wx%edFJH1d{qS;BcM*O_kp#y1hHDbyW;l{)MtL@rBk;?9!j+N
AbIpQ8+J8OMz{K$w^ku_rCd8mjyFM?fYkC25_6O^Yf&9;-rLc<x-T3QNnsuYirx(k_&v|YF!g2KVHksl_2qX|6o=&-_Q;>rZ5P)*2gk?;Hg<gJ8BpM~*WlTV
8jW_Ne|N~&ZIm)&`8Im^6Tuw<d*(b<YUx4R9>S1pmeGYNB<G@qf6JXSjNAMJkLGgE-r(m$fNpv`O329YnPJh9xs)AEM*}@f-@Fpl{npO26$DpCaD(V;+}P>@
P>FA}oj1~tQJf>|GSUFk-TYs^MWM<!up7=`3k{x5Md}<O3kt#h=N+(=ky8&V%+tL(BR3;iU?_NO?R^3)X-<sB)O-6o$bR|NaY34aFjfgle-tU8hqYXW;^??x
wn>&+$%_#ok7aztMSz-`7cM_R(3`sE2TMc?*pMQmY=w|E=O%g4)KOn{*P02#msDnj15p;$5`2#uH@rj(f8_lBnQR0S3hGEqrJCgp^~?l1dYl)4E&gxq9vlM%
g+QABi_ktj{ZT<bp1WN_1pj&rNdV89Q?r3i;cApPLmP<Zx7}f!;cR9;X%o>581Y03wxRv2dy#et!XzL&>bSms9=WK0)W9#q1jGq~Vc!!k(O!(96{7@GU0~O<
he3lK&0zdMq@mdbno-AJz}J4Kq*{WT?xOluniJyj74vyOtU}UBlp^t-ab-pa-9!Y)wrK|P;`}BCAmkY5sA>>tSgUCr@Kyh*zpaG8<jdMwj=J;)84TX7y&(S2
BRHVoxV~tV`ZmxKwyPFOZ54B0Mjijt<JDD+s8Qv}&)5<uSI5cdhq9M)GjfZj59cMXSWZ`sB?w?K1&#)LQgV;mwm*}94!}~s;N|=q{25UFg3fmW%;?F3O+^Z5
{|jld4(qtEX<855Hf4rgUu}Z?*MdV_Omer~Q~L5`%mdo%u{T&Aet>^2YbtcYKY{fO(Y|x-U{fl^+>vc|X3*R{+sSbK)#Zau2^t2Rv+Prck5q^1;XM!&nao2d
-L{7#D)!<TZ==@7Mws=WlOm1&)VoM3*78bp(fVOZ07IHine_{!Y(i|RseXf4Y$+GsZ@WbD!&xgj7T8AU0!2NEsZ+8GTZ>Om_@i!UcD-BS@X;F6EA%eYZE2pE
GpQqv+GKIyR;)H#@1?;7X~ufo*b4x|D4sHC4VX94<=d`AmYQxJGpVbY(wDT&lez~%^bX=E><@irLZULwxB83C&02h<(#`gJq)4)`)0eqc$L|fcH(}c3<z+UW
2r8_>*UOPP6I>d#qEiICt|CyH9i2!7si|Bw31C+lq_2w!Hz}RPFrFx%N~b<?Rs}e|2D%%!)?DR$x;gr+IjCxezsn9?7le-Fo@BwTF!d#0>W<{OJSviIR7qYY
Dg9`NA3pT#*ppF!*OIOdoBdFq&k&%_J9jOeQh$YzEqk{Ah9AOFRu&ODbh%3t3k8if&#<%1nrdL*H5NYwNfEfg2Etf)=-&FuC=O3fm&zbTx1S7^{Iou>-g$m0
`8*>+)LrXN-1fD(ykxd+bepe1lf1H~_`8-fRCeuFyKEcn3bhwW=Q?26QCK57=H@NdKr~(W#9lT$!Le!#KmsUS6%qAOGD_?kD>hl_;+wW^(o#imc#>EEAJ6KY
*fsE4D5^XtRk^fep+km--bwaefEbywDsPYWliQd6aGxDwacrHI4ukIM4%xqn#K8f(DJtWzh)!zC^@4sdK~x%Sh2-XFZKy_3$Vs&mD><g4w;Uj|tL2OI1QVpD
VI$RZQ-}n#-+Cb~b@_6kc^3P7g?ri7rCAt`pdG`|PvWnp=C60%CH?uVYu&2L+nZ~ECX2rz(U~<Pk@!>_)$4jXh0uCtrrHdwnCL(d`Z0e(rWm?Tu{We=AM5^u
f}(EGJgA}P$oIgkG1}7fC-Ew3)-NFaVc9U~bbY^KD@f<xVoCnO38}V>>z{gyzyEVA>N&+@X@yrAm!P1Gke&74jV5vQz7pk29aZ?+T>!zV{4Q6yq%+OX*fj(j
!+ny5D9r8{oG^7CxdgG_bK#V6y<M)Z6kNDV$HWhRzcp@Cg-IVUfPy^=wMQWBMXD|qRH0+#=2qAsF4NOS5#9=lpw5P(9e>zrOTcK5Urd=y&|OZ~tz{4vajZgs
Vk~)!x3j?QQy>5T21dTo)^85vpJ_q*uT}BCAhQhz0+fV)k`gDnHEq1au=RFt@sN#Nlz8@wL*Ka-C}$5(Z;tkwU}=MVz1PquL*Dk?IY*1$zH_jxyHX_+<^b8R
=2|?ai`~(Ua?xIKfM<})+Zf&c@q(=DNT995yq0!)9H4n!st)>l8LqOgXMeDFaRls{ogdS#g7an601qSGvWc3*Tzmj6iS8^)e(|c2>>(k9{aoZ7sIUN67+nMv
v3X`|(h63WCRI}guS#>%mlSYS&q{=tI1tQb8n&V#cs0!`K`etDi#^i6POB}*HznW01mZ3)9-XzmyP56Mm2c7QJemFMgZRg?{Ogb*FP1h?f^yD-pwNrcjnL|M
)tKgG)zT_P4I4+sdm@dds-B9`U{ME{N;OMX8uU9D%39zN5?sO!3g@zxsyjL(cx<96brFL4k5Oxd0MtLg;`o1Xt2)XF;~L-(fFiiKfOy1sV^1NT8>^iN33P$M
OV&9Uj*X><5HKZ3M(Lu<59u+ZmI5#O7=O)F)2yn-_LG+4w6zeNe;Cw;Bh3$5#^O(r)R{346{)W1BJ}%}QN~fMh|L+=6`w`A`y@*x!+G#!SaDNq1^q8K+ViVa
rqp)&`UL+-h?Eo=%6Gg=3|{-7LeR)yO))*Jz;h%ASbYU$jTi#-c6}@~*{Uu8-=sq^7E}Haq~%h=sX{8iJZgJQ%u4c*Kk0|Rx}vDG5OLIyZlI=3U92Bs#{prI
k_Dh-#S)5u<cvVofcrdp{6`P!t+0n@XmqRMo5+VUI7=FO7pKe>p}(R@CG5Dov#vE~GPQ@X)m5CPJ8(HWfFU%5IWRyV(xmhKtdv`~nop)e)**-T70f*#K%Fur
`QI+d0)LzSV$~59o^$~%d4y@2%7;b|EqFl)n7}T<gk5Ku$W2WaTW8{|U12@LOX(53pUirH?|U8e<;y>2st~^)ZX@QV+Zw0S{u88!C)k#BR@X9VXbgelX&Imn
s&DC!Ti#tZ+CmOJOVsn<SsZmnRid5DrqCxc3H=8x2g~pC%U3NrBFZ;bBq-f&ldm^gHX8rX<4jX=B{mkN(kWN01KR?PMs|-=Bpm&(HBS0nuhW-#Yugs7bAh{X
;T>-e(D|Y%@EkgPOmY9F3j4(01rzWm1t}s#LF!f=(@d2pl?RI@0#JTk6!Cm%U%}e2tLr$VJ$enV75dID%=WUZ9mrPtZ<dDFX`vRDyhcx91j1|+fZ><FdOYal
gh>&I^(X}v<w9ktA#++*)~MTg3$ls0T*>13D}00hh%sApV2jG&9D1~NJy#Km+2g~36ljGe`5#R#8UP+DzOYK#QuJX*Zzw-3nTx$7nhf9wfJ9r8V5QwwBqj%z
jhtyjCB%KQxSb{Z*0=CU?@Vn<_R-$Edmj<M$SYbfoM#}=D<LcS{3E9-U}Is`F$O78ix(XF1jXyB>f*)#wGVn`<k^B6(e#6)@}EQRCYp5XcDuA_3neCxZENja
bG5%{H2yqsHf%2cbJMH8mi%dstlC0Cj`SYu<w9Xx%6yV>tS(YLnGja%WNs1s<AdPU87yblhCwrC0DQ5QJO#Ks8cy=mYHwhp5p4AS{SK)~=CxY5Yctp<E!k^!
WHNP|J^9bLH~K$MTrSL<wPIYSZ{(M)ldLF33{(sNy;mmIj`Jw}=oi0KdG}B}wz^uhAWwQ1eUdarx1Z7^UVw0$F;^S4vXs2bmb;U0uKXDj!*GDntdWY(EOu61
uh6fNII@I!S-hEhq7l5x)?RjT&`nxTYlTu2;CzuL;n-Z2s0>Ug5DxH6G#>PG)79`1yg9RSc8=$6RrpqA0>&!WH)C}IZc{t|A)5}(ghV@v&Fu)KWxBIq7vx@o
-?PqjKodZALa7~M^TJ+^woSWfl=dCUhl;ue3Hm)!&p(8uv8DD5-bhF!1t6|h>&YnS4C@6q71*HoAhR$Clbah(iudrkk?@Yvr<b|mCu9*0k_{2diJ4Pc4tL|q
W`ZSUPbh_1M$AU6qDIR|F5!?C8{BaGc|1tm<WP+`mwI|8lDIN_X`gGaIb!a=c$~VOo-NN|xK&YQLqmnw45nW|8htJOZy9qu5Zs4#`H3;i=AQ9)5J$$z8s`XU
ojVn5?5R`7W6SH`l@xm$fl}4(vq#-e$D7ZNDj+E;4zrUj50MNO$duUawuq%SUx67@28=!R*bbUFAM*u1YgO!H6CxtW)Cn(SWyGW9(pO_btn-K~BHsWp=oUzR
*lK8LL{59SpBrN?L$hP$3uo#kPL%~qhUr48;yFK=f*#gQ8Nx$FR+-lb+*W}(M6HzL@!Z4w;fw58t^kbA_!k@&-~9UmMVTMIOAB)q2%hiFzk=MQ&=8+2epG%)
tkb3*b}J+?p^9$jT3;FHtXW$8nt;RE(O>?MXF#H3%svq)Cfr+?*rz><pCw|9LJjes&{c^P%~YBK=_G?EHP+6bg^FaGrCTa5PI63kLQdR-E7FMgfoyo^{dp10
jgV-ulz;8Y(3>laZFjlME*kYjc;@@(nTxIr#4WTr6v<-fbHl{MV||D{Sv7_IOkcD+e#~A2l|@El(uzEfj^w|1AjNrv8YjVL0@$({0OoU#^+JLfx<LbXv)sJ8
;7a+k>f52oz$&lj#TW|c)(VXhA)6{^;^&4=8z1z(Xf=GSqZ;O2hF~p9XMNaE+3_~C1=En((px6X9}ul>kwHwIVPXd|e-CMf`Ab01vJi-Gl>c{gn%P<DMcsqC
G0r9y`0xXIVeT!v&5#de@}=#yGBqO6<^NML$lJxWn%_toDx_EM-dPaD8xukN@4OC)A%vC~DIGaKFHJvQ9pLm3`T%^o+hm;nCRi^6;Z%c05JNGUjtd^9Pm1&L
a*JnES)Pk6XW|7gdM+(3JNch{-^!eoe#|<PRd^q>(~&lFv+XGb2=1}s3NcUX0gqQ-siugR<rzOtkd=)sifF#S(CYunO{;6p9fM=EkFAvsiwYSF{e3&>YpC&Q
y!F$lhQJH;w~}O0p)HB0<<W3Dl1W5_v&H=HA~(@J%)grOGtM5Gq7L%PI<shmmm7SV-W!RL&0xL~m^^ufp|poy`}{{9n=4o>7HPSCN`KeU>Ibv{;84#2&~QT4
;JBRnn{rg49M<t??I%9$Cnikip1fl2I}aW`ZO)97T_G7u*#~GFaY=Xx>@E~uI?$;R${beI-7;7syKD*9&*OGfGL%EQ=D$xm+tAf(3)6NB4T2QxEoNuWg#N)@
5nGbc@`TP_gDukrhj&g&P7zf8C{J$r4{CgX?beW~Ab#Dc^sB7}{GDDfE!gc<<58e}aAI5X;R{Poo)Ry3S@3$qg{v+qJZSfx8*fnU7O(-ECi>zU-OyRVgm8R2
tLMcjD{Nh@bL`;<vKb<0o)39c&Fae}@x;jh3-FpJUIoqXJK)+E&&~%yEC;aS<0}*a|MYKvA_GIK&-y`#6PoC4Ln-wM_=WGWkF)_NmS<c1iEtyC9EMjiw`gfq
>{4}1tw4&cP0#6TV=3^RfvV~mM(wKu3Tka4^4g}>&j|(`Tfj|+A`6%I&rbc1KF}afR1xEr%yA?`Gd7uNRp#F&<*XfHyB`_o*BA`y?bPWUG685)gYu@k0veB%
+k$G0E+QqKn*QdMF&GNKd&}mOmMaihZeIi`mXd<w;u4AKNLJKn62p_RN)rJi5Bba|-sp82J~6qVe`5gpK{<g`Um8At5(_qMg=N;9+HmvH8k$uY8Z5IC9R=Fu
aSYGp3S-$mcp3xW-bb3*xQ2f%%l7G+T^zr%r;U1q_WClBx_ko<VE2bJrb*Q-p415q!~%I!#eWN64B)!*z!a(#wuyUnwI4{xEL)%Mv~5;cKcCAfsE-~Vi}1%$
DypbaV6OCal_eg}4C}k8Q|e2;E9l6%6eGOZK{IU!td{@=ES%Sw_{gE%@niD_&#Xa=YTuM;S~64|MME9zBN=NlC@@CPV}IHS1O}!4jLY%DHLDk&FJW02_*YU)
*Jv6CfS$`GICWLx_=8P|hRKb1)OxyoW}GMYLK|Nq*9M_pc_UN9==hYs{3E{a`NPoqDKacNH*j|GERq>Ql!MKs6kBPjol+Lq^~G=dPezrH@AsCZI1sn>$3~5H
f<LBn7`eu?i`xZ)SSf%cNECm(D;S8MWd<|sfg>HYVJQudC9v4GoL8I|xgyOuuJ93P<lHyUEZaq8!LLM9SG)p;BK{-!9$!<*{KlENG|#dba1~WdHven+J3jO{
J|^?V!s0JobGOTDVDBUjR2dnIU=$P^dampv9>;;tx$I2O-h3=0{m-krrSh}_loxQe-n15WEDv-IGj{Ijz-hdb*2)W55O1t8kQ$Uq;m#m%lK{xp8m6$(=F$Zm
!%5b|zE-9RTOD;$fhut-d}7wCex^hpUo~9BO@6~7@Oq}-Y~2?wwdmW|acgud+;P0PI1a7IZ_jPNOkmW)gPYtgKt5{NO`Xkgjr4VXdU|wyj<T9#2AZb0jfGUW
04WPp0^IGEMu*g8mR<NILpUEroq&qVs;?G9mi)5wg~Yv?Wm_7*TrrwLr{o;kc72w;j!;`3*{y5#4ZQfGiOWtNQe>j$Xyr~=L8u_hR}@vWkUac1tRCV;tkM4W
lh5H&g@mO7H=sjyY^jq5PUT{F=mO}HopVp!!i<@O(ckWeUbE&9f|f5?Tasum3^K;my=53_I$2Eu#fw}#K*F4ggVK?C?uZ+{6Efp(W=1J$YbMX8UlJN#!N<lO
9!$P{$coTe!~Kg2HRAed;T!_uL|tztN2pAczyf&nMaD(S{*_$<i%+0~ml`f0b14W_R|Di=vQ?rEZ0Cd<3_Q>wVKkD6yhfB9qD6oQ7fz*=&oq+l;E-%bvj7r3
kAQ(-9>#8clsCN(>AQf^CeE2aqp(R<Rg*d4txFFGhwFUEhE1H~9!PdsPr&jufyO-);GAIn+TdN+x1+8-uL`sAh&H<Xz=5~G&+1Mda6#|>^N@dne45d*gMx6f
PEjKWR9_oBQgB&j;$d`OEZP-|Nf!`8#*zPD3~@HCU_Dt5h*ex$_nB8uBlDtRsNC|Y&d^<*s_gaPvk->t3Z;zJ;;Wce*QED$n4D0%V}iZWlIF4dbllzm>OU#0
!<a7DxPs$V$2Z>222x%IhdrT{OK$x$2>zvb1N^-FB52BiGbkng!^>owIk!4VTdNhbf?3Z-qnxO|fdu1s0=EQ^Z9nXawMAPiWA<snVq_6wNag9d%(t@ILq!G_
;C(d1Hioz5mnMYt5mjBr!8PC;m$Z@kUt1C(p!=6Anvg9I(cfe;rPb{>kgb18>{a(?Bg7hXBrQHjCLs1)esgT`joA<0$SkvrKe0O8jPKwD0N7cN1B>jU_HYzm
@ev6j6}aBNxAxJw<`fm{MutmTrq{`Az_v00Vl}Kz3CEt$k~n%)>S#vkN~(6V8U1N42s1QP98CM0ZKN`P^Xn}Lc^!Ok6p;BfEo4j}ReSP<n4)+D-30{{e+GZ=
qg>pQs&f+@lO*e(JNA&M(-UZw7LXOMZu-UvrAXt)G(V%SX^{hN{>;?;<>zCkL@~Dur8>j?9Gv6I-EA?^bp8Q8C)I5v)A2ngK`}%1-lY9DqYXg=hu#nJ>GSG4
?EK&r?1u?n;U3HmT*RVn5^=r|+~6WAx(*yNmaMxaz`J2|+xvnv+5mP<{9wi+erxLjIV3reW@b<xf7TO&=lr)_l==yOwBxat1V0E%Pj|XC90()IzN@Q@u804c
KTgO^12p)vSw6#S%Z}}b<OI=cSu5U;0p{IYgCQNMI*Yxyac4T3{Cf9t^GuVoI87XYMpz=_M_Y>hh&VcN)#ma}Z4u6Bu9W_hov|~BsH9lJB$-3l6<>6!_G+(U
=Y@CgS|%FY-!Zk`qNQ_*Pnh!b=R}rB+1IW`A<ZPajH+=IsL5(U-e;xG)NrW2W1mS}5YwYkf3UjU##7+u(3!nzH^cW^;GK|U+NY!mI<TMt2GjutAD59_l#QhI
L2<v6VI9C0b6}PJA*Aq_aS)KXSs45OJNU{F72avkn!l|TIPvlC64enhP(*>fI9PG~YX}i-1}mrN*}m8Rd(^`sWpJQCUQU_EX|&wbq9I;Dh;XoS(tcJb1jfml
Y2x~2#4)E$x~uBh^kAFB&2fwdHId^5L-XU{%j=Uduoh4qJ-DAeUCVY4`C<7nMMnzQKcbat?E8RrFVC#Trhl{E!J(cUTCC5|Y%|uC*Zz><WFm-FxG=iDk$Q(@
`Kv#fT=;}%J9+i(1Ts+L&Ons6^Ucu^vhc3*&owKa7Ixe?Ey#n(AL}@L842*_$)B8*0*9*{Vv+~w?^Wp4G}^;T<xG<91^522IpKzxY&uKk(8$RA8I7pgABhJ>
!f#+n$|18G>$&wERVXkrDSH2@SY%p+-rVJhF0Mc9*8gYq2RfUU*L5~!v}_$;=%E<E)#_F9>q!#kgAyPXq{Z1S-K+~2vOF>-&mo_hyp;IJ-aG)rv)*_TR)nX!
_~a=ZPTvUxfFVhJlkVuLRj~fORJw`J2jLy~T)!nP_%X89!xoKeAnH`qrHo*Ps1AhtGIP4q>34#my<^MSw86m?QEqlr4+GdD_lY-<tu=IhX}?3zoK%BMDjj=~
T!cr?Jnr-fT)NNBp1|!+1NR$BS&7lxssH@O30d^Wb?dc4!QIL7xx2?DpXHvr@xK18fS{D{DO3@B`=>xgfuZSKfT)#GZy{aWnv*ewbH|3JZ1LfScrDL?8~tQZ
$U4Vj33jt{w^UJ=SZ0+a+JSzL&|hk?nmU(QX<p)TxXZPWMf{NH^{*s1ErwaZ1g(|s;GL~Y(Qd8YNYSyOj2GPqXQ3DU<(^CrUCm+%BbNN`u|d~T!<PrCV@DL%
2V0DihozhDU-C-RI7{fX7R;shQTqF=e32W!cAZLSTwF7=V7OEsW{bq37LgHdoB)-hr*FF_)S=3Jqsk!U#5zZ19V!%ro)HqFh~F)WZG^~*8&1mK^r-nZDo5YQ
KN(r8GfvdOLxr1A;q6Tf5MH~W6!s5+G2KyAH$l30r;20EL_hh3)i(0Ef)Kk?47$kKV#c5)7S5GREj9&H>@7!@cT&2rL}kDjP^&v~R<d&==(5j|L&Z7Pk3QHk
%F`2u#7AVf9y@mEs~zmQE|;*$g?W{X{Wkz1RedH(F71GUBc7g>@{!QV5h@PBU)D-41Md>`dD55%3UXXp#iOqDv$AvQ$>tEzcG0)00C?tYOL<&gHa||s%;K?l
{%Qs^$XsapqL?ni+d!in=asI$-1XNpPLW7D;jGx)+rvRKx~e;0L-{rq2F^3yHW{ZzK*h`{tAOLL4dLmlZW|O0k0HM{`^6ayB^dUJ*75<X?(xiuZ!}pEz*9`X
S((ec`+1`fBzUKcI%k=>9OlwQ@OLQM6@2mSu9qT)>|VH{$XGa^)$bTZIn<xFJ$rfTQO!OOcw!Q5$K#~0y3q$>i{U20^ZPxKdK?x8FTN1?%rU+{T?=@drGHCz
k{&M=-I#({WCv_&C<LFl$}XmctMttebH_lpWOdj0sZATGGw@$n;0c!VIhLCxtLjq3VgBbtA3)!Lh;erHCAL?KCLd`%#MXF4i!Yd@C)0!6mguw30K{Roe`4uT
X2-*^P*{|UVwzKVFOQ7os-!z3KN41Sq8nIRI&UORK-n_4s)I(g(98|KI@GXu%+~HRzT+M)hKa6x`7fno!~mz>^;+gY(8>4LIz7lwGG6C+Ff(_v><@*O;Zr<l
6E6i0Dn89r0rX-^v@e;LSrrH*p7Ta$3Wt4X9M|OBkr;LVH*`&Zk*0oTYp2F8$Jw~Fzv1hd&@m(WHFHyOAqH9s#^*c2F*y&B<^}X1K%e&D*rE!FMh*hA&^yM}
)EW<c>ZlZFoXmJ|1d6XGM74q_mJ)&iTRA$&UGd-|yo4JutzAOT@gcDSgj1O#gkHXsj9@V@2RhtiGZSq5&ZZNO>tQH~Fn@SQr)B4WoP-c-WxYr<Gy=<F1K~YR
DmY2d(iSb@{D(nY5&EosbFzBJBBSA7u_c?m7|(i4A7r>UI}u@sVemyQTno0#e)_~LKez6Owsxn5mOy#W6f*6HsJjFY>gS3R=nvw8v>8_x+rVl-z#2`{*{n8@
ZjrQl5;lMH3a|qr&+3;VC-#Qf;-S!rxyRu3sM5fw_CsE>N1-K5p$Q6tay$r#Pr!W$bbRsI;nadLC{T&yziN!sY&8Jx!-AyFMA<sbM&bkt+xY@rCN@(k<N}sH
2p!%)Kvf29;iz7(3Y%ju_7w{ODlEIX7(EJ2IBuP~h>3pqsQA&2(Apa~np{G`cZO7<XhiAA`9GHEJQN{*Y9MS7sN%=y%yC)1-n|UYvfXk*SI!m&Uk!uh;PAK2
XrUd1|Gw`B4sjx3B}JMHhBP%JD#k0@1jft<y$!o7C?m{dzA{SsF|pQqLC4$Ywp~Da7y(nHxCJVU0MAreH@TuTv@-~tmn*lrsQEyW(00q~$dDkS=FOe{Uw0VG
3Gpdq(}zMF(S;zp%xV3Td)ZP5+N-L~AvhMgsMjp%?EK%dPIifCd-ZoKloR)3Rwr&p2npO>ISmFf3tZEABsHb*XGbzPAO~fQ%S&)eQa67c!d)}~mMUD=cp9EI
0A$bU7M4{=(vuw{v8K;WF%vAcJLm!E#xrIq56yhwW%#Gi<e?&3E5;DG@f1Bh<+$@Zp+ewLu0yLwfQoRp(64Td@Zse|h)xKmr;B`t<(sAxP8hTdoI#-T8Qh?@
%<gM7{A-h0Bq2(;`UYMaG?7ONRFcKqmr+7;O59KVZzAwvzcn_>?ImQRuEV^l%WAOZSIDh4t<2%{B&4};r;8&(etZ2x8o$ngXfF=%Gdt8HIc}r(z*s(LdcA1v
xyRj^ECJ;9mQ~l{JkpQiGb}U{r4=kMvn;V(y@Ys`Ow4Uzi?E2v(FEi)3bVWkRZ^NjURQ2<#ND{8Ft<Dq@gtLJhHSG)w`R!(8E~VD-4Vjgw=4YVe5I1goYmR?
xAe!l+(9J=37=p?08D-pZBV^heiW{9n?H=x2-h)CfQY(T)|w>PNe&tWfadUuupmj^@zEpzUK$emTmf$H6G_=49R`AdoK=Lf#07qpLFMbJmp#3<)-p%}ohO!}
9$0L#N+QPOhsH$%c5m&hoWr>1@WGWJE`iN-oKEIz9nXK&EEbyU%u-m6<!Qz`l<SRH;|>F^ZJo`-t~G;vJ-H7|#s2LwnRKP7&SXjK?S7_7iPLmKE?FDlC?U72
dgeCYQoNUf%*1JP6?C105xclr4`^f<s_Xs0mF|qc5`!iD>(`g&2earl^ejAe^x|-#g)7yvQY!3Pa386PEF0ClQJRb{jXQ=AE<4*pPSo|e$y#>N${<Q~0qbX|
IZoqOZ7=|kjD0Nh`#c{}JtS{YG)KP7+q*A*d1USjsJITsn)T?A5#A1Py>ZzTkIpW9J@$~cz$!JBESK+3K1N2A=|dd8MNJRKQ1-mrH#?Kbh?rkQ|MC>*Zb+Y0
^(&f8_kjt3G{313T{U@(h}I_3C3N?XqwC~1W1NhlSXbP6z$3De(*XFhNZ`J~7PS;3E3e?k-hZsT!5NFx{=&Hzy`;gs*7_brP5k$8+}}%Usw*GUJ66|8uHQ~5
y*y5GPlS-LA&=V~XI0~PE6m106-)~~_WbdKd#;+}l&UGLiqLq{Av&|eIq%FuvGV+ZFI3nm!t=*fDjoGqvemUBkeduxd1;Erx=X7Tf3YN~z-5`#rpvP%1hEL(
?x}{Uf{ZY(OgHW>cLbyN)BvO4vou`?WEx5?;0%zcB;;`1=F&bXw{-xHoHRSN6m-Ll?2Q5QrUQ^L@+p`?A&~wDcn$@XPVv!}M{9?A97-wG^F>OFhUu_*badTT
7C|0!kxwKb@jE>etg<n`ULtUj)oqhtvLBpZsXhTU00=r|<drmhAYDRFZxm-kfx|;F14hy~n1iz7hfc+gj<*f7q&P>e&m}Sq^GQ`QR8kVzNWk-^vv#|$RN)ot
=PJsz62!1Pi6~ywu%FUuMX~@&`HP^vLM%&SwUgO|tQrF0lg4C#{_G>`ySnu`GvCqD36e9E@RlZBU4|t@aPC?&VbH0pj*1h3B%_*S(g-OooTjPLOPdP+ZL|(4
y%j1_lCD*z2d&(jUu7ar7>c5pgL5?qHEF+^TiW{vZQCO<8Z2bc+3OsB@GtnUJQQv)G$V6FAsV*_(sdirnvta3R(bq(E+m_s6u-rTZUCRS!**n#ltc0_FH1Wl
@NwBhN8AjyCwmvvUjEFr<r{uP{PYeSz1QIj;0H1M7A|Yh`cxxmqcW@7-#Qy@j&AE4@h@XrEx{VS$tlG#sgDne6(O)ys~bjcf>15aXy}U1dNzCh7vD#fb2LjH
GM)oo{-?1A3(rY(KXK2U2&Ge}TL9Ydw(dJb2ezO;o?V{ROq~yr+0{lMnf7*Kz^o1O)&aJn(9=p2I%PeiK@QIPde%b^We;As&#hv3H)U{B$Z9%dG+GuSoLM>T
eH|<^(zXyv?Yrhb4}yh$>kiKAnaHPW9a=6w^Z+oS@|~5d7OSv>Ub_=!26W_+6eSp|FyjLggG8gq#@%ntxlx_rnHZ<5L`kUS9R(uh0qwvg*)X+;u#YsqXp80p
V~IN26JqV1xhjur9kp7ovya{ZaIDTW0o$a@?o0VJ)Mf)D1oonJL&Jx|^WHVy-eG4zw!av;2(iu)EyPmMi8ea1&a(JUw=VDTKq6Qq34qfI;$R*Z>1IB^q?Wxt
b8LW=Z}y^)fdarij>tT9QWf^?Ro8;QUmr0UA$*1bm7M`WF|l^GEQ1N8nu>AEol*s6Zm)f|;#sADILNxT`|8NZ5wJ4^&ehcm#J)EpT|12*uFc$~`))cM*XrH6
(=U8WusikP(oEO{HR4KiNcIAkn78qEA?zG1jpCC0P-RTdOysnk=>v8NjP7hMh247%=`{hJfwVIzJ%}f_=dB53!T!-*wnA^VBKM^DRrn5=Bo~YG1I8oQ^e^?X
)Yq<o?8jaj3Bo-^?I2X^*MPRo5Vt-WX$D#-eeP6|PD`2J<NP67YE!D6KF3Ry?db6WuC!t<x#gu#j&hwhDSl1Yre|45S#jiwN9W+dPXRI2%<VP4p5+YG^F!n~
h-iEC%hY<EqNN#-f>vBBBZcaMn)Z+>aj9}Zb(}^iUUC=T(rXaD^-vUsKNwaX8KhIx^r`)_+DUD3RNsSBfI!`dgG|b0<c%#a$w@t*N9$MgA-rYo9e)biL&?dQ
^rSj6%l`mFDjE>3a-W}$aR-+US|iQ%-^IJy;HC2@1s=dWK8ft2TQqE;)99#aG|}DKEH5a#Ocd<JBJhSZg9&UnoF^oK4{5_*lcy(i`^~|u0sKbG00B8b&SmNE
Wj&QvSzP~EkB+!36m#d~0q`MERut0hAPMPyPX}tlD9ZCALIBP9=MKlrmUnsu6i=XaK=U;{m2jGzs*tXtgQ24B9k*csgg)FWB5d8gDkZ!?_Fo8|p0&e|bKpp*
|L>};hQzn%l6Yl>!MyzcTbijcOMKIEA_==#A@CX`oDU}L2oJ(7CczuJ&2+XpF#za$W|0rYGzC{L2frZe%LfH(Y4H8oHQHU*K$`c3%hkusBAEpR><|H}bG;F5
ZbuC=?>blZA=RnP>1S|I;{zO~-MZHv-~xK{4I3ovFe-9l`1$59R+5-s#$NCzgZNFO(e=u>@lu(AOMW-vv2~FW#I+w}^_mdiC14+bM&AXHAx0!01k1`ahkzZ5
`iOSJ2_F;k>87%L7@MHA?mkich>H!qv6_K)Zp{auFm-1A=uEHY1{6mb*A|IwFaD10&Eglj{llrmCK$<_P9=d|(0jU$8=4vuOvu8N+Rqh1CHuc)ml!20uZ2K3
GcS28tdS=@bkF%`M;5ZVpZ$gk#$<aRm}Bof6X?A^i;uAgIyqNCZQ7lHE4l9<sVFfe4N~z@Q+MGEZ(Zr@T|r<1boD!uO0?(oO_=|Ko%%G;(-^m8zu<I)oR*3h
_|y$}>*w$)yUbEx7SK+M25JIu|DW>tG-#C5JZNYHMB!~+wCwOx3Dy6q4ooF~nDz*M$lqJIiBj0;6%P}xh_xZdjaF4|m5xgO6khz8$2q>E*(PYoK0cOnmidz|
Jn6v11-&4@-z@gvv7NMfFHMyaCRAS~ho9xc>EWr^RM4P~QUWVd)<?w&0><6SRoF($^uA^Ax0A?$t;jgFUCOc1Xxpe-p4?J#lc!^|FG06m)Qx}Breg}5RM2F!
!T#Kib*bk)Y|7Q2eJShCBZ9H%#$~5xBIjCuZGhRexn^R}S7BtNp1t&lTQUpLjMYw`adv~~L-UJ{Sq(I;b*GD&iFv~=2SB5_ay-<AfA|m0Mqc^V_QOgV1dV19
pkcp)G(DW$m06Y8u9~CIR?FBm_@{q#cC0=y{|m}Y&?cBT8ZS0H4jxo~FFi}KEEzzVc<J6IEyxKTLX_MDqQEHKv=T+NMTII~sn;@=v0jcJGIWFw{8B^*G=#t@
d~G-|&j`FM%c0DMME^yqXvq#&vD<tuLtoDO7GBW~XTXhDtu{3NNM5g7p05_2&mKO{KC32Y9lifo-TCq~fpmbmyw<Gl!H5g_O)%+Do*x7*0@{jGMhg&E+<LKS
#^D5ww3T*pOs#&q@9g(KDkHy}9?@MeAv*>@ya!q`!oI(#aVTzesrC>6-W_`g=KMnu{EO|clFh6x1*Bk<+q4%~JIE5*CF3koRE~(9t31v+n(Q8QHx-V{8@UZ?
d&l*smAe(h=8jbG(<9-gjzS&26l4J;o~e#g0Ca+{Hmaqm*B?Q>anXsmvK~aHL!rO33~-(dt!|Tvz|pR#vc}@tR>>T9#KAf*PXll6ihaZiV!~}8d+JOrx@i8a
UZOTEBc4e`yAauXwUmvRI^4mz;o4v(e=$5X{cqJ0KVJXXfDuAq-8>nAK0lrs+#C1_W5Kx5CaP7eqS|#AuZk_9bYgo8Itax|FW&o*iUyb>D5KO`5d}$*>f<7%
aiIm>Z^qrUx1GEg7xN$|Z}7l68dMInR01_)lntR}N}+<Py=niqQ+=Dd#YK<!<Ns}-EvE`)cWVA?d2}PyWD`m%d5<zeaCiP~ONH)R$B3&7P(p4RU#<Fi7ja3o
0VpKPy=QS%|Hzu+F*(rqCz<=q0!Opi0$5O0mp<)LYq0S|rpGEa|Ls9Se-A~#;fe!ARRjp_Q77=-W^vF@Aj2*h{YEG&^WCFkvXwUlu&y%f#SIND`^Y;RQhVSx
4DfC+3_}R!y6mu{r<Q_xRS*AGz`;9+gJIzk$JA<C9G~Dz3pwsE46?(ZNF8K>?o^L}#$C3d)`QDUeP^&Yt6^1nt9yEDn*`%mzeH#~r7b;r|AmYv)V?^`v<t%W
b+{2Jh%%ZmmUZ*AJXXxUBY92-a4oE0$+OXYoW3+LPBA)kk8L%FarBOz0Y9imm*ERfdr<|`;u85cHZtM=NmZr0doRfyanb8y?;f0l50YhFwG_j8&#|d7#D(8W
W55gx<{6$Ug#2TBd!(*?ZUAzJv3_2unTm6=t(Ur6#y^uc=jLhk!wys*fBF1a($4S3JuR_*qFb-M%39L6K{iwrZ%XD@)NHv0f|BsOOsq_2ev-larVVXsVuDEr
)&ik`SsH1Tmht<wa>ByY`Q~{JSxWegn{ZcQGVflbq5po^<Xk>NojIBwd3KTJIE<@iwku_W(WWs?H`dWHKF4?GODH_3DQSPS<Z1SHO|>54l7?}{xi=JL$7=*z
(e6*$YTw4~vUx#p+W=|Q9BVZbqqIZIG0Q=Pkh*m79hmH-QsA%|3^&qAHMRS;;uF&e8JUpP-2-8OM16A82Ovi8C^A={A1;%;YpxUt95$?kCTR@?5zyIYaTH-Q
kev|yPC#zKA>&V^hNHV8L{JT*nS8cyW2C@dz~?$pbfk6r$$!>$HOScq7lF}(>#PCD(S}%+V9z!DLnmP43vs^qQ`&T@{z5v(V|Ho~<X6RbblV*D9U_KoUJcFX
Jc^6((L6xoy@45WWh@0SB!&%U;!zB)E0(bE#31-(4{8o&U<8w1AP>9G5q2Q1&yQf1F&%?^j0KcH=(XvM+j#6!tmmdRTfokb-w~!EGpLf@A-Eb|<c2ZC`0YE&
qA@jQUM6IWym$q{7VE@P?D40xbH}QG3P#Nb=YyV4=-CYJMYun@i*7=*1LuO8enTnY98utPudp&_OAcZZ9AP|5GtZnPl#x{B9qmcc4R;;!mfxgdI%`8opn<dm
Q>ivLMQ1P<iCs$E4DlVhe6t4do38(qIm9SF68Wz8CjS0g?sBaoYOKbZjy_lqSm+XlSIVxc@B-l+rAN~UE4J`!YacBibgp|2<-J*r)Wkzj%j6&LKaIUsQS2p4
_u8z_MS(a?Zimj~L12^Orf6Sx-$ir{|6!*=S{Xdt+d(nZtjR}P0j|H`UtGr}U)-%5W<Z;}!GW?BO;v%?<3EO;X9)d>EM6gfLD<Wnm8_k{cwXyaifKgevu(2P
%E&FdX;EUmVT>RT6F@MP)13}qw^H*+k%I2zpKqY%-NFvdxZWc{Rw1dgs~t+43@8Bapv!<a=L5ep%5tt2cvp$=`Gmr(_Hm41qN6@sJ8oLw20|)KojOMF!YNAr
75zI(=DYvfXZy;$lNM4}$uF}m*<|3;x*B33dP84XvxsH+H6=NG|Jlyzr0Ro3ZcLfJbXMo5w2hu?=Lg1>q=d<>zXfc5s}Jq_BEqfkG_qP)8!W$+rM41x3Q}3z
R51n3$^HJH*1F^9;#kQ=AIf1r+BJE4_C6j4JJck3@Tr8dH?ck*$)yA62!G|d5VgKV6!Ezi=lOu$V)Uc2cBf-r9#)k0yqb?0a(=a%DHz{GZ6K*g;L|JvYn&qI
q$?VI0EQWoNa!V6n;K-}=tKe(SIox-svM;OlO^(ZWW)QAjg1+rEM)CXeFNm+m7sNLqt^X5g=U&mC94l2iC}d~&U(OV)FlSIkX<9OUbQWa#N>gMt9G3=qLRfB
|Izm;7J#^Jac@jhUHm4XZoDQGi#<NX-Uum52{mTPk*RH>$w6w}3Cl{Ru6x^~$DBH&-wn^!_L*cgq~=G4|9BFnmqQ<vrmBBAp34j>Q3+PwHN%o|RX2T-u^-as
Tmi%0B!BC{x-dB6lDK<cRhPn%!K#|Hh$+O0SjC#DR?FQGJNR*y=07xRsA^JHgUNqp%{ky1i}DDNW)zeVt_s~F;&h+Dv?+^k{67^!{iHmqtMjlf1QQAUzT+yG
e(^(+UdyA2wfZ)0RQg8RrBrn8IcIV}zD277DOL2Img~Mk;8zrk!hiTe4^f*6VGVXC0=SE#?e|z8p4Nok=ZC};anC09Eq@w|sWq)vG)pDNYABaP8t$uN)&kev
sb$B4IA_ltXahX&qUzoH>*5}Hs#{+6hDiyxceqpHXv%(3?4p$*X9$q@$~?HiP6S$x?xh50@Z;kB;uXE^olY*65=U_r)Et}Sg8_9wlyIPgyms%*gjq986))+j
#+;pAxZ1b03Iy;sj0fai9hirSr<|_(Z24}xx8Nb7y3la4!^yhtj!B`hrndB1@--o600000edQ?f{6*|(00Hs20>J4W313-GvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_flavoria.csv')

In [ ]:
# listas/externo_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<JXN`bzJ~58cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ{j
khJT|E4KRKjFNUmimR`l^{%Nzi4CIj3Bjhg_*Tv2VeY<iML9W`cFNRMLkE<>-IR9vIAMD3jO(u34+3mEFoodfu6C14HvnZSzk+>!CDj!t2ML*@Xb3giH&9~m
QoV0RcX^R)4=3q(+3DD5k!dT@q?J2MM>>8k!Lh)J0LHf8;&3!@Z>LVT95$Lv?tYU#t9ZaC;hC@F%i;S&K9VDE+ffB-b`ohyc+)XrUb&zc^u;O?)yJs3H9oo{
4O$IJAuvi4|Dm`U|MBtlStsl@HCa4Eo~$yrN$W@UX--%mAW`jj9huojooOA#*g4Il&o=+?^LB^42}?M$21KtL)+5^ynxR*-yzLFg8rSZ0$BUs1-y6t&dfOB;
r-`QjV*qKirPpI!1paV$6Rdx@xIXRtn3??3GDvj53tHF+&D$hMGg7&C-npAgC}36Mlwz=_g4Byq%6~Q^b~BxGf6qb5Qh6e*YgSvOABn*ENP%{lveOz|6NLuR
CR^o)`pi*<%RIwf{c#;H!UOASD3_f_Zn0~lEz*+l=u9z}%|-T0x6SYZ$p$=<8n_)kW`&N_P!GY%81owTLlHt@Kjrh<kSP^7vUUaToC+<F!^Y^SZd35RoJ4D#
@@8t+3V5sCO0|y}pNQQ%8DgHA7T9=4(%7RbgCW9%R#6k6kPEC`h9+ZIgG$q2kJaP5(YzU7vz!nf7xHLd1CwyVJE98BaXJ6-airQz0&x_Nn+bYc10=<@MIuNq
to39|2!CY$ub*Xk@kDokNqE*8LbcU*HV4u8C7W=pS`s0!!If3TvMhHSQ=RrUn$9}Ku5|2sQze9($X6%<EV=6LGOQiLKJqGR{yFKj{6uZcvk5R;b4z8jOSaUh
7C73^Q5^Q_wc9Gk_4_mrAZpVEd-|N(;<s7lHa`xIO4?gRs~5gyEjYj}GFdrv**0|l5gNFE?Sc`=F*YtbAkPRB<)$lZpk5v@SLMx&iL*#TARKGh+bgAE!6c)$
AN=IOh5}w+-(}c=OsuwCiVoKHfX0_gB|syeAWm90LRXY9jhfaT$tQDU@^#3Umt<zwcxopeB0*tw>f6*dNGNt^_)KDpy*1w}ejSwEh!fytKw;a^GOgV-Bg-Kd
@4H`p_m#yhsuAaDPflpv!};@`dPb3fA$fF#`J_^6SD2H^kCA7a@VmNFW3t0Uwz#Hm4$iYIFZK179QZDS`>;B9I2k}qE)H5v(4b2O-g%E%<>V8lboGobDl_=b
YN24o0+xne^OfB6Tv*$UZ7$8;scSi0O*s{*8`cxyvu_a%ya;KuhwX@lq5P(L@6A^?k^))(;Tc6;v{<K?KZ5vw3fQHC%n{uK(IR|wR_rap<oh?Q>$$2uFpLr~
WxIN0<YF4FOzdX{;9?)Y>E?QpbMY)0qvrbmY3KWWKFi;|ylglW);UeCL1<3Dah=ylYH!=u@fo72^9wYWt$L-vzv!q<P+HQ|nkaD%kb6FAxpQLrW^oMX&nP01
J+<5(vm!cDUJPxxV-?lIoEmls0KaL-UqTv6#QLmpwbTSx_M|TUQ<;x#|Hsv^Vtv)GN`3pG@{C$5uYQJmo@+&1fS>gk+M%i~(In6713BqR3i-aE@tE<u2XQa?
q^!wuusQahgj`Z!@vej{bz4ovGnF3f>Juvp(B}x}K5LE#CD}#+?%$<4eR}J2*1RwzJPl-*+E`1Rp1{&~5<&hgnOZ+_{>Jk(0V6N{g^b?cu%&RSzUwPazXcU}
S@W4CaG%PEint)r&}Lpw^eYnSSo6@KUi(n2($k{hHQQtDI>J|r5|*S*peiry{B=A8?ht8qAx+rM9RQI`ukYQdnJsO4a}Wqa6oNqdHeq#|l_Xx~P^MtRO5-@=
*QxLbfdB4=&@gWqN(KHZo!`YQXVhUecSA}<&q?cufthj_U=0zkFquzmOyO`wc;~ZXy<(Hew^CRR67dR@k&}gf26jc>cR7D$c*|ki3{7FK+dnGO@=Dg@wUS~J
m*f&GH7EEANE-@%uInodA35}2eQeshn=)p6j}(6tCMdSPV7(*vHLPVIcK#m4ryDj31fWHd)U~qAxs9le-zS^ZZmhH)kFr9+7_}=u((bE4XJ;G15C3wS6}6@g
yIjBb+*Z?e{Iia$wU3ACH0Al(L-%;)tZ|a>cAE9kYvLW_3PkG-TM|c<GWy5~V2RmNe<qGvYFgaa{PvWh-lDCMeDvb8kmfsrDr4P7yGs3f?x7ubxGIgqR2`YP
zXkC&Pcs$<(Kj4i;FcgsOtm(5MVh6-XZo2f(*;L1$XWb-tg?U|5lIEoL-`D8bSoJNf=A;Q)@ia8eKR4)t&KQ2$3$}SSDetuJZC+3q$LrcHr;qD{`x#-PwkER
l#uXed<j)aKMpX-o5c?7p7@>*6erT?@s-giZR7_vP_87<Yo6rxB9vx#^Bm)w+F~f}E!@J<d`s4QuVm%O7&rE5eCF9;5Ym-j>33h1x$g8WC_p6I{CUhD%ZU&e
NGGbhS7{k$C`{rru3;sv`!=8_RmfAdL~lk6ga&iygg`b1yO2=0qL`|FkrSNtwxId!P`)g!aMgC`!JprJjVha*EF|@PLscO!JXBVw;j1F$<u;ajsAvuYfa|+n
@&~}t82EDV#R=YWo$|kt9Jc+OWhT5H4Gk9kjAMA5Kf&FEO=2?HQz)Q!ORl#A#ub3>654H!6eCpoq`5v$GqX3LrP8zNd5!k<5Z0FghiGPd0C)D~z<VxYIu3F{
qQ2dZG*ABZdt9NHOGw8kC4jg*Yf;sUU2qUQ$|Bc8wc<KvuIGidlE&N(b7=d)WlzmpEB_<BuXxKD0L`OMemOf&PgAVDQQw)acyJvU#HGu@vWLA%BU~w^BN}}=
&xbaH0-g#SCS=>pWP?l^pY##16B&MYzv>o;Sl7=O8}h9Z0e?L2!4yah`IjVVtGdg2kdfM)y?k0n>O=eHbOx8rlnNeDk5y;rj*DD=_}W+_8hB#ZqFqWQ5^*+`
;0CG>%6AntaEK8%6*UyL@9BQw_-dMA(8VaU?~E|o)~3lX9P*7^7Y>;><HY&-%henmc4bJ=DCpsuKq3!NovqKo_W`(L8;^yC*YbF@CWFSmJX$L(?V##N(`9kt
tFKv-)9W|?zo!OF<BUNPi2GEGG&hdYARhQGp5o}(W07(!3H%X0b8Q~~0T6mkHq}2kz%7x`w;?*djehvB@g=styuEC~i&)*nJs2HuYGJB(3)71rcbarS8{Z;w
h<SdhPiMEgvnA5n3JzFv56k#(d^c?&REC{KOZv^vA=-x9B&Pq>i!>~}d0QH_Dlu?s^Rf0`(bzJ(pRqHnw~*k5$i3{QEx>%VuVP{-EBwM}D??q+LBT|xfC=O|
LQC8hBJkAiJT&m{iS&9TCLBDtg-M9G_Ux<R7k<Ib@n1A0C_qr3zMTtE0)OabTj}@LStEJG-gWw1q^{{D%E2tzR+}`06$yB2XX@dS686LAiX@=jKBNc%eg#1T
NdeRrl_`3ZmkAPu76N@`qVito)rSs9p<{}9N_9&H2YmID7t*I@9?O{BsPTttu}YgO#<!mPluh1w$&$j9%gp7oa_Xq#cs+hI2x$&zGQ;6mYv-e4#ghkK=$&3&
Dx_a|Ey;Rw`V!EGH=l|jl}j7+%RFqN51U!4^&I_593S3*<Z{c_#)D~>HeJOs3Bh5K&yD(lXbVN3rQ)&vr?sK3a&r7Z-)EMV_G5JM(x|))u@A4FZy@{A8)6m5
g*VTZ`v)0)*KTTuLB8x+{O_y;SjZ_vlo~m^jkKCDl#aQ&6x@Fg)qUcJ>Osx8)ZLCfYr6{jX~!97NtA4Sj5N0fm9g;F+p}#Ml>5h3y}9K+G3NG^Hm}=9raDm1
`8q^nz#lk#H(S83Hnhof_cI<SAZo!%a>yqj&e4#D)MMpMSMzi27V33&sBRC|Q+_b78%zR%IuXWtSblMJuBt~WD@LHkP(50Ci+Dr2D_!$!0i^Jy3UKeBC{A(B
Qe^cGI=70eY0eePQ>Ze@29LXgpfPW2Mcnn)+mgn5zO~<|X?$aq<~4v7DHxpR_aj?mM@}N8g$DNq%~xLVDRv6FLie`}G)mW&u>2ud!vd7T47QKnqG5a2f5f1H
E;Fda>MK!CC`J-c;(&RUxs}Vb%S4!Uhyu*!Yqx`{K~91se71Cv<Yi^KY$2!kt(AI&Dbim)$!QC1rI#z|gc{+f#aesOtXZU5J;+!%yFeA;he6IahBfZMvM^Sy
KTZH&3w*iTBY=1mAQ_jMFu}Gw4-NeFjXFBtH@E~g&T=N7*EH+*f`-=gX_EgLUq*ZE`hCPIZZ&<76;ICxFYXL-l@EBjgTFEbttBBsw{SuA#{y^f^x>b$Un7W*
xH?G$FSE&JXuLq81hb~aJ*)5h+1HK0o!;#GH=GVlBw=&a%D{IjG-&TYjE9fk@*dJO27`%ryV^6|qhYx?tC-U=d<`b%lbBzHi2W7(-?Ks%oZt@TS&u%y2i4VU
f2jzz_*3lgp!nE$<%qs;gm7qXJ?X+I##du;8-AK$rt{2{rNcElpT^p0aBT^unp#|(;ZJ6{NfM@F3(YWNS-NGl?2!gSU#0b|M{#1yo%qO`$4&BrZxA`0Z6|X2
KB1j)?yWDuNE8@oqi(q!p5e_;=kqwa79(3{m<UUAo~6*M!o=n^j|U9Z`X$=lY`Y%ls-ikdV<P55d9f_l&N(8-p==jRb0kIm-VQT%?5Q{@4QUBivmiy;*X%@}
_SyCOF8qRMQ|_qGRRHuNaTxhWJM|s#z7{F}9Pw+B{-zJJ9a)_)l5o}t$>L8qL~ALk6pOBpyWYgLP)Bh2^*zXb#svkz;qjz08O}L*;TW^LSx5<Nz({R)OBoAV
#Ti!A)+jGcgX~OLu9NOE{*uUCS1M2US}=2<{62QbiDj6tQdIA$!OrTX^==j9`|g$#{_HhKx32rRxGwVGCNN&67-Z5MPV5(|<pWM4;8#K)XoSb9xYbk$48I$*
&$E|v?A{Q}fpZ|DYB<49EVIvhtrcpMB|0uF)WY=`7pSU4mP>@45?{rt`b>O;_|*Kmm5YI}g%okbsH=jW|H1pbQ1n~@1R&mT9qFQ`L8lS7!>HVVii|+@b>I@&
fjAwEeN3sh3ZwAoun;e_iq|H)E$w9?pSu}NPeV|^N)q_U;>H8urx1<sW1b(QbWNpCRl3kgLGER#tRY`+%P|H&9t}BXA!YcAt`cE5kX)q_rpX;GZ!upB5>Q$c
c3g&d>&%lThatY~TyoFFg!!XCzkG-{F!GMvqxWz4W0Et?eb6h}dS_w^>#G|dBuInBb0_9ZMW@0<PrzFlu+ZEU%p<xS8^kRWyc`Gvq{QxVA9-XS-Va>)4Y&fm
<~obrcVkk!3Ct&iURu*Q3JR;&ns}$)IB`}`2X{K1TLqO?Ay1=bkto!1*7l*GQp0zXcVz634#C4}AHe41sw?;)ySzu<^rf^evRaYk($sGJ#g$-K^kxn$tc@Zv
aVZ!`Z;eV&RVu-JxL;=Ut5BR2c}6+=aja|4CVXPKd+N<tHR9mI=>g~^NpXOmo1+P#Y+YyvZi()kzA@NCMYW<0v6%!191lOeyk}uNRT;QHte<YtrxLDBV>>k1
jjdQw?v?D*MuSEh$4%YLcj2AO*L#Ob1O8igSRSQiLL?`Zq>LldZeAbQ@EextE;%sfJw!?B?+mr2b1g}Y6bWwcNxjS88_5Yv7BssT@yoSXm+X==Lc7^XbVWxA
priYyzT>RC4R9}b4^cjj$}OcM{hbZ~wx2i<!Xr$wuZo*d=YbBR4i-t!s#fHi(}GyXBM-D1`o_DC;~MKvp0@((RWjY_FB#u9Ud0{JfiGQYw?9vXOPzAcxNQF~
CbKWjf!s{1Nb^5?u2l1$yM0+xTBsUo3obw?vfh+=vrhu_GN~$pq9oUY_)DtFxy1cuiiPvClh<ub>)`>zsY$dhzRIhu<3_N~;e;}plN~MeAtPCccxqH2w)g8+
uCycP`VL$~O<>!b2^N^oho*>As6-nolv=5ii5PB!5d6LI)1E6hqTUaM1^oApw5H7Pk8xgp_?wz|7-Ppe(ddj^-$^Wr(W<1%X3=ZQrm*fWTTp=@A~pndQ+#n1
MoL9=L;*L3UtoHys<B8-1lvTM1ry-Vin_^hw<3o-xA%pHq+&3VXugvu+TrPDM?L_S;0R+BFgJSsbaLFAr57i;x)V8RJC%sH-K0}l2*z1dlZ3&1Llvm8Td}#{
E+}gOm!MWx2&h<HHz86rkd^D%@2#A<d{|Z2w%k29nPBIZcU|^skr0c{+BuxXm(tDp(4^KP%DNfj`;<;XIF^8zUt5C_)A}QxbvS=U@;Y?gXkrl=*Zm6x6l!9U
)<K1Cjj`(CBXm*u@(p-eqo!+eD)kQoDs3R`e-+F~UV9T$=N}3w(Npe`9{pel%mj2sk6JESRQLego~D8qpeMGK0sUPqP0tdErfFSaQKkvnO4q45GpVUcf3TYT
b|)!Alvsl?FO-<_8vUiXLrOWhhr3+#NO!a?$Q~8nC@9Z3qyJz@lUHDVauu}e5$2yiY6$C3ht5|<t9@S?KfD{sTzFTq_{n_DUhPT0GS7c!NVbd&ID@7i8Y!i3
WH!yRiYa>=G4WH;Db;Uc?Gg(-dQ(@}76L0APBuN-N>t;ljuK@k0ZdA~KkPZIc`r(uDR!_lCP76TyAdU&ou$glw&aJ#Ko0jvA?+A{Qf6tBY;ab+JM|{kSu<Y6
aUi9cnI)QwrdQ%Js2zmgT{T*7k)_)q;#nXhE|Pb)#wh(}T~l?~MA=TMtrk307}7Hz(bO9xH8}z)mbA1yk8?=78@^^)RevwFUG4H$UOO*;1>^nB`M*r<VPJDo
d$6C#bh2^*q=ea=9@tQH^C0Ab8PtjC&(kWnfv}k5#b>Ph_tdotdI%0i-b1>;Sbhm7FL}nA@DPcF)7qHJ&w0!D%IP3~6w~z&@Xwc3Wj_Nu0P4C3E}TA9R~jNS
&<|&))kKp+@TOxHDNXBlN;>I(Gfw|tzs=>RdN;?Yc{=-4v9O1ZFF*B+z*8<G_M#Exvo)~X)5t0pI5#PKIu<BU`WSzdX^^V$taV}u+Pq9?=8839U$u*kBr2#x
+Bvv>z5N=3j%%FFZeVa9)G=m{x{WrDZH0f1%IR(k5!a#a-iU{yJ%9Jc(qV^KOMj~&=H5Cc3iFAK6AC~{aPCg2hzaRaG`;SlD>A)q8%UhIVWwOm9AQtx=NFrg
+(z~zP~4uk+3-r0`T~j8(cmA-NVC%gUAXINd$~vR<&YPy>k#`7{jPP^s(~b^#sy7i0dnlTlu@14gk{akp;2p?NupbJ;Qa3O{YZ%edYPR!b>LH?#<@&RxpxH@
ZN=;&uc#T!AwcpQy{6g2gRa{pjvCqw*0*yS>{iY)lId$sH&mB|%a$Z-A)P;RR=n{g2!_TL#}DYTLLRFDw17nlCW>D?caqdP+SSGrxHBfB@qzQ6IM-XY6Q?`i
cG`-*NiSV6NSrVWwEaMTPV3tv<?CN4LI2lMro2}P{BMnG0<40?@FaL1vIhq3Ro9}|Fe{qzT>i>Ex);m$UtRVR0-l@-jS|7m0#UkiFxh|vT!R}&NDEH`UsIOl
kh2wr5&1j-{3ug&Rl;$2a&M|dZ!_wCP)*p!r7W%aKYDpPUG@7u;Y4j~au}b;5H^qanCUZFsKv$p<ZH-NL6}crTMn&?P-LCY;O|FUVoJ5>uWo>U?XMz2?tLCh
T%uhb@fY^qV%@b!ro$k^EO~Zb?~7^Q?9FuFsE*yk+<fU(3-kg2BBngHxTA)o+qNO(Q`7}ZKsXIb$)4+X7cG&FpjU2*NaB$7j$mbsTQTcH<q7_G7+xzWTvWlr
@uLa7r4<nU9|mGR0y7Xyc+AP$tKS(7+kOE53q7_#lC@A`iz_IM#O68MG8<(==XQ?A=adcFWU#UmsQil;4EE3Du(}5O;3v+&m^_-!aV<$gXP|ZC6sAZ_!0t@X
eQOI8sT<H0D5wAZ_5*b=a-=UHXW@K%a190e(BtDino2i_wo@6(2r`auZ8nE$u5T{XH?2~%YmJO%l>K%0HQs~Y?a};;DfUGRZZ#G$e;L0Sl2H1CGN`md-nu?h
p8KLgi4+I-1m(*7fV)g0uz|h)2Y`nasWxL>^z~#%ien}Sx|=DzELsQ7>yVxXkmS29AscqBxtO){2g>oAG14JKQ*H{<_~fp;hM9N@RQf2|lgX{NRo`#aftSAT
pC(uEWcJ_gk6XU>*Z57mf#}CdxI(0da{a}zNj#Kbs<x#wbA2h=#mS`vN9=m({jRP@hGv#PWUaMWS8VL#a=MY_PV~DNJ6-@*by?WJm{PsvLR61%0l4|4yB2qY
5<aA_=3XbiaAfEhl`(pwhDV%Fh~9cs@#TX-L1?kSv!2vZjRuqIWX`*I+sTq!aebm>Uhtz~;A%2lbEih4rcl>JivSe_Sx)Yvo3Q<t2ZVH0H-oBR5K6=CgQ$=l
1hr|{U+URi-@Z(}b+ZHdXU2cvK*3a33`6BsSq9}kS@qFCO-G2U8OP8B)C&!piQ?Glra|CO9y3>LPNjq_J|)SrExx7vDS#;58zoXTPL@FqMxCQaeBMmbe$G1J
e7gJV%~LibgKv&y{`a7^#*8r}aiV>J!{B_I!I^}_WU390$|8qOHfAM7PEl9eTRe;097@-*f(FH26PO!K)a?aXx#ftz=&34mJGKZ??S7Luuo9uHy?(ci=-;dj
0aGplvQe+P_NWjqG3<nWr2y9T7Cdz_F4gphY_=b*KAeUKvCLOVX@%S_iKF)a_B!>v)0BAfMFa%0fb%-?7kxOt4_`2oo@4o@7Hw+>dy_1|VNw_hv-z*zP*&>_
LWuAj@SYIQT~~BlVLyO56Akm7ce<QQ^%ph!ED89U4T}>b5+DEYAq+dlpxOexi@mOG>M-kX!>xdEqg3TAsufTB^(%d&*DCGE0nc`DjKluIm*LjJ3CX9ab`^AR
{h7Z#BDtC3D7v-Elq&uy^w8_mN#L^(oo69vV_B<-k@f+g&<7oF!UQUltAv_yQ;V&sdZfsW{e;(enfRJCV;G8s8T#f7i$|H-l>y1EoCxK~n~Qm1jx|dGBUt+C
=)c@#E-|Ha23lzHUI6^T1@MLo!aQVeoB8gMY;zKGG{$JD6onmVZtQU{qEO-zk5%3WD*1_(g$HudGs|)va$3kE2@QXaiOT>ON}hN4amS~}zi~_#?NQI}?#ce0
MY4w31O8k{*mrj7cR&p<btpO&0%=6421YS4l!%_t{Js!askF9k`dgTXi*8Q7fHH5MAEcT`Ed3bykre8T#M2bk{tqyfhxUH-4E8kQoe|f1X0EYmm^=@cDxYmL
gYEI{QV1u)9m<28VNl*QWm~m8{i888v14FNMVk;<<PDhxxFdfZv53&=Y5$dK4)KV+0o>bQ%D_C8HbnAF9U_(jM!f56%H<Ql&Vd`pAUMom$nWFrJ*Pp`*G?)a
RqY;ZV6R{Q0fxA)k{zrCdZA}vtt21lLqRoH0k&(T1Pi*WcQ(d@$-4Z(SYaIlW=Wt8Vwf2-+8^RxKpOS%=vMKl@Z&_S$`Lq%9GC0Ls%$)=d>rrblN1qBkHr;o
Kya85wFyU?+p#87p(Na%g7gVGfJT^^LqSU@SUmUC8p!N=RXfo%nlvukyy#w)Zj-xn6*K6Y8YMBm$>|7zCOG|ntpx}=S0O-*7kBtRn%f$!UC6q8L>goAQPuLh
!HFc`n}xjvn0|S@zt9oW9eowARoF_9VZ8k79IW{#{lhxD`b3ASvS6qBfIm^Hr1FPYNUU=-=7yCL_f4%)x^oViN`M%EmB?fQJ!adqNC^%_BAk`_Fhl*__4fuu
X)jk8hq#Ob@b&v%zjPXN0`T#LwMosmi=xUE8iM*aa}lwzk7Ddq+N&W!yIfB7(!a&EFTXEC-tlo9eZ-Wl%iKhO@co4+M8p>VIQ7R8!LVtVLehxU`@G5sHwfE9
?ocU~>h3WAg29oc-xIK_dQZ=ea)Q(LzPCaMPUsj2E_UVk{LiCU%E~WjPS|GKU?J8zo;;oKjsB#jn^bR2&13$Ug7V*<M6SzQr_N-Yhvm`up)7aFYgW0X0F6wt
bInp;_h+m53^WxA7<-3o6g8YzfMdd=_USLqG##XZ#ZEJ^Q)WMdSEnT#{Oh3n$xQk+|39~KsY-5(%+*Tcg5t`1?|*8-Be5|4arYY4@U-PHWPS;dd;4f_v0LYN
4<5K@_&Q8PZ>M(nb*gjwXzkTBgXoJ;PgDO<#hJJ`tZfuht+>b(5xi(l*Gg@QZE;li2#uwmTEWF++gm~2)jy>(!6;UD=feyJif<xsS8+lKWZmqP3dvB?BmA?z
)^)~_;{HAgoU15|Fb^vacEWs#8n%B$WPKVo&MO0u9dE8&^L0=IcsJe+m*XCZ-G=6MP4;$L8UobPuJO9v5!U9dFuJz|bW^!RD=&Bf6+^hYY3h`<U;i9ah($|*
DRD<E>@|f6PW9_eY0T24?KHKP=DR7*F&X6NJJG5`2fPY%&#<tKk0PPdNFSStC(>d5wo5bs%Xm4`FSQYzp3FvSvha}3o5V8Yqsj#GC5(G<ckN+ercNgI0JT8I
7Y@MK!w0*+CjN*`7gV&OqYNl!_Xs$wn!dQlz7#4!BwWj6+BralrJDuCQ}3o3Ny6UHepDvpW!2-&`F*=R;3V6MZ^1|oAH(7-SRcF?f8zkmjL(0YmjA34-q1o{
OrUcx+qD?<$k@Mm*|-t63JI2pvahP?E+rwn!#7ill@o6%yw+}HAS!&zb?+NYv>o4fl4*`IL#Rbj;+CrhVtMXbiZzc*>Cal0KStwr)*Jr;(WBGzDfVNXniUjr
WD*_=paSwH7`o3bpWCq=qe7xo;<_PM)%Jk!DRxECNS_&}{1ykU<qx0Tzh6aJ$e6z!@5x8(%OvsQ=ys7UjCiVxD_j}v4NnrGz7`EBjlR=**<4xz3ucuv$Ns^?
LhByvGslFa$H%HtO9~^UD^5;m&T<4uVrd##Q9QAjQC2Prnc+x%DfRbzP$c^%%dQ@vyNPD?N?WX+qd~-j*ByEmr!R~cV~(Kl1Ae{upj=G?6)PU9e8#0aZ(34!
X8qNdG!)%)soeoz20!m~!Fma(W#pLdt8E|L26~!VHs*}$LVmvChF+8dRsR)oSj!kg6F-%!$aLn_O<%(qBivtE`<n0`(teh>D(tFQRqdJg4wIwE7tD&e_4;_y
4BQX+tQE1*I?m+oFEsit+5;+4Ufjl%W##k7f=IEq3O;`O^x5;vtpf}~pGCQ=J`T|?x*oVxjmyLETs~%|iDy}+0g0I<AqPFiGG!O&{Lz@7Iw)YGHIg;$)gjWh
066OLgCdQvXQu5R?W9d#OS^CkA!dIb8)MmtEL*D{;^%5{(IFZJo3oSd#daAvpCs#589|}!9;JZkBN=))vJJ_Xd!>-FrM|l9!57anB`2qf3bMq0K2*Pc-vO<a
W$7(1>sv#e^#Gl){sn(XmRY-MsL;H<u4BaA7}Z+WA@||zgY(UwpHhFh|3LItXWakPc$#Nm-na0@Py&RI854*F7j_f~vDgG8`-MAH=oD<1=SSRz?RM>xiZtx2
Eq`H(tib+0V1&!psXztVK%;N3#cS&4W`|B-!_IqwXZa#P4#yQDPvGU21k5eoSf`D$6g%Z?bBk8|h4Z-=(UrC2J|cwlq|Oj{&#{`Ql@3+BZY#3n<x1gxcL`nV
`|(#TQMBwKD+qZcP=F}W8!lEA5~q1_0~=93$DHP}z+KVzU3Az2$f8<l4@xeTZAUvGBRV<}x7(SeqJ^p!AoRn^1f_Y^$3vu5<8*PUC*(HentWkE0%1x0|2=}^
e1|z})z~j9@)9bpB5e#<IU&zDG=kACo?S8ivDGG$H(V)D{1<+kJchBiOZTztbVfA(Sz6f#$5=ERTb3kDG_jm9pl6<F*Jy4!F|xfQRb`4^doZRqC#5*-{WCEV
(%yqDr%7A{W&km;8Kc1@7FMRH(pvguaZtb_Rc+G#Gs+b}Lz?m-6kFscKEB4ZY+0+~({MSj1zury=Ey$@+0Cq}0rXF{n3T2~(uDeY1C<nj)y0;Z>`Vc4`J}U4
rse*ay0$Sm+^*Z~*SfA}HgX@VHX9V9ZB-1Zc0)oX9qg0wyH};KfiA-*p^g_Qy5}o}5DStwd!i#S*5vOuX$Zy*yE$Mj%+Omydvvb~xuj$;uZq#LFn=X4F=TJf
{PDUH4+wkUOX7{K8I6)i=X@8m{u(cWOOBZINFBbe^r%yQ<sF9Sl^*TC^ELEN*);`n6`u8nfyV9$p+d`u`}8T4V+QE%(0;9VsH*9CJ=IUV;%ssGh-yQZjK@$|
MlmC{%(UI#IgRe9-|tv`qIqyW8)0whTza)nGMP7&Vk$U97=y@qGEuu~9khO|Xitoc_PPY~sGDqxv>x!(DoZae07aJ>(*WlTc+I4XYbLOM@se7Alp((9zeQzl
C?B!}EXF~j<-OdDqx{ET40bPrrKZj?6>Hp*^<Z<0?hj#JT^j%so8gJ$84Eh6c*ic%T-KR;Hrre+LtFB0EJGOuaF7xR0Red+SG!d0$2`*FYZ5N^>4Js%&CGm!
$qhjUG~br<H5cFa6bTP`k1kJAzq(0yM2cITq?-`N0^^hB8*DX)7)KokQUhyBDI5S)&V<WE+6FyZr0+mN@91=7(^Q6n1=-P5D92FrLsSY2w@FDjG<xVQYC6GD
=KV&`)^Of^{T(Sy5)t140Yg@C{Wm))dqJ|j#IK>pDLs+2PhysqNOMzLcpTFAZRq+r+#6LVUnLhV@lz@uM{97Ddj93ppfa4$<%o$|k~Z59!~oYpXq-zsV)B|o
)lEcOfN6a(ZD&-;l;tlEjErYlbJ!F$z<o|)XX6!T5BT!4+8HX(xWHWGpq_%9ZALiJ0O{Lb`3$C%fndSRP8L)_4uY4)Go?Qp@(d}AJ@;oPJ3fOc{wc^QN6qd|
k)<~1SV9aPh^Z=oXaJAr@BzLiIUAu5<9s{5^eT-6v&n9?RL^{OR*p9Eax8x5cq>HT54<UG;G(Oq-Ez1ZBAqNB)>+>_*6@EDt_;SfJ~E49ZtOtUlPK5_=Mz@~
n6Zft6okgbYL=h?F<_OZcvk}*NB$0GVmJxb3icf&Ic0hTXx{X4(FkgchZaVtoyNw|rKY;VSPBKpvAWeX>YtEidPD2xJ0@Kp=aoHjrf1Rx0^pA*flX-pV84(9
XnSa^{oCo_BQTqYb#rJY9|_^RjDcD}#0$}F>cC))5p&?LPHu4Y5+<8v2QkeO>b%WpX8q0|-G9rtt-Hzn7jj*I=ZiXr^@u^jN(;K=JqPL*aJIar1=WKTq26Ir
ko4%$$eva8lTl)9@k>d?BQ6CmGYd!{HRm?$v?+3s@d>KMx?Xdw(?6u5aT>j<(68tk<h1cwO~fKHbJ?aMR?FtCkpT*<<-P-v?NZgY?=4but90Fr^k(B`AzP3w
_0W;vLFwJx7X@Xv6ssv@78a##GPJ$@E|F5O0yBQ&0Svr$(4%5BeHDl+82Hdw6jI`5-|q6v5zYVes0y|=v2X-gcJ6aWlf~>;GedK3R^dwcgXW6>;Q{!EMX}vp
$O}rW>RBIVIN*l0=#ukxN2fV7R6dK4IeB6DT@0a+E1fNNGx6jRgIosQFPf7n(K``XV1Itn1Wppzj?(!Ko2!iu9Nggo8@{pt;a7=C+F*29yZP>$P4@HCi6%=v
wE8f6E?<tH_JmiWXofr#cxkR!UFkP@O<**tgJuQ&kA+yCwGF?sv6togTdZBMAaf~S&$(rMk5yLtYISs|rF2*=vGuHx0<F4SjmiS?d40+Rd%?an3g+R3u*%Au
0B|@HDY;W3CH@Vs0m%2Yf5z04On1D7=H?|Nm(WtBA6r1yWjH^jhjl_mVlw4v5e7<X|7ws^!Cq9C$H5x`F>dPmIKpyw!P1Nv1F`E%+fAnbDS<R6A8TZA>oiX&
L{YrGX%dj#RAkrBowRf%ylS6|iT}bTaajiPpTrg7N!QF!BR&6CtuCmR*%{IQK!lYMYJ0dqZFe}XOsWbd8Q0^SuF*tw4B$^_bK1tkEtcHdpnF}_0PQ-XdPJ|e
ac_<4c$t9z$BSXUZ#JHWFt2r+xR2}yq3Ns=K)}?|w4Iizw%S7As{N>iU?`^acL>O*WWGSePs?E9m^qgGQ3@k4z14nCV72K%YBYN2#YtKs8`_Cwb2OOnCyrr;
{(0WyNuQ`rO+S05`b<*I@rst<d)^0CV_zmzi<%rk2fNUMj|0%(5V~qMtB;3}A>fCS3st|0m(td$OGs7qGQuE~=d0?7<hRZAVQ^=>GVSXNV;bsd`se9XFtn?l
*-RPMH@uv(1np6fu6Q0#9J#Xc&ZU0jPt@ph8kvBPPYMlDoqy1+Sb#GNB$+Btkd#4B$C~yVHtFtpXid4KMo9LrGo99-n<wd6=(Fw0cRcI{I2fC``W6Q!K>4s;
<$c7VOrQ`EjD~X<(9T?wsoDWu0~yQ@OmR%q>oVs_44!rcdt%2W2dPP!JX=>zJZ6$u%4$Gk><~gjuy5=bZNCvaO{wHU2!v8$F#DeudpvfpHl-S;KrS|SWV6Gp
Sv!em$0f#|Nt1;N!uI*~O>gH>a^<g?f`~}e)Lqx0DJRg^RaFy>6;3r*1<M9R*?v9<Nb4Ww7wkF83gK4STApu4^YBH#n+csXyX|>v7p{t<t*3He2?gJC9PoEe
i$(;=<QB^uZ5eJ6z{oh(@+gMcP!~TufahuXTN3$5uH^NAHo-!1I)BD5kP4mp;Fjv#Bx@wjDHC3t^D$RpeTJOPWOs3E%3pvr7J)g*POI?T25tsg)daXxVyEOf
`O_?pb`Jsj0iJd>O=$PJ%G2wU3pBr<iG}4ZN4PzI!3r{-q7)1J#?NFrS37-qB3N6jys{RF0ft}(=XZpOX1yY8{bj-bmI>u}&bny(DbS>3d5T)NUBy+S$}o}b
*Tin;^+U-OKnKutyl!_FPpWzo#tIbEr00DGOE=@{BPiq$C_Xc&=2=nu;DHfk>J)?DXuF-Zw4$bBK(X~xclm<h1FW}imj|Mpz@};G#L^$D4qK_@*fyp)J{y;x
kPGb9&ez6&uM}>|7-A=*OXlNJ9EU2YzzXZfj0=;N_5T$`p*~tUt<<z=>lnoSzWZp5kw-n%8h?zdoMOF0>2e9d?B~8N;J9N?%+_#^;8Xps>nj~G0FXmu_h6Gb
mUof1yE!oSV_LDu%SP-PAs+cCWQR>Pu(2m1;qF#R^IUZ)M%JrF{!f@38U!)d3M6yY&biDEN1I-n0Qp%3;Yn@BO)ebP@$z2;=S*4JzyQ2cjIy^cKheF{s1~8|
0Mm8IHubTQ<jj=F=-z&2;7UOBM}am<7T<D!&a8`Bz>qF2ujRqwj4^E6t>G!2b|bp*K+1lZv2&p*e<n250xo<Z3`I>nZ}qYP_RqZs1rrqnI;S{`5|+wNDq5{E
Zsh4L;1VxGQJh^b0A6m50sc{L)my{-In(F;*sN?bqJFg79KS6xeV?H=9uR<^mg|7KXla*qI#m(dRE|!2b11gjAI8~@N1}tCNCS7>qv5qOnu?!d8L&-Q0M^L+
8qz+uj^#u>_%w<9m5-+XR{pFIN^b0h?$@c(JdMGD*HPcGAd28WM98qn2QF(tHF{3BNK5+)S*%N7-!gFQO00-e8i=>o96jiMg2g=J!D#qJ9s<(B`0kxPmU==M
mhhutOE|lB)oUd?_R*z7<R>%DE_gb&3?s?F01^&PAOEpiY-AyK?!0o9>+z*{(k9V)@o(=hF}5xkjV{S#Dj26LkTa-YV5blF`=^hv(r|3xO&D|4G$3Xf7z66+
dbHGv?M>{+)+5F+6|(5xwwLqSvSh(vyl9xwgF(hz#9#FkSQA&`-sr-CQ)mqneJ;(N>A8ev`q3xR-P{<;_IyhYaC6DRV6SR*d!LHqy`7tv!gg|-s~Ir)`_B^-
R45G006tZGKXO)HjcVKj?$B;8Nje&QNulgXg*gr-rzuF62&1srYjd9jT{)2a2wVO=360cQT5CtezP|^1OpF6TPR$Vl5Yb$|S1sf<qO|RS3}N3yL60fhm0G%?
jM9RyPc8@UbssCBWOBlJvFMKlp8UxgnB`{jL+jqe>YXTa<_gT4BV9Vqp`I6zsrXh&Fi^Qw&v}#Le<2)Z7hrkg33RjzpEG;*HhYmiIj7EwZ(mLqJ81~Bc`r~;
@|EppBBmeCjaIENJOwn(Ky=gvWF)4PIu^)Ni%NrO_;%URH3p=htO6!~!5%rwrN_nH-UvfHKT8O59b;YLrz0w{TTX3-TS-2ZMe+dNsFx?C94V5+;)I?YK7d+2
ft9`#t>@|5TeOSGZ(7^=kLa*i=UX!_1ylY==D}gPI9`CBTW1QQ%4h#b&EEuKoFkQ$LEUhfs4h_20xd{*o>|H1qz;)T<5{0ltHtO?@((21ZMq7hRab%Rbe~5I
WgZglOY%GdLTVK`pED7HbrAxu#?f`b_isPkp5mtbKKblDDe1=P|5)}*?I+LhJPN>~o?h&BHc%%~4V1R}wl^YHJ(|^U`JH?~ycSUD>h0A4)LTuf<OhY6akKsF
1kog6-hqtjN!G&1T0Uv915!&GnmXO2nz|XD%=@{6)`_GtoA=xpj_4E<u*r_x_^9AO9e;A1unCbfP@vBTtA7QK@zvSsj#jkd3D#4|ok}Bho1W*KW!%=;Zu?rD
cU?vyyh4o6C5@QXcamGW-<uYU&yPr;xC8H{>J3bFUSLoo^X>?p{=&v6Ct##l<$#Wv?BENN<@D$@2~taELqlhn0V~)vhAA?GfLzA=f&k7S))}17tC0`>Sa)}>
mfYl4u^N$rW)@1FHvlOQ`^!~N8IS6LB7{^I`Y9dX`{!Mohf@Z4A~oRntV1IS5436ZifVM{5$U_80@n4Q0(;dfn~!aN98q7VFvs)U4zHVIz_)GBi<#H?^PA=6
OgN1UbS?z{D320lO6Nm^3E+a%uSIv^dfvYxk4|ycEi3RO(C2=#j%Tl{*tsmLYCqF>c0-zu_nIF4IU&sH)o&%V9RnXy(7AC|D=MiD={&zJ@Z%g0vJ$QJ)gObe
_aKZc0{?%+t`TgfP$&B|^)$~gkQ*HO_eM^UFF{BacnStVSjGT9p(C!&LN9J#o2djcT?<nXMw}%eT86-UQM1JjcGp@4=$-sow5_^e?%6(z_pWdd*^~63B9E<j
P11Py{m{7Nj>~tR@7~s>R!C@m``*mq5;WJau_4K$ivT^WT_qMY58;|%7|y%102Mu$(XM!9<I+*O`Y?dylhl?-T?l_>1%&{jrZvLRFNqdL_02k`665`rmdau7
nKG3%QKWM^HA_?rn5-?TFe2)2<eU5iHZvQSBaz#Z@C1`!jshD$E{cjnK8p*JC7@|pBlI&DupE@eg@B%OUE7TtKkMCy*0hKMBhBl0JVkz<QxXvdaHDy#CNIx~
bs*h&DJyQIn1zvmDM*m2{!Uz=TlRu;@ctaKKysI%-n<hf$N($q?#U><@-%zt>g`=I+AZGqKPz~_hpR3xv&ww%uG|b|<xYo<!2*mY@g#5pzxbRluf@<oaujZO
z7q{qFey2j?nQ%y(@N15D6nP2XgFnF^w7SeC<kGt(JIHCW-FN@gJnqm*2W|wxXhoG529*7K=bGtmj{roFxg~hTU#6gTB+xC^xhLTV=7R^IJQen<!pJW<rA`a
av~@xlRnizNQf0?vDUjcPo1}73RcR4w(2DO`^sf1d)-mozoQ)q2B+Fb@;yAJx-d7m-z?evVO%dd+w_lXY#xde-e8oZ9?kw5eF`f3<6H0}c)<k#ptNY&ruEa+
AIkdY7;u&$rRU;F>1=<WssPv0&Pv8*Q<FZhmaKDLrT0)+*FT6A`M#x@1hpq!;gRh-6OPRUEM29P8P^}h*n7V{2<8SbT}X|pt!sul5y$V=E>%l!sbbC4l-S^Z
C3l`fK$kDE6^kp*`R;x=4dq51bHK<NW#Cp+gKIB(Wi)aP&odNbn#OiUm$CLP0X7oADw37Vx|TX4g#W4~_KT7KXJv(9+(t<cgYHyfRDBh}x}}>oCbn|cp^~Wa
BSSApnSy!#%xf*^!SZK#$~wAP=8}@9G9qIiA~snOXuwbu9K31lwK_q(c%J7IQ;jEPiYON>ct6gfE5Ol_?)nFrda9xo=eC9&Ahh>7(v(G@B8ezv5y=bkcNafd
5#pssfM@K}w%gzxog7wfzj%!W*PS%>Rq$@AUYL`uD{qV9Y#d}2JDhIKb_%?-T@Dw$PNtDog*wBRR{FQ8ByTnI+aCvvx3-;Qq*vFM@&{;Y`p!W>N!HY$hJaB=
K@#G#i6Gvj*C5(fswAm~A8h(KWE5a2scLv1RFHTJOeKqj2L6r<vuO@Bvc*~Z1=EBL8UPW0d3m<j9lRbKoJO>C(DM<*+OA?MC&b8vjys@1Mt&P!w!a)qhTuIy
vw5^KK)j~h)o9TX8)5QYKcfZ5Mf%2>%**B*CTJ;D2Kx2tQ7HR@S_6JhzK2PGBvXyh&prytX=@HKcE4E%3SM5cA`x`}cNxw=axD1Hl+vXp92nGaVBX?y;_nXS
Nqk|kIE`&2%zY^(4^Le-1&Z<BEPj`kyun0t*ktjT-1k@zOV<RQK>9IL#_5}N$N?OjS6B}ZuZ+)Cv0B$zuWkf_!DV^J#xC3cR{8UpEH5^RhnJoR;Wi$p#ljX1
=~g$@(vF1E&&S+7*u|XgH*JZbKAU_0)FpBjG~sc@qym#aHQ9zfy*yCDYTzQ~LZ311weQa=;I9Xwr1G*__>5?YYis$nNDkzc{V5X37@4ZRPIO0Y@67yXkR$+y
%K*6P%)XTPNV;~(xhFK5xf(t6dRnsv-~b(`3e}OQ_AQhVt(xJlr1$-zKcN_f;x(LJ>br`(pmn-ZEIy%muW7e`tBmBiq(D`pmx{?UA1;_))*OF@wTWd(g<nqQ
jWRkfL>_5T+_c0S++%T(IXFOAR830LX^k`$usOzMy9G;so}I4ce_nXHYi^){i4m@ADPM3(J#tqaC}+l>aaX{a7o#xj#Uth!>O*->iEg`57gbf9Pc*#u;CI5?
n{sKcz2lpCqeOb;zsF=SRmr>Jp1N_4x+`_l1u`D0MXd?dy~X+%m7FYJF53p|fMaS0lVj`yX5AT|pyiarxh(ErFZdoj=mBqe%e9_#ZJv~dadQy3brY-c!9)8p
X{h*F1QXvZv-OAosx2^%%hlw`B%;**JFVXwJFI%w{!?)>_vbP_uUf6~QX^T&5cmt&a5*NEx2j;ro&E%Xyd2TF(nN^M-WaKVKPH6>XPkQ$?tXG{HVT@Nb)5wt
z}Ypxgwt^i3G}HQ&iqh_x2CYP_1t6!u@4S46_YQ1Cn{}_&}+ZUJG~G?Y77sG?ft5j0{3_Bgg^mdWW}6a32)dKuV%Y8ca9*SR^F20hIuFSfpV=XA`?0*7qz^3
TJI=DU87Po=bN{`_5kI>)o^(w*(p_=)bebSxDwgis-mA9z3S}0h=NM`kV$V#h<%FbszJmKTq*~H(X>gF{Su$>0KPV}ADTf-K3RkTvo+95%NY{CMJc`9jnmad
O`4wlEUr%>Xu;1N0<z)n822>VXI+%eVlPmn!R?Da0^<JwVWS0Hb1`A^=`!%c`-!(4G@&dGNw;$4ErFd0_Ke<8ivoofY`1Gkj%Qo7ey`M`O*Pxs`1}RJF;^M}
sE(Jrt>Bs<N%T;5&H3Y4a%Awz10*<ZG!M^qK2E`;M$5<{)?(E|*Nl{@tfa*ZCG?wL$T_~hwyH`*aw383rpVVlI*~dy$L1^li{NryV=FCQySI!ZR?&4gbs;m|
ICv{lzsC~-y^xbRCF1-el7f(5VTvyN8;$tunOLy@%k(UUTP{Kdy%G@>Q{99jfF}v^LjS(tjH-ASE~t#~QO58Phzg2MhX81Q65o)gk;~8|M6SWQ%ijG|(kZNt
F^(rHyp4oF4DuV;qdC;2z-B8rVV8AMWJI9z!j)^&E6@~fasXrVjq4h^J3Ua_m5;(Me%hKgGqhDyf@2gq)SN#mPr!ss87)DR6Bk{Y?NsctpghRXB~aTni(DBi
U<vaQGXq(-n_C8_&a174e>XY{+S~hV)8H}0p(|Py_r}A51ZGMoP+<6ep<lAPTTIovR8%<)mxuhq!K4Nk`Hj<U>DrCQlkd&a*(OZx#A>FK$x3Iyvq#i`j{Sm-
-91$JG~S{;v3}*qOQB!%9Fy{zo|R3D%%O`yMc+r)LBNt>fVfSVYmD?HN9q^q&<1#R0s)?Kh78Oq)$KvY@4W!}Y?HWr-y;4__^Gt2%s)FY2u8iTdz$}^;4-?r
6UwNZ$7#^^4R%9+)B0t+fHO}!5*p)vAG?CM&WSwy!K|Kj`H1tL$rcRA%fe6b87FWUAhu<BqmlcOK08<QgE>(G%qtlo1TmL(?<xUlVn3cFj;0$vjhnm@@g}`Y
%v*#yENRsyggIni)(Csz4JA-y>O(mdYQCgG0aAc!=fW$fHGAaiJNU!&DDo4gq`i`yAW*LMMINE+@{75^#5S3_lpC^DC$kR@BPTu;f};?-#rT4FQcGmmi%meu
bO)-%ToG*eblDH~p$?Or+09frntcS%IIlWxKUyGxFYw&N5g8mXA5bmHP(iueZDbdi0)RC{G5;DNP{L_gk&K{jV}>~llVkD*L^HeSkg1wkY}3t8PJu4(GX|bF
E~ed9{?b?Fxp$T6FYRA;D_>-#GgbNT+|$rT8_sWc4X(Kz1`Lqjqm}^Ms5BqiPHf^Q<%o`rJf;?FdG7^Gx`2b*)38L8K_<)Hj+);kFE;FII<8<8rBP{Z3mWGX
J9ivE2?n}aR6`t!8w>U3?@8xVdOAj5!QGN?wL<6yRR+$RstathD}f|`1@T~Zn-^$K9qCM4h)x3IC)s9~c;&t)Tukp0kE%U+U$d?RS8RdAi;+Z#uGXdGV?MN<
!dJS3ugoq_Q2px_bV5NP6<(CSGvzop8_bV}%VDnB$fxm%T1lT}EUs8wb>D2@m#w~+@BcaR4Ai-^e{N<PLh|1q2zq^cY#u3r)DemEbRWHm2mA|4s2_N6nOWuK
xTO5vsfzM^6xh8g730{u0>+WVO@e0Wp@`c0I@6L<H%!iL>ccKyoB#(tFg<o9x%#em`?s(x&*hpYyCjEw&#s~WugziO_Tt-xo*?<&=Lr`T`94!6w-gw<J=S06
8ws0C!!93E^mCgqf9+g;ytlQU|1JXsPm53xxYEHh{fo4+S3|7$`6&<x<IUzpEj+a&QGudt;`Ni8JMXJmrQxEwDP<d~*T`u^T62C172_IuofYA2x^iYQR<LcX
w36B9wuTG<^>`ODpbt6H<_q;uJ%a?%rdpwzY61(Y*ASajEzFMaBvQM`JGUSucPm7@68M*<(<sC@+|(Hi7bj4|mq{X`xVzgvm}ljOJ58`3GVLs;irN)C9|c58
4e6L0K--2jh#ui6k#Ny!b>V`GAc}N&^-9F5OB?9RZi>nQ6vHEiZyU7{y$pecS^@YOPU$4y3;S752z@J3kai``vnd=8%V)@SH*O~2AF-H~ZaMJn8v~8;j3k+Y
UKZC~-J6v#Q*O}USarVgm4#7U8`_q&pjYbb6m37Wmlxv<imV;1pC7E6E{I0cNpvn8$R5Df(Ek^Zjy<e55`h^*TrN|LzT+Uot+HD)FIv$z%H7wA5PTT$kXfRH
^aLVmS0cZOn*c7KjPYxi%8LJcOoDwdpXu;`E*Prh&$zWm<ph&~E#?VCC$*-h!fEPja1v_pMS26!&#?1K?<xHw%7E-Xt8V^vZ!m5spsCBK&p3Ej7cfn>!STRp
vZPC`g#6nK-}A)fOXBat^96F*mXxmv|NJ<}%vV8^#Zbb}Mrqh=GZTj?%))(7*I>wQYc-i5HJ?LX$w#VN%KEngMUa$cANTPT=iE#NR<MxKg*KAfInX2C`-pmJ
Iu-L)$!`3A@zK{`tI!YC7i?CX+^YXJ4HXL&@21<gL+A1fOE1Qj>r`Nd(8#QFOtJrH*!>=KQ;S)F;x0AVb>}Z-_eM6AWxVT7Hh8{qC%*ZevhcI;9vERJ2wm`B
Ms`{6%Nk7lRy{xPx(hboffnsFG$@u0=@QOY$fxb;R*@amN`=807bRQ$s*W_j+AL#JUh4bGtQ=N0x}}z7>&?@AFpAoi^LO0jS#F1_a@GymKMb6ioAu`WCq7$B
y(>hA?yZ$Fp9)CF9G4joXRpwTN1AY&vuZd=$5MdP?OnHcYhGQ)GdDUX__d5eYo%nMcT)4n9t?5YT#F_#g~~5Vlj4VUADk6;=IGhfeK1xVj|vDK7N6r(^AkhR
xCARAODxSpbDV-z@wlsj{x9{3qt=jAn-s}l;C}k5xU5cNV-GRkvz?(tjZ=vDeBlWw*8E1L(nXLZ!xUJgC^2nMqIZVT;C%g0%=+Eyo`{;_gPUB?YrLB@oLM1*
5)oG5Rd>aJz_V?!?!1BnB4rjGLm{QwpcLx{*~@&B%RS$0Mlt?Rv+=mr5W_F*NJ>2W**5MOMO!!NKzN}?#dzDn3*!71ROj+qKch^^JeN@f19+`vKb=t?NeTxo
+SJb86E8^FXpE&t?+C*P6~Khzg)Rwme(;8_T&-ltbO(#}17LQ{eW%F+af<k+&maT!u17fCC|p7*o&@dMJz9uJ$QNy*F~ODVB&XzZcI4Nxd)9b4I9zu3tc$lm
QZ;%-HRV)uM6~pMtmClbjiXjD3tF5!Ov9C>haI!e(RzwMyYpq)C(Wips7&tsCk1U7m5&J9$%j^d{zTDh5ktkAV&b{1%=h|Jw|5s+Gd`No75W&-@JLj&v{cAM
vF@Y;y}-e@Rt}(;6A;zaX$CSErySYSLHcK7DNO>bcIAZ30JfaR61+~fsJxs5Q|Bg0j7JWUXksW)W<0x!<EgK{yN?~1r}DT85q0b&);2l#`#>YcR?xa?y<dM7
ZH+q2Z9HCc9%3+KggB(DN*h6><r2}vINiV%*7+)$ji~$RF7<HG9qB<9(Xm&tBod27u!3_|21*&*Q>No+1@AHzixx~~7-S&j#F+QhRha^@Qzjkr>yI+ksO{;V
U-|+VB=60yOYAHIkbRV}@O3-1lCLeZm@93lHs(*DsQ5(*lLuh-FB|qpz)WtySS>Fq1C(8iAw-Uv5z~}GRY&4r8>nR>72yH7wL!-n5&yxdYBuFwec!j?3sy>0
4)12QJsp;Y=1PaZAHMJIG0-K&Xt;1P0ztPV7posMx*j{tU`JcPSrIa!o0svy1gO2uF1W!XwS(&f5nk1G^#+q%kb|57n(G&|j(N*b$&QyaS?gaVXKzvUy>+jL
BKl$e%7R;I_ic5H8=tXViL8Ygu;TkPky%2$$iuos>hB78aX;VzSbkdh9>8<$0q5p?YbJM{I<AW7LrQR5C!5RZ<neM35T1S?d(tyQ5;WHbDgDGK!rNXhAW3Gg
aLe`}PKO`mzZkDMCY^e53H?;xbYhfyCUAcqeWnK&Jh;8cn4*&f#+dD;)#Jm-xLy&<p?d*Y4}#(Hcen-a0*zeh60j((2y+n|E5w_3u=m0`7h<upBHcAnM$7{q
-@+Z5zQ50?$uSw*@SIJet;UvN<P7h|dn+=r`93EY%wD@wmc11Pl4Ixgh8hlql$754ctC#)4*55|+4TqJd>>^9lTQVLWg5<>RPM5pce_1~1`^L%=qnqwNf98%
5mqyQnYH$Iumum8_8*m`k;Srrg+xIrgRKE&r2`c(I+Slu@a{DMyf3D;ET5E^^<Q&eefwX(!Hc`>4|92YB_3{rrcN3|u(uiGkgfzCV}yGXi59~)4y-$$MwXwW
x)2>ltje0q(|--yUOZk_`NQu%WOuY&*HYr@P8CM2&kjaj)qW9Pb@k*DZL(teFT9cc((Eb7<-JN8xg+e|5VS=8NZ4pxBjbGmDse7wFI+3;_r#etkF(?}4w^nT
K2F2fThmGuOtd*3OcX;u_y-r-#PAv^2RUjY)Nvr{-ZA|eV(p!u2S1Vdb0I^^I|rmlSO)}<gp(h~DDpaKmJA}vUvElm$9q4^g>>mDlC4Tu1M2o>8}98g(J`IB
2VME*ZlD$n@RyNpZun~<n_zz-zlE(JW(oG?z7g7WI>EfTPo6(7jlM)YS2u)73l~s%YlafB+DYH$!-EY^KA*q7vxbIu%os2nJgTWGv^Uqv>YVDBy{G2av;CE0
{ya6ic;iITP-<<pnk<YNfzGBpr}2e^PWI)-+5hJ~w?QEtoPBX!40)PEp|X4O!QY<OOMC--vRrtQ1RjMl!9zrR8E~SdU4&UdToJ%yjm~h&(M&N7F?9a<gd01Y
jy^X0oqBX)*7K4s9kh!vu-BzYqVB~vX(Od5ak<7gMn8~aFZ5Qe@=dVT$f1T#{c0u<KtT#=X^4QPs`(l##73|Y0vjFcLKHZm(DC4ae^2xBI;4+aY4+ZCs7k`2
12|Bm=(ulJUtaSay+=Av8mp&V8%dUb<N9_pp53%scZswDni8}B0g=scIf+H;O>!#+OGT%b4k4dUmwGZ`B$wU(@c^2$h}nM@hEzGPb;uF{AaQri<j`$eQ3fAO
c7ECC%p)}gzlk#;;Br0uMYk18tz(Hbq-~Z5Db-4Qr->Zk@uV3)w+Uj0TMMGB(<n?ezZGwBYu)d+&eH=l!1SZ8sI7pz)szebBfqWaNO(b4IM)UJhkchWIkC|p
W>$yTHetBucyVNY&ydg86fmpwnQHq*ppo@@8wh<im`6QPBBFtzqWFOn>-y2DzZ0mK1K{oyd;rj^<oxd(pfUI51ZuQujDA-A>CUnHsa`ix3uDkIS(F<vQ3)r2
IYAGv47ml(Mj!3ANK?<htee68iyej|st;C)VQ7117JW1-4?MQ;r7`R#5T(di$zVR4=xz;|?QYuAZMeV&OOz9-<Q+gF+NAph*p6-6=bhv(v!`W62GuIbc92Hc
Gda;w^7i9NW8eE<XK206npiEE8%pLOy$nKJznJ7J<%ttbj5=;)SJGOsh+~1fp0S!{&Iu_iE!KD|bHHN{S$kbe<!hz^A)qx4DD1;FPf`HMScr&K9(SgorDT0S
t7(T%FA$?;o(R9wf6awSG<Sv)Kx`vG!H=UWepxk9`H6{%2HU(Xu#&g21`LM(Ju5=!7leK8IRthR<Xb~v5dxle;UlcM5f_PNRX6?1ryW<EFvBvBV8RhebGY+p
hV;9SI;XwiEuSNUtfd`4Mb~|+rfEz3_?Z^XxJYw>?92K~|7#<bX%a*8c6W*K|H@{RnnVn1$_B!3UA8qclmW!l;3=QJ5@~<ok&_Wwu|k$%Lc%1hp8NT1y#W7l
#M0uHR$)^fd!Y|Orx}vHag<V3nx#XYvuYz8OJv>+2D<L&Fcaf+&eaCtm-pgB-Vyi!vsq@IX}lc-IrV1CUr$`vLR}TkOy25P&J|eYuR(Db(dA@%+{tfH^|K^~
p&eFva4;^Qtlx?Q=0Gnu@pKVaExRD;9tcFtDq~iq^qVA*KP63mFl!>Sc85-^Y~SY*8)6;6`y`H`5rB9pR%e<`w*(<ufe!CNfn}!4C%167^v5I0AdfME^%Ox@
Yivxyb|aAG`X4s2cPp*%>D;m6#FOP!)_}@N-t^iu+F!kUeJ`Clfw9h&vNBw{_3k>x!)33=Hlgec@B<%p3*R|W$$ly=Nw4Ol|1h*0;6eXCb7*Zi8`rQYX~9|!
wpHYnw}^==`l9j=^M1{CM@chU9g^VC;V7{H$e$Aa_C+3%2vuGglDv~CgZF~85^t9-j+>ZA#mfZmpNDGL7i<^Qf_B)RmCyv!;Mrc8;||bX1fqJ>5vjxR&GQ4Q
v3I;25gfGW9E}(lWhpK)u?(rk+f1WMD4_M;P8^zFG7m#Pg~%{S#f;tnQ&9Ex&3ip!xD(d1E*jztL}I?MoLuab_jC-EagSsOvspa`VJ9{(u}v!4;_iM?n_!MD
{;+LHUrDl%_xc2qVbram*A<PEFrsXfJ+8@7KsY2~?U+gsv0&eU5hsX5rdx?5?q0n^#BhprK}^A`WenCUsKg~ct9XJ-5p`DrltFjCrNct>Gfr-hr9vjBmvi@{
JVN$O`q@w3&@)CN>srTpB{MJ-hGQkl^991JpKRv^{*vNJRzP|Wr}ak<3D9DIVrd7eWLR@#O_PBb!cX*I%-YM>rLM-;`^Id>>rgs$X?j4&S~4xkpY1V%`*wUC
A*1tecxBWCrO72`E8GdJmmStK3M<T1u=N0I2YXa8Jc?PI)IlsDXfcj{Ox54Ln}@Bn(bKCrx-?_wjCR^qw_-P!jyyqj^oQ+s#q-B2Bg?M}b-}D&=q#=B5ToF1
Oz45XTlXnnLUb3pk_%$QdtW@eeQW_bnd#@aNUS(kfTw#98rQ2_Zd6zk`!(+FSI|mhvD-ROLRu|vrTKsYB%3wmQ98@kFP_x?%12N2()H`{nOC>tUM3^1wMGQs
*9($q0{R9;U{=o$@bo0s+@dJ_(3J{NL*(ViNTS^%v6LUo%Q>#g<T4B`{_5$1R0PL%fCtxXp<nQ4!QM=E<lv!7tTR(*0KPqB5se*o?r;`B+g%5}6ELhljymzL
J<f9={xg^#s^;u7pa%++ilc{G#5{Tq=tE8PxJOQ@(64#_Z{LJEdg;|gdv5?~oD%QT2J;S5AgxtZ>L^P>`GS!m@W8)=_`Vmfb9c>F^v4*qZ2;OinK@}Q5G?5&
G8!ci<@+dsPokG)BDCZUB|oZ8sNM(j3I1c=EFSiQey@1vb=s6)+#{Duxexa}Yi4g=(Sk3368Jzi?(!Qu%2OV@f=4tPhc8~&zdYn}j-c#i!Atlc%96htDHQ#$
)Bq@n?$ZUO$pmy~9&JLCU!bG@d>Lun;0xrd#WEnmgJClQ0luria}osA%Y}oW7Xz%jx{Ft0jr5m4Y!?8FY6rqTP$OLG0X$7%xvAKY@c@#tS?YK0Us~uobxmV0
aO+TQeWm^jw|3a{5l;ihQxfMO;?RdPqVzVPwUoF^(R}yg=S=(K?;JUwPbzC0U)#~8rBG<e?xU#Q=Sn+S`w}t+k{nhr&`=w-m{h@bAUNMuFaao2j%3B>MC?R0
rPUWj5_6xBAw?2@7ueqE(YGWp#EV%RQq~=LtgD!|Y^VtxQ_)U?1ocM6JQa^LLAR^_HP~fIRNuT5xFf+<?2&A}(Kpu#6bXoJeQ=%dI7IMp8d7jr2iZC(RLr3B
x)%fgB`{C~Oy9!@C%ujz^EFtnj{1Fi=fVA-hZxe-ZHu<3;xhVf7$%S97ICl|LN@7IIu6_FYWr+IOYG^lLS8u$#yX1pQ4b&m2dhqcfb*4T*w*7-)PxiLw-p=w
@0e$>9wJm>;75N(ihquD?~dkt<-54YiH)}K+Yq7#Xf&SF?6ze+wWor<*|7Z#XrjYl&9h8+v&sT}3C$5X2an(EWJ+@xnhBN1$i+ft*RaTMcibKPcmm;%tk5%v
z<bl1!34#BDsUJl0+XAy!~m%1@>wx1Jf;lFF34l;u&{436sS)O@f4=S@4TL_H?E1i*dT!UU7nN#*o|lq>u~b>E#OPIhkLVTBjbZEF=}LssK2*E6?uyjxznr*
!mU71Z%Pxfa@|&Am*2|$f5l?w-_4F1t4UbuKD_an=Z#@5Hu+P#DKc(D7tnNOE%IoUw=d7eq#u4nkt{`87j84n(jx-M$`1{dFeWiSZt0RJD+1T0-u#2*OOG3-
*GIaWc|2n2Icc6E2m@9%{ol9k_4JNKm*AC@Y!7U5$4Xj~74(}aCfTR3gxq7G5JH#kRl%=`{4Ge$*^$eOI})S`Q~G672^>Rf0Kyqa#P$;<KR_|Nz(b33faF%+
adFwt=^@=04OA7)!j&}e0<cI}{&$}4jhNcC^lU|&i&7#2I778S@~X2L=Xq_xPMvL@h3KY`Jw3Z{WJ<!3DUkw?BEF2ibsNO^k9VX0ctRgLX^adyS-Hpvmi=Lg
1V)3MifC5X_>yR_ATugL{MER7WdScw%W|q~+k~z-lRehX>e>nsx&wR$0I4hL8FFyJ&%Gj)jQvSweU0z?I{i}m8TzZ*poTB#c-|*op5dcn3n$v{$O+)a!b@;c
1e3Kp?@IZm_(r$Hzb!&8h$INV`7lNjO-2`UpMd)_`qp~tCLXP-T*t4%zcVRsvuGp3BYs?4Qno87UAJN=f*j22SNHj=Y{!1bt(%cA?GM$)p$y~a-}fY<&DLJj
jr#|+_HLcZb4Nk4lpBDQjXB`9(FoR`&aG1|k#?V?#9u^(sjt=w5_1$%<UZ!>|5^&RTCsA)NpDfNZp#aE@s|1XB;(lFcTDcPivho7xXJ{0XF6%d)t24GpIk^y
cHcXNm^C7ZDWbdw->EFp#)kaUDg6`-^j7kc?e*Iml$YnHp1a@BCM9ISS-Z~CFgf~vcaYc>P)JEei}jO!L>^ToYmYVBkakP1G_k);@`8+@p@Qtqzlk)M>^SBo
^QBZr?w!a?_#EF!v|cD7S|@lw&P)_K?ptmvcDT-~w>JX#mj%MQ)7;+wgo}J$WnXSyG6q-`>aT$3eQ$bSV~p3yfLwxIo8{f|;&P1`<sfdEYrNkn(VB<NB{wiS
pHXAiT3xewtr>+s%p19cKDhGBlOC9vc<A9Y@V0|BHQ&#-0c;+?TCPyaQ9JNV5B9vvD-9XcL#Q9Mnq+Xpx99r<IRUMhQ8_GPiWVc|MgN+}`50k~ZA{P+Zio+W
U_yOI1`>9J$en2=MEER+>mBeh_cA2@;am-uM`9ESv<bZK9Bc+W_xdF-TY3eyvxb{XukNBPmq04mu44xiQpW8@#h2(BzC!NZ&nRoWlwfv#QPcVils~3Kf&0Bf
w8*c7eaZAyGS*qVVIOrWZ!BP+B*Dfj&|Rzpa93>(by$1dCZoIwkQ6`^Y&yJv!s`_;H-dqh-~IJL%3Ho*U<T1@7CTeKaT>^Y5qQwMEN~=M-T9|{*p*$Ac1Opj
;Ud>>B2Kk#PoO=mN|TZd*IacJi`lWEeP+0rw#Dfv#Hh8XXdn8Y=6N}ShOn&NzfVA4qFH^gASi0j#@pKg?<2sjD4@c;03q{(FR*1LGi_!(lSit=CW+T)t<#{K
lo>R7!-P{(->0*#exuL6%NF-V?ckbPtdVuXucT9c`Vn(CuIW7856lH9x8Ox=XJAJO6zM-p+<qm_p#fzf^PemU$ADSv%Cwp4BDxa~Qv-z08HCeA{gngWtQ^Hb
uBX*9u?Cv}hUf@3M_`=VXV;b>jms7$xWnBDvXY&6!jXv5_P87@q1%>3(jK6UVMBwMaT*2AwOUJDni2)JGftuJ9Lha2P`ES#1AS)~pTe4Y1R|#`@PthtJ^yF%
bM7x`pP}iIY|NVIQRJG%!p-NDnyX4mwUd2={&nzlNbg;J&jUPMf~pV!xX%^);8F9xU9F2rI8o{#ldmyl5n+=5MiW=m@iLz_ac^q7iKV%QoY0EHrpV>0+iINh
0-aN%A!?MpI;a<{_t)x34!{z03*L7#kQGr+gGQmQSu3bRLI<n_`*d)GuMXVHl~dLB?XxhBq;GJfW7KA&>TG?d!H^6a2m+B-tqwUm`j@sp1G%Y!-jImjsO%ql
8*hc;^{|lZZJj-y#+PKd8z+ql<Gz-b048F2dK*Cs2GUWq8m0mlDtMvqv+ND&^OQ^ax3>a^pzP!YE7;{P#1NEu$wt0k7bnG@uR*?9`U>lVM==PsEJT8^fRCxZ
i4v%u%B@~apQXFe)+UEpTfT~gsr|o$GK~k-<6&B4%f*W1iE0L6QPAY*k4>!KL<a<SV(Mh3jX~ipt`bz6XDihyqbDybcpHZ$b@;M#=1l%lN(89}_-7=jKH@V-
@7eoaeT1Uo*xBE*DAh5@@-=u3gep^#xv_zymTjUvtPnkok;qL3RLes~@Kvhwb(ywZVpyV4yeH-0ffjKUoZn`m!CpRGI{u8;cVPp!r?Si5yXBnb;F?L~t#LYz
`sBw#GzZ6y+3S(dkjOs_jTPp*Z%y4bO`n-Ri0gd^5?qV?W0qkGack|45#hL7VFY6`{Na%@hF#4(>w=KT(cH+)fMQ4pY~|DQYk`|kWVNs;C+K1u`l?7*g|HTZ
jjY=-e)*4bnj})1BX7E{m0O+{hkP&LiJe{%g?>;|d)Ub?)3UXP4LD5!Im0|tX%FMK4E8er5oyf8d6HyFrh1<}K)xBYlLR|%$d_stI=>6}&x);`?TzUU%51X&
CJo*hT^(4s<kS3yh`TBb<!&S1mGFrRViSV@6+`a~lXM7_26)9iReLyXIsE-1-C|`K&47r~$Tr@tbLfqj13#)1Zg#nxN}}n8DCKTlcKoYXIr7vJj8G<sN^IsG
kRMzaG-1P%IbN@C?NDeh8XStSTNsKWES{ZvxckQvlMxHYunOc9t#vReD_d@UGlDNWXcA^Fuc^9MWdhM4!kU9#1nl!S9((#@WhjqBG+Sr@00000^oY%;v5SW4
00EJ?0gkT^2A7ZGvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_lefood.csv')

In [ ]:
# listas/externo_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Skdi9$f%38cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ|U
QJ#v&wd0IhSXHo?ycwa@(Ot2zx7U?<NkXM5n9#DM+4D4$p&uYl1ZgtI>*ILLUdZ&}+!n*kJ6Nso+Gnx73$TxhAy?mVijvJp?=iXBe|JZH=@;2sB$~B3FBRCz
LN}+vs;NJb(%ip%S$_;G7Yzp#Zm%HtN_l>GkbI+RmNcC!+DGNl9dTX50#MrR#YW=bqpX5L(#C(DKe~`ugB>%)(Z_s2ZoAmlMwPDqHWHM#$ol>=f+FnbcwNJ7
3R_B=)oZdRZTyRt%Ti9mJP^|LGmBBJOjbM>pduOEa(qaeLaG=LN?}~XE;CFGD0*R6^!j|ORSBD9r@m{dq+8(zrg_!fmZnM&KG}iQFE=kk>~AJZ_?#0VKbrq=
MRK%VC%rE_tgIYC|EZ#A*J-K#wTjlke4FRv`954dfX_L43mGwNngRq>u~c>vu>e|Qe;rSM!XR7P3)mvx6Bd_b<<*%e0$61A?&3Bzt%@)nwXb^+J*El8mydR3
B(Tfh01MHc>JJ2xYuMb5Tm{}i)fzD?kBh+$9dl2Maeyiji6o(tH&UYbINnUZp=RoX8Bs9IoTuB#$<CNVhFmerzf=7g4aWd&+Pw&dPg5Ag*g!zq+cS*U1*NWF
_}nN#6+l8ETVRHuaE?aaXcr=;Vc7_<nAW6rL>oT1_&TUb5~l$RUz!Kjv-40}ncaJJ*Ln61JRa(dhcm^eI--J~;ZERcF(NBvd-}N@6f90#bb)MdW1;?sjSEF3
IRxu6FiZ-;isfGvIPDw|T@JHA2bjd&2M5XgN*2MAzgF1~KHd&F;UP`_6J&_RJ1yW^Eoh!E&q7vT_Eo=`e^>)m;d!RJ84LE&a|!3$G#IHzY_<vo4y8#oz-(>g
@*La~TAhK5FqeZ(ag?Z*wA_5hB(0(>i+8+>pMOJN3#7py-(%=-!qfZH^#w-C)wd>|3W5ZQX+L4Hv*gpc$}n;?<>6%8j|sZIJq_DPyBd4#HiRPk!MNa8qnuY@
&6wb$Rk`UFmyFIff->#4L>HSwNdw}TDR5`;?O`b++|ujkXs#m_pzXK1(n^{fKW!&%FS0V+u2B)bcM-$pYQlc%&$S<uytTtOdIS}f=cf%y)oQ-b;W$7>=?={4
flry&%vun00~MHwaFMG8{~McjfX7~18aR<bs-~DthOl4Vdosc-C@X*?fmp0gA4ADILi>Vk*+8&zxV_Qkbs3ctM(f++HQ9cC6;65;fC#Do<BNh%@0QFk<~1fi
M~z2oN3s4KInS;C%&H|ydx}->$wZN=ld0c!*&HAu|BI0^2!<(87V+bGT2rc1eLfV4M)p-`?`Ul@0s}6^S{Xsr4rW$osM=Ji!}aXy$`ki;*WBC9sHvmQ%`UgF
DaMblxDZ>*h_ZQ7q`;g?^^o=8x4%J5?^gm_Q_1g38<P$9mMM+!fviVVWndJL<DC062xll{(J7$LIu=+5b`fZozgJNZrVn;BMzT;=UWkOB5wIk`R+0l0uC#`(
sL=B{>j);k%M0?a?nm%2k~6+<j+UB<)SBhOw{^&_C7b9fYdH1uwAF?x*OS&2S+#X9{Oyz3>z^h+(b@XQplU?#3ycF5s}=am+TA6%Zq^h;ODq*-(k&JN0zA4Z
xD_|)lazvmS4?on8<rf|p9=KO<F=aUzZ$R@!SUQaiZlWQ3B@v|gvr`nCyR-p5s5!Uu;ll@;@odR#zh!-Px`ANxA08UX}JKRo6@~!v#tO_o50uNBug4I%m=0(
P?$u1y9)`jW7}s)Y;a6)#fR0UMHJw1ODmqap@$mEDm0fUWr&@L5u$I>RZ4ytf?&@q(buPv_g+Hq7u!RLP2$mn>vu^nPjo<m(sjNSUaR8L&!|6=Ts4(WsZ>&h
+0)^Z^+E`wbC>^T!odx##RJ?bo9>|&!AEdM>Q?SA!4S{OdZo0t0;AmRf~v_Zk;afn3ujHL=wjv^!AM8JmBIYOC<#@%93+8_zh{2qaq<r3p>gVYUUXHhk+_W#
q)GuzQxU}WJn;T*2Ld#oKlk!gBtd;DwX&U_ss5L?U%=;E4QiUJSMy-EPXT`$*{LYo^_}-uj4e&-6XEJR0bkfSyBXEv-Hc5!R*CQ*;$AS4K<#mHnOSW9XFW`<
ia8jdJ>3q}Kb1FW?EAkL(yfP{Bd*CK-H3PB0o=TUv<={yEnO%B7jvF*Ljz-B>xi&#sqLvUhXVQjDp!?@zvouHYVH3kX1ez?I)6OxGcF+Dkq|Apm02yvRSWM6
uKbEren2M;%u(#s^No8|tx2(6yI+a$^%=U?o}-M6UgS%|a^{}5%ibR^h2l+~3?Ea?(9(qeFjK=Z%(VM=`HkM3EOj@^R3wtCMD2!;K9*4*I(ouBmelRSpuvY-
6?&E%@dw#7+}oaE>V-ypnTiy_gNCObq13N>T|^ujZZu^*Q0TW7W)4o;_?*#@hFp!qs&m4Cd2mK-2KB;Tkbp>NU1H)1cCV6m0L`8K$#)sf{wBhNZ)>%Oqhjl#
zFmlUT9DuoKTj?)<s!vM_~YY&VUl}gGN`UidUuf2yQmx_8y@Y}I|;MJBN=|eQ5hC*3?5}#tV-E{MgQBt2QAAVnR&1XaG=W_K*ZABKWsoRnk!~TNDOqpce14I
%AG^m>&IVx3#Em|Ttdtmu{o^p+ldzj2=)J@I2l`qa5IL$wMHq|PxZt)N4d`cpVC6Md92)(`SbU37v9{EF0Sl?V*_6tBWcW;Pm}4XNg3cCu{IlhY*3t%9W){;
4IPL3WZ}X|ilYf45Rw3WN6CGRMadqZBPoO^Gi;#54?s8;f>$6Jcha-ZCJm3$eYK&rMr@d*VJ{hZ(;7K{QkmQzH$-uF16VUv#Gye@dW|9usw4jZG7|ZQqL?X$
qe5RLJ3&^+pa;0Tr;3&EHDF!2@u@WeN4R7>df+gK(Ru+@3VPT*ks}A(2p6l8jy#1*qndUwpjDWu|E@8Ds%rYZ%v$H`ebcX!d<~pIop$LY<%O~HL!Xxx1W*a_
fM)1~{*P<?I^#1|ZABLg=?b6CNhWu>43Dp?p4&_*gcq33POo3y)h6X4JbPu|W~Yu(d}cG8c8G>;u~&nU#W3A_?h;fGM_f2MtJzAJK3Yj8YO(f#*%473{H^=t
r5woN6bw@?<Aa42AJ2JwKUT))CQM7@ZizcO`+-DB&&W5jPiU@FNZ!Be@S53NMTQ-JlL151vcYf)$XoFCKd6iGJP*d!zK4b&n%YRub&ru>^g|k-R>NA~(Q*Bc
m&^kYRxC93)%<K!I{an}!usxy>8;JW%5h);ILK6wEnTLGUD2Al!HghL?I6nRwrcF<ooG%qR+9m}wR}OXAgRnM{?+baJH*bc83+)m8C^m6FJ|c$0tIWGC?Oc4
yKS`4lUIjAu|SmwKPtOw;a_=gvunFqk&6tJZo%uhmg`rhK5cnn;p7(+^7+7Z{S=V+QLMKL&JGaejh(cPQ3A=}@rz$R2RN8<d8c1@iCQUd9l=N-;gQ=)T<RR_
go_s$SAn(-{g)Yp2)m!Ih2t|fEM#Z`>2pc>Bm6`*5k$?9p3V>U_OZv{V*rWv*oMJyQ56z{0Po)Q7xB^%E>Mzabz{{iO`x@^w3Z_t_<%!}!^x_R8=(8iTOC`6
YI{sd8>O9~zNDpPo#t<=pvmNRm)kqtb9*F^U5%>?+k@~FZ}6NMCJasNeZoPK-?0F9fXJu{lhS$%J!8pC*?#VqL)Nl~m0!-F^*CFRoKvdq6au}t!2Nc%PwhA(
REUHhGvAAd*lnVaso_F>27oD{>ytDkScyEZBwrXMSv;M}P)^2i6aIdhhk1WJKg%xdmW{8sBZJ<39$yI~MO4piyZ)a^VT0A(CC*-(OF!Rvopz!?`wmIf^jx|e
q&bB1UCg$_!&Uqs8+?0?2`-FbI4Isb`9eT}h7*}@ay&gPRe7|+;v$DR6(7?nC@|V_xhDyc;Ol|QA94&HE>uC}ODOYWR(ggNfGBQ@SQlV+9HsrwIWivXF(1*(
d9~h(e~%Eq8x-rH45&xc(t6gG<Cu7%l%#FhAzl_CiEQV*MC)uIgL-NCYlU(hO|v0IKTvOUqbGQjJH#oBtujL@%6&ZMYI1A52C}J!6YRQmvK+N5z#4g{AM$2S
Y16E#*pXyV#k4{o4ecag&#W>`i*Wuw&dy7q3!!!>nr;1*DSB8keIaRKr>`(*rOwC>n3|>TT`Q`P_3%H3X&`2ksv#hWn%JFMMK0H@M(#a#Jqp-B;n5#>k}+l;
dfaifBo>5$7Ba<;vLQoOnNCRqg0iZlN7JaZ6|5-?+89k3+MySHC;Tj~`hj9dTuo)U`h!y;x52IbtTtK*i)wZj4$G-mQbvzMCg{lo%<x7HzOdZ<m;}v$##Cid
=6m@1qq(rn<4|cjzE?CwqP9T?0n@<JY_YnS`JVK^@o<u@nGfN-@Xv+&URs4z7~8+sTa#s=oT9<%Qdj_pc*XHjpdFg3oXLR;P<svoth1@A^sx?^CKD{$B^9$N
doI+Q89cXBR=E)jb`G~{3k-j(<*lCEqH5&X0=dU4-H*Sh5ja<f8;e%3@Qw>%B#0puFdukPKD-h3aLEyD#>oMk0uaG^i=o88hse;SRz#9<a^apeaA~0QVyY?)
a%US!m$riw`Qm7&bs#6l8I$ES=>nlqgx|XfJ<Qanq<nTGTan&){4@MdoSELRW>|abagVr}^Q*U}Az|4A8_)~`ArwKhfqE)u$ql(!@8qp@!Jzf}2eqyk#{;ap
D#u4ft<D(&5CTWrN!}G5gvbI~7#GNhMsTV3Xq9EHXhP=T;23hR`~JQfML|NO;KO;!u%a-y-yo6#<SR#3OQTTCW{2!F%<6sBiXhzex25|^%5HhcXNfomUO(^1
6wI2VliIEGWrEwjf>UN(eY`xha_C=6X)`dBnTl1$e>(j4dEUV{`Zj}%tjmCpBC3dStls2WDWFk*a6z$;2ZjXCe0gRVj>w)ISM2sNHg>Mu94K>+!(7ceP2GYR
n(Q1KRfF;157C=ZZa!*_G99-4qufgn)aL`=cJpj-)kvC@_iP>8@JnBdjX=#}fav*~7K~U?lI6${yB<Iqdg3;&&M&r=9bi_RB@GgjeL<KjjdPRq2Ya6{QOsMz
oI}3ATNEl|r92M$T^sQ=GSN$J;ei#y`}mH>e*O7(<=kWVPJJ3k-0#cxnLrLtYlYipX)i+WlHo15$v#;C{6Cg^ik;CY+?<%$<dcwMx#z}vTM}%TeGDajQiMsi
z`_#xzAhUCn4%3NC)3ZpI7~0X^wDO-(}j>6{&b#>q8HQS6aCFnF0<SM|K&S}AZMh3hXdAOy1XgKpC{|oz3>7NfhAFb8Q_m!dqW}aY(ZA4?xfJ@Bln{h5dZ)H
YP>1$o|liW00Fup)S(0bH$>&uvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_mensa.csv')

In [ ]:
# listas/procedencia_r4.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/procedencia_r4.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~)^Sxm^G~f`9=c<Fp)p>rCx{$66ocxA48)4nzoP9DCX6`0NDs?Tl6E^p9u1wQHc;BXPHKLy7}Jp4u&1v%#^Kb0k%6
iO)dymlR5q(EsYY=0)wNVX~Vb@n?nW_<@N64aEOUwp2o&e+v=6+06BaN6_2tvludfaq-ou^Ed`I<)DW&4MGymJ-_tGt1maNv<L&y_3Bq$NbS<nG&})9M4RlJ
%fYTsg=wAs+xxi!Tx7Sgcc3s&i$^#Ed8*7+Qu#D$Zj|Vl0&Y2o^Cj#!71p+F<t^&HN0U~MpbuobjBA_td5^CBEQXa3CQgr07nXB~%uQ>m@<;5(Ajlm+y?Pac
o60f6=M!P6#iQXRxgmFv|0k2`nBma|h9#J3K~q#gM$_G@3}iLveme_aI6~{R1=tPH0z8A-g=3{j9jiEh8+#56LBc?V2eIswGx|Ibj*6)xL|iZ5G`||r=n=LM
bGHo@)54WlP<P-%bXAm6^G$fox?XQ$kD={t{)*FV0@?<%wBX;uDEle^b5oM7^>F*FSUjc#T3n1hcjPSq%xx40__i4aXLjXK$c)MM>eUJcAzFV|00000<dRtn
=3KVW00GqljRyb#@>_b`vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/procedencia_r4.json')

In [ ]:
# listas/ref_r5_prato.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/ref_r5_prato.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;l0g$_gw%o8cNL}xSP$f%+<@`qVenIyA@A}Bd&^EP5}<II>laObrWm3&=J04vdTS5z)<rt-fpe<h`wBBT~$IP-|pJ^
J}Hzx<hk6unR8J(^nC{j^kAV^uM^q7I{M~>h!&Kh^tXFKH%fv5iy8QYK%QQw18J;)@`r#!g~yPz7{ux%%v35M2)(66F7EEQYSf<jU2;r8Cc6P-9Ne8^-mx)g
V9dxdZ?XchB@*x=nars^jgC;5Uj<IXkk4N$o|v^Uo)r6s+wNT<%rI^TTHWC7L0nWgm4&enm12_Ly)j%|gw|)278VXhr<}l}C(7i7@eOWh++NYEI+bI8cs|MV
1P!Cx+u$e4%GNC8vGn;=BeDZNnJO`x8^jA*M?WB*$ZZk4_}Enj1MMiAgb5Ngs=ALRV*TiaDUx*5xFCN7Nxu!%M7pUe9#e>HB&pjYI_S<8zZhW^{OXN2o24Gv
PVV0x;@EDWMQ+x|1i`fBj>8?HS{;K3sGymMoT;4BD@s3rF8C|<{W&}EQ$hnsPbMXL`YwtEMg<_gh5uAnFuPcVysB`(S&ZJa4F2n?w6zdHRAkA9-JSb1=}j0g
GmK(Q=yagtW~9sXpfNITWYqbDAD5gn(U3<PvEge2bIB-0j!15FH9L$cs&mQC`I@|W#wUSU=RxG~%-ANeJ?}Zlw-ZhFj2}*JI0BP16lkL|h&-#LiKf%Dg$I*Z
>+s^Bb-CbjnhI(-Mj=z@OW3mO_Ldk`bK2RTa#;vO8Y}M+{D1yIg(;Oez7n5!FOIUnkdf&C{4KQ%?@I?MLuWu82)n;ZRTjRd*B@y#m|zr{Ho$GFg8bqPdL$2N
Zk$W*-0|peWBhkP=+3Yto~x>;O3^P;`&KfSJSQYUO$g4mqW6Z5|EJLmzDS(&7PH{Yp!n=tV0~|$XM^+;y_^uJ#AYLnzkC2N#?aC08%Up9ZlPxh{AUx5%?_EW
SE$TT@pMFQoI*dlDVz1sM_CMyHy8}6Vx9R#+=+ia|7p2w2ffLG1)3E?p?YLaGGB+5x^I*Bazep9F1s`XVY$k=Siuvx?lEf>)|cR_x&x^=zLb4I6wbf}=C3ZB
a1T!A5;XSK;7Kz-^I`3E>+|m<_=nlW;gCEBt-1rU)mLiRS6*vBGoyxWk~MaQw1<M+Z5dKfz|v2pkP$3>{Ww&x`r`#jv%wY+%uAA{P5hI%V{0Sca9h%jZF7zL
7Sc;gXd|cXB<{L0<nIIQ68#d54e&?YQ2s~RatRI{$vXpQwTM6%%GV^dWqZB9q;MJpA<dMq%oh=$<x?4+(J^1vp)29f5?NqvOc%?1A~3YfD}0iyWT0w3q_V?}
+TLwFC)h<2j(zN)sqY^4{&%}h0!cVl1~Z##B&dTmg&y<<gm%I=6L!4Z(GPRSa=386*}ZmA{Wr(5m&}Iq<3fc40&ax-Ua0_Vc2Mxo#p4Vk{TV!^km228#5)mN
tId?9%EiQ3L8RoBQNIZkJR)GIZsFv{`Zusmr(@y>=!f>QjFs`gcOf}Q@LZ`zR@+}^_XT5otdns&Q9mbDrBU%t5pHCyHk%V>@=GlZG6{~)J(CBvqTVQ_ARr#@
u9C0p%JD<<Tn~10Zn%m|iB}7grk<Zuz1j(9vFDa1ZzT=~+L#?a(F@Cw6i=520U@cE>m6F1XP4wG<hLsxhZnA>Y=p?atkWYWi$W&iv~{JuNCfWl^>!n3?qz?*
Pa+^TTlfODtbA_glV}g}dUI>f*lrKPcVDAj{gC(f`+ocZzhBF9F1#QS$YCG-5Bum9FiQm-%()_#=n;yoySI8)(+7~fe2A@$XrK41m1qpy*VGIM1)O@3o5#4Z
<*+Rb=_2!*?#YPB7NN;Kal&!78j_hfnS`h;BeC)VJ1e<Dz8&B<gobNo<8Ym97K*;;RNPruPu+23#f_gKX_xSYWY4N0jre<j6S_IRe3H5M2>!Ws!>>03jVrMC
yPe5u*jjw2HlEj?!FQjukC?Lf{Q<Nsu#1Jwi2Xs4qtEN1(V(gVbatpED%-_-JrFonrO`f^8C(f$?FnX}Q21?F2UfAo*|XNAvq0QssrP2^d89}(w+%I-B4R1c
0#s58eRVrRX{9qo9_&dNU-PZ&qH<<(FFo6f@pm)#QVuU`?==rvM*dx#!?EW=tym^4AP{Te##0x}?LxuTx+Z3c7zc#0jSN0Sl6G0KvqInuar_EyJVxU+b#%CE
E(y5kurgylUd#lMV+4Dn&&QR7eQw?}if25j<ND=aJyiK=Gc{{kNwgcy{l_^#AeXjkQ`le2yglF#9RAa~^ucG>#yLo$TBaCI(05xNR_*K>W9FlrT{(6ad(f>`
-uYgkR)x6YVjnuc7^?NnxnqYDgxv__7zRK1hMmTxH`%Tj`CJwd`%Jk3)ojF5Boc;8Hh|=7{;uSRP)M9|sf;=*SE3z)jtwiexmTrhQyM^jSanUpz6XV$c8J9X
P??iQ7L{Jm<>W3<GB$S$p0I=Zv}RVr4WQKA#?yO-7zA*UfDz?Vj*A2?EH%-OrdU|@X+mjt>5DY|;GX+$TSr^}UTBf#KfU+(n?Wg*Sfk=q2s<=Gs7FNo!-yht
AMJQdZ;I}4xS~|1O#%bfLGau-Q+ZUNM`Qrj84-(`(~aw_oIx56<DhR%FHZl}d>Lm9Cw|~r)GgYAS*Oa8fs#id#tC!4#aSc^@10?UiqA_cE<`j`E<kbEVG5$$
Q0xvQniEXh3))H)TfuRWc4SFWOeR^;*j;Bimn1)t{J6!(0ZF_S?$A%19He&a^haUDF&=T+x<jN{mpwrYU$1yOH*DD2(qy<OC^zYHQ)XkkoIdb$0E;OHX-#TO
9BYbEp<|v;zR)j~j`IAj{ipqEO?dekt*wNNsxsSfStL+Y*+FQ^h#CE`PW&);m{OV&vf&#zc$ivf&J@<_FY#jAv|CD#d{y-d<NEcL+)~e(8d0qg*>#V3sV9-#
J+~2|(HnCGDXSKcpJ~D0Fzwar4`&w(1EL$~*rVKuLurp>NZpuyvBqLR@x`95jm4_{cx)7oUjAruJsK8n@HgkMFk#W3f$N6yC<@fU_IH#iEmBJBR|GM6*_}K0
?ZS?_%e%Zs1{>{k7q^_@6CyMZyAi+D5xaH-{6pXrSKO$*t}caRC;e?S{dklNVEm4rL*p4l!bR{~dt2SmW@)Qz6s9?o1hr{<kX8?Hr+X*Pp=Qn^$9}sA@eOA0
`XG`gDIY=tOi0cnM3Mfh=WkILdVf<%w9AJIbdS*n=rlSsk|F|?sdVoT%q7FjxC4+;nr~cmWN4rMMtC%_ls^rVQ2oVG3U7mMcpENoXn^Z4cl>0~5%nnLR|i1)
?~Z?=C9Tc-sgF%ok50p~C-<Hl#$z6A&m`;nWs@-VC|CGQO@i5CoGGm$wB}<SycsvmNa?~Bf?QLRu?T}i`{aoaI=T)vGcT1oe<<dUf2SY<Ek0FL?(s8(en4iP
jC6E%-oT(yylEvRr!KklVA}m3cfJbQC((Ih*;ne0c<GryvZ!P$*~Cfto~i0@2xgv*)Jo)!>Koi`u$F5Adj7{J>|d<KFWPCuPeEAsiP`Yzm!t`vGb=nokB^;+
Ea~Ob!B-IyK*lFJ3I{;SY!8dVex~qTKW>A;3C&^REJ(XQB;LssP&yOK+XR2XK~wQQ)e3w8RU#%wCFAB0<z_KcUZ9rHk#Ea&+<}P~qoPa)P*Cb=0X91&dpy~c
dOv_E)Aaz8Gz4wrLD32K;urT1yr=HFzhFuM?3c4sI#BJWjD5FJiKo8mqttsy)seLobU3>{93ypMuA(Ev4%@74P`FKO-(>c?r|G}CAjfImUuBQsv!`jOAf7}{
)Q8~Gy#btJLQL7?m;_!eYvb8=4E`p|e7i}?6vz4nJGw22`18@+9lb5Mt!mM*A1~98vW6<8BP~ka0C|PYS<+a2=C@wI6%0epBT<8eECgGwbAYWE(TCeZY#ORK
H|Ri1@>XxDSLyem{=Z;}7cZUKY}Am|o4JNTqKCUJtuo@$!uu}h-^P@$B>2dDPPe|NSC=aIFx{}Hp)TZXPC@bsJ`H6eh`Gk(wc8FB-5F)ThNMa2-bq42WXY}I
#%+$tJm@Dte$<@g4y~uAJ(@CNn@Dd5^;9Cl#LhNjM_)ssETGRtG}u^y5eU3vTMw~@*4S2~sof)S8ha5r4D#A7CCtizJ6<xUaQ7yomOU`tG^J!f{>4>p+0KTY
$f;>*(1cB-@>t0qkmfQ|@iM6u&tMjinL$cuZoiE2sIUsF4u+>R&$w>;6^XYmtxzRWE#1Jf%hE8VRjQ)ByQZHV9*dqp<PI8h7nJ+A19P3vY*8NV^JvdBHN<tG
;}kDwc7RF4LS)n+OQ5~ob}20St875nP=N4tgto%2l4qLev~n*U<7JslpN`~d+M){jcKkrmu^vH}X(eNNFtWilT&^0<SL61W7_(3UEVX&9lxq8K{d4-Qu=C9&
)WLdXoo3pt+UX`;77!tNMUAxN-gt^AXG-h|HUMj<Qw*rFUTeynpvmty+Zd+}91-K@_@N1X$AllN&#v0OF4P++SPEwvicTZ%1h3DE&?;n6(GJH39}&)wU}mJ3
dMiU1%ILp-z|Cu?!_{|TX}l0k;MwBeKzFh0YEhUKGJ~m&M++m5JRHV|BbLF@v0G~Zq8$DMsuzn2T+}s>wu3klPwq4bDi$UAq*CVU_<KHXM(N%>FFf&dt;dDu
3i@+B>}5+7<FLrc@?Mq*v{?<?%{(DFPT>YfL*W^uW#hY%wh~#}f<oHPV99{`{YAw^N(y@c-R!jFcTG()NoQaD{w{K)c<rj!5aL?+KRlE}C}epd>OroL(CK_M
#iwmQ^5)GwdZsuOo0kpXQewfL(xh{Mw688X+G)0H*4^@+Cj1-I;^HSpLHAhccZYO}$j${yu0rd#0Y|t?h~D?dT9_Ac^6DmzxJnbQ_N?Kd3tMofaXY8pT+bQ!
>^6mJ<iw}CGOI<qZH|&51+n>89HAVPnz>WXe)HiHGJ1YlT#wdZERk!K_$`^^jdqo?m?MV`5&PPCz_ae!t*qNp+}&F^Zf(#*I#M&V99<j~E4F(rRAe5OuCaS#
%2<mrg*QZrf}q}M_nh`j;kMhgcK@!%&l1L-Ove0!iB$mrt*JLmfxQ>x1^;wSALLOmvk46pS`3ej;-=vRr0x=Lp3a>t4IWlsJGl8LQGZ!{5nj2TV}@I{dVA((
atAF+&E74>E)}7!*vwLKwQ>73ZL(%DvsT*c3vG|;9O@4vQ>n&YIYp4sv?B!&h2$5zIL;VH4n?0w)0zGn9USX%gdHYjENW&^HobE)rC4!mln8H+ga%_v_9O^G
fyBPS-=bmW6{aGidmpJAx&G6=>Tks>E7&kh&Va4S7mf7dkJmYOuZ#y2eBC!jHdf#_Z$Yj@c6BL25Ja>0;V@BG^T&PZ=6H#<+V7oHLonMFw_gpuPz0)>ByZPe
1?7U`k<@l_mNXa7Dbt8$kM=U7bE#w1J<3V;Ciud)y-K|OEE?DX&C}8{4Od|TQh8*^jdB1j3I-@!`JtO*;ax&NS^A||*bZ2!ogJE}O|MVlq?e$~Kcz#;B^;U8
BG&iE6BQI{kR?5%*mrT^ftJeKEkkIl?s%4u!zMT>i2UA~AdE!@OpYh~FT8>or{LPJOg%Lkcw!3(7~PA#cM}9q1=}<hvOs=RS8*^y^52trS+FsZov5GeNm1F(
;KWZI4MvmJZIyeN=9*!tDv8;|<6Vzb$WHLXsOSt~k=K&5VR3UrbJ(mEmZPRyKl-I{bM)mHaw6~G<m5e1Na$H$qE|qjp87PwKow0^>NMz3tZzn5RGPm>g#R=J
Yien!=fD=1=Txc|@=EURnohp#aZ8>5zL*cLBl_}!uIfMiQo|2(sjRsEIo(j2&AWpR90%Vfzirn3CW0uh4_%&tjA7oqK3Hjwy>32a%+hB8*@1=2Usv_1s0Zn9
0rzl`@X~)h<?}(}6KS&LvrFBn^z#UR>%CgYM5nDhaTE44JGe>@f;z=d9jB~wfWf6Ki62Hw%i4kUrC~~yjfUWmYMdfsQ`FkF;}Dvi1T@&X)z+Ybk@T-{{Fjr7
O?&)0C>h48R+%}HZ{;lcspOb4c>E{qmzU*@4|H8{nMwy&Z@wLqRn|~;yGDB(k@8z)3a<RW)>GYSMfe6$tkAA06jvf_KMHLLq2<AB3jjo-lP6uV{VWoVuIQnb
B=U1;R-@{ejLY;}lk%uy*=cY{IT#-XC&pqXRr0dR5+a13e`d#3Gt^6gmK)k|2e|~SLk2I0h2p&|^iyIt+&Cv1NJv3pQK=y`8texOAYwQjse#Ia-q3y%(?8wB
4nw^4ak#sQ{;hvkIDujEw*2Pl6cuI%$KrY+TV=v6ydTF(T=|ton`lajzB4j^GzG`C&w&JXmZJziz&+7h-=P*=kDm>o9Za+1GQ}GX*Vf6K=U#gah_74MTjU+4
sz(zU+>6l|9fMjYd#NmF(U2+T<~-I7i^`B0AX!XdaoS{Pt!RG<z7-qB;ho*;Ol~@{-Bl(sAvQ~VnIB#R?(9tyH`~@5F+{Wn1Qk8zAVxh=CK>v331e$^GG@b@
hKQDz{5$v|{NpG!vti;>j@^nu@Lt{{;?FQ93jDM&{rW`Vg=1E7yD=N6wn5L|SASUt#{u#$i;#S>hDtA%_H9axwY8Pt{t}BmJvL%Hp&A7ZAoW@Z0$!QY-=}x2
<|w#WVt?|HenRrD#<xAB_+mmX)jkenRc?{NYu0Bw({3=AD^S%?!tWQGJ|8*7;P*=7A@+<K9O^OP6z(t7)cdpe>O6;O=q?QCBp?+@TKjoZ-qF>C`u*@1-p4L!
FfhT?FRNC|1JzHq?}am+{cyU=<bH=rhILbuQ2c5%9xH%1G>o}+7u*2oy|h;=v3A?WS@e9RQ8YA#OAvwe&fK1Bt)+@T%q&mS`8{gGb4i0btg@<pbLn)K>YlDC
Q6sb*X98G$LuZr+4dcRDo=(ag4f)m9rf^u1+lzTa&GN0XXn`g=WThuaudwh8jC+;-k(&KiP`*Eprin=%Hygr8UDB*K%Ja0<ML*5tM+VdKDJ*X3v$|fJ<1S%7
TkF1Q+)OGqsIV$_PU-CxD07R$@y1gMk3d%FhKI{<RFw!1+y)%=7(?a)Fw+tt3zI~PY4Z&n{N(pYVqP^3>>gueQW}-0{uvAkQ<Qc8+Kt!w=`prsk_qTk=+?I4
aP^{WF4!qfH{@9f1$YQ8snQgFFX#@FzOe?}doRr-<Qv!3em7%3&9(=h%_(tjw#4{)7Xc(+5>5JV(5ucpW~f6UWT|QBk$Ab;6S_o_3*AUqxF6#-Z+#ZN-Vw^%
Rs5D4y|QR}V#6((X-;a?cBU{?WVQc@W2(_Kv^0!(W6E?Ii<<UlcRxWFw^@%NV;Q`_DiPW4<YI8`L83PS^*a@1;D-*|fy5#T1+R@Fr<@vSESWQ+7RC2K*=b7_
1C%=^WgMQYeb^Q%-v^&su$k-?_*HTrxc)Ij?ws7{JP?k#L9HA3IEzqOKMsx5c<ek_gUD%S(6P+-yJEhgqdC0ozv}m0A(X#Hnflsf+K=vMTVRT36ct&CrQ9$)
M#q5kxJdfC>zb7KB5&@$7|9s|Y+Kv%_q}{F<~tloqKF5r1r83Ze3Q|g)yY=&;)I>3wk#1!1IADXymY6#2VMB7SU%^HqWAO3OUE4b`(uxBWxS^2e7+ATB27<p
R6>=R2)iIrzOEb=n(%RmuDlGkQYtSN|F<7ZINgrN$AL<WBuK8gWKBWBmznE`O<2^>Ogur3IxxKz??X-aEHCW^i@z6CC_mDR^HMzw2YHzU#jVaRfLbWRu(MsH
N$5l24~Xu$#`x9pZ_wC&iTjH{fxUl?`8oG35EKyGg8^&|kg+c*gUB4I8*yWSE8s$)ZTgx`R#vfa%;_P-9p{00uj3=NFXr%{juxXUF~xH+OJ6fgjg%w2VhjP9
1xjvL=kiU5R|8bT$33p?5qWy&qu%1_StBXLdbYkKp_(vCas))L%wrqEQdg>(o#N0xP6Ii(*C=5|Rx%ByJa_xwlts9fv5H<oOnMSSjG_mIAWS_d_Xtg*{aF~o
+_D8|F@L#P!w&oFZPPhPMyj8g>8fCz&d5-#R}5+vK0+O$y`^{1>;41!llONsYy|3SOAtF?5bg`U5NkINQQc0YP~xmgyDh_bn+L_>jck4K+uEcui33l6i&*WJ
P76AbOFZ6GVn_;;Y=@$Rkq@06bIR9=9Rm+5@T;Xsad><N;9~4+9o|*8@39A{V1&&<(0!FrSO|~wyuB#(Ml5mCvh>UNnZ6Tp9o&#^>?0EOxWBW?i!OkG8|Rzy
#)6`&Q>Jd#51|oR-0Z#{x9k58CNSC1D0?+;Z7+!W9y<@f!bPD%QWZdNZYn^#{C^B(TiveMsCe9ta^svFF{)Xul9PwzW0W;~h-Y8>nGk;q<unYjPJeg&j-m+V
ZmDms{FlX1D={qWnUP5v7mIp2sX>ZfxH2(etk>N6i%gHU5bcU4d;4BLRuAYoZ@V+K^*BXqz0J}*&k97r?_f;u-&b@qgE6a7G3vfMXBF=4uiH`=flC6vwF84y
OE)qfQ2hLbx8{$&f%Eli4#k&XUj1w=RvM^3I+*Z!for+Bg+CxqgCMdmKqso9Y*mn3KWtk%8U<8dA&z)ELlJ+(Y~-N+@tyd0stju=4ILZwNtmD=^PClA0|Dp^
WcaDG=I&R%1GBA{J8+u+br~Y2Xv6sWKl#pb(cJIcHobmRXPoBZyNXoXbR39o17Ov%HsO8i&;t%J)`E|*I>;0Yy3fH7uG6Lz$C~0UMa`Gy&Xa1W7|eA6$Ic_V
N;~10x;M3_u^=SNTS4BBl1_WvY!2*KkXop@;CKD_8JnMoq&DqvW$SCYiEr1BmssYhtZvW`G$Wn?EB<q8H4G`)ooK^a;G;g~=62D}KRu^-q6Z`Lt=KsVr;Yc?
jah#Mrg4K+DJ&y!b#~<_fCgewi-K^YN{#y9a(?!qb6#Xj5ck-z@FS|F&&2iN;x-z;{eo}9?rd&rS-&?N%ej^v4z?6YVxuB}OVy`uwE{*L)|kEaY={SJ8xm!q
k(>oas;Vs-pmrk7jv<X~<BQ<_a{!jYV+?la5!Ur*toJ-L{k2(!h$EPv3qZ6-is#~cv)^MQrIDN#(7WT?RRgGQZP~E@rE}1H9gl4|HjGt*Q^HS$H37!Mg_sIx
CAW~ZPH@NTY7&Hbh44?n95270kjd7R@4HB~W$q&~Cu^`Qj54GuBa7JouDFD~#$AG%k8Um!=bQ7gCN|nDR@PH!R*C$igM62%7m?Ao8Bi$FeL8O(Jk_jQ2H(zj
M`yFQ+Q}c(n~A+_D74071u-Oq*?5~6ff?q^9?cRQkOQR~w}*%|uN~GhHVtjNeBN_2w6w$vIf+S=GCnIDI42po-@X94pT#%7gF^k`A*P4X1MacO$h}0*cbRF5
-%DD!StpRb+klQ15Q>M0rmPqt^o(q!o|e|A+G-q+EVYRP&7E2<a9G0wH2o&BnSL>U_yVQ89EpEQ@JPwgPi%u_gcNv@J4zl9(&`5UGt(0)8z2IgZN)sZzz)i2
L@!Dw>SQtiUwAKsbJOS;>hEa#ytoJ{%)8l;7mw}?DK<t)$Tuuq3HZJd$*fgKpdw~4X8k#;WVurJ5TS_>U7KZ&PT?r>*^*g%IF>3v!D2N<8H?%<ZjRiHIDtT-
D`GF7%*0U1NCwcW;CLmRaf6)op)vF{txHb?34d2fOn3viHntjGDkitSbt#!E;xeP=TL*grX<FL9A50Zvg+uZBpNboCg834>3Fj^x0{)*7OT(;eK(Axh%|CRN
?mYaV#D_eu)U73fAnAzjK{_xUa2L&0E9qJy-ZuoxavT3KK@bdgY-cOKw=sDdIiGN>4pmAxzM_7Vfq`I+#yYFf(rx>UbQU!CdTtxOE`RnOW=I$>IuKs;GuoId
3O1Z584fOdYp3}tpuzYv#%W*{SL)soa`a`DoEmB!*dY1x4|rW{QLOU+9Q)qt`iOCEXt}z7iIjRr{fx(Fp=n8yiFu!Sw~vmp-JW_;qfTB%9E3qw$7h1pKbe0X
p&v1SEA(V}DHf9Q0#$)uptFKXu>Ts!7O`|8%6L72ffw_Mm#u&XBMj%K7&$7MGvNI{g5VeC4{{l;7*BkJ6OTBPTb9=M4}x0sOz{DQ(STGT8!up+vKsP`mnIp~
mjsAvIgO5s2OoClDaGqRRGN{NhwI`NO3r^~CgGI{M4bhv0HUtbtz7bvr#it3Tg4vt)bb27arg%KU%%4F%-yOs^2la6ZkC3@0kw$Z--9&8ft<>#=_i(HC1FUU
^y&6>;F)P^&=!(fJJX_tRdknlSgj!xx=K^6?RU@Yw6}Em%Q9$7zN@XRf{Jpgk$z`6EljYtI)q@~+%1;B>Y8+sv11%J=~JFSwXFS&RFy#PO6!?0BWkxtUpcF2
a5OVtEJ)a1;^L9<zw$Zc<{X`WdCn3u3KxUK7P_eiRfYRSd@|U3I0t@7slelgl~HwwewU;oSckTzenEo82%lZ(D}Y!A4ub#HzuNo!WZ<2uoPkovD9PNmSvV%c
6Nz7_9`CdgL4<)o$@@G&>I7q`7wWQh*0urP=j{-_l;LjE)33&OjeD}Xm`4B+RLOv+q#NYYKgj?_&4Op!Faj1x4Pzq2yx3IgFn5DbE*EQjHp7sDDvCwlfG5e6
8KMu{Zb!p|EX!%i5YNCVNSs(VUe_+>fVKOV-nQMHHN1jZslPeMGcEnO>Gx>_`eROl$QNxL=J-M`N~-}C2ki*A<CG+j6o-kn!0Dtia18}{gXny#N49I$=K#7`
h|F$j6{Xh*EL+@4o&Vy~)<9<t!@TkfgG2j~k)&w)yV4cRGe-(0cDye|QCmKce+8B+doxHH;-QUn#1m&Mo^Sl+VEFAmy9E4swFJ}y44mXcXh{pH<mHVC%tB0e
t8vJ#B{lhRB}uye54TEW-1W(8CHDjGP;vOW?V7}rG-CNOnSTj0RtHIt(=$<HHqy0#T>V#3+<;-`06~QFDGJA015W81bR6R_cQ5wE_X`5?!`zyn;CJS;*d12g
W+fisej2e1v!bvF_BO_(AVl)Zhq+Z!@PSmKvN)`<AFKq|iu5RH1JJ#YnA6gY?h<(4m+zZzkeiS+Pqh$_0glxaj>9R%-J4%|8JchGhAxPQUoggtPfm=!J5xD{
J>4+_4mQg;n~_f>dAG1t)i0SkeY6MovPi;OsR{v<bl7_%GoLnN%Sk|biESC={v>MBuXUfSdF!bxU{UhwI-AEmTiP4k4xnkYZwy_kYA28D(}fWVh*kFnS=2&C
4XU#*oq3kOY{I#eiPT!zr7N=B4!uNsa*1CPEy<s0Rl-j_5a<WoC6r3zp9Ky)d@~XHY51$h)b9#npB`<AJkZY$4M!8r%AlEbj6}iLH2K-Cg7_`Y`<?0Af0Ly#
!IU%yGo-Gickv_WBYA8kaNU4I;apNjo%oR0?W&Bxf`p_UDqWU8dc`uK-j_LWze*^VS4XrKSw+##tVv75fTq`|o!I>w{pec`1b;9&d-VURqZ1>329>2#jlM*L
5FeZpdmmt&xssE!*a{&3NHPN0EsYyRG7=g|u!1?<xgh2KdO`Sg2)q?dYDE1o@q9bc=<Fe4t38b%?~HvQ6vBMZ$|4I%C@9cLao!W$P43KBCfJs^g#cgSCq$aR
&<USZIr!Q2jx=kxvW6@h6|?6Ek#-%}<Yo7e5{Ujw1Z!khmwg&`LXes)N!N+^L=7VrfM>s%R4kTj6cqp@q)A>NG*`W51y?*{;HzDmzb+A&DDBK*4)Xp`y+OB_
(*G;eBTyT6OH1uWm-DkYOO*$)qyLe|6g@mRJhh_OJ(K2@;f`w!qfCKA`=4$wrHseKra;QTVU&#O#&RjGeYPiwU`@^tOfLSRlepvI0NFnO8qMp8g80zrzmk~q
7L06#aqux@D6frWVtmZL9AD_%uAXyanvY&yQ#|e#W~y+-z{m9!I0Tl#$CJYV)l%5re0qE9QEzwGZ~4&wfUQSCOrXWTGUD7wOv2KGGB|)mLtP!E=4{J3h5wsp
|4-*34Ey$v%8cLY`$HdELkb;<C+gh9PS5ZJPP4e{!9NOYjce%%OW+i}ZNP#JFvaM$?A8aP48<(ohzg$Mww_6`DlG_#*;WWh2W`NH3k5w`C8sNniyWBa`0_s;
eh?te*V~WEgnrm=^z=f5DZSZK-0Iz`#~N4SL)6Wh8`R0dQPvWWp;J`=(LA`T8W>H;P(?=o_w?0dfDhJ>sQrv8LVL<#!>^BFq0Tlrrl;mZ38=}bxImBIj@5;W
%t${#oI3p25sH^|!C4DSy3JS%16KbmIn@#@H3qLltghK+4bgc>#S>36N5sOkL2B3UqHdS0U#MHF8;_7eS<}W_Kn1mnw@=5{75@l!MvFGceYbJG{hH!9e!k%+
|2y?|aViHET@+q8{X&E$0?F9SO^~UQsf4|IhCZtRn_-yoEG_2;u#-q>vi3RtV9JJM*)>HbpcbQ&BtQK+8YLS7inyL#PNhN?fs79E38|W=UCIKu6GL+QjPw?j
O^T3mk&{tG%Yw$LMouZsIB>Wm9eE84VB9C_S-8ZRdq1xWvQ*V)Q<<HTVJ{cei;6Nn%^yUo1xl`(NhFet8jIiwyEQ}O(t=RMbby33=iA{Iwg~E8{C`D}GhAod
pRZieLA7ZXGNY8Rzo`3}@M2=S599f_$vRrPlW;@^R}u-O{4<#RVh(h$8ao)SZ{dT{_>q1H9!&)Oq~$Ml+3kB)w-cR1J3WXt{ye<5G{|r?aFERI$_r;8p2H{g
{HB0}oez8CT3=u&%7+a1UVPf#&QQ`&hGV{*+ktwvFi^xF_et?=0VQ~#o=_MV2H0NP)Sw-mI@jZZr|ViwF1&}UyCjK1%R~kLxH+o0zXAo>bLxrZLKi@w0+!vt
y|Pf1e=Z6)we58x{;3=c>%VMcJvxRM&tI3CmbZ2u7aJvR?brF{JB;;<BoG|r1mq6Z*^nsrfAYg4nR&{lQ+oK5i|J!~Mj0}pEOMRVMK^L%>yB+B^%T*<rXLnG
IB@%WKw98}6fn`R>@4K8>eO5Su#jgsjNOKWy6`DBaEwl$>BJq_<%`T;=1S^masZUF!;~Iwo!(B3#M@geGg_Q8nLOTWU(*;4vg*?9G0MC%w3XqWe&Jv)3Ba7e
EpD$0?z;NjsThYn8Iq4zm-zFP;YOLNO|KINjtLYwV#`zDT`}JaQ8fG7Op6a;UiJh^dA;-YQO8876)2Lk5HiaKIY|hEF=%H4W^rFyNOvib{nPV5x&-I|WbthM
uim1~xFoaD@VR>SIH+X}k@Pkb=9zPLBe*wDBaffR46;#eBFuml9!93j`1=OekK>k<;+ZxOlN9i<M?(M5k{8xCng<1uLf(gv%Wc^-_6b)4V(qERSc&V_8a>h>
r*uV`vaWwE+*)DF&Wgv_(2>Y15^Du4%AIJRZ3hS46v8S%Nlez@fN9)N1vi?2W1x9$Yan_te$F|^a4yF3x7sERcziXTl`FA{`*9Vg8Tu@)zq@t{9*k2$FVfxC
J^Bf2f#Gx*6AV*gqDht8f9te|!gEFZuSUZbiesn@-$e`8?&9KpQNONb6tx0vur2IKpm;B6#S1bEvY=0~{3IQG@nfeTt3>b~7}nN~3IB?seX-s+v+A&vs+kCG
o~T>QWf$c0s;5I&kxq3pPW;wX=MS~dJ*njvnN#}!KUIS`xciybD7Xh>|Iw5S1R60F4*_8nBcbjNKzwXrkBGu{!@*6av1mJh4!uufcv-L#lOJ<~!0;SOZhRO?
1KW{fkdR3o_o2FC;)NH-I$l6dL@10kyD<fnq(to<mib`k4cOWt)#LQd(#$IR#@FH$z(ry$2!dF;L(pi?hN%hi0ZbH@|Idrgqg5}WL*3@Qa$LD;RaM|L$i6o;
Vg!HylZD>oHuI$MXY?D!s=hlst_tyXhkyy(G4{xkKd_v57*<d4DI0*OrPaM5<PTgbMLH~3hbvHA64WKM$cc78O7lwAaz^P6sjCF!pq)?(M`f-K8l1}2{WIjj
way<GC?S8Oxf{>x>+*yee@O?atrcsT@M}Xfu|3?2`F(zIW2`ChxtPoileOGfKqh^4nCc+o7k|nsO8GV>`>VFk?2Xy9#WXtXPk*^q=wm7%KG#AdV@<(C0+4I$
yE+PV`5N4l$J8oslANO;n~AN;&8w`4x_i{Wpu1EapXd~i^0&q{`5uM=kK+2~Xl`F!2(Wa)ptxvw#at@9QHx8n*mriUG-$0)Et%4FJT?bN<sA)i27FUf_-%un
Wy||}<|cct0EJokQP>BQy9eF&=8UjM^~_2r*8^pX?>KM=K{)_$iVggXK+0!lQ)k8fnJh}2C%jzAFxNYdq<8Vi6SIbx1O#l%%oUkr!wqjStjriud#u;S;I(Uc
QX-ENa?y}Ky9>vDsM*L7)*NBvm}cG=JLi^@`}J$iT-8uMlEGz5<}K-+TWn1MLR6GAjX|UJ#-@Y*0((bTf&}0zNt5wvAJb07DK4jf5GfBGfAgl-9jq!ed@FFR
idSL29qJ*navX2u2644Q9&t*P1YXXJ+e0HlrEac5=EGm)wClVM(su`9W~{m}k7q?MB@lAPnu7LTv7CnNqtAIDrOoKMRWG^NS~?IE0nJeTab%89aAK$Tbsz%)
7HCf{fNFCEcVmMBk7@B(gdNOHh}`e)F)|aaR~rLzB?`@DRAd{^x}hS-8QwV(T)4E{Ku}5-bu6ALLB2JFk=$h3!gn5=8)m$wtundx0js;#)1+d;S42UXsT%!a
BZG_*Dn$-9));)p#T<ln&E9A|WVT^zFqxpU$`WJ~W{^g>rG78DwnQ*OH{dm)`yZxF`GZ^cYOwPZO=RMH7-TOR83etp+Zip0U0D|oQ8X4>Pg78HJA7%hQ;#o|
Uq;TMK6=+{XugtXZbuJZ%6()qdM_3(VKV5`^o)W6Brm{I4AOk%yt&iiaGMsfRm8G_N&p96{i^{!V+qRF;$v#)o#7r;B`9bLYE{Mtul8<+q+J`rshlOZJa7*h
&|RS6wk6^%zrX($crJ{!?h;QIwkstFI{TmMW^9)+$94I6wo|vF_GP!qg+&jh3@o$s&KTD9DR|h-56eC3wyaaP4XaS17~is)@VR$X;agdMYCd&2DCf5W+_!8*
Id-Qy1$&_S7bE0rMBebALM!u$IQ7}1n{L@k+ho9UvT2Zsi5y~jdaF%txXM<rb3xtfD@q<c9Ca0YOKfe+^!&5;RJ{ExfMtorF7yNwapIT!q_X_%tl>}0Wf2zM
A1%<hVLUaWY|K}n3P2&t&{o|Woql)tQEvF6s?3iJT}&_)8-S$b%!&F+0E;@IG5XxY2j#)Ok1#(!d3Z6HHl7U2v|NqIpQ*8e+v&;J<S}ix=5=k`=VX&rCg5>n
wb>KMtPei(y{HIvHN$b*D=}oqFo1*^CsB1})X?grc^$XIllwb8bxF0~0)6nJaR%%Gc6@4TMfuT_Cnb#J7zgXQmqCWQLzvq=Vlw=vXmkBEwInYfYR6DW`)POJ
+gXsrcs$%!nRa7{*IfXOcVaq%znsY0)@9#qer2K)5I5|Pv)yv9>CU*QP+l}BTMG}QwUnTQW-F3YK=#K(y+eratxTVuoDcCyk5HEAw_3d$?BX1=>oJXG^{Fr2
bV;kpIwEYB2Y=vG2C^)t9_$eC;5?2Yu1EBYt4&P%)IH+&{)|!HpW@l}@e(+$&Yq+kS=|r-RlbtW9qR+sA&UN(7^pS@UJ@#JeCLQ*f@oIj1_L@mgH$IxDT<y_
JqeGM_?z2L+D<5qK@*SL2!P8qs#;POSJpS_`w+b275k!S*TG#HesUPF>nupD65jj=)?C_tOJD$_`V*gxJ&+v}ejzS~q2cY8{sfck*O`*m{O<+>oYNj-;2Z35
TXPxnDhQxukS7&PYL+@UjOZ~8n|YCgSo~^q&6Hfc@=erR5gO0%xRyvD^$o09u`qqhKzzp?<NWqUK*6a!Pd%S;86lA|3vGd%Qx;vPm-Zv&J+)t>S`$&vcEck{
`zPqCyWQ$_T`;I^!t4+snUn(IS@^+OV(eO-4+)ip#a6tbiqx}jc~5Cr(9>uSF{he*Z#ovGUl{&$N1d~ivu!lSoK%CV+aWnN=S%Y^Xr~L`Q!cL)A$=`NkfHd}
bhi2JZC^ZmPPX`|W6C<1pe)b&Aog>|!4kt2wB?n=C7ue%$1Vu*?c$_PAx#F9ZAm1`x}a$0EQILo3-$ApdFAQ2D;6N_7y$tL1NSWyq}_mMKv1X>@8XT7<Fp5K
jcV>!&tgI(F(`nSTe#fyRn%_HWMV82{HKjqssu?cyBFQLA<sf~=T)g(f9{se#F!__DI+v6U)oMh`Uh9QFm;5s2Wdsc>IFbJb%{xOrp<>3Vkl5}ph|SOfAi(e
#@S5()e-_3i8|4>q-E;6#c!`kR9uGJS9-d@;5>V@+?6G(<DqvFt%Z>l5A2`KG4b<QlTvy8|12^UK)GWn6CbKs&;T@WEy2={7FDP!{zly;UvrG9^;|7gjPY;W
p9sI;`0g%SQut9j(d6RRsRh*EEl#}JOr5@#H4_SD-t*U{+#>FG3;T&`Z!{4YF?-cw<K9ycTU^aSs}i^Xspzpc=Kxid{@^7&z&W1Lts($`ZN*n8hE{8)iqU%=
nl7uy-1m$vhz_4!-qQEGc{8Cd!a62A);oEOdsPBYasHboG?dNA^5P#bjx03VE?(N%bvR@&utPwv8CsI!dfz&lS+EuJUF~%<c6kC-nm*5*Bio{N@H=>MzupVO
;#f4QQVmd7UIIOp<n@%Lt{nsE!ElM^v4D+qJ-w_aPW?U3pd=BM_jj5}*HU`fbH_b>AMUu%QJ3}Svl6o5x<d09nKa-VJ=|s5*kB=XmV2Eg#XpyF@6OwMsn{vX
T*5>GpBuU*VX&Ut$s1beVLi3&^fU-Oa}`Iw6j!^PD(z&NEVze2)_Q7{07bW|Y91)Op<0aYn@gEnxetu@Y_sgMAbk`nC8Ke9me$SfpjDa;K89>zMm+~OHja_J
Et(5H*deB5RxdJpu4HJ)TKi*h^eNRP50>rpt%^0`mXNjF+h3+CbGShBm}6%{UKg#>VdtH;&`niWldICPs5rV+*<fR9U9ooD*zl_;q{Lm;ldg4lRn!(#eyg;F
C1okUAb>k`65hfbKXFdFylWM&vm;{Yw!5M0pYi?n*sNHjPIimJ<vzgKty&fe9DiVLDQVSAB;9W2c3v~Y6kTUBwYxy;*~3Kgi6)hlaivEP>-PRM%;2;0jNWgQ
{*wBdI-`i|R#Au51zG@aVkJgOsW^HXTC>p=Wssn^ei7k9v{ql2tH39*ion&TT&pvxd8Da4pv<=sO3&Co(b~_3Kz_S1oA}S@;p_ci!&XnmobDMdIMgNv+UANy
Q;hk<VC*i{u`I*t%?{CQ{K6zW3k;QWs7(l$lQ@XM7u|G`1UwFt!ApTSAgiZD+3n=H&wN~~!HPez(Mn~1k$tN*3#;a^d>-c7h6fh}(R$U0CN}6~P<Q{8I7#4c
f`w{w+7dU4LjlDUg!_?#1-l98df?-q0u`-JUl8YAvo5$J&SW(Su?x{T>&}-hl)dyU^>gBO%RqxjCUZ3pemWTCRGJina3qBQxtC+w&Tx=hJ&UTX+PR$4Az0)?
DVN;QT+ZM&7cqEz022)Rr%0+N)7K6gh=>FHG0;dEO@G~yJK`~wa>JE}%mJ0^KEX@$f4DZ+r7!|sOlzOp9P&j^bJn{R$a$*}MGTFKz`GxEdxI3D2kbjLDkLdB
ChDP~rg?8zsmUMM7~zbkWCRFKyZoH8_iWN7F%91TUL)|WWnb!}%Gi=Lz(J4iX^cbO47Cf?V*XP~&mDDh|G&s)EHlW}q|JmVx<|7B@6ajx-3p~9`obR3(8lD3
r$}*0dPEi9_SBLWk*47w+hjX4G4tHEB=1T^e)pQSuq<Sl-uYu-A5f0FSp@koil-j*o#ll4x}angi~f&eE<0TB27$?vZPGc?*$AqNE9-my3QM&fB}P*}-MhmW
S6aP60e&+?{l$-lq8`3)!y!m&`@mr9F6%V&80R2moz!xxScLjzImJ=K(z4XJ942phjJ}rBIU(+tSfps8YTY=^5{67l&#~-r-xvy|f`3)`Si#2U4a<vR@RDv9
Nh#8c@|9&;*=3yWlBL<mMOllh7U*}M0EJwS$J<^zV+`X}Yubp2J555~K=Q)b7XT``(UghpL?g_7d$BDkt?Q_E;c`wHQ<EJ(DISUIrh*fHSMG_0EN7R`+x`@f
iyil>qjII(?5@M%q&ximJr(V2Jg;(lwYEwB2i^ePSA#*gV+Y(mXIMl5*S?aP_^o0H$vFC9rXie!>QDn^8&!?LP#<FJeg-g!505$^QSs7!9WK!lC__91f|X?E
r|kF|YK@KS;j!aMLb3a_uT9z{qfs*jHGRLDMJsJ{<jO)59zUKouL%l4il90WNq^L)lNBHhIaR5)INIdn2C#-#D$4kS5F>)~sCL48-&oL25blQA7sU50=9*K+
PsI0AQ7cdi&AQMrU~E9g)=8jmK@7OneHfC0DeqcXE5(p$GjydHMDUd5YRUj#tcf74R$Gi(*#RbPpRO0v%?VO(A#g~prDe3God({{Dw3p1JXwJ6J)bc#$wT@V
3t?5;k|_sl*Glc0TJlhf2~_#xeRxXQ)lD}PU?q@zhzT14uR17yY0-ukukjhWc(w**RBw`{UCOt}TPWQa{ZVQ5)hLe{GKWZrI!1%ytSFJ%rQ1LOordy_?5AkX
-4O8!?=uS2Awe3wq3*U*HOdk~*9gr_b&FT2n>@e2dh2Nl>X90n?NY-qN#J{(@MDB5e;}4_^$yc%MZsy!3;PJ3%KnhS&RI+Pgrg*%Wzj%3ZL|!B1z+g~=l?CL
Mgi$}M&r$|iGY*Db=}zF6OWqx-OVoq7Op1&0SQof!y$S)k%mAVOKd~6ZbzlqYW?qeyxkuWaJ^-90(@=xrMRB3*YZ7AgqL_QqL2+0<FO&m-9W7=YXuLz=FQ=`
c|n~-2}=fAKMLqV0{|02fbKOuIX|b!_c~Kx{8{od>35#ur`|^&KiN4Pfy|At-ngK1B(3Klj?6|N=wx>VQD*jy!KLX;3@>li^uDyOQq41l$9OKgNtoTkFCZsf
UQ`{<r{vUJ1K{Go0CcQ^8^+<7F0<rMC%*E8=h4+|x&!UX-6#FKev<5|H!+2T8(--N>*3cP6-J1M?bTuG#|#<ip;=9XWZ9t~Tc~~SusC6?K!uN|R&Wp^(unDd
Q$!!nT2QpC6(4Iu_Nt{xrMMNhCZHBJb{-F&sXL>Guz+;JMlgoAI7+^+&6OEUy`keVFs>+@(ega!x8=7tux_$p%pnfrI7k%r45}D&g5`A#A*S7BXY)<Bh+d4w
fE*#*pih&1_!>pW4+j^%-_?VHH)8JmJ@EX7G38rIik4~0V6qUR*+dEN0WxZ(ZptYnS6!S9OBh@aTlw2zcEoDQ+K(P$o|-HBu^GPZZ<^RRLPFX*X8H%9_=V&K
KYW!JH#{OMT*?=+ujgia{MJ(2Pbo#)CuHr1GvczJy?rA%st4D4)>>WU@*|?-B1q#z@<xxhldaCWy@yQMHUr0s3SLiCxGO;&moPm(r$c#NF`}!&k5|jAiv0_g
GNfm?;a4wMGL&}_UzM-mU;)-m0?X-8kZH0w-AlZA=m`MrEi$0P6vHYTx{`Y-j(E&%SB`H*du~MbPr=-Q4;k_1r8KT&azP%J{L=>sD9(`1t~k1I&>GIXb{ClQ
KA|AZF*nXX9nzvbSO`bsbaqjd>cYllF#(pl|2!Kw86MP3X@`oGX^61#8V<$*BjNS-_p!kAR)BxS^@B0qF(!0tf3(xl4O#OV-CX-ChzQ`HZ$&4ehlP5HrMqGQ
UeUdGy&ZnjZbKF)q7TPXBI=(?Y0$B*bUfvOUatlCqgO}-{i%LsHWd;{3W@11y)X4AGPwe?q50RD8R438lBE=UwntH!Hnop>*S6Kt1AlO5?Bc`6jhnBE&!_o9
RlTEpyPEMx(T0({+)+n}aGhjTMX_B8S^^d!)62<$A(`36p)GP`(NoeuF0+r;mP$~uEI}5rst`ri725!S93-TNj4Cr(Q1+uzEP`UR6w#3=1Co`_8;vaDxdx|5
`sAv;l<)aq#%^Dl_io`b6qTbn{c-E8se#1<j@(ygPK#wv14Idb7<M;C396yJWeBPZPs+l&W}_Q-wUe=c?9$^%_Nrys#~<<K7Cgvx(?1`R!{ZpsC{VLVa3e;*
mlega$Obn2nA>Bw)<b9VIT@ALrK!zO*jW6CtCc0|TrAocvpWUXKCg?gTVbp6YXmGcu9JgF2U}hB1B$}d_)*$q@^PgZM@3xO=JQ(3PYX~TVCdEH%F4T0n(FvB
iSUbGT9)fnjv~=yBpky&DTsHgemRvx(OB>qIpY0Z#*6EY6tyvJ%@dpK#0X-dPXiju#r0zGFyjSOOJ$UL<R8gBO#6WV=fR=1qy?Vo@vr81nLA#R{0}~LCDL`*
!OJpdrsz2zu-O02Tyr|;L5gl~3ysf05DJ1~KD@B~&ba;=inV?}AHf+O>U|y-8nP?mnW%zlH=*o!O<h7yqSX_=6ys%}?;n6lFL*U^u}3Mr{pgSre04RvKNGiR
&qS9Ihz7X_v{`9Vdt5Ey5k<;(Kc*3_>Pni^4(x0wc7GMDnf04v5PaCB;KcNLk+0=2x6nh7JQ(EdQPv69_LPXB@p_<asz%=xrwZ2(VS=>hhiNhxrGT*`BK?x&
#J?8L$3NmJU;`H4<AIWh(0!18O`jF+;3U~remCBv0oFT>VmND-ipu}N84Gx%PqOLsj;ym4gls1jH63II4%h+owZGTn^*Y6STI0oZ-{}Hyv8=p^1KAe}XDAk(
-6=9N>M_&Ix8GY~iy`5^1*#{*Q@{wIZLhl94|UZVK+T4MS)=H{uwz_Jn6JuOzH|YaQ{OJkU3|+7{+#t>AL(H$B=)pPwe;W$m*u}UO+f2ln7v0A#+`I8&5gdf
D;{qe6(eB58}IkOuWeL4@4g1+v<~zRw2!}d#|!N&2dHKt1@_$EMXGXPL5aR7!F6+q{L3C!sp#8oR%*mTxhtCO%yen^ORTSz6P#6fe6jqQyog{VFosw5!k%9%
Op;#{*>qSkHE^|^^i|1o_nU8*A!2Le6YV*Qp^WQ31a(3b;pv3PY5i9TUZ!bYa13#{TD}u&QO=Z3KdS#l-9{}F34z^TK~kb!JQ%f4)grOGYp?zJjlI}cJ1znX
L4tlgA8>1VU(&fY)|de0#iIQ|1YAOg8XXRj1<bvjFT*)~@v(Bm>?2~?$u0-)UqBmMSYYUlMOpezM+CjO+ve>`HeD&YBMP@LX$NFKeZ8`G*UE`6;qN1u@9kv`
mzte`Fh|u0-8p86+}Ce>lnQ3kG^GsEzy9t+H>QAv?e*nsCd~}-hdaOyI4P71vG-F~Il)1aY}`;4!fzj2`9zJM^j!{W?xtB=2~&y}1O;6AkoE;AnE=#(oj(!O
?298hkSLY!p}Fq<`a|K%bH^#FAVnb(@dQ;lYis^x;zb0@5uD<!A<D#UZAr{WY2Z{8X&6`C+FzuijYWKZu~e~MnH5vW7~EOdp?b5UIAyNc1s4J=lz#p&y0Src
EI@9Sa?Nm0kP}l7)j~$l@ZYzmkuh}PDGd2-Dw|v;$vBm#sz<i&`5Aw}d_;T5wmC=ddX4vmYcn^yRVxh4=2g%QtL-XJ9?I7Yo+JY>`N&gSnJ2cePop4jraj0?
e#ChlYy^zzaZ`^(ujMtORTwD)hM_~aSLdz$**rK&Quk8W@8w7G9u%TFL8CVnP>4KnzD2>K_4nPfhDsZgNYJ($wbgFhq5c~(`ep`Dl<t3NzL?sLW0+{WG{_u8
wb68~KXvF)EJl2YuF5AJNF)!Yz;JtS-{53z6Mj-#d#NWaZmZ|ZV$qFI9ctHV@XX87bmX1Ad|&^0oz3Exa^Z^*myFFutgJA{qbCvk4~hvz3`}Jy!Y2du8sa?M
YFs+}|Cn{vop+lA3^|?(H5NigP(=AiFaya=muO|7BEZL=GUU!1kagQiW|YAuF{40#SIpTRWGJRj1Nvd)h(<@dq-@OY`tXmLnSf(6UcT8}tDFud7Y^q?OwR{4
j)OT5P=$O&P87g7ooLYMsu4-mSt)rG^f0|)W(PSSimT%O^H?*Y2u8UgqzNx`IpR(w>X4k}b95%Ul+Ec&N>R!e#<hXG#QR0x4L3$k1PZ4etdx+2Jq9F?JTTl+
;5eSYSU>xVf*MQy*x}q<^XAN3b4L0I#TzMUf%@AO1KY2M3*NMEAZ%?gjNX5I`t8$3newFYLs1|~S*2$R5ECfz#qabD?pJ7<6yF~Y=gIqBZ?uC9bplj$xTp@(
Id<qo`Ox49!Zpui(OUmDwJMVv?wkZ|Hj!S8T-OiTgAv7e94WWLirm|$)|LjJ_GQ%~kuW2~vX&obz%G{i3@+|(At2)Niy5vJ&c=RS)KT>oDd=H+5G}+RSrf+s
0I_K}Se1ze@&Fs-O*f>~AIkuPBHNBINx8+(Sfm!k3FS&5tj>tY{AZ>9Lj?)G03<F^weR@D<>g^1YAo<oRcfl*>nypVn>=X@=OF}7*K%pa#j)$I$C0jWOV{Xu
oEycIf8mX{d0NO^0rBTaY|wZm=JuEFxgnIBW~|z?DSCTpul<tTa?79xD$aww>!Vfc%FrFTF&syiOvh|0XmEJve>LH;)_;0R4GY0tdWnTe&9JoCO3+4xXtK2m
&B|S}$xyBdO*6V5<2fY#ZUupEOH;l<V2vE$caFt@Ya{{k8WU<K_Bxu^OLIN4vVhYGT|5{7>p*{{Jwq6e@+GS+xCE(a7*`~C@tWzrwx}YA6$5kxK3Ffwcm;cb
5M=?20-z9B9QL#1*S?xC><uHdC)s8e0m-wt{s@lp+}yn<HPAlWPB?Uiv*gtBHxlnLU^z`4VYP6EwF9Gf>2$nr0+bPXlJ_DP8fe0JhmxSN0Xyeh&DdzOg16$9
R9coMBTV8A{i^d-O|0|^gR_V0;Ur~d_+6v_7w?Sz;0&VzEi|h-8KgcmGO7bIc)4J#c*0AK-zYvq{=#!WjZc%jtcj$%J$`^w?UOI{Y+<j4sXB$(Qfa<$(`QU|
phdesj(x1fFJVxuftOpXy>&+#6%kD~s_6JvMUUF8X{BVef_PAs;(y>mu+NPz@2z#rr%0wE?vlun+C6;9`UN6D8|fk75KeN{7_fl}G%sxHG94_xfj3l%%ECi3
gCQ8M`Tm5LXy9FFeDlnS(cIeyOl>ESqL*<YTpiMXgLUk<E@CNQ5AFDXe@ku&S{!lGXH9z|_}_X*Gp}}=Im{NZZ1jm907PU90)>DWOqe9W|32n<&Z?qTO=K`U
mDc5gg(j(?lE-K%v*ca!xG<hdv2h%HP7t`A$H=d^f2XxQ6@3_@Fh<jkG~?Y(Fv3u4r00$04EHraF!6=tQg!)bJ`clN+f=Xy=s@5XU_wuFQ1rTZt>hv1sfGpU
)h4Vo-%nQ!lOOknd26t7U**XvydEn|o{p9S%F`=OSurn_uweo{KoBB%MipLNdi`^Nh2`xBgJ}`_wx#CNoeW_BG33Nfj>ql8B$&45W@`Xk!umf_1~<#=Qd=8Q
$VA5~VRJB_i@NK)qK0v*^Eo#*J7W2d0<+;+Irm$lKm#cb3c%?Xg?)1DtZ|dT0vTd&iPI?@97Ij?kP3L^=BwWIBXK!Zw)Y<m6F}~`pu}MDwiF)z91O{a)@uO~
%jhXgQ-rMSUADof{U+volJk;7TEBknF}QKxyx*<3riCn-;P7Dtj8xnvkoF_=iikdnzGhbaAtlpY`2qHqw-rlmGtGrt{>{uQ2aMXAev19T5vGLz_sxa6ldSx}
^7!@K+C67ybqzIZcR#seiQPdMt&f3kC<G?ICBDlnTThx!9m_4d(m@+sJEgj7*t%4mm3%b4c3Zzx*?~$1)MEj32r%La0|^@wOUkBK%rC^T(kem7!uAm=n!~RL
7qpSj@4Uhxg3a0z&R?aDY8$q|=*sEv5d(fsZt1<|2l8&VQT(jg@7htHyv*)@V)!*7ou3x~P{L)W?FZQJPVPD$%A{RuZVR*lbOHZtu_~1*M=@=w%hiwby@Rvi
8Z5>{*7Zg=feLrQvn#REeD43NOD#a9ZOuDc@4<uqWDrV|k}+=cYbO=QzZg=H^$C5-dfcWet}lZ_^>>xG9B&|U+oUR0+@*ugWG-AXIWI~zAp-|;_%D(RW8<w@
rTw`J@Z(db7!Q|9h5BCZa9zH&7s&`2H9*2&;?Uu1^Ce82n_Qex+_)0|IevE#c<cR?rms?yU}<<0q0*|C2c`G()7Zhk{a3qN0c%Cy^H8b)PH~Blz+!zG*Mv^n
^EIGDg-^yVfcn<@MVQ)d?>T+?(e~_XK``0zR^bw8`wk~#CbxOMdgKLL+cJNK)Q&TL$Oqe9s21o5HhI<76+BViL{$<UH#L-UGZ;mEiFZ}6PelJu>#@v&o262f
Y~^qekm&{u1$2wM-iDkQSe;%lO>S31`;GY{>l}$fDS6ob!b2((<IE|*?WwT_L9IW;VH!cypIPq3)GIG%ld?u%K({I)g@M+|=>svit@uKiy|HKuQLuu6;&?Ll
3<^6BWLV9+N3SdAT^W;VkVspQsBNCQVI13}U6p2q2)k#whMiGGoowm7AUx?TS?RI9!h`JHSuc-~$0*&W2z}f>aIDG)U(<1j@w#{hr~uMpk-}oa06TUY7@O7Z
BAJm<pI9`Miy?`(Q!C~5M>QM9$R^^*i(o8oY0OP_fx`+~w6g96HKEcd!CCxu<r!2)6TPX6FI08NBGBEZ{b?hSeUjd7g_#s%<NT9zZ^l>sBFEo})b;MkJ++hD
2In-Il|RpZJ^q8>_+Uet@ZcMQ5My}uy|aYT2;9`Ns8R!FI=F^H1H5Qp@Gq{u4WOx+GW6=mayoQ213arW33g~lM9*VrRx1c_9K{_xiYu{C#6jN}I89h3Kl{+k
BA|KFvnxa^Svz_1I@Pa&^xd~4ZOfnLQ}Rt_VrE$iC77!_-$BTtnIKOw(iXqAhQ1|LGAgC2ZzKlGB7|GG%(NXh2<Y75cA}*231TL?Mm-6;W^fmI1&Rg(K<Y*W
w}JUF*38hWks~9_lo*J(jcjB^?xKN==Tn-jWAhTP(JrJ<bUkqlu|Vr^fmnQ+kacPB4Tq|gE_L{wq0M!7#hPchgmwmSx?4m}^Coya64q%x9C1%0fPM0OI(FT2
q1);S3CA?XDDT@5oJb*^Lc6B{twyiem9abtTop7+;O+-^Tr<`O{5YR1iOu%MVb5vB(UDp!CXZFwvJdkQc=lJSaUXw+Zcs@~6Rz*(O5orDl%O71GyOj!30D_F
U>)8?^F!iny9WZ;+?t-NdEsj)M>9f);o4jnEl&JKtj@SVR`1|v!~H5#C*?3(lVpcVyPr=9>FI3oraKxZ%P^*0<l1JSc3fDJyKWR(X)p5nC>RcJt{m)>3?<`O
);njn_8AkZ#KZhJwq$7-iqh5L9!|iOb}`K%VBg)5PJZmQzz{_ArKmMmt7`aBM^iT3KEhs*-|vM*uGAL5R==ioi<7GqW7lx(QOQ&o!ZfnK+SszFuc#P2;X66T
q4-F2YKvG@{JrcR-*A{M=59YXVxfBa3^ygVB^b(ppT;iRRv{P^9Uc3NOf>=%*Vy=Wgk$C%W%+oh#~LlI&4S(71<`-4)r8}9cI`}kk2ckzurbe4`{IgWFYdC?
<$^8Y1_9okC(rSC;sO<gT-!KmFenMgB3LdU#7U3?o}%<0jUk6coqpmEh~hj3R*MoQbd#Q>SFgfp#D!~d{<yJV`g0d#>m=D9&^|oBb75wKrMG?G5HQ41M?{=Y
bG?7s0*zzsEGI9IM3$S8>Wzv2(tCLKs_jh`*-)1xZ~N~XeharqWeUrep218&Q)6CA>tH_l#hcVrzw=bUc7KvIR%mV!&+2W~dmM~$otHen=ha6NO*qb?CHk?M
;dZmFNI2p9f^Vj}I<()Aw!Gj8j~1P=3kK*iSs^H(C}8O0fV_utw7C-frX_~8fE})EL~~_?cvsv5Q7fNS1~iLj;Wuur`Ox=QSP(9-^cBvQr+mrdJL0iTEOs~p
8)<sI2%eUzAPg9*#nle99i@lc{kf3KF2s267^ke_2+uaBaA@UJy>!}1ll{7k0rxR5@b~v`%a2X29Ahg2fb+Uh({VplncR=aBu2~{=r>=%)?2EitnN31D5Q2Q
fvL@gRFC{tQTxD#@^Ys=x*H2{?&=FaE>%-^PMnB5{%Jg)vBGviNlJ7k$sy8JIz>mQ1sJieVt{mAY4~G4VT5{&-U5DMk_Ta8%0sL77TQ5_WzrrS;S43C&1FWb
e5U*UOSG6YQkmS2NuIQ(2;_xgR;v?%C*~!e5aqrfvUo@~L`|d(Q93hx9E5*(XNBtWnhOw`r$(tUqvwL4)mh*7&E`6-XldAi8k)=sJR2W=GUN#=hIGUk1B*mp
#uO=pq~3`^I|ewRbL~i?79b_MQ*$fC5>S@)f+rO|rEy%tZ_X0#ddJF~5JeKQI86tk<=eiiwlCX!V6(smk3wth?}miZjYoM412c~QCC=>6NGQT+bmx(`#;qA_
HrC&%7{IxXXGhWNa<*l3Go)Owui62kbJuedvWswAm(lGh#|)!@bRm2@b1s0z&T16QTnsMgm{7x<4qRXx8cG#m{Vo4q3mHY#YJPUi^&Lan&({0rNm6Kneyy9C
o~J#i?D@jtx$3M#75VjH5YmQwa%~+Dqm#^WAio?)9IZnCN&?Hr^VVs(1WZU^Zs=ibmST_<EauFC8``88n-Z6bmBNv|c2?CW|1|bo)-Lnm+OzI+RxF>mOwams
bm2U(291KTkY2tB@W<?h4*A;$5Ba9xPj%{)yjPVmZD2mg;*tQGvA3w?Yi^vWqJW_n@}Q=R6>7c3C0ru4PR?=f0b=YckM3#r<IH&=wXyh%Tn}n>OT-N_T=M24
DV#Ox;ouCbQToSDh>?Yx*cPV~mI_~sfBZJuCM~*P9+3||p_7Kv%v`U$BV1UP7!v2?ab=24%g#GpI&K2!ZwA86Fh92jDj*aG8htLZVP2Wn@I+8{ip2Ysz_qk&
890o4wY3Bc@qrDjsrhCE_Ei0=IY<cspe{huraVmGTDVC=b@1*FA>N}xI9|Z!OA`i73cAcrj`3@X$nwuo!r&g~UNjfLhQajy`D}hoo-WYEuv;P&+Dk#RpQF{*
K0q7*b>00s#S#v&caq^1G9O~I*4+|DREb7H#=0EkQ2P782FyO<T)#)4@wrQywo$>6$wK@7c8g=Y<ScAU)01tjJDGrp&4M>%7@t|aN}MZLpaG#=1#q6QF-vY5
-8b0Lm@}dlEy(f6{>z3{A-5rRxgxSlv<3rro0b&9D|Feou+Vtc0w0CVV+=rn({!re=Ti){EAj!)S9EC^5;#sIZvL<BtRN*}4uX^}&-snO?}4?_V@Mys^EmaU
QvyA#<t1j>!2)Run4%_=3vE8nE8v>@A_jV21r?IDyRGBb!7lXgiJ}+ne%5o&>d?x-jAca}t>GLIsW#{WkwFc|V9FW&uO$mF$RkJ<sd>Er5zpY2Xh-n_j^d15
bBz7&Yrt0G)l!(ab7NfL^3r2b-7m0}t`{b|ZxYS#7UEhWO6f@9YS?6`@|p-DeFFG+=KQikI&P!G7oT{&B~AoaR^K^h$P*ZILyj|t#aju1{SOwW#{{Hrh}Yh^
=uwf%HWoEmui!bxAC~yWT_I;<7;Q7Qw;8K&xI!su&ur7@7NzZA)tp+e_WuvUDYXpSt*pJTH2rd=M${16gB&oS8!kGx6jcoZ*+wt5HY>G#A#1&IE5(K6kBj{N
&|?N!`O?B6bL(Ea!b>~n+^ig@j>#L74ueH?wBo$YvG&R`82#rvZh-fY-Ox{5j4a6C8soIX_lWd#hj}wSE)^vat&t4}T=~`Ilah_Os+LtDsZH%*G1rF3<nh;#
#Q_30t-ln-RsC>2QSQk`lhiUR6m!2?olJDHI4ZPd8O&Zl?bxMP8fr5$H^X<B>LE|*dczu?inU<IKuI>P)f8aPDyUtDMCPn20-ztVnxP#SGO<-@iVi3j&inR$
4}SVuh&XLDJrmgkPrc)@buX%W4wq0M%x54V#6E|qtbzP^<ck4sH)xz2oRWt;o-K!RRl@}F=AN3vjlhwF@DA`AZh78JiF2XZuNf$A-Jstpw2ZYU9JR1;^A^_O
g-br_&yAPSR^`fb2>S|$cn08zvql{QD>cH(wYkMXDJ49;aiVmQeokWE#$Z$K1Or4~bmha|?4a(?8*tDeXzjMTdYdP~lyte2yS7o-=%Af%@I2nyZZ*4K5F@w{
NV5G%xO~}A+O;ZJf@581!cXA^9+DvgmVuxsyWj;{A2he<?d5crNfTzx!dBux?A+z2ER(1vdA2sGpmiVuFFsG6MbI0X_UE&k1t}m_YDSR6rWDN7`<p#Faspva
k(SlUmmO3~s=LO+kQN%U5lh<TaN7%7_}3oUHHoP{N#TKBzk_zcOp)r_9(L>5lc>BuwK~iIdA~+y-41itJk>>Tqcf;0#8Bu|{u2o&&<y#dUKI)F1t~CcFRI9U
Uq@dt#kFR?2r~hBo7y2GfI}}smqy;2ZclS=(fy4dZYF@QL{?z6DKbC4HU1G@**P5dJPn?m?1b9d&A<%X9HbWp^s>w2_VVw5Jy_5Tc13v{>&V>*ojDB4vdd(t
L>zo-EF)5_`%&!;b-gZmFOQ?vOtq4(5~cldLhp&DTx~Et1R3?Qm?D9v4I0=Ugu&NbgKzRwX>ww%x0}~$Zd+#YenRigA-VjZ0JzZAt~ZERXc~1UB8+(P0O4W?
8$HIg`gFkhjZT}@SFgoVfL^BXFA|ltpOiX{!CMc~!r0i8%(U*C>3f<>gX*nwIVto2r<1qu@hz+7S9z8y^L@0*+C)GVaKq&);2l$8xE+@`C-Mv*|6z0L_=1VE
zCm}&pi1HCbp0#5(znQWaT$gUjapgQrhf}jb*uHIQGt1sb|0DdO>2kZ0;g+k*6gOepm|Gcy;XyLl-f#a$d6)u?^Fuk0&26Y;i7|QGkuGWxwz~-;41XIpqH3s
Y|a(b(XDy?<NXG4M4+adDlli!kWO0?0p@JVITj$q<R=X`kblTtfsPY=43pv+U8XCU2_9_H?+BkRCZ!<^%%tZGx?W8uF8cQ%3<P9<3I@AGT4r!wk=sIw>h@j3
jRC|b8v_PI@HDADh%{i)o4S+uNjLPHJw=tRNNdRQ+9iRlSnE+3tGfsWpLY>#Lvlco<20FCm(k6T+0L3dyR%o|6KUb_m`?3ij}NE8-B5P1=neHYS(rY|8Sl^i
%hZ}m6ayJ&x=_aq3>D63QlTD@RcP{A{Bd!wrYdwgGX966HybF5?4OJx#8H}12{GBYnPur(e0tW_1=(qsRz?#k=+?OPmUAeUopw--<nvlmq9u~drNdlrm-d{Z
pqu~uP&G>!N3^lkOUX<>V{(+imd+b7>)FiO`UkVDfd;`+Q^5<Z^MQaLe$^pfr8*u;F|de2delL};82ZyzDGgcEC;*`_JJrFF^#GMV|ACnzjIkHLo0_)RJU5=
gGGzK;eZI_F+P!ijM)kVol>I1Fg9e9QaQC7F^dM5n&-<>uFThbE2${&?M@ddS6tncaHZlLujGX%*_qb4Dxr^+K>Z=zrZCpR@J{T~E{5R9f}0zF%NU+#bmD`;
nd(TU4DVJa5&>(Q*H>QB4Me9|;zdoIFdUz|>-cY%gcJJKm;sd`WbVQg2jO_I-T*Up)ls5Hq8#i9&-rvwnvbnriE<nNB<uAWmeZC7>$KGg<j(VF1Be?rn0}_Y
kSphsW+W<Rx|s3o^T+%98DuJVE_AA)?TEcDj?2YpqnIy8tr1qKHB%;OSsou_1XX|Kn@CJkt#fJhN)`(5Ky~4+{1;K|YXCXfxxPqzFf1-6PirpN)}EZ=TV9Pu
(1n2#J?~8a9YC?eV-W6|&WyAlZ5@i`N()D-)Ze6Owr5k39J%d&&d2-<4Z3ZxqlLGlvB2<nPAHHqjXwQk@m%IG%mRd_j<aItYJcY=zR##9(e<Faq!wr*YZ4Tw
Byj*#-tA;n!Ls62Z5^v2%Pnu6)(<kjdxY_nCyEaZ1xNZU%ap2)owr$>6I9f;aaJ8>__)#Le0SAc$iXcy=8Tz_Dxh1rBk!V-@h{{4q-hvmrwZn4(ln+Vn74K+
!L+`OgaGXF*qT<u`&0Qx|E^oSTCBn#tIZ>KB^zdHpMt(%I{d{HfjN@&ij}u9kbeB31Z*Xgp}YM?`{B(QgFs28WUAaNKN;hNsgsBA5(KUoMGq|&%-8~=04exp
;%P5&4fw>+AAXwU%e(O`Ewjj<r^hw>B-`AHZbiamUFk9H)%#v46-Q`p-@I|HSOtx7$Yh_FXM{C^n;N^U*(b_7>3x9x<lk}3+F$@<<CXI831=qP*?jq=g?rT6
?D0%(emAV={&vcjF6;axEq83P0eS8F!Dkx|#+~~dM*ulj+~5pde}<H$7>a2tYJ62k@+-deQ`gr0As^^LpcJ*lTVi9=8cBV}Un)=a7;a3X5&X9W!!QH7`UJVe
)aW?@v<nt6p5!S#=Uj^Flg9@`**cUfuKaj1NNEqi+e_&lUXOfidxmueQ^y!6+ipKs2{WwW+W^R)8=etdm-we|s)VbScJ`BX`!mK8t@(f9n;MtCy_k1e4HjkS
nRK@6EXoo@Pnz5MJjDr+HZy{k$_YrrnGUC6`LkPRQoaE<EF#=iVcG5!bxzA@bR0uQy>y}i<_Q*c171H3iX|-&T3LuDKHNyaEZH0aH&h{qdB38vSTa_UA=wY1
3vs~uR9FKX$=pR<xg>zSuR4L5Z|Yx&L8tivp%SUJA2H}^JrGsYVQ{rcf5_RQ=N9F*ZoKUlbRQN%;(FsfoOJIb1nOR`y01E?FGuQP6O^l%7X}h<UB%OrA44&l
dHN5<XTR*W=YA2$8osgJFWSS4_H7Ty@JTSK$J|VNUt7<VN%Er}vW0H+9`EmKwDu2?{OOstt04Q|;Tt4mLZ2eG{S#A3fL$tIW>y+`3*;3s+~dmqU*Cl`Ik@U}
_bHnj-xKIkrO5ev>O>4fp+Qg<;X3C%?(Z^zSlBB3ux5=E!eY!@yMU>05Gw4$aVa=?MX$-h1~H6aCtr97wm6bySk%%ML)DG&xbi2WF1#tudUT|y32Aw5e7klH
K9(C2)m}ERMjF+ZZ_N>`n1qbqF3_(w4zjY&er!wV*$yf`a5l##PnBcv&Q2&mz#WJoYjPO+al{Uxyp1E+?84|w_knqI&kzr~OtZ0APfv^;n}`+|E+v%Pu5WT8
s)RkHc==m_)ECk;Rq(MueVx+=^cmjn4GD#r*0p<Hi)^0~M3*{I1?{vr10re(J}nZT2k}wcx%{yY(`o9pG)NMOLV42~!$_M&Hf3=%<k;(Jg;V9HmBN!3K(}R=
=^?Y(Ko4$0sV&bfGn#Q9rwi|AZ@xp$Xw&Twx&Fx`_?{Cg??|)D+$i~TkWzx=*iYuJk``YQWRT1yPIx5UJIaRi%Yv>-@9KM9oxD=L9F|8s^>^#)H1cK6Hsdi3
Ct#SWZL&U$4rh%MZ}G^AmG^NdJ0*_5QD%$nks|W>Z;=zF*~cjp@ljaxg}*eN3Vny0&QMGuEXY}sTTE6wtF(LO$gQBPRPoL?jf{jFhnx=umJ^&l6~3K-KZk#b
qYPgdfTlNyBoMkF{N0uY1I*a?GI}Clj6l=JWL2)|m+aIU)r47M&ZmY*4}x?^z4xCYpmrJ|;d0UUIZ!@ZtliY+sS>A#fVJPWeI}FI5Y^~*g%?c<0-THcK)iLz
-EfZ)`oTN2jALoWMHtJF#_{ZAqd${Av);dKZW4xVmp5WpMv!8Znxe<5fNtV=!+?VAE{6s#ObE7s%$x+uOYm5g-3`FOxdUW1<|?4=2v%|AmGLNvArjmbwtxoO
e)6|Q>|b(6*GSzc%8$<u>c4F(pHkh_Lnb^32wi|+Q5rSlv?gzd9qo%s6h#2p@~6K(B=rqL(%!RwX#m|6e3w$h<|UK~pleU0jhBJ1I6-AwF4z(`@vDv6VO8`}
>?h^J_<O|(LNI(Pg}liTyFlwEYwKtle)eybaCCqCqu@y4Uz$1Ias8y`Lf0gdeH2mO%~QakDYg*l5x>|%&>L?Juavu6+_`eLWBE{x$wab1cae-8d_r-@7|&Wi
VcJ<e6)1BeXEFd`+0fJUDWZCO8m7fkLx5CCQT+iSvD_GC`RZ%${Krnrljj|8l4qFwG5wrI!EeG#!1bdA-*N+k^PJ|v+}CdAKDJgG2fwHfOT6LPM4chT-W)fx
_uh;_!K@)T)4#iZ{XkSZDua5jYX;HnD>K%2lgl7X%QDEa_7_^2dbnEEH(|rCE8Z<E0PA^;jfoHyo8Dd=nv%wO?f3Nz#q+_vw-bOTzuGtc0p_p^=}w5DqE~S$
*qS;^Pr@(!qtq)=&BtBa&ElM^>%>9^eX{fIgNymz#WNV_i3E%mHr<3g7&k5}?Sm9iGcg%-uQCO`B+p3_m+doySASbvDN4mD;r%dwBh(2d06@Q8s4I`Ar5b>o
nk-Z(g$34{Oz@cAJ(L7BgWC#(H+C6cU;=gy@H<U)$Cn+W1{FexCPBs_6Y_|LD>gP+S8JS<{YH+ToF#eA<xNC_CKn*UD$>W0kaS9+HcoISWaEW#2)T+9sQlBj
^Xh*$(N`34mgNiR(n$U^Dhjk^;qre@jcc#;o0ib1Z5P23UjG;tk4wuN%!Cn{13;;UvAH)%6%%1LM_FFOfTPPT^~&JrnKxa>9^6Q1-ZL`$4S6vQ3(}P(M5-Nh
2FS#(wyvU$HvoEh$Uj(uE?M16`j7jb&XV{f7e}>S-<uDKy9q=f<NeR5PZiOXiF-4FE&dq{{~l4|R;B$PJ{m*`;3l9K<8;9*BDe6p>N(Op@f;Zv63%DH{^KBv
-pKl~f0K%l5Y%UTcf`}9gCDcZTkSwdUoXWA5fT9rSJS7|-3fD1W=^jR$lgh@>XiY$eRD&iPSRO{_-wShhZ%#648iRq3rp@G+}B7<+5iHGWdw5C=y;2+FY1Ow
L*T`49&Z<qET{EB`G|Ub-r)q7+KsB=%T8=RXI2OrIti8>-h7!a_0_=lx@UW4tN7mk1aImuGM5b3L!huM!;^#<{2%4;5wgAXyB~!cGcYnOM}$M3yP)J8Vaw=^
Nt9d9Mk7DPFYx;-&JjYjNd&I9Kz`q$X`qI`I7}`x<MV{p7G7Uqb<aKEI5_FN`lOhRPU~rf334OB+Sy5ah~Kzx3IX?KyUO)ZvonDD6DD<3N~9#35<Gurkb}gQ
&6Ziyir5q`vlX)%WzOP^Bj66E7w;?R?_YkQ&5frgvaqF<yomLy;R2#P7Y$pdiO~CykEg4~bqtYGr4OQ{2`jymRn%bwpZzSZ@ooqV%HV_Bx@ZO5g~p{f$8uq}
Cp}?n|8e9P--P&ZbAz-!E5)RO?^2Y(&34Y}4V*2?rg=-wCI<6g%)=1v3sW8)M3FlTKT=+0#x^W3%AjO{fXAax)W3_NyrSWnaDQWEZ$8MhhT)fawq-!kF17ct
kn^ZuckI;DD#xSpWF4^_u6(Su(jcR`n^~Ff$cq+J&i)B@Tl%!(a4<x<^{R1kJde2bJYC>^o?~YeAF8CL-mH9!CEAloE@|Pz(6VM~)e}C3aK&?%7X|#9J;MWx
t~P_HhV>ExYWd&#4cfXtM9Rny&i$6)o&odLS|Q-czTd#!jM<{BUqp&1CSEVtON27*&|K4hKFCAo=GOt222Dtx^qy>U{AdOmc8KkRe2?DGWGgjlxcT!NjvH-I
p#p88_w>sc%td;K9op=C*uageVLkLd>yQ|<z#88NaGQg=rS`{vyya?Fb~knAHQ1hb<$3YzAQ9b<<L8^GW}8&GwEVNeVor(KU`$98E*wB)oak{%a_vBN$i9Ss
|0XEEMZB3ovweA_!BA?+^*rc-0e&?{bL}vtk^_X$U@Edk9CBbGuz8)K^vj^h2_pol`b=Rc>PMk`ul_G9<Ff(C2;Cyx?GU`_6RDdlhhn0MuG!k(Q~78rj(aPc
sPhV2DOKnrMDJbDwI$X6K8qbXhf7DTwlOZQ3B)o6F3vSm5hnHmR)f`t-<rfQ6r<(~rcMC{iG+jpMVIV=neyzCkZ*>&m{Q|@k0HbK@@@iW-`piuKG8W%?T=$n
@RLb!n>wD`2%6486!Z)tO}JLHCjHPr<C88{=zNBJZ4`ctuWtnR!#9u55Oku`5Pq}uVHC$JjHyF`Vq-^1Y)R6HS9xt7#p`vEGR;zQo1R1LFP`UzjOD$C0B;oo
+)9a^oxtvh8qJnh+c98IgUp;r)HqG*#AQ^&8mE!LIR^_k!E=5KajsKv2_b7vKIkNHk;o=H5Nd87=(~?UU~!4|m!0Y?onVz3Q0d<6P>mpb{cqQy-tlo$BSeS1
8WQb=e_8by?y-N{7(Rey`JVT0U^owV66x!oRXzP5sR<F#vz%3T`1L^&;_r*~pbbwyovVS=RSXL3+zwlwRG@>nU^ZHi_x0)9BUGs*xP361CaR<F;hrIIt)+F|
Fo`EIc9F{C8!1>{JmsVt4b?=%m3DD{0=rl?2Vnh4a!M5o$-t_)UE?63*8QuFS^=h7xpJfwIV3p^1%s-f31!f6HbIIr%&KAi$PpStf6fy7;A)&}A^9GN7f*lv
yE;FapOzFBE3z+WGs-GUVIi~LwN|-wS!hn2iYFtL6Tmxi?-c;0@bGQQQJVOO3{9=jlvXMT>F_iZ&S2{}dj%4>c@TSL`~bnikmh$u@7ApONLlEks!0nrvsMGw
dh{t=OKHx_kQGOsn)L_>c@cxz9Aj1F@o+o@n2DdsYA#24nL_KPWGTT$oXX<8zt3vaaBWQ%Ov{hyFGwZs;TX1f%(S4KaGJL#m%|qYbQ#8`okTfBtk_!4UopiN
>J1SUS^Boq_bfQ~aZByc^l_G5K)x&K10ya51zr@R?lIq*i)JC2C&&J<&!%cIW!_d)QKI5-zkh>DC9+U8e{l2pEc@&7<p_f}dA=1J(m67E?dRUB&7opXXmNOf
?SrL_UcN0v1=aGgPUnsl%7ct0m+TWBn%)eeVxvpI%61zu|G&5}IEZ4maRc2!^uM)Ja`2d6o$e;Zglbtjc=ns*umVC@6@m`=DbPrYta4=J`kC2*9uHk#BCvq$
X^NM^pvG>Id8nx=m;c;&1ixtfj-_uhBn`EHszJJ7ipoE(ENIG*Xqg<`8FFYa5+CS8WGwQ|+NkP!daak*QfYl+KJEGlw|ErBW6m(kQ0000pO08ZOt!Y(9_IyC
$zTX)bvU!wbvi$KRr=$FMW(hcNIrQsoXlkNxqOSncAD_=-kqGjdD9RRiSUx9oE0_PWfka2ev`l4-pDO75Ln`in^14G{=mjSMnbS$jA10rcvJDO%T_lNuJtf4
V>V$M!NCovn&ye2&cc9c6-3WF^8X;4i}{ks+ef_AF&`$Bqd_@DiZm5lDEwU+(P9zzr)5hI2fuSefhppW7H`?sGQJvuq#)#fuKN#pi5o{v{fB;z&Jy@mn|Kgj
9>z|_Xwy8#Kq93pluLRSS;68wo-%zF)p@Z)Xy#v%)yv3ya9)Y2L{~V)=R7!vvKe^&E@A`FWXtQ8K?GCAt7R%K=y#3U-yxU8a%P$yLIc3*Yl^hp<YwSpHdBnH
`oaJV0I}W?#ChzFEC?`YTv<{z92<22@i0O+^EFW_fx6xLR8J8Wk?$D@veM$wTB2j9<e{Jo@LYUPnJiBIQMr<I`pN(_0Ia)0_mA&7^-fM*c8m`ISCV(>I@Al0
7%&XXoqPHAo5U}3l8&I?a(xECF4r3a&hq{LL(ggVy$dwm)<thO9Ts)5eM4kHa78i!)i)rpD`#}@bs=+o(!9mU5wSrc-8_NBL|Oc%xtz@fsU77;A2&$*=z(at
PwpKars`XV9#J)na?`}?M6Su5?{Esijk+tlxNT9S2IjrYt(bfsL*zm9h>+QdMHE}fke;5wN&wQ3S$WrZJNsc|#{!C*y**p*5b>%k0UXSjk)`M-N=;#psU^Z{
bhN?)N-~D{b;BmHenKrGNr!b>gbj(t2)N#F%ez_sdifVDZOn?|+3F_v)sWi<?b}2F8Zle=O;C7dpIlV9bF6#?tQ~j64yn)O0;JfR6qVS+8z>7BvjRV`7~j4W
7UI>9iZ5b~55GFBBJUULBPxe!<Sbiqx+$Z7jp?u{tx2BRTcJ2P()hAx4WSYap!Qtq3o(BCuVl)r%b=o|s$us?lKWSmT|LkBXj;B+Km7BoySNP2?$?hiZfC*A
5)5u$RNq_xX#Z?An*8H+{8h+{7&x>!=Fsdp_9i2hoCF8$bu|X4ag(P<Ax$AF|7%>YB7MYI(J3}8&&e$~1#t|MRcBcdDrj?3Z9kl#IOndE@?d){&$_NByhHac
iVlxV!l_Re5E7t1TTRu~Owc2zf`$$ZC9KH7p<CdeSBU_`X`;ky!5z*FtzM5Vrn1w5xF#zfN>J&lhQ|rl^g!S%Dp4)X7&{LMnLf%If(pvgi~A4D(Gw(pKZK+H
kZjm0pd$?MIpq}_b#<|f@li0G(zOX<KpV`bGScvxmIB3UeWuV@tF<yDVR)X|ZAQAxZZqqtf&RBNPhNV}+~*GXqU359oT_)IhnOxv7aGJMUyrP=VjeZcU$$`#
Ih{2`GYZew33b|kF$ujfQT-9q${w?}O-6L$o%!mP^0%x4?X4U+z80w>b;F#|*{9whtjLX8%0GF`Z!|xV*AR-jlp-W=cr722tYkp?AiS+$kFd^UklvCYeAUzu
2)~(NXWb{`$66msM_&daHAjOaH|vUHrJq3hInVvzC7ikR!b`0KDV?ioPVfeCJm|=LEKA%`rk|wNhszunhFh4b($dD$IP4ve($+O<HumcZ?Q2spBIILYxbK-P
uJ@<Rv6Cp$K;AL;f?z1bXHtjns}p{nDTXVxzy%aWjfKj7`@la^5DI}X9)+y9TD9T*P=i)o-fT0zLeY?p3Y~Z@0QB&Nr)d({KRM|992@3z@lq7X+XgXN2nEer
Hv=fJSYSglAgFry$KfqL$P5)PyCTER&{Cr&fYKfYxBc|{qz&d*S0PE#K-O#`2A$3~#-~vF9@YoUlycIRLJJjsLlKk|O>9Uu)Hcz#670=0albwrO&<~RUGAlm
x>^r%GlSsE&mozhJ8-P&9I%P5gt(MLYx42yaDOVPl2k5+)A0pFkWM|&#<`L=O)fVTkw>irb5Zl2i}Nl}C{#AcE?O2keiJf-2f}ovkl5wdJL)vhzcNLvT#otD
EP?FM5!;i&SU;U##5>OvndmzJccRc-#lw$TU&g@nUOrPk?;aYB%9Y@4646KanC($cw&&%sd>@%YWhhOh!vOUgG6V->hP?YZuCSNLpVPyTFU_Y<@7)*c|2rUd
bseRV-glfZ;h2o%kW`*h=vO<`K?cTnqXNYc(M4xyBtC>RmwR93NQ3)37zJ}rH*6EL3-O;Oi3kqg&?jYV3{6k$k1kV^pp8=H>_9U|4sd1vDN&SjGVjIAmro#Y
s>sYV-+oL(X&X8gKetp%whQ!9{+!LH%MG&b9|tdyX$DdSBpwl6D1oR5=5MDnBclu_lJM6e#vK?7#b!o;4T<<SHOcZ34=!2IrPfWW%LhkB{rhuIi%;^(rXJZ?
y_!b-H%7|Rf1*}KbS;w;y$(L22}9mR9ffPW|8kKz$H(PM%(L3;y_0)MXVK6A4l<4np7U)!JqSmC+w0H*yPW0V%184zUtG$qFG&#e*6Y#AgzBf}p~2bz4t(Yl
fcSm$sxpiy6w!xBjl?&(IH}W_w^T&m8ls@Vsih$30tw4e>=_t9_?d3(U`IHkM%cTm0oB?PV!8#Q^2FmB@Owz53P%-ygtk<FE{7j<3wE%^CC4`BWQE5YY)j=1
*>}~~t`4BZV`0~3X+DU)kg9~4B&_jf0#f4GiRv~yuU7Ep;?Y2QZmnk5QSW)Tg?0^AOSn&#5E=9CHV(RxetVf=;)3bYq<zkT)BY;sJGnPb+SCWr&g6ZK8*S*V
5a`-y@|@_Z1fNXQ(qZy8n;me3Obe75)IrJx$?QPM0=!|j7d@v3xemTfq3VakAjMY#2Q#`?GHZ_Ae|&CMmf%M<<b@rAv2PWqpR-4rsK*<%JvG8R7bTxuH9$Jz
iOMPv#L@xI`1SvqEoSQeE<Hs{4(Y7;Lt~Ip<wtOq-6BmOJ7I`ZUD+3aeAO=~A`J>G0oy{C3f)ciFKvS!TdR_*O7H}3AIQOq-NctTM!8U$+8O9K>ijtccgev#
W``fQySc>^HEr;}%8xdH)LHJhF;c0}GVX!PJ_x>hTZ)thmwAJ$HoH=AS(<9)IsI3vQ{SRLVbSBTRI=Qm8=fx)@>F8#FNswo+CKh+l7l(+^G7Fy%`O9K|3WCF
(1*YwA4!B(suoiNXe+tg-^Tc6PBZ0QFOLH;y#JKt#7U1BjLZvWg|24&v7W*AM^6thh1I?<_tz1O3;#Wmgho$ZG6SF@Esgo>->tf@YFBUQl%3IXHt2%fTiaky
67mJu6_oX<o>u~}W!&`TdSqme^=*Y+RKyF{D~rLQ(FeU1bb@1i#+9Go-PU??{KH}VHooFxt}=Ipi&H?>X&BO^9T$7?LQlEUuAs}q9mPZ@1fmh+H?#W!TBc|t
W^n!(3NKIG(9<}+mHn~CPZ|85qSm3$XCHrV$uV+8$W5V(M%#JV7?^U$sFq|3?4G8qRD_}20K^TPyl8m^foohXqItGKwk7<Bhw}Z!0z4|^{3#c4?_tu}<6|=o
6!~tzeXt)Lp{2cB3X`VyM#M8tVS9WssIaRt${np&)y0^Ew@vs1=9TG(A8Akixwux5`aG^~Uw*>0ir!h$8pn&-t-d^dpipTJAg}z|*o?~C^*B<rMScsLH5FT5
)nPhHjv1mbZ25Q7c^|0}e5&0GZ&my$zm&_;L<zub^Xqfx$;<iIvlQ#_%-_@=;W)@Dw*tj{d+W|S%_uMXDHaB(!3BMjf(vXU=V*vOS^VS=X@#;YKQUU`#=B$P
aAG)<3Sn#JGLJmorHNxP@9QKmg@L@&7giO#RGE`j8Yu#Y@pnLixh@7zxLqOIv<FyShLcKKJ7UfuvJA%Xuer^$V&JrR$(Ct+Ck~qz3GL9f!9=`LET!%BD$3mi
R>n9!KS8ivbEvY01)+6GFouwIBXd!G=H#<*yw*`FT?Z-nA(9iqCxc+Sw@SKEcp0U=fSoQJFdC=jjI<=t8yx1-@1WKwM>}3?FB@Xh5k_vc(`c`!FtwDjLr)j9
q}PN9i13^(*&4IjIz}&14uP=<two|G{Wx%(A^z&j6j*&t_v}&c%;xGBJ=aL|aRQ^0rLpt1o*-U*3Af%|COq0URp7}+c8Hs7tO2ld$!(&;jQHIH+guaa^|jGz
9Jhle2$%g!+hhJeWz{M&Q7QKJWD_U(%Lc5v^;%cS1J&PcrZu2%dT;{`p#XE}^WmVeKQE$Ll;ZU6zn>xuCf!;!6ehkYqj2O^g)^~Rn`j-ofBV^#&P7{iM+P>4
=ITU5OsQ|&wPk`P-}|b-nTq6+9~_;jF>;UJU*EzGZJw<|QcrP@try^mwd4Jw$!aGiXcpzxA@W0h$(^o}(&S9~J-DPA^g_$(j6OL#4~dYAX5ulI-Wrm|lk8Gn
I5OoOXrSfI(QC6D#EAhRSc^cw)U2D<^^pdV12B2#;o#bmE=@blD-Lk8-+l(rZ6GvaFxFn^YULUMp>8j<DuOepo%FJ_`Gt!M`N<s}#rN=xG+3&4*OGjkqp}f=
MucqtPoxfr&y-}IkK{EH^Xhu!O-Kx?rZ)WAxzK{9>7r&<IZ_gJe*EKUYNnG3E@!g^Cl)MI<YB^Vzj;UM8d;bC>XC4`?uWq9x1Sca+Q(OU4e?p6<DV4TBkgs9
Q|#3)D|uicOgrN1awt{-P5<()TI#j>wvop3m|imZkK{*!Jiz|!>BTLhBhv^SYFc%rz6^zKxtze~RrVnAV*UQ37tm=t?WPDGpRs4qAkxy%dG~Le{uQ}RI=EcT
b0+|uyw7UeL((llC5{)J%Ow}nVA43Ovqg&asO*BB*dhC6RwO=Ad2?&aUe$Xyy~5-V-*NXzuKU$*PL+mHuGuWWhD7{X{Ed=X#3#Z?Q=gZ{1Dmsf&2*uLAn_4Q
cVfJ|CG%w$nY+<HHk%qYqrtVs0qxar>O;aR#QHv#Up;go)`4RJZ7Cc}bX!z`lA4`tWiR9?OZksf)1T^TbXl({Eiv$A3*)`$X*S9IzYzPh%0f-pc3D-HwhQ8T
5kX(Lh{_+v+u8n*Hxtk<D9OR{$a3V33b`aS94}|07(>Lw!}sE;Wt-QTtOd;9WjL^p2a+sk=2P26&*Et?ynGa))WX0QMswxazsD{9QSbI4bmQaa0K11eeb0cN
D78d`niU$@jM6b@l$(=Wx#e6gQeB|nKsC)a1*%6D9TX<v6u1DK$uvPe-@2`4$83lnpR4FOU}>vOw4fAOtPC?@y+<}a(dz^(w-Ai~!$Vp(^}7>!Z>SiyS59mO
G_N^gJLOB4I=n`=2X97-a^KC&5&T&sl4l^ITp{B$aUs`MOV6BTkqCE;`mwkwy&Q;O)5%jl*=wsox1&1N^)=bOwqXkmW-fKHUzq!nu{cx7#2u#T$XZAesqgFj
0t<D;e!0O5LZz^YLN?j0R3CZSFhRY^cA4!sS=<dMvWjD!^|PY;&L%rcI;%ud^WD~UHzxB^3zZupGt79Ju$S%p7kESE#d(TBv7udVo?TD-OG-th7|)|WAH{tm
N~lY9oYW&7xG)BJAbdbkjU7}1tCy~~%YuXC$*HXwg_Gcuknkmutc-F@j=4GTBPMk(%1S-!It*Mpb>=)p;LiL0Wd)5M=Ym~>Y_Auz&%fKV-8Bsd=|>Tp4L`v~
<#vkX0Cvxg#y$t68q;q)g<l=rw14Hm!m;Bsq%wqT7l0U;1pIf?fbn1v|GPaP_ia;j*kI)%o!!vhZkLKoK&G2_$gv4^!OS5NsTsCAG+^!4Etv>|ev!yEhBxs)
9Nfmq!<yK**G9iTFZbnfy(-hv`!94aY3;%f!juup#uqXvYj;WkL(cB>lNEoO&maZwJDx3fIzUmuy=w?$?Amau*hDDQF=+al7;!DLtB1&G{{=Vl=2CkolZb#@
vVR&I&104V9=pNB>V;cYtfA#0;nN*5k_#*UEFC{+m=GnyPe@v+&Iy;BAGzwUlxGyC3%XpXO3Q@%ds^<;_RuXE&7d`XFb$UUc=twNqEA;@HpNB%en4Sep)bG|
%NaYCEiUcls(3@TG+*Z)e2E6k#=g=W<zP@M>(uR}IBX;gH@yAk=mpLOz;kEf1jCT6*KWcUOB5OdP5|;wo!%aBrsPO?X&=2K|2D=B|G&2P1BQ1kZn@T5lweub
BIKj0wUr>GoFP?u0OwxnbUOCX8AT;?!UE0WuYfvH7fay_8eKJDh>4o*eg}OCSzI#dCEPCp#HBU?>oeJjnoW&`5HiaW>DYU!HIjr_=Oxvy=YZCd?jqyx-Lebc
Wq(<GxtY#ti*Y|$gftOD_&L2llFW|_o5v4FvpB?ZS#L-ZF|Rx>PQ!))cbCcXKM|7pr_v)eXH(QB4g0`)uc*KD5N?qsIT2PkEAHBv9Kj~O2{P^{Ho(>NsAd+^
?aF{qgkh1fY0O!q%=#?>&IP=Rj$*+{;Eq<AxfplQ{D-Z0AyzKK;rG+eX+wux{O0ui#rW^s=+X(${hxg2YO6W*8vo<sG1RQUaqVPpHX!HV(R$!w`1sSkYLp1~
il?+CP2yA?g>Cl<N?XS8P$@^;mZkszw>nYOCXwat00EQy0nYmd08GW`vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/ref_r5_prato.csv')

In [ ]:
# listas/treino_v2.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/treino_v2.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<twuA0bKwx8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTl9(JVVGirv%q4sY9B44hig?`BE%r#o^Zq29?%ejt>DGIwE3W{!9Ii
^T8pa9s<-_nLk_t<B_+dejJAjh(+J+N8U?`f2xn<Go0Y8oPTpiTaK?%UG$Sjy(y^*#+tYyjUVk;J>~er0Lq#aW|rHOsotdtZQnaJZQK;e*L~nvL)e~YHZgq%
y|t{-ugR7Dcw@u4+RaA>@wR_)+(1(IvWUB%z5xJsyi@GBgq(^zep`iXdO5Te_WgU(g`i{+aPb>`3Kw-Pg`9|tJMZRWSk!W-!3HvP6$Ci@hZaj>UmZ{C1<}s<
)Ly&9Z13IhYP8cLI@P#*j_IgP${=Sh;pikjg<k4cBq6&X$GVF_^D|*MX<BDM)h=C=H9d!6;tm)sCXTboA$7A?zjnLNiJIEA3P7TGg|;U2bs0wGE+Uz;X9IH!
z#=ArZLUb*-j6G3B)i01JvZ4BkS;O|$sEE2SZah?`|w~RW$Z_}XumaOhKS~2h+4%)6CS1S?>%30l%EgXd)kR#fk@aDKlYnECfq>5pcqHj1xUcnweASV3pvbs
V_4;m!Z{ku(YLn3?*c!?MzW4^ms)>!>1Z)*1|a!?>ZYyi7|-e<5nj}PdxEP#Zk}QYrppJ=L>nijHKG{J0XMrqwF|kEAL-CbpFvkBmv!um%ihllIi6d>y~V<V
lcBE>!N=VU^b2cbk%pDGw1lBU?^(CZ(2o7QaD_VbF~`yoe8em}0kc8)@!GCcWxT+^x(TzSyg2W%ytCvV#}`Irb-G=hDQ<dGzf?LGr;!;iCQEMAX&bXvTh61$
X6nj?R&JLrzm<+e@;)a~-39}3-$K~i4<MhzJ>k@@JMg+GxHM;c9iN30DZ%XBFl({LhYno_)4I5=jz5Rz5X3hRwJ}azCik9{2hUNZkNws?t&}YscR*;$iwL~-
a>(d3AuR~T*Paq~M>fv~b^q8TuZ?M$?ncqB#{s{805GY5RG+G=tD|do66(z<e^F6s$yJEx!QaRzRny<;q_6af)m@!me_x=cK_75!%%l=3i;0$m_g^XX$XA2+
K!=A#1gL>$PKtCQ^>w~TR~o#M$szRRs|fk+$hDY`AswT4vYH;cKej<dW#=_Y<5#p0cxgN>tRC5}Qm*iHOcJv!E9_=|lfV6R!mk1+J{DLKc2uD9uE8GphIqf2
-~H2k^;}CaHT6%3#P3{5@|E}FK(M7ML+0#I&>OF3fzyWT)(-n7=rB>@Y<G54e}WIpV(Bu37Pw2#D&}Zt{vhS20IICED}Ljh083RWl&I$(^}Vh9_w4uV>yr6f
Dvr&*UT&(#;A~@Q?F5I`p!01E4~+%cLs0O{*cMDIy^_q-Ka3=3=&U}NuM6tO3}}+kaXy;Oio0M6=~4nJQi4%{Yo)DJ4xLjN%>~|thmqNY)8*@R_&UGhsI2aC
>g2)d(KxuFwI};W=n9ZO$`Y(+bCBCdu4!JYP`m}P_5~<QP)hA5govSa;^T(nQ@lK#yhB>CwiTiB^Gg=(YBQomCkmTFe|XkEqLc#y53K#u=LF~#7jhBx!@^^C
(1bWyi@gyVo+rXOpQX_-57*~(*q%7f91M~{-*g3_fq*I_njldYTe!t-kwrp<GDTVqjCw)yEtHhOD-0-$(Z^nwL#71~5%i<qzLc^I?KE$<y+3Jqe$iXE)qT?`
o#hLy<#!%$rv&i1E*Z^(`vdP_tWPO>sGp>lqQwreqOPdh_8`vz!R5L1FeqbRj|6uoR+4f#D&!*i!ZW`?fW`XyM=uSfTBdvBPL=7clnx)L))ekjY<$lC1=>g|
Ed`_*1~<W2u*+kh0c2>d4AjrKEb!V?>Xr5XQf(C{RJsyM<pL`5pzlhHi{~8WjFs2O)7b3stNagtbxvW38}XS+^q2U<SS6MTHWF3nGlW%>aW_Sde&q=C+z$;j
z3p+udJkf)`8?0;#k5UYn(z)4yU+LrRL3M^6CgFWizp}XabLn8Z@?Z0qBzaF`b#)fYt;g$qf-II<twwO^?r<{MbM66MKFdyOUmsXD-()WS8C=!)vr0zTilyN
m#ofrqoSo8G01Q`z}Q?{Y?pykTl$z$D3AvHVE2*7%DcwVgVrN+f6g<T4f~lW`l<H8Ema>$ezPa?{-)wJ>F13W@iU>ous&=lQc7U8djcC;r|V35u7$NB9+-@U
`eOllO(`8rs1Nv5d^vP4+!w`AF6S~_oTWlkW)_6*09a$0DG>YHPbL{W6m%s6!y`q6_(QfZq*V%R?N|t2{SrsV0KcVupym5?iIg9G-{lK)tR_lBg2|992cam>
1k*@+S|2iA<Vg!zIP5|&yG?7KVcT5NQE(&08)CY5<TT-tVbTyi5qXKQ&xx7DJ#3*N*z}&7OIs7so7gYY-8T9N)wB4QD<mQH#05=%=PI~Laz<jefi`nD3Vk|%
#M@9-ZY9SP#An4XqQVUo=!Ua@Ul56(%}AfISfQ7yEML}qEe4(Pfoj#|Nq(Hq$zliN%~Z}oFW9@?Q`rvvTFsUkTEj?R(kVFg2&&b5UX?dhzhrq~4q}KrNn}6m
Y6(f$Wa~rCmGX>5ys3}_{hm^kbj!$eh8g%(r3<easce7Io4AL7pX!_^uZ1%)>qSo4d7l^y*QfJ<e&H5P2<mpPm<3dPh?`+XuOHx)zBRP!em=g@)uqX_4Y%iV
s1yz+h-Y+&SsNF4#>?c9BcV(v>lLRdGv1_&7kc)^AW*%vP?ZBfb5sEp8cvmM4^)D>7;7{9Fgj~7fCu1Ve=1^2itK*(k<sj(!m!jY$m9W}c@OxV6`GVB&R0^0
gYuGDu@q*_l(nV#VIORsUzO|V{;5&1&y3I+g_H<H>Kb=<{g|*9g#K2%_S{3}|4VF6Ead^k0GkjPM&xADaw~x2LyHgE&pLzzOX=-jU!h2~o$08Co;cWQHS<k4
m?$&sysl%q`M`E%F8+-V!UHec|B5Czuj*C50MnX7gxs2;3-i`8+|Xi0{J7A3y<C|NPM!8b;-~fMOr#y(4}tEVfQbCKaiM(5G;}?NsNb`vRCz3Owsqk>#@0O6
#LKCgI5(JyODpXZDOj%k-x4hnM7ijF8VZkRX)FW~#SV(&*FZ|{Xyvzuly+mN>hBo|PM&)n?eiU&DHC*dPXT#<OI)(qR_rn=;@@z6EdAvzbce3!y6ba!LUqn{
qN?SOvUEB_P#IynlvZmVmJdG}XXXrR`V_L|H;qKT<d*%%&`vvnGc;GJ&#l<XC~D{Llj{>6Ya&<`oU!OQ1}s6vKVy7TF%C^P#*AAM8qg>_m&p71*f_OSgV~J%
jI|LL6tQX@jQ2b3(C;qwwb(!^13LLg=QS3*+34$+8Ke;t<(zt`)6ud29Gz%yOi#1f<$g9GBgWmg_ul8pnq{LXkM>;B_EMa!oG?5O6q86!)Q-wP@#CsE+o-II
=k4>((<}FO4tFY?z%|Be{yIpB%>!qM$YFei&$9-$DJ~|~WbbZiu(cJw4iZqxW0<1ksf`R&zgrdJ(okFlsOY+gAhRhk9k+~(Sb~~NyPBT$-MG$(fwz$8K5Vs_
*tVZSMqa+dcy>08XQ;VInZ!AbXlI#w<04e=Q$h|jc-FYFG#GOKt*%gLIziC@mXSprW1TbOM}!Fy#_)ziveKn3$xI~;yfp33d#E<VAO&i|kdh7W7(+!Axb_o`
-s&Sh8f4|w<*M6_&T#QhkJ*o&i@EIG7JZ&#EItN4)7X@x4B}N8c%OL49~XClys9ljPe!S3?xtuqU@<}2Dv@cPTDl1ivS2qbaZp7ZFT;u#78x`rIkCI!Hk7wO
;M3I{w}E%}knRHwEgAKQ1D<zid2NMZkgnVuFt+Sx?ZJry6L&vZ#W2yb=O;9LXp&G2W5WoKjGEJ!KF@gmp!>pMQ-o?!HF%Mu&&uPr^5vm?G;+QMh!rZZr(cP-
keG31)=*XX+oIKV-xa5(dxleKD$$o#ST}%^8cxAsPkAvFA_&jc(*gPB2!QTCzfWRdV7%<&ga*lgB}U$xqI=%QsS4T9!5&|lH}ni6E$*Q73@%v*zfdMtWg2jt
f%F27UdS|XBJDmcBXPrEd=?4km%B6^vC|sU%W<xAz1kdFT4Fb;B*!XDsrH(O4?lKR7ke9JOSZ%)V*;*g+bsbW;k`;?m+4Ht>>iuw`jtu`c(pUSLf>QNi6OJl
hOch_vfp`7^7MD?qvh$ED@`;}6zybmCoTt8*24$N*UYaVe1kR&==IPTYYsDdLsAl=_Il2qU9xsX#S(tHh~f(%Xy1153y~+o^O2qB_+n%8K_`?|R`YlMBwoPM
TGoI{)w9RHap?PQus9;Qp2}5$!hcWp&mPY&tR)?1zLN7K<!xSXEYa>1U8nBX#PrWmn(@vr{$1v~UhI!xyvK8rDpk7^VT<Fzwtg_HE`x-|hch;sRVnxAcrR5_
vJUdZi}a*#LDFd>-<us<k97HQtju2FU!Tm@!VlwkyJNgdt~p#KBgES$*(P>ok)R^*jqBDjg@LJUAK?#F&gOy!9Zik)8t$EwXI|Nxv4_81HzqBC;(+{L+|1cV
fn^CfWd>gB4bm`MY84Mh{NsFuyd+gyMg~IajJ$$HR?3ye5qs=I^x{4goPt@12zy;ydYs=cVq*Q<oY>@U)Zz`u`DrgHGo2}QN8F!_cq^|cGZA9JS`{jkcSd*9
V%Ta@UlvXv)U%f!yYo~<*1VH?=cj=^EzonbXo*y*C`N_7Uh_?)R|Jh(tN~1Y;)POWt%Y8G>YAL%@FmhHo8XIa-<P*8;6u~rLTVb0F`(zqn&AC1rv2j7;bxCJ
6wXd;*;48|de8PJbbah3uPd+r(pfJsG6?7d9q9Sq)|8w}@<i0ewkFB3lfdn@zg6dPvyKG2V`)0K42DlPHz`n_vozw&9=BFf^o~tP5%5&^ZWfxQlEX;SJ_-+o
(3KAC%pRz)CZHvu@)TQNm>M2C8|LM~P*%}#RO)OA4Q#LMLm+>JCNK>C#1t!IQ=1%X@Nw@CQpiX7@XHhke#=S+w&D|`7hJ%n(s`&$hj+u7nC5XV(Hq^Eowhrm
V1<sMz>)s6#QDXS7RAVm{5FlFoyx#E5OS#E7nzI=u=-`)a4$`MyiEC6d|$Q(NRn~Zn6T(GA<Dn-nxn^pHcc|=^?FSX!qJ)_8xO$KhJZt#BX_y&qE|IV+l55I
q4rEU&m&nWS(Mm~5P!nQL{4|pq!I|^^Z!X`Pt(<?gB<$@@`*1rg3va1Fa8xA@;@-Z+^WLqfpRc193N?K_py8$PI=A~FIpk6KYm>+(4gRF!r6B~uXvv0bdV)(
s_@thH()jZf#VuQ%SQ78D^NkARjT%79Z^h(7J?St(VG>Bv;sZYg^9-bRC@(+1>E8+#Pn7$W$q~0`t9qy)Ei^0tW+4<9QwEDamyrKQsiUO1YB#_S$cb0n{><Y
$)Z3rEvHS#eX9AJBv=C{HPcObBCL7$WP8SbkWG=v3h_Gl9sV45_Pxo>5OJ^jwxmRm&7i(E=RxcNr*=<4UP<K?1KrIB@7WR@i<~ofS~qp|PgBDzf;!&A5O(-F
`O2b|{2Ymy&YgAfy8ix<J8yTigg0mBAaAxu$)!6C>-)EZiMBGHjVT?@<svl}elgg3$%YV$&U?mI$t<!E(C#fE$qM@(3yv<^mE&`lrK@lS)&9jhjDIln@qi+e
U#w}?*BQNVo?VXkrRW|I4N2K#PR~85E)mXhqJ?a)(8)~7gGMy1imTT;8yn?VdeBvsc+xps1XT9_Dkc{|;^8yzQz)iz6yD_G?B<afX|{rZbB1&BvuQA??>8H-
4n<~Y<v!4xSU6a2QX}N*lC-@}m;dyCU1xh-VA12ZI$JLwQYUohA}@QkBP(?T<zT(}1<sFB0;eUvU3-v_#-e;%r;G3Xr6cBwy7EV=9sANSCgAUJzrCsT&w=Bj
(-kiSb%O1Z2^utzIhL6lS5O>!<NFIbNE$PpiTYllj=nT_2Run7b&gA7{_2MhP^SXhHx)G3>W=EJY*iAkYa=b_&q39o0;Qz!ht2-7-kRY%S;wZP$F1(i3#}+V
m8Ir0Ysy$p4W_P6DOETyxnl;to3>QkyIH!!F_Ge0h!oSs|EUBMOi1SS$TOnqK6}YD|0Ko7j)B=4O6|~rX}bxsn44go)M5`eX~^lL_|NP_0V7p)aPV`Ai8s;-
EnA~-1(!sfMf0K=8q<b+Z*kBX@V2e<ea7EgtS&bT1{z0{++G2Z!LAHB-QSAtc{#)iWEEqF^zw0#G0;5vmL#}^&$9|}<+I+13Wv5{YBEbmE7A#BS@<|@K?!sM
%Q!@y?TXXe)8q|sdz{V)dn|dX$Z;b;AHjE4N<CJ%GBMa7C!!LuHLgp>hVSt(hHLIR$#^+NB#W8`L#Xd5MzZjbie1@{JXb!NqdTE4dtdpFvyz>EOg`F$FHL*^
hMH%KAnx=WLol2ReSGLargPn$oD*};LtVD+d>~?v41B1T{_N4q+AAt>e*8hKeCb@hxXUygRA;?cHMr&GEa9fN&*`4idrs{IRkXx;9ELyBH&;BAj1j1|P5Y^<
co=KbBp|JmA97g!8uF;v59!DMSMPx>(DzPSo(_e+$L12#Yj!ct7gup2mcd5>*M-SvSBx!R40g%fC|~_3QiW0AvYX+o&->AJozbN7w8X<&l?CoM!IqK&rJ^-C
#63P>&9|D^!u$wyNoP2@Wb!W*9TVC>|HA#32TKE2wAZ^w=WE!JqZVoB-VxmEP_eqaI=^_wXxWyzrs?cI_PJ*62nAoRjS5o-mS4(e3X|tdyV*G}7?;2H{AWX)
tzk@cP0qCk4NgqSY0!MKcfXkvsn1xF)`EW8TeM9;mn=DY&!S>GUcYuXCXgz|+K^1#%xRFcta%EzE>EI@JyB(%D)OCvP8(zSB>GhPz?SR*Bl!TAlDUJuY=PYi
0K?Pirk5A}FVYFpp=z9|*c5Yo#V2a|nkmp>Z4DJ_U&i#jG=5VOJY%_0^SxJ&YR>LubksJ6s&2TPleyQ3(-hwp53!Ai`wNgAjVL#0LwwkNryiK<6c?jmG;T8G
5jJ8eraXwD^!hyiGY=<Z(88&{`@WXoSZMV>m9%@-mr7%t<H}Lk(;gC)<!oCm7P{WM)oQ%x13F^I1*klXB8n(CzVwLMKawihkg+FnCUN4VOBxI1@KIM^@$N`R
cEw@={fulQ7?-Q}zcwDI%3iYc#}>_yg3ix|TrP8oD1@d!Ow-iEdGaPX;VjQHXuh{79Ksb-mb)pyb&&k82I7B95*rxj3NGftdB0o2Df~DK!>D-}_8)#RLjp#C
WVTg@ug&3qOJQt?@YhtoV7&p@F`APZ6RCwwFxL_PGLaFxS@rkvG*nhwC~3k!AXYcjneb2)R<`O-t?NR&t-~&lck@czz&zEei$hn~>ZfXphm|&4L+!p+*6fBb
syK00T7Gx8<j&aOd%z8L0E_t}6)Nb=r;f2kd(P#qBRQru8zdTR@w#M7M>5B}HG-(rm&0sT%imrdZkGe%7u&c`9#8ir_hZl&^H7V<G^6qwM92;yZ(!8oZ6RPb
2edtA4?DH9T5(5dzCBpYG?9H%!!*sFrgR(I=&kZ*4QJWY^#lc<kthw7=im65eMH4Q!r}*A2U)3uIrBk*%P^FY|6w!17a%$TB*Pwny#hUISKHauWwp=icEl!f
xsVJFgS=j*g)!#!+>o7XJgBdozAjzEv}Oy-9rZh9_Qx+bw&eMa*S?MxMSl4FZ)UVp(Y%sOJOck+ErgHgMJ9#umQB@nkwsdhP9cDR0gflTHou2QL`9JiX#1+k
bqE5v2Z0i#Rq=!>*}MyI_PBEtb&HE1bJiQp&1_EC{{Zjad0ZTZRQN~b-cVm$l;B{J$TY<`v-095NE{KhGHsldr=#xd4A5)I0+0ClHhs2Ngax%|A=wo$_M|j|
Zo+0psP&mjXp$}b;Af&QXNMG1jr%<G5Q<qYfM^RNJQUr#X>YT4`_OHPh56RR7Ad4(G{TKJEH~1kV(9Ku2T0|=y-#+VR~lBEO@_(yPt$VkbSq($eyPP^^0wvG
Q#$!J1KKKYzm4ZcliV{+T9kLFDLk{A#|>Fk07k$>Lmd?Kj9R9d$m2_j_cJSd95Ss;#RZH~r8a$8YU<Zhq4sAZqu?;IE&?l-3$jPV@rTn;L~eshKWH-z-d7tg
ULVxYBg#6wjM_-D{9gAb$c*Y(LrJ@rTV5eiRG#44G0>oaejM6lKQOh4g?jMwSeiU__}GJKWwc!qLLJ{RU0-<R>-N2B6>4}PIoxUJRnpA~tBM<0!||lLaXvTv
oeBQolKfmEb3J5A_#&PLQ4hM?C(?aQ8mBmLs|rqflcGKcKDGH%L{~4W$9v6R3#K7mXAf)%>ipN6$MJV)FBE`9c10hoCg*G~zdjeud5Sjv&0OQguUim|T@R(~
<M~s)ZV|n`VYm`Ia9&gk2XPOMY(zQzE3ho3(P0SFG}$weCl7UrD%0C}@u8<;(CHpzXPYV}*UH(5KbvntR}!hpr;_l8QKW(u7T06}l%9S!OlRNb2#7s6h?I;p
KZ+Rxj#bP9mF$wI4z`U{8xDdhwtM&NMTX&j|CcSF;}H<K2%zv?B0(`&cHmN*vH3PvFNDbhXF!ph;S-=EfTFNDqPA0BQT(ly3mEklTs-q#xd(F~;pBMv1&s9J
31S9@x9474?^)kBNjmGD*1R=B3*#!!a%<`PdIHMbLg#hV21fuCd-F=)SB0(uSmS}C=Km2QVPDhJu+K>#sWH3tJ2wcHJ<QFI)A?r+s{=8(_GdT)so*NgGh>Vl
d?(ri8^)VAkN7@ope?X%!{0nU8&E=sB>YU(D;nC;N2pPs?kl(m(!d2(c_+T!mM<oL<_QQq7cQL&8Xp(G^~I!?P7-d)!Y2t|Cy{Ecj8Qj{YQHlt9i_nH3YqT$
@^$MWqOm_on~FLj6*mEs3-jw}v-H;X-HPTQE!FC;7o=i+p8Z_|)BUAk8nctt*hcTUQhXYECRC3rj5v4;D}nF-khCannOky@onT2A%pHKGT4%bC<Dfnrc|GcI
7J5i2<F!U$sFt_X?15vj<+}7h{M8;HndlIl7u(^xi;|2E8peF-E!^!2Pm!Cr9NZswztlfEe%*{B%Wl-{{EXfgZ)Y;XRt%x@;9?^FOvv(nY^0$il+&vPSOd8C
Zzir$hR!y*CM0Y<3K-Lx*jS*@J9XHtUeM-T+-_;f%WCB%Mi}YwHqBqeISgRKyJ5_z{@(XN5z`esCTP^#48op(X>eOn#|1FB?|hlU4sW1^>wHHRTq@O8vem<5
QGBcwTIi3>D%8hwt}a!z<JxYpdQC1J36Aq`>e@@3@&m&g4B}iw)}>XuqUlSrV9<~o!n&2qOx)&<!oF(TmezRwRFOVSj|7$}^`fbTrHNyanBau28@e%uM-{NO
4<QTJ57%XhsJbXkMSv<`X8ttPVbg=2M7bf^qiVc~QjIZ0+&wWayotma!p<9N!Gga9O-TZWWJ3`w*~)C^G}Lkt1-&;63?ZN9(ik7edK*wOvq*kHT*AZef~i};
A4Enp9pCqM%KL`(GpK^a0|3$Q<oqExy_vP4wTs=1u?L_)lAofTqWBvTJnp+rU!uwBPCww<mL?4iWmvf7?BoH1?$CdhS?u*4Ohk?rn#)x4jMwkeXm2)AjdIyE
x|AGtOXsF&ws_NmY_J=wM!8Ibn+3wQCGtbuOougwJhUrKgQNVQ4EZSD!8xeR%ujvx`@g*tOzbD|Up-KuYlm0_K(qO9)d?OeT<UzB27J@v-0--r1XW0Wp}!L=
Jn!xQ7Bv~jWXE!v3RZZ7nE!yLAD5v`0qu(8<jk}ufcScAqm;UP<?5s6^Jve3{~Oh_ipND9#WHA_f#?e5ANA51kFKI4#50q#5}x?2pKm;yDvA?GV}c}}u!Y`I
CiSDpWlYKT5OKa7gRQo8-fiUdr(VaNmc-|OBIp5-yDb{*vom=2(hxT#obqlD#znhjI~RUo7rlPF5lyFFQ}k~#U14Qn0tPSbN&z!2@#1zxRaJEj$P{^;Wyvbh
Bt|1r3)kn0<LCcNr@p}eCOcI616Rwqe!3lQgem!vQ^3X~=5o@MI%2V-8qmlCLH&>V%g--x-=A|_9RVtocTPe#3v}iI2M_444}TpM;<+c>))#i-GGkm7n~hU~
tR((QZb9A{rToEfd=xL|2wz5}Ev0f8!v%Dnu6{{Zz7{3>A$mYX(G{7j1_)&5Yo<B*sA{prCAYR*oRm%{moj)H_L>Z{|CY0H8h56eBfZ|jhT`~BVGgFNQhtfU
A>aGsS#m2C7Y1g!#V``&m-!B(M@LwPTJ|j!Gd8Z#Z7HI_DAmKRxoy*@@XmYB^U0|sqJJ7`ELK;~#4IY>Q7>j#0x-NLo48!cF0i(+$dT+ZbLH9><?T(E9%dQQ
Ajg5(z&i?yUPwjoC!DLO(|%XtYOb&g3=Fr80+91_VtTSThMZrcBRZt3GHFl>J4#rg739dSh5uAR$UuMk=A4@}@zmR6OlnohPO90~6cn6VwnS1#w5zypqbnp=
2?*6qBH~qCj#`IuM!}(Mj}>V?TK{)XpoXPl3-bgI7d=0QR}7|DeXGtf$+n|Vcv~f`6%=>ZhOZ*Rbu2hD#@RH_h-R?_5NjmWP-x^005E_J&^~g{^9!bsV*0sl
-)S)N1<0vPTvn639eU<b1pIdt&EX$^yp%|dme$`x{Rwj!<OOlLKJBTf1qx|Mtn;U$dL8F$Yg0cE#5YwePJ}W^n&Sb7@cRlxa(!MuOEh$I{Lzxz*MqAlUvAf_
2@jG01B7Sb56hd4;Dls$I3Se#e@Yey3UmW8SK;K?w^O=kL~@-^hvyutzi>CR5Kp)lM}UaD{4rA6?xt}a+1d4a{8`hdOa9d2JeZQil)S||{I;~JkQyiKO~(@i
K;-G*yLGcl*$zI@t}x#sG{)tE1-`Tv4g5{bT<qf*X`b_vGo4HWp1%GsOs8N7;k3{WWq+pQF6hxMO)}rMupA=q*RlO)55AnJX=5lRR(9BnWDEBB8<NkpY7>*M
Xh_=q+08$a^SB|jnrV?F_A~^OBP+)NmCmG?3!a@$+Bw&5b-e9jnyv?#n6(%kw_lxTzBAj$9`{i>18ZPVZtCL=*O)nKiG6K*p)cspkc-d{UR8M}$tTW86vf}D
LmqIo8yl1=t?)@bYM)<@4~h~@N#DdAHjoAq@M4-JL2&=lz0#rUGxBtniysYzyLpvYF-f*k)`A2~sJOf?tf4mFL}b1DtF2FG611y@MEqu(s#*ynZw4w{lP10I
2{`~<XjPjNhhS#XNEz2K#a+Ru$hc-t*z(=gV{N<4r3uAbLB6mgKbVr>CBv{{x*1mS0;*{c+x!3iq9{f{N8sS?2q{M<8ke;h|E?li5x(|>3gLds4)TQ?YGxb?
vsK;)8C0i8UV<DLvBGUW9SVK@eRdRn-NVbh{r?8Ec%-{MJ9Y2G;|-QA3gf^q6V$=4VfUwx4T|6B2NAaJ_Ju63j&id8{el~Lb;GTvEAP@CRc2`UkX$~QhG#}w
@t^_*NS&2X58)#4F=?laiZZ<%Wo?VEPZS07>B$dElY?P({WgTIoyeBRo4kQAvM|O@gCvj(h40u*VOHrmlViy{2?f6G3fudzk2I&VW`=V6%NSFw>ZxaT%3TPo
ARne^5qpVGKEk05F=-vpEDbXzaRmfm4h1$d!2xQy9Xz<+O-ij0!44#iebeWWQ|yd&b%9LyUNo=;-5l7abKZK23mc9sHme>!@M^i=TE+Ll_NLncsrOM+o*kpc
R3UTzsVArTujqKtvU@*dBV+A@Lr04vl%WM*=w0<Kfrx5|kaP2sADG~9ao_E7;7h>`=h?CvgqP;Q<5dZp?QcnIDhgW{Ga~dkSp)tewn1uC+K0&xS|uWHbq%lC
$F`BTcyPajxGgS!`X%i2h4(!v`4Ur#H8`z`TvU{dnheU`$2VRISJB^Jp&P4eqG{Cx?qT<^5z0E5-w23eng2?GHE3$cIMLEFb=9j70Js*2+Ec4A#J#2D@$8t{
rCS25&6N<A5WD^QDmm+hdKO5h#fvqF9_8pClB;<9%qzpiDXo{0sTO1k2HhfEYYcw#Fq@M6?S5N;^r(Z=)K#_9{9cu4-O#M45XZ@^i3(1TIV>=dilv~U-5SFc
&loLM#Ins}O;FPUGraR^RBh2Us68dx-~e%d5*J}5tm~8X5lVnjc8Pj^#GIIq@YY%dPbD3<0NvPKcTBRy2r)-A=LmP?6rs>_odf(50CxRkK*Lm2hO+|b(#jtf
Ho;(2O`LcUKlVM=0#L*561mPUT2HT|v;dIRCf?ypiY<DmE^pQ&kAzvbD+G^zV)O3z#i0nA(JYSCQ6vCNFKq=(pgv;^W}lGe_f4J|Vgp@9pRZ$kXNCG~nrMU4
ug;A5Q}Q}*yWdTZ%9?&jdmKw=Y1$JNaZ_gBkSq|;3Sj_gZy->|wkVdc>$|NlkirnmBO>FPQo?~cSv@F9Rid$mO<oPJ+P!C+H51^^QIi$Ifdn_FTpdtlQ?9u2
!RSJGn62WsfIp%Q0ZfVx^UgFVU}NCmWPdC^j~&`IR<XSaa6XkEUF5b6jG!t`!l$3u27>-t0n;R?QlDuG0A+s9X70C{6=K+Np-)$UH5<?{3TZ$x1^+@zPw##N
VH)luWsZj*JF~~*g|ZO76)CFNKFX^IYO*H3UQ7FHT9#0snjzuBP>a93<?e~TeztDn8Y+%z+HV|>gV|Ci`@tln5H6nsNQWD!FkL(D)r{|;$H%W9r4|{ho$$80
Rc{D2(w^vR`1<eGE0ioy)?uw5{**VbuBGh(Q8i>zTp!J}l03pbSZ8BNLQ=AzfB)k#LpD~*up)?>Bpz2-zU-^g-`aP^!RQ%Hb(FeB&)^_Y5#Xt*q1(%`CQSQQ
>T{93<CrVev(6d?Q#lW03{pOT46Bv1HDqniKmqLR`-rMuyqTIP-ww>_{VKLnZ$YKD)b%RGK-k`%*&D&7T7LeBB&{Hc_a(oe1zf2(Cvl|bzYOA1{)`O~HLpz7
UOc)A!&?PM`v2OBUs8f{6QoGnKdDi~bFCVVXVXub?JAcJytGOl3EN-qcu*YoZM@w~PLtLnq4_O}PMBDr5YmSMbHs9!&w^8eQ4Uhdlv$=bJ^VQKf*I<*7<^T1
xe_8UrJ-@sR57*?Y$+e<#><C6L*by_gH?W&UBu<ea2QwNN3ro;#5mh_q8GY<68+Mey5qw;0tm1Wcm2F0yY@RU>LLNDsYSi!p5bb78X_+`ouOPY+nDzl$??6Q
D?{W~XuWz5S}A`HcDTHgb7SzVN@@U&Iy@@hVc?P<v-WJemQIV{d8rA@Ud!w4D@_w?S}Z^{W-B433*!}sD0)&`xib0C8xzQ&I1qhUa01|2sRyN5IqgaW9Eq^%
%YB%0Ino&xR{P`*i!r%|ZFAgXmnpemW(7$|Gr-Kzlmv#B7p_*b!6e<>#Vx`@C=2m7--aevdfv^7tW8sgXDyDt5SJK-umJ{I38B4Wn;-kXH9+CN(q~IyY__C?
&e&G;Hy!1Kg~WqbarLG=!wZgI2COUo0;9zopzS-?@I+prDz|3Kt(+820EIC(gD_5Sdp8MltW3#i)RA>s&S63@l{@asOTNr~i?dtTVoOUi*!!+GUq-rLxL{{a
c09t#pRlCyNV}5f6WXSrqM&x)@aJfmI_-RXKZ-zc6}Q?X_2s;mW3f5PK-%@qtjE7gHd&||kd2%%YpJ{`i%J4aki{>!()SsIj9^^`$!<z%K&w_;E@%;hq+^XH
RVH<4D*lwDv<+I$z!j-&c^yDbdU?J7UlVvciA<|gJDY0c1CaijP+6F@Fr!DC0CrH($tLkOoJyKqXLCt>pGF;GvEQN{5rpMtdXQDHKiXEUaARZ!drQAMNhoOM
1jaMGP7u5c1ta<$mbP7CBl)k<LGUxm>R*xdy>d}q*Jof-8z-ZesWN0}`*vIhc3G45nxUjI_34ehW(+rcrPjOS!OH15u&JgF$}T`Z5t)ggJ8;S)9QANoF2jzU
H(vCJMNkPH01qPTGo)MrJf<-d*m%HNC1_pi;X6MzFM`;h=gkA*G5-t=jY4vszTB_{;%c+LcPs_-h}NHlC_FQ~qP_M*sLNN9mKVeM>6!nlf$Fv^>ywUE>B!rO
#b|pZ%0bHRqW;?-bjD*jvaf>gKQ2E|)ezcN;pGlmz*6!`@XNyTYcJ$=wW<jWln5&~LmW|&;&@5fyOT-nc5@Cgd36;Y>b$4&zv-%~*{R96F^Qufdg=x#<IHjI
xjoPGR2i}kaJ(to8%ywQ@6DIGbkE38{m4_z=!cP_7Spns!`9qIR5iJHxJ;Qy6P>*Of2fk@$my=ylYPI?GhsV6(iY1VnQMHY>2VE)=v!V4Uwwdc*5AT&x7?{{
dg8q2-AKmVvhh4vFssZ*b>EJYB=H34eT_w2TiXVum#MtFLS){*UQrVZqsz@rRzIND#Ia5m9%ftJUjHq55bY)Li!CWS&0i11fPoIQd{1Krm(H<#WVBVxIJti6
;BlBtDdb*@UD=uFaA5^2w1MAFEqW$vLhnB-N8GDONV0o>0<8vZ9!XIzE50=a8Ul|TQ*s+?j)bxdS}^(8@|#Q6jwb_;b5uhdOts9^;D@wJmza)zeAYmiQQdla
y|Q)M7VdZgV+r<dt7$3NeRH^~)kutZf-+kIR0RIbT_tt8+p;PiG7jR4ecuvec*3W%xNcMqfFWEh=DP~$sF#w%zXDfcQTIq*y6gj-+qqEb&Zup+a*U5rQ)vVo
*tLS+(!S3_jE7tB2=YhM-%nB71@#dTCtpx?e%Z{HyGs;)B;aHf*NjpUPs}4G3lv{H5`S@0Nm+qe<Yy8L%WHg)ohPY;PIrJA5!|+;8zn-#4=AV;m-*Zu5Dau_
ll;Z5a1C$KA2>y~d%WQ}&&g`1^=cf91I~ikjxMmnPvYdO?=^FN12KxcOLc?gZ*Hj8Bf)nHoM9?J6URfIzy_`NV=2jEc3lEw4Yjylm^K2aB3DgqVU?-%O#>qX
BwofhYCJ|O%>)#-(C+Hk0|WRWpz$lw;QUp3{#_&h-c?&kT@O;1T;+)bX_`V}$*jh5ekTEhxeGUGM!Yb;T;fB-<>B`6*G--lZ-$c~V|}1$dFrw}uOmo2#D~gV
lH0m%gizW_CStw>*#$1Hw}Xl)>h*;w)tkugcBtA_tx(rs7<WjCypGU74PY$UV>k66osjEB=a`=XLa_rX3(jt390*||d?Eh;zS07Y);Z0f#PhpvA}|L7jw>tz
RRRwKEqsISXd)F_3b~6KRtZ~hxam!X4}!9bqlQX8Z1-|49srk=1Ui*yZgDb5a*nZ1FWlHgDsIEdjfllaKhq(vy$mZNg#Db3I2HDqD7L_=qn?0`hVD&S9_X`l
)+B#3%xSEVhB&Qx`d4fyH&ru>*a=vgnR+?ImMS1>S3s|~-fH9^ttpJjxP@(Vs{eF&P(96e+v1T9Yk5SqM(<&xAhNrA1hG8mmtIs$3fJd@TvPiIQ1vLRLnj@A
ZwVe7$L3rB^`b8DrLPY;Vsrh5^iu?uado#PC-qvZUnP}aS7StK2UDd1a(d+K)&BgN*@0ZhBiCyFUqBt~0lj~)1enTAH!nP5S@zWtmx=*wgK+d!%2n&fIzQVZ
cHkn`%{u6jch97Fkd#tt6*!25IAT0S$$Wrm=NM1XnYWxeAnGUlrIU6U7onitcEN6`iJ{i~sva@c=&(5M;~00HN#-86D9JSoYHXNhq^lx#O_55f&R&8nIvHb{
qQ&>%C8L=3)Gh*0c#gJ|3FQSn;_lM?d1n?&e~>XYz3hM?jh(@iPhX*hhVWC0N(M;aVfSS<);)lM>$q(fEdKx4|Gt!EmC*%dR@w`P2ij%?zjmx_P+U6A*M<ge
qv|Uk^3`f4EN760k_a>A0@o0I<J(B1F+HXi5)HGu3d(Wd<z{zivXv8M?SjR`WBVUGx<*BbW0R@}NJs0?c06ItDD=S8ge)BQvNl9JKpP^d2@I;jCnGi@twndz
?ULGwS_pQs07t4R()p>8-*C?4o3X<g8IbnoY%Pv}*<8oKfa1oprbX@mp;%8vVl9qali+*ySGV`c)d#6?YQzoI*D&zbV|heB{%{%kpAw{Y6gFUwkSR`=-O%bA
bX#sdApzr6wCyrpv*LwP0Km`IOUDqv*HqNth^3n~j0W4Cc}vTR<;-zV%(*C~w-Ix_qO|#c9j_+PTmYBMzw5NUC2R4)sboX6RyeCY>_p)j8_J!Vnw)Z$<vTYz
aDs|h2)=(PKYn6db<s_56(7$<3xl9LRR>pm$kcy~Fj+XgS8nZz-mRvYSM})cbA@y^?9F$rk9{`C{4lE&JM}l}O1ryuzcJ~d<RH&kv0w~xag5o`IyFiakhF-2
pPy0gCV{-YK(oplMlcc4NRX%F*Lb!eF`&Nq;}CNwnu?lN`2*|d5U9TX^1IaTPh#-34O1=a8L3cc+Cy-ojC0y)vh}SeUEylwr0u*Q^Q{Dhpv*ijOX1*WrC-3|
2?lHA-Mk{!ej>GMvK|sgDq3lOqY})$^k|?n-{XmCw=zK8SyWlN(q}v)JQK`vK~ok-Vz|B$l@x?C!s0n>WJZtNxP<TExkTrHjB#XWqj%G6yBNr6!LGc-L6Uq2
;mDetbERVAb@*)mRe=D!Csj5HjSs{?B~!SepvSaxyt0BuP-aZeCWu#*hOuSQ;@;-~+2>LL`WELOL$eCDqM~zt26jd8j<jmwi*<S`pbeK}%i?NnZOV3;e{C?&
(Aqk7lwS^ERA@M6ElsBbqriOEg#y{LIMUGGBzAWRUAt-Z23jIJhqMCCTQtL-Ld4>q7U}97Q6l&ShWG`uN!W6n5yRkW!b1=N%y6Hfh>i60lsS1n^3UHMJ1|Lg
AG>5!ZJGWNIvZ8;`9dMxy3JNPPG`IMZvcH{M{M3Da7fpz-C2unZgpb|b;B$dN5?yJZTSyn7{%m60Zz|pIB~}<XJ52nX!&jh39CD>sw%D^>FwF>Q5<vWa3-;u
hb%R-!$kqqM9w0F5l`%myF&ioGF4((O40(K(DFBwj-D}9@Od}oP~RgsmN(L**;do*_4>-qEziG-_4Ol?2PYQTFHjNdHdbHjBNZlwL8Z)Y9LfVo*-1gzEJ_>|
qNYy@hx6QWUXE6II>&t8=>}y@{zZDNG)sqnWDM~cv4}+R6K$2n@=<P$d0{LCR};>GXsOEI(8+Hah_MuaUSGb&`1;XO+^L6!t`N6p1v!NVExWE_qP6m_kq>ls
zL;%w{zvZTDiS{q7}CyyS<I#Zz`vr=Z-H>PlS65W{!M-JMBfjd%9_<TKXi{<%FCL$U6Dq`$4tT8^G^RT!mlYU6~{-pf|_wdq`Dgf@+&fF=3)0-YsRTBD2bzZ
MLU}uoy4#=)u^`JTUJzC4dxxW{&JEKI$~U_Zk}5I7G=PlxMTZmSF5{}wm+DSRW<+kko&rD|5KtR_Oo}$%9Z@M%`fspX?MZo526E@Vk0Lq4~MpPQ4L1RKtCZw
w7}y}?t+5YsjCNb;&>fJer4GvZOo6&wUznmq&$K~Ngj5wLGU9(E;ZyQB#<LzFQeNC7(T0|1-BHH*>5hcOXII)l0QAMUaLusA|aX;sTo9Tn9Do!5TX_~6t9$m
iODC4*V5Yt`pkeXNF#GcoG@Ph)^Ds^p}b)bcv=r0NIHOTID|%<$I*pZsolFbHP5vZKL_ledp~vf$5g^|yh!1mV64IpUpV;i15y>f(A8y$q%!EOFr>K<siX3W
7I|0q7h{qIt8=rgNM;^aUH5u<$vZzjsqu$*8>4uXx>|s=_dm4F^4AqIQuUeN6P|?G_XH_<CksTgA6RoNq9$tnl31TGgza4{Fet-UOL58RdTA<3LRu|tD=NLg
r$LooF^%XTyD__<=mnxjXlvN6mWj2RFu%McXcOmmCbGrnoV2%zs5(?OPcH^%AJPFv%o`78!eA3_mx~nL<qr(HPBSf^{4gDeyx}OA@ZMndppK55a|)CG>O43m
yDulBLvPRlQ#y%=D0Esigy~BwL!6XhM~DF<wv{&QU+vr0J)eKNK{U1!Q~DVJ0Or02h_G{lh>JXG+UwScZh5e>;*{J*a1%!2;%PRlap01F@~r%rUSraS@H;}m
y7G#~IPBo~POIN@hVom$)YxKnMn&OsBP^v*3D?mQK>d3JdSQAU8y>KV!BjA$rg~sZr=I>g2x}FcEzJ`I`pc}Do+O>3E}2Q!G^T_fmMIMySb+ty>KwhM*!P<{
BTv#h36`FLXC*dZa!5Q>xS$`>*Pht@&4LmQqQnauabJ_pIxvs1!G!tMH3V_ocEDK4y>3T}g#Z5);-FWQ&{}h=w0uN^xqo&hf=u`+)ruadOMYXCy@11wYm?`S
)TmP_#NEr~jOlK_y=gg8siOPhD&=#z@L}P@mgBd2+`I0^Z1f*S{TX9o#l%~Jc6ar8p+sC^kCU>vbA1~++u{_vl(lHy4Or~3%MRHKQvUBMhszGDH1aElw~@2h
RbP3(YH$!a8rZinyzJoW;dD&$LeE|V&b8>gPf9^*0`cWW(IcSiH&1_wt1%yNBY6_Qakm(u7M<-PzKO>m=}wj%*HJWmTucKeL>7DPfL70%6LEqBvQ7KKX#3y{
g<RdzXe)_y5u0a3WW9g$oLUajmuu_?=_2|fKSjCoCK!t}70Vc29#NuGRFpE8`eY?|or6xxlBgg3dqG0y56@TAf_PgxEcShjoXP*F+Udp%T79+WtZsHAVZ3Ek
osC~z#o!GY0VWwu14lN2_&5gDRv4ziP{kzxWWyXw(2$1LG7wJ|z&IR+@4l}I(K<zw4-Bkhk6K5ealQU2J|YBKzI>W#)E84EXx@O$mp$J28kRsD{Cq|(0_)uL
LL@u>5v1gbUA3A9MT2$XxK<g9Ku;N;B1RUG-3jSD`yNrj*IthtwssmP>UU_5ytU4QgM_iBjL;wkZEIa27X`DRRwTvsZ9D)AU=Y7xG&{j6k;sOvGP`w9b=1eV
?mKQ17^tmZ72D&G4pzHA<p~CSe!5!z1W!LZqvbaJZx{IP`2b0tVdBQ)2C~|ySu5v4q0Lx)zq94TP`nDP&Z|bPX9<qQ)eSIXC1b9WpjkKd{S7$5=}($s+inm8
v6mVu9?)tl2|umJnud)Ia2W3_l3y;+9%3fvG5ulZ`BfXaf?c2Ocsxu>DN;|MFg6Ki(?hiip9IIhv49DOL!8*qDQZmiD+n^|Ko38x&N&aXb!V(ye?{6Y0zp<Q
!+&sbBqjVe#(?LMiD?T=wX_{yrB^7gJgYR7-o{6^SUw<Ea#9q<ntGc9f;n}kuooLA@v2f=gwKsw9LXd&=1N|ud9zXeR^NaSqOA1fF(7vKGP+q@2})uC`@tXt
?@4`lefmANO60Z*Ey$0wZSq0P;<rNfHb(12j9bqpH5uvC%Wj$WOE`bU4xbEjW1VYmoIGd%Zd#N(cR?D4g3F#_x2lgsKjEkv%ks#-Tow-7WPp52oo_BO>o495
H{eaq+aaxD2R^}FoNCa9^#)B1ZKHE1x~u|eQ48UkW_RT*LH|>{V!*O-6-v@_*=yF82*BDYRtgU3_2-S$@dr#|MxG1uclP-ym+1X7zLOe$S^LZD5C1Vu(@)8m
EG>0d;Gbi|O6FtZ?lLLiI$yCV;ZK7;p&aCVB3R;@*4FdBwv?V?DHtFblY?n=&iS|qchB*5&m=}dTmM1hvm~6Wv}&h}1KjSjr1&^F$2k^kY-pzXSRYn4oxfp7
1G;>xwFi%Eu(Hw5Cq6L|R(4`pSu}lyH_Af2*aZ{BHcP4K7#DL~IL)G1c|+U%`u;Ycv9FMWoZd1kND@MW5FATlLSlB3x^A6jk-Ik}h8OaET(pV{LaUfv#YzNo
z~2;@hf>+ZTH9Y72PpckA}EkS1IGZxCbla@a6&GSyC&_@jJMAonYcv&^+wTP8C=xd9gZP9)c#V3;S!y`KBRxAB8+e^0505!o$;2|i+u1Y?KRfWU7ey20-3a}
4_{pDUA-+M4>Xg1NXKaY6a@OGdli{SORXXb4Vr+xa7){0HZ>G1(D03eYJJx%FYQ=hw)eQTSTgr6#<t%%TQC*q#&+HwI{Dte=3W&hz)<R6=QTPen1y+kX$ggv
hmucfE~D-u3fG)v4MuH8^&1LG=&X$FzWPEG>4IqVpyRxpQx(AbcJN(<pGq!(vXC?XSr;{Il_{8Cs|uCs!bp-H)1#_YZg#V$W!o~?K(yb097u|{`_2v9TlWvR
eX*c0NPka&YxPh&sO0u<M@Mj0J|Y}CL!WDXJ(nTZ$Fg*A6y0wmbb<=sk}2Ur%1*urK*9Niadhf3A*iH^xcj~O1{m1boVS%i-Gzq0WELeaJmR9=fHebA7)#!v
Jm+UZC})qa5CHaz_W@Y9ccs?CA`pwEUfbI_bw}>wH5Tmk%L1q`y+w{jp>(VsKw2Y)`mSp_-jk7jb_b}nQ2+W;^tax!OZ(($+sD~LO+wl6UB21fA>~@H_zXg!
vtTOJEue>b`Hv24sjoqyav9r#JHqW;|DQh4{0gzwL{v5>#i~)bEWG)7clPn8Gbb0&c=1~%hMj;o^a;*XK*WWST2ym-SQx4r6V!9M#{`{$eM?rG8(r4>r=%Q8
W_k3LCW%k`a6>Bh|4g2q<mGwWA&p$u3E>~axLI5OU@b(BuhKI$C=0Q{^#Gda%2M3%jIv{bUXu{PAN4(a@s^|>SCAJvu|hh2TFXX1N-g54Qoau$*a)dcdSoS{
QMGFc|A4CKEfmU$w;B?LyC!w}KCd<rb^N;yo@wi_q_(k7$C``wwsdAFj%Ya>fgUSiY_GK9MG}K<R2vmI&W-T$ynF}>r9!_rS5&}ZQl$a6k^(syojD}}XhnJy
BP0<dULOe&2(a;2NrtQqs82B?gh6OnN)FU^daS+i$TZVpK{P5BJ;HrYM=5m}(mziqRgxk?fxV7NV&mSpGm*;Y=I69`sHbUFj#N}NM2pu=KX<@4RFP0VvA`ec
NWa>fp;(OjCQL99(HS@y(IxMCFZmpWiE)ZaA<3?MM|gC{2c#cF16|@)S65njdFu6)G?BnidrB6aC15=TXY80iw{!3vaE~S3oNWbMlfQerGzVS1<N5*Rd=$@!
pl~2moyE6n8u?dY>qM~Z$f2ThUd~(V*sR1IRFG=hiSY&3Um0iF(It2!6j<%<N?&F&Sd=ELSHS0;*o9N`I~k6}T|!RpVNVOVbA*SQMQw6ELj$pPzQR7S_Sonf
M{8DuFLgoYm;wBKGt*min7q*!ETO9(-gPKk`c11`D<-S5B0j;py6y#hfqszk9VzaFnC>cIE1K-DkGL(CsdjTRhgD$Y5dHaGt9z{cc3oEHaK`L`K_*KLbD`cv
0|S>Sq9O&NN0_5v=bL$Q^#@KFcPckJiN9GjBm6F7N<pp)|2Q+>@e3JiR{~JUZV?4&0Ts??OmxfTc7O`Pn{7l#=u(pLF6^YXFf$tn@M`bFcUJoNp%2~9E;6|7
q-x?lHo}O}2?MH@nNhKaPMz1#vzjZVKhydafq5uYxK9-xFo>2v_)jExNL<I+u2)nKIsg8>*$hIi2oE?=K7}RtBE;Z9KRRmVmudnN>V?Nv2zY-zbL1Ej_*5k?
lTI2(BxbH@#fkj0NYLc;>%*C<nILHv@Sfh8m7c6>oKT?$+^F786qi@(_ezMk=f<>&zzcScdJoWLJ`zM5KJcCw?v*h9-qT_2T<>TssF-2U6OX-LD8O)z%{*<P
bae_C5Mk0+PahQ3F>>i^jH7y91+=T=mJ>HxaBU8)P@>#0>^bmfyI*~ty>hL(x+V9M$mB>>D+Jvap3eYk4eX%c<r?t4!K8U%Hp?r%-1V=Yh|F*f!Gk&PLYnNZ
A8eph-RCht0_#)O!}znoG+-YJ+uJq!HR!hAf&)IUi${N@J8pz4QW^Oj?w<F&qI})z6<N7C`PEz<lug>oJfH7if&U6GwdE>mag*`egzoW6^KTE1T5!HZ%ShYD
)kQDO-X8&MEZE@QodY9!>g?d_>i&IET>6@fywKG8B!cV4qSja-QFA6RZ3P^R&c$3NB0aLCa4nQUouVK;Hf^?Pqt<(E`9u7G8FkVMNRIRCtLOG{4}8f;{APe*
4gyYHxUKO}M>(C+SVG7bZlN6k%VGmP=DHPeJh9^%*Ipl=?&GB3FtWj08?WnNEAl@~`F2~YmEJXuE#^Gk9A~MvaIHL=EYO|}H36X8g9{j#7b|FECy0}6UR)+8
9;8>94gG(9q?op?|8PME;6rT%zQD1z->oeIIrJlkexRxgUT|R-w)qz3gEuaF`7D3Hp4V)OptMcQE0GA&GyA%re$1fYs5muh0HwZXK7R2xP-_^y$O!iIG0et@
h>KUcXcf&yo||7SKTWbDHNbU=d1pSRTKYlH(Duxd<#1M4D)90T1sF8RoDwICIgtMnn<q(2nd0oTMXsn~yT}||3^0|-p5nOPKZNl8^xJ0mdZ`s2c=lJ)wHVe?
Nk4MQ(8tFMN?7f<eR>tyAK8R$;G=3`dKB$t%A%g|O)lNWZ>(z5N6?}OD+E5r0u>6>v^`L(9P8q0K9Hs{S<zH98M2lwa`Rwh@YusSP=P}+gyVyGT;FmTj4Wwx
q!J}ieVXm_^<wrblX?ok?LKvRJ1v>-;A&kVBBh>6q1d&c#?8Ac-+6(z+QW)Yh5aL4Jf8;dAT~?N(C4`pRrz`-@n1c#?di)BbGs!c!|^b}&JY-jfQ!yK_r3fE
-0~!;Wv3vWlRvtM`>s{p!-XeR&VG~~2ijhgLi8)1-f3x=kL9Q_W&c*Ords=w^NItqTJ{#DpjJ%8+}0IU>&H#YRNDy4qm52o)BP`WTwm&UO7>7$3vILt75PIy
TkQogPw#6=8NZvJY*oT0tYT)O=3tYC%fD5Z+5S#oC5Q4!F8Y;vh4Da%?Im|c)9KI~#%g(nOq6W>HvoiF{A;;+*X1f(_9^65rh%ti7b+^??Sum{DzScbXpu7l
2nT}G8*+q+*ka^5<%oYCUll&J2JfTOX$gQ!9eJ&X?Cha87f{LW{^je<$U-1g3LFtEcSopqkX1+2*+lb1(L%YMfxme5ze^;0$OQbm(^U}@?a(t`n}3~|CC${h
xhloX1#oxeVTej&sohP+rzUnYnOyY_C4G|pl_M^NjY3rMD+6Dq>Q)ZcrotKB3gI47X~E=H-ZlAnMwHC<I&w4$GYsO~<dT<cG6$d&b9DFGd(Lw}D1WK!KJGsK
_7$!+3wp|5BGHwt^pOOj6mq6CDOQ>K7gGmoJ!39C1I;phPutbmp&%vn48LgTK_@X?SR;02d)d=Tzu&)L;U4~iCUkg$bSDg3GUJm|*rW<#nggXSna^Gg|M#XA
U9<^ShNJKtYhyRA-pDEZ;ru}g%BI|g2q1Lizf1cp0hNH$3-H2-om1|}a6IrVtapv<55KXn2wmFV*mmW_XM~?j=@#12$x(bxExo%EKBXfHfs1i?kBSok<t8Ou
zbr}-@KIE(JOG%Dgn-DD>V(7J(m%&#0cE%@cU}Q&{yi-$WRwm31<Mo~sAJ;P(i}pg{oN_Qq5j5`0)@eQI#FzF_104<V3#`i?`Q$Nr+o=)-v96DdR7WFACE@R
g#=qvO;ZaIgW$$9E?VTJcrBNoK#O-<@lluev3C$&?|LU=cNB?uT3cV?fXtnDoQ$hFp<4=WWxhm?*rKBmzdO3~AV$^TKtf-t!`FP(&17h4t2@(xh)oJS0~)l^
N5~gzOW&yJUcK%O#rmh^wZIFzfa2Qgkb!KUcyEkeDLi82LmM8x==YVJ=c%w&_21l{ko*T80y<$&lL=Y0A}gzat+}9)aNw1^S&q8M<^5R_?Vrzfke*0l#a{@3
GwnO6ziyubaRM<oBWT7<Ik^`6^0<dV0i&6O)0m!@gIKB@TKV;Kl%(Lw(pv;0!R(gG%=klZ%BV{Ez-E^MhAPOqarrFn(3~7JJe@desV8{OKLJSekQv74OAZxw
bA7~9?}*#NmIDlCI6*F);E9*Uh|X6_5Q>SbywZDmV7>oTy{6-_7qG{kzDt)RaT?)LqM&ortpuV%C6UGoqs<=6lyZ}u)LHPx969FaL6Vh~%HBS(!zTyDFQJtM
59J=!1c=LA?3v(OKfIe$U+6nSuK(Hh>*us6GrN8<L<0F0M<86DOpL-m&$@5y8XnpUqXRP~y4hsrgeo)LIYVy&RZ$O01o9B62j{8EmF8gmxK<cnN~*i5Q&}m1
c>>AC$)_N&WP9}Y@QEL+BkKTsa5J*gYq!#U-6~t+$2>0E2r3c3Y^c>aKYlpWA+&yBSy8&WfDfZZ8%v>86iub9v;#x73LP<a$H7FyJxM+_WR*m;&F!_HJYjfc
I7>drg{EGfIrGZiV~3vA1vDASD^0CQu|3XR$O-|KZ^NTubZP#GDb%WiZ@D{R+$#Td81@R#+Rs&Z;@n3JO^jE5(4)yx@zZM%)m}$8H(Ns)jSDeLG`%DK;maa?
SBf(L%cF$ymO!&<9KyyKJufi$PTchftz($B;N6n}L)Mbw*(xICU%9GU|BXiheO|8XSYg4FU)7qH3QyFL3s6z$+qp&Wd;uh-icB&LU6ennB#qv9v{E%Nr8p3}
>i=044u$JXEm>@U2Dod@{Yp62X$b~|tXsgrOBMhI;Y&*+>CAnOhNslh=mcWCxIJFR?u;ZBZIxnAG#Nf{r~O<cC8d#?-TckVJa${bVsUie*=PM){zq(mcL-_9
1K>OSZByTC9rRp^D@~bIie`h#f_9f6vVkHlVcM_RyGYMX#0w<Up}eC^ye1kvED}{YhjcF$J=a9#u>4Bm@(cec&<L)6+a?=14#~TvQ{ZzZJ(8)G0acTMz>&JF
kH<!t`yna~wL6~l7;?9!J`CRYcsIcczHVXX=;Lp%GCvvs%X29-g0(%vy7dGv4f)=GxW^&oRX8vhzW(*|fP<X(Em&+^0FCBpQ9bcr+|LTt?`MYzU_gBe?vScg
u|mQQWq7tV8q5v4^Z2ZT<c*w)l;$X^5+MwEZYi0Qt(Zr76t0HvvI!^ZVHvT~lUm)gJpAwjO7c@^g*hJaSWlr|8H(`{E@^HESSRWfK$`dn*CS|7lmoqIja}pK
3b$5&QCwU7($2_sn9b{a`baqr<tlh8<;|<}Fp$bI6qu!|I_aoH$^}Fd%KaNI?ge=Xgk!5h8hSIY((bGK$MM5JYS!F%Khk9-bPMVkIS?Jhb@hZP9bvEROhlS5
u0&?)8BO~O`bVg;toRn32+ii(K8Va)8z}zN29&?V2ow*Uc46Uo8K<ggJ4=9@ZxAvPa|0t=e-&>36Vz!0N3XSC%7g7BE#1_P<kKjYfkbC}_(a`@7C}Iw$6)nQ
X7~go0pjv3b%<BQPG7}5bM0gk$B#W54o*BX*|U<H+J@)4enayzXIIjt(o<mBjFhQrWV}h5IWm7$Um1%XfWQY#B4?w>^aoK`{WV4K=zmpQ``|!k%68^dzLI2w
9Kdub4s{5o@eqCF8qTBd5YUWqOUjz6uh8%aB4$*7x?<J>mcj>^`(dQ+nP^+O(Q>^oy0P8XAHA`~dqdS{nG0*NKpZoIpf^ZsINVj9*DICBn1&y^nV;I7YCyyq
1x;QeBX}{19-(W4N76!yj&Doyw<%@3l5C&VE&LA$h5X#vtO`Z1TL>fgW!432ikft^x(hF5&M(whxp^Ib1OjYj@IzDpL9xzRssW`BW%qa<SYmS7b1NJ6F%l^r
GqqM@9r|a6*Iu<ea@XOvnbbH}+nGB1uPnXACj~Sx%7v)<M1sK^2G|5vuK0ay+ye-S2IEabn7di~?(ZWB-(m2)yS8%uflw_Ttier99DP*(5}d{au%%mVC%^1#
l@nBZN>;Zq_oxAQ>gbfr^9pjKy7U4(cW5<|wmJreYlgOK9<a>d;IWSASkG!j5@DgFq9V%n%G)C8GR15Y{L$}mj}VZFOHHuRsJdS{_nVzxI~IxM{9h9zE<eBE
#^=<Mo)9It?C{Y-B5=#r7a&$G^#k43m!6OPT`a*omA(4O?&qN!?|pG`+sO)mVd@)^%8LRUt)3H$4%5=TK!6KdG^Xrre2E}Q=Dw`6<NM_X{;jZo6&-{%I!XWW
^O}Wb?h)+%j7hz&bCse$mvAvW2KNqAFGs0C!s^YrW{#@lDa+!lQ##%;<3R7=)fk(<#!rjc{3Pzu6{$fbkRMjgp0Ch@0g-RBB`}%J=1SE(^tT8OL#C0wcbECh
CJ1Da=pE!|2noNjQd+H@18W2*fp}nX0EEp#Cins1Vz?=+WY>Z3(P&4vzJ@DMhqAnS=nSMnOy|Dq@d=<a)?%<6{mG+qGl$8xjAfr&FDjZyXyDddR&}9R9s&mR
sA#_*cWE=^jH-4MpuApR)M6jiG?Cj&NrSCBAm|joNZJ#|oZ(NXs(~+`-lHfWFC>2|-$g~n0_i>d`?`!N%S)#=^S{(|!#}_M&8-}FE7Q6{2fh%l*=;HijI_vc
J`2vd{dqYb68OA@Z?dPhJYO(FBs?&No66ZTjR(p&qj6v?t~|8#Ym3xs$2@J&GxUCrrJ@<(78zriqQY8Ay?U6!1Qr6Y{@vh>rn?eFq%NwVpMqu!^Q^JE0w$Tf
QkqCP{v`7$$4?lB&IOl{)Xs(_)K1kH)lN(U+FkHSu`<DS<!*>K)_mi$b#fn?%fL$<F*Qy8@hcIGl{(EvP;LjHMi}Ho#fNh{ThM*I+&*}s5$>*WefW8eypk2Q
Nx61#PC%d%EXsrhR-c!F2r*1j?w55IVX*V$#jymH@@M#qz!OCA<mjQm<Bjaat{*&$OFIyHhT?KbC%OT9X`_CHH^(RquP_ow%$qc%2lzJdoA1pa_^}>@waPZO
!oh74ibQmL_2Sg1Iudf*Ak6d<Mo`O)VM0IscB0>r8LVClZ_3z2XSlR9i6)_Om2G_y*}X1A_HM!P2t_c`Oa&?#Xi{#@+`;>65N1)xs4v#r<ym&+F$Ad2mTH{@
(zoV^L>a+JcAQEQ*eA~CRU|n~^YVPuFA2;0w0I1C|CCfA3pfB@Sg2)v9Z0(-ql6Atkj+d-H0+)aApTvG8b6_1MNLIxk8bGO0X@7f^<f|YL<%ZBCEc?JTzT0{
HxeL3KfnEa<|1&r(t8uY32wNG7l!7>)(W#OmI@>ugE}gHE#HPS0i|*_7~M*uzaO2DMJ5|X>_@bs%li0{e;6pBcT>A*(#MwPv+)wBe`mcx26UM=o~AQe_e|}<
vf%T0SAu_4<fS>^I*6Vt=eq*uu2T*eg1wPtp_a$DW|mX*ZjT25I|55eP|A=nEaFb$pUGuBban@y#C6Z1v)5Et?=|9X+G*v>4(D@4^KRCI#mQ3i2sZ?;$pIkH
8u6YwoaNkCLzc>0Iq$s*`xIxEB?|qH@Oe{;3YvXyQsMR&Xokq?%Ix9|0>PP+9LM-0n)>V%>6O2=)h=o^^fWjN1OSq^Q?qxrrm&m|nJ#$(QWK?ID=3>L?F0bP
7btpci4iNKbpQ~{Yv-3j2vtbjd>r3(KBJWvO}p^u+YZli@5dt#IX4sEcmo8AW1}T9n<61Ir@X#qyo!xHbSohm>OBtD?!SiApAY#XSz8VqARaC~{TTrlfJAVe
c&Bll26$ViImu~KGz8{`f1<IW1Ks2hxN4KAPLpVl;G{OJdkt_PP2;MLodeXP2B%_@T92$cR07gh-j*-~j;zD!9F<>w`M||n`>&kg0fi&f#(W^R?YgYqj>bc{
QlPW3k274qz)ZlpWf#?cz}CdFbj4`2K`I5cht=y7(!Lw(&6P+90St2jII=n04Em}un&doW>3WV}g`aLloPnXJu*a*O%P3K!gB9vr#Vuv=jBBa?PscDlRE1cQ
Wgqc5JP&kGQ2@`pr|z-lt<3~=fH-g?&E*x{vwa_7&XZaWo_4HZdz1!nimKUUs9&U#NcelNL>6!bq!{14xMV6RD7m5(gd=a|Tz(3&MttMffauxg6zIVizC+M8
pEJjltrTeH2a2b>mzPQ?3$p~=t%|e}#*&!ANh#Ws>ilEBW`%V`{=Z5!G9Ee~<Edl)>k9xNVU_EQ=p~D-hfTuh#vcU&4Pgs58(6u3gaNp40@{hLg0XM84|X-{
M|Lh@L2V<OTLM*=dOZpt@Jey0l<2#-PoW_)LK{cjHKC)tJI!8Mr>Twx#K7rKrl@Rox`(ONz39^lH`C*02&bR(pzQ`t{ii_-j#<sy&D#g%!G-V)-*iG&_k?0O
U~C$Xp?w3W*q~X=No9<+qAh%nK7D$EOxQBS@XV60BJUh~%C2}^WTYWLpB1*+v}GX3Uq(QAvW^*lbIMl`*_5G7+Q=rY)ziq!|GAgSH`Y3(PE<c1hWC4TuoLnQ
J6PDcR2clq@2j(z$m_Mvp68O7Gt4+i%elh-Pd{sc)U*Oz354Q%OAL$a0iZR$s@YmY)*5VZCBhw!BLoSj)DFflm9r8=eAw?Iz<b!dUt)9i+FM#t^OIr&Jp_7F
X4Pce7e{g@D)aY*82C1l!T-XvT;sIz{-Ed(nO$J4`~AVKN-$Jp{1t_4(m*DOw6j>6ThXKr0vp@8Wy5V|r=Ja~E;!05=1zrd7vWWu5&yyOOGJUXjRalm+|2rd
Q^Dvf^VlVk(DKt=ep?~*va`s8ce|>BKuGPuE8fEr*`mmnGVQZ<YGP0v7^3s>7pc9xG;)x<>Ae(hU7TfQZHi&e2~u9AQ}!Xd#T{ep*~yhBWHC!#u0XEKYyIRU
AAuYg<g)e<@;1~*kV(=B<wM!i<cy;N;Od)DZ`OxMj40qU@Fwq3s)ie|vv=4!)zbr4bKlW5le~D$s|_3PQI`3LjJqnI7j(Z9j<|9TlA2%<kX7e0#{Aft-R=u;
EZ7@l9L~HnQcUs*r3_jv1mkP<npH!D3DHjz(q%l5hk&dVRE98BS9{6F<%W8ZpgK^2Bz$K%(7Lm?GxL{5uveFjCE}U!AC|Iu@n-4*e#$i)qSmX_K=c?G_f@Ub
o0m-F;D(X;(D~o>$-*$O%Up~pvt0|HkF2$G;b8oh&0%Y|)J}YFJzBQw;(oBw*;?g=roT5&1LhIhwufvVj8azSwGfL^#?GNHy0`9L|913H+9#Dfn=y@eN4U-I
XyzNXUHIc1`fB@V9;m~(VHwI~syMC;!Df*({$k{Ve13xAA?GF(h&}Eec$<3N&{C{sJXCW!lq?_Gl6QMOX&J*Kqa<-4eY>N8r7zL6_JBoSlDQphTNQ<)&#~wQ
!S-8QU+qC;q^p-TVzgw!e<zni>S{!=iuuT41YG%e;WVd3nrVzwNAElz;P>A}$hLrqu=9>;YKX)B9F*B7mtFo4nVFodz8_$<!|N_RtQ32dF7*q$Z>!zK;F4~L
^icxf<8Dk@_Oe$X-2o-u_R#}zErF(_Xe`|;)kpy`q(3qYx+z&Ew|uxq9M!wN^IZNFm3j%qfl}xW<R90<40^$Se4FZ^0jbcB1#}&w+<?*bVt*NNr!SY5<#=pH
Y)4+4Zpv&XgUA^#Io-BOm!-Q1B#-E`O)VRo*zo~$i<|7xpICo;<K5<T0~Db$a|;5=9@cBz^z`hSMoi&)w$kgFpyMAMn-Z94XA}>}aHrFnc+tdH!m3}a;E_q`
C^Z_TbMn-ba5}s>Y|Ii6DXWet4IbaeP{&y)*7bEckkJOB?*_`k*q7YkJQtCL4n)Xo$q1j(PE6MdwlL^B8=1cRar9$}jwf(>D(;uVM_t}8$t0T?5uTPZ!D-6p
f20kRVj}I|22n|<uB;%BLp<>zwkry`^LC1WrFh2Hp=G`qMTh%x{cdkTQw}`V7U~_SBs>-CLR&hZ5GSjGi16mdKHUdxmFCR-;*xMix<D$2L`$z7GYU%59em)!
>lsy3M5HS>**QeChVV|SgyUyNiP|h|E4;BJ#jG85*pd^ObPQnT%WOb$u%u5H3Yun>;|s;Zc!z(NYxa4S)^_8Iym~b+-y(&`V*@wLIs(OidO3xs>OYovH)Z|N
)wf{Wg4dmrWs!DU7!t}ws=oZk=?|B_3cb$I=(s+C6Y_|>cnSWobI3$tpZ+r*3rorg(+SkfQVm{St1Bj)R-``Da%>f|d_2+Ldx;$IjI+@&i{0z4mZ=tZ%AhE&
8=U$!R4`xPDI;?)I^5@5e14TD9d#-r4Bh6zyUacU|B1gAy(27!Dg7@6%_Z%XQ)?KOX(O3sDLHwtpzYYAeeE6nSYs7**)mL9e2gSFxT`xwgl+jap7`&7NtiG1
^>2H(8U3I3s(&h+P-FY4F!f&(kp_1!`UquqBg=ZNPHoHt+oo|F3BLMVw(2)`wtqxddoJFx20dG7)9E;HkrCGu<7Et{uYcu1>K8*p;;OEZA-%2jYX+E2IlBaE
M3lHgaz$rnaOH@8M7^BEu%TO-L^(CXIkGaA+a8nq>{1Rzc$3+;paR$jUv<Y}@Is{LRNlRZ$l^)eq=BAf4A0@!2Fo`^0t(y!{+{F$r7)-<oXeq3y6<%|8%RX2
_<Kq&VukOf9Wjruy~cN1uwB+wrI!Mv(ERDNhRJ)qldScdb*;mUaN4Xrf%r1hk4<9BwXhikeeQN|%Qr?P#T#1A?)p_<JH$#Kwzx&G>v}o;f78ga6*!pQR4nPK
L8J)Hb;cJ<cHzFWGF+}ElJSP(PP}+JS>S0T`-V^xIaixg>(1jon^pW1Iy;ai;r&7aKQ2WhClO+MNJ#$s^a$U;AdB=()jz^_JdIdmkHz9*o5je#+csZ)T{G5`
q$BQfV6#xh;v=y3mGvXt`0jHR9xHQV`3asRLu~#j=@9bN`YLwJx%keMlQ-`*CIAQ#O_H1nPd6$aLVP|6!DlSX$ss(4y~1^<C`rpFOESxim2>07_Ctn?hiSsk
2SejaT6JBVAm7;J+|$4L$}oFdN_q$%|7h~^%x?tK!YX1BeX@69S3R`C^ou;mm$DY1E4AF6tLycML4G)&UIreSppmmaC(MpDdQq@zb*sC26Mli@Qx9eEJ#x6)
`!j4KfhlI7D|Yf&Oa_ws1vB}0eoQ&rt7eribFGP~y)xa;d+F7D>M7P=!yxZxafN*jOIP2y_;e^S(4+}wimQdi)g``l&&~8BiLVv4`-P2YY5F~Vfj}hTmXNJP
Q;k$jV!{z&7_<M5U)HL*n*TgcBh~V&#vgTx>T030n;7gc=m~nA|1FNcg64YCsu0B*srIEn@r^kb-PpWq_L4Sd#|s1Qk-{Gb7Uay<e=_TH4FY1wk+bj=e^6-}
yKY>`c`k~bzH+HIPV?3k(`ul?BWn>2<M{YZm*+tR?$C~GBRw$-y`FBYpbAOUGtl3BDoC+Vs6yV-4tefony+JcaBo-?AF{jmsmfKN72?URBAMur`}Z1y&SFqw
OxdjJxGL<p1o8NL2~o+9j*5X&bvV}Wj^}LijQDU}b#@n+I(3#a>Oh3SJ(ufCBebh|kIjLv%Pior@)fmrdWAd->O~r)c=-#@gQRF-X4I^hxALbl!B5ONJ+bEL
zG3Gx*!DQIJYMq1#cft&hSsU4NfBDbI2`duQ@?GQ<Diy|z-m+X<#}UjgOtJXX(aYT{VS+;z68S*C4xRHsB_K%@K+jZ1`U;%F0Kvs<>K;deyJImuw;=YwtU#3
8G#;WG6LxW2MrDFzd2U*(Vke!#av%_8}f^f=@ys{^XBt~YXg-}Bhcmg7qXiIgP)F2KZP5~L0UI6TorHpvbBMY5?K;v9Je$kJ3I`MD<wT)kE^7Qaw3&o6-%p*
qh8es<5Kje019|+1M54e0`vc##`U7!?purjI#bsec?5BOsa2nM7GZKH?Cr{_xZ>=!aL{ti^ABtLOD6g=*l*%h4k-{CUUrO*gzeXxWkOp`<-u5;F>SJi>-(o&
BG$wGa_p(lx=6*auaT|^@SRi0;9?DU?3zwwZn2bM7aLwtkmc9P2K{_0zNj*ch;3w7AhA<0iG1`T05zb-!vLc4j1jM_@(XiGgh4$v4ak9pw^4WEP^OL$P7D^#
T0CcKP9X@SaPg{vv$}m*o+X!zIuc+L8Ue&gylNk5%MO8kVt#!!OtJso!>+*L3%XKM4)UK}DSQpa<)AB@rF?W{$|&wxBpQelp2Ffq9%LT7Mu2c%0oa`9L-n0|
Bm+*23<h&Baf9s>|7x7F1Fr!h?zvv(1LDp~_%M^we@&B|Wl5_?%x)Pylin>@SLCs5g9kMZK*je(a#nEUO>o?Yr;TK(pDt<^xx2C_Ycp6aemKIQp{8#pY4)!(
HeYcN<~Gfob{YP3bwTTKqE}{`{*tAzuc!hjv7qp96@$X&n$y)j(Z?EkeB2QFn!6wN8qu54Yyd2hWtQe5f0DUwe_X@j2rbD`V~*pej!n*^x_FPOW5C@C%5aI^
yv+Ea_H2NHfmB$*K2;qLFv`n{4o+Bb?xvE@jJ)b9mi9P@7nVk1bNJgzi}5cec2KA_ZtC3$v|4*#*b_MkBYo*>(R#T{9aDdFA=Ku<6W?6M%5CiewrALLEhP4G
zghh%TK&E#A09suL@hv!(^wFJ;GgGXMl&ZS@<X${3*FWUTiU7%eipnuz*x6taCZC3#a|Q|AF3;3H<oICSwruZEFGt<7=2!fP<~;%<-R0XKk!<J*wu(ELd7~e
Sh9`0roL7K+$?dXgN3q9&AKQc136wJv##9ct6g)LL$i!^nWYi(Vt^`Kv1Nl^QlqJX!)(=sAf8p+JEQy$>uBD2$R)*@3N5(iM*ob9t`CHm@sBS&0`Com7+&U1
2CRKFh2T)h73sjiUnVS5nwdE4311_({|m3Vm1wo`U(26YNSUVn$v$dqL1N#@r#R{kZc3?0E@%c_VpOGGvbpN<{gBncHd#+j<*{$g*H}U%pW{Ra@&^|MdFUST
+Pp#Buw7Y<9IH0f`E!-?wkS$KsMe9~CJma&lPLl}1mx~qa%QLpO&JbK&<l?hWg>p)K$%zUo$F_&>o)vAtWp|i4bl@g4S#0#<r+(946<3EL6wakTc>0pfCSA$
){V$$NW|J*zC&}Wn8dl}#m`D>rWnu!3#C^{cFAA>vV$af6V|VW_;)<7V0?E!akIu$fW*m*y$EA$zq`GYPSj348S5_$Lh0~mMyl4I*lXaZV;9`=gHdQ>N48b4
kh9_;ARz_hhG9=dx*{AUt7>%$?{u>B#|Xw6w8G;XjIjT1|5pc4Y8e9j82``;L^Iy#s$s&X^ZRYV->GVeTHEMYAmn%TXK0LtUFPoMr5#0u?vkLmTF?-<SR{31
_-@e4|4EE7jD@hIN1sqCv<T3ASDGLm%4*%RApgPap+uPDimn`e9M98-QpE#25o1b^s(XQmby~w4se>6do5Wz3>d;<cr01068#sPMd&0xLpxyT5%Yx0PS4VXJ
TC?hgD86))4xj`rqRM+WZ2xP07qa%#lf~Ie!`kT`62}Nqpqi@BkJHth)?!seWDbU8%<nf)Ml~~>l+=67xk<l5vA^5I4H2Li?3*%c<Rw()`$2`-X4P}SY8ASD
xIl5b69#mDL30NzFh-{}n6PX3Ts411m=ZRfz$rJ!Ch`Uo)a-~VGZo!Z*_WAvOpxZW$zn;c3@SCHt*jywRdFLQ_RK1=sDg)yC`X>HYd>QhGSq2%tH6=e(z6)Y
mYpJ6-Bd0|yI6!g)j`5az+XvA34crd3c;-D7&yRbxf#j$KMS7y#K^t<WYMA==GGRm4}ZXUTq>!@7s2WempE@qJOx55LLp<&8TbHK=%H-995NWNR$&m=+d@0d
B?d9)B{Hj-FhCHP$QlO`#|z8df~rPnb#Fi5$-B>U1G7^M4e1ny|I~sKxySSo=$@6z=P<Shk5Hqn#A0Hu1G?}Wi5&LrWc$&A_VG{_-svYNG(W2X$J+zWRVHta
gTP)+66EH<xp&OLNe@X+6-9Lh({n?*i^T#~)O7RlYa^nF@AA}kDPRC|JpHZtsDUc%WIZk+j-FdTVv4&8yP3a=^QtetV?b$`HNoD3)ZyQy0x{I`5M=QA_=2B-
FhCkv(idEH@+5LW0i@X`LY{id5Ej-K;sOtTIR4RB87A#WBI*m-@4T54Y|Cs;<6jx4`~YWQ0I~-)NOU{{Nn2Q@`7AJlc~?ZwW5f_(2@c-6u*!6eGT;LnJGaSO
fc^DaI==Nrl4b0XGpYYl5b>ktzUWp-xr;5|bs}Evgt;a`cYeVr#70$RbO*Ua{OP@u$A%kNz1uDy^)#Z#PM-6|)dV*{p`<&CE&GU=6V*mw@wbbXXzgDSD9%QC
!NyijdDAIeX2N^)gS{G78aPOQlUSu#^jbyDyEfa=YBo7ei%YK6t^{N)>7qZLmz?{MhkC@$KF_7+tw>6S%)qzVu>aVf2Rma~+JUc_owz7M+^$7+os1}H62JY=
kcoZ$nD(>1j?wOtV?;+m$GKFZQHy(-Kdo*09{Nhg6(lF$Yc;=qpYpHZL2Os>idq<(8)LO9mL#TQJ))~TP_DvgtOj@5D@;=+nG@F>rVNHVyG`%m=)4TR>!<nf
M3e|70$GZA%;*qViwy@)>c_J@I>%bAV}7PSL;1iHXg(bSDx0wqyy5Xc)9f#jyO9%LklWqgN;TOZ5L16r0AFw!p6=t0oSfQ16%I4q^8~$E?BPl@>*P?4Niw+A
Tg1Z+WM=7;12}lBC%BSbwae&*6G?WgN@!GES}%2f7fOY7Q+1(c40%M;_O~lVR#6RV!~7eS$Q3=AqzSP}=Y--9)V83cSG}l8JG%=5?bs>&McD97uGq_$J5_z)
uVB3q0LtFU=w3^F1AurF*rz@0HH7o^hN)gU3Fq!!{?NDo1e^7StY3Yxb&6_K>)`~RmeSa3lV1WVhr;-|+~<NB`IIxVGjC-&xlDMyH6|n^p{@}smna73oBbT{
Em>-=AqKK}BAT6|%XTe3stmR|Qu%F9G=*YPIu3YTXK1?L7U?mlsFN4n4)S?o^)j)vZ2}JOd0p96n#s|NfDC-mZlE?sH|bFBkIx6HO;rGz1^EIaQ0K>$+d#5e
fh(_*+#r%4<3c`qu@o!nLFg1SajoysZ9@37)fls^WK@T*0&S;`D_s@6Ta24ZMm4~WKwj0Rs$8|9m=EQ~0ngwOCM#j)9ik}RP{k<KalhOOWCz@m7<-JrVC$^5
yh>cKGpvlRr!r!>c;~ve4X9semk@;tPOr|U+DXy9G7R-*oPdgprTg~nHSYuLCIr#Vyy{{moEm02a(Tos>?D1=J)Yu&opm0*GeoXC=9L;G29uZu`TF1zCHN|8
Ypj9$D=n<riJ>tW=YaW~>M9H=00JyAO*6|vX_mB{kFQ^?8<DzllDk>4ZBK$>Yej=X37h{zXNnrTm+%IPvgx*x`q}DXcNA28DvZ6K*2+DGUx`_-j(Qg4Eu3l!
;#^xJ=Gpqu6GrHl2(9u^T(*ovQ^D}4xUZh5EV*$^;EbeLlSylZHsK-s8?(E4XvmWaGHX#Tf~Utg8s7#_l<ikXnZ!!5t#>Ny`o?(D^L7=0ugGYokVC_FkU_pi
(jj5(i00|?RQ3efn}v_p<IfzCr)JSFs>zA-UcO_ESmg}pb=wthV>l+ZhFC1X-c>vGwbHT<|82-)EDekC!|NpftT$D+djnz!g$ay~UdVoS+4W3q+-$bKl)VX9
c`4y-h769Q`)l#ZVev}-&XJ8qH;8Y1LL^iponM^OK8t9?0gSc1B)~i2Ld-$lW?PJCX*polBGG*PeTn(WZE|8pc?13yl94a!e}{iw`+@G=pQ>+b1@hvoLbz)8
E2-ThfnYr;XQPM{6mD~H4r7`y5Tt~))9kenf_Iyli)>64{hr}c#Ypx)!aJikkfgDsQrQ7!Etm*IB)mt{LA(l7T@R}VlnHIRZqkuI2ua5J?JZ$V`LOagNdiK8
^KQ_fhCEKH!E$N_If^tDH{c~CUe;>LA*PUkgGhWL>fN}sk3FVk3DHGylxe<Gk!!<r+qm=B?)2CKEQ`(XA_MFx>t=dpHY^f>=Z4}GYJgsC;~id$FB7lD1k#Rm
9<s~5=-PEKeDoqRMfqjxO>aYMSp%Wk@%vuAXYFaMz-Q})NHem-x!CfEv)HF`=RK&a<VqZKM2saJT{LbHjws$;e%PKRPz0PPLw^oX*i{6gs<X7+tG!0iaumAz
evNQE<|r{aa(p+{7olaQ;YOwD3L=1AM*;>|77&&sHDvcI{6l)*ei7$#)q|yoBk69fYX=2y$<ff$F?L)qF<e@Y1dJ)5ZQAC5PLgogV6f~N%F_@|>hl-X$$0Lu
e&}RcIdU4Qq%u^&a&Vq?B+e92s9e`q)6DPR6!&ppc_o{gFLVzub5l5heYt}C0iGDWV?h?9d^wzUTgw|X1aN3vt}vrVilDdC3g?G`6A>%^QCw1x?2p_mV=pmC
M<=KQhn;D-e&t>?*_%Xfz8Ma#aG%0~;A&~;Ywn4uS-Kz+c=8{N^Z>xvWks=$C|0&#6Ox@j4H@hTRL(+#cn@h-`&qPf`9-z++bkzMFiZCdpbA{@*j?$<MGVHA
;8+FZFX01v+4CPpi83e2X^1U+veYuhQ#4xk9_ib6^84ThLWA6EMf^$4$k$XI52QAQw`D!mn57Yg)q(IJBzC|92{u;apX=mYfFaVpPN|aH9PudtlH&v{Dp-H2
lp@d05Z>-BCLgML%e?I)isSJ_!YuGMaD1}`Xr%!fI<CF&D!9r{aVrnG)?uIMSaPGXV#)(7AYb3JRJyJx{Ro_akiCDc)9~ZQ>6b*TJ|RZ2oj6$Sa}|nJc%*LE
-SQ*a)vIWi0Cdumr|0(RZ`?`E?|Q$lfeGU_&Q<x};_wMj@Eq2O`43<tP5&L!{9;mYC3fLv<T&}U|1(rbnuaein2Aw(8+pZe8jRzT_q@O`!`RG2UEhS&2}a1|
eg%kB0M}->y*e%6dISlC*LJU$hl6mhQY)gqPw2IIIhF^nS7VD4lV?o?CyP=OF8&O85sS5a;^9Ie2C`Xfpy|!HIh@}{N~+hTAq}KPC6XZ2EY46b@H`6LQIUA~
5YiW1zwYaSZO(gn2jNL@H9KI6RsCJ;vY{au^^-NHj8RlU5CLkelY6}E5K)NHbAaq%M#*9DS<z7qswC>b`N9Jga4L^&v7AjMt!Q!~Cqw_yXJx}DC%}$Whnj~-
{{w9N$4rkyyl{}lEghy3#?fu-i6mx&0Oqu~muk6cr`Gk*_9kOS5qW8w)qSJQZ$1;FWl`dIogvwXNv_Sp(~?2DY(U#UC6FW(@jf_q3FqP6V+Y~GAz3EI5Nsub
UmqCZ9&md!yxU+cN;~uckHN$%_YNEIDB(U|PyZf}vr_M|CF!u=W)+Bqw-g_F5nbw%UkCyRO5Y{zNFJDx1|qz0LoSdbtqftr<fYk4J)~>5iY8!7im`&W9-_UC
eN$p}*z|2aQ6usk9a-Q&5bC(32glXp_bVSF8K`>-;I`_E45OCgb=w=UA_>^(^$I4KL~gjhyIt<tW-RvCAJ}3V9CF;f7VY|HoWmyq>Vr<9I3)tYqfWzj5lMaA
W)YRKj&yydeL5~dKZW>pqNcZSE0$B7z?7CON0y}2ZbQn2O@Ui%|GeqvJEV*{a2kS~WEd84G?oIsbp2QMQ*=L=949e*^iWVme;+8x;q|;bd}ylkm<<khDLaAT
ZaX$$srHSz6EQ0!@3x2rJs&o&4l>;y1&rVpBJx-yRdzO5y#F|(&kyg@3F<;wvW&VcN|Z42fT+CO2pcDh>`{l*EzeVwSL01@xeQwMQv0>FCC1$f3qF>H5D`Kj
WD>;yU&u!|`GNqe&Fl)_BQ)&+bhXKxqB2e^1_9+Yyna-u0{0Pl<<ONv%}r}V1usA{KPe1QFra*DDu*niFZvtMaeQk;vful5zZR?o6cmIyTe|NSZWf?1NG@ZF
oG`R!Gm`q_HJ|k^|94YwI-<$JBjs{ANWQY|Ad*{9nV8mkUIwE*o2Xx`=V}h$UEJFq&spN}ei85I@sYCv0-~x5MiSJ$(?v^i<1hnUhjvG$Nql1#)n^FNhPEmS
<kF<yrw-Vy*w&6T&CRx7xbk#iSrmtUKDud@2JAx(7j|X~&(Sm19w(mH{daJNn!r^@m-931xv&m7X~tITgVdM9dsQB0JzW`#K;-QsjA-$xn=p(ecD(0H%*@3~
8N*r|T!0#0zC{3ASx{B?0%`3_J1(`T03>*IOyrZ?WJ8>SI?aIot^G{jFDHUCy|oGcUzN`5pl3Ke*AyOI8!}pQKHx%+i{<E*Eiy1y+va_8q%eMgi?~=2^N*F?
hDTXp8JDs@jbZ4(m{jVynLk&qm&J%`EZ9PS%EIW^y_|oENlxSY_ptC!6K~xeLPv@bsf;K!XzA9m$qa_JuK=?UHWiZ=5?108JAU!<E)aThk~4X!1?c$)N9D}t
b9FGU(mQx8b3#A01+`1%leU{Q7z#tEOea7?`H7XY&`7oh7_y9LV|}@nly08Q^aKN9mN^>qgK72$K0Xw&C)2J(8S3x*_<HYo&2mH44T0Kuy?#U!f0L>sE9S;9
{dxLfe))H^tLP{t89;^Z;HpAS_J{2L$J)<@*DLL;91-u_JWXZaKu!gY&&XVZXFR<&VanK9N~-*P`9Uzxkp$vpL8t0&3R54W;#Y><m30CxdE8GRo3SRdWvsaj
%!Ji7ot?`UefJ|sb@m$0fKsfez?ThNDnj7MpjtkWjA<sp{|cE5Pko?CTwby+1wK-eCw_-)^cvyMKHGH#UOdbjDDMAa6Ey5~{)GG&hxwMVoWNRHyFD1P!ef~)
I;w<3ZJ%OBgF=baGB>nr%T{ittM5O&r;_vkR|;q?H}bx*Adp01UV_nA67;_>oZ?enu7(O_fn(=CJwQeY4D~V7XI&{2Get$`(jfrRlH`o|46<2pyHo8ES3DEu
sqLR$?YnBGfOjK+ACRa<0Q#(y^L2jRpkohkI_Sd=vtSXnK@JE^dd<uf`p(OgYMXtogin~+s0f07u2XVYHP@iO_ok!EZ&Gnxn^x=9%fM1t3HRt3ywtTXV0p#s
YH#i?UfW7DG_wx%G)=j|Pc{hC&WBDQYuFU-v-=^@E)eT7LTOA*$UmqsNea+<Z2x4UqvYlWY|ERlsArTrq|nPc$?gMTvIJZ!*Crh}N6y0IyK)NNhh&Y=otR@W
lgCzHGsx^lP92Oo#_=XsI%<fq6^Qx5=k2bolTnjqg;LjAqD>x`I0X*0h-3o_lb^nmsABi7(E5Eu;*V^f)jkv@;UeiXu13Ok!qSq(1q)>Df-|BtY25nm0;@n%
QJ~x(Z`KJQ=1XP!*!xdzN{gFi+4_v7SO7Nb@+~@)YDU2lV(zx-SLKY};~zR|**)(b0JrcW+4o4hRB3-Sl_~Gw&)^s*RX3IgFkd!&7k&$m`Dg;Ph)5=pULBDe
x<$;plr9=F_-6q_46cZQ7-;V9ADKbd5ya#{rj22&*;~JGdC?(!dhI7vv28)S-hEK$<xVM~V1pU+%ETG2G=V!>-UwUm#iosM75hIknMgMtgma}i>!BfFnvI9t
#2P6ou(S^!|E$WRO~G4Vg22*<Tvzvkb(oN@_DKoAq+3fb?{?Zck)*;A$B5PjW(P(UXPwZBe(g5{z7nCGHcvL`Qg3bOWcOFzmLkZl)>bx;NvQ1YW*Ye$Yl#I!
QCNI$F;ON@J*Yjp(Oh#z$Nv~TapEnN8Wgr0;!sm4Bw}wja!T;FRM8BI0JFb8WL2><8H-uH_O9(EqAJyirym>fc*~-sLFrd8XWtyP+}ep5nD1_%mwM`hF*l#5
d5m0H_SZJ0re^b=Pa8ReRVoSy3MVshXaT^+w9F!z`{k==v-#N&$nRZY*)bbt-*{SrI@Y6VtJN0B%Llhp2I|Q%&|&RnUGUIu6*9+@@Cyb_1gD{aDYF?hyO4v=
#W#uJYe6$!<$RReXfZEz6aB!z)Fj{4mp&7*5azNNSwhQJ$c2bC+-U%tCpsmEaHfDHB!BjaVv$!YX0i`#|6<~M^Hk;QxGxF;-Xl?s|3I20!|}Tr2toyGmlr+I
7!dmR&;N~83<row`pmOHIDwH0C<0Xl=_^MnK=dE$h?i^T>Z&+1Zc1(}m14RRtkeGui)(1&wb?*Gjl#jQ7XCX3sFvtxF)>$bF_iEtT%BpRXc~<#rT@_^eL8vl
G9hS0s9P@rFFJv}4=1dOgDdj>=jj%94JDcpV$`#t?VYqsVMg?y%0XWTYeR|8#VqophU&BQU&<~UqUDgSd+7uM*5sLE+C80^+}Rq3<*%{hzD9c4bG<}3(FFWA
!ks@!bFXv_?RCh^V?~MJ>r+yBo+-aXYJ@(FB09RF*rrxO`iB*%E~LZdjJ5NI!Cc+J4cFDUKYKTM4iPy?nEZ$geF*$xP{5Ns@^6o98A_^h9`T$1+tLRYdkc`g
SwQdn2YC%*4$<08m6$nT;G6v$mA4_6;S}E7qu!&~jdk0}I|+JP5eo`$1kaZMUXcPO8m>TJ6HM**XInJCCNCn8BC4Eup#?4i765FB!CsAT=d)_Cqb0J(y$JvG
S^YH2erM%QZ2!tk!6=*`T?x1gPR`WDZMOEt&bC(bnvwQ!`gHFsDQei9;&Rz-Hn7pQ7DDzOnTw({ALv1Vf@AtI$sXHV48O$y;qkcUHnmVZzuW$98nZBOGEofR
K~DL@>fm1uL%2Lg_D^z#5Ei?#_frySj1qcq^hd<}zxH=5o-&t^DY<f|UB5%=_d<j+L%2U_jJ)Wb6GlqbS({D^hg#=?HXwrE@8&vbiA8=Ss4VLLcBR4zD#CUP
gw}XsMP?rX<lru~iON>x8pP5cJ+v;MYyHBimXWPVQ*{?acvm=%i7D#)uta=LJ<3X{Xv+P&G;b-0IvP9#3;l(r;+>1aLE?{0l&2P(@{rvG6O}8b%1K6WM4`n^
ZFDWq2ZhfT^Z?%|)<(&aK_v&_^X#nULpvKa6vxdRdXAC1R`sB(W+*^51)cq9mlP|#K7B}xJD0R^clJl{8$VkcQ)x)Q5*82qF{FDX;)i%rS&FXAQ0S*(y0Bqs
N^BMOv8V~YMQ~jwW|5kRg8P$(sfM7mOGd4+c|g=5$#ftsuZ#+&t<A3vh?tTr={AR%dtt6jz*8J+T2$=*g&b%M7+jihX=TAP2jr#V^XVi-VEW)n`c;skCS5%V
Oq_uy{UgG(3CK4opu&e-{^@ZMk{u6-?W{Ey=DdUW8#Hw`s_XmQ2L`u!L10*e6I(CXnWm2PzYhQ!Bts_L7&;lQn`JBEh)6w(Yh88776#~>^BS%frGfr*^f{Lp
IhCck&hieVRY$f14^yTp=M0$#C~-^I1@DYY`sxDQ8=X18sM|+fkDHXo(cxF(nlr&p6P2f<2xv{gw;P-@D8iHvj=l;4W=TSLpU;`nl6*<fATZ^Jr>o0!g%wZ@
Il9`GRmmL59Q!HhP17zupH&v76ZGC@5m%>Ng6si3w|q{@aVF#R+eB_Eh=f|W=HGA!VVZoBxc%$B*WJ^`GbKx$1~$0|SIgc)5Hu(K+SYahN*3_vX5A3*@Oa&v
5GFPr4g#YlFwQf_C{6nk!!3#K0H5++WNu)cIP;W=hJMA1-=ck4Nk|_e9&;I^WS$0}M)%NNdpoST)l)|zY>*gsN;8FFG1<9>f*ymEYij<w1F^^Q`J5k7f=??W
^?EsxJ9Djg&k%L1znn(g)N;bqV9!;Vgfu?*7K%hF7U*j@GaOK(*-GhaSjET0wcDWaYPvjEiE7d1Ej2#|t5+7rY*H{}^*{oK4BWlu<iIjb!S2bq%_HtRX%w3e
)OAEG6R;OGKQro^=5SlK$dA$Im|0uwnLoWws3zh&!;RQkIm}T^0?Pr{oQ`cX7VM>I|Eg?-BG<t2Tp}!u<-(HG_4+Rqt{ihBjlq9&{eZv_UChx3zk8+^VQ;@6
VPU7Jxd};J+M;wwpdTx_gW<<55Y_BbR}H_xiG|oEDJ>Ow7ecKF4UcHJil`%UW>Kbk8;tY-@M+xf(vXS*paC?=J0EW!s7BBKnZesGcH$}OtP4@Yea{&GW{_~B
$IZ$bSPdlRh}Q<Ej(iz!hHchJ+rg*EL2pKavO`>OD(^`D(At~$uwQgC@y>S>@tn9LX~U`Iv_Pwt+(qs6HJl}`IVuXl$jgfo=jr?&15>}<q~CbVHusa|LbiKF
`be(jTd`>NQw?AseL41>c9WDFIOF_GZETq6DLJb;f;ddn?gE}WV^ZtCXji}jRAzt_mt!A?_?@26FQ}W;{8ibVLYoIa+xN8zA4Ibce-+3otKV1&^*npivG=*>
cA}^nt|ilPxkdyLm`%#0om@nd*IET(RI*2OS<`jO1?RHDZr+6=D$~EL0LY5^_bm<9j@NP0Jc=h0AncI4MblkCP{AUnF1KqOaQXBG890A+7Wv0X*d|S3sGZkE
M2R1sF?RRNiujRFD<U1#Op~+}_qolaxS_U@hI$2P<pJ;9zC*s>T0u*o2nmS!7BGSSxvm$gn<Jci+H^}R{Z$`G*pws7?p`CCTd)>z@g_=QpBY{TDTlZF&h~4A
(K-ignU|+YU<@aL;|H8WhiRFBggoS4eldCs<<>+UgP>7SS3C@pA#t~j!f+EM|9KOAyO!69AduQQ&vAem;^X=B3)Un?<YT1(i8vvFt|V&6vm_(>HX{`7imRp4
`QB;J%!Q)daJ74py8yt$a{qm8@O?QCM&fH9@_nbwnoKqoIuQ88_8_;q+H>$^5q1qBiD0*a^iSO*S~!e$s|R9gAt8zwGz2e)MItLr+(lE;JTf0eIN(Z)00+$Q
O&v!D<#tb2E3eRf{|6V|u=T)=b>*@c^3sapPpzL;$K-yqKyl;ts{i1?ndDD1>X}SfE&&}d)Z@Yi4^?9b7}4qPzUvShxLM&EHD?miRRpDka#AU2`S||;B`zDW
;)+z_u1tCllZtfI$^G;(gP|Zlk;T(QsGf~a1~TiS(9~S6zsrI94-aF;&T@#tmO~q$>P22dHI!r#TL0zN*3d|1>K2z<h%|rU@uy2bl;jT)HN}Xen^0OC!Ibpt
lYoW<RfX5%Zm|k1U4T(Q2X_Jc9vV*uD=d@><BQu{Z*aCG<<v5IQFRPN@g<<S$Yf0KjFPaE`xr5`qEc9?37cLiocw~XI})9I1NkwnZQ=cP3s2*3cW<DqJX{$z
e>X%<d3xU|Q2;27nR|cWPf@+^kJd#!Uk7P%9u}>nrqz)k=m;h8YfbKhDwliakGj@3>|(P(L<4!pCi%zP?D;NKz!(}XP!n)$`=#+AJ$s%1ylX$^u9EfsT*xIm
kW5uFxU?9Hy8<p6N}^7wv6g8&NixMUHV_P?G~NQX<>I|qrTI%o%Xr`Q>~bSnr&4jXV|I4UV*o05qdpo2=!U4o?;%_@>*g!f!_AwuM*D9`yTU1kRifcGa+lPG
@*8qIdf@oluWrFXP;W1?A{;_@S!St{V)RDig?c=V$=@QgS#%a9ZD1Hxn!5cr`9m~380a}jl<m<kzGPb0#PYTSaeJ`WpB{*4x@OcUO4Z^He=o#gveujaZa?xw
7`gm)AwnFs$+!9<z2w?!I`2drnx?pm@$oB^sz1g*yi$OCamOoU^#HF^9IZob0+{p3*lz5aUlg#e^focE&Ty8oBEl2%^09*eogvKg`rqHDz&IO=%Hu5x1?$4n
W8zKb{eaRp0cL>F=WMR<iYa7q8ug<kgqt2Zdlr%|6Vmc$m+n<20uGbua|oe=B_RZK6vw58%BtvGcZ#9XubUQVg_N(yPRqs86X(O$c5t;3J(wh#&s2&_yw={=
Ra_PSWUoGtC)x&VYR;1}7l<Z(*BjVUNJ<JE%2X_r#pMsbg$4whFXCrF5n<BJVckf}9c*Y>7yFSp+eVSb$AD#I_Z=W0P|dxz=xCvLqh=NCt!#pWiH*~<27p~?
r>~ka*Il!aqksSHglDfxLtHPw6jp8w$d5(>p^{z1N<Cecv=V|9yYJ84_nw7PNub@(kWULoykU7=K9Pp3D^C7&_<D0x^TD5RcHaCggyQtm@#yXAdtRGEj!Xdy
4J@b0&idMyi#9FDIKn?bnRF3t^LXPtf}tAF-Oa4AL17A{eH?h1CstA``rt+I)2+IO*{eximi7(w&I!Eg4Zn-5c$FC5p)0p^3$D#*_!n%6YeEos#)_=;i?!5`
#jHXGyM|$%t^7^ekVK6&>~;p^PGG>D5WpN|2<Pt9O}rlrx?d{5Fx*UkA-+m^WUe)GLLl6X&i~3lFr;f^enlLgkP#FvceL8<rMY?$J4IHo8bcPPw&93F`5(XQ
JdgQva0Vw!rxABL!*|Wb@S*R7kyC(-A!9k-S8@YWY`yj<><>N}ax`?6=2u@xfeTLJH_Q<M48#db<ieE6X2eo3)|zqPV%#jYkWCoJ?H6Hn>!-Xz{zGn>zJdtp
;Z-9}E_VVwW^SEE0J3>`#R%!E;x2(y3t3%{(Nri0TK>)YR4sq%{&wz;mOk}cWXo-r*?C-m<f&%+d4S%%L3N6-whp6kr(I{yo9=JP)}q16q&td@1kkF#AW3oH
#z%Qn-_Mpc3vqz^5|WP8r9gxfrY&;Y8>ErB?%$93e%*b2S`t;A;YSJqaOPs>x2``TGmI~%Xa9rgupj=RJgXZ<=z&w!p43av@CdL@tMOgfMs?Gd6ma1VBWyJl
eJF#){P-B~)kr&N$P@NhleK^bq#SO_NN+c3`GmPwO{PkdnS>pt03K+9!M+=r;&&VF7D9|mRT@`qwyQq;l(}^L!;n@6!$2-9Xk{sZdE{v71!to?(r2*rC%x$F
S+h=fz{8d~sH@Bo#h9NLbJ+D#>(&?RQMjMk4}>K&4@6Ni-Vy}}WnS!*5H(r^;6l+ke(4i1McMlS0v(EI`zsxTlItRYt7b=x#n_!@^Wd6dLFy-#BcAyTXJ!3?
+CboA1V7O{&x)02J(@?FBGoeWv9en>P1`(8^96>gl?*<H7a5AI`E#T<#y1IQZ!_v0BeivQvXfQI^yu$7zruGXtPd;6*-SDZlW(u>D~K*ukl?m|=nFAEfF{=Z
OtHH>rJE>}@3wavFC*P~Jc4Tqb9nFSpNhMm!>Rlo_iEcUYzOUb-v;pRp^q9=n2*^)mz+AE7(r%3a>hDBM&$Sx7*&r2#4FiO9pou)B?|*-z*r7uZxb*`YI#!i
!U69d17rJJ$g#}0Y`<3RH+Vnm4~OEsWM`-Fb{g3eG=Ff8i5IpY=Fr9W^su?g6Uth}0KO9k5f;vxu@74pgXec|0n`skfkW*PaJiVJR=?y^Nr|^_(x++Owg>m9
I&_GEIm;hj5YzNxf0YpJ9+vXMZ!*2YI0U5Ss>xz%nK~*1=(-q(cisN5(NA=B*)`L9gptY39}ac7rOiGfU=lJ{i8JSZZx4SZi=8+5K0Hf%wYHcH408y<Tfr!`
5)zk=NjWMqcxobL&j1@MJG0*6<kaCJBeWfHBa-A%miF9;gxr;$b;nk!zhQ8jP7@mRsGf9mK|tLewi7^)n8$7GIz#}b)chJ%VH-T!n?eG;<MR70`2;O^v+Um{
Vc_|N8zzUvgGy2l#CY7rUOo4+y39Ce7x9jv-l7G3Q{u}9mu!b24y4x=v;jo97-JSwkgR&gI4AfdKU`PCBQ_Z=rOMe{(Uredq?&MH9PYb0yq&s`U7cEajPmmw
tjCu^vn_&-{PC=tTSa8mon=8zjRSnkJ3Rmo*Cbsz!5}=3zlR3NKQihfR%G+JA`QwuEoq!X17iHP7JGPE_U<d(-GO3^Y(oS2VDcpuXln;AyDQ;K3=|v8-&?wv
mU$4jdTh7MzIINb{)dS&rtnGJvxRbRklh08+Edq2i4{a@t#S=!0w5{ijgsaW4W_fKFg_d#>6>hMD*yb?xu3dqPIiRyEVmXx1VZmPSs@x72+E~YZgL_RtR0#3
nuZGCnbR^nvojzVaR%KR`2V^kxnq{^*l@$XiJ9t&-^E<A81EV|-I^rY62#rCwI%S)$=b}z;88B7AR>RMV8C%>hlU+++XBaiAeU~vdtIm9)0<fyKzs^&NzgG4
+>eoS2KF?+*L)7$k)dWmfL@^QI7c{U7ZxrFcaB2@aK7h0KJz*sZHPZWgtDq@$E==ToVW>*TcOYvTH_I_?k*?6ok=<6l!2|t_+Ytbdl^cS!fZpi#RDVcSZAlW
nAHdGGjHPPU+{$shyUdDu`BLaa4kV?xVr0}$oprpEBtf`FGxl$HxXWlo_KfXpjuE21?qTb{`RmzenZDG9smw)9pK`o_3VoMA~~E@8{HK=J11%>&ASEemZR!z
k)HGe^ubs_TG=Q@Qfm9p-NkTp7%)ZV#UiaG-5_d0@%ZA+nh5_4_Q~f%CKVo!BbLyBcdEr&qyTF~4djLg;l{+={vLMBQu^Vte$}3I`6BXDr|wPd-?w*4MOOPX
Yc+P^`npsd{5bTG)h!`*w2b!1SbOr4*p%skh~L$$oQ657frFXgWCFusoI^msr~K7qtl(1jZGX8_m0_~d>(NsXbG#RTY9%##ECG*55Hwi@o#imjO|k%#7@a*p
k;Vo@<E?1ep8%vh+g&5xDQx&vCu7THlO@2wl%|<?5myp2)U*g!oKd`jvJT{ed$qtFi+;U3Wn3RlY5S(-b^E&3sEP+V03h#3R=+Cq<A!UU!s*_-OKb2>Ivl4}
g+sS$;-aUwws5g{elyu`kl7G@V`Up2h9YA0=`}qq_6|v_#uKE&tzO14Ldn=$YblMSh+!o~GRMP#vVL-`!J{VVDz4pl%OcE;U<kd8k@IA_CQgIr>AXE`v8DC(
N0}1>4bWqA+m|6%OlVM{<22G<kg7EUc~XHsn?g@qImO6X#*?2Rp4G4QE`vDMy67`!nRkQ+^G^cvu#tvv;AK&v6DBDTKGRkw9ls<aG*zQ1m45SkYW!YL$fFy7
_6xDfR$RZPJ<a?wdJIHRu6jrnDI+6KmfMR8)<yU%d)W1rd<{JVgrT4BasvBlf?yj?n5R1YLqrq0pK4#lm?BV&)SxHx*YFDXQq4%8O;YcqiQJdKudsVcv<O!k
wfyYaQ}E*E(&O(&_yPAaEJ@lilAHM!x>kb_rIhQSOCFSpBP-02KgcqqHtiswRjb7t_ybC8HmSHE6_*rLVYacgG))lG&O^Lzr(ge7k`%9KObo%OZc{mfcogGt
9NrWYaPS??N_rnWVaWN)dBf<WT?%V01ZyeE%kqJ3B_)x{q^ol0xURbLIz8|QotjJ030ZFmvud<7_JQ~|H8RJk;j9^6LwKS$%h<N1fDJ&bITZzyhGBLRm*Zo$
XPy0c7-mV_-%o|ZE%Xj@oJ;_di^j3bFl<5yH%G7=1{H;p3&aPYehVb9NS!FU5VvXsb35_-vP#1a6)@ijJ+;R$bQk{#LGEh+*=v;Wt|i0*OVB`(XI3l*?!}*Z
hUcG<GrTY?>m-`VS7_G7_xQGAAwS^sNAoDnfjV`2$59g!zvs{E4VA`@Tt@zY0-S_}t}a1UrN-Rq6YQU;Ni6G{PY~rj9A;u6l~}XM7QIUAtg_qyT~uM8XforX
f!tQkLBs<N)I`x<RB!6jX33>qrXWnk;?mfLMS(JTB>_9OhQ;I<r@&2a+jVQHI6)$j(Igh8Iuh6RfWwg8LqnuVMXgjVNUZG^fFKAr(*33Z(6Q}&%7L1O-PcFq
9#Q_-2Q_~}Gd|7cQ^2+tKqhq~L<v&2Lc&xsEU~<u%4J2o314Tx1JPHh`q?H`S=Y}`LasHt(m^^O_3yTZxIS#6#jfq>Dg?Q%utIian|8-<u(te<iC>c~d`Icr
O4+@y<zi>w_r*_hh>mMfvg{}(^c@JK7gmCZbG>!}Wt-ix&@h~tnP*2gj4%@sFs&lJok~F@Ld;;L#<uSz2xvf)?mk5)b1l|ZbkY}qS}^>5zoUIbgmy>^3p4A)
fL3WRAihVZv*%;Xp%$D=@iL~Q^KC9=QnTQqYLR-4O{!BC3*V%{|LvjD1p!FfHEsEQQj8+$G9$~sgBx6l_D&Uflm<SPg+n)Q@5?AjWy3btm7kyZ@g+C9^V+uO
@j>2NRDcQTM#-q8pfrWgGo%e~U8x|wYwMNHgn~eCTnch$5a3-}-el<S@rpe($!N3Zc^?pRkEi%|^}sDnp1*B(o(v(j<`_?qd_21&^CJ0SkN>by`z|%qS_Y~-
snIQImBCn1IT}L}H}+H6`0%n5h*oS>elTm(nV>khfqoX1C84c)3y?Fv&jcCUiVCky6%$@03<K(rZd%4d)lukHb!F0aM)B;8(5gq92RUxEbUllX$tqls6TncE
(=w!+JOC<#-t>A9(TueQY1hREvAJgm{LSOa+m*S1mIxM#l`Le%(6MY}-Ohs{Y>mrU-B(j?{@cN*@E{kI4)1WZZ^rz^xT9>K^s%I&hKl0}zIF}#F;X5fZ<vZ@
0V7VgV97^#y`=nH#d+z)JX1(t`9X_}LgCsZrhV5(JDEy5%qXbF6p6NB%{cRKbTp$}lQ)>!i2{Di-HA~lCMI(;f#cVK>WMY<0A~3mk@>*>?<|b-a8g4InwSOH
QT>yPvaqQmO}Uad4zqVvyo*U$(xIy>8s?f1&{>nP5J*<@oeV*K*f{d@J)DW>NgSAAT@2YXAbuS1j>_wovzFqfZok$rqM*8WW<9b)cYbN*cvM3Q%Fd|xqFx11
#Buej@czh%PLlTaPsVJ&w~~{&f!m}IG!3?{=F08bkYTWcL6E#Y)&DwI+(XDTppR~fa~UAAcjnv<T?Cn}^U_PV5fiD$H&)-%Jo2ut)kATg-j@POZ4r}&vu`9c
1uF55f{K48GCjOl)xWQJKIib_CGe7S;&m>C5AUceG-_*Fc)5CEd|R1k?e-&m|8d`SQ*X~SnoZ3B9ngm~81rT^+_MntlP9$0xi8>qM;i&=+l5K=&Qvzf*zELC
ZWkvs2buqdECSR&0ZtLTR0tby;y&j+Mu2=n+Iint86QLCmGuLa=sf%IbYC*mDU+Yxb`=D^73EPt&}$a{k&~}qmd(qfAFUzMy*f`B$Dqm3u%9OT7AJw1EgtZx
#hA>;Pw}GQRg&_Ofy(hOj}iNy<3q=r2Pt9+$QC7+2MBe_o80TQcWY9fQ}Kq6(64t1qYK2VvF2clou%2-V6V6vc&k>m0Cb5~8|km@)?^Ica|01^k(WS}*u5%y
60A8%3I-B)BWEGX3sP%!C+H6Yue*~2U)!;2!KSb5=MaryM*B>Ut0v!)L-xO|_FNFA8!H^iJy3WEf4o`Ssh$-k=>3wue`2*|c^*hq`Q3uc;w1T&<F1RWBjHS_
up%XbZOz~ydl>VWp80dcG)5Q}^2bCmM^mY36Oi!;o$b_#(44h(F#Y-N2G5I=*(q3i0>i||72OeAydxVShZC)UnERj<KpWog1iDlVgd8~309!|4T#+>mH*e$;
;68~;w_%L|T}~K)s$kGLIC1g;4J%762=hX+>x@AA1*>>q=ds)O>|;k-8#6=!iMmJM92bxwDOSu9{{q+5pTY_shmDf~%o~MTYgNW!`khnTtgp(S9nqcyXxCXQ
`wwfE&)|LbO+O8-uPdfSsRYJjSbIq#_*Rq+pjFoo)oFtsFWt>{J>exhyr%7@P!$=CB45aF^Z6+4;WAZrbEE0wwVNdURzDeHQq*|$?mUoK-%6x}c_RafJ1z8O
|8aOMO);23%W%oCPtBNGH5k9K_M>U2zf2V635uKHnb;-FXmDvdeq;AhN=Clr8)#JM-n2y8w{_4eBTIiR8_SCqa!BiH*~(dim7LNVDZP7842yCuWK~`Dzecr<
A`9$r8beD9HcD@e$$LwmEe1M#-4j_6MW0T8kP<>RSZA+}nH#ZI^ukrAxiKIr`+ZYp3OlcAc~Ym?Fg;aw&du&lqWQJ(lrOMMdvyqbg$YoY5wFMx<bWY8BQ7>K
0zgySP${A{fTzoyVI?c+n`%uCgnGa<ovf!--Ds9~im?zFQ1J|{8UgpYzw9^kR)h24K~FhKjwl~mO<GBa9J#?q-PB7kwi#;+up5YzXJf@-=@{4!k5;O;69&PO
i;8uvEu4}lefP$bCb6XrU&9?V!DnChy1!yxjz6Vt()0spkoA(XVCFSsH`GNMi&daceqoH8WiaKA@i=ni=^J;$jCc^%)|GM6(g?4XMFjcJFXc8I;Pl1@<Q&!-
M}XdS=ZV|ZXzd5%u;>OZ{=m4|JBgUI%yW6AW|y_9llzV*sR6H@1TjlJ%0NZr_;W(U?tcmcr|H2lE9<`wTonBx-c^?2#@y&c#((dl$#KjbFK8We$Yp7vUcL3=
VeIqx;MHae1)6!U%J3Z~RflYy5Jj0Hs!!u)ky+V$Xtf=E-N&{Q{~4pS`n=_u(PQFMQjQm{&IX5*(36G%ZximG`h-waBF_O9B*+9GDj=`QDtBOO4-R|jU`#Wz
IP*!{o_^M@32>nb$Q7=kr_^x&>6<3_gynJqv`HzptTGf7y(rXY+D%jx1$HL12D@NdE=f+UBbY9#GtC~(Y=iVixC6WjRP{||sTchgtPF(uh?ah(kIZL7F`)cn
g(Ii=5|$noK<WdII!>PF*w29iC#W8CE>r;GUr_G0g|x<n)$S|Eu3gl9Pt{BoAO0aV55U$2QUwsxOmnVEW~?w)ZmrT)K@Q*n;`nX3A|z)4nSXH`di1*{Jwa58
Nwfg+HhAM=!6vtfeds5nl2`{6%W>yNP(0A1j6G2CyKs9%MEI=M%;2;rIsTR1zH5RRC=#JFja&q;Fq5|xYv2;H>h>jm*ud2r^C|VCysI2nU#X-<OY-!GD1(BG
9{{kM%zF>VZOwh91-lK5H5!YmuC87Z8_J{ZSpXTfvlaoU?>j3=G}D%t3I-*358&*ogzVE%<XQq4t1Hgg7c(r!b`g)|Gv*AF5p2ixM%+}trfgtBazqDGncvSV
%+8rwqlifm0v6guPt;kocWNjc7J%J++d^}LU>ew1-!u(hH^vtXiMG}9fX4EJK$FvbD5s_log&*skf$fNVx(C%AEmsVwL)-5J^KGQd3^JaPA1}FJcG;BD2^ZG
6KTzwXo~i+x5?mOf0GZ7v3cL`^&_n~4hUPVK=-5LFzrQ?+i+)=WqTo#-TklWZ>9ozJ}SB{DGN_pEVfGe*$VK?TJ53W!HPZp&66c#5qpZE15JV>;EIyH!GJlU
>_pI3-fgx-OP8g)xCmm}B{WNe114m_wrAsN6d3qXW}Ie|>-qGYat^tzZrs=e%TB1Jq4znvi~<b@>@q217_P~8e?A_*m3QHiYr8}z=&||IN0~;d;#Ouj)lq;w
bYCOI#kuGD0=fjsBzD(s6VQi>0fzS#>=~@t8d_;28FgL%rBFvI8~e|Uo3y8KxeRNaf*W>!4doj*bwcsdi}#YIMB!!Z(sK~$6Yb#z6SZ-It!e1BvY-_q4|W{7
>+^?JCr{ej#xu~Ftp6+!yA)EWo+yVCqG22Rk|X984uLzbZRlvtH?Bv)k(}A=rWamp7u=6B{>~cP?N=4|3#)mu7V?cg1k#3RWREw%9yJK>^Hnfy2f}3il_wGD
do6O`tY%??>jIZC98AUUVOd{xElq%}M)8Fq1F#mjQv#NgrysC;-NjY$Bu9~p?^jBIPn>uaJvqJFN_kE_u%Z4?^99a;gfy=PM_nF-oJxG;oY1{rSt=%R2}qg|
M1&y<zzVHENyc<X?8F1$icDCjy${|>-J>HWMNeY&JpoG+&U;coaXjr@t{|xds5>GGlBNhi04Px9=|1;TUr?o&Gpwo?fHqy?7kVM*KOBrUAu1t6U2-ZG0s?Se
$5Hr>Oa%2JnJqD3QK=!ZOd#>c$a0;C0X?>H9;ZV)en>>i;3odOZJSlK#6Px+qj<m$4;_J@C7sDKJ^dBRBci;A+ekHcu9)6o>Ie`HdH%k;mf$XNlVl4I?$Y#S
)OE?q>t)K=2w_M5QInGH#}<QSr~Tb!&@Ymg#P~~!@s)u-{KMun8=SL%^*?!mx3>#0VTKr(X$s3D(vzbd=gtfC!2NqLQ46xXOYj%jP7@4JOfEb5{k@}qA=h86
%wvac=<`JvGj+1gVe>fDz>f56sVoQPOa!qU%Sc}iV+)yyY66F%C|B|2aqVj-2~*Ui`~LYVl0e)~__IprkJJDUuN9?J$amEyoT`P!dO@_DPwm7iR^&PBJ2X7T
h-@1aJ={&{M{AWi^|CylB>m@z6qb2=^PSty)1ds~>K?pYg?-7_oIWnS7285lxG)uLg9XJQrjvL&7f0`AI3>=-ckQ({qCV8V7llW~X;KgUluc4P&|dv;`-^SW
Ab{y-xU`1$)zL(X`iOv8<Z%Otkep-gr?XVmK#>M;JD7eafpfC@AIjm0v3i)i)X*MDZ_2;1!mwC@uSe2S)QBYtF*QIL%f6H0g}^K=K<e#YL2H1aBu>mi5LZGd
;i;HHT=_SLZ-sK?wLnKM?WDmthHE>|@H6if?4YIK#=Qc>7MT%+5UX5cuuDGEUgmb}3+i6$skdiH36^xL&|Vfia_im>hC%iGi@^B#!IZOcep=NsK%In{S+ml^
et%wsV-YzBbmtN3wGsT<37MAIVc{={z5M_RxyFsyxRqk*R_JMpR6dU<Iwk#pZ#kC4n%(}y4{8n)`#Apk%I&JB(pBrpPX@1xqM`P!rw->80=Dydl$xRBOhy_Y
vAaiuy?q8*+8c`9!Cx8ED%x!0m2}z-M6TlEG`&J8GVUiOMBFn##us2ord$59r@X2xSH4?&+81D!?wJg+3a~IiGbS)O5M}*|v|>O!<;Jr0`2nHMsG)#lm`T_X
UpOWgcydxu*@D-4t9?@c9g?2u7Jo?MRP&=QRsI`%4g0)2kVCA2k$E6EmT;h_sPe4!;Vug?1Z1ids*P#(@$Q-Zt~qAtm1u10xw%WyIN{$*UY8jOE_65t#MDA~
CWCw-0}l>}0pA;e^F{=4u@w+4LfvhNNwwTZ%^o<o>QU!~yJI=qHdGAhV`CXp)Qn-h!2Ua7<9|2rJr`RcKC`1_?_KrZsmDRD+`C1Qxv!EvxKc4s2=Q!PWL!0B
Z9_yV+bMQd=yxl8D#DUx9IqzsnG+Ff3(n0-wO5X>11gTypfc|T$P5@Z|D`T0elL+e!~$V!%xcn!R3eb>JxTl7{K$yMLcv38A{wX8Isi?AR^<XAt-+jovIK<l
prcR;#`j~sQu`iERheeq{Jdv=_YEJNv&N3oVD8f_fG7*jWIwsvn;obqI+sVz-aOxPJ)ar3uRug8#orFzM-(SZ@LvuY3>&2G_xLXuHkYjDMMOKKB2)2&w;n=c
4vG5vduJEhUkL12A6A9NKl1V&hWW_JCwM7di=7;tSP;z%IE2;(UrF(ufOW%44ptX~#a%-g$YzU;Xz@ZQ-q^((dFc&d)k55IfpotS-E_J;aD)53;;!Ni{f8$`
*+WsBM2H6`xZNG(Ml17~n}tSR26Og)04_9n05KVg$~I(oAfW>rr^@C59RUQlnb4<UzVV*XpCJ5LcXB|n9F#v3A~qeaqYUmUya9VvIqAj|@LE3kL&jq4GUF<c
x<jVcRn4s4&G_WTQiKtK$lx-t(8TNai#_oQEK}N(pL|XcMEWsYZKw3LUsF+4;r7kckq1}VLw_-I;*;=^p~ik)UvHdWN+@C0xQ^)U;X^$-C2)@XJPQL&|71uY
ymZl8ckC+vK0FX+?x#NLq9rsIod=D2s*;sFxpmvO^dD_Y(K9(q@YH%gOg;srXY-V!t>V*`$Y=9?HJKyqxn0mvIPzCE_T@)eeSe)K063wkaMi2dDCG%pDaKXW
kSb<(<~Q~H$Ya243%zjw_qlJ*F)MzvNS>21!oC~4aM1DKMHhBP#fED32UN@o0VIf#{W+4$-6u%qHolK@k3aOStD(*`RTljh&Ydn%^XvV;5~R+P$i`O`@yd;1
d4beL-5HH|+(Z`^u#tI)*LG{}9}tRjr^8q}LDbDFypTYdJI4jHg97DKwgH|8H3y%L>*P=%A?BeQM7qqe`jASOv87(<opa0~8(6MCl4uBPQrSIPfKB~uk9Q%*
%YDJOPrjtCyPt4HSFm^TPu!dfhg$B&oM2{^O)f-nj6q~1)uAqGM*)!&PKAUftj#@q*CBk%ed;fg+@2ksC$;+qBg7`>7D06o@ROO@B~y8)Do(4dIH^Qae|t}|
h_})#<Xhb{)!XXOc*#(g8=ChrKD!BoaCG&RBuKBRNB<EEK)=(Eh*&*af_py1Mq{Q7Zm*TtwmQ(G#LpY>(0Fq6-hZQT-L2G1o_2fjrRIpB9u*vKsC1n%P0lR(
Sos@xpitgt?LP0R-kWQibOqX%BH{U~X7S9KJ22Dm2s<+Qb`EvkXc~7qFAL@kzU|${VnxOC1}pP%$KV`^R-4QxUTuhA!p+&;^ec>eu}?0`KHmT>mWiQ6ez<hS
v-ehRQWL3Vx}nxJqu>$*%~+iTu}|rBZ~;M-K2O~DoUfD5qIcHBmJ#M`%8WZ|-3i3?Q2^gX!u8H=%t{TDYVLKozo1fKs{~zRmCuK(HDUKnZMTCvl)75nlNIUg
*c7I729JT5gW&}1^TCYt=4OJy)p>al^t(hj_o5M9H;B23H&+mdU$r`&=cxt#cq#ivDW1F5`6af>E4T~~Vb9=rEXkjW?KB%=84sPTuwx&<W?f*Zo1<fp$cx|u
x__>vk|_#-;4{=}JRUM+aRe5aaeQE_seEDKJi^+7rbO{e1tBz@+|UAV#t&@IW0CF$4o(JqyaSw+My?FQ)A&ynz`%><*1zczNRL*&Cgn&_czpGU=zmdl2$W+`
aPt}I=J+4vW{!>BAIbc{9(XrF-j85k#dn)Jy1gEIDrP@dl@5xlRT_*}@(nn7G5*OIUP+u$L$p&e_MN8~aAOe2iBjk4d2<F+3adOygA<>k39G{)5Ub=@3O>jr
T@$=RmP8^71>CQMXN<5#!6u&$fT^lE{aEy60C$pOHnYfm^O>V{gM%7+8QAR9@hmZx7dr2CjxJpYVIFWwi7no;S$!u>i&DR1j<V&Vl8F4tR^xGPgnsou<0Vg*
sL|fovVUC&s`Bo;=d#T=Hv=0P1Jrr^1pA-4dt-xUuFw%0TL0klH8NXdikiKB?qgDH&c9!v195CQDLgTaGE;&INvDTp;g@q)p}iN%mL-CPt_Vwbifkved#&vC
-DAIklS;yHADsw~Pc2m**fZ<zzI+-5JxGXW6Atc0S~kQqp=Li8&&&w$Y%qC1SGSK`rY|l?gjJidwr8WJ3VhiP&TwM;!ty{M7{!Dk^MJ)reUnvVeQ>)WA#yNA
_Yug~r1<Djo)F2=c~kv+D@3WFss<UB3HqUhpOLY-1pwd2JsYixpS!6A`EAqDIK%5oVsn&;SYk)TY!HV~ej5|9!(tX<3xs>F9Aac~*{Va39E&zKWP3LJIj2O>
o$-qJEYa%JOMqylzPt5BE{a+wf(3ZA(&iZ7T@B1w&pqtI+18J5VGMUKMDtON6O9n48k6t$|Nk4!f$H%h*W{ZZe@4I)yE++7{2Tb5ffz)$ELtbjWb!2boba}H
lw}zsOz1wO+V4Lw;3MV+p&{xRr#!;EbSJ%P%zeB<y|~H@eyD#BoNi@^80KL-1kjOy*gymkE6-<(td92(L*e~jky?OAW3^G1dc(xJ2JMV+92>jw%*pb>4ascl
9NCsouJ_Wh#Fx-9YL`i;_I%nm0fJH&kyW4vfNA0(A~Qp~^`G_90p<5^3n&kI=HmI;mbtX*ltJ%)h2pUQGLnH23{sfRfVs&OpLT~frESCHRaP^t4BfT*JZxKZ
iiPtP3opFg*`inGCk089+;Q1-<#$ef{~xW*s*K~t;k=0!W1tjA+K+EEmUZAo({wd0)WCb~$r{<qwtr{UoN+=d+7fkyPq#T#q(@4E0_;nQU#CN_^fZjb071Ij
^pdf6n%jUx%x5<7ViI3guqX>B#L5y!(n<Q}6R!hq^WBj=$%>EqdWyF$&W=>?Xzt=VZNj%!Ma3({NnH{9YElp&A(PAsJTuReNEDfxybGMVHw{Im{+e76NioFA
4Z56~rbA&iq2W1%Nr?((<S-g<T*zdQsNZcdbz71jKQ@hBAuDq;=J?z5%Dy%;#Qz+Z#UWowLk_T(&GZi0u1f|gG)lS@lsIz{f#Z1)T4bBs20nLbiXli4{sT7c
j`}y_iU>uGF2iDr{IiS5)h=T*kf3YiO(q5AP@gc)k{<F>EG0(VehU$KK%_3Ap*^zVww53U3b8rytR+W7!CM-DcyT|(py3Q1MVY~4;Yfe5P@wsVYFAW|bea0`
rOGq>Gp(dt^dAK)-tR5jCfAUj=|()Ixe9zE<ImJaq*TpcpbYDkJT$tb66U7cg_9s`1WBlp9tTe?6h+pPvjXbcPGKbK_-GTSC<|kRD_94^$_$ubq7?{fio}rL
U;2BfOjmraofTMah*b#?WKTYdIy*vyss*sPc2_x42QCayUg5TKRAg6LJyClie{gEES7@~I$`EGVyA9jf)IC3q5iFfCQ94y@uwMWE($f`k+u_8eRnb(dj7jzc
@Cv=whbqY7pD(s;f?%@Z=DXg!Pr4-gh8<^fwzU{3@zkM{fpfbGkgMilVQ~i&U4g7cwFF#~s={;15}W+?_+J;Iwm>Iu{M~XIh62ccdxMS~4S%4N?;3Y_N+kxb
;4Z{H$p_|h#S5rO^6%W3LkBO)^%;K1nyA0)9D`j|D0|DKyQiuo-+6Ro+n<}z)E#yYq4%O;;dWXgsNq^6F9MUvUF$T`TEool#t=plIP$2cJ@ed)t-2|6ks^-n
dlu_>-=yzf1n-)KhI{PVq|r)};U`>FqPw(mZZTLU7F!4vm4;S-gjDA0J5>zOSp)f6ygsy(<07Q^3lD~a4m=}=C)0XPA@@p3ugl1mc-*&CB&5h(xm==Of}dyf
?o42+IIImAtHP}N;~jUJ?<JbIJMBkP?+h_yMtPhLwT!mY0Oa!sYpX+|vZEd%ek0u^!^s^|!xh4;++Jz>2HZR3)Z5CBWeo9GP-dOpl{9nRE?r(S=O)GV0_VtS
{ti0v@~C!a^|}IL+*oO3D%<5K4LBA=n1pKPOTA~l|A6wu0kIRudWU+X+KAOGP*44J*#gsr%#{mjR40nhcY5DgChes&QnAQIqwS|{)&38gLCC9n9!ej3!E`V$
P^E#4s6)@wRL7bX`vC1r7a`+2DiepEtny&t;}zomVQ8q2;aPCfAR=)j2xJKlWm7G0>+ULyL#7vXAUUt%a(7$^7TvU*6us?fS^V3-r5j@tLpUhkiel9`d&?C`
I>2*B>#-sP*Z*^9DdH7Ry(`jIWAr=g&ddRH7GbK`_am)bP*q_jVcXKehLN4P*33$VB;-R|h<s(PtiDz1q}JSMms%3Co5Ce$K&`>g9R?`(Lxr>b{+`BMFl0||
5w3CkS$#hS|LZ5FlhteHZ^h>dYvn`K7a~Le<I1powzOs7tE#;0vJoV8DYzm*b}pqJSO}!*j)l-jx4(|%Rq(<{gestPdd*?(2L!)v<-wYsm?~+FFyzm)(wn#_
B~0n@e-Eavz`~Ovi%REufN&jHZhtb&bdbz}Wi2>t)Mf{x6%y~fgX%XIC}+mO=|r@Z*G*GhQu7jt2~g?w;4W*%Z4)|BrBj*&RF&Lk@(!lwbg`Pfn_9{9rzp~;
fj>yjWBq1P1ZN~Pclx-c?YyQnUUnne_<I>jPTMyfS2J^%hw)#}6cA$mlM<&AxBg}&J4m(n>O)7itehq12FUIkG^c-@qJpTh=H(xBH+BxJD7+_5jR_|@q%7$d
_|y&}C5h3-1cdsTN>17j4kUiH3w+)0{iKSni$&&gi{Fl0_{^iM_-SFwk{2BHIA@BsKQ47~(sqifO!NEFF2H%JWux0o2+KNnBRuC~snk)~Tf`<8ote*`o*y#K
EH{UB`gbYYN+m?x5Dou76|9x9d;5EEM(PT5Dtt(W?2CAdZW?d5s-9X1g{ZdTgtA>(fp;Go%pD9at#29kkwX2n3x~%jq-=r%8T{If3MmKFu$Ck)W$W$N%}2X+
I&1DHAnCnzp9`ZaxqTkio#Q0F6RSW<F4>3mo1Z%(aDfxI#$*skL^#Mu4)jbVE+wgA;_OUmI+n$TQ)*m8soW{~*F`inT%$1ffF{s7)>(SvHE<!qaJ;0(=GZvM
5)&wiR$S-6)h_ombp}ht$s*RZ!X@6jrPDUEglxX0SE7)xfsCPJniv0M52gFgVFj4?5Ct;6H!q6fc8cbuqupl##kx-G6N?dFP|!4?!C%^nN<HOBZ$h6BS5ep^
IilkCjw4&7#i1VVhG3)Z-%7qv51*RD>`c|_4*Zsb4_y-k<t=LS|8O1GPt2d?hy}R2&^k?3uOcQA^qVGPmcDc}EpHe)_;RSNH0FUJw3KkQXIQ0$XZYlVK`r>X
tpsOUohjv9+jC9G2h0n4S=IU`8}x+rMI7$FU;YO39x>&<|JVnd00D4+f@D=y;zf1$f#%4#zBseSbY5H=qPeUSr(i@#cY~yPriZ|zFF(g0ZAS^6>JLe^%U~<+
bqH;yrVgAKOw!-KSp%@es_Na9<W0lRX-k;gqa8~{1BfnZCb~<y%ntCW&6fWtmTZudLkH;SYh98~pPM?Gra}6bE2Cl64m=yFbH;0JV*ho%6)SJ;YM($WH>&Dp
p>9QkG5zBGN7Qj7Sgro&yCc^EfBWF~*ES(@17d^jPEB)fyaHF~A{crZEXM_tawL8O0Z2-LI>Jp?EJRXU63i*yRSD{u^!Lf+7CGUGa!<7jtt}IPYmd=MhK>+0
I+?(VdVXoj_^Gb|4kekOIb5A~zlFWMLW3zeMwfQWR&OO`pl9mKm&}7^2V6n^{YHr*MY=w;IdIgZBXicZK7iVvV;yR;*vpGrN(bp64+K!sBOO@ebm~>tc1na%
#|JP0UM%bbGq_JC(%<+dX0TOQT1?^xVLlIES3zFVQy^fLG*lJ8-eusp`&Ej&bG9>pFBYRV`jy%+7b;S)@`Yz{Y?N~;s(LM_8=<}XTYAnkh0o6dZ>srM!vOm|
ucXeB!z7kOpf}v4{20*2TjH@^<e)!u8S}WFQpwTt1FO4)AL?W=D@8RWVQXl8M~YSNqctuzGQ82*41+}Sw>GwO4S39y{a~UK^}V^A{p5e-ks!Y-i;z*d#daaJ
2(L0_dnyrw$`1L-QBGS<q~DO-7bgl)`j}6@*MDiVV(^%OT+mPxHK}nh>=FFNXa42bCVF)(3u{zoa?H_nl<3&q|9&QvTWnxV35C)lvpqP+T)x<Fymi*bSE&hE
#fr4kGl)$I7XT|uN_*}$R-Wq{aEvm>a4;l}jlCZ*8Ai>V7J(YkneRIvE2blI;vEG<!seQfb)-6~Y-f3o5v#9hAi{8>)C;qC(tQe_{65ud{!Ypi6JO7Gt2s(!
@UR0Lp{S2sjD~c3xoIx;Q~$Dy^pIdnAfp2J?gwd9?SGY~WH%>t%bWRZaq@Zz_{e``YD*HtfEBObNr8niWeK|kV58$@C<MY)hwR1hH{%aEhz7CPa63oG9pS~E
VR5}Ya&_i?+F{57xwd0uq;xSYvz9mOf1SlE7N`j?Z~W%JA;E_ZS8ED>YeX}|MEShE4{`TiGR}o*2tWC7HE-E!WY$Th@m90`@SVSST)UqKw#Wu7>}kr_3y8A)
dyFGqpMWCg48~3TMkg^b&5e^V+^hM|^^LzHd~q<)8svmvk`b=F%Q|iy%A<lb4ne}P65Gvc-~fEM;`L-{`9P79P|Wq5jHw?30ZL0LZ%&Ds?2>u0D9@e=x^O&&
=fURf9};MaAQ2;=the4-*D64DM*?is!sxgZh+PX2zkpRGdEdI|efruPV5?Ffwp2{3a)_$$fTr9{2T^W+OK@O5cabF>I@AUI-O@*xKY5aiNMrQAksF!cmiNnB
Zr|i8t~jHH0N&=d3v||&j>oy(>BKUVkOl^ZmJ%GS#YFDU^CgL|1YyZ+_MG20E&1IuLaVvTS3=z_!y&~uJ*g6YM(<)0%GIN+gxyUFQLvMRW+(CcemanoAAbTj
oRs6_(tRr!A(ixdIqK-k4ma{WF*D~yUG~YPmD^O+;+o{9CeJ!^yQlB`m)5$owRy{R3vMU!X_U~lo-ak|g<T+r9yddHl|wBelF$oLS>A@+ip_&T_Aap=_8-VC
?R90@ClCASJyhD_H<TfwK5SB^k0C(rjD?{1jNjxgf+)K3RS8-ieZdZseVXW(if-I+hfy6uHD*?JKBdS4{g_Z0IsU;+PUBdrhF(x9ROM?%i|K-FVsPJy=jOZy
EP;IUGM86WFfj9ASchzzgbCmd$dS-9Qi$`5h@6n}4lLG4S}~MaFDvbkn_UI6Mqab?X7)-8SDPF-q^Nf$@+>R9?dT1I>F5(npZ+Trz)S;vg*GDGGIC2B-AEpQ
8C?1Tz4}j?#A&&v_XPO}bE$3Mk?3t$T8X@ju2iz;(HpvlW)PsX2beyd=<^Wy@nqK+apQ@8QqszLTh*;-+EJ!$6-wD?^&Svp#QP@01x0U(L5Io>rzo|l-3T%n
AML6^N=COV?>>}CLzlSdqel`*;#lxGsu=c)PoOCyWEr~Uw;UjWn~&Vv7Y4e_l;uB=niX00q+qebSpVpuq&BFY+jA9eDMDwRuUqri#mtdRkA=;jM*5L)A=_T5
EXR02cxoe2sQc-lY959Wn#I<%z5dYnSpPBAk7}YcRpIfoe}OY6K2#CN!gFUgU{n*qMQE3=?J#V(UJ)3TRWDI?GV7lPb;;hK3zr7;8sf=k3jCMiA(oW4;Hi<%
-)PXDry4joMtD4X5mi}ziAS$}L7<k$ND)IBkd2QuhN=dHHCeY$KKrboyGU_Bn<`LU@M_^cn}r^j!#E?zg*ecQ?+*d0HI~VEtIAvcG$7jN8%OT^G^<yw7@S-y
2{5~L;fvOp-Cth-EWXswho<td67YDnG{B7@o!&)=se^!!Rb2i+13A@$uWq;;2m9<FvOdI6K>-sVAvxn+UF*zhzG%9Lm$Ci}rQQG6ZXm@L42?{Fpr{S%3#bd$
MrHfF4|54KR5t>leS<hEb!KCNLiBEcor642Z1u&_G1G^Sx7d|(b)e&mZgV}Nc>6Na!O5Hekkg0lPfA%g1w@i3l1rt_D&vW;{8-!mzJjr2ms<g9|Dce6G6C_%
%wQa4BV?;@YrRq*s7&_XGKsC9g5c?&9z<IxltEY0wOuh?D)_hMAIS&BE#<z}G`3~b>9BcBh!=P&p!*AJ-}@an)zw3$H#mBdLGSYw<bQWTy3I=%fw#4SLc&7-
H+C|XAzNOA4%Yvwx1`Ok9iLxumNdA0!+QmOa)P;_*e`nWHM(UQpyx_Bg|23l0;gSo5Q6+l&?8FKuGgY#XJyOLy16xZ$D@64rcR;)l4Yy405MFXORboKNnAeS
0d4RtmDd!V8p<TDM-g&ta(;^RbQb7>QljS3yNW-KIJi^t*j`c~?_(n5!H$ekHUQ(Yb#T=o5Qfz6=(gxFRi{sbvj>D|am<a{Kd;z_X#aD~B;*CM8w97uvH{SP
_p*pS5B{RF@!e&AMvjdep9#y>Je!QtpsK$E&0Vu%%zjausTG4)&_&j4t<|@LEFzzcg#sM^o1R8Uu)Hknvf^PIR-Jn^PC=msb2R2zZF(5zlYVP}*hT%_n3Z6L
g<M5q>Hyj$XMM~S*+SPZ%ku;gUE!-|bQdr_v`E>|F?JuSuxo?H*)TZJ)+RDeQ(bNjH(yu;UlO3;jO3l995z(d*;)BDn;v0ID)v*3)LLY%?uMs%$sc1O;vI#U
OV<LIL$}ot{{d&#nUAK3Ujs3OE_qZg#*!eoMUu+zFWwM0`OBC^St=FAUTGqsLx05{9iZ#+`<c!g&KA3cx0iXDcMyP1rk`bXZg(_{+97{HeuRrqeGGpW9MVaM
)}vt}YF_}oZ(ubi?8iJFyf3BK>lSRnpygOvtD)@(p6!uO?1*cw6as##JHHC(Dngxz`>;djPzPQ}2>b(!pww)Ad*BUH%>`sz`IO+5zS)Y2+rzF=WxT(+!+JB&
KfOOvfBl&9%t?P+y{Eg3pqJ~{Jlg$b-OCD7z}9MSKYF`r*DkxiAfUt|qdT4Rq@sHjC^h`B$y2nN(Yf|qy_Pz|Ao-*7?^iracH^~$1G>nC?s@`iD75q=tzvgJ
ECrYU(8HDDt%QRsL@&8n!MS+ljezI$KgR5X(cHaz3#c?`@I2beUWa5sPX5YUUeQ9D@*e^=psaM|3Vh{8JRGGAw)#kcOj^$E;qv5r)F}`n?ZE`FW)GTkDE?`x
YTr%I0lp8Vsc(-@_m`B%rd5)K2^HzE#8aHx3)*U|-HT9K9k59;9y%2xgxBJFCO*I~;DZw_d=Y^E!5Ai)+36@EQtg}vyivnAm$!P!J&j#<x-U(*l9@cn6Fx<)
@}#iw1)1SGZyj>LZ#W3(fZpaP&+1I#o-S=YQ9ynW9G}Jm9?JE_94d-ixQl`nqQI0S>{ZMCoPxrnVZSbGe28mPixw(p?Pda+lL#P=PtL`pw-Ap*!)bAQ-EkHo
A?QfcYZvuDawy%i%z+!_0#8MXzowQSjlqT5laDi)#>Vg)lx-)toqTB4^)|U@6!!A_+v^OV+ko_%UBcKR2&5baQtcDg9pq0EB1r+k4our4`mE8HP;mS?*;=!(
_T4{s)KpqejjbjNN$@dx%-_EW+aAgYA?njP*$T;6gTIh{jBkkR&MwfDkaQ&)#!Q&k4FYKG{JYsM^{4fr{n-JLNuQ<0r;>1>)x+h9Ikb-rOoEY~PS4BCcyqA3
l<WOK)~cz&m)$$keG%Y<fGP!v!Vk(%i0`R~!}fDP=TwT9#!7;?Y|*;0p*ruzBG(PKOc|{k9DAH@p=*>Svmo^85K)te0_C$1^a?7>eIoXGXfGWIq#GD>3u^lf
H`uJ>wgx@%SW%;0NlfjF`Ls^PDsKe`t1L;mqtaRQ4ROvHuD74r(Y4)MF7D{n^AEO0PYF>GSR@q%y+iMCY@SJlnnp{-Gq1hL5<vNeLL6|zCBHL5PCaYaExch0
hi{-S<A|#`n98GrvOX+*qvV@R^8EClgD_$(&5NiQsLoB_k)buMMl+(gNV}Ro8zN8(RAbe*LHQw2@Rq!7oU_REGWEW4T8tOo+8ObWgrgtdSBw*pxG;sT-VjRP
!;z)@q9TcnD3DX<6EbL(6El%Agy2mtFry<4N+PPJ=VzPj#Tt@SSw#%Q(WV>y=6I@R24OvgQHnzj&&SA;G2cIvsYA;ake7cGhUC*2@h?x+tEgce{fzhfn37qE
v-IRs5-nH7!#=pY6<921441l50L$h9vy;>!wmSg}JUXMqsW3>|Exd%pF88t(ZWfgDB*Zgpz-^X21iAKU%iv5z?51^l9ZFqvJfk}c8t~RP<zr(4jZ^kdQryn(
NhRMn&nqK_AeCxJ_Yf{q4ZMvM;(DcL-Duqv&cq_Qj3k9Q$z<vs$#<02wWv&t)DY~E4F(%5K;d;*yS%bMT^sg%lcK$62M_FmrA^*NVd&k27tE7s{Cd0Gz@%6^
NA$+<h`9T&Ym2NTzdDUjHJqa<r18vXrbeLQ6(FBVd9YUN8~gZz^T7R9iM~QZuYJiCVw<d$z-}1xo?WPBfd2#0i>o?pEo5zz$4uz&BgL@+Dcx6WhY+{DDeFzq
8JH2)l|!PW>Jx#&=S3Uty(tp{ei?p|`7QgQe9s&RH-R8I8v2R9o2B;i&>5`OL-Vge+Txx#7obR4>tkh#nOIOlL%PFH?TvE=;26tXvnmQ;-->^B2vb-NY(A=o
KxKzYgi+ED?T;rcM7QzcH{9Jd1=a6!d068(%(sTEmf;w(Ss7FevmQGzGg;t~n_|G<T1eUZ;S^)zra+NdqhjQ{$(80aGDYOYy;Jx_&?3>y3*c_!mlyR0;)a*{
4X^J*0Sl{(_47$_*m+A{ji|fri<TMIR%I)*E3N0kRF%ag6Eek&H+6FXr@LQUtGFT$qrzq!P66J4ewN8SUXk1jC%?Ji9*3~*Bm(0b`X&gN$N-7G@dRGOUOaFf
l=a0;65F`{3IE@yy>{-&_gO>2(}sNjyT3JorrBFgS+X6nLFY*<w+xZ_H^TTRo9JbjdxM16j}q;M^|P>P+plS$n?FoQrh;>%@kA?jC+@zt+pj_S?zdfqz(;8a
7q2s=%<Bwq(lTnG(U)&CK=Dhj^+X3d38S_e!6sPP6_W+OmNRYhZboS)v0Biyvc^nk|8uB`8`g;8>0qANL;h}qv?W5`Ehv4`(-NCkZJcT(#xP%%UxG@dq?YZ>
U}48@bk)Mx{UR;nbH~bx8XcOmy&TiP39G+Y`-e<t!%%FtC!p%7sUST`67@zpfr_|DoW>d$f4KHk;&x1GM6woD>Akm8=2g+GHvh(oG05qn_Rg4UHywe$^3pY;
^EfKgJ1NNvVaKSWO7dnn%h!r)*K@<8HpRU9B|jn+T7E>LOsloaZqS@epKSkTDM+00HC<rMCP?o|;>)!v#QKD1Zn{yN*&F==4Sl!HVmEk}KZ33bgEv)Bes~-e
-0blhm3xJ^5cQ#=NDtyn>9ihFjpV7B{=f=fMil6L;sS&fJ22F$prt`xhchr?p{kYC`o{`Jmz(wm<GFeNd4g7w+a5OO(~sMQo;Hf2g1PKvDEyqCvXTE@Csulr
DZb-S*p&PxU{BrCN_aS_kfO>zvFNJkQTX%7O=}HTLWR!(Oa)#K@*slF(CBIDN;{vWg(Jv9?{`xrZN^%<gmmtnrBi-4sES#JK;0{@R{u+1ukvDVx1vR?gdS)V
N;wf*@1mV(e%!7XPkW9t-m(Ieg9pa+>Ot>;3!3LDncjSj{pNn1G-s8I_NgH4L%BDQR<OUnNGh~!o}0oz%9$y<J)=Yu&Qxi>NiYub=eTXsxP!m~&>2#e?Xng9
GA>k)6t`)`EdNAxa9;+)0J(Y0N6SmdS;cd<-g$+Ttc`w#BFAr`Bl0f(Yc-?M_<S9EvEGgpfV)e?X+U!eCbw?mBXbVc&NOom=bi8yqIrXK+=`R^Ao#5&{=4wi
N&ISh_*&@q=2{BE^qoaCv@>lgajQyu(_7M1Rdf1f(Bs)7Jk+VZYSkX`IhJKDZv0jwAzfSDzM~@vtWqJy{QWCN6m0e{uejJ-o+?VEw#|BZXD!&eBw%_|5X!Wn
9;!n2!3FLvTMMQhyP2XRazbyZIHsbn_ldTu(%fFq{!#YC?D$dHT^cUoIx<n3Zrm-3sDAUB7Q245UTF3o!E)WM__>~7!xz}K8G>bjV_p=dx2=PyYVF5}THq?z
6iX7Q<{Y=(Sv~6fPl3#8orH>~*r6y<M%RP=eG{S#{++O1YZ2~U*H|gl%Y!)?8n|<ze)EzS@D2DbZ8%s%egqnB*10C88eimyjDb%pgG*jox(n*;r9{&ns0Pn!
6O2z4naKFDHN6uJn`Feh;rkylqOV6ymLTH>f37j$Hoft+CJ2qZC{8dfQ~ROE4iqrVdd;fbq_T<&)RRFxVK~;*g5{R)m_qh9+AI6a!WI5Uesc;oyBA_15N4L^
;mf~*{CeI-ODm>%#A#s>zaa_y?d78Wtswx5362Aebep>qr3K|RG$cSv-;%%W9eg*L2n4}}bTk!o*scY56zjG+?k}DrpZr6_EH4N)H?pDw$Te0@Lz9H66xn21
gl4guZkVI2>Lc%BTMxke#5X=htNlALCI6I$iUU@w%5gn~X1<Z;j-F%?k|wE5=d9H2o$LS8dxvZiMmG&bpG8FNJ%#i|y+D2z$&x|@pISvNZf?~Pu^BLXqSF96
{#~1!NHXZVs1yt}BI1nuM^)lMoQEuyoRQ1$ND56@hWW`~UqwaRKHm$RvnAb__xONv%#SFgZl(GzTZh?uN|GLRQkmmCAgQyd-%uv_569li?qBf!s)3ly$TFtO
2Uq0xjmn5hmBQpDJKMA1UE}mt59^;UW5RT8ff>1_JRbp(74DE|aRGBDUinz&{OIkiAxPGUjYcq2kuj{1{ZE6DyrAf$l{WJIgM{3r3rAE)JJ`dzI{$DN@Twrs
J>bOQ(@n_C5-`)$8)eO#!QXt;QjEM7QFx-4r(S#v@qjT)pP1R5hb-+1Zd$E8)u(*zUCl(gB3UEIhi{ZYZbWgIc4Pdj@{n{}CU%p&^SDLxwbl(x#d=G!pdmS&
(2f;JvZMF$!zVj9lR19w0(Nb{2AAYbew@PSEn27W!wa8-T7knKp%Sux&Go6BP^b2X5EH9i!1<gcp@`ujV!S)a^H}qF1#ldC5fccPO7+rPhgtQq^m8o2(+CyN
<_PV2y@~#+u@)qsud!OjqL{ZBNf7D(lm(2pVUvpGfn^p^63u8;l{UoT7FZn)8dwpU)J;IC=&R5ABCvWFHd(kc0hdV$)O2`;TySbG=*1yjw&TMv_tfqJTbO)=
+-O{l78#SO+x0-bJQ7j?Hn!sAegAdb+SV!-*|hPXd2#8i{R+$XI60W=c7lcE^-7u!I9%EDDko3OP)pKi-q*)$toUl~VigUXK)17UJ5iNAsVn?Pz?BC9fAUi<
Z`>Xb9$cpWUADy^nT}VpztRQLp^DiYSTx-RAS=Gr?T0qDdxCVVO%Td4W(h0C2C3x{ywu9u<Sj3}PAc9K3s&#5xaP5eS9{4o&7eA)|FGL^Q@&ihK|n2OwlMpX
_i)@cEl@e~N)?HsU+{Hvsb^9>2|H6qa3EJL?m&61F~>8W`FBOPlwp05^y(<0-G<-@5ncXE9zaKo4TPs7G%O-${VXL?+y{OkK^0N>U{12ku9X@MoM=GQXGl0%
esb^ji-=Gf@2=^nBDFXdBu8TrfsX4BAV(*q|Hyuu`bmHn*qXY7DgYRu@3JN!5NIFnZuCX3N%1M2av+cz;29Ev*TVl1(5A@sziBwMI7Q8>gQMk0Edy4<q?x@!
rJ)%hZ|yW%P)G8A1@$#P*j4XGDidu)-g&ayHir)@n2Nj+aqfOI$I{8Ra8{K2eKjtuPVRRB>;uD5JB6=H2yph(jyVIG%d#htM1_JIU&^D@7x067Q8#|_QJb-p
Q2?($s(H=X+wz%Dz-L28*8vtY{nNGH3#6?Y0bWjeZgyijXGHA_0Qi<spi01myGFG1`si+P>;yQo2uwxZqj-jxORBsOBN{M|!oOC$XtCgMMwX+<DOAd{c&~MI
JcU3giyzd6@ae`&uxRETp)$c2&$isD+N9L1Xb_Cp)}7R%-h8*><`S_s_MKYv98Q=xx!&JyoTu*oQg&);;2QBXNt50E5+WBM@a`2+a*Zx1f0zTpM1{CN2_ww!
UBJ0*Pj$8wjgIgKwEKj$`CXZm<t;LjzWV@c1f3Ct9-5k0t|9_k&$30~zbm=Pttyoueezb6daOGjx#K$*gV|83P5BE`asC+v<?+Hc_6K@Uh`X--UT%j_Rv``=
uA++O-4B%OWDh%)DubXt$Uaa<5gUhELZf~&%wp)^4SP~SdN&Tf7L~gsRy^o5=y72*_pAyIyV7d^tEe7d7eEbpHK{Oti$)aQqMNGOg2F?w%;{3}_?2dgF%pUF
5gwSU1xSEhNVma}nj0`LxEshGMj{PIx2D3kP2YvO$O^SE#?{Yi0U15+*4(6;MkRef0I_>$!>SF<h!&jwIDY^~-bSaqr)tuF-&wv+|H$2!;uB+X9~!>cW5`a~
jB)Ka-;o-7P42pVD(JA#Vx%d46gNE1JEC9quNIbJ(~>RQJ&Xv-4EG42q#(hP<f6eN<}q~W3Cae&adP0+5++8Szt<d(rL09I$yd=$Mj5)&(6I=B`l%~XB`bCW
HfqV>eaKcJtdbE9xX@BTPGZfa1Ii=Gra4KVHavo81{=iIJj#6^+x#e1d7G!Vksj@XUtxw}#h0x=%zp=m!QAGeDSR<*YBK?$bTIXjc;pg>wROi$ZZlr+$dGGg
*$4UIDgcSINn<MB){iAB%6_5#1A&=cHJZtg(JwZ<8J#+jG`v%$h@%&*`o&}l@4rdi4nG&Hgj{7V2m6}(mVgA7Rx(bqM<*ZDbFMe?BN(JSO6CUXu)sUx%*-h-
{A76|pjCjhSrUZ8sfmty+=|7#K?w_7#Y)E->?(5+k!Qhp3>OIo$Zfn7@dAM-(x$L2a_d*P53C)0lsm%ZXj?pbQ+;C#h0HqUVU(Ai26$5^A~)NFzpfS95~PI5
mLCiMKovJ=1@Ou4On76yE;2Sq3dkD?xMqE1=CxCN<1#)fmhKHHL$GE@*&{}P*DSkdg-%Gi(h_46@}EKlF>sKqS-)q~9zUR;TZFPs1*e_kYb^+mRt6n=e*_BC
H*V2g?b4N@$sYf*oN}-;*~R>atRO+KF#Ke3o$1xA(qDof6{Quat?KKoJMBWf^BKJm2(n|E<oh8lt5wfH#|SVKEbEF<Vog*F((MP__10>-fRD$e?hR#HpC--}
W5}t}x2xbSRAr}u`U*^Dx=-I*z$|F{Hn`RZ--@<nw<?mllo|RTT}*yQ$dG*e@~Yv^N1Dj6SJE6f-mVwpcN}0jzR`V-_p88GtYJIOY#W|rtVQgIkI*KKlpY;Y
Zf<0eJy`VxZ+&}&;zN!Wy#|6zz4wWo)&vWQP9ho<`XW&An2V?B_1e-W@_W>gxL;z{7{V^%#Fh*M#(pM#lA0eiRgyIg4hgufd?MBNWjAeur*az9m{yempQmTO
NpPrj=LJRi5-B;E*EL&Mx^1x&jxF7A?`{7$inU61x{PU(E)%|%ArkzWf$${Eci;>Tfy@1;>Cl`0!QCzb0v<e@&0kqVMg{_~mmzje0=7%>b+W59Dh`c_!e`=7
_Q2fF*Bd^&9FNb<HHF#b>y!;rxRoy5u+5d^8}S4%N9^Z7EtPcIiLg;|7(!+)LWN*0&tru-u7q&uQh8=q{<obmd6@yfiGkd7^T&~#Cm*j;Yt?FaejTwoAlg+-
HqG*aRe)Is%1D)%B_;n~S8wZbJxy2X;=Z$->L_8~dwZOu8==D@Ia*shnHGgX3FEYLWG@YgoYYa49F1hxRbei!d$1fg_;>R(z&9Ee5Me7rwXIu>iNHQtk5_x-
e&V1*KaqXbKD1Oe2;al-rdf(f9XPQFWy(AFFC$RuuX9pB8sg}3&L<K6pS=l&b)}l?+x=f+mB|>F`mo2N$wjmCMsP>H7u^HA-j?<K$@J^N_vyLYGi)X`f&{);
P!4yB)X<(PlO*_Bt4V8JVu{nZ(@-0oh8^ZI3++ez6r?r#to$?tnQ*jtax;JAM>(mTDEoqPt0Hwni%l!RI!^?DF8H#CVgDi?YNC>DBs)S#>EEgM8c*2r#aZ=s
ZFIFiMVL2$P)9${`fwOHYsb5&%lg}v+!cqjZJs7>$~otWyyr{;`^j~b_JIwmk6yF$y06b=b;Jyts%s#m!a6>Dn{k=kfU0c@5n?6S@J<%$e_7_gs@rnYwbw}o
mh6#}p&Y>*QVghWKJFxnSd1hcnM8*BZLzMVs9M!A5$?M}MQQ{0i4KPs<M`7`nykLQii#vAI8iG_{EMlBa6L-|>s`SkS|ZwwpNHYcz#$vDs-0Asij>Ut8swwY
SC@(%hWN(p#Z@^cCdA;;bTABo(CDVK8FT`Cr0qI#e*rGU&Vb_d(-GUDEEf2%gIAl|L!es0hk`b3xV%|<=eI-uo3go_jZh{h-s|x>4BLg(&(<oRPbWog=yJR9
Ad1w^X&w*b3vN54ph4j4sL=Q!ELDygH}zo0prbr7ywV;=k^1KsGC;&zq9Pq;Zmhb`CKh*hx%OBvFYV#QOINp2EG-}VnId*E-g&HKt8KghqW4;9X`!36d~7Ww
k)V+bo<;TtWd#q;c`{e9K@mz(CJkT4l~ycYF%G;#h((RJhkRYq1w4#rC~|ZajYVWeI8IX`DPK`2uRjy9l%r1?RC`|Z5mEx_;-H+^lCFrepeClT3Z%aNV6HOP
j@`7QNe)}Be=o*&GxqM`Xinb%8PZve#lu^{oh;8~qPH3Ym+bH1LKRk!#r7WKsI?R<1qi+9_`^pQ>4)kdC-=4lz<{%jhGiE>u@F~Nj1#z}9Jx74&0xX2_5xJj
@g+2LVh;LQ<8jQ%&mGeMIkr0wz>ATE-YmjDf3LUIES=6laeAC21!(45b(4!&+K}|=zZeLE7(FJ~>`k&BlBjMKTvd^oI-g=sa{hzjl?11i^p;zSUXM1pKRcL7
^<o8>db5VrOn(3dO@^e1Qr$(8NWOz6qW(tNdJNqTK>vg9cU4BgNA#Kgl!jj1<kWE?)eq_TH@^n)ZU-gDZ48pIX}oKV{QN@jvn+ey&)&Rc3%d8>G}^Jcd0<k<
eM_PRsvVk#=^Su{Eu0wCOi3&)$AA;`Q1bgC)U7h;osBK57A`2V2LDYmfqm5`Y%dGO!;DsyNTO5odzmsfxSh6S=nqTLDH_MGl7*%I`Bjp?qhlL}g?kpXds9=H
WJG02gETiFJULjoE>*sp1bABzsR7zw>r*Uv76cf)!9k#hv|U56bjp6kwYMBzzjoBa(+c{a9fI5k0og23)f?02x9r4aSc~)Jm^!JvGRR?34Fn}atru&;0s{3@
x5yDzsHjB5X<5g2Pa)#7l_sO7MlB?_Nw1d|8~81Y7-U?=b*HjB_!V!t3hEGcm4^#<cUln4qx6kswms3GtT+=Y#<2U#F%_dYc{XvnMxV2A6lP`-by_<397&`y
Z$fW$2hz$3)G@Ms_9@fT{wA~}m36icUpzliSEIZqC1PTC0*S7uz+hYM3eu_PC-4dUOgw-_%T&jpw257GpJ+HhX5TRhGW0v*qkt98yFq~dP~ecwgpzP>K-0EM
T)&bNe-woB$}x*TY&(2}<!<z&s81xcA7W=mb%VOBKO`)?@mg`v6~9m9rO&%28d<?O2Htk!3{YrK!gqe6`t*u^;WfbH`^nA8z6rllqz}MJ*Em=>Ia^B;0i8=s
NhJRO(Jo-?B&5yL$%1A3{q_$a?-1!iKTT~LZ=Lx1rp!~vCkCjG0|Ef#NF+vq%%=oxL%@3_-s2yJKM#2p;*H1;S`qtB_uWRkJ43`_=n(3iOQ-<brsM)_;xoSt
H7PE~)EXUx7jIBKsILE=xiQ@Q6;wQxgz`Q${UkopA|V-A&UlwffKei5{21$b8t8?@RSAWpx$}zZVS7Qne$!~xy~g3Rq=K5%32P{LAilQ(I~g^3@Fbo|O(kEp
-lJ9IKZtPtC{y_jy^h@J=*~<IgjI_t+3Y;JuXE)@4YQ=Zo#?_5-mZ}5a<kr_s=$J)rLmGmgJxiFDkzK|emA45E@akBua)lE@FAp?{j{ezK&9Brf(fg9qK&{~
LHadfYsN}NhU4+pRVCo8n96_MOxdsA((gs&y7W@DU`%6bq!K|Gh&Z=}bhWu6Q32m|b0L$PApe3>xL1{k?|yb+#nYatkI6t-$oY5$7Wuq9K`oQ(0JcdO=Jc-;
w{cZ7_kredKBECMxakx-1Dqf|GAM6@N()lnNAj(pssFmFcaZ&&v=htD5VgxX4iOWaMZnKU5PQ!23Lzsv(Tj$Stlwl`5NQ3X5IGpfVFj}ECA`^9vF3F_pQSTn
V?AM(f&UURjZpDvpvUiUyCIejhv&sC)x-M&e_2%xm!OtoU4GwpJqBMfO-*}C_I>74rL^?iS!iee315|ExAc6W`40F#qspGo%-1XBph}pCxLQ$m9&P`?xCG7G
+0dfkUVs0HjwCFq0!6qKWdo<B99;kVh}ORO-r|~(0{X?oP-59g#15Z6rDKT_?)$;FL2n>{BI>X5#rYlxAy6vKGYm(gYp+TjahlmzLxz>IQ*Aax>rUR@HWJGH
yFe=tX5EghoSR9Op)1s|amSXtj%QxkG2O;G(mS9QA%1>y(?V^Ja2xz@`>ZP#meJA^oJMQRrWEswwvJZc-`XK3Oc%bwikq@D!drHf;|Z=>!sf6xU`d4?PZYIu
E|rvmTc5N{(f<}xAh0#L5T_e~ldc)PWMnQXKnxxX*&_P6p{=@5A{V202~RMh1-Y85fSqouk@pxd*>heHVf54Bp7_>qlsg?zzqhl6eV@iT`gFyMdp%tZsQv1o
e2vxwM?31I@!w5wj0LPpJshbH=-#Az&$&HlI@hT*GWK1LkA3ReZX;9TOJw0Sbd#b#XvOSmb5jONdN)%x440ypJV`3#E{^tVQahk?k_d&LQ3`Oi%l%_BCZwLt
FO|4{)S4m|z78G6{-UMYxka<6x(P^zYCrK%-#Yq65@qif(lSBM#lCqaPFP3>Zn1FZvVSN)MkM_&TBY@udc?qg$66GKQw16pb}w9Bm~1vVB%4ymkb57;ZsrA+
COD!l>5(i^{(c1hCO}O2yb3#nk7%kAzAFg|a>u?g8<AxYSZ}GO;KSgBjxqn<i8X?2frn$W<xrt_9Jn|wI)k@zuN0a(f)x^vf5&u|eB}xbf`|QNNBsqGC1A|V
Rd{YTNs}&@Y>JE?IN}r48`jS2fGs?!0hEaWEvsv36B`Yb!NAQjl;i4anPoq|xVaW9H2JB^O`8@{<ju0|_q{~Za#GG)wI4>e-kLTiSUvufh+tVA%Y}J<7pNB5
#$497LM+gsxXG4uh@6|(UGRNr%(q6-At|!Zdk*xWFZIKs+olJnN23R$?lYWNEP33PaH)>XqNtEqe@)@gM#v36fzwiH&1@WfvMc%X1P|~)<ABw@JRtx9tb9|W
1yKgwW7%Hg#UqK>NUbNGwv`gaXlofZv?MlZ)b_$#vs2p6jeY(Vi$~MAlZwmxSc99jnlG@GLKXF-qhLmV_V$oqimIl55tZ=cf2pQ7{cvp=?gV`!;Fb$(v{>BS
KigtWRbd!JTW=h!Z@JiUvU}A3t{G}>?d`Mps?kFXg=u8ME0Bw4JrT*dzbf1xH~z4fWJ90=k|x|_!3NM@)#37~wj!jeZj=ucJX?1~;}8s;SqX^=kfeWFQOZ^Y
y`ssMRm2jNja&UQjd+^pF%|;qJrOkbz`*!z-PPg7cc_-+(8<@IMZZ&TW;AL%&_o|oVwp?Vs|E62)PJniKnW))WiA?y17rRJz|;p}?b}$)_#QSAbX=h;PSA!*
zW{3;tK#y#0p<otV3p>t!fdf8Zi1ylU(Kw{+V005vIfl4<vfUUzb)KaH&~4y(VlHA@3WKY^p~G&ir8b_eS0fTWTxu3=A^{Qw803(Sube$^=sv3c=ed$&!tY|
10LQtI&q{oeepYPM<YD=m+cc_rEjY;2<Q-&`XJT;65YTkzN4>0_%DW6i^l6`EjZNd57tb|B-wREkH^zNZj!gaJU8jd#tK7!&6ssoJ3L7AO;rB0Ci?#Pj{lGi
V^7C#h#?wUS1wd%24oULjzN6Ku4x#Mv+d_N22%%yB*5d8IG;>II$DLf?G0u|Z~|%;hKCZ7XN9JC=N{WS8u2oZ@CXcvn(ra`sHuwdA&(ABX-u2VQ+WKLS?v2B
9ijlYd&C`5xp79w8^PAjKiNK$TI(~~2|<==BhXK0bCWw_B^hFSpUd@ET4)pu-%;DLwkjKp%uq#Jf=)b}W9eUMfn401Pe9Gla6qGosO;b+yr$bV6Rlj}{1Cf`
DTq$i0ukZ0i$K4!UO7A!{>xw%$pX3?ov=lf{Jg~Z04auR_(~-5c>TQ5GBgMM|Ngh3J^%)(K+zv9@4>P9Fd3R_QcrX&M%gS8iEEogTh*(>V$XtQl1$i$sAiho
MFkv#?&O`5I~K?S6d%&P4sl9~%_;3{o^Dg&y8$u6wFH?9UxMfRiK*ju{RHAAj7zM%+?jteHRu>H{OJZ|`Zrx80E@bMqIr--|H#+l+uZ$D<{MH^!7;w_%lmDI
Mysr9VMA<a-D2(>)sxEi;wip0!xyTA|FR91PXomgF?$o73{%-=1y<{V@H}ssKL5lb18iL}*II_%32nSB#JNz@72pzmgUCBU=HTi4ingS<I*>|if_leZm1niS
Vtutz=ko=rX>TPDnPe>Wy~_HoADs-{-15IvONvn34%gDeSZpoGJ2)biZ{6{KMpzxa#p--~CxGs=m~55E9d1SPkVr8VQ7l;}vBk0e;@NZzC|GzCmIikC;aLlI
!V{*cu{cTQX9p=>9}>5miveNUP0VM_NU*RHoj~U;Zk!B?GY}2Aot5T&h!e|DM!2QVsoLQ#W)HCY2<MwPRgULZp}9$UTBS;=pY>BDUP!T%=F5~`c%QFhEkA-2
Ra>^gqy*h%-1Rf~tn7NvD(A@GS_P{JseqQ;UAHpO{PtD0f&k7{`w+>Cvm$OwK1_#_lrDr=&2#&uF8rLALwC>&c%Y{jj}R@j!LlS3D(0%ln#!^3f#=V^3as-m
#iyvKMe{>7=fi#<`zo?eeo8_ci?#$BMsa^pt4eev>_99w?k8OIg69Li=uuOruzB2Pn`-T4V66+P;`nqBUjyij70Ef(VOggMdm|-tD7oEl7$erTOYnXrH79UG
a_%54g~%ndiO1g0?2@`RZ7dgwtypN3**hkDLNb(aqY^zWJ7wo<n=Tj1EbU?_22lFHDMi{&gr13iGQyt_lhP>gV?C_~X*k8G;gCR@fdUaIA;+ToB*Urue@!86
Spv8qwII90nMS+5tYE1o8-{&0)v?-N5h}-JAj$aH>ucd#t*W0amKW9VP~Fs|(6dYPksX<-z(=PMEk`f16Sfb`0szB>?DGwVcFX1Nz8(wYx(5YGM9lU9UY*b<
8?k+3=2<(NI%*qiS!h;odx>Xr5ma)HPx?ZXEmtGDMC*=xwY3E+48bwjL@JJqc{wqS<|fHA9c)H*veR^~3&_3Z1l-AfVqp|z@A?G(?rhwTGFSw8U>VH)rKeLr
MoWc5@YwE`$}~mCe<ZJEKfIS+MVukQq%T(B;ht@D!|6WX`SMI)pb2!-eDuS?e7f_IJ1i;pa9Tm^XdoGlcWAPIE1kTFH&U@A342lHFl*rN85{qVVNu2y55IOn
Ef!&iLkj{KW;8=4kD5eH;KJ&QGFD9-p6k8GQ__*DJvdaU(8n*V<^4s{u!(9Rh;|O!ofWL>7}e2K5XQFepZ>^3zXb?yG1tTUGN+CL4h3`>tFd2B&#c0GtdKV!
4g`2-K!0UFf2U1KyxR%Ituh{!kk<m``J(dT>%P>QpsoYeC7lu_n!c3G^eovkkzw1$zY#+>sMA|XTaAa&vz_bDD)}zq1s0lBWs<EbLZvensU)OJst`rE@{V+W
UOZEyQ|^+1n-8RoP0sq2Csr;W#;QMB=cZAjdufEmS^(shp%H0asJIo{+c!kQBG-_>yr2wQ$H%`s@6FuteEU0W8vxtygXuv<R%BwvA7gjhFFJy>_&9^!!V++M
3R$StCv;%KNv_mL%su|T>M?9c<NNDp|5@<eoP(oXvCvOA;pd_sCnIK=MO~JH7ihjj4FjU)#idrUZLrtdLPJGA#|}G+9zJVGTg0#i=)#S>6KQWG#!H~{@Un}2
dFfU&(li;o94)FC3ArVWB2zi7KAUgX>fc636&%9&ON#*^aa=4SRGI(0P<!G(chOO;oz@UkJE9S98=x=8$A*!`%9s-7El)uN$|X55WQPr2zLr``++4{qLBTFG
CY2y?KH;LmB2tu)eLg8u>MoYC+W>r8I+pK2;2*}4N2$oQ7Q8XDY@^}??ZO$qG1yJLmM(h&X_07VN@l>nx89A$v956ju})2b$c2K;Iv!+aI7nm5)LhD8K|v-Y
+iUNxV$geUP-1frDxH_8fhfNv0lp38Q=P?VG+$<lr_vEi5~Tn9*#D(_^f*lkqiTf5<#av^N;5$d^B;tH?<O*FZbx6ucpMX!!_FhmQrMiSHbLHI)kU{v*r?wA
cxvjwW>xqL75~l7tAT$0^VLfL_Q_PeCR9hbeHn#E^ZapjAP44~CD#TFtA2*l7q`r9)QPEvOLJany!4Rdn=BJ$8a->8HP()>t4VzidOJ=+yTm>HT_(LGi4iaT
5vP7;UW_D(pBjVJ>W{M~4wjxQotdr*-}0QSa+Lcqtr*|ra|}rVGJgCVEbc=l3OLQ5p_IeLOWxg-n)tZoc&iVC4t<G?uJ3)Ki5D_tx9p#91vIz8*56(1A=atb
LsWl{Sd@20aq1R~)lJL~4k=9~kK9DQqMsg5xf*e3xq!N7QG>-?Q&(FFbDf@{cBLS!t;^36n6CJH=lV$`jnCrV5-e{wU)_Dj`qDjpd)wGh(Genh0fm4+&}_lq
^vbO&Pkzt7p%v9GhgM6?sn+2;n!(r120fsjZ(ENL^Z82TsX%$51tVeB2z4>%aDdi~j1P>-s1Vz<!iueCN=bWA5ab3b3M$KqVF&c<<9OFQMC;LDF=D+%&EIPv
TX!*+uch5USxIqbDquUSU#|g-MgY-avaq?uwX2@$2`O9u1PFy46g84};Zw5-tOjAP6+Y7#;6~bh#hTfsx-2ydN8=zYqZ8}5+{61hiKmj6O+9!yY)An6W~8%V
D4Z5-ec+BW&YkHhSc4ZZlYr*9imLALol=tjAfLf_TxnC-O505x(NlLu&wYk|caXY<r_56uz8BVHe1hg9k<%Rg^h$4F*J11kL2)|v6Jj^x`Ow0UlchXDVDK<)
sJU>OHLM0$k%rRoEmR`iBhYQnx?N!!k`EvoZRHy-6op@Lfx)Z`ZN?7jHD5CD=AIWk>o=RofhG7ueycgd+U!5ZhsGI4f`^DH4-rVZP-Z{fmIzNIDUZT!EpGSB
X=ipZrNu88eI!RyYam4~v0zxo2&T?CLMBB&>Ln?gP#IkH4G-;^5KEtmOB39KlHFRb=BE%^^YXCeE?DkAw>vqWJU!@08MgG?LBnF+dOF)(*>-KQ)Rye9-P7--
6%R;n<jsyDM;9+=ux!UxL-RJo9W=5?aHn9MD+Pzaj1ByTt4v&@sS!)9&9iSGS<aE8J?VLE#yKoPz=d*F3&~K1-ojo|Mg<2UDnm7X37aJ2RI*e$B4yhP$)V&T
#jcOK>cC?z(B(?tJv0`^@{2N)3x~=`hb3rYpRUB5pi9WwXyILYblYlMTVjCKsE7QaHb}-{4GKh-|2ID|$T?$rfUz+jU7rmgf#6la61rjxMjfU7x`kyKD~4ve
{W`&TTg0<Z7_BZ3iLnjWh9{q$$J<z*`+`lPPP%!YzQZvvjiH+~{`^aAeZb?sS>T?liQa{Ah2fLpg6tSIZq*1RF=lvk=@rdE#&(zTKWSdaAt*XJRO~LjV=e8{
T_i5&g1v`dSu84?eJ}&G@*XQ8*w!uXOn`<SL$nxhYB$#hWOjZCl#W8}Rz-y{pIXI_=&5=-{#U1{r^`2eE{KJo=(H7)#`-Xqn5&9N+lXnBK5v8ejnQB?y?)zY
j|m8O>qjHOs{@)`5XX=y3r1UsiCeVM4Y^RpEOgIRp{wsWPeH!>Ir)x1tj*7Cq=CJ7+KYvzuCO9w;WM%+_1uTOwsyQ<vm;qA7Z$O54!vM(>AI9N__J$RS5bc>
^&nlW(`k;TEkW?YcUcW;5-EIIDyDEg3h|<Ol0jghh;Z^o7D2_A^M-5uFXXa&=-nhY+ye)`Lr7x*A3PdKh|>ZgykWotPq=j*)icFMSN}tt=EvQUw(C9Jxo~TI
6AKfSi}((jVHv*X;>?a`5aC%wVC~>LAtW*g85Zns#3W<Bl-PY0K_EQh{bjp~WR3|h&Oom+gPYS21AzOj!-A%1I)P;1{^VI)=6jfEw7BSBM*(CZC_!(|kkNcc
YA*~7`htXZW9xmg3!nOvywgO8RMG7OtdSh$uRF-#r>Gn)X1i^s`UzIB6OLuN;86dHC_nonXYVSx{55EE!-zL!Lm&Bc2m1W~lOf3&<GdNl16~*E@MkYyu)8&o
p&=bI<OzIW)|_%<)>iJ<$<CA*9<<wmZ*@b<?ydaYg83%VwyeOp(u?w<TA>WgH?e9gfYph40YcB_{-EI-K*5_hpr?2s%hKXM>ac<;IH`S(fLIO!OTta^#~B(4
vMxM}I$lz{QUpuTBP?NP5C~VlN!7@wF4QOmuM|lW-tx`t_cXPth|meFy^4FK?au<Uk<4)4Y;69eHc$#x&7^}UX3Xqy4}&Hdm+Feo=;UR~xj65Kk7Q#5;RfhY
ArU!x0FZ=wlz%jjt4*mF?kp&ttNG^v@4|n&`~)1^sbHWiQ&bd=8Y^z#?VN~I3u;KjmhQ-0{&a8LFs`JC<v(;vdJwIQlE9W!QXBSZ1HyvcqTucL@FY-bVFYB6
e0EZ(3tMvp%qis>iEKCraY3w_&Fky+!xo4RMc4OIoqJ2;*qC|u$?jkC!wl|QWPDQRE@Tk_a+MvfH3i5jwR4o^q+qVkcF%Fu$jtrc2ycR5Ag~Lb#|#%(tD{;w
Ee;dkEW7*1%ZKmW1gdLy7oj#{!QCp$)sbAOZ5ml5>oOgj9VyKJxZj0YvaYHMFh$|`$w(yB2fl0(qh?NE)B^JIO1js>g`N_H4r;tdaK)a1V|jx&4#Pk$b)YV#
p{B+eJDdv$)iPU>9pj=N#ZVK54&GPSniE|$00R_~?kS{96x0yqFWKr;{V9xZu|Ac7#HUmiKr3C6nc*5*N{DXfjpL+ui>MXH4ax?eU>K*?t-UUeZ^0aqBNfe-
YqQ*PJ}Z33KF~M``G_fVQy(K)YW?_#Cv|;i@3~+EticVWSjFdk_vQqLh7QikyF#7`HXV?i1K$`U?F&|BMiO9Bvj!8HGFWPifaw6W4SNdhWM+h+AJuVQ`XE6@
R#E>A{8BjBZ1mC1c&x;L0I9jDINeSYTF)e$?*Zi(L91$a@3}q2Z_iq=*s3BkZHPnP{LtRY_E=_t$XDD$kP8eq#>S;-_l>oX2TRk=3b9izw#A?3L|xVW3r-Q|
U&MTSy60Z^ne*>HSaeR;u96Js7$rpc2QAW6^9vOKUXs53;Kb}MlW5EgB=Er!*&3Ey1A=$sM9T*d_5#^Gux)a_&FmLC2tN^mk9T*{)!PmC9E4S`;&PkbxSID8
(5I0qWC?$Q`uTPdT4wvI3l82A9#X{W3g3r|fhxd4{h2q=bGf=QRIs2fyxh_R><lrOv8?c}Sek(3dnM$K|Gp^|MFgIbA$~qA%=z2GXhvnLF6jhAlXwsuFhrpm
b~ZsJGiQS`uzwcy>C{6gHnejd$W;`jVy$PZ0{bwU!SaoW^9g1=ly@`)a2mVi2ungbfh)BD`{UuvG87jzozhCLWS{gx%z;0lR^OCLM4`wP#I+(DlbYk<6lu->
#qIDEK+j8g7X%?vsN{9w8GGse)B;SM%LcB#83nHBGR+b&CjH5q)Jo{`&MSxKlKA!mR*<YcgqEDn&TyxyTP@7ImoS~y!F-oKYo5&yIf5hd-Z(wL{{}3TdN#-L
x}Ii8l<Z{1WwLHuy};a<eNlphYyRx@msOrrBDOsPo}3-~>+5X6oWwpHfAhLSRxiS|IbD}gAoI@Aklh(}ibzwHm7(EWbpY_h-kECVG}*-fZa9AoRww7=HUV}5
fZPPqxai&H;m1xBc!xp{eY}L}tj7zzLVI`Z>ATB|`Ev!As@c*smt%IDaia~fp<i@z=I-}VJt%2)xLOrZX)URz#SvrC%I0MSA>KA1Wi;5&0t8n8Nsfm0yOb#R
G0s9?>5AgU<{vrSzCPH_r1YKG<?twRFSR>8pZ%i6;t*z(`8KfQG9X#0+9Zpl_p-biePt{JAZQpj?oO6~c2<vx*wFQMom4pbSERWfw!UPrESmJ?;4sTY#-T>>
4i315LzBzmexhtYC^u9Y=f`;w2%IVFmOalV7Cs5Tff5`vIRL67t+RmU=E*ijDhMkBFG!R`%Q3Rkurm|?=3c{yDOt^BT^~1;z~2?MPjUZRqmsd+qj&2rz8>Db
<bdqg%ytpE$<mw`Klw7b!e~by3RRqhnvd3@%<wOB!(IgGhrppjU*w%^JkR&x{+@q1Hp7jrv-AyxuTPRft)d0ia%5&3%cmmWtp4RQ#t$YGOrs~qtU$V5Vjn(q
x!9|}Cx+Y5Ixi~>wXytLuS^H7my?ef-|z*tWAT8CA{}?i?y5QP&-{aCM(X1C{9hw@E}Nd!J2eHNW8V_eCgCKU>7PRt3Jz~tKpda*Ouh+spdCmOh>A`W<UGDf
`;|(^*DZA1DH3*}oCsfQ$2H#X%X|n*81ea!H@a5Kr;xL)$3#k2+j>q6Qeh`VQb0BM*-EFvLu$)9<FcP0_%)4$)pA{BM~8lG_9(Fad$=pKxuPT^LQCe6A&r%A
CAmS67WwpdJkbX)UrZZ&&NnRQ8{L&Rcu~&nehEW<c!5+F$~penJlkX*!~#UAnitZlq!CUi`lYg~<&>#FnRH;m_OHbmyM|9q`N7{B)~d8-K<`2S;?Vch$fUoH
q7IvK+GB$qZvl*=9j~NyO1t09A(lGUb#GzjkAgC4u;wY`7a}Z$8DX#*v>$s%>pPr2watrM7IXr@&@VViCjuPdG9nq0RQaj5C@kv}22wu#EZULr+B@f6?1ENt
6{ua<m8*iDqas@TIxt9y`y5m)U|?|^ntG<Z{+7ZKYL&+7BS1M_^=!+7lWn{DRZ`(DyeqGfI~V1J9D#Ql?iYwXEsdX_B*wX(YBWLLj{szW?{{Z)=T^7+UAndc
`h%y+Uem~&b<#!S_Th`Lf`M?#F`mRzUb53)=LRF_9O`bWZo6KI?-ZY^pan-qd(dw8x;P2Z!y)6~^=lCfAduQJ3hAP1m7QA#2>Squ!@*RB3-7&@wvCi$OJa%?
EwUD;Gg=_`UgU*}P7(++y-Vt)?*f{g`QxZMNJ?AmYR`3QC=L9j!Zw%(=V%e`(F4?(qJTwfc(5Yw*{8?I#?Q9^(r^O3DD140pCKE!A1=V;&22ntD9)I$5CxiJ
b0X$gLwfC(1O0j#C|nkNj9};5yBpOcdA@wj6^ejUm_3A^Pu5lq{L%~0aCudP<=FS@LAECe4(X}u+!ZO_;$aujz%{yX_A-l!SamkNLM90d_phkCClZS=)khnE
P;xLUByDQ{Ed>C^@0Qe7Pm0^T{YBuvEGEJ(0&`Dw=4iuzCek$uDNppQ94h(2fWq=zGiUFeSoC=M=>CwAhm9gmb%$LZOc;6XC5o)AiW{&>-TYD{dG062_2BS!
9RCD?8%MNrC&F^P+ZAEXF66t|d)o3{Dfd6MBmdt)r)}J`bUfT#Zx1-+EE|we6a*-Lcn&`5-!v1JikeHiP7T}#fULie*I{*)OuNBE$%~lqcnaz{u+ZN}Gd!Gc
5|m-7o4;`^xmVH@?{Nb7d{Hw2H~mqi@1nuiLO(9*pB8qZpbT1XCMIF6AfFQaxA_h3tBg&k`HOT{c8q0P`5rE}{$srlch;QPoHHv^k?n0N8@xQX5dI}{N-464
kJ}7;U`p*RnrCVulk~7m6X-)V+c-3sGoh+Ty~UC`uSgv$*#9Jk2`ldM!j>fiE?eWHdnCGgT<0s?qjWexfySPPSd$L#*|yquV&OxS`o-x{d)UR&Qb)J#bxjO{
<vw%AmaG92zQcXl3bCG`3H2*xg$f0lmf>*w0*(dU!k(&qpAm~JBS5Q~XB5;;61~!p?Za<2R;P#y6dEMtjF_Xwt|(-_BXE;_0(NhBKCSfMWi2ACvn0l!rB3j1
^veQpdK4va*z93bpV1O0Vv&oO@+WmU+<3PmPw=dBICK(g(S~TGWCN~Iq@B}Mw3D)e)hcIS^jq;$J`>rsoi1<$s=k3BXbC1>&}o;f_>l}d?{><W{qq@4b`c8x
Ki*)pH?&@w$GFX4HB74d?-XjPa9mzP8H865$&xeSt$$;}U;8Brk@8!K8lavg@`1V`!Fug-JbHDdxZ%U2$?mtA!hGK$7(n4X9Yoq{){+>mL;M=@7@JxszA{zN
QTP)#dl2XAAFpuFTL)MWYAU<kyxw63<Z^2Yo&c^c`nZh}PvRBT4Fh79gN~n@DNq4C=I9JHmFDixZdq-2F^-9oyidLU4~A3Yzj3*qEu4AS&*H8Kt6uKb!QOa}
OwJZ`YK06Vj2VuO{?v?Xh|vBE56lj4S(6@&i!|k*L)2t=ZVusU#Ax~=QKu#3UMZ(mAW3aA0pPlH7#~~}x##ox_344z5>X38&MIsTpSMnpAHe?)+ssZy@M?zw
uj@6fP>}XW9WZ^$#}#U={#&)4ixU$|6G$sf5<E{S_!`QOo8xD#qi_4Y6%`By=~{?uJH$|QoGG<uC6Stzyt|*ZDlhSTQ(Q`+d7)R5U=Tg!5V=cfXo)UK2q0)3
>KfF+ES0^9BWiQhwb>fKF;EQ&0JY|guYm#9ldN9d=!&3v9sXh{AkjNMu|G6tZ(_PK+@StAV;<L;!%&z#Z95#-yIVZ7n`@+0D?GFjOi?N69LEKA|H~J6^h=3X
sSjJ;6oVUMQG^0tT$c(*bf{GuiyTi~1qSxl>6vDVGH|W1?zaCG_zpTOef>6(y?~zjts462d{;!=8n~t|W(d@Ww;8}T1k&7NtP1E;(T&J{QDXBdhOb?)ISwUl
Hu7fXaKa|of=GEW*F$6%0(kkS%ybsf`%$IL)Li*_Ji^Uv`!_$TXExRUpeeKDQFm|6M*{3cLxz=MD4QplCUdROQ<^bYjml<wsYD~uw`2H;x2T#=3)!pg3nYaS
I*^6~?&x40BrXv8&~c7?4xJBIF<&@JDS6K!>m0O>Bss`1n{)C8?|SN`r1rFww!v4}83{0whDKyI_MV=@ZCj;+#qJ4NJt92{m6g4$HG4>)T<g-<Uc+-ahnkJG
D#gON?ulgm{Hh~I_k3hRVlke$P6riGB}wHUm?`33>@C`rOKgC1XO91WgjX{C+-YF5Cb1s7`4fy>5a3--;CV>V1gW3Cxqt#g`O$GwZTjM}Nx)0xc%8Rra&GY|
vf+cKZtg<tl95aAouax3v`xgd(vd%6DYf^HmVO8OP}(Q3TExigVg>>+X%0dU3CV&1lLL}kY3`-t;a?!rA;k9&0@3q~COwzSJbWT)jT@P2c{DF~Bu!tPV@@)p
5Lu|UdDWW>&#vK-ecL!fG{UhL2_FSub<?M<uV~8Hp!|4sam7}u0g56ZQ=l+TV2|i^BBLE+s}{1>6_K4Yh(<Etk*<5o`4SGGZ%QYuqqXIU$};F$GR1<>7Hnp4
YL&3oUh!w?hlyKmQi6Zb$<qL2999ey5?4?wJ%fRu|Djk-d$0Z8G}=kd$SjEiSMiOV3Q+qf^i^Ame3o{oVJgS5+G4`P0Vw1b6j<(4>v>7BN|V(oBXi<Wne6vb
VURAw1iUa*;gW|V95lFX8<LrmpDS<rTE+N!Le)^hG&6PY^}z(AJLpK5VSSrv$Oqkg{vo|FD>yAn_{TzL<uphWRn^yla1$>zipYq8zUoNv1lw!=mY2tsm)#BK
%VP~WfgcKC&=TvV&u+%a6P7{tJ8s4Q);sCua>K{c<>C_$4Nw1hq8waJEp5loj$4fY0mr7CF1tm}apLmr<WTy%*S#FZ!YH2iSRdnVk6PrRT$|_U<uT<K_EG)?
5DPj<T%EJhX*t*_aM>QPVgNA?_=Ur>5Bpm%gF>xiGo$W4GVkIPeB>tcRKV`Q^7$_o6@}?RDIfwJb}->z8%f4^XRUaDhu?Cz8@p8wxa=W_JI$0hHNBug_@`kg
;)uv1m0^5wtmqRHGH%AkmogRcsE&FjnzoQopc(S)#791}C>?Zi;Ym^-t=r!NesJTJBS3xY<=s0La;52$_6;#W(_#vvzPw79@sr-yWHu@jN<Ba#ig!UgyQm~1
BgM5b&?t*8f=~44i(}2)%DQi!jCsHIrV)rCY8Mz-5-VVhEa54^I(U#`?<OQV3r<!%&75%z7*_F{5+?kqdYy&Rs`7RU7Ryp*4#!>44n-*Cb3_XQ{XD1u62=!S
L@wAkHHW8jO(qshus^$1OQOoa&?N?~*P(p(^Ha|gs@nDq0#5pA*Na&aFUj6l+QNNkV(ri-f~_7?`%O_M>DT!WPFM`IA8y>DWLyOS*|}9t7TP7sr%&@zpPuLs
;NLt_#cVfUQ;IbAklEFLwCrF`+{5ltP?a7`;Tz$nM)!qAB>auC(#a<`1(fLRiotit$AiGqUUj553eGtVAQWn68W>+L!L_`mWc?YWuyC>nos?bQT8Br_dt-ww
5ehFcK6P8lg1l!vQ;fjhK}njU6)8U#>xe?&DOdu3Ju`}d96D{n4l}2%32n^%MEcGu4eb%<^D?#jfQ!ymB+8=t5X^KOZ?x<4XB++CBbg1|IYA%Hw<{<&a#VS#
=t#5LJ)*PZzn6FIC}esnk+Xn91<_x>KO-cH->GmNHLI46ZS_*<C7wMZmoHK@ZKvhFmd$kw1oa3jy9%=Vt|tdqLT#cqG&A(h4l_#spsEfZ6Obz}(oHxri!?i_
=e$d8m_23H;d4Fv<H#aR0M<UrgAKS*yHu&HAF52Tp}0uiTC~GIk(_x&?Wl4M-QxS;i#<!~oc`EIDv|8*9{9_{w6pqWxa*bJZA&c(J_^YN=t-B-!8vv8mJfz@
zk}7b^?rN)RSFjj52tU}^kwEfwUsHjQ#Hu&RV|*Jfl&FQI8#C(9W`aBLrtuo?!~-AQiscmMw7lz<}?$nad0G2L>Nc5a=UIr%34l5YXotBZ`56jpJBVu+%&gJ
y490^B&#v|!a)MRS>VNGe%|T#cSd1zsow?_3eySi6zE|--s%T;9s5zvV+Z#v;``x4-%Ft_Yo}yJJo&yaVXG0{;AekW=vN)*`h%!DhzroR<JUqdiLUTBLmEt|
og`f`z#Sb3>3NH!hkGQd5c%zd6mm9HJF!W5&$5SbAJ%{*6Ar&>1fx-OiQ>>h*_?C*pi+<2W!O@l4b$@Ca=zH?_Qll$uq^>rQ4^!e_1+pshi3>;goRWG&ES);
NwO%_b`8|9TTL@h-x7ab_T^OCTf0>(t~if$t$|p7s>ds7MfGWKB@W!@GAONbh>fn)mQyJ+&Kh%pTyb+uV{>9`f_;CniJ}-H1n5X@Cem@1n&zbml@@(OM&66q
eCV6l1PR&zs~NtX1DetoOGRB_%<k~}Zj9F$V#P}Dt2$0G<8*=JZL!k636@ckS@uR5Z4-EyrX>JOE&sg=vmI8;0Vd0e>Fte2C*h^d5i;FZ&Y#!yr_;1*=F&+^
XFUq6i5>xPF2W39cfB%>N_&xuE*>6Z&W22^mRZY!)Vrv8z5;X_v|I*nG+S8s1L=TC2sFi~@86j(!lnb;k}MDu&d%8eq!QTPc%3&vk!+<jp>oeJ6w~|gY7m+`
sakcv9^149Y712h;w{*kQbhKIqK_K2ovZCksl$f~aER)_95GPb2yk*_bEsG!7lhm)8_1c$1lSm;#S@5^4m@&Pva>S(Xw%z3oWw5Ji2k&0@G7Kc=0?Fb?J<-U
lmft5!6jA>n@cLlE<J)|`$Szdh>tqtxP_%G%JPv=<=Qc1=4^-WyV<F!OAvad!$xR+MDTY(?hVFI_T%3#LZLVHgV^C6^8*St+Vl?EC)J#$m+RmVX77A}QytqY
RQ~WTgxd{nny3}kM-~!OeQQnGP=?#SQ6~oLhj<fJYsUa$hawFixH4Nb5p&LU{5rGc+mck+&66kg(O8W2K|`bu!UIq%=Hbzjpo9vPt}>(N$<{0lZ)gX!DG=@@
F*sGG_lhPa_euKjtxf+c!qXsT-Zc$3%0sbnT61HF9|Q}=WnpEUeek(wIriI1O#D%sG()ucJ`Pm0mlfmlv31U&aR&(&m$+AAkc>0aBHWC;RU(M;Z)im8&MY$~
RqV0L@ezq|VOQnV(F!iR6mAb_ELCnE%23Je>In(e=ecQEv*$?)J)V6>p#*AsGL4NHPCR1~<tmIU(kSqYHA2jj#El0WYVov~bv=1e2O?o$5cLKbYUIF~!56}@
wc?;3Rgz%~W3iST#W~c_g7KccJ?mJKkKUg4lA}H--HSiv`l`v35^)l?KG@h}0J>FvGcSN7r7!`^>!9}iXAhx?W`xVUj_aZx>Hfi>74Q4+i007>A9S_3a+}Ky
vYiEAc=^ki@YFp!QKrxs0k)<nPjxM)59#_<E83@ZsY%C+cIsg^@}q*?x4g;7U-KKxIpM8ra0LqLKu_Aj6@Q<eN&8h$rRjC@r2@fkXl_o|bUE><kZ9i1!8W0h
pqeiF5KdXCA-wuKYO9-6H$**&kr@XSdi%Y`Bv+LmU|;f8erVVI1mRO(9dwiiM9DvcFnYaZCVjMeP28=q>WY86z1m3Sq<S5!Dt>8Ubso(oDoi6tLr9RSBr}0k
u@&*l@w?$APt(jv=lUNEwLLKH%qC}V^LH#TP40I_kYW0#sWb;{-h|AW%re|86Aq_D5<#!RSRMps4A=Joaj@ki*&ABX&fxFQ{aX=Fw3n&IE$nljcuTkZZK~44
2SBLmN+iO$Q7(5Lq{o~eS~M2F5d5wQIL<Z)J}Vq#i!eAFXSZ!U*TevvLinZd5@^&lO|4t5K2niMj8|mh=yLx_F!g}*YwykB=25*}@Ul>l5--E7QCW36#Topp
dlb_?AMIaIU#@=;l&-idy*qm3&-rdX=(X<7-Lwh&Bprd>jA1`eQANEQIA+-9q0<)xYBX>nf_5kmZdCbHhi>3pm!k|u6x_^^9H`6V^`q8_B@oqPG{v;%NbA*>
o}Nv={EV*?y5h~$!YV@lS<{%e>`zaO3(|T~idWDmp_y<S{QGWo=U`5z+A+wt<~ErRK%NWSORH$47XZXV0$IShtM`9>ovh{~Lk9cKpk>^5<l4onFse(;kce4U
(nYX_$xddWmLkX7lr*|eUl*s#?#HF28(fq{qFK%24o$*vPsOlm#NwIVSA9+;1@sqlxL*#{4jW6L&>Z=@R+US4wNb~Wu2l-0aCwbq9n-_iGhT>CD0mq!)O|r1
lA^J@A4I$`LW$o2oC)__Q<@ENFymxUkgyhG5?Ripo><mGJOgiuj4Ufvz&;Y`+%mTTpqr<TGT=0?-9d_fc9x2=rdu(g-}UD=Ac#@aDHX+0C`5hD{2zlc-ehID
dYt-bS-i9^CVUY~lkY3(_;(SN2b7u)*vY$MEslPQ^B%Wn(fBD4aoA^fF6Hi?KS!UW{_?>OFJ|P{R^yjTr{4tmv3APvylUh8o71$#eZ+hr%XB|M6V(Lz#%6t`
n5P~sA$Yfo0;Y)3!8A&tlW8SS?El(TvD(-}T@;Ie4|;aSOtI-6+^JG^(L@#~3*hltvTFh5c?*UAheFEe*pvMwqy;!VS?N~R865lIeIKl-OjPAvT~B-pcGn`0
B>VYNay^y7HV!pSFx=9;F9b~l&IOmhyxB`f=-H29&zpX@-xD;39QBK1nzgIYk(0^<Oob=!TpAJSeijYJ;+JD<b~gcF#{(z$GC;rh`MtYs({9>`{7dQG{cslu
gJIIu*SnjY-D71b&f$*h=n25hp%5<dLkBfy6(IF>B0$G#AT2MwIPKc6<7jTQ_hqsl;`cZShpw*$;F4R9b5pzxPOTg-q?Ubaj}+Ga)NU>7iT^MvwBy5ge@~+E
%9Tt4XQ8OTo<WtbUDCICdFa@;Z;v8YngV#9MD2Q12%WP;er48<3jZ@mptu7{qQPKMsk=baEjxkjKkNn$e7%W8?e7U0MzhbYDV5}GB#_6UV7yD$l8(WbZypib
X;7={47P9eBf|t%W`(OosSQzdz}*zrG`T5)JdMhR&YMi(haFj2AavDU9$e6^U&-^bU+O=0)>(OS@#4d~cpIQ=Ekh&Dan|-k;WVBbRIxP&|HFErTmn=2p!cQe
CoXSL!GKuetIfYSW$oj;=!RhFUPB>I%VH(=2Y_DdatQOgL^sQWd1*-Qt-(Y0E(^O1OxEc9PJdgjCp(@t9?e=qH=JoI@TZ&BI_wv+*!l9AyfPWrk@M>I{{_Bi
+70fLKnA=R!@{b0;U1B4Yf%>X4sPEc&t?)^-xPNXybc~cN*90Jt3ZJhr_DeZdZH(y(k_{p#UlItKMQA$rET~_#gC0B?54N`^rV^oXWrX#C8XyCYP|`kSk&wB
eWYc#rfgsg5Rjivc(m<N2AJ|EEMeAsAAA+7!Fpt=qS>!#TX1~QY+sN$7(NJMJB}6+qhYs_z4FbGn1AB+OZ#aMr8&g^MqZ?T_yjm}Vwpm07G`CzW*7lenF%o5
gCeHqTGg=QMDb<&cEeHBB>(jWj=EK{8?tUSCkGs<s{!%mr2I;d8NN~}hWU>W^mkEnxyjzT!;FH}TthP_2i+nqXj9waL;$gLU#?eN!I^4yI^d92X3K5{82vw9
xA-+y+neGnPHX_hXelHeIDnj}I{&fdX+s?(7}?VdlW#n{-j7SR*gaEed;mZcOM5Pbs5@E!1JWq<wn0AN)yvBNtF@k*m+K}ZpLVh0O#YcBh_o1|l68d`S7%uU
6wRSNWerIM{zLrp=QX9Vt<+(yOtW?Vh%%34S`K1S??iA7H3cl3uWr_rwj}h~LK?5bkG53*5rcqD{1}P!0y`QyJm!vU2a#cb)Ud~8snhm8f|T@z)?Z$`afs?p
Ze}|XzH;h`J_b@jv^%%~N6AQghu7^v05p6!?LCiv>J1}gt(f{>b&hA`9rE=Fk0Y6B6t9Bq?@}xuu2N)3UNqWC2nRYEoRP<`bEn<nGE8>mTdgqBi;fJiJqw5E
N8R}=X3XB9v_%KPl$5*}G1|@`uYKSNl$Z|!%st}GxKN{`+NYQ=NEyMR3K@S!;H7c3<T86BbtBOI?-7x`!WiF=%QCopN;`-xRuy-l#UiuS)-}Og_r7tAdpEc4
!>Wk(P=5Fez_5*TEaI0DyVk%qlUCNnFfIzeYUB_%4D@)C^Z6YIr_ryVbg3UUXIAOiV#(x*+k`VPO8p7sHpT|i>qM)t-rA0~0#6&nrdH*QID8btnLdO&N_I4+
Kjvpub@z9>g1Jm1?W_p&+gUv7M406)Z-rg1`>OW!Z-cmy5738yy6Sj;t^U;R+hDY+GbbX6zcnV@pmH^*1a=SCgL56TEPljd4?_fgBp}dTnXUqCH1Q3$SWByt
Dg&rU$@Q51iy=2-`XKc!ieF<_=+k_Mw((+vJV5;}6x#TdT=7xxuNGD84u#$OltA4pG^W=SblS8=&eIc(hM$EO(!)m}^PRl+cDHl@vgCNNHR&k;iP(}1eGB#5
s}=>0;fO~l>`<dRx!kDh)`ewHbMKIV${1*tu9pmYesWB9k0z~Rw)0>5MhlK4940E9m&k^3pRIa4q46O^{VY=w2^3RaWH2IhJl&Id+CiT7!du=LauXqkV)d$U
1=*Jv;Q{l6qU=)(tkAwexef?QzH+!I?jgm(=H3a3z$9OUxvc4^idonyGWdvRuXXXH!g>sj(EwXVNf2NY9y9-5P0?acgiu$KoiUr7T)+kKdNUwYTz(Z+)$cAw
0uR_>`#=m=@VJ%p%G2T7Sa^8SJQX1$)_gG$XXL3AkSyUx3Fpu$GsE0y>qN+DEKpndp?0X(+Ia|f?1+oS6Nj9sY!D6pi7gZ*G*p^U?MUI^XZr})xr;QGF$vi-
Vh%+M3ejbIEqYwMfG!~AxVBHO1GP*%HbrLE^IRceVD;XG`jc%=i2Q-1UDKYs7nMj;1-L{s2~?*-*M_akVQ)p})k)s4jzbs++yh84zQ)>XYVn!^pkKzT9WQjl
0*S_&fOOdBkhqMw`3`Z6X8km@6+9TNj6;l^@JQ165uEb0!USAg4LMA3nqis5qlSj!r9KXwoFME$T###-HJJw>VsS$D+VsRDfukvyWX9phUJ{Hn5I?zC)oDHg
h2oK#i-^2z8PI8gF>Ii80bQELKx!)VECFXIc=|s7HY&$0AZ+LW-?jzD?l?g#d)DMT(sBwn!-sy<PqQ|g-A1Q&8*|T5EAKoRD<lV*W|z-1n?T_&@NToDPJmB_
3{s-FOCZa7ha355J7db5VhGwL@BRjMrcR2z`X+BT0unVXTi2pN_7cGX)@!-=*kqcV|C4fhKNOtPL$G`uLI>nc`o9v?VEpo!$`&+mf$4(d(}Kg(>%C5|`$R*k
Xhwe~f$zccB5JXOyN3QofmHW$ThUq~c&kmdqF99zneXae5!X)d##D{FpwF>xcAy~*hU0*ydGnD`2db=FN$jPbNz{Jl6zf>e-<7SkTjZCLRCul%@)HH!X69`%
{6qy$J%yK21BEdV6uF9kxJb~C+w~OtK<c>pxdp}zz4CVz&DMY+LJotyxLF#c&dk7jXHMT@0M+0}YAt*h+Gf@Anx4fBQ8zgMk1OJuq740_dki;qa~KY4(zk6D
JZjplt>4B#R&4zueDLUT3tw>aOo(r8XkTb}zwLZJyZbY*`=p`~lr@jW%_yb|3*wgZGoh;!zaY%Sea0Z^q7Y<Uk=|d3_}~y<=hVPH<8}`SgW6CvAWU`stZMNv
xzh4y98EiKAYwu~9tkQ-R-%5{NgpD|&O2nC`61yY7mu5KE9jLZzNHA2Q5Q1x_3-T+&R;$xD9)hBv@-S0P_3Q@F)@N{I{6#^%jDu&GoGYrKYfj>@ywSdh%|qx
M`OXofmMJTv|v(do95)wVr)<_{>H^Jh^mN@AKqjY)%7_IwpSWOde$=I3^B*{ypJ;<L`C~z(}A(x6Fj0aVUUnCd}!rm`LDHx{TI_GRRpHOeJHOCUx<zRd-0yv
fAhrBL<q?i${UCYbFyTYW>kF$Ql_D^hGDnSG38jMY-wnFC!CU8GWyShKlIYm(a|*E5@%=`j^ZyeFpI->FY=$bn_G&`lj~Eu3|#q$2Zo2SxrZL1mzZli$(C8Z
0UWxDq}=fH=MDx?hD1uMN|L>;2CEDw79@|F&X`1zDK&(hjf?i{4*fuaE(#GK^-xSvADIK;-+AM?_REAVol?G_W}nF$qv=vsjgxvF$H^VX$~tKq!_0B{WOm0F
0wu;L95lxUnk};jG^uzNnJzBhAq^|&?t#nH*^Daw&(&23Q>gXofU^(fhdY8%rTy?25Ja1hUHZCMJ2yx}HmaKA9HcA>T^RrY7SR#rY`pNQo}k{F2SrylJ6=1C
m5=pNBMXC3HBuT?3DBrF?$<<cL`Mcn)zJX;VwP(Ko{mS5N*>~ni<qVK95%BHz~_3d=7fC(2(jd|H_5=O)nVoPZmgQX`PK^o=euc$-5L>6UM_A~33>Y{`-j9@
YJpA=GQ(txw08%l9jKmIjbcf^_l)_2%%s~>F81Js<whn>O9+X@Skg<<=#txfG8s~F!@PVYI<>~0&yb19ok;5ExWcAo1UKi=E5-~Ocq=m^ALTdaOK4)K`tn*j
!e}Q#M4;_e2eiicXXb*soq#FmiiZH&rNG$fja!5A*<>xcq+APy!qglcr4okE?06uBAXS9|Qwr8Wn^BXhSi76@)@Yu5Uj=q_jWZXwf$65*TC(secETwJ#dPIL
4RH`?o=ckKyvYjDMj*NlhfRL^K95N}3nyFtzDzrQyU7?vfSisFLzdV{6UHq)%D#>!{2M&^s5LYvqhFR*>-Vxra5qz+W^k4|2qdWmJr0_r{FDqnnFu_&EP6gg
&`xN>icPu8`>rLxU#QUTsh0Z$6yO!PNJSgTymvjy$<*1Y4z$1I-t|t*aB0dUZs63?L9!({1#O--jH&L9rC-!Ls5gor+skxfzh{pmrCSPm3XsrtWBO`M<XI7u
Rc$PF^~^|GPIC)bQht%mlytxvkWVm?Kna#|QLNF3+cs)YMXc+Iuf!<Wrtgsz*`$e2EW~Ey6j1zP)ix^+%q&zcSI4aN7}QI|jt)SO$!jlM8gW@y!tE@NHinZ&
IqpRDn=j#=ajxLEPAvwN20Y*BZxC2ziy(5TYY@i>A;f(FY>!+>F1+*MU}Y+dn*XTzi>g>#*!+)Gx0N;9GP;o-5?f?OMml3ia<$s}#B{6k1f_VK$->`XKAM5P
1Kg>YNPdr7Tk`dp0(b`pQGEcJ5~A~UbV`Qu-8MKWIUM1|bI9Fv0`)5K0{AQ@vziJMmdsOy07uE$r`sx`{4A)F*vFv*YGPgcR!F*xuG76b%+BryeFig025Co6
=1IR*b*+UK4-@k0iG(<uTF=iBpiU6|5<GfeEUESB6YHvdH&Ed;UbaYL3YR;5`3ibe)Oca|BRg-(_q-Nh{?}zf!!ZZyg4(2}^}4Lh#)}#*g;hx&&CE*(@MHoH
m!ex142nP9hYtaf<Sjo+Fy`e^-INo+!@EDx`ZVj^OmlZG|FoCVUuwLb^vTcE<YrqBz&*jjB;NLA<E~#(z7a>RS*cpypxL}sta>%Bntz@Pn{Q7L`2CI(d)efv
9HT^Uno$BRK?)x~=dML8fowL5*|x%3S;t(k3xsZdMt76&MKfXLwi4ekqat6P%;fU6Gz=7LtxE=**LXAg?4ITRid1H^NL9yE`~XrCc8Lp~)Nh*{ZU`9rlTd7I
kJc2VK19x<u?gABT=%k^A@d{7tyesTZ7#*)wW+IO__S6iR^)|>*DI9N-3MM%O8d}4R{!E@g$<6mkYVvKdwWqHakmA@9y!2$ln4rU5S5`foS}UAqutYghIV6<
gYjI79=RlPdD)82%lVkgjA9=>ACHhKtIuU2I%hki&Of)m3CftV$ETht*+xEIF$7Y$95NpeBm&5wgvz|mB;R@C&J1A((!3t5ZvkE?$5?p0p5J0cI`p4~Cz^8=
gucWk?a5f8#Sv+*+{wzhj~GZKCY51PW5G*_AcR=@+X;cWt{+Iee9u$fN(9X%nA7i_ff8%$`_fJ^tYzwjb&+V~6Pj+Fu>FtT2qJZ>ZF0_NnA4^Styy|6UiuV4
tk(i3Lp0!b`|4en9_|X6qc>i@*#$kb$5l|6F}GoNF}`F%2AP)48m~{>5B=av@~Zy~Edutc!h5(={{J^8H0P-`vD7S}75ucvZZZ4vx`%viZIR{<*?u?jlGVj%
+B=O-(B5~sk(Z6rRq9J_xk+Nc9szyAq}lIG$(6?!yK0Jop+uLq=Smi|J0bFamsaz4UNz=&(nb?ME^v!*YkO@5#GSq7v7e^;XSrO*b{k&YIVT<O{<PYV(=X-A
vh%V#hpluG5kr`4^ZmLj(7g-|_ufZ5ZYlLK{aSWV`m3#O+Ly@+D_Kra^U&z<OJ#ZeScMN<<+FN5XneYg00J}DWG-0CW>rZT(NIjqY+aDEa?qMcdf?x*D?3#Q
XJ49_<|gvXAxyDDDE6(3#Ud}l*GrI@FP6_<@F?&O*#tYJe_4BH3&Uel)N6*^px?t~IImy^2hO;sI5bPGUPB!uOM-!AX7U<wl*=6X-=f$gie$+}h`;lE2gtx%
N}Q2Z?{wvVb`4*86N<c`b}XN`$XdegpR~9*;x+(RrYY*$xfMdYU(q=ii@gqJ3m}m3tm9jQlr9E$`cUgh3KVYOjs&5T(;Vg=Y6=SW32h1`X5B+2n`m%O%c?v0
fQQaS76^Dm)A9oy3-K`1krRLsiOu-fv9N8c)w7Gi1=Skp2n+(a*>O0uIR=ey<3|_L;ty`sZI6Edj1;xM=Ua5KSxWI!kd(!+fPGa8@^E3pUu~(HYvZ?F4Xn_3
GMF&(%%wXJq<5WB<Av@QpmT$HT<>dEeIH?ku=~sEU<qB3kk$RPbSY9Z9HXaQM>)>ML|9<w>Fd-#rZ1by6}Q5j63tO%BinVGhhJTKUzJ*4N-NZ@rU-<t7rtBk
^wqmM2lK6=d=y$KaAD-(Eix7|xS+;GNT}X4fROD0=YFSfx)6ABZ!P_LuO@wMH^Y!Or(r44q^;*7M=!AU=*jk~w$u#&rZJI@qp6eIZC6^R<5~j7BfEPg`Q~Tz
QGsf@Ej<HkJX}<}kE+E#f*w~HcQKCgsvq$&qZ#Z^k8Bl`-z-ywjVX&{+ow>Jb`=j1l3UAyZH6(nc(Epu$vT}WWl|LR-`$}@Cq}fkM!A?SqBTG+zYF8O-5l?h
!3B$#D5XS&a(N;NF+5d%^|mfmTf30>y=+a#*0YGG%OXSVQhyGv55=*R?KO;A{DwZCI(LX7gy_|Bxp($<tx>T2&~*8sq1=anyjs&eFUH(6i8e=gxz1wC70hO+
knN!pqETlN@gVj0HZi=%gO)7>m@l0;GmZkUOwKFXy5jU%UN*&jml43yi{v5oN~1FgoX`My$J~+RJ$(P17;j9-F-+)oZ5}`EdMY?%*)8LHT`7Msi3NaV&jO>r
`ZKkX0~$Nye8-u<K{@oG3T@iH-Ypq08tLL?(yyt;*BV9S2=7o-Ar=KxCxe2ZIt#DE$;QaJPo19X!zm@{y|CLpd11vjIJS~h3FL)I$2ZVZgPJ(QFp9vTmfm0+
HwuJ2ckIT6#FKsz9wVrE;I>Dti02Q%yjMw9uCFjXxE=f!ZM%q?GjD=kK<FiU3%7cyyj~sm(2DLlckT0vr~}`ZIJ!AWvmjSgw$T#4JvHpzK41$`D=R>{JW%=h
{dh6`aw0M(?+XbCiSMnD{NHW0zd5NlO4CV`H;IBY`3%meaznV}tB$j!fvtJtEW)M75(h>cBUv4+^Cn~kGvkUzAB40rx&v9@qzB|{pKN-eS7M4q`xP|#=Di}c
20S@~%OJ`>v5eFSzSk9w{Z|Bl32YJ_BibIB7NSm-7AJ0Rdz~*3(~8`)<H!&<)F`+H^m3Mg)76$b3N-a{Q9-WJI5prV3IfJY<oJp~n4b0=-RnvyRXYnm_4Ll@
#ZXnjAgR4i7DHil<b_9RWX9L}Jj<YbSp2t8V>RU=ozcTA4b*-V!|!El8YyA&*DEjVXws`Tp*e69_H<H786ZmLK2CrsQJ#DGdy<q#cUuG}i=Vg<_~y#YRzt0j
&C(H3%#P9AXn!Gw)qUk!fOZ^pX~(TNGHWGanKs4-d#xtCLs+Zu`oC!wmDZI%+4&TEdgOcehcSju3&Z6Cb9aFlYOeB8-q0HtXXL8@OzG#m-jvV=;^GpuZX9>g
IBIB<Kx5Q?(gwJ>&-qiD%zmK6={qWS#UIkjHY{QIr3C1BBF#NtV~DkK9Ru2?mxZ*4CDG1es~la>P*fTrTl?Az2Dhgb5(ujR;L%7$Rn)I0T5?lTlUe_U;*S+h
eghppQ5%aO*h+l?)U*h}1i+N8W)wOD)Ca(z3-B_#koy(XDF42f`2#TliQ6)(vD|fNf!K!*7cIuBi&irkD~&Xal-wt?X7>+Ep_ZLQGUYWOOQ(drU(7(&7se;x
G29(QaPHT$uJ1cp7c<2Kd(|H9f@|jR<tS5^IXHy~CQ?HO<T&Zy8F-Szvh_*LOx+mqUHR-l;j%}eE#gK4?yo@gej);GpO(nXJt)vYZ6Yv_1J+Il-HjFyREr5l
r%q#AlC;eAKys06evY0;y*OoID=NmLK5?$1s~$nQc3hdQ`pVM56QTEo7KeA6A^!{FSrB@H5}NFE?!95F_6Wyty^~(AN7{j_vF9k!m}5K1Tjc?LP4NEKyYhS7
$Z0h$U<7igB{F)p>i=d5t4_DRejaySkGgE&4Vz{E(JQ7AoyQka0oI)VkI9EJn;$ceV8k3=v{px5^PRLf3c@T(&E%2}j!gxWA|RTHjg+w3EH%4tu4BbKIMjl}
l#G7AZ;;zJw}ipH8rJchi0K~9K^K7V-ml7m6h7_Vl!*xj-=mymM5X^{WBG0_fq^NR-IjG-_le+;r6fgaodhF+p!M4l<&ofpdn?Ll`nR#f&eBg+4H0X_Ch9g_
Pr&8#9klJt469;L;H&}x|G67uCPNiHWQsdOh3sVi>=)G99DE2VmpC8GUC^G3#&B|9Z~GUoRfC(`yES0Rufc=LyLQzIF+9l<0nj0vK;of)T@=O%f)rBsPA$Vk
D2v*$Awk-S(6?PoL{baf&9`78YIR27*77c>#evA1qiZVlA^<Lj)-_6ue}tR~m>SCyCZe=(_W999RAskd6%pSJti+L;u@+ErC*o+y6*tpq808e?N0~Pjs4iDR
59YQ%b$6#-X5cOK5u|h&JEfE%QZ`NKu3P_7?40)vcOuFKJ|mvucGhds<E)%&02xpYDRO*@lKr{kK1MPH$B<r<$7Ak#@2JlbJS3R(Ovr5v4RkYgFj3~tvd)Db
Nyh~n+cji_iWAEdW`Y98hMS=3pw=!c9mN_=0^&G3O{x}?KJ8j%{PJ4*XLHqfuNqTR*Y1nC$tHiMF7UqU>V(M=+^w#WrRI2h7MBLRR7$FO9EYKfF5?>>6fHE3
G;IatK}IV$4#0i%hj#Z-aiX=pkC}x%vgL|RLzK{SdtiU+L9a%{l>VoXSfGw=P<tCDXZ!i;oD=j>+#fGBpN`>6f}R-809TE*A32z$xW1kSo#hsI^#E$p&|0&V
<U$+(f=9O%I7PIVBX!1uAG=}KC(0gihpvEA?HTV2o8w=;{EpQT^3vQ$S$~^t3z6dM*v5IcY*tVcYQKvtMoJa>H(^)`N6@@iWO<fv^jP-d^wf_S16nxnvIS=X
`F2YEM#9<kQ0?2WTuWENq>A1>B0Z2Nr%Dm4LqsfkG~cahq!pZ5YFP8*_LE*g=2AfNrrgb3VcPmFJap`tqtbC>*TS~TB%0bKXtD+W)xg8x?7_<!lJaBTdnsAf
^A}c-Ocl;xZpLLPMH3txVbWr-_2!FI1+xErbJonbNNmsfQ*wh2e*XUNa0=OUdtc2n$X#oK6fpn?Z<4FmWba(>yxFiTux#Z?3VT_*i5^o8GR{lRbk*-n|Eyo_
_d%h>m7!z_LF*|k{UGoJE693WXRPLOaJ=@+R`<ae{aWCh4x7$^FTK8&3>r+U)qb53U7Gg5?I;m->8=hb>5}z$Da5g9{^F3mny6ah_FtGJ14L2<q|G5XEK@ZD
nWfcwtWt@~5QtSf%Wm26L4G1MIQq|MTB`t+keKkpSh{auskzHt2v+e}na|J^+5oQ*d^v(}QBOPb+H;Dr@onpN2l~ITxGi>or^BfZX*tHZ+z+q6G`75XHt6Ze
K^el|JtqYU^Z<1FfD?sT-HLa4bCtv7uQ?YFx^nwgM@{eU5SSqg`?`{<mecC|dLBK%VDKm*Gw*2J6Ys4TNoFpbN!dyq17u?)XbWLqLG7TfTRjnAnzBo3H@%&y
AZ0LrR_4t>QJ~dcMZC)^$LBY+b%Ai*{FHRss`y3Tn(UtXBqqzQyI4)49({k>l09$Qx)+8)jTLEe^>eL)(p3AnL5T26qEiy#Z+wdLvNX^YFU^1jyXlc}0hLrh
NSML~dOg?wfhb0sz5a*7T5M?7KCgVZgAWefud1&$KVFx3J&m3Yc%`cdV8BK^_Ql1$^>AI<%~N*G{=NVJg0cNJK{&TO00G>{1kn2!)TslRvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/treino_v2.csv')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/treino_gpu.py', '--rodada', RODADA, '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())   # não levanta erro: o que ficou pronto continua na pasta de saída

In [ ]:
import json, os
m = os.path.join(SAIDA, 'manifesto.json')
if os.path.exists(m):
    m = json.load(open(m))
    for nome, c in m['configs'].items():
        print(f"{c['estado']:14s} {nome:46s} serviço: {c.get('servico', {}).get('estado')}")
    print('paridade (sexta execução: R4c e H0c refeitas × quinta):', m.get('paridade_d1'))
    print('recorte pelo prato:', {k: v for k, v in (m.get('prato') or {}).items() if k in ('regra', 'consultas', 'reprovados', 'n5k_ok')})
    print('confiança:', m.get('confianca'))
    print('bases externas:', {k: v.get('situacao') for k, v in (m.get('dominios') or {}).items()})
    print('tamanho da saída:', m.get('tamanho_saida'))
print(sorted(os.listdir(SAIDA)))